# 파이썬을 활용한 수치해석(Numerical Analysis with Python) 이인호 (북스힐, 2026)

## Chapter 1 계산과 수학   파이썬을 활용한 수치해석(Numerical Analysis with Python) 이인호 (북스힐, 2026)

In [ ]:
import numpy as np
# float64 타입의 정보 객체를 가져온다.
info = np.finfo(np.float64)
# 유효 십진수 자릿수
dig = info.precision
# 기계 엡실론
epsilon = info.eps
print(f"Numpy float64 Precision (precision): {dig}")
# 출력 예시: Numpy float64 Precision (precision): 15
print(f"Numpy float64 Epsilon (eps): {epsilon}")
# 출력 예시: Numpy float64 Epsilon (eps): 2.220446049250313e-16

In [ ]:
def simple_sum(data):
    """일반적인 부동 소수점 합산"""
    total = 0.0
    for x in data:
        total += x
    return total	
def kahan_sum(data):
    """카한 합산 알고리즘을 사용한 합산"""
    total = 0.0  # 최종 합계
    c = 0.0      # 누적 오차 (보정 변수)
    for x in data:
        y = x - c             # 1. 현재 값에서 이전 오차를 뺀 값
        t = total + y         # 2. 중간 합계
        c = (t - total) - y   # 3. 새로운 오차 계산 (y를 잃은 부분)
        total = t             # 4. 합계 업데이트
    return total
# 테스트 데이터 생성
# 매우 큰 수(1.0)와 매우 작은 수(0.0000001)를 반복적으로 더하여 오차를 유발
N = 100000
large_num = 1.0
small_num = 1e-7
data = [large_num] * N + [small_num] * N
# 계산 및 비교
sum_actual = large_num * N + small_num * N
sum_simple = simple_sum(data)
sum_kahan = kahan_sum(data)
print(f"--- 카한 합산 알고리즘 비교 ---")
print(f"N: {N}회")
print(f"더하는 값의 구성: 1.0 (N회) + 1e-7 (N회)")
print("-" * 30)
print(f" 실제 수학적 합계: {sum_actual}")
print(f" 일반 합산 결과: {sum_simple}")
print(f" 카한 합산 결과: {sum_kahan}")
# 오차 계산
error_simple = abs(sum_actual - sum_simple)
error_kahan = abs(sum_actual - sum_kahan)
print("-" * 30)
print(f"일반 합산 오차: {error_simple:.10e}")
print(f"카한 합산 오차: {error_kahan:.10e}")

In [ ]:
def pairwise_summation(data):
    """
    재귀적인 방식으로 쌍별 합산을 수행하는 함수.
    :param data: 합산할 부동 소수점 숫자의 리스트
    :return: 계산된 합계
    """
# 1. 종료 조건 (Base Case): 리스트에 원소가 1개 이하일 경우
    if not data:
    # 빈 리스트일 경우 합계는 0
        return 0.0
    if len(data) == 1:
    # 원소가 1개일 경우 그 값 자체가 합계
        return data[0]
# 2. 분할 (Divide): 리스트를 중앙을 기준으로 두 절반으로 나눔
    mid = len(data) // 2
    left_half = data[:mid]
    right_half = data[mid:]
# 3. 정복 및 결합 (Conquer and Combine): 각 절반을 재귀적으로 합산하여 결과를 합침
# 이 과정에서 비슷한 크기의 중간 합끼리 더해져 오차를 줄임
    left_sum = pairwise_summation(left_half)
    right_sum = pairwise_summation(right_half)
    return left_sum + right_sum
# --- 예제 사용 ---
# 수치 오차가 발생하기 쉬운 예시 데이터
# 10000000.0 이라는 큰 값과 0.0000001 이라는 작은 값이 섞여 있을 때
data_large_small = [10000000.0] * 5 + [0.0000001] * 5 
# 일반적인 합산 (파이썬의 기본 sum() 함수는 C 구현에 가까우며,
# 이는 내부적으로 순차적 합산이나 최적화된 방법으로 작동)
naive_sum = sum(data_large_small)
# 쌍별 합산 적용
pairwise_result = pairwise_summation(data_large_small)
# --- 결과 출력 ---
print(f"데이터: {data_large_small}")
print("-" * 40)
print(f"원소 개수: {len(data_large_small)}")
print(f"기대되는 정확한 합계 (수동 계산): 50000000.0000005") # 5 * 10^7 + 5 * 10^-7
print("-" * 40)
print(f"**순차적 합산 (sum()) 결과:** {naive_sum}")
print(f"**쌍별 합산 결과 (Pairwise):** {pairwise_result}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.patches as mpatches
# 목표: 근을 분리하여 W=0, W=1, W=3 단계를 모두 보여줌
# 설정한 근: z = 0.5 (내부), z = 2, z = -2 (외부)
# 식: P(z) = (z - 0.5)(z - 2)(z + 2) = (z - 0.5)(z^2 - 4) 
#          = z^3 - 0.5z^2 - 4z + 2
def P(z):
    return z**3 - 0.5*z**2 - 4*z + 2
def P_prime(z):
# P(z) 미분: 3z^2 - z - 4
    return 3*z**2 - 1.0*z - 4.0
def calculate_winding_number(R, num_points=6000):
    theta = np.linspace(0, 2*np.pi, num_points)
    d_theta = theta[1] - theta[0]
    z = R * np.exp(1j * theta)
    dz = 1j * z * d_theta
# 적분: (1/2pi*i) * integral(P'/P dz)
    integrand = P_prime(z) / P(z)
    winding_number = np.sum(integrand * dz) / (2j * np.pi)
    return int(np.round(winding_number.real))
def add_arrows_to_curve(ax, x, y, color, num_arrows=5, arrow_size=15, alpha=1.0):
    if len(x) < num_arrows + 2: return
    indices = np.linspace(0, len(x) - 2, num_arrows + 1, dtype=int)[:-1]
    for idx in indices:
        dx = x[idx+1] - x[idx]
        dy = y[idx+1] - y[idx]
        if np.sqrt(dx**2 + dy**2) < 1e-9: continue
        angle = np.rad2deg(np.arctan2(dy, dx))
        ax.plot(x[idx], y[idx], marker=(3, 0, angle - 90), markersize=arrow_size,
            color=color, markeredgewidth=0, alpha=alpha, linestyle="None")
def visualize_winding_number_1_case():
    # 반지름 설정:
    # 0.3: 근 없음 (W=0)
    # 1.3: 안쪽 근(0.5) 하나만 포함 (W=1) -> 이 부분이 추가됨
    # 3.0: 모든 근(0.5, 2, -2) 포함 (W=3)
    radii = [0.3, 1.3,  3.0]
    fig, ax = plt.subplots(figsize=(10, 10))
    colors = cm.plasma(np.linspace(0, 0.9, len(radii))) # 색상 변경
    theta = np.linspace(0, 2*np.pi, 4000)
    print(f"{'Radius (R)':<12} | {'Calc. Winding Number (W)':<25}")
    print("-" * 40)
    for i, r in enumerate(radii):
        # 1. 계산
        wn = calculate_winding_number(r)
        print(f"{r:<12.3f} | {wn:<25}")
        # 2. 데이터 생성
        z_circle = r * np.exp(1j * theta)
        w_curve = P(z_circle)
        # 3. 그리기
        label = f'|z|={r:.2f}  (W={wn})'
        # 선 굵기와 투명도 조절
        lw = 2 + 0.5*i
        alpha = 0.6 + 0.3*(i/len(radii))
        ax.plot(w_curve.real,w_curve.imag,color=colors[i],lw=lw,alpha=alpha,label=label)
        # 화살표 추가 (W=1인 경우를 잘 보여주기 위해 개수 조절)
        n_arrows = 6 if wn > 0 else 4
        add_arrows_to_curve(ax, w_curve.real, w_curve.imag, color=colors[i], 
            num_arrows=n_arrows, arrow_size=15, alpha=min(1.0, alpha+0.1))
    # 그래프 꾸미기
    ax.axhline(0, color='gray', linewidth=1, linestyle='--')
    ax.axvline(0, color='gray', linewidth=1, linestyle='--')
    ax.plot(0, 0, 'ro', markersize=12, label='Origin (Target)', zorder=10)
    ax.set_title(r"Visualizing W.N. : $0 \rightarrow 1 \rightarrow 3$", fontsize=16)
    ax.set_xlabel("Re(P(z))")
    ax.set_ylabel("Im(P(z))")
    ax.grid(True, linestyle=':', alpha=0.5)
    # 텍스트 박스: 사용된 다항식 설명
    text_str = (
        r"$P(z) = (z-0.5)(z^2-4)$" + "\n"
        "Roots at: $0.5, 2.0, -2.0$")
    props = dict(boxstyle='round', facecolor='white', alpha=0.8)
    ax.text(0.05, 0.95, text_str, transform=ax.transAxes, fontsize=12,
    verticalalignment='top', bbox=props)
    ax.legend(loc='lower right', fontsize=11)
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.savefig('winding.png')
    plt.show()
if __name__ == "__main__":
    visualize_winding_number_1_case()

In [ ]:
import numpy as np
# 1. 근을 찾고자 하는 함수 f(x)
def f(x):
    return x**3 - 6 * x**2 + 11 * x - 6
# 2. 함수 f(x)의 미분 함수 f'(x)
def f_prime(x):
    return 3 * x**2 - 12 * x + 11
def hybrid_newton_bisection(f, f_prime, a, b, tol=1e-6, max_iter=100):
    """
    뉴턴-랩슨법과 이분법을 결합한 하이브리드 해 찾기 함수이다.
    :param f: 함수 f(x)
    :param f_prime: 함수 f'(x)
    :param a, b: 근을 포함하는 초기 구간 (f(a)와 f(b)의 부호가 달라야 함)
    :return: 근사 해
    """
    if f(a) * f(b) >= 0:
        print(" 오류: f(a)와 f(b)의 부호가 달라야 한다.")
        return None
    a_n, b_n = a, b
    x_n = (a + b) / 2 # 초기 추정값은 구간의 중간으로 시작
    print(f"--- 하이브리드 시작 (초기 구간 [{a:.6f}, {b:.6f}], x0={x_n:.6f}) ---")
    for i in range(max_iter):
        f_x = f(x_n)
        # 1. 수렴 조건 확인
        if abs(f_x) < tol or (b_n - a_n) < tol:
            print(f" {i+1}번째 반복에서 수렴. 근: {x_n:.10f}")
            return x_n
        # 2. 뉴턴-랩슨법 시도
        f_prime_x = f_prime(x_n)
        # 미분값이 0에 가깝거나 뉴턴법으로 계산한 x_next가 불안정할 때 이분법으로 전환
        if abs(f_prime_x) < 1e-10:
            is_newton_safe = False
        else:
            x_next_newton = x_n - f_x / f_prime_x
            # 뉴턴법 안전성 검사:
            # A. x_next가 현재 구간 [a_n, b_n] 안에 있는지
            # B. f(x_next)의 값이 f(x_n)보다 작아졌는지 (수렴 방향인지)
            is_in_range = (x_next_newton > a_n) and (x_next_newton < b_n)
            is_better = abs(f(x_next_newton)) < abs(f_x)  
            is_newton_safe = is_in_range and is_better
        # 3. 다음 단계 결정 및 구간 업데이트
        if is_newton_safe:
            # 뉴턴법 성공: 뉴턴법 결과 사용
            x_next = x_next_newton
            print(f"반복 {i+1} (Newton): x = {x_next:.6f}")
        else:
            # 뉴턴법 실패 또는 불안정: 이분법 강제 실행
            x_next = (a_n + b_n) / 2
            print(f"반복 {i+1} (Bisection): x = {x_next:.6f} [Fallback]")
       # 4. 구간 [a_n, b_n] 업데이트 (이분법 로직 적용)
        f_next = f(x_next)
        if f(a_n) * f_next < 0:
            b_n = x_next # 해는 [a_n, x_next]에 있음
        else:
              a_n = x_next # 해는 [x_next, b_n]에 있음
        x_n = x_next
    print(" 최대 반복 횟수에 도달했다.")
    return x_n  
# --- 실행 예제 ---
# 근 x=1을 찾기 위해 구간 [0.5, 1.5] 사용 (f(0.5)=-3.375, f(1.5)=-1.125) -> 부호가 같아 실패!
# 근 x=1을 찾기 위해 구간 [0.5, 1.2] 사용 (f(0.5)=-3.375, f(1.2)=0.048) -> 성공!
root_hybrid = hybrid_newton_bisection(f, f_prime, a=0.6, b=1.2)
print(f"\n최종 찾은 근 (하이브리드): **{root_hybrid}**")
root_hybrid = hybrid_newton_bisection(f, f_prime, a=1.2, b=2.2)
print(f"\n최종 찾은 근 (하이브리드): **{root_hybrid}**")
root_hybrid = hybrid_newton_bisection(f, f_prime, a=2.2, b=3.2)
print(f"\n최종 찾은 근 (하이브리드): **{root_hybrid}**")

In [ ]:
from scipy.optimize import root_scalar
import numpy as np
def f(x):
	return x**3 - 6 * x**2 + 11 * x - 6
	# --- 브렌트의 방법 적용 ---
	# 근 x=3을 찾기 위해 [2.5, 3.5] 구간 설정
try:
	result = root_scalar(f, bracket=[2.5, 3.5], method='brentq')
	print("--- SciPy를 이용한 Brent's method ---")
	print(f"찾은 근: **{result.root:.10f}**")
	print(f"수렴 여부: {result.converged}")
	print(f"반복 횟수: {result.iterations}회")
except ValueError as e:
	print(f"오류: {e}")	

In [ ]:
import numpy as np
# 계수: [x^3, x^2, x^1, x^0] 순서
coefficients = [6, -4, 7, -19]
x_value = 3
# numpy.polyval을 사용하여 계산
result = np.polyval(coefficients, x_value)
print(f"P({x_value}) = {result}") 
# 출력: P(3) = 128 
# 계산 과정 (호너의 방법): ((6 * 3 - 4) * 3 + 7) * 3 - 19 = 128
def horner_eval(coefficients, x):
	"""
	호너의 방법을 사용하여 다항식의 값을 계산한다.
	:param coefficients: (가장 높은 차수부터: [a_n, a_{n-1}, ..., a_0])
	:param x: 값을 평가할 x 값
	:return: P(x) 값
	"""
	result = 0
	# 리스트를 가장 높은 차수부터 순회하며 호너 규칙 적용
	for coeff in coefficients:
	    result = result * x + coeff
	return result
# 예시: P(x) = 6x^3 - 4x^2 + 7x - 19 를 x=3에서 평가
coeffs = [6, -4, 7, -19]
x_val = 3
print(f"P({x_val}) = {horner_eval(coeffs, x_val)}") # 출력: 128

In [ ]:
import numpy as np
# 1. 다항식의 값을 계산하는 함수 (Horner's Method 등을 쓸 수도 있지만 여기선 직관적으로 구현)
def polynomial_value(coeff, x):
    """
    coeff: 다항식의 계수 리스트 (최고차항부터 상수항 순서, 예: [1, -6, 11, -6])
    x: 값을 계산할 지점
    """
    result = 0
    degree = len(coeff) - 1
    for i in range(len(coeff)):
        result += coeff[i] * (x ** (degree - i))
    return result
# 2. Durand-Kerner 알고리즘 구현
def durand_kerner(coeff, max_iter=100, tol=1e-6, initial_roots=None):
    """
    coeff: 다항식 계수
    max_iter: 최대 반복 횟수
    tol: 허용 오차 (수렴 조건)
    initial_roots: 초기 추정값 (없으면 자동으로 복소수 평면에 원형으로 배치)
    """
    n = len(coeff) - 1  # 다항식의 차수    
    # 계수 정규화 (최고차항 계수로 나눔) - 모닉 다항식으로 변환
    coeff = np.array(coeff, dtype=complex)
    if coeff[0] != 1:
        coeff = coeff / coeff[0]
    # 초기값 설정 (매우 중요)
    if initial_roots is not None:
        roots = np.array(initial_roots, dtype=complex)
    else:
        # 0을 중심으로 하는 단위 원 위에 균등하게 배치 (복소수 초기값)
        # 초기값이 겹치지 않게 하는 것이 핵심이다.
        roots = np.exp(2j * np.pi * np.arange(n) / n)
        # 0.4 + 0.9j 같은 오프셋을 주어 대칭성을 깨는 테크닉도 자주 쓰인다.
        roots = roots ** np.random.rand() +np.random.random()-0.5  
    print(f"Initial roots: {roots}")
    # 반복 계산
    for iter_count in range(max_iter):
        new_roots = np.copy(roots)
        for i in range(n):
            # 분자: P(x_i)
            numerator = polynomial_value(coeff, roots[i])
            # 분모: product(x_i - x_j) for j != i
            denominator = 1.0
            for j in range(n):
                if i != j:
                    denominator *= (roots[i] - roots[j])
            # 업데이트 공식 적용
            new_roots[i] = roots[i] - numerator / denominator
        # 수렴 여부 확인 (변화량이 tol보다 작으면 종료)
        if np.all(np.abs(new_roots - roots) < tol):
            print(f"Converged after {iter_count+1} iterations.")
            return new_roots
        roots = new_roots
    print("Max iterations reached.")
    return roots
# --- 실행 예제 ---
# 다항식: x^3 - 6x^2 + 11x - 6 = 0
# 정답 근: 1, 2, 3
coefficients = [1, -6, 11, -6]
found_roots = durand_kerner(coefficients)
print("-" * 30)
print("찾은 근 (Calculated roots):")
for r in found_roots:
    print(f"{r:.6f}") # 복소수 형태로 출력됨 (허수부가 0에 가까우면 실수 근)
print("-" * 30)
print("실제 정답과 비교 (Real roots): 1, 2, 3")

In [ ]:
import numpy as np
from numpy.polynomial import Polynomial
# 다항식: x^3 - 3x^2 + 3x - 5 = 0
# 주의: numpy.roots와 달리 계수 순서가 [상수항, 1차항, ..., 최고차항] 순서이다.
p = Polynomial([-5, 3, -3, 1]) 
roots = p.roots()
print("Numpy Polynomial Roots:", roots)
from sympy import symbols, solve
x = symbols('x')
# 다항식 정의
equation = x**3 - 3*x**2 + 3*x - 5
# solve 함수 사용
roots = solve(equation, x)
print("Sympy Exact Roots:")
for root in roots:
    print(root.evalf()) # 수치로 보고 싶으면 evalf()
# 그냥 print(root)를 하면 복잡한 수식 형태(세제곱근 포함)로 출력된다.
import mpmath
# 정밀도를 50자리로 설정
mpmath.mp.dps = 50
# 다항식 계수 (최고차항부터)
coeffs = [1, -3, 3, -5]
# polyroots 함수 사용
roots = mpmath.polyroots(coeffs)
print("Mpmath High-Precision Roots:")
for r in roots:
    print(r)
import scipy.linalg
import numpy as np
# x^3 - 3x^2 + 3x - 5 = 0
coeffs = [1, -3, 3, -5]
# 1. 모닉 다항식으로 변환 (최고차항 계수로 나눔)
coeffs = np.array(coeffs, dtype=float)
coeffs /= coeffs[0]
a = coeffs[1:] # 최고차항 제외한 나머지 계수 (부호 반대로 써야 함에 주의)
n = len(a)
# 2. 동반 행렬 구성
C = np.zeros((n, n))
# 서브 대각선에 1 채우기
for i in range(n-1):
    C[i+1, i] = 1.0
# 마지막 열에 계수 채우기 (부호 반대)
C[:, -1] = -a[::-1] # 계수 순서를 뒤집어서 넣어야 함 (상수항이 맨 위)
# 3. 고유값 계산
eigenvalues = scipy.linalg.eigvals(C)
print("Eigenvalues (Roots):", eigenvalues) 

In [ ]:
def calculate_sqrt2_by_iterations(iterations, initial_guess=1.0):
    """
    뉴턴-랩슨 방법(헤론의 공식)을 사용하여 2의 제곱근을 계산한다.
    
    Args:
        iterations (int): 반복 횟수.
        initial_guess (float): 초기 추측값 (기본값: 1.0).
        
    Returns:
        float: 근사된 2의 제곱근 값.
    """
    x = initial_guess
    
    print(f"--- {iterations}회 반복 계산 과정 ---")
    print(f"초기 추측값 (x0): {x}")
    
    for i in range(iterations):
        # 헤론의 공식: x_next = 0.5 * (x_n + 2 / x_n)
        x_next = 0.5 * (x + 2 / x)
        
        # 현재 근사값과 다음 근사값의 차이를 사용하여 수렴도를 확인 가능
        error = abs(x_next - x) 
        
        x = x_next
        print(f"반복 {i+1}: x = {x:.15f}, 오차 변화량 = {error:.15f}")
        
    return x

# 5회 반복으로 계산
sqrt2_approx = calculate_sqrt2_by_iterations(iterations=5)
print(f"\n최종 근사값: {sqrt2_approx:.15f}")

In [ ]:
import math

def original_formula(x):
    """ 
    원래 수식: f(x) = sqrt(x + 1) - 1
    x가 0에 가까울 때 재앙적 상쇄 발생
    """
    return math.sqrt(x + 1) - 1

def stable_formula(x):
    """
    변형 수식: f(x) = x / (sqrt(x + 1) + 1)
    뺄셈을 덧셈으로 대체하여 상쇄 오차 회피
    """
    return x / (math.sqrt(x + 1) + 1)

# 테스트 값: 0에 매우 가까운 수
x_test = 1e-16

# 계산 및 비교
result_original = original_formula(x_test)
result_stable = stable_formula(x_test)

# 실제 수학적 값 (테일러 급수 근사 또는 더 높은 정밀도로 계산)
# f(x) ≈ x/2 - x^2/8 + ... 이므로, x가 매우 작으면 x/2가 근사값
result_actual_approx = x_test / 2.0

print(f"--- 재앙적 상쇄 회피 비교 ---")
print(f"테스트 x 값: {x_test}")
print(f"실제 근사 값 (x/2): {result_actual_approx:.18e}")
print("-" * 30)
print(f" 원래 수식 결과 (상쇄 발생): {result_original:.18e}")
print(f" 변형 수식 결과 (안정적): {result_stable:.18e}")

# 오차 계산
error_original = abs(result_actual_approx - result_original)
error_stable = abs(result_actual_approx - result_stable)

print("-" * 30)
print(f"원래 수식 오차: {error_original:.10e}")
print(f"변형 수식 오차: {error_stable:.10e}")

from scipy.optimize import brentq

def func(x):
    return x**3 - 2*x - 5

# [2, 3] 구간에서 근을 찾음
root = brentq(func, 2, 3) 
print(f"근: {root}") # 출력: 근: 2.0945514815423265

from scipy.optimize import minimize_scalar

# 함수 정의 (예시)
def func(x):
    return (x - 2)**2 + 10

# 기존 오류 코드 (예상):
# result = minimize_scalar(func, bounds=(0, 5), method='brent') #  ValueError 발생!

# 해결: bounds를 제거한다.
result = minimize_scalar(func, method='brent')

print(f"최솟값 x 위치: {result.x}")

from scipy.optimize import minimize_scalar

# 함수 정의 (예시)
def func(x):
    return (x - 2)**2 + 10

# 해결: method를 'bounded'로 변경하고 bounds를 유지한다.
result = minimize_scalar(func, bounds=(0, 5), method='bounded') 

print(f"최솟값 x 위치: {result.x}")

In [ ]:
import numpy as np
a = np.float32(1e20)
b = np.float32(-1e20)
c = np.float32(3.14)

print((a + b) + c)  # 3.14?
print(a + (b + c))  # 0.0?

def kahan_sum(xs):
    s = 0.0
    c = 0.0
    for x in xs:
        y = x - c
        t = s + y
        c = (t - s) - y
        s = t
    return s

import numpy as np
# Generate an array of numbers with high precision issues
xs = np.array([1e16, 1, -1e16])
# Standard summation
standard_sum = np.sum(xs)
# Kahan summation
kahan_result = kahan_sum(xs)
# Display results
print(f"Standard sum: {standard_sum}")
print(f"Kahan sum: {kahan_result}")

# Generate a large array of numbers with small increments
xs_large = np.random.rand(1_000_000) * 1e-10
# Standard summation
standard_large_sum = np.sum(xs_large)
# Kahan summation
kahan_large_result = kahan_sum(xs_large)
# Display the results
print(f"Standard large sum: {standard_large_sum}")
print(f"Kahan large sum: {kahan_large_result}")

# 부동 소수점 덧셈의 한계를 보여주는 예제
a = 0.1
b = 0.2
c = 0.3
# 0.1과 0.2를 더한 결과
sum_ab = a + b
print(f"a: {a}")
print(f"b: {b}")
print(f"a + b: {sum_ab}")
print(f"예상 결과 (c): {c}")

# 예상 결과와 실제 결과가 다른지 확인
# True가 나와야 할 것 같지만, 실제로는 False가 나온다.
print(f"a + b == c: {sum_ab == c}")
print("\n--- 실제 값 비교 ---")
# 실제 저장된 sum_ab의 값은 0.3보다 아주 약간 크다.
print(f"a + b의 실제 값: {sum_ab}")
print(f"c의 실제 값: {c}")

In [ ]:
def pairwise_summation(data):
    """
    재귀적인 방식으로 쌍별 합산을 수행하는 함수.
    
    :param data: 합산할 부동 소수점 숫자의 리스트
    :return: 계산된 합계
    """
    
    # 1. 종료 조건 (Base Case): 리스트에 원소가 1개 이하일 경우
    if not data:
        # 빈 리스트일 경우 합계는 0
        return 0.0
    if len(data) == 1:
        # 원소가 1개일 경우 그 값 자체가 합계
        return data[0]

    # 2. 분할 (Divide): 리스트를 중앙을 기준으로 두 절반으로 나눔
    mid = len(data) // 2
    left_half = data[:mid]
    right_half = data[mid:]
    
    # 3. 정복 및 결합 (Conquer and Combine): 각 절반을 재귀적으로 합산하여 결과를 합침
    # 이 과정에서 비슷한 크기의 중간 합끼리 더해져 오차를 줄임
    left_sum = pairwise_summation(left_half)
    right_sum = pairwise_summation(right_half)
    
    return left_sum + right_sum

# --- 예제 사용 ---
# 수치 오차가 발생하기 쉬운 예시 데이터
# 10000000.0 이라는 큰 값과 0.0000001 이라는 작은 값이 섞여 있을 때
data_large_small = [10000000.0] * 5 + [0.0000001] * 5 

# 일반적인 합산 (파이썬의 기본 sum() 함수는 C 구현에 가까우며,
# 이는 내부적으로 순차적 합산이나 최적화된 방법으로 작동)
naive_sum = sum(data_large_small)

# 쌍별 합산 적용
pairwise_result = pairwise_summation(data_large_small)

# --- 결과 출력 ---
print(f"데이터: {data_large_small}")
print("-" * 40)
print(f"원소 개수: {len(data_large_small)}")
print(f"기대되는 정확한 합계 (수동 계산): 50000000.0000005") # 5 * 10^7 + 5 * 10^-7
print("-" * 40)
print(f"**순차적 합산 (sum()) 결과:** {naive_sum}")
print(f"**쌍별 합산 결과 (Pairwise):** {pairwise_result}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

def add_arrows_to_curve(ax, x, y, color, num_arrows=5, arrow_size=15, alpha=1.0):
    """
    곡선 데이터(x, y)를 받아 일정한 간격으로 화살표 마커를 추가하는 함수
    """
    # 점이 너무 적으면 화살표를 그리지 않음
    if len(x) < num_arrows + 2:
        return

    # 곡선의 인덱스를 균등하게 나눔
    indices = np.linspace(0, len(x) - 2, num_arrows + 1, dtype=int)[:-1]
    
    for idx in indices:
        # 현재 점과 다음 점 사이의 벡터 계산 (접선 방향)
        dx = x[idx+1] - x[idx]
        dy = y[idx+1] - y[idx]
        
        # 벡터 길이가 너무 짧으면 건너뜀 (계산 오류 방지)
        if np.sqrt(dx**2 + dy**2) < 1e-9:
            continue

        # 각도 계산 (라디안 -> 각도)
        angle = np.rad2deg(np.arctan2(dy, dx))
        
        # 화살표 마커 그리기
        # marker=(3, 0, angle - 90)은 회전된 삼각형
        # alpha 값을 곡선보다 약간 높여서 더 선명하게 보이게 함
        arrow_alpha = min(1.0, alpha + 0.2) 
        ax.plot(x[idx], y[idx], marker=(3, 0, angle - 90), markersize=arrow_size,
                color=color, markeredgewidth=0, alpha=arrow_alpha, linestyle="None")

def generate_book_cover_art():
    # 1. 다항식 정의: P(z) = z^3 - z + 1 (Winding number 3 -> 0)
    def P(z):
        return z**3 - 1.0*z + 1

    # 2. 설정
    # 고해상도 출력을 위한 DPI 설정
    fig = plt.figure(figsize=(12, 12), dpi=300)
    # 축을 포함한 모든 프레임을 제거하기 위해 전체 영역을 axes로 사용
    ax = fig.add_axes([0, 0, 1, 1])

    # 데이터 포인트 밀도 및 반지름 설정
    theta = np.linspace(0, 2*np.pi, 3000) # 부드러운 곡선을 위해 점 개수 증가
    # 반지름 범위를 조절하여 시각적으로 균형 잡힌 분포를 만듦
    radii = np.concatenate([
        np.linspace(0.1, 0.8, 2),   # 안쪽은 조금 더 촘촘하게
        np.linspace(1.0, 1.3, 2)    # 바깥쪽
    ])
    
    # 색상 맵 설정 (안쪽: 밝음 -> 바깥쪽: 어두움)
    colors = cm.viridis_r(np.linspace(0, 1, len(radii)))

    # 3. 그래픽 요소 그리기
    for i, r in enumerate(radii):
        z_circle = r * np.exp(1j * theta)
        w_curve = P(z_circle)
        
        x_vals = w_curve.real
        y_vals = w_curve.imag
        
        # 바깥쪽으로 갈수록 더 선명하고 두껍게
        progress = i / (len(radii) - 1)
        alpha_val = 0.5 + 0.5 * progress
        linewidth = 1.5 + 1.5 * progress
        
        # 메인 곡선 플롯
        ax.plot(x_vals, y_vals, color=colors[i], lw=linewidth, alpha=alpha_val)
        
        # 화살표 추가 (적절한 간격으로 선택적 추가)
        # 너무 복잡해지지 않게 일부 곡선에만 화살표를 추가하거나 개수를 조절
        if i % 2 == 0 or i == len(radii) - 1: 
            num_arrows = int(4 + 6 * progress) # 바깥쪽일수록 화살표 개수 증가
            arrow_size = 10 + 8 * progress     # 바깥쪽일수록 화살표 크기 증가
            add_arrows_to_curve(ax, x_vals, y_vals, color=colors[i], 
                                num_arrows=num_arrows, arrow_size=arrow_size, alpha=alpha_val)

    # 원점 (Root가 있어야 할 곳) 강조 - 핵심 포인트
    # 약간 빛나는 효과를 위해 겹쳐서 그리기
    ax.plot(0, 0, 'o', color='red', markersize=18, alpha=0.6, markeredgewidth=0)
    ax.plot(0, 0, 'o', color='darkred', markersize=10, alpha=1.0, markeredgewidth=0)

    # 4. 모든 텍스트 및 축 요소 제거 (책 표지용)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.axis('off') # 축, 테두리, 배경 모두 숨김

    # 비율 고정 및 뷰 제한 설정 (그림이 잘리지 않게)
    ax.set_aspect('equal')
    
    # 데이터 범위에 맞춰 뷰 한계 설정 (약간의 여백 포함)
    all_x = []
    all_y = []
    for r in radii:
        w = P(r * np.exp(1j * theta))
        all_x.extend(w.real)
        all_y.extend(w.imag)
    max_val = max(np.max(np.abs(all_x)), np.max(np.abs(all_y))) * 1.1
    ax.set_xlim(-max_val, max_val)
    ax.set_ylim(-max_val, max_val)

    # 배경을 투명하게 설정 (저장 시 적용됨)
    fig.patch.set_alpha(0.0)
    plt.savefig('backfigure.png')
    plt.show()
    # 이미지를 저장하려면 아래 주석을 해제하세요. 투명 배경 PNG로 저장된다.
    # fig.savefig("fundamental_theorem_cover_art.png", transparent=True, dpi=300, bbox_inches='tight', pad_inches=0)

if __name__ == "__main__":
    generate_book_cover_art()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def generate_clean_art(output_filename='clean_cover_art.png', dpi=300):
    """
    텍스트와 수식을 모두 제거한 순수 그래픽 버전 생성
    """
    print("이미지 생성 중... (텍스트 제거 버전)")

    # 1. 고해상도 캔버스 설정
    width, height = 2000, 2600 
    
    # 복소평면의 범위 설정
    x_min, x_max = -3.5, 3.5
    y_min, y_max = -4.5, 4.5

    # 격자 생성
    x = np.linspace(x_min, x_max, width)
    y = np.linspace(y_min, y_max, height)
    X, Y = np.meshgrid(x, y)
    Z = X + 1j * Y

    # ---------------------------------------------------------
    # 2. 복소 함수 정의 (패턴 생성)
    # ---------------------------------------------------------
    numerator = (Z**3 - 1) * (Z - (1 + 2j)) 
    denominator = (Z**2 + 2j) 
    
    with np.errstate(divide='ignore', invalid='ignore'):
        W = numerator / denominator

    # ---------------------------------------------------------
    # 3. 색상 매핑 (Domain Coloring)
    # ---------------------------------------------------------
    # 위상 -> 색상
    phase = np.angle(W)
    hue = (phase + np.pi) / (2 * np.pi)

    # 크기 -> 명암 (등고선 효과)
    mag = np.abs(W)
    log_mag = np.log(mag + 1e-9)
    n_contours = 15 
    shading = 0.6 + 0.4 * np.sin(n_contours * log_mag)

    # HSV -> RGB
    saturation = np.ones_like(hue) * 0.9 
    value = shading
    hsv_img = np.dstack((hue, saturation, value))
    rgb_img = mcolors.hsv_to_rgb(hsv_img)

    # ---------------------------------------------------------
    # 4. 궤적 및 화살표 그리기
    # ---------------------------------------------------------
    fig, ax = plt.subplots(figsize=(width/dpi, height/dpi), dpi=dpi)
    
    # 배경
    ax.imshow(rgb_img, extent=[x_min, x_max, y_min, y_max], origin='lower')
    
    # 궤적 생성
    t = np.linspace(0, 2*np.pi, 500)
    path_r = 2.5 + 0.3 * np.cos(3*t)
    path_x = path_r * np.cos(t)
    path_y = path_r * np.sin(t)
    
    # 흰색 선 (궤적)
    ax.plot(path_x, path_y, color='white', linewidth=2.5, linestyle='-', alpha=0.9)
    # 그림자 (입체감)
    ax.plot(path_x+0.03, path_y-0.03, color='black', linewidth=3, alpha=0.5)

    # 화살표 (방향 표시)
    num_points = len(t)
    idx_arrows = np.linspace(0, num_points - 1, 8, endpoint=False).astype(int)
    
    for i in idx_arrows:
        next_i = (i + 10) % num_points
        dx = path_x[next_i] - path_x[i]
        dy = path_y[next_i] - path_y[i]
        
        ax.arrow(path_x[i], path_y[i], dx, dy, 
                 shape='full', lw=0, length_includes_head=True, 
                 head_width=0.15, color='white', zorder=10)

    # ---------------------------------------------------------
    # 5. 마무리 (텍스트 없음)
    # ---------------------------------------------------------
    ax.axis('off') # 축 숨기기
    
    # 여백 없이 꽉 채워서 저장
    plt.savefig(output_filename, bbox_inches='tight', pad_inches=0, dpi=dpi)
    print(f"이미지 저장 완료: {output_filename}")
    plt.show()

# 실행
generate_clean_art()

In [ ]:
import numpy as np

# 1. 근을 찾고자 하는 함수 f(x)
def f(x):
    return x**2 - 4

# 2. 함수 f(x)의 미분 함수 f'(x)
def f_prime(x):
    # f'(x) = 2x
    return 2 * x

# 3. 뉴턴-랩슨법 구현 함수
def newton_raphson(f, f_prime, x0, tol=1e-6, max_iter=100):
    """
    뉴턴-랩슨법으로 f(x)=0의 근을 찾는다.
    
    :param f: 함수 f(x)
    :param f_prime: 함수 f'(x)
    :param x0: 초기 추정값
    :param tol: 허용 오차 (수렴 조건)
    :param max_iter: 최대 반복 횟수
    :return: 근사 해
    """
    x_n = x0 # 현재 근사 해
    
    print(f"초기 추정값: {x0}")
    
    for i in range(max_iter):
        f_x = f(x_n)
        f_prime_x = f_prime(x_n)
        
        # 분모가 0인 경우, 알고리즘이 실패할 수 있으므로 처리
        if f_prime_x == 0:
            print(f"오류: 미분값이 0이 되어 나누기를 수행할 수 없다 (x={x_n})")
            return None
        
        # 다음 근사 해를 계산
        delta_x = f_x / f_prime_x
        x_n_plus_1 = x_n - delta_x
        
        # 수렴 조건 확인
        if abs(x_n_plus_1 - x_n) < tol:
            print(f"\n {i+1}번째 반복에서 수렴")
            return x_n_plus_1
        
        x_n = x_n_plus_1 # 다음 반복을 위해 값 업데이트
        print(f"반복 {i+1}: x = {x_n:.6f}")
    
    print("\n 최대 반복 횟수에 도달했다. 수렴에 실패했다.")
    return x_n

# --- 실행 예제 ---
# 초기 추정값을 양수(3.0)로 설정하여 +2 근처의 근을 탐색
root_positive = newton_raphson(f, f_prime, x0=3.0)
print(f"\n찾은 근 (초기값 3.0): **{root_positive}**")
print("\n" + "="*30 + "\n")
# 초기 추정값을 음수(-1.0)로 설정하여 -2 근처의 근을 탐색
root_negative = newton_raphson(f, f_prime, x0=-1.0)
print(f"\n찾은 근 (초기값 -1.0): **{root_negative}**")

In [ ]:
import numpy as np
# 1. 근을 찾고자 하는 함수 f(x)
def f(x):
    return x**3 - 6 * x**2 + 11 * x - 6
# 2. 함수 f(x)의 미분 함수 f'(x)
def f_prime(x):
    return 3 * x**2 - 12 * x + 11
def hybrid_newton_bisection(f, f_prime, a, b, tol=1e-6, max_iter=100):
    """
    뉴턴-랩슨법과 이분법을 결합한 하이브리드 해 찾기 함수이다.
    :param f: 함수 f(x)
    :param f_prime: 함수 f'(x)
    :param a, b: 근을 포함하는 초기 구간 (f(a)와 f(b)의 부호가 달라야 함)
    :return: 근사 해
    """
    if f(a) * f(b) >= 0:
        print(" 오류: f(a)와 f(b)의 부호가 달라야 한다.")
        return None
    a_n, b_n = a, b
    x_n = (a + b) / 2 # 초기 추정값은 구간의 중간으로 시작
    print(f"--- 하이브리드 시작 (초기 구간 [{a:.6f}, {b:.6f}], x0={x_n:.6f}) ---")
    for i in range(max_iter):
        f_x = f(x_n)
        # 1. 수렴 조건 확인
        if abs(f_x) < tol or (b_n - a_n) < tol:
            print(f" {i+1}번째 반복에서 수렴. 근: {x_n:.10f}")
            return x_n
        # 2. 뉴턴-랩슨법 시도
        f_prime_x = f_prime(x_n)
        # 미분값이 0에 가깝거나 뉴턴법으로 계산한 x_next가 불안정할 때 이분법으로 전환
        if abs(f_prime_x) < 1e-10:
            is_newton_safe = False
        else:
            x_next_newton = x_n - f_x / f_prime_x
            # 뉴턴법 안전성 검사:
            # A. x_next가 현재 구간 [a_n, b_n] 안에 있는지
            # B. f(x_next)의 값이 f(x_n)보다 작아졌는지 (수렴 방향인지)
            is_in_range = (x_next_newton > a_n) and (x_next_newton < b_n)
            is_better = abs(f(x_next_newton)) < abs(f_x)
  
            is_newton_safe = is_in_range and is_better
        # 3. 다음 단계 결정 및 구간 업데이트
        if is_newton_safe:
            # 뉴턴법 성공: 뉴턴법 결과 사용
            x_next = x_next_newton
            print(f"반복 {i+1} (Newton): x = {x_next:.6f}")
        else:
            # 뉴턴법 실패 또는 불안정: 이분법 강제 실행
            x_next = (a_n + b_n) / 2
            print(f"반복 {i+1} (Bisection): x = {x_next:.6f} [Fallback]")
       # 4. 구간 [a_n, b_n] 업데이트 (이분법 로직 적용)
        f_next = f(x_next)
        if f(a_n) * f_next < 0:
            b_n = x_next # 해는 [a_n, x_next]에 있음
        else:
              a_n = x_next # 해는 [x_next, b_n]에 있음
        x_n = x_next
    print(" 최대 반복 횟수에 도달했다.")
    return x_n
  
# --- 실행 예제 ---
# 근 x=1을 찾기 위해 구간 [0.5, 1.5] 사용 (f(0.5)=-3.375, f(1.5)=-1.125) -> 부호가 같아 실패!
# 근 x=1을 찾기 위해 구간 [0.5, 1.2] 사용 (f(0.5)=-3.375, f(1.2)=0.048) -> 성공!
root_hybrid = hybrid_newton_bisection(f, f_prime, a=0.6, b=1.2)
print(f"\n최종 찾은 근 (하이브리드): **{root_hybrid}**")
root_hybrid = hybrid_newton_bisection(f, f_prime, a=1.2, b=2.2)
print(f"\n최종 찾은 근 (하이브리드): **{root_hybrid}**")
root_hybrid = hybrid_newton_bisection(f, f_prime, a=2.2, b=3.2)
print(f"\n최종 찾은 근 (하이브리드): **{root_hybrid}**")

In [ ]:
from scipy.optimize import root_scalar
import numpy as np

# 동일한 삼차 함수
def f(x):
    return x**3 - 6 * x**2 + 11 * x - 6

# --- 브렌트의 방법 적용 ---
# 근 x=3을 찾기 위해 [2.5, 3.5] 구간 설정
try:
    result = root_scalar(f, bracket=[2.5, 3.5], method='brentq')
    
    print("--- SciPy를 이용한 Brent's method ---")
    print(f"찾은 근: **{result.root:.10f}**")
    print(f"수렴 여부: {result.converged}")
    print(f"반복 횟수: {result.iterations}회")

except ValueError as e:
    print(f"오류: {e}")

In [ ]:
import numpy as np

def brent_method(f, a, b, tol=1e-8, max_iter=100):
    """
    Brent's method (브렌트 알고리즘)을 사용하여 근을 찾는다.

    :param f: 근을 찾을 함수 (f(x) = 0).
    :param a: 근이 존재하는 구간 [a, b]의 시작점 (f(a)와 f(b)는 부호가 달라야 함).
    :param b: 근이 존재하는 구간 [a, b]의 끝점.
    :param tol: 허용 오차 (수렴 조건).
    :param max_iter: 최대 반복 횟수.
    :return: 근사 근 x 또는 None (수렴 실패 시).
    """
    # 초기 조건 검사: 근이 구간 [a, b] 안에 갇혀 있어야 함 (부호가 달라야 함).
    fa = f(a)
    fb = f(b)
    if fa * fb >= 0:
        raise ValueError("f(a)와 f(b)의 부호가 달라야 한다 (근이 구간에 갇혀 있어야 함).")
    # |f(b)|가 더 작도록 a와 b를 조정 (b를 더 좋은 근사치로 유지)
    if abs(fa) < abs(fb):
        a, b = b, a
        fa, fb = fb, fa
    c = a  # c는 이전의 b 값. 초기에는 a로 설정.
    fc = fa
    d = c  # d는 이전 단계에서 사용된 s 값. 초기에는 c로 설정.
    mflag = True  # mflag는 보간법이 아닌 이분법을 사용했는지 여부를 나타냄.
    for i in range(max_iter):
        # 수렴 검사: |b - a|가 허용 오차보다 작으면 종료.
        if abs(b - a) < tol:
            return b
        # 보간법을 사용할지 결정 (역이차 보간법 또는 할선법)
        if fa != fc and fb != fc:
            # 역이차 보간법 (Inverse Quadratic Interpolation)
            # 3점 (a, f(a)), (b, f(b)), (c, f(c))를 지나는 포물선의 근을 계산.
            # a: f(a)가 f(b), f(c)와 부호가 다른 점
            # b: 현재까지의 최적 근사치
            # c: b의 이전 값
            # 여기서 La, Lb, Lc는 라그랑주 다항식의 기저를 계산한 것이다.
            s = (
                a * fb * fc / ((fa - fb) * (fa - fc))
                + b * fa * fc / ((fb - fa) * (fb - fc))
                + c * fa * fb / ((fc - fa) * (fc - fb))
            )
            
        else:
            # 할선법 (Secant method)
            # 2점 (a, f(a)), (b, f(b))를 지나는 직선의 근을 계산.
            s = b - fb * (b - a) / (fb - fa)

        #  보간법 결과(s)가 적절한지 확인하고, 부적절하면 이분법으로 대체 
        
        # 1. s가 구간 [b, (3a+b)/4] 밖에 있거나 (보간법 추측이 너무 과함),
        # 2. 이전 단계에서 이분법이 사용되었고 |s-b|가 |b-c|의 절반보다 크거나,
        # 3. 이전 단계에서 이분법이 사용되지 않았고 |s-b|가 |c-d|의 절반보다 크면,
        #    => 이분법(Bisection)으로 대체한다.
        
        # bisection_check_condition을 확인한다.
        bisection_check_condition = False
        
        # s가 (3a+b)/4와 b 사이의 안전 구간 내에 있는지 확인
        # (3a+b)/4는 a와 b 사이의 4분의 1 지점으로, 안전 구간을 정의한다.
        safe_zone_min = (3 * a + b) / 4
        
        if (s <= safe_zone_min) or (s >= b): # 조건 1: s가 안전 구간 밖에 있는지
            bisection_check_condition = True
        
        if mflag: # 조건 2: 이전 단계가 이분법이었을 때
            if abs(s - b) >= abs(b - c) / 2:
                bisection_check_condition = True
        else: # 조건 3: 이전 단계가 보간법이었을 때
            if abs(s - b) >= abs(c - d) / 2:
                bisection_check_condition = True


        if bisection_check_condition:
            # 이분법 (Bisection method)으로 대체
            s = (a + b) / 2
            mflag = True
        else:
            # 보간법(할선법 또는 역이차 보간법)을 사용한 경우
            mflag = False

        fs = f(s)
        d = c  # d를 업데이트 (이전의 c)
        c = b  # c를 업데이트 (이전의 b)
        fc = fb # fc를 업데이트 (이전의 fb)

        # 새로운 근사치 s에 따라 구간 [a, b] 업데이트
        if fa * fs < 0:
            # 근은 [a, s] 사이에 있음
            b = s
            fb = fs
        else:
            # 근은 [s, b] 사이에 있음 (이때 f(a)와 f(s)의 부호가 같으므로)
            a = s
            fa = fs
            
        # |f(b)|가 더 작도록 a와 b를 조정
        if abs(fa) < abs(fb):
            a, b = b, a
            fa, fb = fb, fa

    # 최대 반복 횟수 초과 시
    return None
# 근을 찾을 함수 정의
def example_function(x):
    return x**3 - 2*x - 5

# 근이 존재하는 구간 설정 (예: f(2) = -1, f(3) = 16 이므로 [2, 3] 사이에 근이 있음)
a_initial = 2.0
b_initial = 3.0
# 브렌트 알고리즘 실행
root = brent_method(example_function, a_initial, b_initial)
if root is not None:
    print(f"함수의 근: {root}")
    print(f"근에서의 함수 값 (f(root)): {example_function(root)}")
else:
    print("수렴에 실패했다.")    

In [ ]:
import math

def banach_fixed_point_demo():
    # 1. 초기값 설정 (아무거나 넣어도 된다)
    x = 100 
    print(f"초기값: {x}")
    # 2. 반복 계산 (Iteration)
    for i in range(1, 21): # 20번만 반복
        prev_x = x
        # 핵심: 나온 값을 다시 넣는다. x_new = g(x_old)
        x = math.cos(x)
        print(f"[{i}회차] x = {x:.9f}")
        # 수렴 판정 (변화가 거의 없으면 종료)
        if abs(x - prev_x) < 1e-9:
            print(f"\n>>> 수렴 성공! 고정점(Fixed Point)은 {x:.9f} 이다.")
            break

banach_fixed_point_demo()

In [ ]:
import numpy as np

# 1. 5차 방정식의 계수를 정의한다.
# 계수는 높은 차수부터 낮은 차수 순서로 배열에 넣는다.
# P(x) = a*x^5 + b*x^4 + c*x^3 + d*x^2 + e*x + f
# [a, b, c, d, e, f]
# 예시: x^5 - 2x^4 - 6x^3 + 8x^2 + 5x - 4 = 0
coefficients = [1, -2, -6, 8, 5, -4]

# 2. numpy.roots 함수를 사용하여 근을 계산한다.
# 이 함수는 수치 해석적 방법을 사용하여 근의 근사치를 반환한다.
roots = np.roots(coefficients)

# 3. 결과 출력
print("### 5차 방정식의 근사 근 (NumPy 이용) ###")
print(f"방정식 계수: {coefficients}")
print(f"총 근의 개수: {len(roots)}개")
print("\n[계산된 5개의 근]")
for i, root in enumerate(roots):
    print(f"근 {i+1}: {root}")

# 근이 실수인지 허수인지 구분하여 출력 (매우 작은 허수부는 무시)
# print("\n[실수/복소수 구분]")
# for root in roots:
#     if np.isclose(root.imag, 0):
#         print(f"실수 근: {root.real:.6f}")
#     else:
#         print(f"복소수 근: {root}")

In [ ]:
import numpy as np

# 다항식의 값을 계산하는 함수
def polynomial_value(coeff, x):
    result = 0
    for i in range(len(coeff)):
        result += coeff[i] * (x ** (len(coeff) - i - 1))
    return result

# 다항식의 도함수 값을 계산하는 함수 (첫 번째 도함수)
def polynomial_derivative(coeff, x):
    result = 0
    for i in range(len(coeff) - 1):
        result += (len(coeff) - i - 1) * coeff[i] * (x ** (len(coeff) - i - 2))
    return result

# 다항식의 두 번째 도함수 값을 계산하는 함수
def polynomial_second_derivative(coeff, x):
    result = 0
    for i in range(len(coeff) - 2):
        result += (len(coeff) - i - 1) * (len(coeff) - i - 2) * coeff[i] * (x ** (len(coeff) - i - 3))
    return result

# Laguerre's method로 근을 찾는 함수
def laguerre_method(coeff, x0, max_iter=100, tol=1e-6):
    n = len(coeff) - 1  # 다항식 차수
    x = x0  # 초기 추정값

    for _ in range(max_iter):
        # 다항식과 도함수 계산
        fx = polynomial_value(coeff, x)
        f_prime = polynomial_derivative(coeff, x)
        f_double_prime = polynomial_second_derivative(coeff, x)

        # G와 H 계산
        G = f_prime / fx
        H = G**2 - f_double_prime / fx

        # Laguerre's method 공식
        denominator = max(G + np.sqrt((n - 1) * (n * H - G**2)), G - np.sqrt((n - 1) * (n * H - G**2)))
        denominator = denominator if denominator != 0 else 1e-10  # 0으로 나누지 않도록 처리
        delta_x = n / denominator

        # 근 업데이트
        x = x - delta_x

        # 수렴 조건
        if abs(delta_x) < tol:
            return x

    return x

# 예시 다항식: x^3 - 6x^2 + 11x - 6
coeff = [1, -6, 11, -6]

# 초기 추정값 설정 (임의로 선택)
initial_guess = 3.1
# Laguerre's method 실행
root = laguerre_method(coeff, initial_guess)
print(f"근: {root}")
# 다항식의 값을 계산하여 정확성 체크
print(f"다항식 값: {polynomial_value(coeff, root)}")

# 초기 추정값 설정 (임의로 선택)
initial_guess = 2.1
# Laguerre's method 실행
root = laguerre_method(coeff, initial_guess)
print(f"근: {root}")
# 다항식의 값을 계산하여 정확성 체크
print(f"다항식 값: {polynomial_value(coeff, root)}")

# 초기 추정값 설정 (임의로 선택)
initial_guess = 0.9
# Laguerre's method 실행
root = laguerre_method(coeff, initial_guess)
print(f"근: {root}")
# 다항식의 값을 계산하여 정확성 체크
print(f"다항식 값: {polynomial_value(coeff, root)}")

In [ ]:
import numpy as np

def horner_method(coefficients, alpha):
    """
    호너의 방법을 사용하여 다항식 P(x)의 값 P(alpha)를 계산하고,
    몫 다항식 Q(x)의 계수를 반환한다.

    Args:
        coefficients (list/np.array): 다항식의 계수 리스트 (가장 높은 차수부터 시작: [an, an-1, ..., a0])
        alpha (float): 값을 계산할 x 값

    Returns:
        tuple: (P(alpha) 값, 몫 다항식 Q(x)의 계수 리스트)
    """
    # 계수를 복사하여 수정할 리스트 b를 만든다. (b는 몫의 계수와 나머지 역할을 한다.)
    b = list(coefficients)
    
    # 순환 계산 시작 (가장 높은 차수 계수부터 시작)
    # n-1 차 계수부터 0차 계수까지 (a[n-1]부터 a[0]까지) 반복
    for i in range(1, len(b)):
        # b[i] = a[i] + b[i-1] * alpha
        # 여기서 b[i-1]은 이전 단계에서 계산된 b_k이다.
        b[i] = b[i] + b[i-1] * alpha
        
    # 최종 결과
    remainder = b[-1]            # P(alpha) 값 (가장 마지막 요소)
    quotient_coeffs = b[:-1]     # 몫 Q(x)의 계수 (마지막 요소를 제외한 나머지)
    
    return remainder, quotient_coeffs

# 예제 다항식: P(x) = 3x^3 - 2x^2 + 5x - 6
# 계수 리스트: [3, -2, 5, -6] (a3, a2, a1, a0 순서)
coefficients = [3, -2, 5, -6]
alpha = 2

# 호너의 방법 적용
result_P_alpha, quotient = horner_method(coefficients, alpha)

print(f"다항식: P(x) = {coefficients[0]}x^3 + {coefficients[1]}x^2 + {coefficients[2]}x + {coefficients[3]}")
print(f"계산할 x 값 (alpha): {alpha}")
print("-" * 30)

print(f"P({alpha})의 계산 결과 (나머지): {result_P_alpha}")

# 몫 다항식의 출력
if quotient:
    # 몫의 차수가 원래 다항식보다 1 낮다.
    power = len(quotient) - 1 
    quotient_str = " + ".join([f"{c}x^{p}" if p > 0 else str(c) 
                               for c, p in zip(quotient, range(power, -1, -1))])
    print(f"몫 다항식 Q(x)의 계수: {quotient}")
    print(f"몫 다항식: Q(x) = {quotient_str.replace('+ -', '- ')}")
else:
    print("몫 다항식이 없다 (상수 다항식인 경우).")

# P(x) = Q(x)(x-alpha) + R 확인
# P(2) = 20, Q(x) = 3x^2 + 4x + 13
# Q(2)(2-2) + 20 = (3*4 + 4*2 + 13) * 0 + 20 = 20 (확인 완료)

In [ ]:
import numpy as np
def horner_method(coefficients, c):
    """
    호너의 방법(조립제법)을 사용하여 다항식을 x - c 로 나누고 
    나머지 (다항식의 값 P(c)) 와 몫의 계수를 반환한다.

    Args:
        coefficients (list): 다항식 P(x)의 계수 리스트 (고차항부터 상수항 순서).
        c (float or int): 나누는 값 (x - c 에서 c).

    Returns:
        tuple: (remainder, quotient_coefficients)
            remainder (float or int): 나머지 R (즉, P(c) 값).
            quotient_coefficients (list): 몫 Q(x)의 계수 리스트.
    """
    
    # 몫의 계수를 저장할 리스트 초기화
    quotient_coefficients = []
    
    # 초기 나머지 값 (가장 높은 차수의 계수)
    current_result = coefficients[0]
    quotient_coefficients.append(current_result)

    # 두 번째 계수부터 상수항까지 반복
    for i in range(1, len(coefficients)):
        # 1. 이전 결과에 c를 곱함 (곱하기)
        multiplied_value = current_result * c
        
        # 2. 현재 계수와 곱한 값을 더함 (더하기)
        current_result = coefficients[i] + multiplied_value
        
        # 마지막 반복 (상수항)을 제외하고 몫의 계수에 추가
        if i < len(coefficients) - 1:
            quotient_coefficients.append(current_result)
            
    # 최종 결과가 나머지 R이 된다.
    remainder = current_result
    
    return remainder, quotient_coefficients

    
def verify_horner(coefficients, c):
    """
    호너의 방법으로 구한 나머지 P(c)와 NumPy를 사용한 P(c)를 비교하여 
    호너의 방법이 잘 작동하는지 검증하는 함수.
    
    Args:
        coefficients (list): 다항식 P(x)의 계수 리스트 (고차항부터 상수항 순서).
        c (float or int): 다항식에 대입할 값 (x - c 에서 c).
        
    Returns:
        bool: 두 결과가 일치하면 True, 아니면 False.
    """
    
    # --- 1. 호너의 방법으로 계산 ---
    horner_remainder, _ = horner_method(coefficients, c)
    
    # --- 2. 다항식의 정의에 따라 직접 계산 (NumPy 사용) ---
    # numpy.polyval(p, x)는 다항식 p에 x를 대입한 값을 계산한다.
    # 여기서 p는 계수 리스트이다.
    standard_value = np.polyval(coefficients, c)
    
    # --- 3. 두 결과를 비교하여 검증 ---
    # 부동 소수점 오차를 고려하여 근사적으로 비교 (isclose 사용)
    is_correct = np.isclose(horner_remainder, standard_value)
    
    print(f"--- 검증 결과 (c = {c}) ---")
    print(f"호너의 방법으로 구한 값 (P(c)): {horner_remainder}")
    print(f"NumPy로 구한 값 (P(c)): {standard_value}")
    print(f"결과 일치 여부: {' 일치한다.' if is_correct else ' 불일치한다.'}")
    
    return is_correct    
    
# 다항식 P(x)의 계수 [4, 5, 0, -12]
coefficients = [4, 5, 0, -12]
c = 3

remainder, quotient = horner_method(coefficients, c)

# 출력
print(f"나누는 값 (c): {c}")
print(f"다항식의 계수: {coefficients}")
print("-" * 30)
print(f"나머지 R (P({c})): {remainder}")
print(f"몫 Q(x)의 계수: {quotient}") 
# 결과: 나머지 R: 141, 몫 Q(x)의 계수: [4, 17, 51]
# 몫 Q(x) = 4x^2 + 17x + 51

# 다항식 P(x)의 계수 [3, 0, -2, 1, -15]
coefficients_2 = [3, 0, -2, 1, -15]
c_2 = -1

remainder_2, quotient_2 = horner_method(coefficients_2, c_2)

# 출력
print(f"나누는 값 (c): {c_2}")
print(f"다항식의 계수: {coefficients_2}")
print("-" * 30)
print(f"나머지 R (P({c_2})): {remainder_2}")
print(f"몫 Q(x)의 계수: {quotient_2}") 
# 결과: 나머지 R: -15, 몫 Q(x)의 계수: [3, -3, 1, 0]
# 몫 Q(x) = 3x^3 - 3x^2 + x

coefficients_1 = [4, 5, 0, -12]
c_1 = 3

logic1=verify_horner(coefficients_1, c_1)


coefficients_2 = [3, 0, -2, 1, -15]
c_2 = -1

logic2=verify_horner(coefficients_2, c_2)

In [ ]:
import numpy as np

# 다항식의 값을 계산하는 함수
def polynomial_value(coeff, x):
    result = 0
    for i in range(len(coeff)):
        result += coeff[i] * (x ** (len(coeff) - i - 1))
    return result

# Durand-Kerner method로 다항식의 근을 찾는 함수
def durand_kerner(coeff, max_iter=100, tol=1e-6, initial_roots=None):
    n = len(coeff) - 1  # 다항식의 차수
    roots = initial_roots if initial_roots is not None else np.random.rand(n) + 1j * np.random.rand(n)  # 초기값 설정

    for _ in range(max_iter):
        new_roots = np.copy(roots)
        
        for i in range(n):
            numerator = polynomial_value(coeff, roots[i])
            denominator = 1
            for j in range(n):
                if i != j:
                    denominator *= (roots[i] - roots[j])  # 다른 근들과의 차를 곱함
            new_roots[i] = roots[i] - numerator / denominator

        # 수렴 조건 체크: 근이 충분히 변하지 않으면 종료
        if np.all(np.abs(new_roots - roots) < tol):
            break
        roots = new_roots

    return roots

# 예시 다항식: x^3 - 6x^2 + 11x - 6
coeff = [1, -6, 11, -6]

# 다양한 초기값 설정
# 1. 균등한 분포로 초기값 설정
roots_uniform = np.exp(2j * np.pi * np.arange(3) / 3)

# 2. 무작위 복소수 값으로 초기값 설정
roots_random = np.random.rand(3) + 1j * np.random.rand(3)

# 3. 예상되는 근 근처로 초기값 설정
roots_approx = np.array([1.1, 2.1, 3.1])

# 4. 일정한 간격으로 초기값 설정
roots_linear = np.linspace(0, 5, 3)

# 각 초기값에 대해 Durand-Kerner 방법 실행
roots_du_random = durand_kerner(coeff, initial_roots=roots_random)
roots_du_uniform = durand_kerner(coeff, initial_roots=roots_uniform)
roots_du_approx = durand_kerner(coeff, initial_roots=roots_approx)
roots_du_linear = durand_kerner(coeff, initial_roots=roots_linear)

# 결과 출력
print(f"균등 분포 초기값 근: {roots_du_uniform}")
print(f"무작위 복소수 초기값 근: {roots_du_random}")
print(f"예상 근 근처 초기값 근: {roots_du_approx}")
print(f"일정 간격 초기값 근: {roots_du_linear}")

In [ ]:
import numpy as np
from numpy.polynomial import Polynomial
# 다항식: x^3 - 3x^2 + 3x - 5 = 0
# 주의: numpy.roots와 달리 계수 순서가 [상수항, 1차항, ..., 최고차항] 순서이다.
p = Polynomial([-5, 3, -3, 1]) 
roots = p.roots()
print("Numpy Polynomial Roots:", roots)

In [ ]:
from sympy import symbols, solve
x = symbols('x')
# 다항식 정의
equation = x**3 - 3*x**2 + 3*x - 5
# solve 함수 사용
roots = solve(equation, x)
print("Sympy Exact Roots:")
for root in roots:
    print(root.evalf()) # 수치로 보고 싶으면 evalf()
    # 그냥 print(root)를 하면 복잡한 수식 형태(세제곱근 포함)로 출력된다.

In [ ]:
import mpmath
# 정밀도를 50자리로 설정
mpmath.mp.dps = 50
# 다항식 계수 (최고차항부터)
coeffs = [1, -3, 3, -5]
# polyroots 함수 사용
roots = mpmath.polyroots(coeffs)
print("Mpmath High-Precision Roots:")
for r in roots:
    print(r)

In [ ]:
import scipy.linalg
import numpy as np
# x^3 - 3x^2 + 3x - 5 = 0
coeffs = [1, -3, 3, -5]
# 1. 모닉 다항식으로 변환 (최고차항 계수로 나눔)
coeffs = np.array(coeffs, dtype=float)
coeffs /= coeffs[0]
a = coeffs[1:] # 최고차항 제외한 나머지 계수 (부호 반대로 써야 함에 주의)
n = len(a)
# 2. 동반 행렬 구성
C = np.zeros((n, n))
# 서브 대각선에 1 채우기
for i in range(n-1):
    C[i+1, i] = 1.0
# 마지막 열에 계수 채우기 (부호 반대)
C[:, -1] = -a[::-1] # 계수 순서를 뒤집어서 넣어야 함 (상수항이 맨 위)
# 3. 고윳값 계산
eigenvalues = scipy.linalg.eigvals(C)
print("Eigenvalues (Roots):", eigenvalues)

In [ ]:
import numpy as np

# 1. 근을 찾고자 하는 함수 f(x)
def f(x):
    return x**2 - 4

# 2. 함수 f(x)의 미분 함수 f'(x)
def f_prime(x):
    # f'(x) = 2x
    return 2 * x

# 3. 뉴턴-랩슨법 구현 함수
def newton_raphson(f, f_prime, x0, tol=1e-6, max_iter=100):
    """
    뉴턴-랩슨법으로 f(x)=0의 근을 찾는다.
    
    :param f: 함수 f(x)
    :param f_prime: 함수 f'(x)
    :param x0: 초기 추정값
    :param tol: 허용 오차 (수렴 조건)
    :param max_iter: 최대 반복 횟수
    :return: 근사 해
    """
    x_n = x0 # 현재 근사 해
    
    print(f"초기 추정값: {x0}")
    
    for i in range(max_iter):
        f_x = f(x_n)
        f_prime_x = f_prime(x_n)
        
        # 분모가 0인 경우, 알고리즘이 실패할 수 있으므로 처리
        if f_prime_x == 0:
            print(f"오류: 미분값이 0이 되어 나누기를 수행할 수 없다 (x={x_n})")
            return None
        
        # 다음 근사 해를 계산
        delta_x = f_x / f_prime_x
        x_n_plus_1 = x_n - delta_x
        
        # 수렴 조건 확인
        if abs(x_n_plus_1 - x_n) < tol:
            print(f"\n {i+1}번째 반복에서 수렴")
            return x_n_plus_1
        
        x_n = x_n_plus_1 # 다음 반복을 위해 값 업데이트
        print(f"반복 {i+1}: x = {x_n:.6f}")
    
    print("\n 최대 반복 횟수에 도달했다. 수렴에 실패했다.")
    return x_n

# --- 실행 예제 ---

# 초기 추정값을 양수(3.0)로 설정하여 +2 근처의 근을 탐색
root_positive = newton_raphson(f, f_prime, x0=3.0)
print(f"\n찾은 근 (초기값 3.0): **{root_positive}**")

print("\n" + "="*30 + "\n")

# 초기 추정값을 음수(-1.0)로 설정하여 -2 근처의 근을 탐색
root_negative = newton_raphson(f, f_prime, x0=-1.0)
print(f"\n찾은 근 (초기값 -1.0): **{root_negative}**")

## Chapter 2 유한 차분    파이썬을 활용한 수치해석(Numerical Analysis with Python) 이인호 (북스힐, 2026)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline, lagrange
# ==========================================
# 1. 방데르몽드 행렬을 이용한 보간법 (추가됨)
# ==========================================
def vandermonde_interpolation(x_nodes, y_nodes, x_dense):
    """
    Va = y 선형 시스템을 풀어 다항식의 계수 a를 구한다.
    V는 x_nodes로 만든 방데르몽드 행렬이다.
    """
    # 1. 방데르몽드 행렬 생성 (N x N)
    # increasing=False 옵션: [x^n, x^(n-1), ..., 1] 순서로 생성 (np.polyval과 호환)
    V = np.vander(x_nodes, increasing=False) 
    # 2. 선형 시스템 풀기 (Va = y) -> 계수 벡터 a 구하기
    # 주의: N이 커지면 조건수(Condition Number)가 급증하여 오차가 커진다.
    coeffs = np.linalg.solve(V, y_nodes)
    # 3. 다항식 평가 (Horner's Method 이용)
    y_eval = np.polyval(coeffs, x_dense)
    return y_eval, coeffs
# ==========================================
# 2. 기존 방법들 (뉴턴, 네빌)
# ==========================================
def newton_divided_difference_coeffs(x, y):
    n = len(x)
    coef = np.zeros([n, n])
    coef[:, 0] = y
    for j in range(1, n):
        for i in range(n - j):
            coef[i][j] = (coef[i + 1][j - 1] - coef[i][j - 1]) / (x[i + j] - x[i])
    return coef[0, :]
def newton_evaluate(coeffs, x_nodes, x_val):
    n = len(x_nodes) - 1
    p = coeffs[n]
    for k in range(1, n + 1):
        p = coeffs[n - k] + (x_val - x_nodes[n - k]) * p
    return p
def neville_interpolate(x_nodes, y_nodes, x_target):
    n = len(x_nodes)
    Q = np.zeros((n, n))
    Q[:, 0] = y_nodes
    for i in range(1, n):
        for j in range(1, i + 1):
            numerator=(x_target-x_nodes[i-j])*Q[i,j-1]-(x_target-x_nodes[i])*Q[i-1,j-1]
            denominator = x_nodes[i] - x_nodes[i-j]
            Q[i, j] = numerator / denominator
    return Q[n-1, n-1]
# ==========================================
# 3. 메인 실행 및 비교
# ==========================================
# (1) 데이터 생성 (룽게 함수)
def true_function(x):
    return 1 / (1 + 25 * x**2)
num_nodes = 11
x_nodes = np.linspace(-1, 1, num_nodes)
y_nodes = true_function(x_nodes)
x_dense = np.linspace(-1, 1, 400)
y_true = true_function(x_dense)
# --- A. 방데르몽드 (New!) ---
y_vander, vander_coeffs = vandermonde_interpolation(x_nodes, y_nodes, x_dense)
# --- B. 뉴턴 ---
newton_coeffs = newton_divided_difference_coeffs(x_nodes, y_nodes)
y_newton = [newton_evaluate(newton_coeffs, x_nodes, val) for val in x_dense]
# --- C. 네빌 ---
y_neville = [neville_interpolate(x_nodes, y_nodes, val) for val in x_dense]
# --- D. 라그랑주 ---
lagrange_poly = lagrange(x_nodes, y_nodes)
y_lagrange = lagrange_poly(x_dense)
# --- E. 스플라인 ---
cs = CubicSpline(x_nodes, y_nodes)
y_spline = cs(x_dense)
# (2) 값 비교 (x=0.5)
check_point = 0.5
exact_val = true_function(check_point)
vander_val = np.polyval(vander_coeffs, check_point) # 방데르몽드 결과값
print(f"=== 보간 방법별 비교 (x = {check_point}) ===")
print(f"1. 참값 (Exact)      : {exact_val:.8f}")
print("-" * 40)
print(f"2. 방데르몽드(Vander): {vander_val:.8f} (선형대수)")
print(f"3. 뉴턴 (Newton)     : {newton_evaluate(newton_coeffs, x_nodes, check_point):.8f}")
print(f"4. 네빌 (Neville)    : {neville_interpolate(x_nodes, y_nodes, check_point):.8f}")
print(f"5. 라그랑주(Lagr.)   : {lagrange_poly(check_point):.8f}")
print("-" * 40)
print(f"6. 스플라인(Spline)  : {cs(check_point):.8f}")
# 조건수 확인 (방데르몽드 행렬의 위험성)
V = np.vander(x_nodes, increasing=False)
cond_number = np.linalg.cond(V)
print(f"\n[주의] 방데르몽드 행렬의 조건수(Condition Number): {cond_number:.2e}")
print("-> 이 값이 클수록(10^10 이상) 해가 불안정하여 오차가 발생하기 쉽다.")
# (3) 시각화
plt.figure(figsize=(12, 8))
# 원본
plt.plot(x_dense, y_true, 'k-', alpha=0.3, linewidth=2, label='True function')
plt.plot(x_nodes, y_nodes, 'ko', markersize=8, label='Nodes')
# 1. 방데르몽드 (가장 굵은 노란 투명선 - 바탕)
plt.plot(x_dense, y_vander, 'y-', linewidth=8, alpha=0.4, label='Vandermonde matrix')
# 2. 뉴턴 (빨간 실선)
plt.plot(x_dense, y_newton, 'r-', linewidth=2, label='Newton form')
# 3. 라그랑주 (초록 점선)
plt.plot(x_dense, y_lagrange, 'g--', linewidth=2, label='Lagrange form')
# 4. 네빌 (파란 점)
plt.plot(x_dense[::15], y_neville[::15], 'b.', markersize=10, label='Neville points')
# 5. 스플라인 (유일하게 다른 경로)
plt.plot(x_dense, y_spline, 'm-', linewidth=2, label='cubic spline(best)')
plt.title(f"All polynomial methods vs spline(Nodes N={num_nodes})")
plt.xlabel(r"$x$", fontsize=16)
plt.ylabel(r"$y$", fontsize=16)
plt.ylim(-0.5, 2.0)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig('four.png')
plt.show()

In [ ]:
from scipy.interpolate import interp1d
import numpy as np
# x	데이터 점의 독립 변수 값 배열. 오름차순으로 정렬되어 있어야 한다.
# y	데이터 점의 종속 변수 값 배열. x와 길이가 같아야 한다.
x = [i*1.0 for i in range(10)]
y = [(i*1.0)**2 for i in range(10)]
x = np.array(x)
y = np.array(y)
f_interp = interp1d(x, y, kind='linear')
n_features=100
x_new = np.linspace(min(x), max(x), num=n_features, endpoint=True)
f_new = f_interp(x_new)

In [ ]:
import numpy as np
def f(x):
    """미분할 함수: sin(x)"""
    return np.sin(x)
def central_difference(f, x, h):
    """
    중심 차분 공식을 이용한 미분 근사 (수렴 차수 p=2)
    """
    return (f(x + h) - f(x - h)) / (2 * h)
# 분석 지점
x_val = 1.0 
# 정확한 해 (참값)
exact_value = np.cos(x_val) 
print(f"--- 참값 (Exact Value) ---\nf'(1) = cos(1) ≈ {exact_value:.10f}\n")
# 1단계: h 값으로 근삿값 계산
h1 = 0.1
D1 = central_difference(f, x_val, h1)
print(f"1. h={h1} 일 때의 근삿값 D1: {D1:.10f}")
print(f"   오차: {abs(D1 - exact_value):.10f}")
print("-" * 30)
# 2단계: h/2 값으로 근삿값 계산
h2 = h1 / 2  # h2 = 0.05
D2 = central_difference(f, x_val, h2)
print(f"2. h={h2} 일 때의 근삿값 D2: {D2:.10f}")
print(f"   오차: {abs(D2 - exact_value):.10f}")
print("-" * 30)
# 리처드슨 외삽법 적용 (p=2)
p = 2
richardson_extrapolated = D2 + (D2 - D1) / (2**p - 1)
print(f"3. 리처드슨 외삽값 (Phi_new): {richardson_extrapolated:.10f}")
print(f"   외삽 후 오차: {abs(richardson_extrapolated - exact_value):.10f}")	

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import lagrange
# 1. 룽게 함수 정의
def runge_function(x):
    return 1.0 / (1.0 + 25.0 * x**2)
# 보간 차수 (N+1 개의 보간점)
N = 15 
N = 25
# 플롯을 위한 x 축 생성 (촘촘한 간격)
x_plot = np.linspace(-1, 1, 200) 
y_exact = runge_function(x_plot)
# ----------------- 2.1. 균일 간격 보간 (Equispaced Nodes) -----------------
# 보간점 생성 (균일 간격)
x_equi = np.linspace(-1, 1, N + 1)
y_equi = runge_function(x_equi)
# Lagrange 보간 다항식 생성 및 계산
poly_equi = lagrange(x_equi, y_equi)
y_interp_equi = poly_equi(x_plot)
# ----------------- 2.2. Chebyshev 보간 (Chebyshev Nodes) -----------------
# Chebyshev 마디 생성 (구간 [-1, 1])
k = np.arange(1, N + 2) # k = 1, 2, ..., N+1
x_cheby = np.cos((2 * k - 1) * np.pi / (2 * (N + 1)))
# 마디를 오름차순으로 정렬 (플롯 가독성 위함)
x_cheby = np.sort(x_cheby)
y_cheby = runge_function(x_cheby)
# Lagrange 보간 다항식 생성 및 계산
poly_cheby = lagrange(x_cheby, y_cheby)
y_interp_cheby = poly_cheby(x_plot)
# ----------------- 3. 결과 시각화 -----------------
plt.figure(figsize=(12, 6))
# A. 균일 간격 보간 결과
plt.subplot(1, 2, 1)
plt.plot(x_plot, y_exact, 'k-', linewidth=2, label='Exact Runge function')
plt.plot(x_equi, y_equi, 'o', color='red', markersize=5, label='Equispaced nodes')
plt.plot(x_plot, y_interp_equi, 'r--', linewidth=1.5, label=f'Equispaced interp. (N={N})')
plt.title(f'Runge phenomenon: Equispaced nodes (N={N})')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.ylim(-1.0, 1.5)
plt.legend()
plt.grid(True)
# B. Chebyshev 보간 결과
plt.subplot(1, 2, 2)
plt.plot(x_plot, y_exact, 'k-', linewidth=2, label='Exact Runge function')
plt.plot(x_cheby, y_cheby, 'o', color='blue', markersize=5, label='Chebyshev nodes')
plt.plot(x_plot, y_interp_cheby, 'b--', linewidth=1.5, label=f'Chebyshev interp. (N={N})')
plt.title(f'Chebyshev interpolation (N={N})')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.ylim(-1.0, 1.5)
plt.legend()
plt.grid(True)
plt.savefig('rp_ch.png')
plt.tight_layout()
plt.show()

In [ ]:
def weights(z, x, nd, m):
    c1 = 1
    c4 = x[0] - z
    c = np.zeros((nd+1, m+1))
    c[0, 0] = 1
    for i in range(1, nd+1):
        mn = min(i, m)
        c2 = 1
        c5 = c4
        c4 = x[i] - z
        for j in range(0, i):
            c3 = x[i] - x[j]
            c2 = c2*c3
            if j == i-1:
                for k in range(mn, 0, -1):
                    c[i, k] = c1*(k*c[i-1, k-1] - c5*c[i-1, k])/c2
                c[i, 0] = -c1*c5*c[i-1, 0]/c2
            for k in range(mn, 0, -1):
                c[j, k] = (c4*c[j, k] - k*c[j, k-1])/c3
            c[j, 0] = c4*c[j, 0]/c3
        c1 = c2
    return c
def get_yprimes(nleft, m, x, y):
    npt = len(x)
    y1 = np.zeros(npt)
    y2 = np.zeros(npt)
    nright = nleft
    nd = nleft+nright
    xsten = np.zeros(nd+1)
    for j in range(npt):
        z = x[j]
        tmp = 0.
        tmq = 0.
        if j-nleft < 0:
            j0 = 0
            j1 = nd+1
            xsten[0:nd+1] = x[j0:j1]
        elif j-nleft+nd+1 > npt-1:
            j1 = npt
            j0 = j1-nd-1
            xsten[0:nd+1] = x[j0:j1]
        else:
            j0 = j-nleft
            j1 = j0+nd+1
            xsten[0:nd+1] = x[j0:j1]
        c = weights(z, xsten, nd, m)
        for k in range(nd+1):
            tmp = tmp+c[k, 1]*y[j0+k]
            tmq = tmq+c[k, 2]*y[j0+k]
        y1[j] = tmp
        y2[j] = tmq
    return y1, y2    

import matplotlib.pyplot as plt
import numpy as np
npt = 100
x = np.linspace(0, 10.0, npt, endpoint=True)
y = np.linspace(0, 10.0, npt, endpoint=True)
for i in range(npt):
    y[i] = np.cos(x[i])
nleft = 2
m = 2
y1, y2 = get_yprimes(nleft, m, x, y)
plt.figure(figsize=(12, 5))
plt.plot(x, y, linestyle='-')
plt.plot(x, y1, linestyle='--')
plt.plot(x, y2, linestyle='-.')
plt.xlabel('$x$',fontsize=16)
plt.ylabel('$y$',fontsize=16)
plt.savefig("fornberg.png")
plt.show()

In [ ]:
import numpy as np, math
def fd_weights(x, x0, m):
    x  = np.asarray(x, dtype=float).reshape(-1)
    x0 = float(x0)
    n  = x.size
    if m >= n:
        raise ValueError("스텐실 점수 n > 미분차수 m 필요")
    if np.unique(x).size != n:
        raise ValueError("스텐실 좌표 중복 존재")
    A = np.vander(x - x0, N=n, increasing=True)  
    # [ (x-x0)^k ]_{k=0..n-1}
    b = np.zeros(n); b[m] = math.factorial(m)
    w = np.linalg.solve(A, b)
    return w

In [ ]:
import numpy as np
print("=== 1. 정수 오버플로우 (NumPy int8 사용) ===")
# int8은 -128 ~ 127까지 표현 가능
max_int8 = np.array([127], dtype='int8')
print(f"현재 값: {max_int8}")
# 1을 더하면 128이 되는 게 아니라, 비트 범위를 넘어 최솟값(-128)으로 돌아감 (Wrapping)
overflowed = max_int8 + 1
print(f"1 더한 결과 (Overflow): {overflowed}")
# 비교: 파이썬 기본 int는 오버플로우 없음
py_int = 127
print(f"파이썬 기본 int (127 + 1): {py_int + 1} (오버플로우 안 됨)")
import sys
print("\n=== 2. 부동소수점 오버플로우 (Float Overflow) ===")
# 표현 가능한 가장 큰 float 값 확인
max_float = sys.float_info.max
print(f"표현 가능한 최대 float: {max_float}")
# 최댓값에 1.0001 같은 작은 수를 곱해도 오버플로우는 안 나지만,
# 2를 곱해서 한계를 넘겨버리면 'inf' (무한대)가 됨
overflow_float = max_float * 2
print(f"한계를 넘은 결과: {overflow_float}")
# 무한대 여부 확인
print(f"무한대인가?: {overflow_float == float('inf')}")
print("\n=== 3. 부동소수점 언더플로우 (Float Underflow) ===")
# 아주 작은 수에서 시작
small_num = 1e-300
print(f"시작 값: {small_num}")
# 계속해서 1000으로 나누어 봄
for i in range(10):
    small_num = small_num / 100000
    print(f"{i+1}회 나눔: {small_num}")
    if small_num == 0.0:
        print(">>> 언더플로우 발생! (0.0으로 소멸됨)")
        break

In [ ]:
import numpy as np
import scipy.sparse as sparse
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
def solve_1d_fdm():
    # 1. 격자 설정
    N = 100          # 내부 격자점 개수
    L = 1.0          # 구간 길이
    h = L / (N + 1)  # 격자 간격 (dx)
    x = np.linspace(0, L, N + 2) # 경계 포함 전체 좌표
    # 2. 희소 행렬(Sparse Matrix) A 생성
    # 식: y(i-1) - 2y(i) + y(i+1) = h^2 * f(i)
    # A는 삼중 대각 행렬(Tridiagonal)이 된다.
    diagonals = [np.ones(N-1), -2*np.ones(N), np.ones(N-1)]
    offsets = [-1, 0, 1]
    # 3-point stencil Matrix
    A = sparse.diags(diagonals, offsets, shape=(N, N), format='csr')
    # 3. 우변 벡터 b 생성
    # 미분방정식의 우변 f(x)
    x_internal = x[1:-1] # 경계 제외한 내부 점
    f = -4 * (np.pi**2) * np.sin(2 * np.pi * x_internal)
    b = f * (h**2) # h^2을 우변으로 넘겨줌
    # 4. 선형 시스템 풀기 (Ax = b)
    # spla.spsolve는 희소 행렬에 최적화된 직접 풀이법(Direct Solver)
    y_internal = spla.spsolve(A, b)
    # 5. 경계 조건 결합 (0, ..., 0)
    y_sol = np.concatenate(([0], y_internal, [0]))
    # 6. 결과 비교
    y_exact = np.sin(2 * np.pi * x) # 이론적 정답
    plt.figure(figsize=(8, 5))
    plt.plot(x, y_exact, 'k-', label='Exact solution')
    plt.plot(x, y_sol, 'r--', label='FDM solution (N={})'.format(N))
    plt.title("1D Finite difference method (O(h^2))")
    plt.legend()
    plt.grid(True)
    plt.show()
solve_1d_fdm()	

In [ ]:
import numpy as np
import scipy.sparse as sparse
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
def solve_2d_poisson_fdm():
    # 1. 문제 설정
    N = 50   # 한 변의 격자 수 (전체 변수 개수 N^2 = 2500)
    L = 1.0
    h = L / (N + 1)
    x = np.linspace(0, L, N + 2)
    y = np.linspace(0, L, N + 2)
    X, Y = np.meshgrid(x[1:-1], y[1:-1]) # 내부 격자만 생성
    # 2. 2D Laplacian 행렬 구성 (Kronecker Product 활용)
    # 2D Laplacian = D_xx + D_yy
    # D_xx = I (kron) D_1d
    # D_yy = D_1d (kron) I
    # 1D 2nd deriv matrix (tridiagonal: 1, -2, 1)
    e = np.ones(N)
    D_1d = sparse.spdiags([e, -2*e, e], [-1, 0, 1], N, N)
    I_1d = sparse.eye(N)
    # 2D Laplacian Matrix (Sparse)
    # kron(I, D) + kron(D, I)는 5-point stencil 행렬을 자동으로 만들어줌
    A = sparse.kron(I_1d, D_1d) + sparse.kron(D_1d, I_1d)
    # 3. 우변 벡터 b 생성
    # Source term: f(x,y) = 2(2-x^2-y^2) 라고 가정 (임의 설정)
    # 정답 모양이 산처럼 솟아오르게 하기 위함
    f = np.exp(-10*((X-0.5)**2 + (Y-0.5)**2)) * 100
    b = f.flatten() * (h**2) # 2D 배열을 1D 벡터로 펼침
    # 4. 풀기 (Matrix Solver)
    print(f"Solving Linear System Size: {N*N} x {N*N}")
    u_vec = spla.spsolve(A, -b) # Laplacian u = -f  =>  Au = -b
    # 5. 결과 시각화
    u_sol = u_vec.reshape((N, N)) # 다시 2D로 복원 
    # 경계 조건(0) 패딩
    u_full = np.pad(u_sol, pad_width=1, mode='constant', constant_values=0)
    plt.figure(figsize=(8, 6))
    plt.imshow(u_full, extent=[0, 1, 0, 1], origin='lower', cmap='inferno')
    plt.colorbar(label='u(x, y)')
    plt.title(f"2D Poisson equation solution (N={N}x{N})")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.show()
solve_2d_poisson_fdm()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as la
def solve_1d_fem():
    # ---------------------------------------------------------
    # 1. 문제 설정 및 메쉬 생성
    # ---------------------------------------------------------
    L = 1.0              # 영역 길이
    ne = 10              # 요소(Element) 개수 (늘릴수록 정확해짐)
    nn = ne + 1          # 노드(Node) 개수
    h = L / ne           # 요소 길이 (일정한 간격)
    # 노드 좌표 (Nodes)
    x_nodes = np.linspace(0, L, nn)
    # 연결성 정보 (Connectivity): [노드1, 노드2]
    # 1차원에서는 i번째 요소가 i, i+1번 노드를 연결함
    elements = np.array([[i, i+1] for i in range(ne)])
    # 소스 함수 f(x) (RHS)
    def f_func(x):
        return (np.pi**2) * np.sin(np.pi * x)
    # ---------------------------------------------------------
    # 2. 행렬 초기화
    # ---------------------------------------------------------
    # 글로벌 강성 행렬 (Global Stiffness Matrix) K
    K_global = np.zeros((nn, nn))
    # 글로벌 하중 벡터 (Global Load Vector) F
    F_global = np.zeros(nn)
    # ---------------------------------------------------------
    # 3. 조립 (Assembly) - FEM의 핵심
    # ---------------------------------------------------------
    # 각 요소(Element)를 순회하며 글로벌 행렬에 기여분을 더함
    for i in range(ne):
        # 현재 요소의 노드 인덱스
        idx = elements[i] 
        # 현재 요소의 x 좌표들
        x_elem = x_nodes[idx] 
        # (A) 로컬 강성 행렬 (Local Stiffness Matrix) k
        # 선형 요소(Linear Element)의 경우:
        # integral(phi_i' * phi_j') dx = 1/h * [[1, -1], [-1, 1]]
        k_local = (1.0 / h) * np.array([[1, -1], 
                                        [-1, 1]])
        # (B) 로컬 하중 벡터 (Local Load Vector) f
        # f_i = integral(f(x) * phi_i(x)) dx
        # 간단히 중점 법칙(Trapezoidal) 등으로 근사: f_avg * h / 2
        #  여기서는 양 끝점 값의 평균을 사용
        f_val = f_func(x_elem)
        f_local = (h / 2.0) * np.array([f_val[0], f_val[1]])
        # (C) 글로벌 행렬에 더하기 (Assembly)
        # k_local의 (a, b) 성분을 K_global의 (idx[a], idx[b])에 더함
        for a in range(2): # 로컬 행
            row = idx[a]
            F_global[row] += f_local[a] # 우변 벡터 조립
            for b in range(2): # 로컬 열
                col = idx[b]
                K_global[row, col] += k_local[a, b] # 강성 행렬 조립
    # ---------------------------------------------------------
    # 4. 경계 조건 적용 (Boundary Conditions)
    # ---------------------------------------------------------
    # u(0) = 0, u(L) = 0
    # 행렬의 첫 번째(0)와 마지막(nn-1) 행/열을 제거하고,
    # 내부 노드(1 ~ nn-2)에 대해서만 푼다.
    inner_indices = np.arange(1, nn - 1)
    # 내부 행렬 슬라이싱 (Slicing)
    K_inner = K_global[np.ix_(inner_indices, inner_indices)]
    F_inner = F_global[inner_indices]
    # ---------------------------------------------------------
    # 5. 선형 방정식 풀이 (Solve)
    # ---------------------------------------------------------
    # K * u = F
    u_inner = la.solve(K_inner, F_inner)
    # 전체 해 벡터 구성 (경계값 0 포함)
    u_sol = np.zeros(nn)
    u_sol[inner_indices] = u_inner
    # ---------------------------------------------------------
    # 6. 결과 검증 및 시각화
    # ---------------------------------------------------------
    # 정답 (Exact Solution)
    x_fine = np.linspace(0, L, 200)
    u_exact = np.sin(np.pi * x_fine)
    print(f"Nodes: {nn}")
    print(f"Max error: {np.max(np.abs(u_sol - np.sin(np.pi * x_nodes))):.2e}")
    plt.figure(figsize=(8, 5))
    # 정답 곡선
    plt.plot(x_fine, u_exact, 'k-', alpha=0.6, label='Exact solution')
    # FEM 해 (점으로 표시 + 선형 보간)
    plt.plot(x_nodes, u_sol, 'r-o', markerfacecolor='white', label='FEM solution(Linear basis)')
    # Hat Function 개념 시각화 (선택적)
    # plt.fill_between(x_nodes, 0, u_sol, color='red', alpha=0.1)
    plt.title(f"1D FEM for Poisson equation(Elements={ne})")
    plt.xlabel("x", fontsize=18)
    plt.ylabel("u(x)", fontsize=18)
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.show()
if __name__ == "__main__":
    solve_1d_fem()	

In [ ]:
import numpy as np
import scipy.sparse as sparse
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
import matplotlib.tri as tri
def solve_fem_2d_poisson():
    # ---------------------------------------------------------
    # 1. 메쉬 생성 (Mesh Generation)
    # ---------------------------------------------------------
    # 사각형 영역을 삼각형들로 쪼갠다.
    nx, ny = 20, 20  # x, y 방향 분할 수
    x = np.linspace(0, 1, nx + 1)
    y = np.linspace(0, 1, ny + 1)
    X, Y = np.meshgrid(x, y)
    # 노드(Node) 좌표 배열 (N_nodes x 2)
    nodes = np.column_stack((X.flatten(), Y.flatten())) 
    n_nodes = nodes.shape[0]
    # 요소(Element) 연결 정보 생성 (삼각형)
    # 사각형 격자를 대각선으로 잘라 두 개의 삼각형으로 만듦
    elements = []
    for j in range(ny):
        for i in range(nx):
            # 사각형의 네 모서리 노드 인덱스
            n0 = j * (nx + 1) + i
            n1 = n0 + 1
            n2 = (j + 1) * (nx + 1) + i
            n3 = n2 + 1
            # 삼각형 1 (n0, n1, n2)
            elements.append([n0, n1, n2])
            # 삼각형 2 (n1, n3, n2)
            elements.append([n1, n3, n2])
    elements = np.array(elements)
    n_elements = elements.shape[0]
    print(f"FEM 설정: 노드 {n_nodes}개, 요소 {n_elements}개")
    # ---------------------------------------------------------
    # 2. 로컬 강성 행렬 (Local Stiffness Matrix) 계산 함수
    # ---------------------------------------------------------
    # 선형 삼각형 요소의 강성 행렬 계산
    # K_local = Area * (dN/dx * dN/dx^T + dN/dy * dN/dy^T)
    def get_local_matrix(coords):
        # coords: 3x2 matrix (세 꼭짓점의 x, y 좌표)
        x, y = coords[:, 0], coords[:, 1]
        # 행렬식(면적의 2배) 계산 및 그래디언트 준비
        # b_i = y_j - y_k, c_i = x_k - x_j
        b = np.array([y[1]-y[2], y[2]-y[0], y[0]-y[1]])
        c = np.array([x[2]-x[1], x[0]-x[2], x[1]-x[0]])
        area = 0.5 * np.abs(x[0]*(y[1]-y[2]) + x[1]*(y[2]-y[0]) + x[2]*(y[0]-y[1]))
        # B matrix (Gradient matrix)
        B = np.zeros((2, 3))
        B[0, :] = b / (2 * area) # dN/dx
        B[1, :] = c / (2 * area) # dN/dy
        # K_local = Area * (B.T @ B) (Integrate over area)
        # 선형 요소이므로 B가 상수라 적분이 단순 곱셈이 됨
        K_loc = area * (B.T @ B)
        return K_loc, area
    # ---------------------------------------------------------
    # 3. 글로벌 강성 행렬 조립 (Assembly)
    # ---------------------------------------------------------
    # 희소 행렬 생성을 위한 리스트 (Coordinate format 용)
    row_ind = []
    col_ind = []
    data = []
    # 우변 벡터 F (Load Vector)
    F = np.zeros(n_nodes)
    # 모든 요소에 대해 루프
    for el in elements:
        # 현재 요소의 노드 인덱스들
        idx = el 
        # 현재 요소의 좌표
        el_coords = nodes[idx]
        # 로컬 매트릭스 계산
        k_local, area = get_local_matrix(el_coords)
        # 글로벌 행렬에 더해넣기 (Assembly)
        # 3x3 로컬 행렬의 각 성분을 적절한 글로벌 위치에 배치
        for a in range(3):
            for b in range(3):
                row_ind.append(idx[a])
                col_ind.append(idx[b])
                data.append(k_local[a, b])
        # 우변 벡터 조립 (Source term f = 1)
        # 선형 요소에서 상수 하중은 각 노드에 1/3씩 분배
        F[idx] += 1.0 * area / 3.0
    # COO 포맷으로 희소 행렬 생성 (중복된 인덱스는 자동으로 더해짐)
    K_global = sparse.coo_matrix((data, (row_ind, col_ind)), shape=(n_nodes, n_nodes)).tocsr()
    # ---------------------------------------------------------
    # 4. 경계 조건 적용 (Boundary Conditions)
    # ---------------------------------------------------------
    # Dirichlet BC: u = 0 at boundary
    # 경계 노드 찾기 (x=0, x=1, y=0, y=1)
    tol = 1e-6
    boundary_nodes = np.where(
        (nodes[:, 0] < tol) | (nodes[:, 0] > 1.0 - tol) |
        (nodes[:, 1] < tol) | (nodes[:, 1] > 1.0 - tol)
    )[0]
    # "Penalty Method" 대신 "Direct Zeroing" 사용
    # 행렬의 해당 행을 0으로 밀고 대각 성분을 1로 만듦, 우변을 0으로 설정
    # 희소 행렬 구조를 변경하지 않기 위해 리스트 수정 대신 수학적 트릭 사용
    # (큰 수(1e9)를 대각선에 더해서 강제로 값을 고정하는 방식이 코드상 간단함)
    penalty = 1e9
    for node_idx in boundary_nodes:
        K_global[node_idx, node_idx] += penalty
        F[node_idx] = 0.0 * penalty # u=0
    # ---------------------------------------------------------
    # 5. 해 구하기
    # ---------------------------------------------------------
    print("선형 방정식 풀이 중...")
    u = spla.spsolve(K_global, F)
    # ---------------------------------------------------------
    # 6. 시각화
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 8))
    # Triangulation 객체 생성
    triang = tri.Triangulation(nodes[:, 0], nodes[:, 1], elements)
    # 등고선 그리기
    plt.tripcolor(triang, u, shading='gouraud', cmap='inferno')
    plt.colorbar(label='Solution u(x,y)')
    # 메쉬 그리기 (요소 모양 확인용)
    plt.triplot(triang, 'k-', lw=0.5, alpha=0.3)
    plt.title("2D FEM solution: Poisson equation")
    plt.xlabel("x", fontsize=18)
    plt.ylabel("y", fontsize=18)
    plt.axis('equal')
    plt.show()
solve_fem_2d_poisson()	

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------------------------------
# A. 뉴턴의 분할 차분 보간법 (Newton's Divided Difference Interpolation)
# ----------------------------------------------------

def newton_divided_difference(x_data, y_data):
    """
    주어진 데이터 점들을 사용하여 분할 차분 계수들을 계산한다.
    """
    n = len(x_data)
    # y_data를 복사하여 분할 차분 테이블의 첫 번째 열로 사용한다.
    coef = y_data.copy()
    
    for i in range(1, n):
        for j in range(n - 1, i - 1, -1):
            # 분할 차분 공식: f[x_i, ..., x_k] = (f[x_{i+1}, ..., x_k] - f[x_i, ..., x_{k-1}]) / (x_k - x_i)
            coef[j] = (coef[j] - coef[j-1]) / (x_data[j] - x_data[j-i])
            
    # 첫 번째 요소들(f[x0], f[x0, x1], f[x0, x1, x2], ...)이 보간 다항식의 계수이다.
    return coef

def newton_interpolation(x, x_data, coef):
    """
    뉴턴 보간 다항식에 기반하여 주어진 x 값에서 보간 값을 계산한다.
    """
    n = len(x_data)
    result = coef[n-1]
    
    # Horner's method와 유사하게 역순으로 다항식을 계산한다.
    for i in range(n - 2, -1, -1):
        # P_k(x) = P_{k-1}(x) * (x - x_k) + f[x0, ..., x_k]
        result = result * (x - x_data[i]) + coef[i]
        
    return result

# ----------------------------------------------------
# B. 네빌 보간법 (Neville's Algorithm)
# ----------------------------------------------------

def neville_interpolation(x, x_data, y_data):
    """
    주어진 x 값에서 네빌의 알고리즘을 사용하여 보간 값을 계산한다.
    """
    n = len(x_data)
    # P_i, i 테이블 (1차원 배열로 시작)
    P = y_data.copy()
    
    for k in range(1, n): # k는 다항식의 차수
        for i in range(n - k): # i는 시작 인덱스
            # 네빌의 알고리즘 공식: 
            # P_{i, i+k}(x) = [(x - x_i)P_{i+1, i+k}(x) - (x - x_{i+k})P_{i, i+k-1}(x)] / (x_{i+k} - x_i)
            P[i] = ((x - x_data[i]) * P[i+1] - (x - x_data[i+k]) * P[i]) / (x_data[i+k] - x_data[i])
            
    # 최종 결과는 P 테이블의 첫 번째 요소 P[0]에 저장된다. (P_{0, n-1}(x))
    return P[0]

# ----------------------------------------------------
# C. 실행 및 시각화 예시
# ----------------------------------------------------

# 1. 보간에 사용할 데이터 점 생성
# f(x) = cos(x) 함수를 보간해본다.
x_nodes = np.linspace(0, 2 * np.pi, 7) # 7개의 보간 노드 (0부터 2파이까지)
y_nodes = np.cos(x_nodes)

# 2. 보간을 수행할 x 값 범위
x_range = np.linspace(0, 2 * np.pi, 100)
y_true = np.cos(x_range)

# 3. 뉴턴 보간 수행
newton_coefs = newton_divided_difference(x_nodes, y_nodes)
y_newton = np.array([newton_interpolation(x, x_nodes, newton_coefs) for x in x_range])

# 4. 네빌 보간 수행
y_neville = np.array([neville_interpolation(x, x_nodes, y_nodes) for x in x_range])

# 5. 시각화
plt.figure(figsize=(10, 6))
plt.plot(x_range, y_true, label='True function: $f(x) = \cos(x)$', color='black', linestyle='--')
plt.plot(x_range, y_newton, label='Newton interpolation (7 points)', color='blue')
plt.plot(x_range, y_neville, label='Neville interpolation (7 points)', color='red', linestyle=':')
plt.scatter(x_nodes, y_nodes, label='Interpolation nodes', color='green', marker='o')

plt.title('Newton vs. Neville interpolation of $f(x) = \cos(x)$')
plt.xlabel('$x$')
plt.ylabel('$y$')
plt.legend()
plt.grid(True)
plt.show()

# 6. 특정 값에서의 보간 결과 비교
x_test = np.pi / 4
y_test_true = np.cos(x_test)
y_test_newton = newton_interpolation(x_test, x_nodes, newton_coefs)
y_test_neville = neville_interpolation(x_test, x_nodes, y_nodes)

print("\n--- 특정 값에서의 보간 결과 비교 ---")
print(f"Test x = {x_test:.4f} ($\pi$/4)")
print(f"True value: {y_test_true:.8f}")
print(f"Newton interpolation: {y_test_newton:.8f}")
print(f"Neville interpolation: {y_test_neville:.8f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline, BarycentricInterpolator
from scipy.special import comb

# ==========================================
# 1. 함수 정의 및 도구 설정
# ==========================================
# (1) 타겟 함수: 룽게 함수 (Runge Function)
def runge_function(x):
    return 1 / (1 + 25 * x**2)

# (2) 체비쇼프 노드 생성 함수
def get_chebyshev_nodes(n, a=-1, b=1):
    k = np.arange(1, n + 1)
    x_cheb = np.cos((2 * k - 1) / (2 * n) * np.pi)
    return 0.5 * (a + b) + 0.5 * (b - a) * x_cheb

# (3) 번스타인 다항식 (Bernstein Polynomial) - 근사
def bernstein_approx(func, n, x_eval, a=-1, b=1):
    """
    일반 구간 [a, b]에서의 번스타인 다항식 근사 계산
    """
    # x 좌표를 [0, 1] 구간의 t로 변환 (매핑)
    t = (x_eval - a) / (b - a)    
    y_approx = np.zeros_like(x_eval)
    for k in range(n + 1):
        # 1. 제어점(가중치) 계산: 등간격 위치에서의 함수 값
        # 주의: 번스타인 다항식은 k/n 지점의 값을 계수로 사용함
        x_node = a + (b - a) * (k / n)
        coeff = func(x_node)
        # 2. 번스타인 기저 함수: nCk * t^k * (1-t)^(n-k)
        basis = comb(n, k) * (t**k) * ((1 - t)**(n - k))
        y_approx += coeff * basis
    return y_approx

# ==========================================
# 2. 메인 실행 및 비교
# ==========================================

# 설정: 다항식의 차수 (Node 개수 - 1)
N_nodes = 15      # 점의 개수
Degree = N_nodes - 1 # 다항식 차수 (14차)
# --- A. 데이터 준비 ---
# 1. 등간격 노드 (뉴턴, 라그랑주, 번스타인용)
x_equi = np.linspace(-1, 1, N_nodes)
y_equi = runge_function(x_equi)
# 2. 체비쇼프 노드 (체비쇼프 보간용)
x_cheb = get_chebyshev_nodes(N_nodes, -1, 1)
y_cheb = runge_function(x_cheb)
# 3. 평가용 고밀도 x축
x_dense = np.linspace(-1, 1, 500)
y_true = runge_function(x_dense)
# --- B. 각 방법별 계산 ---
# 1. [보간] 뉴턴/라그랑주 (등간격) - BarycentricInterpolator가 수치적으로 더 안정적임
# (수학적으로 뉴턴, 네빌, 라그랑주와 동일한 다항식)
poly_equi = BarycentricInterpolator(x_equi, y_equi)
y_poly_equi = poly_equi(x_dense)
# 2. [보간] 체비쇼프 노드 활용
poly_cheb = BarycentricInterpolator(x_cheb, y_cheb)
y_poly_cheb = poly_cheb(x_dense)
# 3. [보간] 3차 스플라인 (등간격)
cs = CubicSpline(x_equi, y_equi)
y_spline = cs(x_dense)
# 4. [근사] 번스타인 다항식 (등간격 정보 이용)
y_bernstein = bernstein_approx(runge_function, Degree, x_dense, -1, 1)
# ==========================================
# 3. 결과 시각화
# ==========================================
plt.figure(figsize=(14, 8))

# 0. 참값 (배경)
plt.plot(x_dense, y_true, color='black', linewidth=3, alpha=0.2, label='True function(Runge)')

# 1. 등간격 다항식 보간 (뉴턴/라그랑주) -> 실패 (룽게 현상)
plt.plot(x_dense, y_poly_equi, 'r--', linewidth=1, label=f'Polynomial Interpolation (Equidistant, N={N_nodes})')
# plt.plot(x_equi, y_equi, 'ro', markersize=4) # 점이 너무 많아 복잡하면 주석 처리

# 2. 체비쇼프 보간 -> 성공 (완벽함)
plt.plot(x_dense, y_poly_cheb, 'b-', linewidth=2, alpha=0.8, label=f'Chebyshev Interpolation (Nodes N={N_nodes})')
plt.plot(x_cheb, y_cheb, 'bo', markersize=6) # 체비쇼프 노드 위치 표시

# 3. 3차 스플라인 -> 성공 (안정적)
plt.plot(x_dense, y_spline, color='purple', linestyle='-.', linewidth=2, label='Cubic spline')

# 4. 번스타인 근사 -> 성공 (부드러움, 그러나 오차 있음)
plt.plot(x_dense, y_bernstein, color='green', linewidth=3, label=f'Bernstein Approximation (Degree {Degree})')

# 그래프 꾸미기
plt.title(f"Comprehensive comparison: Interpolation vs Approximation (N={N_nodes})", fontsize=15)
plt.ylim(-0.5, 1.5) # 룽게 현상의 발산으로 인해 y축 제한
plt.xlabel("x")
plt.ylabel("y")
plt.legend(loc='upper right', framealpha=0.9, shadow=True)
plt.grid(True, linestyle='--', alpha=0.5)

# 설명 텍스트 추가
info_text = (
    "1. Red (Equidistant Poly): Fails (Runge phenomenon)\n"
    "2. Blue (Chebyshev): Perfect interpolation\n"
    "3. Purple (Spline): Good interpolation (Piecewise)\n"
    "4. Green (Bernstein): Smooth approximation (Does not hit points)"
)
plt.text(-0.8, 1.25, info_text, fontsize=10, bbox=dict(facecolor='white', alpha=0.9))
plt.savefig('bernstein.png')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import BarycentricInterpolator

# 1. 룽게 함수 정의
def runge_func(x):
    return 1 / (1 + 25 * x**2)

# 2. 보간을 수행할 함수
def interpolate_compare(n_points):
    x_fine = np.linspace(-1, 1, 1000) # 그래프를 그리기 위한 촘촘한 x
    y_true = runge_func(x_fine)
    
    # Case A: 등간격 노드 (Equidistant nodes) - 룽게 현상 발생
    x_equi = np.linspace(-1, 1, n_points)
    y_equi = runge_func(x_equi)
    # 고차 보간에는 수치적으로 안정적인 BarycentricInterpolator 사용
    P_equi = BarycentricInterpolator(x_equi, y_equi)
    y_poly_equi = P_equi(x_fine)
    
    # Case B: 체비쇼프 노드 (Chebyshev nodes) - 룽게 현상 해결
    # 체비쇼프 노드 공식: cos((2k-1)/(2n)*pi)
    k = np.arange(1, n_points + 1)
    x_cheb = np.cos((2 * k - 1) / (2 * n_points) * np.pi)
    y_cheb = runge_func(x_cheb)
    P_cheb = BarycentricInterpolator(x_cheb, y_cheb)
    y_poly_cheb = P_cheb(x_fine)
    
    # --- 시각화 ---
    plt.figure(figsize=(12, 6))
    
    # 원본 함수
    plt.plot(x_fine, y_true, 'k-', linewidth=2, label='True Function $1/(1+25x^2)$', alpha=0.6)
    
    # 등간격 보간 결과
    plt.plot(x_fine, y_poly_equi, 'r--', linewidth=1.5, label=f'Equidistant (n={n_points})')
    plt.plot(x_equi, y_equi, 'ro', markersize=6)
    
    # 체비쇼프 보간 결과
    plt.plot(x_fine, y_poly_cheb, 'g-', linewidth=1.5, label=f'Chebyshev (n={n_points})')
    plt.plot(x_cheb, y_cheb, 'go', markersize=6)
    
    plt.title(f"Runge's Phenomenon Analysis (Degree: {n_points-1})")
    plt.ylim(-0.5, 1.5) # 진동이 심해 그래프가 깨지는 것 방지
    plt.legend()
    plt.grid(True)
    plt.savefig('rp_c.png')
    plt.show()

# 차수를 높여서 테스트 (n=15 정도면 룽게 현상이 확연히 보임)
interpolate_compare(n_points=15)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------------------------------
# A. 뉴턴의 분할 차분 보간법 (Newton's Divided Difference Interpolation)
# ----------------------------------------------------

def newton_divided_difference(x_data, y_data):
    """
    주어진 데이터 점들을 사용하여 분할 차분 계수들을 계산한다.
    """
    n = len(x_data)
    # y_data를 복사하여 분할 차분 테이블의 첫 번째 열로 사용한다.
    coef = y_data.copy()
    
    for i in range(1, n):
        for j in range(n - 1, i - 1, -1):
            # 분할 차분 공식: f[x_i, ..., x_k] = (f[x_{i+1}, ..., x_k] - f[x_i, ..., x_{k-1}]) / (x_k - x_i)
            coef[j] = (coef[j] - coef[j-1]) / (x_data[j] - x_data[j-i])
            
    # 첫 번째 요소들(f[x0], f[x0, x1], f[x0, x1, x2], ...)이 보간 다항식의 계수이다.
    return coef

def newton_interpolation(x, x_data, coef):
    """
    뉴턴 보간 다항식에 기반하여 주어진 x 값에서 보간 값을 계산한다.
    """
    n = len(x_data)
    result = coef[n-1]
    
    # Horner's method와 유사하게 역순으로 다항식을 계산한다.
    for i in range(n - 2, -1, -1):
        # P_k(x) = P_{k-1}(x) * (x - x_k) + f[x0, ..., x_k]
        result = result * (x - x_data[i]) + coef[i]
        
    return result

# ----------------------------------------------------
# B. 네빌 보간법 (Neville's Algorithm)
# ----------------------------------------------------

def neville_interpolation(x, x_data, y_data):
    """
    주어진 x 값에서 네빌의 알고리즘을 사용하여 보간 값을 계산한다.
    """
    n = len(x_data)
    # P_i, i 테이블 (1차원 배열로 시작)
    P = y_data.copy()
    
    for k in range(1, n): # k는 다항식의 차수
        for i in range(n - k): # i는 시작 인덱스
            # 네빌의 알고리즘 공식: 
            # P_{i, i+k}(x) = [(x - x_i)P_{i+1, i+k}(x) - (x - x_{i+k})P_{i, i+k-1}(x)] / (x_{i+k} - x_i)
            P[i] = ((x - x_data[i]) * P[i+1] - (x - x_data[i+k]) * P[i]) / (x_data[i+k] - x_data[i])
            
    # 최종 결과는 P 테이블의 첫 번째 요소 P[0]에 저장된다. (P_{0, n-1}(x))
    return P[0]

# ----------------------------------------------------
# C. 실행 및 시각화 예시
# ----------------------------------------------------

# 1. 보간에 사용할 데이터 점 생성
# f(x) = cos(x) 함수를 보간해본다.
x_nodes = np.linspace(0, 2 * np.pi, 7) # 7개의 보간 노드 (0부터 2파이까지)
y_nodes = np.cos(x_nodes)

# 2. 보간을 수행할 x 값 범위
x_range = np.linspace(0, 2 * np.pi, 100)
y_true = np.cos(x_range)

# 3. 뉴턴 보간 수행
newton_coefs = newton_divided_difference(x_nodes, y_nodes)
y_newton = np.array([newton_interpolation(x, x_nodes, newton_coefs) for x in x_range])

# 4. 네빌 보간 수행
y_neville = np.array([neville_interpolation(x, x_nodes, y_nodes) for x in x_range])

# 5. 시각화
plt.figure(figsize=(10, 6))
plt.plot(x_range, y_true, label='True function: $f(x) = \cos(x)$', color='black', linestyle='--')
plt.plot(x_range, y_newton, label='Newton interpolation (7 points)', color='blue')
plt.plot(x_range, y_neville, label='Neville interpolation (7 points)', color='red', linestyle=':')
plt.scatter(x_nodes, y_nodes, label='Interpolation nodes', color='green', marker='o')

plt.title('Newton vs. Neville interpolation of $f(x) = \cos(x)$')
plt.xlabel('$x$')
plt.ylabel('$y$')
plt.legend()
plt.grid(True)
plt.show()

# 6. 특정 값에서의 보간 결과 비교
x_test = np.pi / 4
y_test_true = np.cos(x_test)
y_test_newton = newton_interpolation(x_test, x_nodes, newton_coefs)
y_test_neville = neville_interpolation(x_test, x_nodes, y_nodes)

print("\n--- 특정 값에서의 보간 결과 비교 ---")
print(f"Test x = {x_test:.4f} ($\pi$/4)")
print(f"True value: {y_test_true:.8f}")
print(f"Newton interpolation: {y_test_newton:.8f}")
print(f"Neville interpolation: {y_test_neville:.8f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline, lagrange

# ==========================================
# 1. 뉴턴 분할 차분 보간법 (Newton)
# ==========================================
def newton_divided_difference_coeffs(x, y):
    n = len(x)
    coef = np.zeros([n, n])
    coef[:, 0] = y
    for j in range(1, n):
        for i in range(n - j):
            coef[i][j] = (coef[i + 1][j - 1] - coef[i][j - 1]) / (x[i + j] - x[i])
    return coef[0, :]

def newton_evaluate(coeffs, x_nodes, x_val):
    n = len(x_nodes) - 1
    p = coeffs[n]
    for k in range(1, n + 1):
        p = coeffs[n - k] + (x_val - x_nodes[n - k]) * p
    return p

# ==========================================
# 2. 네빌 알고리즘 (Neville)
# ==========================================
def neville_interpolate(x_nodes, y_nodes, x_target):
    n = len(x_nodes)
    Q = np.zeros((n, n))
    Q[:, 0] = y_nodes
    for i in range(1, n):
        for j in range(1, i + 1):
            numerator = (x_target - x_nodes[i-j]) * Q[i, j-1] - (x_target - x_nodes[i]) * Q[i-1, j-1]
            denominator = x_nodes[i] - x_nodes[i-j]
            Q[i, j] = numerator / denominator
    return Q[n-1, n-1]

# ==========================================
# 3. 테스트 및 실행 (Main)
# ==========================================
# (1) 참값 함수: 룽게 함수
def true_function(x):
    return 1 / (1 + 25 * x**2)
# (2) 데이터 포인트 생성 (-1 ~ 1 사이 11개 점)
num_nodes = 11
x_nodes = np.linspace(-1, 1, num_nodes)
y_nodes = true_function(x_nodes)
# (3) 촘촘한 x축 (그래프용)
x_dense = np.linspace(-1, 1, 400)
y_true = true_function(x_dense)
# --- A. 뉴턴 보간법 ---
newton_coeffs = newton_divided_difference_coeffs(x_nodes, y_nodes)
y_newton = [newton_evaluate(newton_coeffs, x_nodes, val) for val in x_dense]
# --- B. 네빌 보간법 ---
y_neville = [neville_interpolate(x_nodes, y_nodes, val) for val in x_dense]
# --- C. 라그랑주 보간법 (추가됨) ---
# SciPy의 lagrange 함수를 사용하여 다항식 객체를 생성한다.
# 주의: 라그랑주 방식은 노드가 많아지면(20개 이상) 수치적으로 매우 불안정해질 수 있다.
lagrange_poly = lagrange(x_nodes, y_nodes)
y_lagrange = lagrange_poly(x_dense)
# --- D. 3차 스플라인 ---
cs = CubicSpline(x_nodes, y_nodes)
y_spline = cs(x_dense)
# (4) 오차 비교 (x=0.5 지점)
check_point = 0.5
exact_val = true_function(check_point)
print(f"=== 보간 방법별 비교 (x = {check_point}) ===")
print(f"1. 참값 (Exact)    : {exact_val:.6f}")
print("-" * 40)
print(f"2. 뉴턴 (Newton)   : {newton_evaluate(newton_coeffs, x_nodes, check_point):.6f}")
print(f"3. 네빌 (Neville)  : {neville_interpolate(x_nodes, y_nodes, check_point):.6f}")
print(f"4. 라그랑주(Lagr.) : {lagrange_poly(check_point):.6f}")
print("-" * 40)
print(f"5. 스플라인(Spline): {cs(check_point):.6f}")
print("\n>>> 확인: 뉴턴, 네빌, 라그랑주는 모두 동일한 값을 가진다 (보간 다항식의 유일성).")
# (5) 시각화
plt.figure(figsize=(12, 8))
# 원본 함수
plt.plot(x_dense, y_true, 'k-', linewidth=2, alpha=0.3, label='True function')
# 데이터 노드
plt.plot(x_nodes, y_nodes, 'ko', markersize=8, label='Nodes')
# 세 가지 방법 비교 (겹침 확인을 위해 선 스타일을 다르게 설정)
# 1. 뉴턴: 빨간 실선
plt.plot(x_dense, y_newton, 'r-', linewidth=4, alpha=0.5, label='Newton form')
# 2. 라그랑주: 초록 점선 (뉴턴 위에 덧칠해짐)
plt.plot(x_dense, y_lagrange, 'g--', linewidth=2, label='Lagrange form')
# 3. 네빌: 파란 점 (드문드문 찍어서 겹침 확인)
plt.plot(x_dense[::10], y_neville[::10], 'b.', markersize=8, label='Neville points')
# 4. 스플라인 (유일하게 다름)
plt.plot(x_dense, y_spline, 'm-', linewidth=2, label='Cubic spline(best)')
plt.title(f"Comparison of interpolation methods(Nodes N={num_nodes})")
plt.xlabel("x")
plt.ylabel("y")
plt.ylim(-0.5, 2.0)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig('three.png')
plt.show()

In [ ]:
import numpy as np
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt

# 1. 데이터 준비 (알고 있는 드문드문한 데이터 포인트)
# x 값은 반드시 정렬되어 있어야 한다.
x = np.array([0, 1, 2, 3, 4, 5, 6])
y = np.array([3, 1, 4, 1, 5, 9, 2]) # 예시 데이터 (원주율 숫자들)

# 2. 스플라인 보간 함수 생성
# 이 단계에서 데이터 포인트들을 연결하는 스플라인 함수 'cs'가 만들어진다.
# 이 'cs'는 마치 일반 함수처럼 호출할 수 있다.
cs = CubicSpline(x, y)

# 3. 보간할 새로운 지점 생성
# x의 최소값부터 최대값 사이를 촘촘하게 채우는 새로운 x 좌표들을 만든다.
x_new = np.linspace(x.min(), x.max(), 100) # 100개의 촘촘한 점

# 4. 보간 값 계산
# 만들어둔 스플라인 함수 cs에 새로운 x 좌표를 넣으면 보간된 y값이 나온다.
y_new = cs(x_new)

# 5. 결과 시각화
plt.figure(figsize=(8, 5))
plt.plot(x, y, 'o', label='Original data', markersize=8) # 원래 데이터 점
plt.plot(x_new, y_new, '-', label='Cubic spline interpolation') # 보간된 부드러운 선
plt.title("Cubic spline interpolation example")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline, make_interp_spline, UnivariateSpline

def compare_splines():
    # 1. 데이터 생성 (노이즈가 섞인 사인파)
    np.random.seed(42)
    x = np.linspace(0, 10, 15)  # 데이터 점 15개 (듬성듬성)
    y_exact = np.sin(x)
    y_noise = y_exact + np.random.normal(0, 0.12, len(x)) # 노이즈 추가
    # 2. 보간을 위한 촘촘한 x축
    x_new = np.linspace(0, 10, 500)
    # -------------------------------------------------------
    # A. CubicSpline (점들을 정확히 통과)
    # -------------------------------------------------------
    # bc_type='natural': 양 끝 곡률을 0으로 만듦
    cs = CubicSpline(x, y_noise, bc_type='natural')
    y_cubic = cs(x_new)
    # -------------------------------------------------------
    # B. B-Spline (5차 스플라인, 점들을 정확히 통과)
    # -------------------------------------------------------
    # k=5 (Quintic Spline) -> 4계 미분까지 연속, 매우 부드러움
    bs_model = make_interp_spline(x, y_noise, k=5)
    y_bspline = bs_model(x_new)
    # -------------------------------------------------------
    # C. UnivariateSpline (평활화 스플라인, 점을 지나지 않음)
    # -------------------------------------------------------
    # s=1.0: 평활화 계수 (클수록 더 뻣뻣해짐/직선에 가까워짐)
    # 데이터의 노이즈를 무시하고 전체적인 경향(Trend)만 따라감
    us = UnivariateSpline(x, y_noise, s=1.0) 
    y_smooth = us(x_new)
    # -------------------------------------------------------
    # 3. 시각화
    # -------------------------------------------------------
    plt.figure(figsize=(12, 7))
    # 원본 데이터 점
    plt.plot(x, y_noise, 'ko', label='Noisy data points')
    plt.plot(x, y_exact, 'k--', alpha=0.3, label='True signal(hidden)')
    # 스플라인 결과
    plt.plot(x_new, y_cubic, 'r-', label="CubicSpline(Exact, k=3)")
    plt.plot(x_new, y_bspline, 'g-.', label="B-Spline(Exact, k=5)")
    plt.plot(x_new, y_smooth, 'b', linewidth=2, label="UnivariateSpline(Smoothing, s=1.0)")
    plt.title("Comparison of spline interpolations in Python")
    plt.legend()
    plt.grid(True)
    plt.savefig('spline.png')
    plt.show()
    # 4. 미분값 확인 (스플라인의 장점: 해석적 미분 가능)
    # CubicSpline 객체는 .derivative() 메서드로 도함수 객체를 반환
    y_prime = cs.derivative()(x_new)
    print(f"x={x_new[10]:.2f} 에서의 기울기(미분값): {y_prime[10]:.4f}")
compare_splines()

In [ ]:
from scipy.interpolate import interp1d
import numpy as np
# x	데이터 점의 독립 변수 값 배열. 오름차순으로 정렬되어 있어야 한다.
# y	데이터 점의 종속 변수 값 배열. x와 길이가 같아야 한다.
x = [i*1.0 for i in range(10)]
y = [(i*1.0)**2 for i in range(10)]
x = np.array(x)
y = np.array(y)
f_interp = interp1d(x, y, kind='linear')
n_features=100
x_new = np.linspace(min(x), max(x), num=n_features, endpoint=True)
f_new = f_interp(x_new)

In [ ]:
from scipy.interpolate import griddata
import numpy as np

# 흩어진 데이터 지점 (2D)
points = np.random.rand(100, 2) * 10 
values = np.sin(points[:,0]) + np.cos(points[:,1])
# 보간할 새로운 지점 (규칙 격자)
grid_x, grid_y = np.mgrid[0:10:100j, 0:10:100j]
# 선형 보간 수행
interp_linear = griddata(points, values, (grid_x, grid_y), method='linear')

from scipy.interpolate import RegularGridInterpolator

# 격자 좌표 (x, y)
x = np.arange(0, 5)
y = np.arange(0, 3)
# 격자점의 함수 값 (2D 배열)
values = x[:, None] * y[None, :]
# 보간 함수 생성
interp_func = RegularGridInterpolator((x, y), values, method='linear')
# 새로운 지점 (1.5, 1.5)에서 값 추정
point_new = np.array([[1.5, 1.5]])
result = interp_func(point_new)
# result는 (1.5 * 1.5)에 근사한 값을 반환

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RectBivariateSpline

def demo_2d_spline_interpolation():
    # 1. 가상의 지형 데이터 생성 (Ground Truth)
    # 복잡한 지형 함수: f(x, y) = sin(x) * cos(y) 모양의 언덕과 골짜기
    def terrain_func(x, y):
        return np.sin(x) * np.cos(y) * np.exp(-(x**2 + y**2) / 20)
    # 2. 저해상도 데이터 수집 (Low Resolution Input)
    # 가로세로 10x10 개의 드문드문한 격자 데이터만 있다고 가정
    x_coarse = np.linspace(-3, 3, 10)
    y_coarse = np.linspace(-3, 3, 10)
    X_coarse, Y_coarse = np.meshgrid(x_coarse, y_coarse)
    Z_coarse = terrain_func(X_coarse, Y_coarse)
    # ======================================================
    # 3. 2D 스플라인 객체 생성 (Interpolation)
    # ======================================================
    # RectBivariateSpline(x좌표, y좌표, z값)
    # 기본적으로 kx=3, ky=3 (Bicubic Spline)이 적용된다.
    spline_model = RectBivariateSpline(x_coarse, y_coarse, Z_coarse)
    # 4. 고해상도 그리드에서 값 추출 (Upscaling)
    # 가로세로 100x100 (원본보다 100배 촘촘함)
    x_fine = np.linspace(-3, 3, 100)
    y_fine = np.linspace(-3, 3, 100)
    # spline_model(x, y)를 호출하면 해당 좌표의 보간된 z값을 반환 (grid=True가 기본)
    Z_fine = spline_model(x_fine, y_fine)
    # ======================================================
    # 5. 시각화 (비교)
    # ======================================================
    fig = plt.figure(figsize=(14, 6))
    # (1) 원본 저해상도 (Coarse Grid)
    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    # 보기 좋게 와이어프레임(Wireframe)이나 산점도로 표현
    ax1.plot_surface(X_coarse, Y_coarse, Z_coarse, cmap='viridis', edgecolor='k', alpha=0.8)
    ax1.set_title(f"1. Low-res. input (10x10)\n(jagged & limited)", fontsize=12)
    ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Height')
    # (2) 스플라인 보간 결과 (High Resolution Spline)
    X_fine, Y_fine = np.meshgrid(x_fine, y_fine) # 시각화용 메쉬그리드
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    ax2.plot_surface(X_fine, Y_fine, Z_fine, cmap='viridis', edgecolor='none', alpha=0.9)
    ax2.set_title(f"2. Bicubic spline output (100x100)\n(smooth surface)", fontsize=12)
    ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Height')
    plt.tight_layout()
    plt.savefig('2dspline1.png')
    plt.show()
    # 평면 이미지(히트맵)로도 차이 확인
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(Z_coarse, interpolation='nearest', cmap='terrain') # nearest: 픽셀 그대로
    plt.title("Original low-res. image(pixelated)")
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.imshow(Z_fine, interpolation='none', cmap='terrain')
    plt.title("Spline interpolated image(smooth)")
    plt.axis('off')
    plt.savefig('2dspline2.png')
    plt.show()
demo_2d_spline_interpolation()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def barycentric_coords(p, a, b, c):
    """
    점 p가 주어졌을 때, 삼각형 abc에 대한 무게중심 좌표 (lambda1, lambda2, lambda3)를 계산한다.
    사용 원리: Cramer's rule 또는 면적 비율 방식
    """
    x, y = p
    x1, y1 = a
    x2, y2 = b
    x3, y3 = c
    # 전체 삼각형의 2배 면적 (Determinant)
    det_T = (y2 - y3) * (x1 - x3) + (x3 - x2) * (y1 - y3)
    # 각 lambda 계산
    lambda1 = ((y2 - y3) * (x - x3) + (x3 - x2) * (y - y3)) / det_T
    lambda2 = ((y3 - y1) * (x - x3) + (x1 - x3) * (y - y3)) / det_T
    lambda3 = 1.0 - lambda1 - lambda2
    return lambda1, lambda2, lambda3

def plot_barycentric_triangle(ax, vertices, title):
    """
    주어진 꼭짓점(vertices)으로 삼각형을 그리고, 내부를 무게중심 좌표 기반 RGB 색상으로 채운다.
    """
    v1, v2, v3 = vertices
    # 1. 그리드 생성 (삼각형을 포함하는 사각형 영역)
    x_min = min(v1[0], v2[0], v3[0])
    x_max = max(v1[0], v2[0], v3[0])
    y_min = min(v1[1], v2[1], v3[1])
    y_max = max(v1[1], v2[1], v3[1])
    # 해상도 설정
    grid_size = 300
    x_vals = np.linspace(x_min, x_max, grid_size)
    y_vals = np.linspace(y_min, y_max, grid_size)
    X, Y = np.meshgrid(x_vals, y_vals)
    # 이미지 버퍼 준비 (R, G, B, Alpha)
    img = np.zeros((grid_size, grid_size, 4))
    # 2. 모든 픽셀에 대해 무게중심 좌표 계산 및 색상 매핑
    # (벡터 연산으로 최적화 가능하지만, 이해를 돕기 위해 풀어서 씀)
    # 미리 계산된 행렬식 값들 (속도 향상용)
    x1, y1 = v1; x2, y2 = v2; x3, y3 = v3
    det_T = (y2 - y3) * (x1 - x3) + (x3 - x2) * (y1 - y3)
    # 벡터화된 연산
    L1 = ((y2 - y3) * (X - x3) + (x3 - x2) * (Y - y3)) / det_T
    L2 = ((y3 - y1) * (X - x3) + (x1 - x3) * (Y - y3)) / det_T
    L3 = 1.0 - L1 - L2
    # 3. 삼각형 내부 판별 (모든 lambda가 0 이상 1 이하)
    mask = (L1 >= 0) & (L2 >= 0) & (L3 >= 0)
    # 4. 색상 할당 (L1 -> Red, L2 -> Green, L3 -> Blue)
    img[..., 0] = np.where(mask, L1, 1.0) # R
    img[..., 1] = np.where(mask, L2, 1.0) # G
    img[..., 2] = np.where(mask, L3, 1.0) # B
    img[..., 3] = np.where(mask, 1.0, 0.0) # Alpha (외부는 투명)
    # 5. 플로팅
    ax.imshow(img, extent=[x_min, x_max, y_min, y_max], origin='lower')
    ax.plot([x1, x2, x3, x1], [y1, y2, y3, y1], 'k-', linewidth=2) # 테두리
    # 꼭짓점 라벨링
    offsets = [(-0.05, 0.05), (0.05, -0.05), (0, 0.05)]
    labels = [
        f"V1(1,0,0)\nRed", 
        f"V2(0,1,0)\nGreen", 
        f"V3(0,0,1)\nBlue"
    ]
    for i, (v, label) in enumerate(zip(vertices, labels)):
        ax.text(v[0], v[1]-0.03, label, fontsize=8, 
                ha='center', va='center', fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.axis('off')
# --- 메인 실행부 ---
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
# 1. 정삼각형 (Equilateral Triangle)
# 한 변의 길이가 1인 정삼각형
v_equi = [
    (0.0, 0.0),            # V1
    (1.0, 0.0),            # V2
    (0.5, np.sqrt(3)/2)    # V3
]
plot_barycentric_triangle(axes[0], v_equi, "Equilateral triangle(Barycentric color map)")
# 2. 직각삼각형 (Right Triangle)
v_right = [
    (0.0, 0.0),    # V1 (직각 부분)
    (1.0, 0.0),    # V2
    (0.0, 1.0)     # V3
]
plot_barycentric_triangle(axes[1], v_right, "Right triangle(Barycentric color map)")
# 3. 특정 점 테스트 출력
test_point = (0.25, 0.25) # 직각 삼각형 내부의 한 점
l1, l2, l3 = barycentric_coords(test_point, v_right[0], v_right[1], v_right[2])
print(f"--- Example calculation (Right Triangle) ---")
print(f"Vertices: V1={v_right[0]}, V2={v_right[1]}, V3={v_right[2]}")
print(f"Point P: {test_point}")
print(f"Barycentric coordinates (λ1, λ2, λ3): ({l1:.3f}, {l2:.3f}, {l3:.3f})")
print(f"Check sum (λ1+λ2+λ3): {l1+l2+l3:.3f}")
plt.tight_layout()
plt.savefig("barycentric.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

def calculate_triangle_area(p1, p2, p3):
    """
    세 점의 좌표가 주어졌을 때 삼각형의 넓이를 계산 (신발끈 공식 / 외적 활용)
    Area = 0.5 * |x1(y2-y3) + x2(y3-y1) + x3(y1-y2)|
    """
    val = p1[0]*(p2[1]-p3[1]) + p2[0]*(p3[1]-p1[1]) + p3[0]*(p1[1]-p2[1])
    return 0.5 * abs(val)

# --- 1. 설정: 삼각형 꼭짓점과 내부의 점 P ---
V1 = np.array([1, 1])   # Red Node
V2 = np.array([9, 1])   # Green Node
V3 = np.array([5, 8])   # Blue Node
P = np.array([5, 4])    # 내부의 점 (위치를 바꿔보며 테스트 가능)
# 색상 테마 (V1:Red, V2:Green, V3:Blue)
colors = ['#FFCCCC', '#CCFFCC', '#CCCCFF'] # 연한 R, G, B
edge_colors = ['red', 'green', 'blue']
# --- 2. 면적 계산 ---
# 전체 면적
area_total = calculate_triangle_area(V1, V2, V3)
# 부분 삼각형 면적 (반대편 꼭짓점 규칙 적용)
# A1: P와 V2, V3가 만드는 면적 (V1의 가중치 결정)
area_1 = calculate_triangle_area(P, V2, V3)
# A2: P와 V1, V3가 만드는 면적 (V2의 가중치 결정)
area_2 = calculate_triangle_area(P, V1, V3)
# A3: P와 V1, V2가 만드는 면적 (V3의 가중치 결정)
area_3 = calculate_triangle_area(P, V1, V2)
# 가중치(Lambda) 계산
lambda1 = area_1 / area_total
lambda2 = area_2 / area_total
lambda3 = area_3 / area_total
# --- 3. 시각화 ---
fig, ax = plt.subplots(figsize=(10, 8))
# (1) 부분 삼각형 그리기 (Polygon Patch 사용)
# Sub-Triangle 1 (Opposite to V1) -> Red 계열
poly1 = Polygon([P, V2, V3], closed=True, facecolor=colors[0], edgecolor=edge_colors[0], alpha=0.8, label='Area 1 (for V1)')
ax.add_patch(poly1)
# Sub-Triangle 2 (Opposite to V2) -> Green 계열
poly2 = Polygon([P, V1, V3], closed=True, facecolor=colors[1], edgecolor=edge_colors[1], alpha=0.8, label='Area 2 (for V2)')
ax.add_patch(poly2)
# Sub-Triangle 3 (Opposite to V3) -> Blue 계열
poly3 = Polygon([P, V1, V2], closed=True, facecolor=colors[2], edgecolor=edge_colors[2], alpha=0.8, label='Area 3 (for V3)')
ax.add_patch(poly3)
# (2) 점과 텍스트 표시
# 꼭짓점
ax.plot(V1[0], V1[1], 'o', color='red', markersize=10)
ax.text(V1[0]-0.3, V1[1], f"V1\n(λ1={lambda1:.2f})", ha='right', va='center', fontsize=12, fontweight='bold', color='red')
ax.plot(V2[0], V2[1], 'o', color='green', markersize=10)
ax.text(V2[0]+0.3, V2[1], f"V2\n(λ2={lambda2:.2f})", ha='left', va='center', fontsize=12, fontweight='bold', color='green')
ax.plot(V3[0], V3[1], 'o', color='blue', markersize=10)
ax.text(V3[0], V3[1]+0.3, f"V3\n(λ3={lambda3:.2f})", ha='center', va='bottom', fontsize=12, fontweight='bold', color='blue')
# 내부 점 P
ax.plot(P[0], P[1], 'ko', markersize=8)
ax.text(P[0], P[1]-0.3, f"P", ha='center', va='top', fontsize=12, fontweight='bold')
# (3) 각 부분 삼각형의 중심에 면적 텍스트 표시
# 무게중심 구하기 (텍스트 위치용)
c1 = (P + V2 + V3) / 3
c2 = (P + V1 + V3) / 3
c3 = (P + V1 + V2) / 3
ax.text(c1[0], c1[1], f"Area 1\n= {area_1:.1f}", ha='center', va='center', fontsize=10, color='darkred', fontweight='bold')
ax.text(c2[0], c2[1], f"Area 2\n= {area_2:.1f}", ha='center', va='center', fontsize=10, color='darkgreen', fontweight='bold')
ax.text(c3[0], c3[1], f"Area 3\n= {area_3:.1f}", ha='center', va='center', fontsize=10, color='darkblue', fontweight='bold')
# --- 4. 그래프 설정 및 결과 텍스트 ---
ax.set_aspect('equal')
ax.set_xlim(0, 10)
ax.set_ylim(0, 9)
ax.set_title(f"Barycentric weights by area calculation\nTotal area = {area_total:.1f}", fontsize=15)
ax.grid(True, linestyle='--', alpha=0.5)
# 요약 설명 박스
summary_text = (
    f"Calculating Weights for P({P[0]}, {P[1]}):\n\n"
    f"1. Total Area = {area_total:.2f}\n"
    f"2. Sub-Areas:\n"
    f"   Area 1 (Red)   = {area_1:.2f}  -> λ1 = {area_1}/{area_total:.0f} = {lambda1:.3f}\n"
    f"   Area 2 (Green) = {area_2:.2f}  -> λ2 = {area_2}/{area_total:.0f} = {lambda2:.3f}\n"
    f"   Area 3 (Blue)  = {area_3:.2f}  -> λ3 = {area_3}/{area_total:.0f} = {lambda3:.3f}\n\n"
    f"Check Sum: {lambda1:.3f} + {lambda2:.3f} + {lambda3:.3f} = {lambda1+lambda2+lambda3:.1f}"
)
# 그래프 옆이나 안에 텍스트 박스 배치
props = dict(boxstyle='round', facecolor='white', alpha=0.9)
ax.text(0.2, 8.5, summary_text, transform=ax.transData, fontsize=7,
        verticalalignment='top', bbox=props)
plt.savefig('barycentric_weights.png')
plt.show()

In [ ]:
from scipy.interpolate import griddata
import numpy as np

# 흩어진 데이터 지점 (2D)
points = np.random.rand(100, 2) * 10 
values = np.sin(points[:,0]) + np.cos(points[:,1])
# 보간할 새로운 지점 (규칙 격자)
grid_x, grid_y = np.mgrid[0:10:100j, 0:10:100j]
# 선형 보간 수행
interp_linear = griddata(points, values, (grid_x, grid_y), method='linear')

from scipy.interpolate import RegularGridInterpolator

# 격자 좌표 (x, y)
x = np.arange(0, 5)
y = np.arange(0, 3)
# 격자점의 함수 값 (2D 배열)
values = x[:, None] * y[None, :]
# 보간 함수 생성
interp_func = RegularGridInterpolator((x, y), values, method='linear')
# 새로운 지점 (1.5, 1.5)에서 값 추정
point_new = np.array([[1.5, 1.5]])
result = interp_func(point_new)
# result는 (1.5 * 1.5)에 근사한 값을 반환

In [ ]:
import numpy as np

def f(x):
    """미분할 함수: sin(x)"""
    return np.sin(x)

def central_difference(f, x, h):
    """
    중심 차분 공식을 이용한 미분 근사 (수렴 차수 p=2)
    """
    return (f(x + h) - f(x - h)) / (2 * h)

# 분석 지점
x_val = 1.0 
# 정확한 해 (참값)
exact_value = np.cos(x_val) 

print(f"--- 참값 (Exact Value) ---\nf'(1) = cos(1) ≈ {exact_value:.10f}\n")

# 1단계: h 값으로 근사값 계산
h1 = 0.1
D1 = central_difference(f, x_val, h1)
print(f"1. h={h1} 일 때의 근사값 D1: {D1:.10f}")
print(f"   오차: {abs(D1 - exact_value):.10f}")
print("-" * 30)

# 2단계: h/2 값으로 근사값 계산
h2 = h1 / 2  # h2 = 0.05
D2 = central_difference(f, x_val, h2)
print(f"2. h={h2} 일 때의 근사값 D2: {D2:.10f}")
print(f"   오차: {abs(D2 - exact_value):.10f}")
print("-" * 30)

# Richardson 외삽법 적용 (p=2)
p = 2
richardson_extrapolated = D2 + (D2 - D1) / (2**p - 1)

print(f"3. Richardson 외삽값 (Phi_new): {richardson_extrapolated:.10f}")
print(f"   외삽 후 오차: {abs(richardson_extrapolated - exact_value):.10f}")

In [ ]:
import numpy as np

# 검증할 함수 및 그 도함수 (f(x) = sin(x), f'(x) = cos(x))
def f(x):
    return np.sin(x)

def f_prime_exact(x):
    return np.cos(x)

# 1. 중앙 차분 공식 (Central Difference Formula)
# 기본 근사 방법 A(h)
def central_difference(f, x, h):
    """f'(x)의 중앙 차분 근사값 A(h)를 계산한다."""
    return (f(x + h) - f(x - h)) / (2 * h)

# 2. 리처드슨 외삽법 (Richardson Extrapolation, p=2)
# R(h) = (2^p * A(h/2) - A(h)) / (2^p - 1)
def richardson_extrapolation_step(A_h, A_h_half, p=2):
    """주어진 두 근사값과 오차 차수 p를 사용하여 리처드슨 외삽을 수행한다."""
    return (2**p * A_h_half - A_h) / (2**p - 1)

def verify_richardson(x_val, initial_h, max_steps=4):
    """리처드슨 외삽법을 반복적으로 적용하여 정밀도를 검증한다."""
    print(f"--- 리처드슨 외삽법 검증 (f'(x) at x={x_val:.2f}) ---")
    exact_value = f_prime_exact(x_val)
    print(f"정확한 값 (Exact Value): {exact_value:.10f}\n")

    R = np.zeros((max_steps, max_steps))
    h = initial_h

    # 1. 초기 중앙 차분 근사값 계산 (R[i][0])
    for i in range(max_steps):
        R[i, 0] = central_difference(f, x_val, h)
        h /= 2  # h를 절반으로 줄임
    
    h = initial_h # h를 초기값으로 다시 설정 (출력용)

    # 2. 리처드슨 외삽 반복 적용 (R[i][j])
    for j in range(1, max_steps):
        p = 2 * j # 오차 차수: 2, 4, 6, ...
        for i in range(j, max_steps):
            # R[i-1][j-1]은 A(h)의 역할, R[i][j-1]은 A(h/2)의 역할
            R[i, j] = richardson_extrapolation_step(R[i-1, j-1], R[i, j-1], p=p)
            
            # 정밀도 검증을 위해 오차 계산
            error = abs(R[i, j] - exact_value)
            
            print(f"h={h / (2**i):.4f} (단계 {j}): 근사값={R[i, j]:.10f}, 오차={error:.2e}")
        h = initial_h # 다음 열 계산을 위해 초기 h를 다시 사용
    
    print("\n최종 리처드슨 표 (Richardson Tableau):")
    print(R)
    print(f"\n**최고 정밀도:** R[{max_steps-1}][{max_steps-1}] = {R[max_steps-1, max_steps-1]:.10f}")


# 실행
x_value = np.pi / 4 # x=pi/4에서 f'(x) = cos(pi/4) = 0.70710678...
initial_h = 0.1
verify_richardson(x_value, initial_h)

# 네빌 방법을 위해 함수 f(x) = cos(x) 사용
def g(x):
    """검증할 함수 g(x) = cos(x)"""
    return np.cos(x)

def neville_method(x_data, y_data, x_target):
    """
    네빌 방법을 사용하여 x_target에서의 함수값을 근사한다.
    x_data: 알려진 x 좌표 (노드)
    y_data: 알려진 y 좌표 (f(x_data))
    x_target: 추정하려는 x 값
    """
    n = len(x_data)
    P = np.zeros((n, n))
    
    # 1. 초기값 설정: P[i][0] = y_i
    for i in range(n):
        P[i, 0] = y_data[i]

    # 2. 재귀적 공식 적용
    for j in range(1, n): # 열 (다항식 차수)
        for i in range(n - j): # 행 (시작 인덱스)
            # P[i][j] = P_{i, j}(x_target)
            # 분자: (x - x_i) * P_{i+1, j-1} - (x - x_{i+j}) * P_{i, j-1}
            numerator = (x_target - x_data[i]) * P[i + 1, j - 1] - \
                        (x_target - x_data[i + j]) * P[i, j - 1]
            
            # 분모: x_{i+j} - x_i
            denominator = x_data[i + j] - x_data[i]
            
            P[i, j] = numerator / denominator
            
    return P, P[n - 1, n - 1] # 네빌 표와 최종 근사값 반환

def verify_neville(x_data, x_target):
    """네빌 방법을 내삽과 외삽으로 검증한다."""
    y_data = g(x_data)
    
    print(f"\n--- 네빌 방법 검증 (f(x) = cos(x)) ---")
    print(f"데이터 범위: [{x_data.min():.2f}, {x_data.max():.2f}]")
    print(f"추정 목표 x_target: {x_target:.2f}")

    P_table, approx_value = neville_method(x_data, y_data, x_target)
    exact_value = g(x_target)
    error = abs(approx_value - exact_value)

    if x_target >= x_data.min() and x_target <= x_data.max():
        mode = "내삽 (Interpolation)"
    else:
        mode = "외삽 (Extrapolation)"

    print(f"\n모드: {mode}")
    print(f"정확한 값 (Exact Value): {exact_value:.10f}")
    print(f"최종 근사값: {approx_value:.10f}")
    print(f"오차: {error:.2e}")
    print("\n네빌 표 (Neville Tableau):")
    # 마지막 값 P[n-1, n-1]이 최고 차수 다항식의 근사값이다.
    print(P_table)


# **검증 1: 내삽 (Interpolation)**
x_nodes = np.array([0.0, 0.5, 1.0, 1.5]) # 데이터 범위 [0.0, 1.5]
target_int = 0.75 # 범위 내
verify_neville(x_nodes, target_int)

# **검증 2: 외삽 (Extrapolation)**
target_ext = 2.0 # 범위 밖
verify_neville(x_nodes, target_ext)

In [ ]:
import numpy as np
from tabulate import tabulate # 결과를 깔끔하게 테이블로 출력하기 위해 tabulate 라이브러리 사용 (없다면 'pip install tabulate' 필요)

# --- 1. 대상 함수 및 정답 정의 ---

def f(x):
    """미분 대상 함수: f(x) = exp(x)"""
    return np.exp(x)

EXACT_FIRST_DERIVATIVE = 1.0    # f'(0) = 1.0
EXACT_SECOND_DERIVATIVE = 1.0   # f''(0) = 1.0

# --- 2. 수치 미분 공식 정의 ---

# 1차 미분 공식 (First Derivative Formulas)

def d1_O_h2(x, h_step):
    """O(h^2) 1차 미분 공식"""
    return (f(x + h_step) - f(x - h_step)) / (2 * h_step)

def d1_O_h6(x, h_step):
    """O(h^6) 1차 미분 공식"""
    numerator = (
        -f(x - 3 * h_step)
        + 9 * f(x - 2 * h_step)
        - 45 * f(x - h_step)
        + 45 * f(x + h_step)
        - 9 * f(x + 2 * h_step)
        + f(x + 3 * h_step)
    )
    return numerator / (60 * h_step)

# 2차 미분 공식 (Second Derivative Formulas)

def d2_O_h2(x, h_step):
    """O(h^2) 2차 미분 공식"""
    # f''(x) = (f(x+h) - 2f(x) + f(x-h)) / h^2
    return (f(x + h_step) - 2 * f(x) + f(x - h_step)) / (h_step**2)

def d2_O_h6(x, h_step):
    """O(h^6) 2차 미분 공식 (수정됨)"""
    # f''(x) = [2f(x-3h) - 27f(x-2h) + 270f(x-h) - 490f(x) + 270f(x+h) - 27f(x+2h) + 2f(x+3h)] / (180h^2)
    numerator = (
        +2 * f(x - 3 * h_step)
        - 27 * f(x - 2 * h_step)
        + 270 * f(x - h_step)
        - 490 * f(x)
        + 270 * f(x + h_step)
        - 27 * f(x + 2 * h_step)
        + 2 * f(x + 3 * h_step)
    )
    return numerator / (180.0 * h_step**2)

# --- 3. 평가 실행 및 결과 저장 ---
x_eval = 0.0          # 평가 지점
h_initial = 0.1       # 초기 그리드 간격 H
h_A = h_initial       # 시스템 A의 간격
h_B = h_initial / 2   # 시스템 B의 간격
results = []
# 1차 미분 평가
approx_d1_A = d1_O_h6(x_eval, h_A)
error_d1_A = abs(approx_d1_A - EXACT_FIRST_DERIVATIVE)
results.append(["1차 미분", "O(h^6)", f"h={h_A}", approx_d1_A, error_d1_A])
approx_d1_B = d1_O_h2(x_eval, h_B)
error_d1_B = abs(approx_d1_B - EXACT_FIRST_DERIVATIVE)
results.append(["1차 미분", "O(h^2)", f"h/2={h_B}", approx_d1_B, error_d1_B])
# 2차 미분 평가
approx_d2_A = d2_O_h6(x_eval, h_A)
error_d2_A = abs(approx_d2_A - EXACT_SECOND_DERIVATIVE)
results.append(["2차 미분", "O(h^6)", f"h={h_A}", approx_d2_A, error_d2_A])
approx_d2_B = d2_O_h2(x_eval, h_B)
error_d2_B = abs(approx_d2_B - EXACT_SECOND_DERIVATIVE)
results.append(["2차 미분", "O(h^2)", f"h/2={h_B}", approx_d2_B, error_d2_B])
# --- 4. 결과 출력 ---
headers = ["미분 차수", "공식 정확도", "그리드 간격", "근사값", "절대 오차"]
print("\n---  수치 미분 정확도 비교 (f(x)=exp(x), x=0) ---")
print(f"**1차/2차 미분 정답:** {EXACT_FIRST_DERIVATIVE:.4f}\n")
print(tabulate(results, headers=headers, tablefmt="fancy_grid", floatfmt=(".0f", ".0f", ".3f", ".12f", ".10e")))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import dct

# 1. 룽게 함수 정의
def runge_function(x):
    """룽게 함수 f(x) = 1 / (1 + 25x^2)"""
    return 1.0 / (1.0 + 25.0 * x**2)

# 2. Chebyshev 계수 계산 함수 (DCT 이용)
def compute_chebyshev_coeffs(f, N):
    """
    N차 보간 다항식의 Chebyshev 계수 (c_k)를 계산한다. 
    (N+1개의 Chebyshev 마디 이용)
    """
    # N+1 개의 Chebyshev 마디 생성 (구간 [-1, 1])
    # x_j = cos((2j + 1) * pi / (2(N+1))) for j = 0 to N
    j = np.arange(N + 1)
    # Note: Scipy의 dct는 첫 번째 인덱스(j=0)를 cos((j) * pi / N)으로 정의하므로,
    # 여기서는 고유한 Chebyshev 마디 정의를 사용해야 한다.
    # 실제로 Chebyshev 마디에서 함수 값을 구하고 DCT-I (또는 DCT-III)와 유사한 형태를 사용한다.
    
    # Chebyshev-Gauss-Lobatto 마디 (경계를 포함)를 사용하면 DCT-I을 바로 사용할 수 있으나,
    # 여기서는 표준 Chebyshev 마디 (근)를 사용하고 명시적으로 계수를 계산한다.
    
    # 보간점 (Chebyshev 마디)
    x_nodes = np.cos( (np.pi * (j + 0.5)) / (N + 1) )
    y_nodes = f(x_nodes)
    
    # 계수 c_k 계산
    coeffs = np.zeros(N + 1)
    for k in range(N + 1):
        # 직교성을 이용하여 계수 c_k 계산
        c_k = (2.0 / (N + 1)) * np.sum(y_nodes * np.cos(k * np.pi * (j + 0.5) / (N + 1)))
        
        # c_0는 다른 계수의 절반
        if k == 0:
            c_k /= 2.0
            
        coeffs[k] = c_k
        
    return coeffs

# 3. Chebyshev 다항식 평가 함수
def chebyshev_series_eval(x, coeffs):
    """
    Chebyshev 계수(c_k)를 이용하여 x에서의 다항식 값을 계산한다.
    Clenshaw 알고리즘을 사용하면 효율적이지만, 여기서는 T_k(x)를 직접 계산한다.
    """
    N = len(coeffs) - 1
    T = np.zeros((N + 1, len(x))) # T_k(x) 값을 저장할 배열
    
    # T_0(x) = 1
    T[0, :] = 1.0
    
    if N >= 1:
        # T_1(x) = x
        T[1, :] = x
        
        # 재귀 관계 T_{k+1}(x) = 2x * T_k(x) - T_{k-1}(x) 이용
        for k in range(1, N):
            T[k + 1, :] = 2.0 * x * T[k, :] - T[k - 1, :]
            
    # P_N(x) = sum(c_k * T_k(x))
    P_N = np.dot(coeffs, T)
    return P_N

# ----------------- 4. 실행 및 시각화 -----------------
N = 15 # 보간 차수 (N+1개의 보간점)

# 플롯을 위한 x 축 생성 (촘촘한 간격)
x_plot = np.linspace(-1, 1, 200)
y_exact = runge_function(x_plot)

# Chebyshev 계수 계산
cheby_coeffs = compute_chebyshev_coeffs(runge_function, N)

# 보간 다항식 값 계산
y_interp_cheby = chebyshev_series_eval(x_plot, cheby_coeffs)

# 보간에 사용된 Chebyshev 마디 (Lagrange 방식과 동일)
j = np.arange(N + 1)
x_nodes = np.cos( (np.pi * (j + 0.5)) / (N + 1) )
y_nodes = runge_function(x_nodes)


# 시각화
plt.figure(figsize=(8, 6))
plt.plot(x_plot, y_exact, 'k-', linewidth=2, label='Exact Runge function')
plt.plot(x_nodes, y_nodes, 'o', color='blue', markersize=5, label=f'Chebyshev nodes (N+1={N+1})')
plt.plot(x_plot, y_interp_cheby, 'b--', linewidth=1.5, label=f'Chebyshev series interp. (N={N})')

plt.title(f'Chebyshev series interpolation of Runge function (N={N})')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.ylim(-0.2, 1.2) # y축 범위를 조정하여 결과를 명확히 표시
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def create_1d_laplacian(N, h):
	# 주 대각선 (main diagonal)
	main_diag = np.full(N, -2.0 / h**2)
	# 위/아래 대각선 (off diagonals)
	off_diag = np.full(N - 1, 1.0 / h**2)
	
	# SciPy의 diags 함수를 사용하여 희소 삼중대각 행렬 생성
	A = diags([main_diag, off_diag, off_diag], [0, 1, -1], shape=(N, N), format='csc')
	return A
def create_3d_laplacian(Nx, Ny, Nz, hx, hy, hz):
	# 1. 1차원 미분 행렬 및 단위 행렬 생성
	Ax = create_1d_laplacian(Nx, hx)
	Ay = create_1d_laplacian(Ny, hy)
	Az = create_1d_laplacian(Nz, hz)
	Ix = identity(Nx, format='csc')
	Iy = identity(Ny, format='csc')
	Iz = identity(Nz, format='csc')
	# 2. 텐서 프로덕트 (Kronecker Product)를 이용한 3차원 연산자 구성
	# L_x = A_x ⊗ I_y ⊗ I_z
	Lx = kronecker(Ax, kronecker(Iy, Iz, format='csc'), format='csc')
	# L_y = I_x ⊗ A_y ⊗ I_z
	Ly = kronecker(Ix, kronecker(Ay, Iz, format='csc'), format='csc')
	# L_z = I_x ⊗ I_y ⊗ A_z
	Lz = kronecker(Ix, kronecker(Iy, Az, format='csc'), format='csc')
	# 3. 세 연산자를 합하여 최종 3D 라플라스 연산자 생성
	L_3D = Lx + Ly + Lz
	return L_3D

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import correlate1d

def compute_3d_laplacian_4th_order(data, h):
    """
    3차원 데이터에 대해 4차 정확도 중심 차분을 적용하여 라플라시안을 구한다.
    Method: Dimension-by-Dimension application (Matrix-free)
    """
    # 1. 4차 정확도 중심 차분 계수 (Five-point stencil for 2nd derivative)
    # f''(x) ≈ (-f(x-2h) + 16f(x-h) - 30f(x) + 16f(x+h) - f(x+2h)) / (12h^2)
    # weights 배열 순서: [i-2, i-1, i, i+1, i+2]
    weights = np.array([-1, 16, -30, 16, -1]) / (12 * h**2)
    # 2. 각 축(Axis)별로 1차원 컨볼루션(Correlation) 적용
    # correlate1d는 1D 연산자를 지정된 축 방향으로만 적용한다.
    # x축 방향 2계 미분 (∂²f/∂x²)
    d2f_dx2 = correlate1d(data, weights, axis=0, mode='constant', cval=0.0)
    # y축 방향 2계 미분 (∂²f/∂y²)
    d2f_dy2 = correlate1d(data, weights, axis=1, mode='constant', cval=0.0)
    # z축 방향 2계 미분 (∂²f/∂z²)
    d2f_dz2 = correlate1d(data, weights, axis=2, mode='constant', cval=0.0)
    # 3. 라플라시안 합산 (Laplacian = ∂xx + ∂yy + ∂zz)
    laplacian = d2f_dx2 + d2f_dy2 + d2f_dz2
    return laplacian
# --- 메인 실행 코드 ---
# 1. 그리드 설정 (Grid Setup)
N = 60  # 격자 크기 (60^3 = 216,000 포인트)
L = 2 * np.pi
x = np.linspace(0, L, N)
y = np.linspace(0, L, N)
z = np.linspace(0, L, N)
h = x[1] - x[0] # 간격
# meshgrid 생성 (indexing='ij'는 행렬 표기법 순서인 x, y, z를 따름)
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
# 2. 테스트 함수 정의: f = sin(x)sin(y)sin(z)
F = np.sin(X) * np.sin(Y) * np.sin(Z)
# 3. 수치해석적 미분 수행
# (텐서곱 행렬을 만들지 않고 함수로 처리)
numerical_laplacian = compute_3d_laplacian_4th_order(F, h)
# 4. 해석해(Exact Solution) 계산
# ∇²(sin x sin y sin z) = -3 sin x sin y sin z
exact_laplacian = -3 * np.sin(X) * np.sin(Y) * np.sin(Z)
# 5. 오차 분석 (경계면 제외: 차분법은 경계에서 오차가 크므로 내부만 비교)
# Ghost cell 영역(가장자리 2칸)을 제외하고 비교
interior = slice(2, -2)
error = np.abs(numerical_laplacian - exact_laplacian)
max_error = np.max(error[interior, interior, interior])
print(f"격자 크기: {N}x{N}x{N}")
print(f"공간 간격 h: {h:.6f}")
print(f"최대 오차 (내부 영역): {max_error:.6e}")
# 시각화 (Z축의 중간 단면)
mid_idx = N // 2
plt.figure(figsize=(10, 4))
plt.subplot(1, 3, 1)
plt.title("Numerical Laplacian (Slice)")
plt.imshow(numerical_laplacian[:, :, mid_idx], cmap='bwr')
plt.colorbar()
plt.subplot(1, 3, 2)
plt.title("Exact Laplacian (Slice)")
plt.imshow(exact_laplacian[:, :, mid_idx], cmap='bwr')
plt.colorbar()
plt.subplot(1, 3, 3)
plt.title("Error (Difference)")
plt.imshow(error[:, :, mid_idx], cmap='viridis')
plt.colorbar()
plt.tight_layout()
plt.savefig('lap4.png')
plt.show()

In [ ]:
import numpy as np
#  Fornberg formula in python
#  Input Parameters
#    z            -  location where approximations are to be accurate
#    x(0:nd)      -  grid point locations, found in x(0:n)
#    nd           -  dimension of x- and c-arrays in calling
#                    program x(0:nd) and c(0:nd, 0:m), respectively
#    m            -  highest derivative for which weights are sought
#
#  Output Parameter
#    c(0:nd,0:m)  -  weights at grid locations x(0:n) for
#                    derivatives of order 0:m, found in c(0:nd, 0:m)
#
#  References:
#      Generation of Finite Difference Formulas on Arbitrarily
#          Spaced Grids, Bengt Fornberg,
#          Mathematics of compuation, 51, 184, 1988, 699--706,
#          doi: 10.1090/S0025-5718-1988-0935077-0

def weights(z, x, nd, m):
    c1 = 1
    c4 = x[0] - z
    c = np.zeros((nd+1, m+1))
    c[0, 0] = 1
    for i in range(1, nd+1):
        mn = min(i, m)
        c2 = 1
        c5 = c4
        c4 = x[i] - z
        for j in range(0, i):
            c3 = x[i] - x[j]
            c2 = c2*c3
            if j == i-1:
                for k in range(mn, 0, -1):
                    c[i, k] = c1*(k*c[i-1, k-1] - c5*c[i-1, k])/c2
                c[i, 0] = -c1*c5*c[i-1, 0]/c2
            for k in range(mn, 0, -1):
                c[j, k] = (c4*c[j, k] - k*c[j, k-1])/c3
            c[j, 0] = c4*c[j, 0]/c3
        c1 = c2
    return c

def get_yprimes(nleft, m, x, y):
    npt = len(x)
    y1 = np.zeros(npt)
    y2 = np.zeros(npt)
    nright = nleft
    nd = nleft+nright
    xsten = np.zeros(nd+1)
    for j in range(npt):
        z = x[j]
        tmp = 0.
        tmq = 0.
        if j-nleft < 0:
            j0 = 0
            j1 = nd+1
            xsten[0:nd+1] = x[j0:j1]
        elif j-nleft+nd+1 > npt-1:
            j1 = npt
            j0 = j1-nd-1
            xsten[0:nd+1] = x[j0:j1]
        else:
            j0 = j-nleft
            j1 = j0+nd+1
            xsten[0:nd+1] = x[j0:j1]
        c = weights(z, xsten, nd, m)
        for k in range(nd+1):
            tmp = tmp+c[k, 1]*y[j0+k]
            tmq = tmq+c[k, 2]*y[j0+k]
        y1[j] = tmp
        y2[j] = tmq
    return y1, y2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Fornberg 가중치 계산 함수 (변경 없음) ---
def weights(z, x, nd, m):
    """
    Fornberg의 방법을 사용하여 유한 차분 가중치를 계산한다.
    z: 미분/보간 값을 구할 위치
    x: 스텐실(참조) 포인트들의 좌표 배열
    nd: 다항식의 차수 (스텐실 크기 - 1)
    m: 구할 최대 미분 차수
    """
    c1 = 1
    c4 = x[0] - z
    c = np.zeros((nd+1, m+1))
    c[0, 0] = 1
    
    for i in range(1, nd+1):
        mn = min(i, m)
        c2 = 1
        c5 = c4
        c4 = x[i] - z
        
        for j in range(0, i):
            c3 = x[i] - x[j]
            c2 = c2*c3
            
            if j == i-1:
                for k in range(mn, 0, -1):
                    c[i, k] = c1*(k*c[i-1, k-1] - c5*c[i-1, k])/c2
                c[i, 0] = -c1*c5*c[i-1, 0]/c2
            
            for k in range(mn, 0, -1):
                c[j, k] = (c4*c[j, k] - k*c[j, k-1])/c3
            c[j, 0] = c4*c[j, 0]/c3
            
        c1 = c2
    return c

# --- 2. 경계 조건을 반영한 보간 및 미분 함수 ---
def calc_fornberg_with_boundaries(x_eval, x_data, y_data, n_stencil):
    """
    x_eval: 값을 구할 임의의 점들 (Dense Grid)
    x_data: 주어진 데이터의 x좌표 (Coarse Grid)
    y_data: 주어진 데이터의 y값
    n_stencil: 사용할 점의 개수 (홀수 권장, 예: 5)
    """
    npt = len(x_data)
    n_eval = len(x_eval)
    nd = n_stencil - 1  # 다항식 차수
    
    # 이상적인 왼쪽 포인트 개수 (중심 차분을 위해)
    # 예: 5개 점이면 중심 기준 왼쪽 2개, 오른쪽 2개 -> n_left = 2
    n_left = nd // 2 
    
    # 결과 저장 배열
    results = {
        0: np.zeros(n_eval), # 보간값
        1: np.zeros(n_eval), # 1차 미분
        2: np.zeros(n_eval)  # 2차 미분
    }
    
    for i in range(n_eval):
        z = x_eval[i]
        
        # 1. z와 가장 가까운 데이터 포인트의 인덱스(nearest_idx)를 찾음
        nearest_idx = np.abs(x_data - z).argmin()
        
        # 2. 스텐실의 시작 인덱스(j0) 결정
        # 기본적으로는 nearest_idx를 중심으로 하려 함
        j0 = nearest_idx - n_left
        
        # --- [중요] 경계 조건 처리 (제공해주신 로직 반영) ---
        # (1) 좌측 끝 처리: 인덱스가 0보다 작아지면 0으로 고정
        if j0 < 0:
            j0 = 0
            
        # (2) 우측 끝 처리: 스텐실이 배열 끝을 넘어가면, 끝에 딱 맞게 당김
        # 스텐실 끝 인덱스(j1) = j0 + n_stencil
        # j1이 npt보다 크면 안 됨 -> j0를 (npt - n_stencil)로 고정
        elif j0 + n_stencil > npt:
            j0 = npt - n_stencil
            
        # 3. 확정된 스텐실 좌표 및 값 추출
        # x_data[j0] 부터 x_data[j0 + nd] 까지 사용
        x_sten = x_data[j0 : j0 + n_stencil]
        y_sten = y_data[j0 : j0 + n_stencil]
        
        # 4. 가중치 계산 (z 위치에서의 가중치)
        # m=2 (2차 미분까지 계산)
        c = weights(z, x_sten, nd, m=2)
        
        # 5. 내적을 통한 값 계산
        # c[:, k]는 k차 미분에 대한 각 점의 가중치
        results[0][i] = np.dot(c[:, 0], y_sten) # 보간
        results[1][i] = np.dot(c[:, 1], y_sten) # 1차 미분
        results[2][i] = np.dot(c[:, 2], y_sten) # 2차 미분
        
    return results[0], results[1], results[2]

# --- 3. 메인 실행 및 시각화 ---
if __name__ == "__main__":
    # 데이터 생성 (Coarse Grid)
    # 0 ~ 2pi 구간을 10개 점으로 나눔
    N_data = 20
    x_data = np.linspace(0, 2*np.pi, N_data)
    y_data = np.cos(x_data)

    # 평가 지점 (Fine Grid) - 그래프를 부드럽게 그리기 위함
    x_eval = np.linspace(0, 2*np.pi, 100)
    
    # 참값 (Analytical Solutions)
    true_y0 = np.cos(x_eval)
    true_y1 = -np.sin(x_eval)
    true_y2 = -np.cos(x_eval)

    # Fornberg 적용
    # n_stencil=5 -> 4차 다항식 사용 (점 5개 참조)
    y0_calc, y1_calc, y2_calc = calc_fornberg_with_boundaries(x_eval, x_data, y_data, n_stencil=7)

    # 그래프 설정
    plt.figure(figsize=(10, 12))

    # (1) 보간 결과 (Interpolation)
    plt.subplot(3, 1, 1)
    plt.plot(x_eval, true_y0, 'k-', alpha=0.3, linewidth=3, label='True cos(x)')
    plt.plot(x_eval, y0_calc, 'r--', label='Fornberg interp')
    plt.plot(x_data, y_data, 'bo', markersize=6, label='Data points')
    plt.title('Interpolation (m=0) with boundary handling')
    plt.legend()
    plt.grid(True)

    # (2) 1차 미분 결과 (1st Derivative)
    plt.subplot(3, 1, 2)
    plt.plot(x_eval, true_y1, 'k-', alpha=0.3, linewidth=3, label='True -sin(x)')
    plt.plot(x_eval, y1_calc, 'g--', label='Fornberg 1st deriv')
    plt.plot(x_data, -np.sin(x_data), 'gx', alpha=0.5, label='True deriv at nodes') # 참고용 점
    plt.title('1st derivative (m=1)')
    plt.legend()
    plt.grid(True)

    # (3) 2차 미분 결과 (2nd Derivative)
    plt.subplot(3, 1, 3)
    plt.plot(x_eval, true_y2, 'k-', alpha=0.3, linewidth=3, label='True -cos(x)')
    plt.plot(x_eval, y2_calc, 'm--', label='Fornberg 2nd deriv')
    plt.title('2nd derivative (m=2)')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import scipy.sparse as sparse
import scipy.sparse.linalg as splinalg
import matplotlib.pyplot as plt

def solve_poisson_compact(N):
    """
    u''(x) = f(x) 문제를 Compact Scheme(4차)과 Standard Scheme(2차)으로 풀고 비교
    """
    # 1. 그리드 설정
    L = 1.0
    x = np.linspace(0, L, N+1)
    h = L / N
    
    # 2. 테스트 문제 설정 (정해를 알고 있는 문제)
    # 정해: u(x) = sin(k*x) -> u''(x) = -k^2 * sin(k*x)
    k = 4 * np.pi
    u_true = np.sin(k * x)
    f = - (k**2) * np.sin(k * x)  # RHS source term

    # --- 행렬 생성 (Sparse Matrix 사용) ---
    # A Matrix: 좌변의 2계 미분 연산자 (1, -2, 1)
    # Standard와 Compact 모두 좌변 행렬 형태는 같다 (우변 처리만 다름)
    diagonals = [np.ones(N-1), -2*np.ones(N-1), np.ones(N-1)]
    offsets = [-1, 0, 1]
    A = sparse.diags(diagonals, offsets, shape=(N-1, N-1), format='csr')

    # B Matrix: Compact Scheme을 위한 우변 가중치 행렬 (1/12, 10/12, 1/12)
    # 식: (u_{i-1} - 2u_i + u_{i+1}) / h^2 = 1/12 * (f_{i-1} + 10f_i + f_{i+1})
    # 따라서 A * u = h^2 * B * f
    b_diag = [np.ones(N-1)/12, 10*np.ones(N-1)/12, np.ones(N-1)/12]
    B = sparse.diags(b_diag, offsets, shape=(N-1, N-1), format='csr')

    # RHS 벡터 준비 (경계조건 u(0)=0, u(L)=0 이므로 내부 점만 계산)
    f_inner = f[1:-1]

    # --- 3. Standard 2nd Order Solver ---
    # 식: A * u = h^2 * f
    rhs_std = (h**2) * f_inner
    u_std_inner = splinalg.spsolve(A, rhs_std)
    
    # 경계조건 포함하여 전체 해 구성
    u_std = np.zeros(N+1)
    u_std[1:-1] = u_std_inner

    # --- 4. Compact 4th Order Solver (Mehrstellen) ---
    # 식: A * u = h^2 * B * f
    rhs_compact = (h**2) * (B @ f_inner) # 행렬-벡터 곱
    u_compact_inner = splinalg.spsolve(A, rhs_compact)

    # 경계조건 포함
    u_compact = np.zeros(N+1)
    u_compact[1:-1] = u_compact_inner

    return x, u_true, u_std, u_compact

# --- 실행 및 시각화 ---
if __name__ == "__main__":
    # 그리드 포인트 개수 (작게 설정하여 오차 차이를 극대화해서 보여줌)
    N_grid = 45 
    x, exact, std_sol, cpt_sol = solve_poisson_compact(N_grid)

    # 오차 계산
    err_std = np.abs(exact - std_sol)
    err_cpt = np.abs(exact - cpt_sol)

    print(f"Grid Points: {N_grid}")
    print(f"Max Error (Standard 2nd Order): {np.max(err_std):.2e}")
    print(f"Max Error (Compact 4th Order):  {np.max(err_cpt):.2e}")
    print(f"-> 콤팩트 방식이 약 {np.max(err_std)/np.max(err_cpt):.1f}배 더 정확함")

    plt.figure(figsize=(10, 8))

    # 결과 그래프
    plt.subplot(2, 1, 1)
    plt.plot(x, exact, 'k-', alpha=0.3, lw=5, label='Exact solution')
    plt.plot(x, std_sol, 'b-o', label='Standard (2nd Order)')
    plt.plot(x, cpt_sol, 'r-^', label='Compact (4th Order)')
    plt.title(f'Solution comparison (N={N_grid})')
    plt.legend()
    plt.grid(True)

    # 오차 그래프 (Log Scale)
    plt.subplot(2, 1, 2)
    plt.semilogy(x, err_std + 1e-16, 'b-o', label='Error: Standard')
    plt.semilogy(x, err_cpt + 1e-16, 'r-^', label='Error: Compact')
    plt.title('Absolute error (Log scale)')
    plt.ylabel('|Error|')
    plt.xlabel('x')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import pandas as pd

def weights(z, x, nd, m):
    """
    Fornberg의 방법을 사용하여 유한 차분 가중치를 계산한다.
    z: 미분할 지점
    x: 스텐실(Stencil) 포인트 배열
    nd: 스텐실 포인트의 수 - 1 (nd = len(x) - 1)
    m: 계산할 최대 미분 차수 (0차, 1차, ..., m차)
    반환: 크기 (nd+1, m+1)의 가중치 행렬 c
    """
    c1 = 1
    c4 = x[0] - z
    # c[i, k]는 x[i]를 포함하는 스텐실을 사용하여 k차 미분을 계산하기 위한 가중치
    c = np.zeros((nd+1, m+1)) 
    c[0, 0] = 1 # 0차 미분 (함수 값)의 가중치 초기화
    
    for i in range(1, nd+1):
        mn = min(i, m)
        c2 = 1
        c5 = c4
        c4 = x[i] - z
        
        for j in range(0, i):
            c3 = x[i] - x[j]
            c2 = c2*c3
            
            if j == i-1:
                # 새로운 포인트 x[i]를 추가하여 계산
                for k in range(mn, 0, -1):
                    # 새로운 점 x[i]에 대한 k차 미분 가중치 c[i, k] 계산
                    c[i, k] = c1*(k*c[i-1, k-1] - c5*c[i-1, k])/c2
                c[i, 0] = -c1*c5*c[i-1, 0]/c2
            
            # 기존 포인트 x[j]에 대한 가중치 c[j, k] 업데이트
            for k in range(mn, 0, -1):
                c[j, k] = (c4*c[j, k] - k*c[j, k-1])/c3
            c[j, 0] = c4*c[j, 0]/c3
            
        c1 = c2
    return c

def get_yprimes(nleft, m, x, y):
    """
    주어진 함수 y에 대해 x 배열의 모든 점에서 1차 및 2차 미분값을 계산한다.
    nleft: 현재 점의 왼쪽에서 사용할 점의 수
    m: 계산할 최대 미분 차수 (코드에서는 항상 m=2를 사용함)
    x: 독립 변수 배열
    y: 함수 값 배열 (y = f(x))
    반환: 1차 미분 배열 y1, 2차 미분 배열 y2
    """
    npt = len(x)
    y1 = np.zeros(npt)
    y2 = np.zeros(npt)
    
    # nright는 오른쪽에서 사용할 점의 수. 여기서는 nleft와 같게 설정
    nright = nleft 
    nd = nleft+nright # 스텐실의 절반 크기, nd+1이 스텐실 포인트 수
    xsten = np.zeros(nd+1) # 스텐실 포인트를 저장할 배열
    
    for j in range(npt): # 모든 점에 대해 미분 계산
        z = x[j] # 미분할 지점
        tmp = 0. # 1차 미분값
        tmq = 0. # 2차 미분값
        
        # 스텐실 포인트 (j0부터 j1-1까지) 결정
        if j-nleft < 0: # 시작 부분
            j0 = 0
            j1 = nd+1
            xsten[0:nd+1] = x[j0:j1]
        elif j-nleft+nd+1 > npt: # 끝 부분 (원래 코드의 인덱스 오류 수정: npt-1 -> npt)
            j1 = npt
            j0 = j1-nd-1
            xsten[0:nd+1] = x[j0:j1]
        else: # 내부
            j0 = j-nleft
            j1 = j0+nd+1
            xsten[0:nd+1] = x[j0:j1]
            
        c = weights(z, xsten, nd, m) # 가중치 계산
        
        # 1차, 2차 미분값 계산
        for k in range(nd+1):
            tmp = tmp+c[k, 1]*y[j0+k] # c[k, 1]은 1차 미분 가중치
            tmq = tmq+c[k, 2]*y[j0+k] # c[k, 2]는 2차 미분 가중치
            
        y1[j] = tmp
        y2[j] = tmq
        
    return y1, y2

# --- 제공된 함수 끝 ---

def analyze_precision(f, f_prime_exact, f_double_prime_exact, target_x=1.0, 
                      n_points=1001, range_start=0.0, range_end=2.0, max_order=2):
    """
    Fornberg 미분법의 정밀도를 분석하고 결과를 출력한다.
    """
    print(f"--- Fornberg 미분 정밀도 분석: f(x) = exp(-x) ---")
    
    # 1. x 값 생성
    x = np.linspace(range_start, range_end, n_points)
    y = f(x)
    
    # 2. x=1.0에서의 참값 계산
    true_y1_at_x1 = f_prime_exact(target_x)
    true_y2_at_x1 = f_double_prime_exact(target_x)
    
    print(f"대상 x = {target_x}")
    print(f"함수값 f({target_x}) = {f(target_x):.10f}")
    print(f"1차 미분 참값 f'({target_x}) = {true_y1_at_x1:.10f}")
    print(f"2차 미분 참값 f''({target_x}) = {true_y2_at_x1:.10f}")
    print("-" * 50)

    # 3. 다양한 nleft 값(왼쪽/오른쪽 스텐실 크기)에 대한 정밀도 비교
    results = []
    
    # nleft는 스텐실 크기를 제어한다. (스텐실 크기 = 2*nleft + 1)
    # 최소 nleft는 1 (3점 스텐실)이다. 최대는 (n_points - 1) // 2 이다.
    max_nleft = (n_points - 1) // 2
    
    # nleft 값의 범위를 설정 (예: 1, 2, 3, 5, 7, 10, 20)
    nleft_values = [1, 2, 3, 5, 7, 10, 20]
    
    # 데이터 포인트가 충분하지 않은 경우, nleft 값을 조정
    if max_nleft < 20:
        nleft_values = [n for n in nleft_values if n <= max_nleft]
    
    for nleft in nleft_values:
        # 스텐실 포인트 수: 2 * nleft + 1
        stencil_size = 2 * nleft + 1 
        
        # 4. 미분값 계산
        y1_num, y2_num = get_yprimes(nleft, max_order, x, y)
        
        # 5. x=1.0에 가장 가까운 인덱스를 찾아 미분값 추출
        # x 배열이 정렬되어 있으므로, np.argmin을 사용하여 찾을 수 있다.
        target_index = np.argmin(np.abs(x - target_x))
        
        num_y1_at_x1 = y1_num[target_index]
        num_y2_at_x1 = y2_num[target_index]
        
        # 6. 절대 오차 계산
        error_y1 = np.abs(num_y1_at_x1 - true_y1_at_x1)
        error_y2 = np.abs(num_y2_at_x1 - true_y2_at_x1)
        
        # 7. 결과 저장
        results.append({
            'n_left': nleft,
            'Stencil Size (2n+1)': stencil_size,
            "1st Derivative Num": f"{num_y1_at_x1:.10f}",
            "1st Derivative Error": f"{error_y1:.2e}",
            "2nd Derivative Num": f"{num_y2_at_x1:.10f}",
            "2nd Derivative Error": f"{error_y2:.2e}"
        })

    # 8. 결과 출력
    df_results = pd.DataFrame(results)
    print(f"계산된 미분값과 정밀도 (절대 오차) - x={target_x} 지점:")
    print(df_results.to_markdown(index=False, numalign="left", stralign="left"))
    print("\n 해석: n_left (스텐실 크기)가 커질수록 일반적으로 오차(Error)가 감소하여 정밀도가 향상된다.")

# --- 실행 부분 ---

# 함수 정의: f(x) = exp(-x)
f = lambda x: np.exp(-x)

# 해석적 미분 (참값) 정의:
# f'(x) = -exp(-x)
f_prime_exact = lambda x: -np.exp(-x)
# f''(x) = exp(-x)
f_double_prime_exact = lambda x: np.exp(-x)

# 정밀도 분석 실행
analyze_precision(f, f_prime_exact, f_double_prime_exact, target_x=1.0)

In [ ]:
import numpy as np
import scipy.sparse as sp

# 예시: 간단한 1차원 중심 차분 행렬 ([-0.5, 0, 0.5])
def get_1d_diff_matrix(N):
    # 실제로는 Fornberg 알고리즘 등을 사용해 고차 행렬을 생성
    diags = [np.ones(N)*0.5, -np.ones(N)*0.5]
    return sp.diags(diags, [1, -1], shape=(N, N))

# ------------------------------------------------
# 방법 1: 텐서곱 사용 (수학적으로 아름다우나, N이 작을 때만 가능)
# ------------------------------------------------
def method_tensor_product(N):
    Dx = get_1d_diff_matrix(N)
    Iy = sp.eye(N)
    Iz = sp.eye(N)
    # x방향 3D 미분 연산자 생성 (Dx ⊗ Iy ⊗ Iz)
    # 순서는 데이터 저장 방식(C-order vs F-order)에 따라 달라질 수 있음
    D_3d_x = sp.kron(sp.kron(Dx, Iy), Iz)
    print(f"3D 행렬 크기: {D_3d_x.shape}")
    return D_3d_x

# ------------------------------------------------
# 방법 2: 차원별 적용 (실전용, 메모리 절약)
# ------------------------------------------------
def method_apply_along_axis(data_3d, stencil_weights):
    # data_3d shape: (Nx, Ny, Nz)
    # x축 미분 (axis 0)
    # 실제로는 np.convolve나 상관함수 등을 사용하여 고속 처리
    grad_x = np.apply_along_axis(
        lambda m: np.convolve(m, stencil_weights, mode='same'), 
        axis=0, 
        arr=data_3d
    )
    # y축, z축도 동일하게 axis만 바꿔서 적용
    return grad_x

if __name__ == "__main__":
    N = 10  # N이 100만 되어도 방법 1은 메모리 터짐
    # 방법 1 확인
    try:
        method_tensor_product(N)
        print("텐서곱 행렬 생성 성공")
    except MemoryError:
        print("메모리 부족!")

In [ ]:
import numpy as np
from scipy.sparse.linalg import LinearOperator, cg

# 1. 행렬-벡터 곱을 정의하는 함수 (오퍼레이터 A의 역할)
def matvec_laplacian(v):
    """
    3차원 이산 라플라시안 오퍼레이터의 행렬-벡터 곱 A*v를 계산하는 함수.
    (예시를 위해 간단한 3x3 대각선 행렬처럼 동작하도록 정의)
    """
    # 실제 라플라시안 행렬은 매우 크고 복잡하지만, 여기서는 개념을 위해 간단히 정의
    # A = [[4, -1, 0],
    #      [-1, 4, -1],
    #      [0, -1, 4]] 처럼 동작하도록 정의
    n = len(v)
    Av = np.zeros_like(v)
    # 대각 성분 (4 * v[i])
    Av = 4 * v
    # 비대각 성분 (-1 * v[i-1] 및 -1 * v[i+1])
    if n > 1:
        Av[0] -= v[1]
        Av[n-1] -= v[n-2]
    if n > 2:
        Av[1:n-1] -= v[0:n-2] # v[i-1]
        Av[1:n-1] -= v[2:n]   # v[i+1]
    return Av

# 2. LinearOperator 생성
N = 100 # 벡터의 크기 (행렬 A의 크기는 N x N)
A_op = LinearOperator((N, N), matvec=matvec_laplacian, dtype=float)
# 3. 상수 벡터 b 생성
# 임의의 벡터 b를 생성
b = np.random.rand(N)
# 4. 공액 기울기법(Conjugate Gradient, CG)으로 해 x 계산
# cg 함수는 A 대신 LinearOperator를 인수로 받을 수 있다.
# tol: 수렴 허용 오차
x_cg, info = cg(A_op, b, rtol=1e-8)
# 5. 결과 확인
if info == 0:
    print(f" CG Method: Converged successfully in {info} iterations.")
    # 잔차 노름 확인: ||Ax - b||
    residual_norm = np.linalg.norm(A_op.dot(x_cg) - b)
    print(f"최종 잔차 노름: {residual_norm:.2e}")
else:
    print(f" CG Method: Did not converge. info = {info}")
# LinearOperator의 기본 속성 확인
print(f"\nLinearOperator의 차원: {A_op.shape}")

In [ ]:
import numpy as np, math
def fd_weights(x, x0, m):
    x  = np.asarray(x, dtype=float).reshape(-1)
    x0 = float(x0)
    n  = x.size
    if m >= n:
        raise ValueError("스텐실 점수 n > 미분차수 m 필요")
    if np.unique(x).size != n:
        raise ValueError("스텐실 좌표 중복 존재")

    A = np.vander(x - x0, N=n, increasing=True)  # [ (x-x0)^k ]_{k=0..n-1}
    b = np.zeros(n); b[m] = math.factorial(m)
    w = np.linalg.solve(A, b)
    return w

h = 0.1
w = fd_weights([-h, 0.0, h], 0.0, 1)   # 리스트로 줘도 동작
# 또는
w = fd_weights(np.array([-h, 0.0, h]), 0.0, 1)


x0 = 0.0; h = 0.1
x = [-h, 0.0, h]
w = fd_weights(x, x0, 1)      # [-1/(2h), 0, +1/(2h)]에 해당
approx = sum(w_i*np.sin(xi) for w_i, xi in zip(w, x))


x0 = 0.0; h = 0.1
x = [0.0, h, 2*h]
w = fd_weights(x, x0, 1)      # [-3/(2h), 2/h, -1/(2h)]에 해당
approx = sum(w_i*np.exp(xi) for w_i, xi in zip(w, x))   # exp'(0)=1과 비교


x0 = 0.0
x = [-0.7, 0.0, 0.3]          # 간격 불균등
w = fd_weights(x, x0, 1)
approx = sum(w_i*np.exp(xi) for w_i, xi in zip(w, x))   # exp'(0)=1


x0 = 0.0; h = 0.1
x = [-2*h, -h, 0.0, h, 2*h]
w = fd_weights(x, x0, 2)      # (-1,16,-30,16,-1)/(12h^2)에 해당
approx = sum(w_i*np.exp(xi) for w_i, xi in zip(w, x))   # exp''(0)=1


def D1_matrix_on_grid(x):
    n = len(x); D = np.zeros((n, n))
    for i in range(n):
        idx = [0,1,2] if i==0 else ([n-3,n-2,n-1] if i==n-1 else [i-1,i,i+1])
        w = fd_weights([x[j] for j in idx], x[i], 1)
        for k, j in enumerate(idx): D[i, j] = w[k]
    return D

x = np.linspace(0.0, 1.0, 6)
D1 = D1_matrix_on_grid(x)
u = np.sin(x)
Du_num = D1 @ u               # 수치미분
Du_true = np.cos(x)
max_err = np.max(np.abs(Du_num - Du_true))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(x):
    return np.sin(x)

def df_exact(x):
    return np.cos(x)

# 전진 차분법 (Forward Difference)
def df_approx(x, h):
    return (f(x + h) - f(x)) / h

x_target = 1.0
exact_val = df_exact(x_target)
# h 값을 10^-1 부터 10^-18 까지 줄여나감 (log scale)
h_values = np.logspace(-1, -18, 100)
errors = []
for h in h_values:
    approx_val = df_approx(x_target, h)
    # 절댓값 오차 계산
    error = abs(approx_val - exact_val)
    errors.append(error)
# --- 시각화 ---
plt.figure(figsize=(10, 6))
plt.loglog(h_values, errors, 'b-', label='Total error', linewidth=2)
# 보조선: 절단 오차 (기울기 1)
plt.loglog(h_values, h_values * 0.5, 'g--', label='Truncation error O(h)', alpha=0.5)
# 보조선: 반올림 오차 (기울기 -1)
# 기계 입실론(approx 1e-16)을 고려하여 스케일 조정
epsilon = 1e-16
plt.loglog(h_values, epsilon / h_values, 'r--', label='Round-off error O(1/h)', alpha=0.5)
plt.xlabel('Step size (h)')
plt.ylabel('Absolute error')
plt.title('Truncation error vs. Round-off error')
plt.gca().invert_xaxis() # x축을 큰 h -> 작은 h 순서로 (오른쪽이 Truncation 영역이 되도록)
plt.legend()
plt.grid(True, which="both", ls="-")
plt.savefig('tr_ro.png')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------
# 1. 수정된 함수 정의 및 실제값
# -----------------

def f(x):
    """수정된 함수 f(x) = exp(-x/2)"""
    return np.exp(-x / 2)

# x=0에서의 실제 미분값 (True Value)
# f'(x) = -1/2 * exp(-x/2) 이므로, f'(0) = -0.5 이다.
TRUE_DERIVATIVE_AT_0 = -0.5

# -----------------
# 2. O(h^2) 중앙차분 공식 (수정 필요 없음)
# -----------------

def centered_difference_oh2(f, x, h):
    """
    O(h^2) 정확도를 갖는 중앙차분 공식 (First Derivative)
    D_h^(2)f(x) = [f(x+h) - f(x-h)] / (2h)
    """
    return (f(x + h) - f(x - h)) / (2 * h)

# -----------------
# 3. 계산 및 오차 분석
# -----------------

# 그리드 간격 h 값들 생성 (logarithmic scale)
# 10^-1 에서 10^-15 까지 50개의 h 값 사용
h_values = np.logspace(-1, -15, 50) 
x_point = 0.0 # 미분할 지점

# 각 h에 대한 수치 미분값과 오차 계산
numerical_derivatives = centered_difference_oh2(f, x_point, h_values)
errors = np.abs(numerical_derivatives - TRUE_DERIVATIVE_AT_0)

# -----------------
# 4. 결과 시각화
# -----------------

plt.figure(figsize=(10, 6))

# 오차 vs h 플롯
plt.loglog(h_values, errors, 'o-', label=r'$O(h^2)$ Centered difference error')

# O(h^2) 기울기 가이드라인 추가
# 이론적인 수렴 기울기를 나타내기 위해 첫 번째 오차 지점으로부터 h^2 기울기를 그린다.
plt.loglog(h_values, h_values**2 * (errors[0] / h_values[0]**2), 'r--', label=r'Theoretical $O(h^2)$ slope')

# 그래프 설정
plt.title(r'Finite Difference error for $f^\prime(0)$ of $f(x) = e^{-x/2}$', fontsize=16)
plt.xlabel(r'Grid spacing, $h$', fontsize=14)
plt.ylabel(r'Absolute error $|D_h f(0) - f^\prime(0)|$', fontsize=14)
plt.grid(True, which="both", ls="--")
plt.legend(fontsize=12)
plt.xlim(h_values[-1], h_values[0]) # x축 범위를 h 값의 순서대로 설정
plt.gca().invert_xaxis() # x축을 작은 h에서 큰 h로 보이게 뒤집기

plt.show()

print(f"함수 f(x) = exp(-x/2)")
print(f"x=0 에서의 실제 미분값 f'(0) = {TRUE_DERIVATIVE_AT_0}")
print("\n[h 값에 따른 오차의 변화]")
# 결과를 간략하게 보기 위해 일부 h 값만 출력
for h, error in zip(h_values[::5], errors[::5]):
    print(f"h = {h:.2e},  Error = {error:.4e}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def get_comp12(nleft, npt):
    x = np.zeros(npt)
    y = np.zeros(npt)
    for i in range(npt):
        x[i] = -1.0+(1.0-(-1.0))*float(i)/float(npt-1)
    h = x[1]-x[0]
    print(h)
    for i in range(npt):
        y[i] = np.exp(-x[i]/2.0)
    i0 = int((npt-1)/2)
    m = 2
    y1, y2 = get_yprimes(nleft, m, x, y)
    y1true =-0.5
    y2true = 0.25
    return h, np.abs(y1[i0]-y1true), np.abs(y2[i0]-y2true)
alist = []
blist = []
clist = []
dlist = []
elist = []

nleft = 4
for npt in [11, 21, 31, 41, 61, 81, 101, 121, 141, 161, 181, 191, 201, 211, 221, 231]:
    h, tmp, tmq = get_comp12(nleft, npt)
    alist.append(h)
    blist.append(tmp)
    clist.append(tmq)

alist=np.array(alist)
blist=np.array(blist)
clist=np.array(clist)

nleft = 2
for npt in [11, 21, 31, 41, 61, 81, 101, 121, 141, 161, 181, 191, 201, 211, 221, 231]:
    h, tmp, tmq = get_comp12(nleft, npt)
    dlist.append(tmp)
    elist.append(tmq)
dlist=np.array(dlist)
elist=np.array(elist)

plt.figure(figsize=(10, 6))
plt.loglog(alist, blist, 'o-')
plt.loglog(alist, clist, '+-')
#plt.loglog(alist, dlist, 'b-', ms=12)
#plt.loglog(alist, elist, '*-', ms=12)
# 그래프 설정
plt.title(r'Finite difference errors: $f^{(1)}(0)$, $f^{(2)}(0)$ of $f(x) = e^{-x/2}$', fontsize=16)
plt.xlabel(r'Grid spacing $h$', fontsize=14)
plt.ylabel(r'Absolute error $|D_h f(0) - f^{(1,2)}|$', fontsize=14)
plt.grid(True, which="both", ls="--")
#plt.legend(fontsize=12)
plt.xlim(alist[-1], alist[0]) # x축 범위를 h 값의 순서대로 설정
plt.gca().invert_xaxis() # x축을 작은 h에서 큰 h로 보이게 뒤집기

plt.savefig('error_h4.png')
plt.show()
plt.close()

In [ ]:
for i in range(12):
    print(np.log((blist[i+1]/blist[i]))/np.log((alist[i+1]/alist[i])))
for i in range(12):
    print(np.log((clist[i+1]/blist[i]))/np.log((alist[i+1]/alist[i])))    

plt.figure(figsize=(10, 6))
#plt.loglog(alist, blist, 'o-')
#plt.loglog(alist, clist, '+-')
plt.loglog(alist, dlist, 'x-', ms=12)
plt.loglog(alist, elist, '*-', ms=12)
# 그래프 설정
plt.title(r'Finite difference errors: $f^{(1)}(0)$, $f^{(2)}(0)$ of $f(x) = e^{-x/2}$', fontsize=16)
plt.xlabel(r'Grid spacing $h$', fontsize=14)
plt.ylabel(r'Absolute error $|D_h f(0) - f^{(1,2)}|$', fontsize=14)
plt.grid(True, which="both", ls="--")
#plt.legend(fontsize=12)
plt.xlim(alist[-1], alist[0]) # x축 범위를 h 값의 순서대로 설정
plt.gca().invert_xaxis() # x축을 작은 h에서 큰 h로 보이게 뒤집기

plt.savefig("error_h2.png")
plt.show()
plt.close()
for i in range(12):
    print(np.log((dlist[i+1]/dlist[i]))/np.log((alist[i+1]/alist[i])))
for i in range(12):
    print(np.log((elist[i+1]/elist[i]))/np.log((alist[i+1]/alist[i])))    

In [ ]:
import numpy as np
print("=== 1. 정수 오버플로우 (NumPy int8 사용) ===")
# int8은 -128 ~ 127까지 표현 가능
max_int8 = np.array([127], dtype='int8')
print(f"현재 값: {max_int8}")
# 1을 더하면 128이 되는 게 아니라, 비트 범위를 넘어 최솟값(-128)으로 돌아감 (Wrapping)
overflowed = max_int8 + 1
print(f"1 더한 결과 (Overflow): {overflowed}")
# 비교: 파이썬 기본 int는 오버플로우 없음
py_int = 127
print(f"파이썬 기본 int (127 + 1): {py_int + 1} (오버플로우 안 됨)")
import sys
print("\n=== 2. 부동소수점 오버플로우 (Float Overflow) ===")
# 표현 가능한 가장 큰 float 값 확인
max_float = sys.float_info.max
print(f"표현 가능한 최대 float: {max_float}")
# 최대값에 1.0001 같은 작은 수를 곱해도 오버플로우는 안 나지만,
# 2를 곱해서 한계를 넘겨버리면 'inf' (무한대)가 됨
overflow_float = max_float * 2
print(f"한계를 넘은 결과: {overflow_float}")
# 무한대 여부 확인
print(f"무한대인가?: {overflow_float == float('inf')}")
print("\n=== 3. 부동소수점 언더플로우 (Float Underflow) ===")
# 아주 작은 수에서 시작
small_num = 1e-300
print(f"시작 값: {small_num}")
# 계속해서 1000으로 나누어 봄
for i in range(10):
    small_num = small_num / 100000
    print(f"{i+1}회 나눔: {small_num}")
    if small_num == 0.0:
        print(">>> 언더플로우 발생! (0.0으로 소멸됨)")
        break

In [ ]:
import math

def compare_precision_100_factorial():
    print(f"--- 100! (100 Factorial) Precision Test ---\n")

    # 1. 정수(int)를 사용한 완벽한 계산 (다중 정밀도)
    exact_value = math.factorial(100)
    
    # 2. 실수(float)로 변환 (정보 손실 발생!)
    float_value = float(exact_value)
    
    # 3. 비교를 위해 실수를 다시 정수로 복원
    # (주의: float가 기억하지 못하는 아랫자리들은 0으로 채워지거나 뭉개짐)
    reconstructed_value = int(float_value)
    
    # 4. 손실된 정보(오차) 계산
    loss = exact_value - reconstructed_value

    # --- 결과 출력 ---
    print(f"1. [int]   정확한 값 (길이: {len(str(exact_value))}자리):")
    print(f"{exact_value}")
    print("-" * 50)
    
    print(f"2. [float] 실수 변환 값 (컴퓨터 메모리 저장 형태):")
    print(f"{float_value}")
    print("-" * 50)
    
    print(f"3. [복원]  float를 다시 int로 바꾼 값:")
    print(f"{reconstructed_value}")
    print("-" * 50)

    print(f"4. [비교]  사라진 정보 (오차 = 원래 값 - 복원 값):")
    print(f"{loss}")
    
    # 유효숫자 비교
    match_count = 0
    s_exact = str(exact_value)
    s_recon = str(reconstructed_value)
    for a, b in zip(s_exact, s_recon):
        if a == b:
            match_count += 1
        else:
            break
            
    print(f"\n>>> 결론: 앞부분 {match_count}자리만 똑같고, 나머지 {len(s_exact) - match_count}자리는 모두 증발했다.")

# 실행
compare_precision_100_factorial()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_frey_curve(A, B, n):
    """
    주어진 가상의 A, B, n 값에 대한 프라이 곡선 형태를 그린다.
    방정식: y^2 = x(x - A^n)(x + B^n)
    
    주의: 실제로는 A^n + B^n = C^n (n>2)을 만족하는 정수가 없으므로,
    이 곡선은 이론적으로만 존재하는 가상의 곡선이다.
    """
    
    # n이 2보다 커야 페르마의 마지막 정리 조건에 부합한다.
    if n <= 2:
        print("경고: n은 2보다 커야 프라이 곡선의 정의에 맞다. 하지만 시각화를 위해 계속 진행한다.")

    # A^n 과 B^n 계산
    An = A**n
    Bn = B**n

    print(f"프라이 곡선 그리기 설정: A={A}, B={B}, n={n}")
    print(f"방정식: y^2 = x(x - {An})(x + {Bn})")

    # 1. 그리드(Grid) 범위 설정
    # 곡선은 x축과 x=0, x=An, x=-Bn 에서 만난다.
    # 이 뿌리(roots)들을 포함하도록 x 범위를 넉넉하게 잡는다.
    x_min = -Bn - (Bn * 0.5) # 왼쪽 뿌리보다 조금 더 왼쪽으로
    x_max = An + (An * 0.5)  # 오른쪽 뿌리보다 조금 더 오른쪽으로
    
    # y값은 x의 3차식의 제곱근이므로 빠르게 커진다. 적절한 범위를 설정한다.
    # x 범위에 비례하여 적당히 설정해본다.
    y_limit = max(abs(x_min), abs(x_max)) * 1.5 
    
    # 2. 2D 메쉬그리드 생성
    # 포인트 개수(N)를 늘리면 곡선이 더 부드러워지지만 계산이 느려진다.
    N = 1000 
    x = np.linspace(x_min, x_max, N)
    y = np.linspace(-y_limit, y_limit, N)
    X, Y = np.meshgrid(x, y)

    # 3. 프라이 곡선 방정식 계산
    # 음함수 형태 f(x, y) = y^2 - x(x - A^n)(x + B^n) = 0 으로 만든다.
    Z = Y**2 - X * (X - An) * (X + Bn)

    # 4. 그래프 그리기 (Contour Plot 활용)
    plt.figure(figsize=(10, 8))
    
    # Z 값이 0인 지점만 등고선으로 그린다. (levels=[0])
    contour = plt.contour(X, Y, Z, levels=[0], colors='blue', linewidths=2)
    
    # 그래프 꾸미기
    plt.title(f"Hypothetical Frey Curve\n$y^2 = x(x - {A}^{n})(x + {B}^{n})$  (A={A}, B={B}, n={n})", fontsize=14)
    plt.xlabel("x axis", fontsize=12)
    plt.ylabel("y axis", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    
    # x, y 축 선 그리기
    plt.axhline(0, color='black', linewidth=1)
    plt.axvline(0, color='black', linewidth=1)
    
    # x축과 만나는 뿌리(roots) 표시
    roots = [0, An, -Bn]
    plt.scatter(roots, [0, 0, 0], color='red', zorder=5, label='Roots (x-intercepts)')
    for root in roots:
        plt.annotate(f'{root}', (root, 0), textcoords="offset points", xytext=(0,10), ha='center')
        
    plt.legend()
    plt.tight_layout()
    
    print("그래프를 출력한다...")
    plt.show()

# =============================================
# 실행 예시
# =============================================

# 임의의 정수 A, B, n을 설정한다.
# (실제 페르마의 정리를 만족하는 숫자가 아니어도 시각화를 위해 입력한다)

# 예시 1: 비교적 작은 숫자로 형태 확인
# y^2 = x(x - 1^3)(x + 2^3) => y^2 = x(x-1)(x+8)
# x절편은 -8, 0, 1 이 된다.
plot_frey_curve(A=1, B=2, n=3)

# 예시 2: n이 커지면 값이 급격히 커져 그래프가 매우 가파르게 변한다.
# 주석을 풀고 실행해보세요. (범위가 넓어져서 형태가 찌그러져 보일 수 있다.)
# plot_frey_curve(A=2, B=3, n=3) 
# x절편은 -27, 0, 8 이 된다.

In [ ]:
import numpy as np
import scipy.sparse as sparse
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

def solve_1d_fdm():
    # 1. 격자 설정
    N = 100          # 내부 격자점 개수
    L = 1.0          # 구간 길이
    h = L / (N + 1)  # 격자 간격 (dx)
    x = np.linspace(0, L, N + 2) # 경계 포함 전체 좌표
    # 2. 희소 행렬(Sparse Matrix) A 생성
    # 식: y(i-1) - 2y(i) + y(i+1) = h^2 * f(i)
    # A는 삼중 대각 행렬(Tridiagonal)이 된다.
    diagonals = [np.ones(N-1), -2*np.ones(N), np.ones(N-1)]
    offsets = [-1, 0, 1]
    # 3-point stencil Matrix
    A = sparse.diags(diagonals, offsets, shape=(N, N), format='csr')
    # 3. 우변 벡터 b 생성
    # 미분방정식의 우변 f(x)
    x_internal = x[1:-1] # 경계 제외한 내부 점
    f = -4 * (np.pi**2) * np.sin(2 * np.pi * x_internal)
    b = f * (h**2) # h^2을 우변으로 넘겨줌
    # 4. 선형 시스템 풀기 (Ax = b)
    # spla.spsolve는 희소 행렬에 최적화된 직접 풀이법(Direct Solver)
    y_internal = spla.spsolve(A, b)
    # 5. 경계 조건 결합 (0, ..., 0)
    y_sol = np.concatenate(([0], y_internal, [0]))
    # 6. 결과 비교
    y_exact = np.sin(2 * np.pi * x) # 이론적 정답
    plt.figure(figsize=(8, 5))
    plt.plot(x, y_exact, 'k-', label='Exact solution')
    plt.plot(x, y_sol, 'r--', label='FDM solution (N={})'.format(N))
    plt.title("1D Finite difference method (O(h^2))")
    plt.legend()
    plt.grid(True)
    plt.show()
solve_1d_fdm()

In [ ]:
import numpy as np

arr = np.array([1, 2, 3])
# 왼쪽에 1개, 오른쪽에 2개 0으로 채우기
padded = np.pad(arr, pad_width=(1, 2), mode='constant', constant_values=0)

print("원본:", arr)
print("결과:", padded)
# 결과: [0, 1, 2, 3, 0, 0]

In [ ]:
import numpy as np

# 2x2 격자 (내부 솔루션이라고 가정)
u_inner = np.array([[1, 2],
                    [3, 4]])

# 모든 방향으로 1칸씩 0으로 둘러싸기 (pad_width=1)
# 이것이 FDM 예제에서 u_full을 만들 때 사용한 방법이다.
u_full = np.pad(u_inner, pad_width=1, mode='constant', constant_values=0)

print("--- 원본 (2x2) ---")
print(u_inner)
print("\n--- 패딩 결과 (4x4) ---")
print(u_full)

In [ ]:
arr = np.array([10, 20, 30])

# 1. Edge (Neumann BC - 온도 유지)
print("Edge:    ", np.pad(arr, (1, 1), mode='edge'))
# 결과: [10, 10, 20, 30, 30] (10과 30이 연장됨)

# 2. Wrap (Periodic BC - 지구 한 바퀴)
print("Wrap:    ", np.pad(arr, (1, 1), mode='wrap'))
# 결과: [30, 10, 20, 30, 10] (오른쪽 끝 30이 왼쪽으로, 왼쪽 10이 오른쪽으로)

In [ ]:
import numpy as np

# 2x2x2 큐브 (모두 1로 채워짐)
# 마치 작은 주사위 8개가 모인 형태
cube = np.ones((2, 2, 2))

print(f"원본 크기: {cube.shape}")

# 모든 면에 1칸씩 0을 덧붙임 (Dirichlet BC)
padded_cube = np.pad(cube, pad_width=1, mode='constant', constant_values=0)

print(f"패딩 후 크기: {padded_cube.shape}")
# 결과: (4, 4, 4) -> 각 축마다 양쪽으로 1씩 늘어났으므로 2 + 1 + 1 = 4

print("\n--- 결과 확인 (단면) ---")
# 정가운데를 잘라서 보면, 가운데 1들이 있고 테두리가 0으로 감싸져 있음
print(padded_cube[1]) # 깊이 축의 1번 인덱스 단면

In [ ]:
import numpy as np

# 2(깊이) x 3(높이) x 3(너비) 배열
data = np.full((2, 3, 3), 7) # 숫자 7로 채움

# Axis 0(깊이)만 앞뒤로 1칸씩 추가하고, 나머지는 추가 안 함(0)
# ((앞, 뒤), (위, 아래), (좌, 우))
padding_shape = ((1, 1), (0, 0), (0, 0))

result = np.pad(data, padding_shape, mode='constant', constant_values=0)

print(f"원본 shape: {data.shape}")     # (2, 3, 3)
print(f"결과 shape: {result.shape}")   # (4, 3, 3) -> 깊이만 2 늘어남

print("\n--- 첫 번째 층 (새로 생긴 0 패딩) ---")
print(result[0])

print("\n--- 두 번째 층 (원본 데이터 시작) ---")
print(result[1])

In [ ]:
# FEniCS 설치 필요 (conda install -c conda-forge fenics)
from fenics import *
import matplotlib.pyplot as plt

def solve_fenics_poisson():
    # ---------------------------------------------------------
    # 1. 메쉬 생성 (Mesh Generation)
    # ---------------------------------------------------------
    # UnitSquareMesh(nx, ny): 0~1 사이의 단위 사각형을 삼각형으로 쪼갠다.
    # 앞선 코드에서 elements 리스트를 직접 만들던 고생을 한 줄로 끝낸다.
    mesh = UnitSquareMesh(32, 32)

    # ---------------------------------------------------------
    # 2. 함수 공간 정의 (Function Space)
    # ---------------------------------------------------------
    # 'P', 1 : Lagrange Linear Elements (선형 삼각형 요소)
    V = FunctionSpace(mesh, 'P', 1)

    # ---------------------------------------------------------
    # 3. 경계 조건 정의 (Boundary Conditions)
    # ---------------------------------------------------------
    # 경계(boundary)를 판별하는 함수
    def boundary(x, on_boundary):
        return on_boundary # FEniCS가 자동으로 가장자리를 찾아준다.

    # u = 0 (Constant(0)) 조건을 경계에 적용
    bc = DirichletBC(V, Constant(0.0), boundary)

    # ---------------------------------------------------------
    # 4. 변분 문제 정의 (Variational Problem) - 핵심!
    # ---------------------------------------------------------
    u = TrialFunction(V)  # 우리가 구할 해 (Unknown)
    v = TestFunction(V)   # 가중치 함수 (Test function)
    f = Constant(1.0)     # 소스 항 (열원)

    # 수식: integral( dot(grad(u), grad(v)) ) dx
    # 앞선 코드에서 B.T @ B 하던 과정이 이 한 줄로 표현된다.
    a = dot(grad(u), grad(v)) * dx  # 좌변 (Bilinear form)
    L = f * v * dx                  # 우변 (Linear form)

    # ---------------------------------------------------------
    # 5. 풀이 (Compute)
    # ---------------------------------------------------------
    u_sol = Function(V) # 결과를 담을 객체
    solve(a == L, u_sol, bc)

    # ---------------------------------------------------------
    # 6. 시각화
    # ---------------------------------------------------------
    plt.figure(figsize=(8, 6))
    c = plot(u_sol, mode='color')
    plot(mesh, linewidth=0.3, alpha=0.5) # 메쉬도 같이 그리기
    plt.colorbar(c, label='Solution u')
    plt.title("FEniCS Solution: Poisson Equation")
    plt.show()

if __name__ == "__main__":
    solve_fenics_poisson()

In [ ]:
!pip install fenics

## Chapter 3 고속 푸리에(Fourier) 변환    파이썬을 활용한 수치해석(Numerical Analysis with Python) 이인호 (북스힐, 2026)

In [ ]:
def merge_sort(arr):
    # Base Case: 리스트 길이가 1 이하면 이미 정렬된 상태
    if len(arr) <= 1:
        return arr
    # 1. 분할 (Divide): 가운데를 기준으로 쪼갬
    mid = len(arr) // 2
    left_half = arr[:mid]
    right_half = arr[mid:]
    # 재귀 호출을 통해 끝까지 쪼갬
    left_sorted = merge_sort(left_half)
    right_sorted = merge_sort(right_half)
    # 2. 병합 (Merge): 두 개의 정렬된 리스트를 하나로 합침
    return merge(left_sorted, right_sorted)
def merge(left, right):
    sorted_list = []
    i = 0 # 왼쪽 리스트 인덱스
    j = 0 # 오른쪽 리스트 인덱스
    # 두 리스트를 앞에서부터 비교하며 작은 값을 결과에 추가
    while i < len(left) and j < len(right):
        if left[i] < right[j]:
            sorted_list.append(left[i])
            i += 1
        else:
            sorted_list.append(right[j])
            j += 1
    # 남은 데이터들을 뒤에 이어 붙임
    sorted_list.extend(left[i:])
    sorted_list.extend(right[j:])
    return sorted_list
# 실행 예시
data = [38, 27, 43, 3, 9, 82, 10]
print(f"정렬 전: {data}")
print(f"정렬 후: {merge_sort(data)}")    

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pywt
from scipy.signal import stft
def compare_stft_vs_wavelet():
	# 1. 복합 신호 생성 (Sample Rate: 2000Hz, 1초)
	fs = 2000
	t = np.linspace(0, 1, fs)
	# 신호 A: 50Hz ~ 400Hz로 올라가는 Chirp (Linear Chirp)
	sig_chirp = np.sin(2 * np.pi * 150 * t * t) 
	# 신호 B: 0.5초에 아주 짧게 발생하는 고주파(800Hz) 충격 (Transient)
	sig_transient = np.exp(-10000 * (t - 0.5)**2) * np.sin(2 * np.pi * 800 * t)
	# 신호 C: 전체에 깔린 아주 낮은 저주파 (10Hz)
	sig_low = np.sin(2 * np.pi * 10 * t)
	signal = sig_chirp + sig_transient + sig_low
	# ==========================================
	# 2. STFT 수행 (고정된 창 크기)
	# ==========================================
	# nperseg: 창의 크기. 
	# - 크면(256): 주파수는 잘 보이나 시간(가로)이 뭉개짐
	# - 작으면(32): 시간은 잘 보이나 주파수(세로)가 뭉개짐
	f_stft, t_stft, Zxx = stft(signal, fs, nperseg=64) 
	# ==========================================
	# 3. 웨이브렛 변환 수행 (가변 창 크기)
	# ==========================================
	scales = np.arange(1, 150)
	coef, freqs_wt = pywt.cwt(signal, scales, 'cmor1.5-1.0', sampling_period=1/fs)
	# ==========================================
	# 4. 시각화 비교
	# ==========================================
	plt.figure(figsize=(14, 10))
	# (1) 원본 신호
	plt.subplot(3, 1, 1)
	plt.plot(t, signal, 'k', linewidth=0.8)
	plt.title("Original signal(Low freq + chirp + High freq spike)")
	plt.xlim(0, 1)
	# (2) STFT 결과 (Spectrogram)
	plt.subplot(3, 1, 2)
	plt.pcolormesh(t_stft, f_stft, np.abs(Zxx), shading='gouraud', cmap='inferno')
	plt.title("STFT(Fixed window): Compromise between time & freq")
	plt.ylabel("Frequency [Hz]")
	plt.ylim(0, 1000)
	# 한계점 표시
	plt.text(0.5,900,"Blurry spike(Bad time res.)",color='cyan',ha='center',fontweight='bold')
	plt.text(0.1,50,"Blurry low freq(Bad freq res.)",color='cyan',ha='left',fontweight='bold')
	# (3) Wavelet 결과 (Scalogram)
	plt.subplot(3, 1, 3)
	plt.pcolormesh(t, freqs_wt, np.abs(coef), shading='gouraud', cmap='inferno')
	plt.title("Wavelet transform(Variable window): Multi-resolution analysis")
	plt.ylabel("Frequency [Hz]")
	plt.xlabel("Time [sec]")
	plt.ylim(0, 1000)
	# 장점 표시
	plt.text(0.5,900,"Sharp spike(Good time res.)",color='cyan',ha='center',fontweight='bold')
	plt.text(0.1,50,"Clear low freq(Good freq res.)",color='cyan',ha='left',fontweight='bold')
	plt.tight_layout()
	plt.savefig('stft_wavelet.png')
	plt.show()
compare_stft_vs_wavelet()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq
# 1. 신호 생성 및 노이즈 추가
# ----------------------------------
# 샘플링 매개변수
Fs = 1000  # 샘플링 주파수 (Hz)
T = 1.0    # 신호 길이 (초)
N = int(Fs * T) # 총 샘플 수
t = np.linspace(0.0, T, N, endpoint=False) # 시간 벡터
# 원본 신호 (50Hz와 120Hz 두 사인파의 합)
f1 = 50.0  # 주파수 1
f2 = 120.0 # 주파수 2
signal_clean = 0.7 * np.sin(2 * np.pi * f1 * t) + 1.0 * np.sin(2 * np.pi * f2 * t)
# 노이즈 추가 (랜덤 노이즈)
noise = 0.8 * np.random.randn(N)
signal_noisy = signal_clean + noise
# 2. FFT 수행
# ----------------------------------
# FFT 계산
yf = fft(signal_noisy)
# 주파수 축 생성
xf = fftfreq(N, 1/Fs)[:N//2] # 양의 주파수만 사용
# 3. 노이즈 필터링 (주파수 도메인에서)
# ----------------------------------
# 필터링을 위한 임계값 설정 (노이즈로 간주되는 낮은 진폭 제거)
# 이 임계값은 신호의 특성에 따라 조정해야 한다.
threshold = 100 
# 복소수 스펙트럼의 진폭을 기준으로 임계값 미만인 주파수 성분을 0으로 설정
yf_filtered = yf.copy()
# 양수 주파수 영역에서 필터링
yf_filtered[np.abs(yf_filtered) < threshold] = 0
# 음수 주파수 영역은 대칭적으로 처리해야 하지만, 
# 일반적으로 대칭적인 FFT 결과에서 양수 주파수를 필터링하면 음수도 자동으로 처리되거나,
# rfft/irfft를 사용하면 더 간편하다. 여기서는 전체 yf를 필터링하는 방식으로 단순화한다.
# 4. 역 FFT (iFFT) 수행
# ----------------------------------
# 필터링된 스펙트럼을 시간 도메인으로 변환 (iFFT)
# 결과는 복소수이므로, 실수 부분만 취한다.
signal_filtered = np.real(ifft(yf_filtered))
# 5. 결과 시각화
# ----------------------------------
plt.figure(figsize=(12, 8))
# 시간 도메인 플롯
plt.subplot(3, 1, 1)
plt.plot(t, signal_noisy, label='Noisy Signal', alpha=0.6)
plt.plot(t, signal_clean, label='Clean Signal', color='red', linestyle='--')
plt.title('Time Domain: Noisy vs Clean Signal')
plt.legend()
plt.xlim(0, 0.1) # 일부 구간만 확대
plt.subplot(3, 1, 2)
# 주파수 스펙트럼 플롯 (진폭의 절대값)
plt.plot(xf, 2.0/N * np.abs(yf[0:N//2]), label='Noisy Spectrum')
plt.plot(xf,2.0/N*np.abs(yf_filtered[0:N//2]),label='Filtered Spectrum',linestyle='--')
plt.title('Frequency Domain: Magnitude Spectrum')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude')
plt.legend()
# 필터링된 시간 도메인 플롯
plt.subplot(3, 1, 3)
plt.plot(t, signal_filtered, label='Filtered Signal', color='green')
plt.plot(t, signal_clean, label='Clean Signal', color='red', linestyle='--')
plt.title('Time Domain: Filtered vs Clean Signal')
plt.legend()
plt.xlim(0, 0.1) # 일부 구간만 확대
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
def naive_polynomial_multiplication(P, Q):
    """
    일반적인 O(N^2) 방식으로 두 다항식의 계수 배열을 곱한다 (합성곱).
    P와 Q는 다항식의 계수 리스트이다. (낮은 차수부터)
    """
    N = len(P)
    M = len(Q)
    # 결과 다항식의 차수는 N + M - 2 이므로, 계수 리스트 길이는 N + M - 1 이다.
    result_len = N + M - 1
    R = [0] * result_len
    for k in range(result_len):
        # r_k = sum(p_i * q_{k-i}) 를 계산한다.
        # 합의 시작 인덱스 i_min: max(0, k - M + 1)
        i_min = max(0, k - M + 1)
        # 합의 끝 인덱스 i_max: min(k, N - 1)
        i_max = min(k, N - 1)
        r_k = 0
        for i in range(i_min, i_max + 1):
            r_k += P[i] * Q[k - i]
        R[k] = r_k
    return R
def fft_polynomial_multiplication(P, Q):
    """
    FFT를 사용하여 두 다항식의 곱셈을 O(N log N)으로 가속한다.
    """
    N = len(P)
    M = len(Q)
    # 1. 패딩 (Padding)
    # FFT를 위한 길이는 N + M - 1 보다 크거나 같은 2의 거듭제곱으로 설정해야 한다.
    L = N + M - 1
    fft_size = 1
    while fft_size < L:
        fft_size *= 2
    # 패딩된 계수 배열
    P_padded = np.pad(P, (0, fft_size - N), 'constant')
    Q_padded = np.pad(Q, (0, fft_size - M), 'constant')
    # 2. FFT 변환
    Fp = np.fft.fft(P_padded)
    Fq = np.fft.fft(Q_padded)
    # 3. 주파수 영역에서 곱셈 (점별 곱셈)
    Fr = Fp * Fq
    # 4. 역 FFT (IFFT)를 이용해 시간 영역으로 복원
    R_complex = np.fft.ifft(Fr)
    # 결과는 실수여야 하며, 작은 부동 소수점 오차가 발생할 수 있으므로 반올림한다.
    # 합성곱의 계수는 정수이므로, 결과를 실수부만 취하고 반올림하여 정수로 만든다.
    R_coeffs = np.round(R_complex.real).astype(int)
    # 최종 결과는 L 길이만큼만 취한다.
    return R_coeffs[:L].tolist()
## --- 계산 및 검증 부분 ---
print("--- FFT 기반 다항식 곱셈 예제 및 검증 ---")
# 다항식 P(x) = 1 + 2x + 3x^2 + 4x^3  (계수: [1, 2, 3, 4])
P_coeffs = [1, 2, 3, 4]
# 다항식 Q(x) = 5 + 6x + 7x^2         (계수: [5, 6, 7])
Q_coeffs = [5, 6, 7]
# 1. 일반 곱셈으로 계산 (검증 기준)
R_naive = naive_polynomial_multiplication(P_coeffs, Q_coeffs)
# 2. FFT 곱셈으로 계산 (검증 대상)
R_fft = fft_polynomial_multiplication(P_coeffs, Q_coeffs)
# 3. 계산 결과 출력
print(f"\n다항식 P(x) 계수 (P): {P_coeffs}")
print(f"다항식 Q(x) 계수 (Q): {Q_coeffs}")
print("-" * 30)
# R(x) = P(x) * Q(x) = 5 + 16x + 38x^2 + 52x^3 + 46x^4 + 28x^5 이다.
print(f"일반 곱셈 (O(N^2)) 결과 계수 (R_naive): {R_naive}")
print(f"FFT 곱셈 (O(N log N)) 결과 계수 (R_fft): {R_fft}")
print("-" * 30)
## 4. 검증
# numpy의 allclose 함수를 사용하여 부동 소수점 오차를 감안하여 비교한다.
# 하지만 이 예제에서는 np.round를 통해 정수 비교가 가능하다.
# 길이가 같고 모든 요소가 같으면 True
is_verified = (R_naive == R_fft)
# 검증 결과 출력
print(f" 검증 결과: 두 계산 결과가 일치하는가? {is_verified}")
if not is_verified:
	# 길이가 다르거나 요소가 다를 경우 디버깅 정보 출력
	print(" 오류: 두 결과가 일치하지 않다.")
	print(f"차이: {np.array(R_naive) - np.array(R_fft)}")
else:
	print("\nFFT를 사용한 다항식 곱셈이 일반 곱셈 결과와 정확히 일치함을 확인했다.")
# --- 추가 검증: 큰 정수 곱셈 (Carry 처리 미포함) ---
print("\n--- 추가 검증: 큰 정수 곱셈 (올림 처리 전) ---")
# 큰 정수 A = 9998 (계수 [8, 9, 9, 9])
A_coeffs = [8, 9, 9, 9] 
# 큰 정수 B = 1001 (계수 [1, 0, 0, 1])
B_coeffs = [1, 0, 0, 1] 
# A * B = 9998 * 1001 = 10007998
# 나이브 방식
R_int_naive = naive_polynomial_multiplication(A_coeffs, B_coeffs)
# FFT 방식
R_int_fft = fft_polynomial_multiplication(A_coeffs, B_coeffs)
print(f"정수 A 계수: {A_coeffs}")
print(f"정수 B 계수: {B_coeffs}")
print(f"FFT 결과 (올림 전): {R_int_fft}")
# 실제 정답 (10007998)의 계수 배열: [8, 9, 9, 7, 0, 0, 1]
# 여기서 FFT 결과 [8, 9, 9, 17, 9, 9, 8]는 올림 처리를 해야 실제 정답과 일치한다.
# 예시 R_int_fft의 결과를 보면, [8, 9, 9, 17, 9, 9, 8]이 나온다.
# 여기서 17을 올림 처리하면 8, 9, 9, (7+10), 9, 9, 8 -> 8, 9, 9, 7, (9+1), 9, 8 
#  -> ... 순으로 변환된다. 
# 이 과정이 큰 정수 곱셈 알고리즘의 최종 단계이다.
# FFT 자체의 계산은 이 '올림 전의 합성곱'까지 정확하게 수행함을 검증한다.
is_int_verified = (R_int_naive == R_int_fft)
print(f" 검증 결과 (큰 정수): 두 계산 결과가 일치하는가? {is_int_verified}")

In [ ]:
import math
import random
# 최대공약수(GCD) 계산: math.gcd 사용으로 최적화
def gcd_optimized(a, b):
    return math.gcd(a, b)
# 모듈러 역원(Modular Inverse) 계산: pow(a, -1, m) 사용으로 최적화
def mod_inverse_optimized(a, m):
# a와 m이 서로소인지 확인 (RSA 요건)
    if gcd_optimized(a, m) != 1:
        raise ValueError(f"{a}는 {m}과 서로소가 아니다. 모듈러 역원이 존재하지 않는다.")    
# pow(a, -1, m)은 d * a ≡ 1 (mod m)인 d를 반환
    return pow(a, -1, m)
# 소수 판별 (간단 버전 유지)
def is_prime(num):
    if num < 2:
        return False
# 2부터 num의 제곱근까지만 확인
    for i in range(2, int(num**0.5) + 1):
        if num % i == 0:
            return False
    return True
##  RSA 키 생성
def generate_key_pair_optimized(p, q):
    if not (is_prime(p) and is_prime(q)):
        raise ValueError('p와 q는 소수여야 한다.')
    if p == q:
        raise ValueError('p와 q는 달라야 한다.')
    n = p * q
    phi = (p - 1) * (q - 1)
# 공개 지수 e 선택 (일반적으로 사용되는 65537을 먼저 시도)
    e = 65537
    while gcd_optimized(e, phi) != 1:
# 65537이 안되면 phi보다 작은 홀수 중 랜덤 선택
        e = random.randrange(3, phi, 2) 
# 비밀 지수 d 계산: 최적화된 mod_inverse 사용
    d = mod_inverse_optimized(e, phi)
# 공개 키: (e, n), 비밀 키: (d, n)
    return ((e, n), (d, n))
##  암호화 (M^e mod n)
def encrypt_optimized(public_key, plaintext):
    e, n = public_key
# pow(ord(char), e, n)으로 Modular Exponentiation 최적화
    cipher = [pow(ord(char), e, n) for char in plaintext]
    return cipher
##  복호화 (C^d mod n)
def decrypt_optimized(private_key, ciphertext):
    d, n = private_key
# pow(char, d, n)으로 Modular Exponentiation 최적화
    plain = [chr(pow(char, d, n)) for char in ciphertext]
    return ''.join(plain)
# --- 실행 예제 ---
P = 61
Q = 53
message = "OPTIMIZED RSA message"
public_key_opt, private_key_opt = generate_key_pair_optimized(P, Q)
encrypted_msg_opt = encrypt_optimized(public_key_opt, message)
decrypted_msg_opt = decrypt_optimized(private_key_opt, encrypted_msg_opt)
# --- 결과 출력 ---
print(f"선택된 소수 P: {P}, Q: {Q}")
print(f"공개 키 (e, n): {public_key_opt}")
print(f"비밀 키 (d, n): {private_key_opt}")
print(f"원문 메시지: {message}")
print(f"암호화된 메시지 (전부): {encrypted_msg_opt[:]}")
print(f"복호화된 메시지: {decrypted_msg_opt}")

In [ ]:
import numpy as np

def bit_reversal_permutation(n):
    """
    길이 n인 배열의 비트 반전 인덱스 순서를 반환한다.
    (n은 2의 거듭제곱이어야 함)
    """
    num_bits = int(np.log2(n))
    indices = np.arange(n)
    reversed_indices = np.zeros(n, dtype=int)
    
    for i in range(n):
        # 1. 2진수 문자열로 변환 (예: 1 -> '001')
        binary_str = format(i, f'0{num_bits}b')
        # 2. 문자열 뒤집기 (예: '001' -> '100')
        reversed_str = binary_str[::-1]
        # 3. 다시 10진수로 변환
        reversed_indices[i] = int(reversed_str, 2)
        
    return reversed_indices

def recursive_split_order(arr):
    """
    재귀적으로 짝수/홀수를 나누어 순서를 확인하는 함수
    (FFT의 분할 과정 시뮬레이션)
    """
    if len(arr) <= 1:
        return arr
    
    even = recursive_split_order(arr[0::2]) # 짝수 인덱스
    odd = recursive_split_order(arr[1::2])  # 홀수 인덱스
    
    return even + odd

# --- 실행 및 비교 ---
N = 8
original_data = list(range(N))

# 1. 비트 반전 알고리즘 결과
br_indices = bit_reversal_permutation(N)
br_result = [original_data[i] for i in br_indices]

# 2. 재귀적 짝/홀 분할 결과
split_result = recursive_split_order(original_data)

print(f"Original:   {original_data}")
print(f"Bit-Rev:    {br_result}")
print(f"Recursive:  {split_result}")

if br_result == split_result:
    print("\n>> 일치함! 비트 반전은 재귀적 분할의 최종 순서와 같다.")

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter # ScalarFormatter 임포트

def measure_fft_complexity_latex_exponent(max_n_exponent=20):
    """
    FFT의 계산 복잡도를 측정하고, 결과를 T(n) vs n log(n) 그래프로 시각화한다.
    x축 레이블을 LaTeX 스타일의 지수 표기법 (예: 10^7)으로 표시한다.
    """
    n_values = []
    times = []
    # 입력 크기 N을 2의 거듭제곱으로 증가시키며 실험
    for k in range(8, max_n_exponent + 1):
        n = 2**k
        n_values.append(n)
        
        # 무작위 복소수 배열 생성
        data = np.random.rand(n) + 1j * np.random.rand(n)
        
        # 시간 측정
        start_time = time.perf_counter()
        
        # FFT 실행
        np.fft.fft(data)
        
        end_time = time.perf_counter()
        elapsed_time = end_time - start_time
        times.append(elapsed_time)
        
        print(f"N = {n}: Time = {elapsed_time:.6f} seconds")

    # --- 시각화 ---
    plt.figure(figsize=(8, 6))
    
    # x축 값을 n log(n)으로 계산한다 (log base 2 사용)
    n_log_n = np.array(n_values) * np.log2(np.array(n_values))
    
    # 1. 시각화 (n log(n) vs Time)
    plt.plot(n_log_n, times, 'o-', label='Measured time $T(N)$')
    
    # --- 핵심 수정 부분: x축 포맷 설정 ---
    ax = plt.gca()
    
    # 과학적 표기법(scientific notation)을 강제한다.
    # scilimits=(0,0)은 모든 값에 대해 지수 표기법을 사용하도록 한다.
    ax.ticklabel_format(axis='x', style='sci', scilimits=(0,0))
    
    # 지수 표기법의 텍스트 형식을 사용자 정의하여 '10^x' 형태로 만든다.
    # $...$ 를 사용하여 LaTeX 수식 모드를 활성화하고, 지수를 수식처럼 표시한다.
    formatter = ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((-2, 6)) # 이 설정은 지수 표기법을 사용할 범위를 지정
    ax.xaxis.set_major_formatter(formatter)
    plt.xlabel('Operation count proxy ($N \log_2 N$)')
    plt.ylabel('Time (seconds)')
    plt.title('FFT complexity check: $T(N)$ vs $N \log_2 N$')
    plt.legend()
    plt.grid(True)
    plt.savefig('nlogn.png')
    plt.show()
    # 
# 실험 실행
if __name__ == "__main__":
    measure_fft_complexity_latex_exponent(max_n_exponent=27)

In [ ]:
def merge_sort(arr):
    # Base Case: 리스트 길이가 1 이하면 이미 정렬된 상태
    if len(arr) <= 1:
        return arr
    # 1. 분할 (Divide): 가운데를 기준으로 쪼갬
    mid = len(arr) // 2
    left_half = arr[:mid]
    right_half = arr[mid:]
    # 재귀 호출을 통해 끝까지 쪼갬
    left_sorted = merge_sort(left_half)
    right_sorted = merge_sort(right_half)
    # 2. 병합 (Merge): 두 개의 정렬된 리스트를 하나로 합침
    return merge(left_sorted, right_sorted)

def merge(left, right):
    sorted_list = []
    i = 0 # 왼쪽 리스트 인덱스
    j = 0 # 오른쪽 리스트 인덱스
    # 두 리스트를 앞에서부터 비교하며 작은 값을 결과에 추가
    while i < len(left) and j < len(right):
        if left[i] < right[j]:
            sorted_list.append(left[i])
            i += 1
        else:
            sorted_list.append(right[j])
            j += 1
    # 남은 데이터들을 뒤에 이어 붙임
    sorted_list.extend(left[i:])
    sorted_list.extend(right[j:])
    return sorted_list

# 실행 예시
data = [38, 27, 43, 3, 9, 82, 10]
print(f"정렬 전: {data}")
print(f"정렬 후: {merge_sort(data)}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 샘플링 주파수 (Sampling frequency, Hz)
Fs = 500 
# 시간 간격 (Time step)
T = 1.0 / Fs 
# 신호의 길이 (샘플 개수)
L = 1500 
# 시간 벡터
t = np.linspace(0.0, L*T, L, endpoint=False)

# 두 개의 사인파 신호 생성
# 1. 주파수 50Hz, 진폭 0.7
f1 = 50.0 
A1 = 0.7
signal1 = A1 * np.sin(2 * np.pi * f1 * t)

# 2. 주파수 120Hz, 진폭 2.0
f2 = 120.0 
A2 = 2.0
signal2 = A2 * np.sin(2 * np.pi * f2 * t)

# 두 신호를 합친 복합 신호
signal = signal1 + signal2

# 시간 영역 그래프 출력
plt.figure(figsize=(12, 4))
plt.plot(t, signal)
plt.title('Time domain signal (50 Hz + 120 Hz)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.grid()
plt.show()


# FFT 수행
# signal_fft는 복소수 형태의 결과이다.
signal_fft = np.fft.fft(signal)

# 진폭 스펙트럼 계산 (Magnitude Spectrum)
# np.abs()를 사용하여 복소수의 크기(진폭)를 구한다.
# FFT 결과는 대칭적이므로, 절반만 사용하고, 정규화(L로 나누고 2를 곱함)한다.
# 첫 번째 요소(DC 성분)는 2를 곱할 필요가 없다.
P2 = np.abs(signal_fft/L)
P1 = P2[0:L//2]
P1[0] = P1[0] / 2
P1[1:] = 2*P1[1:] 

# 주파수 축 생성 (Frequency Axis)
# 샘플링 주파수와 신호 길이를 이용해 주파수 벡터를 만든다.
f = Fs * np.arange(L/2) / L 

# 주파수 영역 그래프 출력
plt.figure(figsize=(12, 4))
plt.plot(f, P1)
plt.title('Frequency domain spectrum')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude (Amplitude)')
plt.xlim(0, Fs/2) # Nyquist 주파수까지 표시
plt.grid()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq

# 1. 신호 생성 및 노이즈 추가
# ----------------------------------
# 샘플링 매개변수
Fs = 1000  # 샘플링 주파수 (Hz)
T = 1.0    # 신호 길이 (초)
N = int(Fs * T) # 총 샘플 수
t = np.linspace(0.0, T, N, endpoint=False) # 시간 벡터

# 원본 신호 (50Hz와 120Hz 두 사인파의 합)
f1 = 50.0  # 주파수 1
f2 = 120.0 # 주파수 2
signal_clean = 0.7 * np.sin(2 * np.pi * f1 * t) + 1.0 * np.sin(2 * np.pi * f2 * t)

# 노이즈 추가 (랜덤 노이즈)
noise = 0.8 * np.random.randn(N)
signal_noisy = signal_clean + noise

# 2. FFT 수행
# ----------------------------------
# FFT 계산
yf = fft(signal_noisy)
# 주파수 축 생성
xf = fftfreq(N, 1/Fs)[:N//2] # 양의 주파수만 사용

# 3. 노이즈 필터링 (주파수 도메인에서)
# ----------------------------------
# 필터링을 위한 임계값 설정 (노이즈로 간주되는 낮은 진폭 제거)
# 이 임계값은 신호의 특성에 따라 조정해야 한다.
threshold = 100 

# 복소수 스펙트럼의 진폭을 기준으로 임계값 미만인 주파수 성분을 0으로 설정
yf_filtered = yf.copy()
# 양수 주파수 영역에서 필터링
yf_filtered[np.abs(yf_filtered) < threshold] = 0
# 음수 주파수 영역은 대칭적으로 처리해야 하지만, 
# 일반적으로 대칭적인 FFT 결과에서 양수 주파수를 필터링하면 음수도 자동으로 처리되거나,
# rfft/irfft를 사용하면 더 간편하다. 여기서는 전체 yf를 필터링하는 방식으로 단순화한다.

# 4. 역 FFT (iFFT) 수행
# ----------------------------------
# 필터링된 스펙트럼을 시간 도메인으로 변환 (iFFT)
# 결과는 복소수이므로, 실수 부분만 취한다.
signal_filtered = np.real(ifft(yf_filtered))

# 5. 결과 시각화
# ----------------------------------
plt.figure(figsize=(12, 8))

# 시간 도메인 플롯
plt.subplot(3, 1, 1)
plt.plot(t, signal_noisy, label='Noisy signal', alpha=0.6)
plt.plot(t, signal_clean, label='Clean signal', color='red', linestyle='--')
plt.title('Time domain: Noisy vs Clean signal')
plt.legend()
plt.xlim(0, 0.1) # 일부 구간만 확대

plt.subplot(3, 1, 2)
# 주파수 스펙트럼 플롯 (진폭의 절댓값)
plt.plot(xf, 2.0/N * np.abs(yf[0:N//2]), label='Noisy spectrum')
plt.plot(xf, 2.0/N * np.abs(yf_filtered[0:N//2]), label='Filtered spectrum', linestyle='--')
plt.title('Frequency domain: Magnitude spectrum')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude')
plt.legend()

# 필터링된 시간 도메인 플롯
plt.subplot(3, 1, 3)
plt.plot(t, signal_filtered, label='Filtered signal', color='green')
plt.plot(t, signal_clean, label='Clean signal', color='red', linestyle='--')
plt.title('Time domain: Filtered vs Clean signal')
plt.legend()
plt.xlim(0, 0.1) # 일부 구간만 확대

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 신호 정의
f_signal = 100  # 원래 신호의 주파수 (100 Hz)
t_max = 0.15     # 시뮬레이션 시간

# 2. 연속 신호 (고해상도 시간 벡터) 생성
# 플로팅을 위해 매우 높은 샘플링 (10000 Hz) 사용
t_continuous = np.linspace(0, t_max, 1000, endpoint=False)
x_continuous = np.sin(2 * np.pi * f_signal * t_continuous)

# 3. 앨리어싱을 유발하는 낮은 표본화 주파수 정의
f_sampling = 15  # 매우 낮은 표본화 주파수 (15 Hz)
# 나이퀴스트 주파수 (f_s / 2)는 7.5 Hz. 신호 주파수 100 Hz보다 훨씬 낮음.

# 4. 표본화된 시간 벡터 생성
n_samples = int(f_sampling * t_max)
t_sampled = np.linspace(0, t_max, n_samples, endpoint=False)
x_sampled = np.sin(2 * np.pi * f_signal * t_sampled)

# 5. 앨리어싱 주파수 계산 (접힌 주파수)
# 앨리어싱 주파수 f_alias = |f_signal - n * f_sampling|
n = round(f_signal / f_sampling) # 가장 가까운 정수 n
f_alias = abs(f_signal - n * f_sampling)

# 6. 플로팅
plt.figure(figsize=(12, 6))

# A. 원래 신호 (연속선)
plt.plot(t_continuous, x_continuous, 
         label=f'Original signal ({f_signal} Hz)', color='gray', linestyle='--')

# B. 표본화된 점들
plt.scatter(t_sampled, x_sampled, 
            label=f'Sampling ({f_sampling} Hz)', color='red', marker='o', s=50)

# C. 앨리어싱 주파수 (저주파로 보이는 신호)
t_alias = np.linspace(0, t_max, 1000, endpoint=False)
x_alias = np.sin(2 * np.pi * f_alias * t_alias)
plt.plot(t_alias, x_alias, 
         label=f'Aliasing result ({f_alias:.2f} Hz)', color='blue')

plt.title(f'Aliasing phenomenon: {f_signal} Hz signal -> {f_sampling} Hz Sampling', fontsize=16)
plt.xlabel('Time (sec)')
plt.ylabel('Signal (arb. unit)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft

# --- 설정 변수 ---
fs = 1000  # 샘플링 주파수 (Hz)
T = 1.0    # 총 시간 (초)
t = np.linspace(0, T, int(fs * T), endpoint=False) 

# --- 주파수 변동 정의 ---
# 4개의 주파수 성분과 각 성분이 유지되는 시간의 비율
frequencies = [25, 100, 50, 150]  # Hz
segments = 4                     # 분할된 세그먼트 수
seg_len = len(t) // segments     # 각 세그먼트의 길이 (250 샘플)

# 1. 시계열 데이터 생성 (여러 번 변동)
# --------------------------------------
y = np.zeros_like(t)

for i in range(segments):
    start_idx = i * seg_len
    end_idx = (i + 1) * seg_len
    
    # 해당 세그먼트의 시간 벡터와 주파수
    t_segment = t[start_idx:end_idx]
    f = frequencies[i]
    
    # 해당 세그먼트에 사인파 추가
    y[start_idx:end_idx] = np.sin(2 * np.pi * f * t_segment)

# 2. 단시간 푸리에 변환 (STFT) 수행
# -------------------------------------
# STFT 파라미터는 이전과 동일하게 유지한다.
f, t_stft, Zxx = stft(y, fs=fs, nperseg=256, noverlap=128)

# 진폭을 얻기 위해 절댓값을 취한다.
Zxx_amplitude = np.abs(Zxx)

# 3. 결과 시각화
# ----------------

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# 3-1. 원본 시계열 데이터 (시간 도메인)
ax1.plot(t, y)
ax1.set_title('1D time series data (multiple frequency changes)')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')
ax1.grid(True)

# 3-2. 스펙트로그램 (STFT 결과, 시간-주파수 도메인)
c = ax2.pcolormesh(t_stft, f, Zxx_amplitude, shading='gouraud')
fig.colorbar(c, ax=ax2, label='Amplitude')

ax2.set_title('Spectrogram(multiple frequency changes)')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Frequency (Hz)')
# 주파수 축을 0에서 200Hz까지만 표시하여 주요 성분을 잘 보이게 조정한다.
ax2.set_ylim(0, 200)

plt.tight_layout()
plt.savefig('stft.png')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pywt
from scipy.fft import fft, fftfreq

def compare_fft_wavelet():
    # 1. 신호 생성 (Sample Rate: 1000Hz)
    N = 1000
    T = 1.0 / N
    t = np.linspace(0, 1, N, endpoint=False)
    # A. Chirp Signal: 주파수가 0Hz에서 100Hz로 증가 (이차함수 꼴 위상)
    # sin(2 * pi * (k * t^2)) -> 주파수가 선형적으로 증가함
    signal = np.sin(2 * np.pi * 50 * t**2)
    # B. Transient (갑작스러운 변화): 0.5초에 튀는 값 추가
    signal[500] += 3.0 
    # ==========================================
    # 2. FFT 수행 (Fast Fourier Transform)
    # ==========================================
    yf = fft(signal)
    xf = fftfreq(N, T)[:N//2] # 양의 주파수 대역만 사용
    fft_magnitude = 2.0/N * np.abs(yf[0:N//2]) # 크기 정규화
    # ==========================================
    # 3. 웨이브렛 변환 수행 (Continuous Wavelet Transform)
    # ==========================================
    # 복소 모렛(Complex Morlet) 웨이브렛 사용 -> 주파수 분석에 탁월
    scales = np.arange(1, 128)
    coef, freqs = pywt.cwt(signal, scales, 'cmor1.5-1.0', sampling_period=T)
    # ==========================================
    # 4. 시각화 및 비교
    # ==========================================
    plt.figure(figsize=(12, 10))
    # (1) 원본 신호 (Time Domain)
    plt.subplot(3, 1, 1)
    plt.plot(t, signal, 'k', linewidth=1)
    plt.title("1. Time domain signal(Chirp + Spike at 0.5 s)")
    plt.xlabel("Time (s)")
    plt.grid(True)
    # (2) FFT 결과 (Frequency Domain)
    plt.subplot(3, 1, 2)
    plt.plot(xf, fft_magnitude, 'r')
    plt.title("2. FFT Result(Frequency domain)")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Magnitude")
    plt.grid(True)
    plt.text(110, np.max(fft_magnitude)*0.8, "Check: We see frequencies,\nbut WHEN did they happen?", 
             bbox=dict(facecolor='white', alpha=0.9))
    # (3) 웨이브렛 결과 (Time-Frequency Domain)
    plt.subplot(3, 1, 3)
    # x축: 시간, y축: 주파수, 색상: 에너지
    plt.pcolormesh(t, freqs, np.abs(coef), shading='gouraud', cmap='jet')
    plt.title("3. Wavelet transform (Time-Frequency domain)")
    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (Hz)")
    plt.ylim(0, 150) # 관심 주파수 대역만 표시
    # 설명 화살표
    plt.annotate('Frequency rising', xy=(0.8, 80), xytext=(0.5, 100),
                 arrowprops=dict(facecolor='white', shrink=0.05), color='white')
    plt.annotate('Spike at 0.5 s', xy=(0.5, 0), xytext=(0.6, 20),
                 arrowprops=dict(facecolor='white', shrink=0.05), color='white')
    plt.tight_layout()
    plt.savefig('time-freq.png')
    plt.show()
compare_fft_wavelet()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq

# 1. 신호 생성 및 노이즈 추가
# ----------------------------------
# 샘플링 매개변수
Fs = 1000  # 샘플링 주파수 (Hz)
T = 1.0    # 신호 길이 (초)
N = int(Fs * T) # 총 샘플 수
t = np.linspace(0.0, T, N, endpoint=False) # 시간 벡터

# 원본 신호 (50Hz와 120Hz 두 사인파의 합)
f1 = 50.0  # 주파수 1
f2 = 120.0 # 주파수 2
signal_clean = 0.7 * np.sin(2 * np.pi * f1 * t) + 1.0 * np.sin(2 * np.pi * f2 * t)

# 노이즈 추가 (랜덤 노이즈)
noise = 0.8 * np.random.randn(N)
signal_noisy = signal_clean + noise

# 2. FFT 수행
# ----------------------------------
# FFT 계산
yf = fft(signal_noisy)
# 주파수 축 생성
xf = fftfreq(N, 1/Fs)[:N//2] # 양의 주파수만 사용

# 3. 노이즈 필터링 (주파수 도메인에서)
# ----------------------------------
# 필터링을 위한 임계값 설정 (노이즈로 간주되는 낮은 진폭 제거)
# 이 임계값은 신호의 특성에 따라 조정해야 한다.
threshold = 100 

# 복소수 스펙트럼의 진폭을 기준으로 임계값 미만인 주파수 성분을 0으로 설정
yf_filtered = yf.copy()
# 양수 주파수 영역에서 필터링
yf_filtered[np.abs(yf_filtered) < threshold] = 0
# 음수 주파수 영역은 대칭적으로 처리해야 하지만, 
# 일반적으로 대칭적인 FFT 결과에서 양수 주파수를 필터링하면 음수도 자동으로 처리되거나,
# rfft/irfft를 사용하면 더 간편하다. 여기서는 전체 yf를 필터링하는 방식으로 단순화한다.

# 4. 역 FFT (iFFT) 수행
# ----------------------------------
# 필터링된 스펙트럼을 시간 도메인으로 변환 (iFFT)
# 결과는 복소수이므로, 실수 부분만 취한다.
signal_filtered = np.real(ifft(yf_filtered))

# 5. 결과 시각화
# ----------------------------------
plt.figure(figsize=(12, 8))

# 시간 도메인 플롯
plt.subplot(3, 1, 1)
plt.plot(t, signal_noisy, label='Noisy signal', alpha=0.6)
plt.plot(t, signal_clean, label='Clean signal', color='red', linestyle='--')
plt.title('Time domain: Noisy vs Clean signal')
plt.legend()
plt.xlim(0, 0.1) # 일부 구간만 확대

plt.subplot(3, 1, 2)
# 주파수 스펙트럼 플롯 (진폭의 절댓값)
plt.plot(xf, 2.0/N * np.abs(yf[0:N//2]), label='Noisy spectrum')
plt.plot(xf, 2.0/N * np.abs(yf_filtered[0:N//2]), label='Filtered spectrum', linestyle='--')
plt.title('Frequency domain: Magnitude spectrum')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude')
plt.legend()

# 필터링된 시간 도메인 플롯
plt.subplot(3, 1, 3)
plt.plot(t, signal_filtered, label='Filtered signal', color='green')
plt.plot(t, signal_clean, label='Clean signal', color='red', linestyle='--')
plt.title('Time domain: Filtered vs Clean signal')
plt.legend()
plt.xlim(0, 0.1) # 일부 구간만 확대

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2 # OpenCV

def visualize_2d_gabor():
    # 설정: 필터 크기, 표준편차(sigma), 방향(theta), 파장(lambda), 감마(gamma)
    ksize = 31
    sigma = 4.0
    lambd = 10.0 # 파장 (주파수의 역수)
    gamma = 0.5  # 타원형 비율
    psi = 0      # 위상 오프셋
    # 방향을 다르게 하여 4개의 가보 필터 생성
    thetas = [0, 45, 90, 135] # 각도 (Degree)
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for i, angle_deg in enumerate(thetas):
        # 각도를 라디안으로 변환
        theta = np.deg2rad(angle_deg)
        # OpenCV로 2D Gabor 커널 생성
        # (수식: Gaussian * Cosine)
        kernel = cv2.getGaborKernel((ksize, ksize), sigma, theta, lambd, gamma, psi, ktype=cv2.CV_32F)
        # 시각화
        axes[i].imshow(kernel, cmap='gray')
        axes[i].set_title(f"Gabor filter\nAngle: {angle_deg}°")
        axes[i].axis('off')
    plt.suptitle("2D Gabor wavelets(directional edge detectors)", fontsize=16)
    plt.savefig('gabor1.png')
    plt.show()
    # 3D로 하나만 자세히 보기 (구조 이해용)
    from mpl_toolkits.mplot3d import Axes3D
    # 90도 필터 선택
    kernel_90 = cv2.getGaborKernel((ksize, ksize), sigma, np.pi/2, lambd, gamma, psi, ktype=cv2.CV_32F)
    x, y = np.meshgrid(np.arange(ksize), np.arange(ksize))
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(x, y, kernel_90, cmap='jet')
    ax.set_title("3D structure of Gabor wavelet(90 deg)")
    plt.savefig('gabor2.png')
    plt.show()
visualize_2d_gabor()

In [ ]:
import numpy as np

def big_int_multiply_fft(A, B):
    """
    FFT를 사용하여 두 큰 정수 A와 B를 곱한다.
    """
    # 1. 정수를 계수 배열로 변환
    # (각 자릿수를 다항식의 계수로 간주)
    # 123 -> [3, 2, 1] (낮은 차수부터)
    def int_to_coeffs(n):
        return np.array([int(d) for d in str(n)][::-1], dtype=complex)

    coeffs_a = int_to_coeffs(A)
    coeffs_b = int_to_coeffs(B)

    # 2. 패딩 (길이 맞추기)
    n_a = len(coeffs_a)
    n_b = len(coeffs_b)
    
    # 합성곱의 결과 길이는 n_a + n_b - 1. FFT를 위해 2의 거듭제곱으로 패딩
    result_len = n_a + n_b - 1
    fft_size = 1
    while fft_size < result_len:
        fft_size *= 2
        
    # 3. FFT 수행 (주파수 영역 변환)
    # 패딩된 배열에 FFT 적용
    fft_a = np.fft.fft(coeffs_a, n=fft_size)
    fft_b = np.fft.fft(coeffs_b, n=fft_size)

    # 4. 원소별 곱셈 (주파수 영역에서 곱)
    fft_c = fft_a * fft_b

    # 5. IFFT 수행 (계수 영역 복귀)
    coeffs_c = np.fft.ifft(fft_c)
    
    # 부동 소수점 오차를 줄이기 위해 반올림 후 정수로 변환
    # IFFT 결과는 복소수 형태이므로 실수 부분만 취함
    raw_coeffs = np.round(coeffs_c.real).astype(int)

    # 6. 자리올림 처리 (Carry Propagation)
    final_result = []
    carry = 0
    
    for coeff in raw_coeffs:
        # 현재 자릿수 값에 이전 자리에서 넘어온 올림을 더한다.
        current_sum = coeff + carry
        
        # 실제 자릿값 (10으로 나눈 나머지)
        digit = current_sum % 10
        final_result.append(digit)
        
        # 다음 자릿수로 넘길 올림 (10으로 나눈 몫)
        carry = current_sum // 10

    # 마지막에 남은 올림이 있다면 추가
    while carry > 0:
        final_result.append(carry % 10)
        carry //= 10

    # 결과를 정수 문자열로 조합 (다시 역순으로)
    result_str = "".join(map(str, final_result[::-1]))
    
    # 결과를 int로 변환하여 반환
    return int(result_str)

# --- 사용 예시 ---
A = 1234567890123456789
B = 9876543210987654321

result = big_int_multiply_fft(A, B)

print(f"A = {A}")
print(f"B = {B}")
print(f"A * B (FFT) = {result}")

# 파이썬 기본 연산으로 검증
expected = A * B
print(f"A * B (Python) = {expected}")
print(f"결과 일치 여부: {result == expected}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pywt

def plot_scalogram_interpretation():
    # 1. 신호 생성 (Sample Rate: 1000Hz, 1초)
    fs = 1000
    t = np.linspace(0, 1, fs)
    # (A) 저주파 (10Hz): 처음부터 끝까지
    sig1 = np.sin(2 * np.pi * 10 * t)
    # (B) 고주파 (100Hz): 0.5초부터 등장
    sig2 = np.zeros_like(t)
    sig2[500:] = np.sin(2 * np.pi * 100 * t[500:])
    # (C) 순간 충격 (Impulse): 0.2초에 쾅!
    sig3 = np.zeros_like(t)
    sig3[200] = 10.0 # 큰 스파이크
    signal = sig1 + sig2 + sig3
    # 2. CWT 수행
    # 스케일을 1부터 128까지 설정
    scales = np.arange(1, 128)
    wavelet = 'cmor1.5-1.0' # 복소 모렛 (주파수 분석에 좋음)
    coefficients, frequencies = pywt.cwt(signal, scales, wavelet, sampling_period=1/fs)
    # 3. 스칼로그램 시각화
    plt.figure(figsize=(12, 8))
    # (1) 시간 영역 신호
    plt.subplot(2, 1, 1)
    plt.plot(t, signal, 'k', linewidth=1)
    plt.title("Time domain signal")
    plt.grid(True)
    # (2) 스칼로그램 (Time-Frequency Domain)
    plt.subplot(2, 1, 2)
    # pcolormesh를 사용하여 히트맵 그리기
    # Y축을 frequencies로 설정해야 '스케일'이 아닌 'Hz'로 보임
    plt.pcolormesh(t, frequencies, np.abs(coefficients), cmap='jet', shading='gouraud')
    plt.title("Scalogram(CWT power spectrum)")
    plt.ylabel("Frequency (Hz)")
    plt.xlabel("Time (sec)")
    plt.ylim(0, 150) # 관심 영역만 확대
    # 해석 주석 달기
    plt.annotate('Spike(impulse)\nWide frequency spread', xy=(0.2, 50), xytext=(0.1, 100),
                 arrowprops=dict(facecolor='white', shrink=0.05), color='white', ha='center')
    
    plt.annotate('Low Freq.(10 Hz)\nSteady line', xy=(0.8, 10), xytext=(0.8, 40),
                 arrowprops=dict(facecolor='white', shrink=0.05), color='white', ha='center')

    plt.annotate('High Freq.(100 Hz)\nStarts at 0.5s', xy=(0.6, 100), xytext=(0.4, 130),
                 arrowprops=dict(facecolor='white', shrink=0.05), color='white', ha='center')

    plt.colorbar(label='Magnitude')
    plt.tight_layout()
    plt.show()

plot_scalogram_interpretation()

import numpy as np
import matplotlib.pyplot as plt
import pywt

def definition_cwt_demo():
    # 1. 신호 생성: 10Hz -> 30Hz로 변하는 Chirp 신호
    t = np.linspace(0, 1, 400)
    signal = np.sin(2 * np.pi * 10 * t + 10 * t**2)
    
    # 2. CWT 파라미터 설정
    # 연속적인 스케일 설정 (1부터 30까지 촘촘하게)
    scales = np.arange(1, 31)
    wavelet = 'cmor' # 복소 모렛 웨이브렛
    
    # 3. CWT 수행
    # 결과 coeffs는 (스케일 개수 x 시간 길이)의 2차원 행렬이 됨
    coeffs, freqs = pywt.cwt(signal, scales, wavelet, sampling_period=1/400)
    
    # 4. 시각화
    plt.figure(figsize=(10, 6))
    
    # 등고선 그래프로 표현 (실수부만 표시)
    plt.imshow(np.real(coeffs), aspect='auto', cmap='coolwarm', 
               extent=[0, 1, 1, 30]) # extent=[x_min, x_max, y_min, y_max]
    
    plt.title("Definition of CWT: Similarity map")
    plt.ylabel("Scale (a)")
    plt.xlabel("Time (b)")
    plt.colorbar(label="Coefficient value W(a,b)")
    plt.show()

definition_cwt_demo()

In [ ]:
import numpy as np
import pywt

def find_exact_convolution_match():
    # 1. 데이터 및 웨이브렛 설정
    # 정렬 확인을 위해 비대칭적인 데이터를 사용한다.
    data = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=float)
    wavelet_name = 'db2' # db1, db2, sym4 등 변경해서 테스트 가능
    print(f"--- Testing Wavelet: {wavelet_name} ---")
    # 2. PyWavelets DWT 결과 (목표값)
    # mode='zero': 패딩을 0으로 처리하여 비교를 단순화
    cA_target, _ = pywt.dwt(data, wavelet_name, mode='zero')
    # 3. 직접 합성곱 구현 준비
    w = pywt.Wavelet(wavelet_name)
    dec_lo = np.array(w.dec_lo) # 분해 필터 계수
    # [중요] 합성곱 vs 상관(Correlation)
    # DWT 수식은 보통 Correlation(내적) 형태이다.
    # np.convolve는 필터를 뒤집어서 계산한다.
    # 따라서 DWT와 맞추려면 np.convolve에 넣을 때 필터를 '미리 뒤집거나' vs '그대로 쓰거나' 확인해야 한다.
    # 일반적으로 DWT 구현체들은 Correlation 방식을 쓰므로, convolve를 쓸 땐 필터를 뒤집는 게 맞다.
    filters_to_test = {
        "Original Filter": dec_lo,       # 그냥 합성곱
        "Flipped Filter ": dec_lo[::-1]  # 뒤집어서 합성곱 (Correlation 효과)
    }
    match_found = False
    # 4. 모든 가능성 테스트 (Brute Force Alignment)
    for filter_type, flt in filters_to_test.items():
        # 합성곱 수행 (Full mode: 모든 겹치는 구간 계산)
        conv_result = np.convolve(data, flt, mode='full')
        # 가능한 모든 시작 위치(Shift)를 탐색
        # 필터 길이만큼의 범위 내에서 시작점이 결정됨
        for shift in range(len(flt) + 2):
            # shift부터 2칸씩 건너뛰며(Downsampling) 가져옴
            cA_manual = conv_result[shift : : 2]
            # 길이 맞추기 (Target 길이만큼만 잘라서 비교)
            if len(cA_manual) >= len(cA_target):
                cA_manual = cA_manual[:len(cA_target)]
            else:
                continue # 길이가 부족하면 패스
            # 오차 계산
            error = np.sum((cA_target - cA_manual)**2)
            # 오차가 0에 가까우면 찾은 것임
            if error < 1e-10:
                print(f"\n>>> [성공!] 일치하는 조합을 찾았다.")
                print(f"1. 필터 방식: {filter_type}")
                print(f"2. 시작 위치(Shift): {shift}")
                print("-" * 30)
                print(f"PyWavelets: {cA_target}")
                print(f"Convolution: {cA_manual}")
                print(f"Total Error: {error:.20f}")
                match_found = True
                break
        if match_found:
            break
    if not match_found:
        print("\n>>> 실패: 일치하는 조합을 찾지 못했다.")
find_exact_convolution_match()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pywt

def cwt_for_cnn_preprocessing():
    # 1. 1D 신호 생성 (예: 기계 고장 진단 신호)
    # 정상 신호 + 고주파 노이즈 + 간헐적 충격(Impulse)
    t = np.linspace(0, 1, 1000)
    signal = np.sin(2 * np.pi * 10 * t) + \
             0.5 * np.sin(2 * np.pi * 50 * t) + \
             0.2 * np.random.randn(1000)
    
    # 0.5초에 충격 추가
    signal[500:520] += 2.0 * np.sin(2 * np.pi * 200 * t[500:520])
    # ==========================================
    # 2. CWT 수행 (Feature Extraction)
    # ==========================================
    # CNN의 입력 크기에 맞춰 스케일 개수 조절 (예: 64x64 이미지를 원한다면 scale 64개)
    scales = np.arange(1, 65) 
    coeffs, freqs = pywt.cwt(signal, scales, 'morl') # Morlet Wavelet 사용
    # 계수(Complex)의 절대값 -> 에너지(크기)
    cwt_image = np.abs(coeffs)
    # ==========================================
    # 3. CNN 입력용 정규화 (Normalization)
    # ==========================================
    # 이미지는 보통 0~255 또는 0~1 사이의 값을 가짐
    cwt_image = (cwt_image - np.min(cwt_image)) / (np.max(cwt_image) - np.min(cwt_image))
    # CNN은 보통 (Batch, Height, Width, Channel) 형태를 원함
    # 현재: (64, 1000) -> 리사이즈 필요 (여기선 시각화만 함)
    # ==========================================
    # 4. 시각화: "이것이 CNN이 보게 될 그림이다"
    # ==========================================
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    # (1) 원본 1D 신호
    axes[0].plot(t, signal, 'k')
    axes[0].set_title("1. Input 1D Signal (Time series)")
    axes[0].set_xlabel("Time")
    axes[0].grid(True)
    # (2) CWT 변환 결과 (2D Image)
    # 이것이 CNN의 Input Feature Map이 된다.
    im = axes[1].imshow(cwt_image, cmap='jet', aspect='auto', origin='lower')
    axes[1].set_title("2. CWT Scalogram (Input for CNN)")
    axes[1].set_xlabel("Time (pixel)")
    axes[1].set_ylabel("Scale / Frequency (pixel)")
    plt.colorbar(im, ax=axes[1])
    plt.tight_layout()
    plt.show()
    print(f"변환된 이미지 크기: {cwt_image.shape} (Height: Scales, Width: Time)")
    print("이제 이 2D 배열을 CNN(Conv2D) 레이어에 넣으면 학습이 시작된다.")

cwt_for_cnn_preprocessing()

In [ ]:
import pywt
import matplotlib.pyplot as plt

def plot_wavelet_families():
    # 비교할 웨이블릿 리스트
    # haar: 각짐, db4: 표준, sym5: 대칭형, coif5: 더 복잡함
    wavelets = ['haar', 'db4', 'sym5', 'coif5']
    
    fig, axes = plt.subplots(len(wavelets), 2, figsize=(12, 10))
    fig.suptitle("Visualizing Wavelet Basis Sets (Phi & Psi)", fontsize=16)
    
    for i, name in enumerate(wavelets):
        w = pywt.Wavelet(name)
        
        # wavefun: 웨이블릿 함수와 스케일링 함수의 근사치를 반환
        phi, psi, x = w.wavefun(level=8)
        
        # 1. Scaling Function (Phi) - 저주파(Trend) 담당
        axes[i, 0].plot(x, phi, 'k')
        axes[i, 0].set_title(f"{name} - Scaling Function (Phi)")
        axes[i, 0].grid(True, alpha=0.3)
        
        # 2. Wavelet Function (Psi) - 고주파(Detail) 담당
        axes[i, 1].plot(x, psi, 'b')
        axes[i, 1].set_title(f"{name} - Wavelet Function (Psi)")
        axes[i, 1].grid(True, alpha=0.3)
        
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.show()

plot_wavelet_families()

import numpy as np
import matplotlib.pyplot as plt
import pywt

def sym5_denoising_demo():
    # 1. 신호 생성: 대칭적인 가우시안 펄스 2개 (생체 신호나 레이더 반사파 흉내)
    t = np.linspace(0, 1, 500)
    
    # 0.3초와 0.7초 위치에 피크 생성
    clean_signal = 3.0 * np.exp(-500 * (t - 0.3)**2) + \
                   2.0 * np.exp(-300 * (t - 0.7)**2)
    
    # 노이즈 추가
    np.random.seed(42)
    noise = 0.5 * np.random.randn(len(t))
    noisy_signal = clean_signal + noise
    
    # ==========================================
    # 2. 웨이브렛 변환 및 노이즈 제거 (Using 'sym5')
    # ==========================================
    wavelet_name = 'sym5'
    
    # (1) 다단계 분해 (Multilevel Decomposition) - 레벨 4까지 분해
    # coeffs = [cA4, cD4, cD3, cD2, cD1] 형태로 반환됨
    coeffs = pywt.wavedec(noisy_signal, wavelet_name, level=4)
    
    # (2) 임계값 설정 (Thresholding)
    # 노이즈는 주로 작은 계수값을 가지므로, 일정 크기 이하는 0으로 깎아버림
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745 # 노이즈 레벨 추정
    threshold = sigma * np.sqrt(2 * np.log(len(noisy_signal))) # Universal Threshold
    
    # Soft Thresholding 적용 (값을 부드럽게 깎음 -> 신호가 더 매끄러워짐)
    new_coeffs = []
    # 근사 계수(cA4)는 건드리지 않고(append), 상세 계수들(cD)만 처리
    new_coeffs.append(coeffs[0]) 
    
    for detail_coeff in coeffs[1:]:
        # pywt.threshold 함수 사용 (mode='soft')
        new_coeffs.append(pywt.threshold(detail_coeff, threshold, mode='soft'))
        
    # (3) 역변환 (Reconstruction)
    reconstructed_signal = pywt.waverec(new_coeffs, wavelet_name)
    
    # 데이터 길이 보정 (경계 처리로 인해 1~2개 차이날 수 있음)
    reconstructed_signal = reconstructed_signal[:len(t)]

    # ==========================================
    # 3. 시각화
    # ==========================================
    plt.figure(figsize=(12, 8))
    
    # 원본(Clean) vs 노이즈(Noisy)
    plt.subplot(2, 1, 1)
    plt.plot(t, noisy_signal, 'k', alpha=0.5, label='Noisy Signal')
    plt.plot(t, clean_signal, 'g--', linewidth=2, label='Original Pulse')
    plt.title(f"Input: Gaussian Pulses with Noise")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    
    # 복원 결과 비교
    plt.subplot(2, 1, 2)
    plt.plot(t, clean_signal, 'g--', linewidth=1, alpha=0.7, label='Original')
    plt.plot(t, reconstructed_signal, 'b', linewidth=2, label=f'Reconstructed (sym5)')
    
    plt.title(f"Output: Denoising Result using '{wavelet_name}'")
    plt.xlabel("Time")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    
    # 심렛의 장점 설명 텍스트
    plt.text(0.5, 2.5, "Notice: Peak positions are preserved accurately.\n(Symmetry of sym5 minimizes phase shift)", 
             ha='center', bbox=dict(facecolor='white', alpha=0.9))
    
    plt.tight_layout()
    plt.show()

sym5_denoising_demo()

In [ ]:
import numpy as np

def naive_polynomial_multiplication(P, Q):
    """
    일반적인 O(N^2) 방식으로 두 다항식의 계수 배열을 곱한다 (합성곱).
    P와 Q는 다항식의 계수 리스트이다. (낮은 차수부터)
    """
    N = len(P)
    M = len(Q)
    # 결과 다항식의 차수는 N + M - 2 이므로, 계수 리스트 길이는 N + M - 1 이다.
    result_len = N + M - 1
    R = [0] * result_len

    for k in range(result_len):
        # r_k = sum(p_i * q_{k-i}) 를 계산한다.
        
        # 합의 시작 인덱스 i_min: max(0, k - M + 1)
        i_min = max(0, k - M + 1)
        
        # 합의 끝 인덱스 i_max: min(k, N - 1)
        i_max = min(k, N - 1)
        
        r_k = 0
        for i in range(i_min, i_max + 1):
            r_k += P[i] * Q[k - i]
        
        R[k] = r_k
        
    return R

def fft_polynomial_multiplication(P, Q):
    """
    FFT를 사용하여 두 다항식의 곱셈을 O(N log N)으로 가속한다.
    """
    N = len(P)
    M = len(Q)
    
    # 1. 패딩 (Padding)
    # FFT를 위한 길이는 N + M - 1 보다 크거나 같은 2의 거듭제곱으로 설정해야 한다.
    L = N + M - 1
    fft_size = 1
    while fft_size < L:
        fft_size *= 2
        
    # 패딩된 계수 배열
    P_padded = np.pad(P, (0, fft_size - N), 'constant')
    Q_padded = np.pad(Q, (0, fft_size - M), 'constant')
    
    # 2. FFT 변환
    Fp = np.fft.fft(P_padded)
    Fq = np.fft.fft(Q_padded)
    
    # 3. 주파수 영역에서 곱셈 (점별 곱셈)
    Fr = Fp * Fq
    
    # 4. 역 FFT (IFFT)를 이용해 시간 영역으로 복원
    R_complex = np.fft.ifft(Fr)
    
    # 결과는 실수여야 하며, 작은 부동 소수점 오차가 발생할 수 있으므로 반올림한다.
    # 합성곱의 계수는 정수이므로, 결과를 실수부만 취하고 반올림하여 정수로 만든다.
    R_coeffs = np.round(R_complex.real).astype(int)
    
    # 최종 결과는 L 길이만큼만 취한다.
    return R_coeffs[:L].tolist()

## --- 계산 및 검증 부분 ---
print("--- FFT 기반 다항식 곱셈 예제 및 검증 ---")

# 다항식 P(x) = 1 + 2x + 3x^2 + 4x^3  (계수: [1, 2, 3, 4])
P_coeffs = [1, 2, 3, 4]
# 다항식 Q(x) = 5 + 6x + 7x^2         (계수: [5, 6, 7])
Q_coeffs = [5, 6, 7]

# 1. 일반 곱셈으로 계산 (검증 기준)
R_naive = naive_polynomial_multiplication(P_coeffs, Q_coeffs)

# 2. FFT 곱셈으로 계산 (검증 대상)
R_fft = fft_polynomial_multiplication(P_coeffs, Q_coeffs)

# 3. 계산 결과 출력
print(f"\n다항식 P(x) 계수 (P): {P_coeffs}")
print(f"다항식 Q(x) 계수 (Q): {Q_coeffs}")
print("-" * 30)

# R(x) = P(x) * Q(x) = 5 + 16x + 38x^2 + 52x^3 + 46x^4 + 28x^5 이다.

print(f"일반 곱셈 (O(N^2)) 결과 계수 (R_naive): {R_naive}")
print(f"FFT 곱셈 (O(N log N)) 결과 계수 (R_fft): {R_fft}")
print("-" * 30)

## 4. 검증
# numpy의 allclose 함수를 사용하여 부동 소수점 오차를 감안하여 비교한다.
# 하지만 이 예제에서는 np.round를 통해 정수 비교가 가능히다.

# 길이가 같고 모든 요소가 같으면 True
is_verified = (R_naive == R_fft)

# 검증 결과 출력
print(f" 검증 결과: 두 계산 결과가 일치하는가? {is_verified}")

if not is_verified:
    # 길이가 다르거나 요소가 다를 경우 디버깅 정보 출력
    print(" 오류: 두 결과가 일치하지 않는다.")
    print(f"차이: {np.array(R_naive) - np.array(R_fft)}")
else:
    print("\nFFT를 사용한 다항식 곱셈이 일반 곱셈 결과와 정확히 일치함을 확인했다.")

# --- 추가 검증: 큰 정수 곱셈 (Carry 처리 미포함) ---
print("\n--- 추가 검증: 큰 정수 곱셈 (올림 처리 전) ---")

# 큰 정수 A = 9998 (계수 [8, 9, 9, 9])
A_coeffs = [8, 9, 9, 9] 
# 큰 정수 B = 1001 (계수 [1, 0, 0, 1])
B_coeffs = [1, 0, 0, 1] 
# A * B = 9998 * 1001 = 10007998

# 나이브 방식
R_int_naive = naive_polynomial_multiplication(A_coeffs, B_coeffs)
# FFT 방식
R_int_fft = fft_polynomial_multiplication(A_coeffs, B_coeffs)

print(f"정수 A 계수: {A_coeffs}")
print(f"정수 B 계수: {B_coeffs}")
print(f"FFT 결과 (올림 전): {R_int_fft}")

# 실제 정답 (10007998)의 계수 배열: [8, 9, 9, 7, 0, 0, 1]
# 여기서 FFT 결과 [8, 9, 9, 17, 9, 9, 8]는 올림 처리를 해야 실제 정답과 일치한다.
# 예시 R_int_fft의 결과를 보면, [8, 9, 9, 17, 9, 9, 8]이 나온다.
# 여기서 17을 올림 처리하면 8, 9, 9, (7+10), 9, 9, 8 -> 8, 9, 9, 7, (9+1), 9, 8 -> ... 순으로 변환된다. 
# 이 과정이 큰 정수 곱셈 알고리즘의 최종 단계이다.
# FFT 자체의 계산은 이 '올림 전의 합성곱'까지 정확하게 수행함을 검증한다.
is_int_verified = (R_int_naive == R_int_fft)
print(f" 검증 결과 (큰 정수): 두 계산 결과가 일치하는가? {is_int_verified}")

In [ ]:
import numpy as np

def naive_polynomial_multiplication(P, Q):
    """ 검증용 일반 곱셈 함수 """
    N = len(P)
    M = len(Q)
    result_len = N + M - 1
    R = [0] * result_len

    for k in range(result_len):
        i_min = max(0, k - M + 1)
        i_max = min(k, N - 1)
        r_k = 0
        for i in range(i_min, i_max + 1):
            r_k += P[i] * Q[k - i]
        R[k] = r_k
    return R

def fft_polynomial_multiplication_verbose(P, Q):
    """
    FFT 중간 단계를 출력하는 다항식 곱셈 함수
    """
    N = len(P)
    M = len(Q)
    
    # 1. 패딩 (Padding)
    L = N + M - 1
    fft_size = 1
    while fft_size < L:
        fft_size *= 2
        
    P_padded = np.pad(P, (0, fft_size - N), 'constant')
    Q_padded = np.pad(Q, (0, fft_size - M), 'constant')
    
    print(f"\n[Step 1] 패딩된 입력 배열 (길이 {fft_size}):")
    print(f"  P (23) -> {P_padded}")
    print(f"  Q (45) -> {Q_padded}")

    # 2. FFT 변환
    Fp = np.fft.fft(P_padded)
    Fq = np.fft.fft(Q_padded)
    
    # --- 요청하신 중간 단계 출력 ---
    print(f"\n[Step 2] FFT 변환 결과 (주파수 영역):")
    # 보기 좋게 복소수를 포맷팅하여 출력
    print("  FFT(P): ", [f"{val:.2f}" for val in Fp])
    print("  FFT(Q): ", [f"{val:.2f}" for val in Fq])
    # ---------------------------

    # 3. 주파수 영역에서 곱셈
    Fr = Fp * Fq
    
    print(f"\n[Step 3] 주파수 영역 곱셈 (Point-wise Multiplication):")
    print("  Fr = FFT(P) * FFT(Q): ", [f"{val:.2f}" for val in Fr])

    # 4. 역 FFT (IFFT)
    R_complex = np.fft.ifft(Fr)
    R_coeffs = np.round(R_complex.real).astype(int)
    
    # 최종 결과는 L 길이만큼만 취함
    return R_coeffs[:L].tolist()

def process_carry(coeffs):
    """
    다항식 계수(합성곱 결과)를 실제 정수 곱셈 결과로 변환 (자리 올림 처리)
    예: [15, 22, 8] -> 1035
    """
    result = []
    carry = 0
    # 리스트 복사 (원본 보존)
    temp_coeffs = coeffs[:]
    
    # 각 자릿수 처리
    for val in temp_coeffs:
        total = val + carry
        digit = total % 10
        carry = total // 10
        result.append(digit)
        
    # 남은 carry 처리
    while carry > 0:
        result.append(carry % 10)
        carry //= 10
        
    # 역순으로 뒤집어서 정수 만들기 (낮은 자릿수가 인덱스 0이므로)
    final_number_str = "".join(map(str, reversed(result)))
    return int(final_number_str)

# --- 메인 실행 ---

# 숫자 23 -> [3, 2] (3*10^0 + 2*10^1)
P_coeffs = [3, 2] 
# 숫자 45 -> [5, 4] (5*10^0 + 4*10^1)
Q_coeffs = [5, 4]

print(f"입력 숫자: 23 (계수 {P_coeffs}) x 45 (계수 {Q_coeffs})")

# FFT 수행 및 중간 결과 출력
R_fft = fft_polynomial_multiplication_verbose(P_coeffs, Q_coeffs)

print(f"\n[Step 4] IFFT 결과 (합성곱 계수): {R_fft}")

# 검증
R_naive = naive_polynomial_multiplication(P_coeffs, Q_coeffs)
print(f"검증용 일반 곱셈 결과: {R_naive}")

# 자리 올림 처리 (최종 정수 계산)
final_result = process_carry(R_fft)
print(f"\n[Step 5] 자리 올림(Carry) 처리 후 최종 결과:")
print(f"  {R_fft} -> {final_result}")
print(f"  실제 계산 검증: 23 * 45 = {23 * 45}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_aliasing():
    # 설정
    fs = 10.0           # 샘플링 주파수 (10 Hz)
    T = 1.0             # 전체 시간 (1초)
    f_input = 9.0       # 입력 신호 주파수 (9 Hz) - 고주파
    
    # 앨리어싱된 주파수 계산: |9 - 10| = 1 Hz
    f_alias = abs(f_input - fs) 

    # 1. 고해상도 시간축 (원래 아날로그 신호를 그리기 위함)
    t_continuous = np.linspace(0, T, 1000)
    
    # 2. 샘플링 시간축 (실제 샘플링되는 시점)
    t_samples = np.linspace(0, T, int(T * fs) + 1)

    # 3. 신호 생성
    # (A) 원래 고주파 신호 (파란색 실선)
    y_input = np.cos(2 * np.pi * f_input * t_continuous)
    
    # (B) 샘플링된 데이터 (빨간색 점)
    y_samples = np.cos(2 * np.pi * f_input * t_samples)
    
    # (C) 앨리어싱된 저주파 신호 (초록색 점선) - 착시 현상
    y_alias = np.cos(2 * np.pi * f_alias * t_continuous)

    # --- 그래프 그리기 ---
    plt.figure(figsize=(10, 6))
    
    # 원래 신호
    plt.plot(t_continuous, y_input, 'b-', alpha=0.3, label=f'Original High Freq ({f_input} Hz)')
    
    # 앨리어싱된 가상의 신호
    plt.plot(t_continuous, y_alias, 'g--', linewidth=2, label=f'Aliased Low Freq ({f_alias} Hz)')
    
    # 실제 샘플 포인트
    plt.plot(t_samples, y_samples, 'ro', markersize=8, label='Sample Points')
    
    # 설명 추가
    plt.title(f"Aliasing Effect: {f_input}Hz signal sampled at {fs}Hz looks like {f_alias}Hz")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.legend(loc='upper right')
    plt.grid(True)
    
    plt.show()

plot_aliasing()

In [ ]:
def sieve_of_eratosthenes(N):
    """
    에라토스테네스의 체를 사용하여 N까지의 모든 소수를 찾는다.

    Args:
        N (int): 소수를 찾을 범위의 상한 (N은 포함되지 않음, N-1까지).
                 일반적으로는 N까지의 소수를 찾는 것이므로,
                 여기서는 N-1까지의 소수를 찾는 것으로 가정한다.
    
    Returns:
        list: N까지의 모든 소수가 담긴 리스트.
    """
    if N <= 1:
        return []

    # 1. 초기화: N+1 크기의 불리언 리스트 생성
    #   - list[i]가 True이면 i는 소수일 가능성이 있음
    #   - list[i]가 False이면 i는 합성수임
    #   - 인덱스를 숫자 자체로 사용하기 위해 크기를 N+1로 설정
    is_prime = [True] * (N + 1)
    # 0과 1은 소수가 아님
    is_prime[0] = is_prime[1] = False
    # 2. 체질 과정 (Sifting)
    #   - 2부터 시작하여 sqrt(N)까지만 반복
    #   - N의 제곱근까지만 확인해도 충분함 (앞서 설명된 원리)
    for p in range(2, int(N**0.5) + 1):
        # p가 소수(True)라면, p의 배수를 모두 지움 (False로 변경)
        if is_prime[p]:
            # p*p부터 시작: p*2, p*3 등은 이미 p보다 작은 소수들(예: 2)에 의해 지워졌을 가능성이 높음
            for i in range(p * p, N + 1, p):
                is_prime[i] = False
    # 3. 결과 리스트 생성
    primes = []
    for num in range(2, N + 1):
        if is_prime[num]:
            primes.append(num)
    return primes

# --- 사용 예시 ---
N = 30
result = sieve_of_eratosthenes(N)
print(f"{N}까지의 모든 소수:")
print(result)
# --- 사용 예시 2 ---
N_large = 100
result_large = sieve_of_eratosthenes(N_large)
print(f"\n{N_large}까지의 모든 소수 ({len(result_large)}개):")
print(result_large)

In [ ]:
import time
import matplotlib.pyplot as plt

def sieve_of_eratosthenes(N):
    """
    에라토스테네스의 체를 사용하여 N까지의 모든 소수를 찾는다.
    (이전 코드와 동일)
    """
    if N <= 1:
        return []
    is_prime = [True] * (N + 1)
    is_prime[0] = is_prime[1] = False
    for p in range(2, int(N**0.5) + 1):
        if is_prime[p]:
            for i in range(p * p, N + 1, p):
                is_prime[i] = False
    primes = []
    for num in range(2, N + 1):
        if is_prime[num]:
            primes.append(num)
    return primes

def measure_and_plot_sieve_time(max_N, step=1000):
    """
    다양한 N 값에 대해 에라토스테네스의 체 알고리즘의 실행 시간을 측정하고 그래프로 표시한다.
    Args:
        max_N (int): 측정할 N 값의 최대 범위.
        step (int): N 값을 증가시킬 간격.
    """
    N_values = []
    times = []
    print("--- 에라토스테네스의 체 성능 측정 시작 ---")
    current_N = 100 # 너무 작은 N 값은 측정 의미가 적으므로 100부터 시작
    while current_N <= max_N:
        start_time = time.perf_counter() # 고해상도 타이머 시작
        sieve_of_eratosthenes(current_N)
        end_time = time.perf_counter()   # 타이머 종료
        elapsed_time = end_time - start_time
        N_values.append(current_N)
        times.append(elapsed_time)
        print(f"N={current_N:<8}: {elapsed_time:.6f} sec")
        current_N += step
    print("--- 성능 측정 완료 ---")
    # 그래프 그리기
    plt.figure(figsize=(10, 6))
    plt.plot(N_values, times, marker='o', linestyle='-', color='b')
    plt.title('Execution time of sieve of Eratosthenes')
    plt.xlabel('N (Upper limit for primes)')
    plt.ylabel('Time (seconds)')
    plt.grid(True)
    plt.show()
    # 생성된 그래프를 이미지로 저장
    # plt.savefig("sieve_of_eratosthenes_performance.png") # 필요한 경우 파일로 저장
# --- 프로그램 실행 ---
# 최대 N 값을 설정한다. 이 값을 높게 설정하면 실행 시간이 오래 걸릴 수 있다.
# 예를 들어, 100,000까지 1,000 간격으로 측정
measure_and_plot_sieve_time(max_N=1000000, step=10000)

In [ ]:
def check_fermat_little_theorem(a, p):
    """
    a^(p-1) mod p == 1 인지 확인하는 함수
    """
    if a % p == 0:
        return "조건 불만족: a는 p의 배수면 안 된다."
    
    # a^(p-1) % p 계산
    result = pow(a, p - 1, p)
    
    print(f"Test: {a}^{p-1} ≡ {result} (mod {p})")
    
    if result == 1:
        return True
    else:
        return False

# 1. 작은 수 테스트
print("--- Small Number Test ---")
check_fermat_little_theorem(3, 7)   # 3^6 mod 7
check_fermat_little_theorem(2, 5)   # 2^4 mod 5

# 2. 아주 큰 소수 테스트 (Mersenne Prime 예시: 2^13 - 1 = 8191)
print("\n--- Large Prime Test ---")
p_large = 8191
a_large = 1234
check_fermat_little_theorem(a_large, p_large)

# 3. 소수가 아닌 경우 (반례)
print("\n--- Composite Number Test (p=15, not prime) ---")
# 2^14 mod 15 -> 결과가 1이 아니면 15는 소수가 아님
check_fermat_little_theorem(2, 15)

In [ ]:
def euler_phi(n):
    """
    오일러 피 함수 (Euler's Totient Function)
    n보다 작고 n과 서로소인 양의 정수의 개수를 반환한다.
    """
    result = n  # 초기값 설정
    p = 2       # 가장 작은 소수부터 시작
    
    # 소인수분해를 진행하며 공식을 적용
    while p * p <= n:
        # p가 n의 소인수인 경우
        if n % p == 0:
            # 공식: result = result * (1 - 1/p)
            # 정수 연산을 위해 아래와 같이 변형: result = result - (result // p)
            result -= result // p
            
            # n에서 해당 소인수 p를 모두 나누어 제거
            while n % p == 0:
                n //= p
        p += 1
        
    # 반복문 종료 후 n이 1보다 크다면, 남은 n은 소수임
    if n > 1:
        result -= result // n
        
    return result

# --- 사용 예제 ---
if __name__ == "__main__":
    test_num = 10
    print(f"phi({test_num}) = {euler_phi(test_num)}") 
    # 예상 결과: 4 (1, 3, 7, 9)

    test_num2 = 31
    print(f"phi({test_num2}) = {euler_phi(test_num2)}")
    # 예상 결과: 30 (31은 소수이므로 31-1)
    
    test_num3 = 99
    print(f"phi({test_num3}) = {euler_phi(test_num3)}")
    # 99 = 3^2 * 11 -> 99 * (1 - 1/3) * (1 - 1/11) = 99 * 2/3 * 10/11 = 60
    for test_num in [1, 2, 3, 4, 5, 6]:
        print(test_num, euler_phi(test_num))
    for test_num in [2, 3, 5, 7, 11, 13]:
        print(test_num, euler_phi(test_num))     
    for test_num in [6, 35]:
        print(test_num, euler_phi(test_num))        

In [ ]:
import hashlib

# ==========================================
# 1. 수학적 도구 (RSA 키 생성을 위한 함수들)
# ==========================================
def extended_gcd(a, b):
    """확장 유클리드 호제법: d(개인키)를 구하기 위함"""
    if a == 0:
        return b, 0, 1
    else:
        g, y, x = extended_gcd(b % a, a)
        return g, x - (b // a) * y, y

def mod_inverse(e, phi):
    """모듈러 역원 구하기: (e * d) % phi = 1 인 d 찾기"""
    g, x, y = extended_gcd(e, phi)
    if g != 1: raise Exception('역원이 존재하지 않는다.')
    return x % phi

# ==========================================
# 2. 키 생성 및 서명/검증 로직
# ==========================================
class MiniRSA:
    def __init__(self):
        # 이해를 돕기 위해 작은 소수 p, q를 고정해서 사용한다.
        p = 61
        q = 53
        
        self.n = p * q                      # Modulo n
        phi = (p - 1) * (q - 1)             # 오일러 피 함수 (3120)
        
        self.e = 17                         # 공개 지수 (공개키의 일부)
        self.d = mod_inverse(self.e, phi)   # 개인 지수 (개인키의 일부, 비밀!)
        
        print(f"[초기화] 공개키(e, n): ({self.e}, {self.n})")
        print(f"[초기화] 개인키(d, n): ({self.d}, {self.n}) <- 철수만 알고 있음\n")

    def get_hash_int(self, message):
        """메시지를 해시(SHA-256)한 후 숫자로 변환"""
        # 실제 RSA는 긴 문서를 직접 암호화하지 않고 해시값(요약본)을 서명한다.
        msg_bytes = message.encode('utf-8')
        hex_digest = hashlib.sha256(msg_bytes).hexdigest()
        # 계산을 위해 16진수를 10진수 정수로 변환 (간략화된 방식)
        return int(hex_digest, 16) % self.n 

    def sign(self, message):
        """[서명 생성] 송신자(철수)가 자신의 '개인키'로 암호화"""
        m_hash = self.get_hash_int(message)
        
        # 핵심 공식: Signature = (Hash)^d mod n
        signature = pow(m_hash, self.d, self.n)
        
        print(f"--- [서명 생성] ---")
        print(f"원본 메시지: '{message}'")
        print(f"메시지 해시값(숫자화): {m_hash}")
        print(f"생성된 전자서명: {signature} (개인키 {self.d}로 잠금)")
        return signature

    def verify(self, message, signature):
        """[서명 검증] 수신자(영희)가 철수의 '공개키'로 검증"""
        print(f"\n--- [서명 검증] ---")
        
        # 1. 받은 메시지의 해시값 계산 (내가 직접 요약해보기)
        expected_hash = self.get_hash_int(message)
        
        # 2. 서명 복호화: Decrypted = (Signature)^e mod n
        decrypted_hash = pow(signature, self.e, self.n)
        
        print(f"받은 메시지의 해시값: {expected_hash}")
        print(f"서명을 공개키({self.e})로 푼 값: {decrypted_hash}")
        
        # 3. 비교
        if expected_hash == decrypted_hash:
            print(">> [검증 성공] 철수가 보낸 것이 맞다. (부인 불가)")
            return True
        else:
            print(">> [검증 실패] 서명이 위조되었거나 메시지가 변조되었다!")
            return False

# ==========================================
# 3. 실행 시나리오
# ==========================================
if __name__ == "__main__":
    rsa = MiniRSA()
    
    # 시나리오 1: 정상적인 서명 및 전송
    contract = "계약서: 철수는 영희에게 100만원을 준다."
    
    # 철수가 서명함
    digital_signature = rsa.sign(contract)
    
    # 영희가 검증함
    rsa.verify(contract, digital_signature)

    # ---------------------------------------------------------
    
    # 시나리오 2: 해커가 메시지를 조작함 (데이터 무결성 훼손)
    print("\n\n[!!!] 해커가 전송 중에 금액을 '0원'으로 조작함")
    fake_contract = "계약서: 철수는 영희에게 0원을 준다."
    
    # 영희는 조작된 메시지와 원래 서명을 가지고 검증을 시도
    rsa.verify(fake_contract, digital_signature)

In [ ]:
def verify_general_fermat(a, p):
    """
    페르마의 소정리 일반형: a^p ≡ a (mod p) 검증
    a가 p의 배수여도 성립함을 보여줌.
    """
    # 1. 좌변 계산: a^p mod p
    # pow(base, exp, mod) 함수는 거듭제곱의 나머지 연산을 효율적으로 수행한다.
    lhs = pow(a, p, p)
    
    # 2. 우변 계산: a mod p
    rhs = a % p
    
    # 3. 결과 출력
    print(f"--- 검증: a={a}, p={p} ---")
    print(f"좌변 (a^p mod p): {a}^{p} ≡ {lhs} (mod {p})")
    print(f"우변 (a   mod p): {a} ≡ {rhs} (mod {p})")
    
    if lhs == rhs:
        print(f">>> 결과: 성립함! ({lhs} == {rhs})")
        if a % p == 0:
            print("    (특이사항: a는 p의 배수이다. 양변이 모두 0이 된다.)")
    else:
        print(">>> 결과: 성립하지 않음 (p가 소수가 아니거나 오류 발생)")
    print()

# 테스트 실행

# Case 1: a가 p의 배수인 경우 (사용자 요청 사항)
# p=7, a=14 (14는 7의 배수)
print("[Case 1] a가 p의 배수인 경우")
verify_general_fermat(a=14, p=7)

# Case 2: a가 p의 배수인 경우 (0인 경우)
# p=5, a=0
print("[Case 2] a가 0인 경우")
verify_general_fermat(a=0, p=5)

# Case 3: a가 p의 배수가 아닌 경우 (일반적인 경우)
# p=7, a=3
print("[Case 3] a가 p의 배수가 아닌 경우 (서로소)")
verify_general_fermat(a=3, p=7)

# Case 4: 아주 큰 수 테스트
# p=17, a=34 (17의 배수)
print("[Case 4] 큰 수에서 배수 관계")
verify_general_fermat(a=34, p=17)

In [ ]:
import math

def phi_naive(n):
    """
    정의 그대로 구현: 1부터 n까지 gcd가 1인 개수를 셈
    시간복잡도: O(N log N)
    """
    count = 0
    coprimes = []
    
    for i in range(1, n + 1):
        if math.gcd(n, i) == 1: # 최대공약수가 1이면 서로소
            count += 1
            coprimes.append(i)
            
    return count, coprimes

# 테스트: n = 10일 때
# 1, 3, 7, 9 가 10과 서로소임 -> 답은 4
n = 10
count, numbers = phi_naive(n)
print(f"phi({n}) = {count}")
print(f"서로소인 수들: {numbers}")

In [ ]:
def phi_optimized(n):
    """
    오일러 곱 공식 이용: 소인수분해를 통해 계산
    시간복잡도: O(sqrt(N))
    """
    result = n  # 초기값 n
    
    # 2부터 sqrt(n)까지의 소수로 나누어 봄
    p = 2
    while p * p <= n:
        if n % p == 0:
            # p가 n의 소인수라면 공식을 적용: result = result * (1 - 1/p)
            # 정수 연산을 위해: result = result - (result // p)
            while n % p == 0:
                n //= p
            result -= result // p
        p += 1
        
    # 반복문 종료 후 n이 1보다 크면, 남은 n은 소수임 (가장 큰 소인수)
    if n > 1:
        result -= result // n
        
    return result

# 테스트
print("-" * 30)
test_numbers = [10, 13, 99, 100]
for num in test_numbers:
    print(f"phi({num}) = {phi_optimized(num)}")
    
# 소수 p에 대해 phi(p) = p-1 임을 확인
print(f"phi(7) = {phi_optimized(7)} (소수 7)")

In [ ]:
import math

def get_phi_table(limit):
    print(f"{'n':<5} | {'phi(n)':<8} | {'서로소 목록'}")
    print("-" * 40)
    
    for n in range(1, limit + 1):
        # 서로소 목록 구하기 (시각화용)
        coprimes = [str(i) for i in range(1, n + 1) if math.gcd(n, i) == 1]
        count = len(coprimes)
        
        # 목록이 너무 길면 줄임표 처리
        coprimes_str = ", ".join(coprimes)
        if len(coprimes_str) > 20:
            coprimes_str = coprimes_str[:20] + "..."
            
        print(f"{n:<5} | {count:<8} | {coprimes_str}")

# 실행
get_phi_table(15)

In [ ]:
import math
import random

# 최대공약수(GCD) 계산: math.gcd 사용으로 최적화
def gcd_optimized(a, b):
    return math.gcd(a, b)
# 모듈러 역원(Modular Inverse) 계산: pow(a, -1, m) 사용으로 최적화
def mod_inverse_optimized(a, m):
    # a와 m이 서로소인지 확인 (RSA 요건)
    if gcd_optimized(a, m) != 1:
        raise ValueError(f"{a}는 {m}과 서로소가 아니다. 모듈러 역원이 존재하지 않는다.")    
    # pow(a, -1, m)은 d * a ≡ 1 (mod m)인 d를 반환
    return pow(a, -1, m)
# 소수 판별 (간단 버전 유지)
def is_prime(num):
    if num < 2:
        return False
    # 2부터 num의 제곱근까지만 확인
    for i in range(2, int(num**0.5) + 1):
        if num % i == 0:
            return False
    return True
##  RSA 키 생성
def generate_key_pair_optimized(p, q):
    if not (is_prime(p) and is_prime(q)):
        raise ValueError('p와 q는 소수여야 한다.')
    if p == q:
        raise ValueError('p와 q는 달라야 한다.')
    n = p * q
    phi = (p - 1) * (q - 1)
    # 공개 지수 e 선택 (일반적으로 사용되는 65537을 먼저 시도)
    e = 65537
    while gcd_optimized(e, phi) != 1:
        # 65537이 안되면 phi보다 작은 홀수 중 랜덤 선택
        e = random.randrange(3, phi, 2) 
    # 비밀 지수 d 계산: 최적화된 mod_inverse 사용
    d = mod_inverse_optimized(e, phi)
    # 공개 키: (e, n), 비밀 키: (d, n)
    return ((e, n), (d, n))
##  암호화 (M^e mod n)
def encrypt_optimized(public_key, plaintext):
    e, n = public_key
    # pow(ord(char), e, n)으로 Modular Exponentiation 최적화
    cipher = [pow(ord(char), e, n) for char in plaintext]
    return cipher
## 복호화 (C^d mod n)
def decrypt_optimized(private_key, ciphertext):
    d, n = private_key
    # pow(char, d, n)으로 Modular Exponentiation 최적화
    plain = [chr(pow(char, d, n)) for char in ciphertext]
    return ''.join(plain)
# --- 실행 예제 ---
P = 61
Q = 53
message = "OPTIMIZED RSA message"
public_key_opt, private_key_opt = generate_key_pair_optimized(P, Q)
encrypted_msg_opt = encrypt_optimized(public_key_opt, message)
decrypted_msg_opt = decrypt_optimized(private_key_opt, encrypted_msg_opt)
# --- 결과 출력 ---
print(f"선택된 소수 P: {P}, Q: {Q}")
print(f"공개 키 (e, n): {public_key_opt}")
print(f"비밀 키 (d, n): {private_key_opt}")
print(f"원문 메시지: {message}")
print(f"암호화된 메시지 (일부): {encrypted_msg_opt[:5]}...")
print(f"암호화된 메시지 (전부): {encrypted_msg_opt[:]}")
print(f"복호화된 메시지: {decrypted_msg_opt}")

In [ ]:
import random

# 최대공약수(GCD) 계산
def gcd(a, b):
    while b != 0:
        a, b = b, a % b
    return a

# 모듈러 역원(Modular Inverse) 계산 (확장 유클리드 호제법)
def mod_inverse(a, m):
    m0, x0, x1 = m, 0, 1
    while a > 1:
        q = a // m
        m, a = a % m, m
        x0, x1 = x1 - q * x0, x0
    if x1 < 0:
        x1 += m0
    return x1

# 소수 판별 (간단 버전)
def is_prime(num):
    if num < 2:
        return False
    for i in range(2, int(num**0.5) + 1):
        if num % i == 0:
            return False
    return True

# RSA 키 생성
def generate_key_pair(p, q):
    if not (is_prime(p) and is_prime(q)):
        raise ValueError('p와 q는 소수여야 한다.')
    if p == q:
        raise ValueError('p와 q는 달라야 한다.')

    # 1. 모듈러스 (Modulus) N = p * q
    n = p * q

    # 2. 오일러 피 함수 (Euler's Totient function) phi(N) = (p-1) * (q-1)
    phi = (p - 1) * (q - 1)

    # 3. 공개 지수 (Public Exponent) e 선택 (1 < e < phi 이고, gcd(e, phi) = 1)
    # 일반적으로 65537이 사용되지만, 예제에서는 작은 값 선택
    e = 65537 
    while gcd(e, phi) != 1:
        e = random.randrange(3, phi)

    # 4. 비밀 지수 (Private Exponent) d 계산 (d * e ≡ 1 (mod phi))
    d = mod_inverse(e, phi)

    # 공개 키: (e, n), 비밀 키: (d, n)
    return ((e, n), (d, n))

# 암호화
def encrypt(public_key, plaintext):
    e, n = public_key
    # 암호문 C = M^e mod n
    # 텍스트를 숫자로 변환 (예: 아스키 코드)
    cipher = [pow(ord(char), e, n) for char in plaintext]
    return cipher

# 복호화
def decrypt(private_key, ciphertext):
    d, n = private_key
    # 평문 M = C^d mod n
    # 숫자를 텍스트로 변환
    plain = [chr(pow(char, d, n)) for char in ciphertext]
    return ''.join(plain)
# --- 실행 ---

# 1. p와 q, 두 개의 소수 선택 (실제로는 수백 자리의 매우 큰 소수를 사용해야 안전함)
P = 61
Q = 53

print(f"선택된 소수 P: {P}, Q: {Q}\n")

# 2. 키 쌍 생성
public_key, private_key = generate_key_pair(P, Q)

print(f"공개 키 (e, n): {public_key}")
print(f"비밀 키 (d, n): {private_key}\n")

# 3. 암호화할 메시지
message = "HELLO RSA"
print(f"원문 메시지: {message}")

# 4. 메시지 암호화 (공개 키 사용)
encrypted_msg = encrypt(public_key, message)
print(f"암호화된 메시지 (숫자 리스트): {encrypted_msg}\n")

# 5. 메시지 복호화 (비밀 키 사용)
decrypted_msg = decrypt(private_key, encrypted_msg)
print(f"복호화된 메시지: {decrypted_msg}")    

In [ ]:
import math
import random

# 최대공약수(GCD) 계산: math.gcd 사용으로 최적화
def gcd_optimized(a, b):
    return math.gcd(a, b)
# 모듈러 역원(Modular Inverse) 계산: pow(a, -1, m) 사용으로 최적화
def mod_inverse_optimized(a, m):
    # a와 m이 서로소인지 확인 (RSA 요건)
    if gcd_optimized(a, m) != 1:
        raise ValueError(f"{a}는 {m}과 서로소가 아니다. 모듈러 역원이 존재하지 않는다.")    
    # pow(a, -1, m)은 d * a ≡ 1 (mod m)인 d를 반환
    return pow(a, -1, m)
# 소수 판별 (간단 버전 유지)
def is_prime(num):
    if num < 2:
        return False
    # 2부터 num의 제곱근까지만 확인
    for i in range(2, int(num**0.5) + 1):
        if num % i == 0:
            return False
    return True
##  RSA 키 생성
def generate_key_pair_optimized(p, q):
    if not (is_prime(p) and is_prime(q)):
        raise ValueError('p와 q는 소수여야 한다.')
    if p == q:
        raise ValueError('p와 q는 달라야 한다.')
    n = p * q
    phi = (p - 1) * (q - 1)
    # 공개 지수 e 선택 (일반적으로 사용되는 65537을 먼저 시도)
    e = 65537
    while gcd_optimized(e, phi) != 1:
        # 65537이 안되면 phi보다 작은 홀수 중 랜덤 선택
        e = random.randrange(3, phi, 2) 
    # 비밀 지수 d 계산: 최적화된 mod_inverse 사용
    d = mod_inverse_optimized(e, phi)
    # 공개 키: (e, n), 비밀 키: (d, n)
    return ((e, n), (d, n))
##  암호화 (M^e mod n)
def encrypt_optimized(public_key, plaintext):
    e, n = public_key
    # pow(ord(char), e, n)으로 Modular Exponentiation 최적화
    cipher = [pow(ord(char), e, n) for char in plaintext]
    return cipher
## 복호화 (C^d mod n)
def decrypt_optimized(private_key, ciphertext):
    d, n = private_key
    # pow(char, d, n)으로 Modular Exponentiation 최적화
    plain = [chr(pow(char, d, n)) for char in ciphertext]
    return ''.join(plain)
# --- 실행 예제 ---
P = 61
Q = 53
message = "OPTIMIZED RSA message"
public_key_opt, private_key_opt = generate_key_pair_optimized(P, Q)
encrypted_msg_opt = encrypt_optimized(public_key_opt, message)
decrypted_msg_opt = decrypt_optimized(private_key_opt, encrypted_msg_opt)
# --- 결과 출력 ---
print(f"선택된 소수 P: {P}, Q: {Q}")
print(f"공개 키 (e, n): {public_key_opt}")
print(f"비밀 키 (d, n): {private_key_opt}")
print(f"원문 메시지: {message}")
print(f"암호화된 메시지 (일부): {encrypted_msg_opt[:5]}...")
print(f"암호화된 메시지 (전부): {encrypted_msg_opt[:]}")
print(f"복호화된 메시지: {decrypted_msg_opt}")

In [ ]:
import math
import random

# 최대공약수(GCD) 계산: math.gcd 사용으로 최적화
def gcd_optimized(a, b):
    return math.gcd(a, b)
# 모듈러 역원(Modular Inverse) 계산: pow(a, -1, m) 사용으로 최적화
def mod_inverse_optimized(a, m):
    # a와 m이 서로소인지 확인 (RSA 요건)
    if gcd_optimized(a, m) != 1:
        raise ValueError(f"{a}는 {m}과 서로소가 아니다. 모듈러 역원이 존재하지않는다.")    
    # pow(a, -1, m)은 d * a ≡ 1 (mod m)인 d를 반환
    return pow(a, -1, m)
# 소수 판별 (간단 버전 유지)
def is_prime(num):
    if num < 2:
        return False
    # 2부터 num의 제곱근까지만 확인
    for i in range(2, int(num**0.5) + 1):
        if num % i == 0:
            return False
    return True
##  RSA 키 생성
def generate_key_pair_optimized(p, q):
    if not (is_prime(p) and is_prime(q)):
        raise ValueError('p와 q는 소수여야 한다.')
    if p == q:
        raise ValueError('p와 q는 달라야 한다.')
    n = p * q
    phi = (p - 1) * (q - 1)
    # 공개 지수 e 선택 (일반적으로 사용되는 65537을 먼저 시도)
    e = 65537
    while gcd_optimized(e, phi) != 1:
        # 65537이 안되면 phi보다 작은 홀수 중 랜덤 선택
        e = random.randrange(3, phi, 2) 
    # 비밀 지수 d 계산: 최적화된 mod_inverse 사용
    d = mod_inverse_optimized(e, phi)
    # 공개 키: (e, n), 비밀 키: (d, n)
    return ((e, n), (d, n))
##  암호화 (M^e mod n)
def encrypt_optimized(public_key, plaintext):
    e, n = public_key
    # pow(ord(char), e, n)으로 Modular Exponentiation 최적화
    cipher = [pow(ord(char), e, n) for char in plaintext]
    return cipher
##  복호화 (C^d mod n)
def decrypt_optimized(private_key, ciphertext):
    d, n = private_key
    # pow(char, d, n)으로 Modular Exponentiation 최적화
    plain = [chr(pow(char, d, n)) for char in ciphertext]
    return ''.join(plain)
# --- 실행 예제 ---
P = 911
Q = 857
P= 7907
Q= 8011
message = "OPTIMIZED RSA message"
public_key_opt, private_key_opt = generate_key_pair_optimized(P, Q)
encrypted_msg_opt = encrypt_optimized(public_key_opt, message)
decrypted_msg_opt = decrypt_optimized(private_key_opt, encrypted_msg_opt)
# --- 결과 출력 ---
print(f"선택된 소수 P: {P}, Q: {Q}")
print(f"공개 키 (e, n): {public_key_opt}")
print(f"비밀 키 (d, n): {private_key_opt}")
print(f"원문 메시지: {message}")
print(f"암호화된 메시지 (일부): {encrypted_msg_opt[:5]}...")
print(f"암호화된 메시지 (전부): {encrypted_msg_opt[:]}")
print(f"복호화된 메시지: {decrypted_msg_opt}")

In [ ]:
# --- 기존 필수 함수 (생략) ---

def extended_gcd(a, b):
    """확장 유클리드 호제법"""
    if a == 0:
        return b, 0, 1
    gcd, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    return gcd, x, y

def mod_inverse(a, m):
    """모듈로 역수 (Modular Inverse)"""
    gcd, x, y = extended_gcd(a, m)
    if gcd != 1:
        raise Exception('Modular inverse does not exist')
    return x % m

def point_addition(P, Q, a, p):
    """타원 곡선 점 덧셈 (P + Q)"""
    x_P, y_P = P
    x_Q, y_Q = Q

    if P == ('O', 'O'): return Q
    if Q == ('O', 'O'): return P
    
    if x_P == x_Q and y_P != y_Q:
        return ('O', 'O')

    if P == Q:
        # 점 2배 (Point Doubling): λ = (3x_P^2 + a) * (2y_P)^-1 (mod p)
        numerator = (3 * x_P**2 + a) % p
        denominator_inv = mod_inverse(2 * y_P, p)
        lambda_val = (numerator * denominator_inv) % p
    else:
        # 일반 덧셈: λ = (y_Q - y_P) * (x_Q - x_P)^-1 (mod p)
        numerator = (y_Q - y_P) % p
        denominator_inv = mod_inverse((x_Q - x_P) % p, p)
        lambda_val = (numerator * denominator_inv) % p
    
    # R = P + Q의 좌표 계산
    x_R = (lambda_val**2 - x_P - x_Q) % p
    y_R = (lambda_val * (x_P - x_R) - y_P) % p
    
    return (x_R, y_R)

# --- 핵심: 스칼라 곱셈 함수 (Double-and-Add) ---
def scalar_multiplication(k, G, a, p):
    """
    스칼라 k와 기본점 G를 사용하여 k * G를 계산한다 (더블 앤 애드 알고리즘 사용).
    """
    if k == 0:
        return ('O', 'O') # k=0이면 무한 원점 반환
    if k < 0:
        # 음수 스칼라를 다루는 것은 복잡하므로, 여기서는 양수만 가정
        # 실제 구현에서는 -k * (-G)로 변환 필요
        raise ValueError("Scalar k must be a non-negative integer for this simplified function.")
    
    result = ('O', 'O') # 최종 결과 (무한 원점 O로 초기화)
    current_P = G       # 현재 더할 점 (G로 초기화)

    # 더블 앤 애드 알고리즘 시작
    while k > 0:
        # 1. 'Add' 단계: k의 최하위 비트가 1이면, result에 current_P를 더한다.
        if k & 1:
            result = point_addition(result, current_P, a, p)
        
        # 2. 'Double' 단계: current_P를 두 배로 만든다. (다음 비트로 이동)
        current_P = point_addition(current_P, current_P, a, p)
        
        # k를 오른쪽으로 1비트 시프트
        k >>= 1
        
    return result

# --- 실행 예시 ---

# 곡선 정의: E: y^2 ≡ x^3 + 2x + 3 (mod 17)
A = 2
P = 17

# 1. 기본점 G 설정 (곡선 위의 점이어야 함)
# G = (5, 1)을 기본점으로 사용
G = (5, 1)

# 2. 개인 키 k (스칼라) 설정
K_A = 7 # 앨리스의 개인 키

# 3. 스칼라 곱셈 (공개 키 계산)
P_A = scalar_multiplication(K_A, G, A, P)

print(f"**타원 곡선 E: y^2 ≡ x^3 + {A}x + 3 (mod {P})**")
print("-" * 30)
print(f"기본점 G: {G}")
print(f"개인 키 k_A: {K_A}")
print("-" * 30)
print(f"**공개 키 P_A = {K_A} * G**")

# 중간 계산 과정 시뮬레이션
# 1*G = (5, 1)
R_1 = scalar_multiplication(1, G, A, P)
# 2*G = 1*G + 1*G = (5, 1) + (5, 1) = (13, 11)
R_2 = scalar_multiplication(2, G, A, P)
# 3*G = 2*G + 1*G = (13, 11) + (5, 1) = (15, 15)
R_3 = scalar_multiplication(3, G, A, P)
# 4*G = 2*G + 2*G = (13, 11) + (13, 11) = (14, 16)
R_4 = scalar_multiplication(4, G, A, P)
# 7*G = 4*G + 3*G = (14, 16) + (15, 15) = (1, 3) (실제 계산해 보면 됨)
# 7*G 결과 확인:
print(f"\n1 * G: {R_1}")
print(f"2 * G: {R_2}")
print(f"3 * G: {R_3}")
print(f"4 * G: {R_4}")
print(f"7 * G (최종 결과): {P_A}")

print("-" * 30)

import numpy as np
import matplotlib.pyplot as plt

def plot_elliptic_curve(A, B, x_range=(-5, 5)):
    """
    타원 곡선 y^2 = x^3 + Ax + B 를 그린다.
    """
    # 1. x^3 + Ax + B 계산 (우변)
    x = np.linspace(x_range[0], x_range[1], 400)
    y_squared = x**3 + A*x + B

    # 2. 실수 해를 가지는 부분만 필터링 (y^2 >= 0)
    # y^2이 음수인 부분은 실수 해가 존재하지 않는다.
    x_real = x[y_squared >= 0]
    y_squared_real = y_squared[y_squared >= 0]

    # 3. y 값을 계산 (양수와 음수 두 가지 해)
    y_plus = np.sqrt(y_squared_real)
    y_minus = -np.sqrt(y_squared_real)

    # 4. 그래프 설정
    plt.figure(figsize=(8, 6))
    
    # 5. 곡선 그리기 (위쪽과 아래쪽 부분)
    plt.plot(x_real, y_plus, color='blue', label=f'$y^2 = x^3 + {A}x + {B}$')
    plt.plot(x_real, y_minus, color='blue')

    # 6. 특이점 확인 및 표시
    # 판별식 Delta = 4A^3 + 27B^2
    discriminant = 4 * A**3 + 27 * B**2
    if discriminant == 0:
        plt.title(f"(Singular curve), $\Delta = 0$")
    elif discriminant > 0:
        plt.title(f"(Elliptic curve), $\Delta > 0$")
    else: # discriminant < 0
        plt.title(f"(Elliptic curve), $\Delta < 0$")
        
    plt.axhline(0, color='gray', linestyle='--')
    plt.axvline(0, color='gray', linestyle='--')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.ylim(-5, 5) # y축 범위 고정
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()
    plt.show()

# --- 실행 예시 ---

## 예시 1: 표준적인 타원 곡선 (A=0, B=1)
# 4A^3 + 27B^2 = 27 (특이점 없음)
plot_elliptic_curve(A=0, B=1)

## 예시 2: ECC에서 사용된 형태와 유사한 곡선 (A=2, B=2)
# 4A^3 + 27B^2 = 4(8) + 27(4) = 32 + 108 = 140 (특이점 없음)
# x_range를 [-3, 3]으로 좁혀서 봅시다.
plot_elliptic_curve(A=2, B=2, x_range=(-3, 3))

## 예시 3: 특이점을 가지는 곡선 (A=-3, B=2)
# 4A^3 + 27B^2 = 4(-27) + 27(4) = -108 + 108 = 0 (첨점(Cusp)을 가진다)
# 이 곡선은 군 구조를 형성하지 못한다.
# plot_elliptic_curve(A=-3, B=2, x_range=(-3, 3))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 곡선 매개변수와 유한체 P
P = 17
A = 2
B = 2

# 유효한 점들을 저장할 리스트
points = []

# 가능한 모든 x (0부터 P-1)에 대해 반복
for x in range(P):
    # 우변 (RHS) 계산: x^3 + Ax + B mod P
    rhs = (x**3 + A*x + B) % P
    # 좌변 (LHS) 계산: y^2 = rhs를 만족하는 y 찾기
    y_values = []
    for y in range(P):
        if (y**2) % P == rhs:
            y_values.append(y)
    # 유효한 y 값이 존재하면 점 리스트에 추가
    for y in y_values:
        points.append((x, y))
# 무한 원점 (O)을 포함하여 총 점의 개수
num_points = len(points) + 1 # +1 for O
# --- 시각화 ---
x_coords = [p[0] for p in points]
y_coords = [p[1] for p in points]
plt.figure(figsize=(7, 7))
plt.scatter(x_coords, y_coords, color='blue', marker='o', s=50) # 점들을 표시
plt.title(f'$E: y^2 \\equiv x^3 + 2x + 2 \  mod{{\ 17}}$')
#plt.title(f'E: y^2 = x^3 + 2x + 2 (mod 17)')
plt.xlabel('x')
plt.ylabel('y')
# x, y 축의 눈금 설정 (0부터 P-1)
plt.xticks(np.arange(0, P, 1))
plt.yticks(np.arange(0, P, 1))
# 격자 설정
plt.grid(True, linestyle=':', alpha=0.7)
# 전체 점의 개수 표시
plt.text(0.5, 1.05, f'Total points(N) = {num_points}', 
         transform=plt.gca().transAxes, fontsize=12, color='red', ha='center')
plt.xlim(-0.5, P - 0.5)
plt.ylim(-0.5, P - 0.5)
plt.gca().set_aspect('equal', adjustable='box') # 종횡비 유지
plt.savefig('eec17.png')
plt.show()

In [ ]:
import math

# 타원 곡선: y^2 = x^3 + ax + b (mod p)의 '실수' 버전 시뮬레이션
# (실제 ECC는 유한체 위에서 모듈러 연산을 사용한다.)

# 곡선 정의: a=2, b=3. (y^2 = x^3 + 2x + 3)
A = 2
B = 3

# ----------------------------------------------------------------------
# 1. 점 덧셈 함수 (Point Addition: P + Q = R)
# ----------------------------------------------------------------------
def point_addition(P, Q):
    """
    타원 곡선 위의 두 점 P와 Q를 더하는 함수.
    P와 Q가 동일한 경우 (P+P)와 P와 Q가 다른 경우 (P+Q)를 처리한다.
    """
    if P is None:  # 무한원점 O
        return Q
    if Q is None:
        return P

    x1, y1 = P
    x2, y2 = Q

    # P와 Q가 y축 대칭인 경우 (P + (-P) = O):
    if x1 == x2 and y1 == -y2:
        return None  # 무한원점 O 반환

    # 1-1. P와 Q가 다른 경우 (P != Q)
    if x1 != x2:
        # 기울기 (slope, m) 계산: m = (y2 - y1) / (x2 - x1)
        m = (y2 - y1) / (x2 - x1)
    
    # 1-2. P와 Q가 같은 경우 (P == Q), 접선의 기울기 사용
    else:  # x1 == x2, y1 == y2 (P와 Q가 동일한 점)
        # 기울기 (m) 계산: m = (3*x1^2 + A) / (2*y1) (미분 공식)
        m = (3 * x1**2 + A) / (2 * y1)

    # 새로운 점 R(x3, y3) 계산
    # x3 = m^2 - x1 - x2
    x3 = m**2 - x1 - x2
    
    # y3 = m(x1 - x3) - y1
    y3 = m * (x1 - x3) - y1

    return (x3, y3)


# ----------------------------------------------------------------------
# 2. 스칼라 곱셈 함수 (Scalar Multiplication: k * G = P)
# ----------------------------------------------------------------------
def scalar_multiplication(G, k):
    """
    생성점 G를 k번 더하는 함수. ECC에서 공개키 P를 계산하는 과정이다.
    G: 생성점 (Generator Point)
    k: 스칼라 (개인 키)
    """
    P = None  # 결과점 P (초기값: 무한원점 O)
    current = G  # 현재 더할 점 (G, 2G, 4G, 8G, ...)
    
    # '이진법을 이용한 거듭제곱' 알고리즘을 사용해 효율적으로 계산
    while k > 0:
        # k의 최하위 비트가 1이면 현재 점을 결과에 더한다.
        if k & 1:
            P = point_addition(P, current)
        
        # current 점을 두 배 한다 (2G, 4G, 8G, ...)
        current = point_addition(current, current)
        
        # k를 오른쪽으로 1비트 이동 (2로 나누기)
        k >>= 1 
        
    return P


# ----------------------------------------------------------------------
# 3. ECC 원리 시뮬레이션 실행
# ----------------------------------------------------------------------

# (1) 생성점 G (Generator Point): 공개적으로 알려진 곡선 위의 점
# 이 점은 곡선 y^2 = x^3 + 2x + 3을 만족해야 한다.
# 3^2 = 3^3 + 2*3 + 3  =>  9 = 27 + 6 + 3 = 36 (만족하지 않음, 실수 곡선은 근사값 사용)
# (x=1, y=2.449)는 예시를 위해 단순화
# 실수에서는 x=1, y=2.449... 이지만, 여기서는 계산이 가능한 정수 근처의 점을 가정
# 실제 유한체 위에서는 계산이 항상 정확하다.
G_x, G_y = 1.0, 2.449489742783178 # G (1.0, 약 2.45)
G = (G_x, G_y)

# (2) 개인 키 k (Private Key): 사용자만 아는 매우 큰 정수
# 실제로는 256비트 이상의 정수를 사용하지만, 여기서는 작은 정수를 사용
k = 7 

print(f"** ECC 원리 시뮬레이션 **")
print("-" * 30)
print(f"타원 곡선: y^2 = x^3 + {A}x + {B}")
print(f"생성점 G: ({G[0]:.4f}, {G[1]:.4f})")
print(f"개인 키 k: {k} (랜덤하게 선택된 정수)")
print("-" * 30)

# (3) 공개 키 P 계산: P = k * G
# 이 과정이 "스칼라 곱셈"이며, 개인 키(k)를 이용하여 공개 키(P)를 만든다.
Public_Key_P = scalar_multiplication(G, k)

print(f"공개 키 P (P = k * G): ({Public_Key_P[0]:.4f}, {Public_Key_P[1]:.4f})")
print("-" * 30)

# (4) ECDLP의 난이도 (원리의 핵심)
print("### ECDLP (이산 로그 문제)의 어려움 ###")
print(f"P = ({Public_Key_P[0]:.4f}, {Public_Key_P[1]:.4f})")
print(f"G = ({G[0]:.4f}, {G[1]:.4f})")
print("이 두 점(P와 G)만 가지고 곱해진 정수 k=7을 찾는 것은")
print("실제 암호 환경에서는 매우 큰 소수 위에서 정의되어 거의 불가능하다.")

def ecc_points_on_finite_field(a, b, p):
    """
    유한체 F_p 위에서 정의된 타원 곡선 y^2 = x^3 + ax + b (mod p) 위의 모든 점을 찾는다.
    """
    points = []
    # 무한 원점 O (Point at Infinity)
    points.append(('O', 'O'))

    # x = 0 부터 p-1 까지 반복
    for x in range(p):
        # 1. 우변 계산 (RHS: Right Hand Side)
        rhs = (x**3 + a*x + b) % p
        
        # 2. 좌변 y^2의 가능한 값 찾기 (LHS: Left Hand Side)
        # 즉, rhs ≡ y^2 (mod p)를 만족하는 y 값을 찾음
        for y in range(p):
            if (y**2) % p == rhs:
                # 점 (x, y) 추가
                points.append((x, y))

    return points

# 예시: 타원 곡선 E: y^2 ≡ x^3 + 2x + 3 (mod 17)
# a = 2, b = 3, p = 17
A = 2
B = 3
P = 17

# 곡선 위의 점 찾기
curve_points = ecc_points_on_finite_field(A, B, P)

## 결과 출력
print(f"**타원 곡선 E: y^2 ≡ x^3 + {A}x + {B} (mod {P})**\n")
print(f"총 점의 개수: {len(curve_points)}")
print("\n**곡선 위의 점들 (x, y):**")
# 5개씩 줄바꿈하여 출력
for i in range(0, len(curve_points), 5):
    print(curve_points[i:i+5])

def ecc_points_on_finite_field(a, b, p):
    """
    유한체 F_p 위에서 정의된 타원 곡선 y^2 = x^3 + ax + b (mod p) 위의 모든 점을 찾는다.
    """
    points = []
    # 무한 원점 O (Point at Infinity)
    points.append(('O', 'O'))

    # x = 0 부터 p-1 까지 반복
    for x in range(p):
        # 1. 우변 계산 (RHS: Right Hand Side)
        rhs = (x**3 + a*x + b) % p
        
        # 2. 좌변 y^2의 가능한 값 찾기 (LHS: Left Hand Side)
        # 즉, rhs ≡ y^2 (mod p)를 만족하는 y 값을 찾음
        for y in range(p):
            if (y**2) % p == rhs:
                # 점 (x, y) 추가
                points.append((x, y))

    return points

# 예시: 타원 곡선 E: y^2 ≡ x^3 + 2x + 3 (mod 17)
# a = 2, b = 2, p = 17
A = 2
B = 2
P = 17

# 곡선 위의 점 찾기
curve_points = ecc_points_on_finite_field(A, B, P)

## 결과 출력
print(f"**타원 곡선 E: y^2 ≡ x^3 + {A}x + {B} (mod {P})**\n")
print(f"총 점의 개수: {len(curve_points)}")
print("\n**곡선 위의 점들 (x, y):**")
# 5개씩 줄바꿈하여 출력
for i in range(0, len(curve_points), 5):
    print(curve_points[i:i+5])

# 기존 타원 곡선 위의 점 찾는 함수 (생략)
def ecc_points_on_finite_field(a, b, p):
    """
    유한체 F_p 위에서 정의된 타원 곡선 y^2 = x^3 + ax + b (mod p) 위의 모든 점을 찾는다.
    """
    points = []
    points.append(('O', 'O')) # 무한 원점
    for x in range(p):
        rhs = (x**3 + a*x + b) % p
        for y in range(p):
            if (y**2) % p == rhs:
                points.append((x, y))
    return points

def extended_gcd(a, b):
    """확장 유클리드 호제법을 사용하여 ax + by = gcd(a, b)를 만족하는 x, y를 찾는다."""
    if a == 0:
        return b, 0, 1
    gcd, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    return gcd, x, y

def mod_inverse(a, m):
    """모듈로 역수(Modular Inverse)를 찾는다. a^-1 (mod m)"""
    gcd, x, y = extended_gcd(a, m)
    if gcd != 1:
        # 역수가 존재하지 않음 (이 경우, ECC에서는 p가 소수이므로 발생해서는 안 됨)
        raise Exception('Modular inverse does not exist')
    # 결과가 양수가 되도록 보장
    return x % m

# --- 핵심: 점 덧셈 함수 ---
def point_addition(P, Q, a, p):
    """
    타원 곡선 E: y^2 ≡ x^3 + ax + b (mod p) 위에서 P + Q를 계산한다.
    P와 Q는 튜플 (x, y) 형태이며, 무한 원점은 ('O', 'O')이다.
    """
    x_P, y_P = P
    x_Q, y_Q = Q

    # 1. P가 무한 원점일 때: P + O = Q
    if P == ('O', 'O'):
        return Q
    # 2. Q가 무한 원점일 때: O + Q = P
    if Q == ('O', 'O'):
        return P
    
    # 3. P와 Q의 x좌표가 같고 y좌표가 다를 때: P + (-P) = O (P와 -P는 서로 역원)
    if x_P == x_Q and y_P != y_Q:
        # P = (x, y), -P = (x, p - y) 이므로, y_Q == p - y_P (mod p)
        return ('O', 'O')

    # 4. P = Q 일 때 (점 2배, Point Doubling)
    if P == Q:
        # 기울기 λ = (3x_P^2 + a) * (2y_P)^-1 (mod p)
        # 분자 (Numerator)
        numerator = (3 * x_P**2 + a) % p
        # 분모 (Denominator)의 모듈로 역수
        denominator_inv = mod_inverse(2 * y_P, p)
        
        # 기울기 λ (Lambda)
        lambda_val = (numerator * denominator_inv) % p
    
    # 5. P != Q 일 때 (일반적인 덧셈)
    else:
        # 기울기 λ = (y_Q - y_P) * (x_Q - x_P)^-1 (mod p)
        # 분자
        numerator = (y_Q - y_P) % p
        # 분모의 모듈로 역수
        denominator_inv = mod_inverse((x_Q - x_P) % p, p)
        
        # 기울기 λ (Lambda)
        lambda_val = (numerator * denominator_inv) % p
    
    # R = P + Q의 좌표 (R = (x_R, y_R)) 계산
    
    # x_R ≡ λ^2 - x_P - x_Q (mod p)
    x_R = (lambda_val**2 - x_P - x_Q) % p
    
    # y_R ≡ λ(x_P - x_R) - y_P (mod p)
    y_R = (lambda_val * (x_P - x_R) - y_P) % p
    
    return (x_R, y_R)

# --- 실행 예시 ---

# 곡선 정의: E: y^2 ≡ x^3 + 2x + 3 (mod 17)
A = 2
B = 2
P = 17

# 곡선 위의 점 찾기
curve_points = ecc_points_on_finite_field(A, B, P)
print(f"**타원 곡선 E: y^2 ≡ x^3 + {A}x + {B} (mod {P})**")
print(f"총 점의 개수: {len(curve_points)}")
print("-" * 30)
# 5개씩 줄바꿈하여 출력
for i in range(0, len(curve_points), 5):
    print(curve_points[i:i+5])
    
# 1. 점 덧셈 예시 (P != Q)
P1 = (5, 1) # 곡선 위의 한 점
P2 = (6, 3) # 곡선 위의 다른 점

R1 = point_addition(P1, P2, A, P)
print(f"1. 일반 덧셈: {P1} + {P2} = {R1}") # (5, 1) + (6, 3) = (10, 16)

# 2. 점 2배 예시 (P = Q)
P3 = (5, 1)

R2 = point_addition(P3, P3, A, P)
print(f"2. 점 2배: 2 * {P3} = {R2}") # 2 * (5, 1) = (13, 11)

# 1. 점 덧셈 예시 (P != Q)
P1 = (5, 1) # 곡선 위의 한 점
P2 = (6, 3) # 곡선 위의 다른 점

R1 = point_addition(P1, P2, A, P)
print(f"3. 일반 덧셈: {P1} + {P2} = {R1}") # (5, 1) + (6, 3) = (10, 16)


# 3. 역원 덧셈 예시 (P + (-P) = O)
P4 = (5, 1)
P_neg = (5, 17 - 1) # -P4 = (5, 16)
R3 = point_addition(P4, P_neg, A, P)
print(f"4. 역원 덧셈: {P4} + {P_neg} = {R3}") # (5, 1) + (5, 16) = ('O', 'O')

print("-" * 30)

## Chapter 4 행렬 연산    파이썬을 활용한 수치해석(Numerical Analysis with Python) 이인호 (북스힐, 2026)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# 예시 데이터 (X: 입력 변수, Y: 출력 변수)
# 예: X = [1, 2, 3, 4, 5], Y = [1, 2, 1.9, 4.2, 5.1]
X = np.array([1, 2, 3, 4, 5])
Y = np.array([1, 2, 1.9, 4.2, 5.1])
# 입력 데이터 행렬 A 구성 (선형 회귀에서는 1을 추가하여 상수항을 포함)
A = np.vstack([X, np.ones_like(X)]).T
# 최소제곱법을 통한 회귀 계수(beta) 계산
# beta = (A^T * A)^(-1) * A^T * Y
beta = np.linalg.inv(A.T @ A) @ A.T @ Y
# 회귀 직선 방정식
slope, intercept = beta
# 예측 값 계산 (회귀 직선 상의 Y 값)
Y_pred = A @ beta
# 결과 출력
print(f"기울기 (slope): {slope}")
print(f"절편 (intercept): {intercept}")
# 원본 데이터와 회귀 직선 시각화
plt.scatter(X, Y, color='blue', label='Original data')
plt.plot(X, Y_pred, color='red', label='Fitted line')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.show()

In [ ]:
import numpy as np
from scipy.linalg import lu, lu_solve, lu_factor
# 1. 행렬 및 벡터 정의
A = np.array([
[2.0, 1.0, 1.0],
[4.0, 1.0, 0.0],
[-2.0, 2.0, 1.0]
])
b = np.array([4.0, 6.0, 2.0])
print("--- 입력 행렬 및 벡터 ---")
print("행렬 A:\n", A)
print("벡터 b:\n", b)
# --- 2. LU 분해 단계 (PA = LU) ---
# lu_factor를 사용하여 L과 U 행렬 및 피벗 인덱스(piv)를 얻는다.
# L과 U는 하나의 행렬에 압축되어 저장된다.
lu_and_piv = lu_factor(A)
L, U, piv = lu(A) # L, U, P를 명시적으로 얻는 방법 (확인용)
print("\n--- LU 분해 결과 (확인용) ---")
# L은 하삼각, U는 상삼각 행렬이다. P는 행 순서를 나타내는 치환 행렬이다.
print("L (하삼각):\n", L.round(4))
print("U (상삼각):\n", U.round(4))
# --- 3. 해 계산 단계 ---
# lu_solve를 사용하여 두 대입 과정을 한 번에 처리한다.
# lu_solve는 내부적으로 L*y = P*b (전방 대입)와 U*x = y (후방 대입)을 수행한다.
x = lu_solve(lu_and_piv, b)
print("\n--- 최종 해 (x) ---")
print("x_1, x_2, x_3:", x.round(4))
# --- 검증 ---
# A @ x 가 b와 동일한지 확인
b_check = A @ x
print("\n--- 검증 (A @ x) ---")
print("b_calculated:", b_check.round(4))
print("b_original:  ", b.round(4))

In [ ]:
import numpy as np
# 1. 목표 공분산 행렬 정의 (3x3)
Sigma = np.array([
    [1.0, 0.8, 0.2],
    [0.8, 2.0, -0.5],
    [0.2, -0.5, 1.5]
    ])
# 샘플 수
N_samples = 100000
# 2. 숄레스키 분해
try:
    L = np.linalg.cholesky(Sigma)
    print("--- 숄레스키 인자 L (3x3) ---")
    print(L)
except np.linalg.LinAlgError:
    print("오류: 입력 행렬이 양의 정부호가 아니다. 숄레스키 분해 불가.")
    exit()
# 3. 독립적인 표준 정규 난수 생성 (3차원 벡터 N개)
# Z: (3, N_samples) 행렬
Z = np.random.normal(0, 1, size=(3, N_samples))
# 4. 숄레스키 인자를 이용한 변환 (X = L @ Z)
# L: (3, 3) @ Z: (3, N) -> X: (3, N)
X = L @ Z
# X1, X2, X3 변수 분리
X1 = X[0, :]
X2 = X[1, :]
X3 = X[2, :]
# 5. 생성된 변수의 통계량 확인
print("\n--- 생성된 변수의 통계량 ---")
print(f"분산 X1: {np.var(X1):.4f} (목표: 1.0)")
print(f"분산 X2: {np.var(X2):.4f} (목표: 2.0)")
print(f"분산 X3: {np.var(X3):.4f} (목표: 1.5)")
# 공분산 행렬 계산
# np.cov는 (3, N) 형태의 입력에 대해 공분산 행렬을 반환
cov_matrix_result = np.cov(X)
print("\n--- 생성된 공분산 행렬 ---")
print(cov_matrix_result.round(4))
# 공분산 값 비교 (X2와 X3의 음의 상관 관계 확인)
generated_cov_23 = cov_matrix_result[1, 2]
print(f"\n**X2와 X3의 공분산: {generated_cov_23:.4f} (목표: -0.5)**")

In [ ]:
import numpy as np
# 3x3 정방 행렬 정의
A = np.array([[12, -51, 4], 
[6, 167, -68], 
[-4, 24, -41]])
print("--- 입력 행렬 A ---")
print(A)
# QR 분해 수행: Q, R = np.linalg.qr(A)
Q, R = np.linalg.qr(A)
print("\n--- Q 행렬 (직교 행렬) ---")
print(Q)
print("\n--- R 행렬 (상삼각 행렬) ---")
print(R)
# A = Q @ R 검증
A_reconstructed = Q @ R
print("\n--- Q @ R 재구성 결과 ---")
print(A_reconstructed)
# 원래 행렬 A와 재구성된 행렬이 같은지 확인
print("\n--- A == Q @ R 확인 (오차 허용 범위 내) ---")
print(np.allclose(A, A_reconstructed))
# Q의 전치 행렬 Q_T
Q_T = Q.T
# Q^T @ Q 계산
QT_Q = Q_T @ Q
print("\n--- Q^T @ Q 결과 ---")
# 결과가 단위 행렬 I (주대각선은 1, 나머지는 0)과 거의 같은지 확인한다.
print(QT_Q)
# 단위 행렬과 같은지 검증
is_orthogonal = np.allclose(QT_Q, np.eye(A.shape[0]))
print(f"\nQ는 직교 행렬인가? (Q^T Q = I): {is_orthogonal}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# 1. 데이터 정의
x_data = np.array([1, 2, 3, 4])
y_data = np.array([2.1, 3.9, 6.2, 8.1])
# 2. 행렬 A (설계 행렬) 구성
# 선형 모델: y = m*x + c
# A 행렬은 첫 번째 열이 x_data, 두 번째 열이 상수 1로 구성되어야 한다.
# A = [[x1, 1], [x2, 1], ...]
A = np.vstack([x_data, np.ones(len(x_data))]).T
print("--- 설계 행렬 A ---")
print(A)
# 3. 최소제곱 해 구하기: np.linalg.lstsq(A, b, rcond=None)
# coeffs[0]: 계수 (m, c)
# residuals[1]: 잔차 제곱합
coeffs, residuals, rank, s = np.linalg.lstsq(A, y_data, rcond=None)
# m (기울기) = coeffs[0], c (y-절편) = coeffs[1]
m, c = coeffs
print("\n--- 최소제곱 결과 ---")
print(f"기울기 (m): {m:.4f}")
print(f"y-절편 (c): {c:.4f}")
print(f"최소제곱합 (잔차): {residuals[0]:.4f}")
# 4. 근사된 모델 (직선)
y_fit = m * x_data + c
# 5. 시각화 (선택 사항)
plt.figure(figsize=(8, 5))
plt.scatter(x_data, y_data, label='Original data (observed)', color='red')
plt.plot(x_data, y_fit, label=f'Best fit line (y = {m:.2f}x + {c:.2f})', color='blue')
plt.title('Linear least squares fit')
plt.xlabel('x', fontsize=16)
plt.ylabel('y', fontsize=16)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
from numpy.polynomial import Polynomial
def solve_polynomial_fitting_modern(x_data, y_data, degree=4):
    """
    NumPy의 최신 Polynomial API를 사용한 피팅 및 최적점 탐색 함수
    """
    # 1. 최신 API를 이용한 피팅
    # Polynomial.fit은 내부적으로 x 데이터를 [-1, 1] 구간으로 맵핑(domain windowing)하여
    # 수치적 오차를 획기적으로 줄인다.
    p_obj = Polynomial.fit(x_data, y_data, degree)
    # 2. 미분 및 근(Roots) 찾기 (f'(x) = 0)
    p_deriv = p_obj.deriv()
    roots = p_deriv.roots() # 메소드 호출 방식
    # 3. 실수 근만 필터링 및 범위 제한
    # 복소수 허수부가 거의 0인 경우를 고려해 np.isreal 사용
    real_roots = roots[np.isreal(roots)].real
    x_min = np.min(x_data)
    x_max = np.max(x_data)
    # 데이터 범위(Domain) 내에 있는 근만 후보로 선정
    candidates = [r for r in real_roots if x_min <= r <= x_max]
    # 4. 극소점 판별 (2계 미분 테스트)
    p_deriv2 = p_obj.deriv(2)
    local_minima = []
    for c in candidates:
        # f''(c) > 0 이면 아래로 볼록(Convex) -> 극소점
        if p_deriv2(c) > 0:
            local_minima.append((c, p_obj(c)))
    # 5. 최적값 결정 로직
    if local_minima:
        # 찾은 극소점들 중 가장 작은 함숫값을 가진 지점 선택
        # min 함수는 튜플의 두 번째 요소(함숫값)를 기준으로 최소를 찾음
        best_x, min_val = min(local_minima, key=lambda item: item[1])
        found_local_min = True
    else:
        # 극소점을 찾지 못한 경우 (단조 증가/감소 등), 데이터 구간 내 최솟값으로 대체
        min_idx = np.argmin(p_obj(x_data))
        best_x = x_data[min_idx]
        min_val = p_obj(best_x)
        found_local_min = False
    return p_obj, best_x, min_val, found_local_min

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
def find_local_minima():
    # 1. 가상의 신호 생성 (Peak와 Valley가 모두 있는 신호)
    x = np.linspace(0, 4*np.pi, 200)
    # y = sin(x) + 노이즈
    # sin(x)는 x=3pi/2, 7pi/2 등에서 극소점을 가짐
    true_y = np.sin(x)
    noise = np.random.normal(0, 0.05, size=len(x))
    y_raw = true_y + noise
    # 2. Savitzky-Golay로 미분값 추출
    window = 17
    poly = 3
    # 스무딩된 원본 (비교용)
    y_smooth = savgol_filter(y_raw, window, poly, deriv=0)
    # 1차 미분 (기울기)
    dy = savgol_filter(y_raw, window, poly, deriv=1)
    # 2차 미분 (곡률)
    d2y = savgol_filter(y_raw, window, poly, deriv=2)
    # 3. 극소점 찾기 알고리즘
    # Step A: 1차 미분이 0을 지나가는 지점(Zero-crossing) 찾기
    # 부호가 바뀔 때(음수 -> 양수 또는 양수 -> 음수)를 감지
    sign_change = np.diff(np.sign(dy))
    zero_crossings_indices = np.where(sign_change != 0)[0]
    minima_indices = []
    for idx in zero_crossings_indices:
        # Step B: 2차 미분 판별법 (Second Derivative Test)
        # 해당 지점에서 2차 미분이 양수(+)이면 극소점(Valley)
        if d2y[idx] > 0:
            minima_indices.append(idx)
            print(x[idx])
        # (만약 d2y[idx] < 0 이면 극대점(Peak)이다)
    # 4. 시각화
    fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
    # [1] 원본 신호 및 찾은 극소점
    axes[0].plot(x, y_raw, '.', color='lightgray', label='Raw Data')
    axes[0].plot(x, y_smooth, 'k-', alpha=0.6, label='Smoothed')
    # 찾은 극소점 표시
    if minima_indices:
        axes[0].plot(x[minima_indices], y_smooth[minima_indices], 'ro', 
            markersize=10, label='Detected Minima')
#   for i in minima_indices:
#    axes[0].text(x[i],y_smooth[i]-0.5, "Min",ha='center',color='red',fontweight='bold')
    axes[0].set_title(f"1. original signal(minima count: {len(minima_indices)})")
    axes[0].legend()
    axes[0].grid(True)
    axes[0].set_ylabel('y', fontsize=18)
    # [2] 1차 미분 (0이 되는 곳이 후보)
    axes[1].plot(x, dy, 'g-', label="1st Deriv (f')")
    axes[1].axhline(0, color='black', linestyle='--')
    # 후보 지점 표시
    axes[1].plot(x[minima_indices], dy[minima_indices], 'ro')
    axes[1].set_title("2. first derivative(zero crossing = critical points)")
    axes[1].grid(True)
    # [3] 2차 미분 (양수인지 확인)
    axes[2].plot(x, d2y, 'purple', label="2nd Deriv (f'')")
    axes[2].axhline(0, color='black', linestyle='--')
    # 극소점 위치에서의 2차 미분값 표시
    for idx in minima_indices:
        val = d2y[idx]
        axes[2].plot(x[idx], val, 'ro')
# axes[2].text(x[idx], val+0.05,"Positive (+)",ha='center',color='red',fontweight='bold')
    axes[2].set_title("3. second derivative(positive = minima/convex)")
    axes[2].grid(True)
    axes[2].set_xlabel('x', fontsize=18)
    plt.tight_layout()
    plt.savefig('sg1.png')
    plt.show()
if __name__ == "__main__":
    find_local_minima()

In [ ]:
import numpy as np
# 3x2 행렬 A 정의 (정방 행렬이 아님)
A = np.array([[1, 1], 
[0, 1], 
[1, 0]])
print("--- 입력 행렬 A (3x2) ---")
print(A)
# np.linalg.svd() 함수를 사용하여 SVD 수행
# U: 좌측 특이 벡터 행렬
# s: 특이값 (Sigma 행렬의 대각 성분)
# Vt: 우측 특이 벡터 행렬의 전치 (V^T)
U, s, Vt = np.linalg.svd(A)
print("\n--- U 행렬 (좌측 특이 벡터, 3x3) ---")
print(U)
print("\n--- 특이값 s (2개) ---")
print(s)
print("\n--- Vt 행렬 (우측 특이 벡터의 전치, 2x2) ---")
print(Vt)
# A = U * Sigma * Vt 검증을 위한 Sigma 행렬 재구성
# Sigma는 A와 같은 크기인 3x2 행렬로 만들어야 한다.
Sigma = np.zeros(A.shape)
# 특이값 s를 Sigma의 주대각선에 배치
Sigma[:A.shape[1], :A.shape[1]] = np.diag(s)
print("\n--- 재구성된 Sigma 행렬 (3x2) ---")
print(Sigma)
# U * Sigma * Vt 계산 및 검증
A_reconstructed = U @ Sigma @ Vt
print("\n--- U * Sigma * Vt 재구성 결과 ---")
print(A_reconstructed)
# 원래 행렬 A와 재구성된 행렬이 같은지 확인
print("\n--- 재구성된 행렬이 원본과 같은가? (오차 허용 범위 내) ---")
print(np.allclose(A, A_reconstructed))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def svd_recommender_system():
    # 1. 영화 평점 행렬 생성 (0은 안 본 영화)
    # 영화: [매트릭스(SF), 스타워즈(SF), 다크나이트(액션) | 타이타닉(로맨스), 노트북(로맨스)]
    # 사용자 A, B: SF/액션 팬
    # 사용자 C, D: 로맨스 팬
    ratings = np.array([
    [5, 5, 0, 0, 0],  # 사용자 A: 매트릭스, 스타워즈는 봤는데 나머지는 안 봄
    [5, 0, 4, 0, 0],  # 사용자 B: 매트릭스, 다크나이트 봄
    [0, 0, 0, 5, 5],  # 사용자 C: 로맨스만 봄
    [0, 0, 0, 5, 4],  # 사용자 D: 로맨스만 봄
    ])
    movie_names = ["Matrix", "StarWars", "DarkKnight", "Titanic", "Notebook"]
    user_names = ["User A (SF Fan)", "User B (SF Fan)", "User C (Romance)", "User D (Romance)"]
    print("--- 1. 원본 평점 행렬 (0은 결측치) ---")
    print(ratings)
    # 2. SVD 수행
    # full_matrices=False로 해야 차원을 맞춰준다.
    U, S, Vt = np.linalg.svd(ratings, full_matrices=False)
    # 3. 차원 축소 (k=2) 
    # 왜 2개인가? -> 데이터가 크게 'SF/액션'과 '로맨스' 두 가지 경향으로 나뉠 것이라 가정
    k = 2
    U_k = U[:, :k]
    S_k = np.diag(S[:k])
    Vt_k = Vt[:k, :]
    # 4. 행렬 복원 (예측)
    # 원본에는 0이었던 곳에 '예측 평점'이 채워지게 됨
    pred_ratings = np.dot(np.dot(U_k, S_k), Vt_k)
    print("\n--- 2. SVD로 예측한 평점 행렬 ---")
    print(np.round(pred_ratings, 1))
    # 5. 시각화: 영화와 사용자를 '취향 지도' 위에 그리기
    # 2차원 공간(Latent Space)에 투영
    plt.figure(figsize=(10, 6))
    # (1) 영화 위치 (Vt의 행벡터들이 영화의 좌표가 됨)
    # x축: 잠재요인 1 (예: SF 성향), y축: 잠재요인 2 (예: 로맨스 성향)
    for i, movie in enumerate(movie_names):
        x = Vt_k[0, i]
        y = Vt_k[1, i]
        plt.scatter(x, y, c='red', marker='s', s=100) # s=square
        plt.text(x+0.02, y, movie, fontsize=12, fontweight='bold', color='red')
    # (2) 사용자 위치 (U와 S를 곱해서 좌표로 변환)
    # 사용자가 어느 영화들 근처에 위치하는지 볼 수 있음
    user_coords = np.dot(U_k, S_k)
    for i, user in enumerate(user_names):
        x = user_coords[i, 0]
        y = user_coords[i, 1]
        plt.scatter(x, y, c='blue', marker='o', s=100) # o=circle
        plt.text(x+0.02, y, user.split(' ')[0] + " " + user.split(' ')[1], 
            fontsize=10, color='blue')
    plt.title("SVD latent space: Mapping users and movies")
    plt.xlabel("Latent factor 1 (concept 1)")
    plt.ylabel("Latent factor 2 (concept 2)")
    plt.grid(True)
    plt.axhline(0, color='black', linewidth=0.5)
    plt.axvline(0, color='black', linewidth=0.5)
    plt.savefig('svd_movies.png')
    plt.show()
    # 결과 해석 (사용자 A에게 추천)
    predicted_score = pred_ratings[0, 2] # 사용자 A의 다크나이트(Index 2) 예측 점수
    print(f"\n[추천 결과] 사용자 A는 'DarkKnight'를 안 봤지만, SVD 예측 점수는 {predicted_score:.1f}점이다.")
    print("-> 추천 시스템: '사용자 A님, 다크나이트를 추천한다!'")
svd_recommender_system()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.datasets import face # 샘플 이미지 로드용
# ==========================================
# 1. 노이즈 추가 함수 (시뮬레이션용)
# ==========================================
def add_gaussian_noise(image, mean=0, sigma=25):
    """이미지에 가우시안 노이즈를 추가한다."""
    row, col = image.shape
    gauss = np.random.normal(mean, sigma, (row, col))
    noisy_image = image + gauss
    # 0~255 범위를 벗어나는 값을 잘라낸다.
    noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)
    return noisy_image
# ==========================================
# 2. SVD 노이즈 제거 함수
# ==========================================
def denoise_with_svd(noisy_image, keep_ratio=0.1):
    """
    SVD를 사용하여 노이즈를 제거한다.
    keep_ratio: 상위 몇 %의 특이값을 남길지 결정 (0.0 ~ 1.0)
    값이 작을수록 노이즈가 많이 제거되지만 이미지도 흐려진다.
    """
    # SVD 계산을 위해 float 타입으로 변환
    img_float = noisy_image.astype(np.float64)
    # 1) SVD 수행
    U, S, Vt = np.linalg.svd(img_float, full_matrices=False)
    # 2) 남길 특이값 개수(k) 계산
    k = int(len(S) * keep_ratio)
    k = max(1, k) # 최소 1개는 남김
    print(f"-> 전체 특이값 {len(S)}개 중 상위 {k}개({keep_ratio*100:.1f}%)만 사용하여 복원한다.")
    # 3) 특이값 필터링 (대각 행렬 생성)
    S_truncated = np.diag(S[:k])
    # 4) 행렬 복원 (U_k * S_k * Vt_k)
    denoised_img = np.dot(np.dot(U[:, :k], S_truncated), Vt[:k, :])
    # 이미지 형식(0~255 uint8)으로 다시 변환
    denoised_img = np.clip(denoised_img, 0, 255).astype(np.uint8)
    return denoised_img
# ==========================================
# 3. 메인 실행 흐름
# ==========================================
# --- Step 1: 이미지 읽기 (Grayscale) ---
# [내 이미지를 사용할 경우] 아래 주석을 해제하고 경로를 입력하세요.
img_path = 'stft_wavelet.png' 
original_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
# [샘플 이미지 사용]
#original_img = face(gray=True)
original_img = cv2.resize(original_img, (512, 384)) # 속도를 위해 크기 조정
if original_img is None:
    print("이미지를 파일을 찾을 수 없다.")
    exit()
print(f"원본 이미지 크기: {original_img.shape}")
# --- Step 2: 노이즈 추가 (테스트를 위해 일부러 망가뜨림) ---
print("노이즈 추가 중...")
noisy_img = add_gaussian_noise(original_img, sigma=30) # sigma가 클수록 노이즈 심함
# --- Step 3: SVD 노이즈 제거 시도 ---
print("\nSVD 노이즈 제거 시작...")
# keep_ratio를 조절해보세요. (0.05 = 상위 5%만 남김)
# 0.01 (매우 흐릿) ~ 0.2 (노이즈 남음) 사이에서 실험 권장
denoised_img = denoise_with_svd(noisy_img, keep_ratio=0.05) 
# --- Step 4: 결과 시각화 (비교) ---
plt.figure(figsize=(15, 5))
# 원본
plt.subplot(1, 3, 1)
plt.imshow(original_img, cmap='gray')
plt.title("1. Original clean image")
plt.axis('off')
# 노이즈 추가됨
plt.subplot(1, 3, 2)
plt.imshow(noisy_img, cmap='gray')
plt.title("2. Corrupted with noise (Input)")
plt.axis('off')
# SVD 복원 결과
plt.subplot(1, 3, 3)
plt.imshow(denoised_img, cmap='gray')
plt.title("3. Denoised with SVD (Top 5%)")
plt.axis('off')
plt.tight_layout()
plt.savefig('denoising.png')
plt.show()	

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def pseudo_inverse_demo():
    # 1. 데이터 생성 (y = 2x + 1 근처에 점들이 흩어져 있음)
    # 식은 4개인데 미지수는 2개(기울기, 절편)인 '과결정' 상태
    x = np.array([1, 2, 3, 4])
    y = np.array([2.96, 5.13, 7.2, 8.8]) # 완벽한 직선 아님 (노이즈 있음)
    # 2. 행렬 A 만들기 (Ax = b 형태)
    # y = mx + c  =>  mx + c * 1 = y
    # A = [[x1, 1], [x2, 1], ...]
    A = np.column_stack((x, np.ones_like(x)))
    b = y
    print("Matrix A (4x2):")
    print(A)
    # 3. 의사 역 행렬 계산 (numpy.linalg.pinv)
    # 내부적으로 SVD를 사용한다.
    A_pinv = np.linalg.pinv(A)
    print("\nPseudo-inverse of A (2x4):")
    print(np.round(A_pinv, 3))
    # 4. 해 구하기 (x = A^+ b)
    # solution[0]은 기울기(m), solution[1]은 절편(c)
    solution = np.dot(A_pinv, b)
    m_calc, c_calc = solution
    print(f"\n[결과] 기울기(m): {m_calc:.3f}, 절편(c): {c_calc:.3f}")
    print("-> 이 값이 오차 제곱 합을 최소로 만드는 '최적의 해'이다.")
    # 5. 시각화
    plt.figure(figsize=(8, 6))
    # 원본 데이터 점
    plt.scatter(x, y, color='red', label='Data points(Noisy)')
    # 의사 역 행렬로 구한 추세선
    x_line = np.linspace(0, 5, 100)
    y_line = m_calc * x_line + c_calc
    plt.plot(x_line,y_line,color='blue',
        label=f'Best fit line: y={m_calc:.2f}x + {c_calc:.2f}')
    plt.title("Least squares solution using pseudo-inverse")
    plt.legend()
    plt.grid(True)
    plt.savefig('lss_pi.png')
    plt.show()
pseudo_inverse_demo()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def under_determined_demo():
    # 1. 문제 정의: 1x + 2y = 10
    # 행렬 형태: [1, 2] * [x, y]^T = [10]
    # 미지수는 2개인데 식은 1개뿐인 '부정(Under-determined)' 상태 (Fat Matrix)
    A = np.array([[1, 2]]) 
    b = np.array([10])
    print(f"Matrix A shape: {A.shape} (Fat Matrix)")
    # 2. 의사 역 행렬을 이용한 해 구하기
    A_pinv = np.linalg.pinv(A)
    x_solution = np.dot(A_pinv, b) # x = A^+ b
    print(f"\n[의사 역 행렬이 선택한 해]")
    print(f"x = {x_solution[0]:.2f}, y = {x_solution[1]:.2f}")
    # 이 해의 크기(Norm, 원점으로부터의 거리) 계산
    norm_solution = np.linalg.norm(x_solution)
    print(f"-> 원점과의 거리(Norm): {norm_solution:.4f}")
    # 3. 다른 해들과 비교 (무작위로 다른 답을 넣어봄)
    # x + 2y = 10 을 만족하는 다른 해들: (10, 0), (0, 5), (-2, 6)
    other_solutions = [
        np.array([10, 0]),
        np.array([0, 5]),
        np.array([-2, 6])]
    print("\n[다른 가능한 해들과 비교]")
    for sol in other_solutions:
        dist = np.linalg.norm(sol)
        print(f"해 {sol}: 원점 거리 = {dist:.4f}")
    # 4. 시각화
    plt.figure(figsize=(8, 8))
    # (1) 해 집합 직선 그리기 (x + 2y = 10 => y = -0.5x + 5)
    x_vals = np.linspace(-2, 12, 100)
    y_vals = -0.5 * x_vals + 5
    plt.plot(x_vals, y_vals, 'k--', label='All possible solutions (x+2y=10)')
    # (2) 다른 해들 찍기
    for sol in other_solutions:
        plt.scatter(sol[0], sol[1], color='blue', s=50)
        plt.text(sol[0]+0.2,sol[1]+0.2,f"Norm:{np.linalg.norm(sol):.1f}",color='blue')
        # 원점 연결선 (흐리게)
        plt.plot([0, sol[0]], [0, sol[1]], 'b:', alpha=0.3)
    # (3) 의사 역 행렬 해 찍기
    plt.scatter(x_solution[0], x_solution[1], color='red', s=100, 
        zorder=5, label='Pseudo-inverse solution')
    plt.text(x_solution[0]+0.5, x_solution[1]-0.5, f"Min norm:{norm_solution:.2f}", 
        color='red', fontweight='bold')
    # (4) 원점에서 의사 역 행렬 해까지 벡터 (수직임을 보여줌)
    plt.arrow(0, 0, x_solution[0], x_solution[1], head_width=0.3, head_length=0.3, 
        fc='red', ec='red', width=0.05)
    plt.xlim(-2, 12)
    plt.ylim(-2, 8)
    plt.axhline(0, color='black', linewidth=1)
    plt.axvline(0, color='black', linewidth=1)
    plt.gca().set_aspect('equal') # 정사각형 비율 유지
    plt.legend()
    plt.title("Under-determined system: Minimum norm solution")
    plt.grid(True)
    plt.savefig('minnorm_pi.png')
    plt.show()
under_determined_demo()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
# 방향성 그래프(Directed Graph) 생성
G = nx.DiGraph()
# 노드(페이지) 추가
pages = ['A', 'B', 'C', 'D']
G.add_nodes_from(pages)
# 엣지(링크) 추가 (링크: From -> To)
# A -> B
# B -> C
# C -> A, C -> B
# D -> A
# (이전에 사용했던 예제 그래프와 동일한 구조)
G.add_edges_from([
	('A', 'B'),
	('B', 'C'),
	('C', 'A'),
	('C', 'B'),
	('D', 'A')
	])
# PageRank 계산
# alpha는 댐핑 팩터 'd'에 해당한다. (기본값: 0.85)
pr_scores = nx.pagerank(G, alpha=0.85)
print("--- NetworkX PageRank 점수 결과 ---")
# 결과를 PageRank 점수가 높은 순서로 정렬하여 출력
for page, score in sorted(pr_scores.items(), key=lambda item: item[1], reverse=True):
	print(f"페이지 {page}: {score:.4f}")
# PageRank 점수에 비례하여 노드 크기 조정
node_size = [v * 10000 for v in pr_scores.values()]
# 그래프 레이아웃 설정 (예: Spring Layout)
pos = nx.spring_layout(G, seed=42) 
# 그래프 그리기
plt.figure(figsize=(8, 6))
nx.draw(
	G, 
	pos, 
	with_labels=True, 
	node_size=node_size, 
	node_color='skyblue', 
	font_weight='bold', 
	edge_color='gray',
	arrows=True,
	arrowsize=20
	)
plt.title("PageRank scores visualization (Node size proportional to score)")
plt.savefig('pagerank.png')
plt.show()    

In [ ]:
import numpy as np
# 2x2 정방 행렬 정의
A = np.array([[4, 2], 
              [1, 3]])
# np.linalg.eig() 함수를 사용하여 고유값과 고유벡터 계산
# w: 고유값(eigenvalues), v: 고유벡터(eigenvectors)
w, v = np.linalg.eig(A)
print("--- 입력 행렬 A ---")
print(A)
print("\n--- 고유값 (Eigenvalues) ---")
print(w)
print("\n--- 고유벡터 (Eigenvectors) ---")
# 각 열(column)이 해당 고유값에 대응하는 고유벡터이다.
# w[0] = 5.0 에 대응하는 고유벡터는 v[:, 0] = [0.8944..., 0.4472...] 이다.
# w[1] = 2.0 에 대응하는 고유벡터는 v[:, 1] = [-0.7071..., 0.7071...] 이다.
print(v)
# 첫 번째 고유값과 고유벡터 추출
lambda_1 = w[0]
v_1 = v[:, 0]
print("--- 첫 번째 고유값과 고유벡터 ---")
print(f"고유값 (λ1): {lambda_1}")
print(f"고유벡터 (v1):\n{v_1}")
# A * v1 계산 (Av)
Av_1 = A @ v_1 
# lambda_1 * v1 계산 (λv)
lambda_v_1 = lambda_1 * v_1
print("\n--- Av1 계산 결과 ---")
print(Av_1)
print("\n--- λ1*v1 계산 결과 ---")
print(lambda_v_1)
# 두 결과가 거의 같은지 확인
print("\n--- Av1 == λ1*v1 확인 (오차 허용 범위 내) ---")
print(np.allclose(Av_1, lambda_v_1))

In [ ]:
import numpy as np
import time
def rayleigh_quotient_iteration_hermitian(A, x0,max_iter=10,tolerance=1e-10):
    """
    에르미트 행렬을 위한 Rayleigh Quotient Iteration (RQI) 함수.
    """
    x_k = x0 / np.linalg.norm(x0)
    lambda_k = (np.conj(x_k).T @ A @ x_k).real
    for k in range(max_iter):
        lambda_prev = lambda_k
        B = A - lambda_k * np.eye(A.shape[0])
        try:
            z_k_plus_1 = np.linalg.solve(B, x_k)
        except np.linalg.LinAlgError:
            break
# 정규화 부분 (Orthonormality 조건 중 Normalization을 만족시킴)
        x_k_plus_1 = z_k_plus_1 / np.linalg.norm(z_k_plus_1)
        lambda_k_plus_1 = (np.conj(x_k_plus_1).T @ A @ x_k_plus_1).real
        eigenvalue_diff = np.abs(lambda_k_plus_1 - lambda_prev)
        if eigenvalue_diff < tolerance:
            return lambda_k_plus_1, x_k_plus_1, k + 1
        lambda_k = lambda_k_plus_1
        x_k = x_k_plus_1
    return lambda_k, x_k, max_iter
def main():
# 행렬 크기 설정
    N = 100
    print(f" 행렬 크기: {N} x {N}")
    print("-" * 30)
# 1. 에르미트 행렬 생성
    C = np.random.rand(N, N) + 1j * np.random.rand(N, N)
    A = C + np.conj(C).T
# 2. 초기 벡터 생성
    x0 = np.random.rand(N) + 1j * np.random.rand(N)
    start_time_rqi = time.time()
# 3. RQI 실행
    final_lambda,final_x,iterations=rayleigh_quotient_iteration_hermitian(A,x0,max_iter=20)
    end_time_rqi = time.time()
# 4. NumPy 검증 (모든 고유값 계산)
    start_time_numpy = time.time()
    w, v = np.linalg.eigh(A)
    end_time_numpy = time.time()
# 5. 결과 출력
    print("\n[RQI 계산 결과]")
    print(f"계산된 고유값 (λ): {final_lambda:.6f}")
    print(f"반복 횟수: {iterations}")
    print(f"계산 시간: {end_time_rqi - start_time_rqi:.4f} 초")
# RQI 결과가 NumPy 결과 중 어떤 고유값에 가까운지 확인
    min_diff_index = np.argmin(np.abs(w - final_lambda))
    print("\n[NumPy 검증 및 전체 스펙트럼]")
    print(f"NumPy 고유값 중 RQI 결과와 가장 가까운 값 (w[{min_diff_index}]): {w[min_diff_index]:.6f}")
# 잔차 노름 검증
    residual_norm = np.linalg.norm(A @ final_x - final_lambda * final_x)
    print(f"잔차 노름 (||Ax - λx||): {residual_norm:.2e}")
# **추가된 부분: Orthonormality (정규화) 조건 체크**
    norm_x = np.linalg.norm(final_x)
# L2 노름이 1과 얼마나 가까운지 확인
    orthonormal_check = np.abs(norm_x - 1.0)
    print("\n[직교 정규성 (Orthonormality) 검사]")
    print(f"계산된 고유벡터의 L2 노름 (||x||): {norm_x:.10f}")
    if orthonormal_check < 1e-9:
        print(" 정규화 조건 만족: ||x||는 1에 매우 근접한다.")
    else:
        print(" 정규화 조건 불만족.")
    print("-" * 30)
# 모든 고유값 출력
    print(f"NumPy로 계산된 **모든 고유값 (총 {N}개)**:")
    print(np.array2string(w, precision=4, separator=', ', suppress_small=True))
    print(f"(참고: 고유값의 범위는 약 {w.min():.2f} 에서 {w.max():.2f} 이다.)")
    print("-" * 30)
    print(f"NumPy 전체 고유값 계산 시간: {end_time_numpy - start_time_numpy:.4f} 초")
if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
def solve_block_davidson_7point():
	# ---------------------------------------------------------
	# 1. 문제 설정
	# ---------------------------------------------------------
	# 7-point는 정밀도가 높아서 N을 200~300으로 줄여도 매우 정확하다.
	N = 300 
	L = 8.0
	dx = (2 * L) / (N - 1)
	x = np.linspace(-L, L, N)
	V_pot = 0.5 * x**2 
	# ---------------------------------------------------------
	# 2. 7-point Stencil 계수 및 Preconditioner 설정
	# ---------------------------------------------------------
	# 공식: f''(x) approx (2f(i+3) - 27f(i+2) + 270f(i+1) - 490f(i)+ ...)/180h^2
	# Kinetic T = -0.5 * d^2/dx^2 이므로 계수에 -0.5를 곱함
	denom = 180.0 * (dx**2)
	factor = -0.5 / denom
	c3 = factor * 2.0     # i +/- 3
	c2 = factor * -27.0   # i +/- 2
	c1 = factor * 270.0   # i +/- 1
	c0 = factor * -490.0  # Center (i)
	# [핵심] Davidson Preconditioner용 대각 성분 (D)
	# 3-point일 때는 1/dx^2 이었지만, 여기서는 c0가 대각 성분임
	diag_kinetic = c0 
	D = diag_kinetic + V_pot
	# ---------------------------------------------------------
	# 3. Matrix-Free Hamiltonian (7-point)
	# ---------------------------------------------------------
	def apply_hamiltonian(Psi_block):
	    # Psi_block shape: (N, block_size)
	    T_psi = np.zeros_like(Psi_block)
	    # 7-point stencil slicing (벡터화 연산)
	    # Boundary: 양 끝 3칸(0,1,2 / -1,-2,-3)은 0으로 둠 (Dirichlet BC)
	    # Center
	    term_c = Psi_block[3:-3, :]
	    # Neighbors
	    term_p1 = Psi_block[4:-2, :] # i+1
	    term_m1 = Psi_block[2:-4, :] # i-1
	    term_p2 = Psi_block[5:-1, :] # i+2
	    term_m2 = Psi_block[1:-5, :] # i-2
	    term_p3 = Psi_block[6:, :]   # i+3
	    term_m3 = Psi_block[0:-6, :] # i-3
	    # 수식 적용
	    # T_psi[3:-3] = c0*Center + c1*(i±1) + c2*(i±2) + c3*(i±3)
	    T_psi[3:-3, :] = (c0 * term_c + 
	                      c1 * (term_p1 + term_m1) + 
	                      c2 * (term_p2 + term_m2) + 
	                      c3 * (term_p3 + term_m3))
	    # Potential Part
	    # V_pot (N,) * Psi_block (N, k) -> Broadcasting
	    V_psi = V_pot[:, None] * Psi_block
	    return T_psi + V_psi
	# ---------------------------------------------------------
	# 4. Block Davidson Solver
	# ---------------------------------------------------------
	print("Block Davidson solver (7-Point High precision) 시작...")
	n_roots = 5        
	block_size = 10    
	max_subspace = 40  
	tol = 1e-9  # 고차 미분을 쓰므로 목표 오차를 더 낮게(정밀하게) 설정 가능
	max_iter = 500
	V = np.random.rand(N, block_size)
	V, _ = la.qr(V, mode='economic')
	final_evals = None
	final_evecs = None
	last_iter_best_vecs = None # 안전장치
	for iteration in range(max_iter):
	    # (1) Projection
	    AV = apply_hamiltonian(V)
	    T = V.T @ AV
	    # (2) Diagonalization
	    evals_sub, evecs_sub = la.eigh(T)
	    current_evals = evals_sub[:n_roots]
	    # 현재 베스트 해 저장 (Restart 대비)
	    last_iter_best_vecs = V @ evecs_sub[:, :n_roots]
	    # (3) Residual Check & Correction
	    correction_vectors = []
	    converged_count = 0
	    for i in range(block_size):
	        lambda_i = evals_sub[i]
	        x_i = V @ evecs_sub[:, i]
	        # r = Ax - lambda*x
	        Av_i = AV @ evecs_sub[:, i]
	        r = Av_i - lambda_i * x_i
	        norm_r = np.linalg.norm(r)
	        if i < n_roots and norm_r < tol:
	            converged_count += 1
	        # (4) Preconditioning: delta = r / (D - lambda)
	        denom = D - lambda_i
	        denom = np.where(np.abs(denom) < 1e-6, 1e-6, denom)
	        delta = r / denom
	        correction_vectors.append(delta)
	    # 진행 상황 출력
	    if iteration % 10 == 0 or converged_count >= n_roots:
	        print(f"Iter {iteration+1}: Converged {converged_count}/{n_roots},E_0 = {current_evals[0]:.9f}")
	    # 수렴 완료
	    if converged_count >= n_roots:
	        print(">>> 모든 목표 고유값 수렴 완료!")
	        final_evals = current_evals
	        final_evecs = last_iter_best_vecs
	        break
	    # (5) Expansion
	    new_vecs = np.array(correction_vectors).T
	    combined = np.hstack([V, new_vecs])
	    Q_combined, _ = la.qr(combined, mode='economic')
	    # (6) Restart Logic
	    if Q_combined.shape[1] > max_subspace:
	        # Subspace가 너무 커지면 중요한 벡터만 남김
	        best_ritz_vecs = V @ evecs_sub[:, :block_size]
	        V, _ = la.qr(best_ritz_vecs, mode='economic')
	    else:
	        V = Q_combined
	if final_evals is None:
	    print(">>> 최대 반복 도달. 현재 최적값을 반환한다.")
	    final_evals = evals_sub[:n_roots]
	    final_evecs = last_iter_best_vecs
	print("\n[Block Davidson 7-Point 결과]")
	print(f"{'State':<6} | {'Calculated':<12} | {'Theory':<10} | {'Error':<12}")
	print("-" * 55)
	for i, e in enumerate(final_evals):
	    theory = 0.5 + i
	    error = abs(e - theory)
	    print(f"{i:<6} | {e:.10f}   | {theory:.1f}        | {error:.2e}")
	plt.figure(figsize=(10, 6))
	plt.plot(x, V_pot, 'k--', alpha=0.3, label="Potential")
	scale = 2.0
	plt.xlim(-5.0,5.0)
	plt.ylim(0.0, 6.0)
	for i in range(n_roots):
	    psi = final_evecs[:, i]
	    if psi[N//2] < 0: psi = -psi # 위상 정렬
	    psi = psi / np.linalg.norm(psi) * scale
	    plt.plot(x, final_evals[i] + psi, label=f'n={i}')
	    plt.axhline(final_evals[i], color='gray', linestyle=':', alpha=0.3)
	plt.title("Block Davidson + 7-Point stencil(High precision)")
	plt.xlabel("x(atomic unit)", fontsize=18)
	plt.ylabel("Energy(atomic unit)", fontsize=18)
	plt.legend()
	plt.savefig('blockDavidson.png')
	plt.show()
solve_block_davidson_7point()	

In [ ]:
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
import time
def solve_3d_harmonic_oscillator_complete():
	# ---------------------------------------------------------
	# 1. 3D 격자 설정
	# ---------------------------------------------------------
	# N=40 -> 전체 행렬 크기 64,000 x 64,000
	N = 40  
	L = 5.0 
	dx = (2 * L) / (N - 1)
	# 1D 좌표
	x_1d = np.linspace(-L, L, N)
	# 3D Grid (Indexing='ij'는 행렬 방식 순서)
	X, Y, Z = np.meshgrid(x_1d, x_1d, x_1d, indexing='ij')
	# 3D Potential: V = 0.5 * (x^2 + y^2 + z^2)
	V_pot_3d = 0.5 * (X**2 + Y**2 + Z**2)
	# Preconditioner 계산을 위해 1차원으로 펼침
	V_pot_flat = V_pot_3d.ravel()
	# 전체 차원 수
	dim = N**3
	print(f"System Dimension: {N}^3 = {dim}")
	# ---------------------------------------------------------
	# 2. 7-point Stencil 계수 설정 (고정밀)
	# ---------------------------------------------------------
	# f''(x) approx (2f(i+3) - 27f(i+2) + ... ) / 180h^2
	denom = 180.0 * (dx**2)
	factor = -0.5 / denom # Hamiltonian Kinetic term (-0.5 * laplacian)
	c3 = factor * 2.0
	c2 = factor * -27.0
	c1 = factor * 270.0
	c0 = factor * -490.0
	# 3D Laplacian의 대각 성분 (x, y, z 방향 모두 고려)
	# D_ii = 3 * (T_ii_1d) + V_ii
	diag_kinetic = 3.0 * c0
	D_flat = diag_kinetic + V_pot_flat
	# ---------------------------------------------------------
	# 3. Matrix-Free 3D Hamiltonian Function
	# ---------------------------------------------------------
	def apply_hamiltonian_3d(Psi_block):
	    # Psi_block: (N^3, block_size) -> 2D array
	    n_vecs = Psi_block.shape[1]
	    # 3D 연산을 위해 (N, N, N, k) 형태로 뷰(View) 변환
	    psi_3d = Psi_block.reshape((N, N, N, n_vecs))
	    # 결과 배열
	    H_psi_3d = np.zeros_like(psi_3d)
	    # [Helper] 특정 축(axis) 방향으로 7-point stencil 적용
	    def add_kinetic_along_axis(axis_idx):
	        # 슬라이싱 헬퍼: axis_idx 방향으로만 [start:end] 적용
	        def s(start, end):
	            slices = [slice(None)] * 4 # (x, y, z, block)
	            slices[axis_idx] = slice(start, end)
	            return tuple(slices)
	        # Central term
	        H_psi_3d[s(3,-3)] += c0 * psi_3d[s(3,-3)]
	        # Neighbors (+/- 1)
	        H_psi_3d[s(3,-3)] += c1 * (psi_3d[s(4,-2)] + psi_3d[s(2,-4)])
	        # Neighbors (+/- 2)
	        H_psi_3d[s(3,-3)] += c2 * (psi_3d[s(5,-1)] + psi_3d[s(1,-5)])
	        # Neighbors (+/- 3)
	        H_psi_3d[s(3,-3)] += c3 * (psi_3d[s(6,None)] + psi_3d[s(0,-6)])
	    # x, y, z 축 각각 적용
	    add_kinetic_along_axis(0)
	    add_kinetic_along_axis(1)
	    add_kinetic_along_axis(2)
	    # Potential Energy 추가 (Broadcasting)
	    H_psi_3d += V_pot_3d[:, :, :, None] * psi_3d
	    # 다시 1D로 펴서 반환
	    return H_psi_3d.reshape((dim, n_vecs))
	# ---------------------------------------------------------
	# 4. Block Davidson Solver Execution
	# ---------------------------------------------------------
	print("Block Davidson (3D, 7-point High Precision) 시작...")
	start_time = time.time()
	n_roots = 6        # 바닥상태(1) + 1차 여기(3) + 2차 여기 일부(2)
	block_size = 12    # 블록 크기
	max_subspace = 60  # 최대 부분공간 크기
	tol = 1e-6         # 수렴 조건
	max_iter = 300     # 최대 반복 횟수
	# 초기화
	V = np.random.rand(dim, block_size)
	V, _ = la.qr(V, mode='economic')
	# 안전장치용 변수 초기화
	final_evals = None
	for iteration in range(max_iter):
	    # (1) Subspace Projection
	    AV = apply_hamiltonian_3d(V)
	    T = V.T @ AV
	    # (2) Diagonalization of Subspace Matrix
	    evals_sub, evecs_sub = la.eigh(T)
	    # [중요 수정] 현재 스텝의 근삿값을 매번 저장 (오류 방지)
	    current_evals = evals_sub[:n_roots]
	    final_evals = current_evals 
	    # (3) Residual Check & Correction Vector Generation
	    correction_vectors = []
	    converged_count = 0
	    for i in range(block_size):
	        lam = evals_sub[i]
	        # r = AV*y - lam*V*y (이미 계산된 행렬 활용)
	        r = AV @ evecs_sub[:, i] - lam * (V @ evecs_sub[:, i])
	        norm_r = np.linalg.norm(r)
	        # 수렴 체크
	        if i < n_roots and norm_r < tol:
	            converged_count += 1
	        # Davidson Preconditioning: delta = r / (D - lambda)
	        denom = D_flat - lam
	        # 0으로 나누기 방지 (특이점 처리)
	        denom = np.where(np.abs(denom) < 1e-5, 1e-5, denom)
	        delta = r / denom
	        correction_vectors.append(delta)
	    # 진행상황 출력
	    if iteration % 5 == 0:
	        print(f"Iter {iteration}: Converged {converged_count}/{n_roots},Ground E = {current_evals[0]:.6f}")
	    # (4) Convergence Exit
	    if converged_count >= n_roots:
	        print(">>> 모든 목표 상태 수렴 완료!")
	        break
	    # (5) Expansion (New Vectors Addition)
	    new_vecs = np.array(correction_vectors).T
	    V_next = np.hstack([V, new_vecs])
	    # 직교화 (QR)
	    Q, _ = la.qr(V_next, mode='economic')
	    # (6) Restart Logic
	    if Q.shape[1] > max_subspace:
	        # 부분공간이 너무 커지면 중요한 벡터(best Ritz vectors)만 남기고 축소
	        # print("--- Restarting Subspace ---")
	        best_vecs = V @ evecs_sub[:, :block_size]
	        V, _ = la.qr(best_vecs, mode='economic')
	    else:
	        V = Q
	# ---------------------------------------------------------
	# 5. 결과 분석 및 출력
	# ---------------------------------------------------------
	end_time = time.time()
	print(f"\nTotal Time: {end_time - start_time:.2f} sec")
	print("\n[3D Harmonic Oscillator Energy Levels (7-point Stencil)]")
	print(f"{'State':<6} | {'Calculated':<10} | {'Theory':<10} | {'Note'}")
	print("-" * 55)
	theory_levels = [1.5, 2.5, 2.5, 2.5, 3.5, 3.5] 
	for i in range(n_roots):
	    val = final_evals[i]
	    # n_roots가 이론값 리스트보다 길 경우 대비
	    th = theory_levels[i] if i < len(theory_levels) else 0.0
	    # 축퇴 여부 주석
	    note = ""
	    if abs(val - 1.5) < 0.1: note = "Ground"
	    elif abs(val - 2.5) < 0.1: note = "1st Excited (Degenerate x3)"
	    elif abs(val - 3.5) < 0.1: note = "2nd Excited"
	    print(f"{i:<6} | {val:.6f}   | {th:.1f}        | {note}")
solve_3d_harmonic_oscillator_complete()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
# ---------------------------------------------------------
# 1. 대규모 문제 생성기 (N=500)
# ---------------------------------------------------------
def create_massive_problem(n):
	np.random.seed(100) # 결과 재현을 위한 시드	
	# N x N 행렬 생성 (0~1 사이 난수)
	A = np.random.rand(n, n)
	# 수렴 보장을 위한 '강한 대각 지배' 만들기
	# 대각 성분 = (행의 절대값 합) + 추가값(n)
	# n을 더해주는 이유는 대각 지배력을 높여 수렴 속도를 확보하기 위함이다.
	row_sums = np.sum(np.abs(A), axis=1)
	np.fill_diagonal(A, row_sums + n/2) 
	# 정답을 미리 설정 (모두 1인 벡터)
	x_true = np.ones(n)
	# b 벡터 역계산
	b = np.dot(A, x_true)
	return A, b, x_true
# ---------------------------------------------------------
# 2. 솔버 함수 (N=500에서도 작동하도록 최적화)
# ---------------------------------------------------------
def solve_jacobi_fast(A, b, tol=1e-6, max_iter=2000):
    n = len(b)
    x = np.zeros(n)
    history = []	
    # 행렬 분리 (대각 D, 나머지 R) -> 벡터화 연산을 위해 필수
    D = np.diag(A)
    R = A - np.diag(D)
    start = time.time()
    for k in range(max_iter):
        # Jacobi 공식 (행렬 연산으로 한방에 처리) -> 파이썬에서 매우 빠름
        x_new = (b - np.dot(R, x)) / D
        diff = np.linalg.norm(x_new - x)
        history.append(diff)
        if diff < tol:
            return x_new, k+1, history, time.time() - start
        x = x_new
    return x, max_iter, history, time.time() - start
def solve_sor_python(A, b, omega, tol=1e-6, max_iter=2000):
    # 주의: 순수 파이썬 루프는 N이 커지면 매우 느림
    # Gauss-Seidel은 omega=1.0인 SOR과 같으므로 통합 구현
    n = len(b)
    x = np.zeros(n)
    history = []	
    start = time.time()
    for k in range(max_iter):
        x_old_iter = x.copy()
        # 순차적 의존성 때문에 for 루프 필수
        for i in range(n):
            # 최적화: 전체 행렬곱 대신 필요한 부분만 계산
            # s = sum(A[i,j]*x[j]) - A[i,i]*x[i]
            # np.dot을 쓰되 현재 업데이트된 x를 사용함
            row_val = np.dot(A[i, :], x) 
            s = row_val - A[i, i] * x[i]
            x_gs = (b[i] - s) / A[i, i]
            x[i] = (1 - omega) * x[i] + omega * x_gs
        diff = np.linalg.norm(x - x_old_iter)
        history.append(diff)
        if diff < tol:
            return x, k+1, history, time.time() - start
    return x, max_iter, history, time.time() - start
# ---------------------------------------------------------
# 3. 실행 (N=500)
# ---------------------------------------------------------
N = 500
print(f"Generating {N}x{N} System (Dense Matrix)...")
print("This might take a few seconds due to Python loops in GS/SOR...")
A, b, x_true = create_massive_problem(N)
tol = 1e-6
# 1. Jacobi (벡터화되어 빠름)
print("Running Jacobi...")
x_j, it_j, h_j, t_j = solve_jacobi_fast(A, b, tol)	
# 2. Gauss-Seidel (SOR with w=1.0)
print("Running Gauss-Seidel (Please wait)...")
x_g, it_g, h_g, t_g = solve_sor_python(A, b, omega=1.0, tol=tol)	
# 3. SOR (w=1.2)
print("Running SOR (w=1.2)...")
x_s, it_s, h_s, t_s = solve_sor_python(A, b, omega=1.1, tol=tol)
# ---------------------------------------------------------
# 4. 결과 비교
# ---------------------------------------------------------
print("\n" + "="*65)
print(f"{'Method':<15} | {'Iter':<5} | {'Time(s)':<8} | {'Accuracy (Max error)'}")
print("="*65)
print(f"{'Jacobi':<15} | {it_j:<5} | {t_j:.4f}   | {np.max(np.abs(x_j - x_true)):.2e}")
print(f"{'Gauss-Seidel':<15} | {it_g:<5} | {t_g:.4f}   | {np.max(np.abs(x_g - x_true)):.2e}")
print(f"{'SOR (w=1.1)':<15} | {it_s:<5} | {t_s:.4f}   | {np.max(np.abs(x_s - x_true)):.2e}")
print("="*65)
# 그래프 그리기
plt.figure(figsize=(12, 6))
plt.plot(h_j, label=f'Jacobi ({it_j} iters)', linestyle=':', linewidth=2)
plt.plot(h_g, label=f'Gauss-Seidel ({it_g} iters)', linestyle='--', linewidth=2)
astring=r'SOR $\omega=1.1$ ' + f'({it_s} iter)' 
plt.plot(h_s, '*', label=astring, linewidth=2, linestyle='--')
#plt.plot(h_s, label=f'SOR w=1.2 ({it_s} iters)', linestyle='-', linewidth=2)
plt.yscale('log')
plt.title(f'Convergence comparison (N={N})')
plt.xlabel('Iterations', fontsize=18)
plt.ylabel('Error (L2 norm, Log scale)', fontsize=18)
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.savefig('jgs.png')
plt.show()

In [ ]:
import numpy as np
import scipy.sparse as sparse
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
import time
def demo_krylov_gmres():
    # ---------------------------------------------------------
    # 1. 문제 설정: 대형 희소 행렬
    # ---------------------------------------------------------
    N = 2000 
    np.random.seed(42)
    A = sparse.rand(N, N, density=0.005, format='csr')
    A = A + sparse.eye(N) * 2.0
    x_true = np.random.rand(N)
    b = A.dot(x_true)
    # ---------------------------------------------------------
    # 2. GMRES Solver 실행
    # ---------------------------------------------------------
    print(f"Matrix Size: {N}x{N}, Non-zeros: {A.nnz}")
    print("GMRES 알고리즘 시작...")
    residuals = []
    # 콜백 함수: 현재 잔차(residual)의 크기를 리스트에 저장
    def callback_func(pr_norm):
        residuals.append(pr_norm)
    start_time = time.time()
    # [수정됨] 
    # 1. tol -> rtol (상대 오차)
    # 2. callback_type='pr_norm' 추가 (경고 해결 및 최신 표준)
    x_sol, info = spla.gmres(
        A, b, 
        rtol=1e-10, 
        callback=callback_func, 
        callback_type='pr_norm'
    )
    end_time = time.time()
    # ---------------------------------------------------------
    # 3. 결과 분석 및 시각화
    # ---------------------------------------------------------
    if info == 0:
        print(f"수렴 성공! (소요 시간: {end_time - start_time:.4f}초)")
        print(f"반복 횟수: {len(residuals)}")
        final_error = np.linalg.norm(x_sol - x_true) / np.linalg.norm(x_true)
        print(f"최종 상대 오차: {final_error:.2e}")
        plt.figure(figsize=(8, 6))
        # 잔차 그래프 그리기
        plt.semilogy(residuals, 'b-o', markersize=4, linewidth=1.5)
        plt.title(f"GMRES convergence (N={N})")
        plt.xlabel("Iteration", fontsize=18)
        plt.ylabel("Residual norm (Log scale)", fontsize=18)
        plt.grid(True, which="both", linestyle='--', alpha=0.7)
        plt.savefig('gmres.png') 
        plt.show()
    else:
        print("수렴 실패")
demo_krylov_gmres()	

In [ ]:
import numpy as np
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
import time
def solve_matrix_free_gmres():
    # ---------------------------------------------------------
    # 1. 문제 설정: 격자 크기
    # ---------------------------------------------------------
    N = 200  # 격자 한 변의 길이
    size = N * N # 전체 변수의 개수 (40,000개)
    # ---------------------------------------------------------
    # 2. Matrix-Free 연산자 정의 (핵심)
    # ---------------------------------------------------------
    # 행렬 A를 만드는 대신, 함수 mv(v)를 정의한다.
    # 이 함수는 입력 벡터 v를 받아서, 행렬곱 결과 Av를 리턴한다.
    # 물리적 의미: 2차원 라플라스 연산 (5-point stencil)
    # 4*u(i,j) - u(i-1,j) - u(i+1,j) - u(i,j-1) - u(i,j+1)
    def laplace_operator(v):
        # 1차원 벡터를 2차원 격자로 변환
        grid = v.reshape((N, N))
        result = np.zeros_like(grid)
        # 유한 차분법 (Finite Difference) 적용
        # 가장자리(Boundary)는 0으로 가정 (Dirichlet Condition)
        # 중심부 계산: 4*중앙 - 상 - 하 - 좌 - 우
        result[1:-1, 1:-1] = (4 * grid[1:-1, 1:-1] 
            - grid[0:-2, 1:-1]   # 상
            - grid[2:, 1:-1]     # 하
            - grid[1:-1, 0:-2]   # 좌
            - grid[1:-1, 2:])    # 우
        # 다시 1차원 벡터로 펴서 반환
        return result.ravel()
    # SciPy의 LinearOperator로 함수를 포장 (행렬인 척함)
    # shape: (행렬의 행 개수, 열 개수)
    A_op = spla.LinearOperator((size, size), matvec=laplace_operator)
    # ---------------------------------------------------------
    # 3. 정답 및 우변(b) 설정
    # ---------------------------------------------------------
    # 실제 정답 (가우시안 분포 형태의 언덕)
    x = np.linspace(-1, 1, N)
    X, Y = np.meshgrid(x, x)
    u_true_grid = np.exp(-10 * (X**2 + Y**2)) # 중앙이 볼록한 모양
    u_true = u_true_grid.ravel()
    # b = A * u_true (우리가 정의한 연산자로 b를 계산)
    b = A_op.matvec(u_true)
    # ---------------------------------------------------------
    # 4. GMRES 실행
    # ---------------------------------------------------------
    print(f"시스템 크기: {size} x {size} (행렬 생성 시 메모리 약 {size*size*8/1024**3:.2f} GB 필요)")
    print("Matrix-Free GMRES 시작...")
    start_time = time.time()
    residuals = []
    def callback(pr_norm):
        residuals.append(pr_norm)
    # A 대신 함수 덩어리(A_op)를 넣는다.
    u_sol, info = spla.gmres(A_op, b, rtol=1e-6, callback=callback, callback_type='pr_norm')
    end_time = time.time()
    # ---------------------------------------------------------
    # 5. 결과 확인
    # ---------------------------------------------------------
    if info == 0:
        print(f"수렴 성공! 시간: {end_time - start_time:.4f}초, 반복: {len(residuals)}")
        # 시각화
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        # (1) GMRES로 구한 해
        im1 = axes[0].imshow(u_sol.reshape(N, N), extent=[-1,1,-1,1], origin='lower')
        axes[0].set_title("GMRES solution")
        plt.colorbar(im1, ax=axes[0])
        # (2) 실제 정답
        im2 = axes[1].imshow(u_true_grid, extent=[-1,1,-1,1], origin='lower')
        axes[1].set_title("True solution")
        plt.colorbar(im2, ax=axes[1])
        # (3) 오차 (Residuals)
        axes[2].semilogy(residuals, 'r-o', markersize=3)
        axes[2].set_title("Convergence history")
        axes[2].set_xlabel("Iterations", fontsize=18)
        axes[2].set_ylabel("Residual norm", fontsize=18)
        axes[2].grid(True)
        plt.tight_layout()
        plt.savefig('gmres.png')
        plt.show()
    else:
        print("수렴 실패")
solve_matrix_free_gmres()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
def setup_problem(n):
	"""
	N x N 크기의 대칭 양의 정부호(SPD) 행렬 생성
	"""
	np.random.seed(42)
	# 1. 임의 행렬 생성 및 대칭화
	A = np.random.rand(n, n)
	A = 0.5 * (A + A.T)
	# 2. 대각 지배력 강화 (모든 알고리즘의 안정적 수렴을 위해)
	# 대각 성분을 행의 절대값 합보다 크게 설정
	row_sums = np.sum(np.abs(A), axis=1)
	np.fill_diagonal(A, row_sums + 1.0)
	# 3. 정답(x_true) 및 b 설정
	x_true = np.random.rand(n)
	b = A @ x_true
	return A, b, x_true
# --- Solver 구현 (오차 기록 기능 추가) ---
def jacobi(A, b, x0, x_true, max_iter=1000, tol=1e-12):
	n = len(b)
	x = x0.copy()
	D = np.diag(A)
	R = A - np.diag(D)
	error_history = []
	for k in range(max_iter):
	    # 오차 기록 (현재값 - 정답)
	    err = np.linalg.norm(x - x_true)
	    error_history.append(err)
	    if err < tol:
	        break
	    # Jacobi Update: x = (b - (L+U)x) / D
	    x = (b - R @ x) / D
	return x, error_history
def gauss_seidel(A, b, x0, x_true, max_iter=1000, tol=1e-12):
	n = len(b)
	x = x0.copy()
	error_history = []
	for k in range(max_iter):
	    err = np.linalg.norm(x - x_true)
	    error_history.append(err)
	    if err < tol:
	        break
	    for i in range(n):
	        # 벡터화된 내적 연산으로 속도 최적화
	        sigma = np.dot(A[i, :i], x[:i]) + np.dot(A[i, i+1:], x[i+1:])
	        x[i] = (b[i] - sigma) / A[i, i]
	return x, error_history
def sor(A, b, x0, x_true, omega, max_iter=1000, tol=1e-12):
	n = len(b)
	x = x0.copy()
	error_history = []
	for k in range(max_iter):
	    err = np.linalg.norm(x - x_true)
	    error_history.append(err)
	    if err < tol:
	        break
	    for i in range(n):
	        sigma = np.dot(A[i, :i], x[:i]) + np.dot(A[i, i+1:], x[i+1:])
	        x_gs = (b[i] - sigma) / A[i, i]
	        x[i] = (1 - omega) * x[i] + omega * x_gs
	return x, error_history
def conjugate_gradient(A, b, x0, x_true, max_iter=1000, tol=1e-12):
	x = x0.copy()
	r = b - A @ x
	p = r.copy()
	rs_old = r @ r
	error_history = []
	for k in range(max_iter):
    	# 오차 기록
	    err = np.linalg.norm(x - x_true)
	    error_history.append(err)
	    if err < tol: # 정답과의 오차가 기준치 미만이면 종료
	        break
	    # 잔차(residual) 기준 종료 조건 (일반적인 CG 종료 조건)
	    if np.sqrt(rs_old) < tol: 
	        break
	    Ap = A @ p
	    alpha = rs_old / (p @ Ap)
	    x = x + alpha * p
	    r = r - alpha * Ap
	    rs_new = r @ r
	    p = r + (rs_new / rs_old) * p
	    rs_old = rs_new
	return x, error_history
# --- 실행 및 시각화 ---
# 1. 문제 설정 (Python 속도를 고려하여 N=300 정도로 설정)
# N이 너무 크면 Gauss-Seidel/SOR 루프가 오래 걸려 그래프 그리는데 시간이 소요된다.
N = 300 
MAX_ITER = 200 # 그래프 확인용이므로 100회만 수행
A, b, x_true = setup_problem(N)
x0 = np.zeros(N)
print(f"문제 크기: {N}x{N}, 최대 반복: {MAX_ITER}")
print("계산 중...", end="")
# 2. 각 방법 실행
start = time.time()
_, hist_jacobi = jacobi(A, b, x0, x_true, max_iter=MAX_ITER)
print(".", end="")
_, hist_gs = gauss_seidel(A, b, x0, x_true, max_iter=MAX_ITER)
print(".", end="")
_, hist_sor = sor(A, b, x0, x_true, omega=1.2, max_iter=MAX_ITER) # 과완화 계수 1.2
print(".", end="")
_, hist_cg = conjugate_gradient(A, b, x0, x_true, max_iter=MAX_ITER)
print(" 완료!")
# 3. 그래프 그리기
plt.figure(figsize=(10, 6))
# 로그 스케일로 그려야 수렴 속도 차이를 명확히 볼 수 있다.
plt.plot(hist_jacobi, label='Jacobi', linestyle=':', linewidth=2, marker='x')
plt.plot(hist_gs, label='Gauss-Seidel', linestyle='--', linewidth=2, marker='+')
plt.plot(hist_sor, label='SOR (omega=1.2)', linestyle='-.', linewidth=2, marker='*')
plt.plot(hist_cg, label='Conjugate Gradient', linestyle='-', linewidth=2.5, 
    marker='o', markersize=4, markevery=5)
plt.yscale('log') # Y축 로그 스케일
plt.xlabel('Iterations', fontsize=18)
plt.ylabel('Error (L2 Norm) - Log scale', fontsize=18)
plt.title(f'Convergence comparison (N={N})')
plt.grid(True, which="both", ls="-", alpha=0.3)
plt.legend()
plt.savefig('cg.png')
plt.show()

In [ ]:
import numpy as np
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
def solve_bicgstab_advection_diffusion_complete():
    # ---------------------------------------------------------
    # 1. 문제 설정 (2D 이류-확산 방정식)
    # ---------------------------------------------------------
    # 방정식: -nu * Laplacian(u) + (vx, vy) * Gradient(u) = f
    # 특징: 이류항(Gradient) 때문에 행렬이 '비대칭'이 되어 CG를 사용할 수 없음 -> BiCGSTAB 사용    
    N = 50   # 격자 크기 (50x50)
    L = 1.0
    h = L / (N - 1)
    # 2D Grid 생성
    x = np.linspace(0, L, N)
    y = np.linspace(0, L, N)
    X, Y = np.meshgrid(x, y)
    # 물리 파라미터
    vx = 10.0   # x방향 풍속 (이류)
    vy = 10.0   # y방향 풍속 (이류)
    nu = 0.1    # 확산 계수
    # ---------------------------------------------------------
    # 2. LinearOperator (Matrix-Free A) 정의
    # ---------------------------------------------------------
    # 거대 행렬을 만들지 않고, 연산 A * x 의 결과만 반환하는 함수
    def matvec_op(u_vec):
        # 1D 벡터를 2D 격자로 변환
        u = u_vec.reshape((N, N))
        res = np.zeros_like(u)
        # 유한 차분법 (Central Difference)
        # 1) 확산 항 (Diffusion): u_xx + u_yy
        #    u[i+1, j] + u[i-1, j] + u[i, j+1] + u[i, j-1] - 4u[i, j]
        laplacian = (u[2:, 1:-1] + u[:-2, 1:-1] + 
                     u[1:-1, 2:] + u[1:-1, :-2] - 
                     4 * u[1:-1, 1:-1]) / h**2
        # 2) 이류 항 (Advection): vx * u_x + vy * u_y
        #    u_x approx (u[i, j+1] - u[i, j-1]) / 2h
        du_dx = (u[1:-1, 2:] - u[1:-1, :-2]) / (2*h)
        du_dy = (u[2:, 1:-1] - u[:-2, 1:-1]) / (2*h)
        # 3) 전체 연산자 적용: -nu * Lap + v * Grad
        #    경계(테두리)는 0으로 고정(Dirichlet BC)되므로 계산에서 제외 (res=0 유지)
        res[1:-1, 1:-1] = -nu * laplacian + (vx * du_dx + vy * du_dy)
        return res.ravel() # 다시 1D 벡터로 풀어서 반환
    # SciPy Solver가 사용할 수 있는 Operator 객체 생성
    A = spla.LinearOperator((N*N, N*N), matvec=matvec_op)
    # ---------------------------------------------------------
    # 3. 우변 벡터 b (Source Term) 생성
    # ---------------------------------------------------------
    # 중앙에 위치한 열원(Source)
    # 가우시안 분포 형태의 오염원이나 열원을 가정
    f = np.exp(-100 * ((X-0.5)**2 + (Y-0.5)**2))
    b = f.ravel()
    # ---------------------------------------------------------
    # 4. BiCGSTAB 실행 (SciPy 활용)
    # ---------------------------------------------------------
    print(f"BiCGSTAB solving 시작 (Grid: {N}x{N}, Unknowns: {N*N})...")
    # [수정 완료] tol 대신 rtol 사용
    # rtol: 상대 오차 (Relative Tolerance)
    x_sol, info = spla.bicgstab(A, b, rtol=1e-6)
    if info == 0:
        print(">>> 수렴 성공! (Converged)")
    elif info > 0:
        print(f">>> 수렴 실패 (목표 정밀도 도달 못함, info={info})")
    else:
        print(">>> 수렴 실패 (입력 오류)")
    # ---------------------------------------------------------
    # 5. 결과 시각화
    # ---------------------------------------------------------
    u_final = x_sol.reshape((N, N))
    plt.figure(figsize=(12, 5))
    # 왼쪽: 계산된 해 (u)
    plt.subplot(1, 2, 1)
    plt.imshow(u_final, origin='lower', extent=[0,1,0,1], cmap='jet')
    plt.colorbar(label='Concentration u(x,y)')
    plt.title(f"BiCGSTAB solution\n(Advection: vx={vx}, vy={vy})")
    plt.xlabel("x")
    plt.ylabel("y")    
    # 오른쪽: 오염원/열원 (Source f)
    plt.subplot(1, 2, 2)
    plt.imshow(f, origin='lower', extent=[0,1,0,1], cmap='gray_r')
    plt.colorbar(label='Source f')
    plt.title("Source term input")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.tight_layout()
    plt.savefig('bicgstab.png')
    plt.show()
# 프로그램 실행
if __name__ == "__main__":
    solve_bicgstab_advection_diffusion_complete()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
def setup_scaled_problem(n):
    """
    Jacobi Preconditioner가 가장 잘 작동하는 상황을 만든다.
    변수들의 스케일(크기)을 강제로 다르게 만들어 조건수를 악화시킨다.
    """
    np.random.seed(42)
    # 1. 기본적으로 성질이 좋은(Well-conditioned) 무작위 대칭 행렬 생성
    A_base = np.random.rand(n, n)
    A_base = 0.5 * (A_base + A_base.T) + n * np.eye(n) # 대각 지배적
    # 2. 스케일링 행렬 D 생성 (대각 성분이 1부터 100까지 극단적으로 변함)
    scales = np.linspace(1, 100, n) 
    D_scale = np.diag(np.sqrt(scales)) 
    # 3. 행렬 A를 강제로 찌그러뜨림 (Ill-conditioned 생성)
    A = D_scale @ A_base @ D_scale
    x_true = np.random.rand(n)
    b = A @ x_true
    return A, b, x_true
# --- Solver ---
def conjugate_gradient(A, b, x0, x_true, max_iter=1000, tol=1e-8):
    x = x0.copy()
    r = b - A @ x
    p = r.copy()
    rs_old = r @ r
    history = []
    for k in range(max_iter):
        history.append(np.linalg.norm(x - x_true))
        if np.sqrt(rs_old) < tol:
            break
        Ap = A @ p
        alpha = rs_old / (p @ Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        rs_new = r @ r
        p = r + (rs_new / rs_old) * p
        rs_old = rs_new
    return history
def preconditioned_cg(A, b, x0, x_true, max_iter=1000, tol=1e-8):
    x = x0.copy()
    r = b - A @ x
    # Jacobi Preconditioner
    M_diag = np.diag(A) 
    z = r / M_diag # z = M^-1 r
    p = z.copy()
    rz_old = r @ z
    history = []
    for k in range(max_iter):
        history.append(np.linalg.norm(x - x_true))
        if np.linalg.norm(r) < tol:
            break
        Ap = A @ p
        alpha = rz_old / (p @ Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        z = r / M_diag 
        rz_new = r @ z
        beta = rz_new / rz_old
        p = z + beta * p
        rz_old = rz_new
    return history
# --- 실행 및 비교 ---
N = 500
A, b, x_true = setup_scaled_problem(N)
x0 = np.zeros(N)
cond_val = np.linalg.cond(A)
print(f"Condition number (조건수): {cond_val:.2f}")
hist_cg = conjugate_gradient(A, b, x0, x_true, max_iter=N)
hist_pcg = preconditioned_cg(A, b, x0, x_true, max_iter=N)
plt.figure(figsize=(10, 6))
plt.plot(hist_cg, label=f'Standard CG', linestyle='--', color='red', linewidth=2)
plt.plot(hist_pcg, label=f'Preconditioned CG', linestyle='-', color='blue', linewidth=2)
plt.yscale('log')
plt.xlabel('Iterations', fontsize=18)
plt.ylabel('Error (L2 norm) - Log scale', fontsize=18)
plt.title(rf'Effect of Jacobi preconditioning (Condition number $\approx$ {int(cond_val)})')
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.savefig('pcg.png')
plt.show()	

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def thomas_solver(a, b, c, d):
    """
    토마스 알고리즘을 사용하여 삼중 대각 선형 시스템을 푼다.
    """
    n = len(d)
    cp = np.zeros(n)
    dp = np.zeros(n)
    x = np.zeros(n)
    # Forward Elimination (전방 소거)
    # 첫 번째 방정식 처리
    cp[0] = c[0] / b[0]
    dp[0] = d[0] / b[0]
    # 나머지 방정식 처리
    for i in range(1, n):
        denom = b[i] - a[i] * cp[i-1]
        if i < n-1: # 마지막 방정식에서는 c[i]를 계산할 필요 없음
            cp[i] = c[i] / denom
        dp[i] = (d[i] - a[i] * dp[i-1]) / denom
    # Backward Substitution (후방 대입)
    x[n-1] = dp[n-1]
    for i in range(n-2, -1, -1):
        x[i] = dp[i] - cp[i] * x[i+1]
    return x
# --- 경계값 문제 설정 및 토마스 알고리즘 적용 ---
# 문제 설정
L = 1.0  # 핀의 길이 (m)
n_internal_nodes = 3 # 내부 노드 수
total_nodes = n_internal_nodes + 2 # 총 노드 수 (경계 포함)
# 노드 간격
dx = L / (total_nodes - 1)
print(f"노드 간격 (dx): {dx:.2f}m")
# 미분 방정식 계수
# d^2T/dx^2 - T = 0  -> T_i-1 - (2 + dx^2)T_i + T_i+1 = 0
main_diag_val = -(2 + dx**2) # -2.0625
# 경계 조건
T_left = 100 # 왼쪽 끝 온도 (°C)
T_right = 200 # 오른쪽 끝 온도 (°C)
# 토마스 알고리즘 입력 벡터 구성 (내부 노드만 고려)
# 내부 노드는 T1, T2, T3 (총 3개)
# a, b, c 벡터의 길이는 n_internal_nodes 여야 한다.
a = [1] * n_internal_nodes # 하단 대각 (첫 번째 a[0]은 실제로는 사용되지 않음, dummy)
b = [main_diag_val] * n_internal_nodes # 중앙 대각
c = [1] * n_internal_nodes # 상단 대각 (마지막 c[n-1]은 실제로는 사용되지 않음, dummy)
# d 벡터 (우변) 구성
d = [0] * n_internal_nodes
d[0] -= T_left # 첫 번째 내부 노드에 왼쪽 경계 조건 영향
d[n_internal_nodes-1] -= T_right # 마지막 내부 노드에 오른쪽 경계 조건 영향
# 토마스 알고리즘으로 내부 온도 계산
internal_temperatures = thomas_solver(a, b, c, d)
# 전체 온도 분포 (경계 조건 포함)
# x축 좌표 생성
x_coords = np.linspace(0, L, total_nodes)
full_temperatures = np.concatenate(([T_left], internal_temperatures, [T_right]))
print("\n--- 계산된 온도 분포 ---")
for i, temp in enumerate(full_temperatures):
    print(f"x = {x_coords[i]:.2f}m: T = {temp:.2f}°C")
# --- Matplotlib을 이용한 시각화 ---
plt.figure(figsize=(10, 6)) # 그래프 크기 설정
plt.plot(x_coords, full_temperatures, marker='o', linestyle='-', color='b', 
label='Numerical solution(Thomas algorithm)')
plt.title('Temperature distribution along the fin')
plt.xlabel('Position (m)', fontsize=16)
plt.ylabel('Temperature (°C)', fontsize=16)
plt.grid(True) # 격자선 표시
plt.legend() # 범례 표시
plt.show()
# --- 노드 수 증가 시도 ---
print("\n--- 노드 수 증가 (n_internal_nodes = 100) ---")
n_internal_nodes_large = 100
total_nodes_large = n_internal_nodes_large + 2
dx_large = L / (total_nodes_large - 1)
main_diag_val_large = -(2 + dx_large**2)
a_large = [1] * n_internal_nodes_large
b_large = [main_diag_val_large] * n_internal_nodes_large
c_large = [1] * n_internal_nodes_large
d_large = [0] * n_internal_nodes_large
d_large[0] -= T_left
d_large[n_internal_nodes_large-1] -= T_right
internal_temperatures_large = thomas_solver(a_large, b_large, c_large, d_large)
x_coords_large = np.linspace(0, L, total_nodes_large)
full_temperatures_large = np.concatenate(([T_left], internal_temperatures_large, [T_right]))
plt.figure(figsize=(10, 6))
plt.plot(x_coords_large, full_temperatures_large, linestyle='-', color='r', 
label=f'Numerical solution(N={n_internal_nodes_large})')
plt.title('Temperature distribution along the fin (More nodes)')
plt.xlabel('Position (m)', fontsize=16)
plt.ylabel('Temperature (°C)', fontsize=16)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 예시 데이터 (X: 입력 변수, Y: 출력 변수)
# 예: X = [1, 2, 3, 4, 5], Y = [1, 2, 1.9, 4.2, 5.1]
X = np.array([1, 2, 3, 4, 5])
Y = np.array([1, 2, 1.9, 4.2, 5.1])

# 입력 데이터 행렬 A 구성 (선형 회귀에서는 1을 추가하여 상수항을 포함)
A = np.vstack([X, np.ones_like(X)]).T

# 최소 제곱법을 통한 회귀 계수(beta) 계산
# beta = (A^T * A)^(-1) * A^T * Y
beta = np.linalg.inv(A.T @ A) @ A.T @ Y

# 회귀 직선 방정식
slope, intercept = beta

# 예측 값 계산 (회귀 직선 상의 Y 값)
Y_pred = A @ beta

# 결과 출력
print(f"기울기 (slope): {slope}")
print(f"절편 (intercept): {intercept}")

# 원본 데이터와 회귀 직선 시각화
plt.scatter(X, Y, color='blue', label='Original data')
plt.plot(X, Y_pred, color='red', label='Fitted line')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.show()

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# 1. 예제 데이터 (x, y) 정의
# x: 독립 변수
x_data = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5])
# y: 관측된 종속 변수 (노이즈 포함)
y_data = np.array([1.1, 1.8, 2.5, 2.1, 1.5, 1.8, 3.3, 6.5, 12.0, 20.8])

# 2. 다중 매개변수를 갖는 모델 함수 정의 (5차 다항식)
# 매개변수 p: [beta5, beta4, beta3, beta2, beta1, beta0] 순서
def polynomial_5th_order(x, b5, b4, b3, b2, b1, b0):
    """5차 다항식 모델: y = b5*x^5 + b4*x^4 + b3*x^3 + b2*x^2 + b1*x + b0"""
    return (b5 * x**5 + b4 * x**4 + b3 * x**3 + 
            b2 * x**2 + b1 * x + b0)

# 3. 비선형 최소자승법 (NLLS) 적용
# 매개변수 6개에 대한 초기 추정값 설정. 
# 0으로 설정하면 curve_fit이 수렴하지 못할 수 있으므로, 적절한 작은 값이나 1을 사용한다.
initial_guess = [0.1, -0.5, 1.0, -1.0, 1.0, 1.0] # 6개의 초기값

print(f"매개변수 개수: {len(initial_guess)}개\n")

try:
    # curve_fit 실행
    popt, pcov = curve_fit(
        f=polynomial_5th_order,
        xdata=x_data,
        ydata=y_data,
        p0=initial_guess,  # 6개의 초기 추정값
        maxfev=5000 # 반복 횟수를 늘려 수렴 가능성 높임
    )

    # 최적화된 매개변수 추출
    b5_opt, b4_opt, b3_opt, b2_opt, b1_opt, b0_opt = popt

    print("## 다중 매개변수 (5차 다항식) 최적화 결과")
    print(f"b5 (x^5 계수): {b5_opt:.4f}")
    print(f"b4 (x^4 계수): {b4_opt:.4f}")
    print(f"b3 (x^3 계수): {b3_opt:.4f}")
    print(f"b2 (x^2 계수): {b2_opt:.4f}")
    print(f"b1 (x^1 계수): {b1_opt:.4f}")
    print(f"b0 (상수항): {b0_opt:.4f}")

    # 4. 결과 시각화
    x_fit = np.linspace(min(x_data), max(x_data), 100)
    y_fit = polynomial_5th_order(x_fit, *popt) # popt의 모든 요소를 인수로 전달

    plt.figure(figsize=(10, 6))
    plt.scatter(x_data, y_data, label='Original data (Observation)', color='red', marker='o')
    plt.plot(x_fit, y_fit, label='Fitted 5th Order Polynomial (Best model)', color='blue')
    plt.title('Non-linear least squares fitting: 5th order polynomial')
    plt.xlabel('X', fontsize=18)
    plt.ylabel('Y', fontsize=18)
    plt.legend()
    plt.grid(True)
    plt.show()
except RuntimeError as e:
    print(f"\n최적화에 실패했다: {e}")
    print("다중 매개변수 문제에서는 수렴 실패가 흔하다. 초기 추정값(p0)이나 maxfev를 조정해 보세요.")

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# 1. 예제 데이터 (x, y) 정의
# x: 시간 또는 독립 변수
x_data = np.array([0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])
# y: 관측된 종속 변수 (노이즈 포함)
y_data = np.array([0.8, 1.4, 2.5, 4.1, 6.5, 8.0, 9.0, 9.5, 9.8])

# 2. 비선형 모델 함수 정의 (로지스틱 함수)
# 매개변수 p: [L, k, x0]
def logistic_model(x, L, k, x0):
    """로지스틱 함수 모델: y = L / (1 + exp(-k * (x - x0)))"""
    return L / (1 + np.exp(-k * (x - x0)))

# 3. 비선형 최소자승법 (NLLS) 적용
# initial_guess: 최적화를 위한 초기 추정값.
# L (최대값)은 데이터의 최대값 근처 (10 근처)
# k (성장률)는 양수 (1 근처)
# x0 (중앙값)은 y가 L/2가 되는 x값 (4 근처)
initial_guess = [10.0, 1.0, 4.0]

try:
    # curve_fit 실행
    popt, pcov = curve_fit(
        f=logistic_model,
        xdata=x_data,
        ydata=y_data,
        p0=initial_guess,  # 초기 추정값
        method='lm'        # Levenberg-Marquardt 알고리즘 명시 (기본값)
    )

    # 최적화된 매개변수 추출
    L_opt, k_opt, x0_opt = popt

    print("## 로지스틱 성장 모델 최적화 결과")
    print(f"최적 최대 포화값 L: {L_opt:.4f}")
    print(f"최적 성장률 k: {k_opt:.4f}")
    print(f"최적 중앙값 x0: {x0_opt:.4f}")
    print("\n최적 모델 수식:")
    print(f"y = {L_opt:.4f} / (1 + exp(-{k_opt:.4f} * (x - {x0_opt:.4f})))")

    # 4. 결과 시각화
    x_fit = np.linspace(min(x_data), max(x_data), 100)
    y_fit = logistic_model(x_fit, L_opt, k_opt, x0_opt)

    plt.figure(figsize=(10, 6))
    plt.scatter(x_data, y_data, label='Original data (Observation)', color='red', marker='o')
    plt.plot(x_fit, y_fit, label=f'Fitted Logistic Curve (Best model)', color='blue')
    plt.title('Non-linear least squares fitting: Logistic growth model')
    plt.xlabel('X', fontsize=18)
    plt.ylabel('Y', fontsize=18)
    plt.legend()
    plt.grid(True)
    plt.show()
except RuntimeError as e:
    print(f"\n최적화에 실패했다: {e}")
    print("초기 추정값 (p0)을 조정해 보세요.")

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# 1. 예제 데이터 (x, y) 정의
# x: 시간 또는 독립 변수
x_data = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5])
# y: 관측된 종속 변수 (노이즈 포함)
y_data = np.array([9.8, 7.2, 6.1, 4.4, 3.2, 2.5])

# 2. 비선형 모델 함수 정의
# 이 함수는 curve_fit에 의해 최적화될 모델이다.
# 매개변수 p: [A, k]
def exponential_model(x, A, k):
    """지수 함수 모델: y = A * exp(-k * x)"""
    return A * np.exp(-k * x)

# 3. 비선형 최소자승법 (NLLS) 적용
# initial_guess: 최적화를 위한 초기 추정값. NLLS에서 중요하다.
# 데이터 플롯을 보고 대략 A는 10 근처, k는 0.5 근처로 추정한다.
initial_guess = [10.0, 0.5]

# curve_fit 실행: 데이터를 가장 잘 설명하는 매개변수를 찾는다.
# popt: Optimal parameters (최적 매개변수)
# pcov: Covariance matrix (공분산 행렬) - 매개변수 추정치의 불확실성을 평가하는 데 사용됨
popt, pcov = curve_fit(
    f=exponential_model,
    xdata=x_data,
    ydata=y_data,
    p0=initial_guess  # 초기 추정값
)

# 최적화된 매개변수 추출
A_opt, k_opt = popt

print("## 최적화 결과 (비선형 최소자승법)")
print(f"최적 진폭 A: {A_opt:.4f}")
print(f"최적 감쇠율 k: {k_opt:.4f}")
print(f"최적 모델: y = {A_opt:.4f} * exp(-{k_opt:.4f} * x)")

# 4. 결과 시각화
x_fit = np.linspace(min(x_data), max(x_data), 100)
y_fit = exponential_model(x_fit, A_opt, k_opt)

plt.figure(figsize=(10, 6))
plt.scatter(x_data, y_data, label='Original data (Observation)', color='red', marker='o')
plt.plot(x_fit, y_fit, label=f'Fitted curve (Best model)\ny = {A_opt:.2f}exp(-{k_opt:.2f}x)', color='blue')
plt.title('Non-linear least squares fitting')
plt.xlabel('X', fontsize=18)
plt.ylabel('Y', fontsize=18)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def pure_python_gaussian_elimination(A, b):
    """
    순수 Python 리스트를 사용하여 가우스 소거법으로 Ax = b의 해를 구한다.
    A: 계수 행렬 (list of lists)
    b: 상수 벡터 (list)
    """
    n = len(b)
    # 확대계수행렬(M) 생성: A와 b를 합친다.
    M = [A[i] + [b[i]] for i in range(n)]
    # 1. 전방 소거법 (Forward Elimination) - 행사다리꼴로 만들기
    for k in range(n): # 피벗 행 (pivot row)
        # 피벗이 0인 경우 행 교환 (실제 구현에서는 필수적이지만, 여기서는 단순화)
        # if M[k][k] == 0:
        #     # 더 큰 피벗을 가진 행과 교환하는 코드가 필요하지만 생략
        for i in range(k + 1, n): # 피벗 아래 행들
            # 배율 인수(multiplier) 계산
            multiplier = M[i][k] / M[k][k]
            # 현재 행(i)을 갱신: (행 i) = (행 i) - multiplier * (피벗 행 k)
            for j in range(k, n + 1):
                M[i][j] -= multiplier * M[k][j]
    # 2. 후진 대입법 (Back Substitution) - 해(x) 구하기
    x = [0] * n # 해 벡터 초기화
    for i in range(n - 1, -1, -1): # 마지막 행부터 거꾸로 계산
        # Mi, n은 확대계수행렬의 마지막 열(상수 b)
        sum_of_terms = M[i][n] 
        # 이미 구한 x 값들을 이용하여 우변에서 해당 항들을 뺀다.
        for j in range(i + 1, n):
            sum_of_terms -= M[i][j] * x[j]
        # 최종적으로 x[i]를 계산
        x[i] = sum_of_terms / M[i][i]
    return x

# 예제 행렬 A와 벡터 b
A = [
    [1, 1, 1],
    [2, 4, -3],
    [3, 6, -5]
]
b = [9, 1, 0]

solution = pure_python_gaussian_elimination(A, b)
print(f"**순수 Python 결과**")
print(f"해 (x, y, z): {solution}")
# 예상 결과: [7.0, -1.0, 3.0]

In [ ]:
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt

def qr_algorithm_simulation(A, max_iter=50):
    """
    QR 알고리즘을 통해 행렬이 상삼각 행렬(슈어 폼)로 변하는 과정을 시각화한다.
    """
    Ak = A.copy()
    n = Ak.shape[0]
    
    print(f"--- Initial Matrix A ---\n{np.round(Ak, 2)}\n")
    
    # 반복 과정 기록
    off_diagonal_history = []
    
    for i in range(max_iter):
        # 1. QR 분해
        Q, R = np.linalg.qr(Ak)
        
        # 2. 순서 바꿔 곱하기 (Update)
        Ak = R @ Q
        
        # 대각선 아래 성분들의 크기 합 (수렴 확인용)
        # 하삼각 부분만 추출 (k=-1)
        lower_tri = np.tril(Ak, k=-1) 
        norm = np.linalg.norm(lower_tri)
        off_diagonal_history.append(norm)
        
        if i < 5 or i % 10 == 0:
             print(f"[Iter {i+1}] Off-diagonal Norm: {norm:.5f}")

    print(f"\n--- Result after {max_iter} iterations (Approx. Schur Form) ---")
    print(np.round(Ak, 3))
    
    print("\n--- Diagonal Elements (Eigenvalues approximation) ---")
    print(np.diagonal(Ak))
    
    return Ak, off_diagonal_history

# --- 실행 예제 ---

# 1. 예제 행렬 생성 (실수 고유값을 가지도록 대칭행렬 + 노이즈 사용)
# 랜덤 행렬
np.random.seed(42)
M = np.random.randn(4, 4)
# 고유값이 실수가 되도록 대칭행렬로 변환 (설명 편의상)
A = M + M.T 

# 2. 직접 구현한 QR 알고리즘 실행
schur_approx, history = qr_algorithm_simulation(A)

# 3. SciPy의 정식 Schur 분해와 비교
T, Z = scipy.linalg.schur(A)
print(f"\n--- SciPy True Schur Form (Target) ---")
print(np.round(T, 3))
print("\nTrue Eigenvalues (Diagonal of T):")
print(np.diagonal(T))

# --- 시각화: 대각선 아래 성분들이 사라지는 과정 ---
plt.figure(figsize=(8, 5))
plt.plot(history, 'o-', label='Off-diagonal Norm')
plt.title("Convergence of QR Algorithm to Schur form")
plt.xlabel("Iteration")
plt.ylabel("Norm of Lower Triangular Part")
plt.yscale('log') # 로그 스케일로 보면 선형 수렴(Linear Convergence) 확인 가능
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import numpy as np

# 계수 행렬 A 정의
A_np = np.array([
    [1, 1, 1],
    [2, 4, -3],
    [3, 6, -5]
])
# 상수 벡터 b 정의
b_np = np.array([9, 1, 0])

# NumPy의 선형 대수 솔버 사용
# 이 함수는 가우스 소거법의 원리를 이용한 효율적인 방법으로 해를 찾는다.
try:
    x_np = np.linalg.solve(A_np, b_np)
    print("\n**NumPy linalg.solve 결과**")
    print(f"행렬 A:\n{A_np}")
    print(f"벡터 b: {b_np}")
    print(f"해 (x, y, z): {x_np}")
    # 결과 검증: Ax = b
    print(f"검증 (Ax): {A_np @ x_np}")
except np.linalg.LinAlgError as e:
    print(f"\n오류 발생: {e}")
    print("행렬 A가 역행렬을 가지지 않거나 (특이행렬), 해가 유일하지 않을 수 있다.")

In [ ]:
from scipy.linalg import lu

# P: 치환 행렬 (Permutation Matrix), L: 하삼각 행렬, U: 상삼각 행렬
P, L, U = lu(A_np)
print("\n**Scipy LU 분해 결과**")
print(f"상삼각 행렬 (U):\n{U}")

In [ ]:
import numpy as np
from scipy.linalg import lu, lu_solve, lu_factor

# 1. 행렬 및 벡터 정의
A = np.array([
    [2.0, 1.0, 1.0],
    [4.0, 1.0, 0.0],
    [-2.0, 2.0, 1.0]
])
b = np.array([4.0, 6.0, 2.0])

print("--- 입력 행렬 및 벡터 ---")
print("행렬 A:\n", A)
print("벡터 b:\n", b)

# --- 2. LU 분해 단계 (PA = LU) ---
# lu_factor를 사용하여 L과 U 행렬 및 피벗 인덱스(piv)를 얻는다.
# L과 U는 하나의 행렬에 압축되어 저장된다.
lu_and_piv = lu_factor(A)
L, U, piv = lu(A) # L, U, P를 명시적으로 얻는 방법 (확인용)

print("\n--- LU 분해 결과 (확인용) ---")
# L은 하삼각, U는 상삼각 행렬이다. P는 행 순서를 나타내는 치환 행렬이다.
print("L (하삼각):\n", L.round(4))
print("U (상삼각):\n", U.round(4))

# --- 3. 해 계산 단계 ---
# lu_solve를 사용하여 두 대입 과정을 한 번에 처리한다.
# lu_solve는 내부적으로 L*y = P*b (전방 대입)와 U*x = y (후방 대입)을 수행한다.
x = lu_solve(lu_and_piv, b)

print("\n--- 최종 해 (x) ---")
print("x_1, x_2, x_3:", x.round(4))

# --- 검증 ---
# A @ x 가 b와 동일한지 확인
b_check = A @ x
print("\n--- 검증 (A @ x) ---")
print("b_calculated:", b_check.round(4))
print("b_original:  ", b.round(4))

# 해석적 해: x1=1, x2=2, x3=0

In [ ]:
import numpy as np

# 2x2 정방 행렬 정의
A = np.array([[4, 2], 
              [1, 3]])

# np.linalg.eig() 함수를 사용하여 고윳값과 고유벡터 계산
# w: 고윳값(eigenvalues), v: 고유벡터(eigenvectors)
w, v = np.linalg.eig(A)

print("--- 입력 행렬 A ---")
print(A)

print("\n--- 고윳값 (Eigenvalues) ---")
print(w)

print("\n--- 고유벡터 (Eigenvectors) ---")
# 각 열(column)이 해당 고윳값에 대응하는 고유벡터이다.
# w[0] = 5.0 에 대응하는 고유벡터는 v[:, 0] = [0.8944..., 0.4472...] 이다.
# w[1] = 2.0 에 대응하는 고유벡터는 v[:, 1] = [-0.7071..., 0.7071...] 이다.
print(v)

# 첫 번째 고윳값과 고유벡터 추출
lambda_1 = w[0]
v_1 = v[:, 0]

print("--- 첫 번째 고윳값과 고유벡터 ---")
print(f"고윳값 (λ1): {lambda_1}")
print(f"고유벡터 (v1):\n{v_1}")

# A * v1 계산 (Av)
Av_1 = A @ v_1 

# lambda_1 * v1 계산 (λv)
lambda_v_1 = lambda_1 * v_1

print("\n--- Av1 계산 결과 ---")
print(Av_1)

print("\n--- λ1*v1 계산 결과 ---")
print(lambda_v_1)

# 두 결과가 거의 같은지 확인
print("\n--- Av1 == λ1*v1 확인 (오차 허용 범위 내) ---")
print(np.allclose(Av_1, lambda_v_1))

import numpy as np

# Hermitian 행렬 A 정의
# i는 파이썬에서 1j로 표현된다.
A = np.array([[2, 3 + 1j], 
              [3 - 1j, -1]])

print("--- 행렬 A ---")
print(A)

# 켤레 전치 (Conjugate Transpose, A†) 계산
# np.conj(A).T 또는 A.T.conj() 또는 A.conj().T
A_dag = A.conj().T

print("\n--- 켤레 전치 (A†) ---")
print(A_dag)

# Hermitian 성질 검증 (A == A†)
# np.allclose는 부동 소수점 오차를 허용하면서 두 행렬이 같은지 비교한다.
is_hermitian = np.allclose(A, A_dag)

print(f"\nA와 A†가 같은가? (Hermitian): {is_hermitian}")

# A의 고윳값(eigenvalues) 계산
eigenvalues = np.linalg.eigvals(A)

print("--- 행렬 A의 고윳값 ---")
print(eigenvalues)

# 고윳값의 허수부 확인 (매우 작은 값은 부동 소수점 오차로 간주)
imag_parts = np.imag(eigenvalues)
is_all_real = np.allclose(imag_parts, 0.0)

print(f"\n고윳값의 허수부가 0에 가까운가? (모두 실수인가?): {is_all_real}")

import numpy as np

# 실수 대칭 행렬 정의
A = np.array([[6, 1, 1], 
              [1, 5, 2], 
              [1, 2, 5]])

# np.linalg.eigh 사용
# w: 고윳값(eigenvalues), v: 고유벡터(eigenvectors)
w, v = np.linalg.eigh(A)

print("--- 입력 대칭 행렬 A ---")
print(A)

print("\n--- 고윳값 (Eigenvalues) ---")
# 모든 고윳값은 실수이다.
print(w)

print("\n--- 고유벡터 (Eigenvectors) ---")
# v의 각 열은 대응하는 고윳값에 대한 고유벡터이다.
# 대칭/Hermitian 행렬의 고유벡터는 정규 직교(orthonormal)한다.
print(v)

# 복소수 Hermitian 행렬 H 정의 (i는 1j로 표현)
H = np.array([[3, 2 - 1j], 
              [2 + 1j, 4]])

# np.linalg.eigh 사용
w_h, v_h = np.linalg.eigh(H)

print("\n--- 입력 Hermitian 행렬 H ---")
print(H)

print("\n--- 고윳값 (Eigenvalues) ---")
# 고윳값은 실수이다. (매우 작은 허수부는 부동 소수점 오차로 간주)
print(w_h)

print("\n--- 고유벡터 (Eigenvectors) ---")
# 고유벡터는 복소수일 수 있다.
print(v_h)

import numpy as np

# 3x2 행렬 A 정의 (정방 행렬이 아님)
A = np.array([[1, 1], 
              [0, 1], 
              [1, 0]])

print("--- 입력 행렬 A (3x2) ---")
print(A)

# np.linalg.svd() 함수를 사용하여 SVD 수행
# U: 좌측 특이 벡터 행렬
# s: 특잇값 (Sigma 행렬의 대각 성분)
# Vt: 우측 특이 벡터 행렬의 전치 (V^T)
U, s, Vt = np.linalg.svd(A)

print("\n--- U 행렬 (좌측 특이 벡터, 3x3) ---")
print(U)

print("\n--- 특잇값 s (2개) ---")
print(s)

print("\n--- Vt 행렬 (우측 특이 벡터의 전치, 2x2) ---")
print(Vt)

# A = U * Sigma * Vt 검증을 위한 Sigma 행렬 재구성
# Sigma는 A와 같은 크기인 3x2 행렬로 만들어야 한다.
Sigma = np.zeros(A.shape)
# 특잇값 s를 Sigma의 주대각선에 배치
Sigma[:A.shape[1], :A.shape[1]] = np.diag(s)

print("\n--- 재구성된 Sigma 행렬 (3x2) ---")
print(Sigma)

# U * Sigma * Vt 계산 및 검증
A_reconstructed = U @ Sigma @ Vt

print("\n--- U * Sigma * Vt 재구성 결과 ---")
print(A_reconstructed)

# 원래 행렬 A와 재구성된 행렬이 같은지 확인
print("\n--- 재구성된 행렬이 원본과 같은가? (오차 허용 범위 내) ---")
print(np.allclose(A, A_reconstructed))

import numpy as np

# 3x3 정방 행렬 정의
A = np.array([[12, -51, 4], 
              [6, 167, -68], 
              [-4, 24, -41]])

print("--- 입력 행렬 A ---")
print(A)

# QR 분해 수행: Q, R = np.linalg.qr(A)
Q, R = np.linalg.qr(A)

print("\n--- Q 행렬 (직교 행렬) ---")
print(Q)

print("\n--- R 행렬 (상삼각 행렬) ---")
print(R)

# A = Q @ R 검증
A_reconstructed = Q @ R
print("\n--- Q @ R 재구성 결과 ---")
print(A_reconstructed)

# 원래 행렬 A와 재구성된 행렬이 같은지 확인
print("\n--- A == Q @ R 확인 (오차 허용 범위 내) ---")
print(np.allclose(A, A_reconstructed))

# Q의 전치 행렬 Q_T
Q_T = Q.T

# Q^T @ Q 계산
QT_Q = Q_T @ Q

print("\n--- Q^T @ Q 결과 ---")
# 결과가 단위 행렬 I (주대각선은 1, 나머지는 0)과 거의 같은지 확인한다.
print(QT_Q)

# 단위 행렬과 같은지 검증
is_orthogonal = np.allclose(QT_Q, np.eye(A.shape[0]))
print(f"\nQ는 직교 행렬인가? (Q^T Q = I): {is_orthogonal}")

# 4x3 직사각형 행렬 정의
B = np.array([[1, 2, 3], 
              [4, 5, 6], 
              [7, 8, 9],
              [10, 11, 12]])

Q_B, R_B = np.linalg.qr(B)

print("\n--- 입력 행렬 B (4x3) ---")
print(B)

print("\n--- Q 행렬 (4x4) ---")
print(Q_B)

print("\n--- R 행렬 (4x3) ---")
print(R_B)

In [ ]:
def pagerank(M, d=0.85, max_iter=100, tol=1e-6):
    """
    PageRank 알고리즘을 계산한다.

    :param M: (N x N) 인접 행렬 (M[i, j]는 j에서 i로의 링크를 나타냄).
    :param d: 댐핑 팩터 (보통 0.85).
    :param max_iter: 최대 반복 횟수.
    :param tol: 수렴 임계값.
    :return: PageRank 점수를 포함하는 Numpy 배열.
    """
    N = M.shape[0]  # 페이지 수 (노드 수)
    
    # 1. 인접 행렬 M을 전이 확률 행렬 P로 변환
    # M[i, j]는 j가 i에게 주는 투표를 의미하므로, 
    # j 열의 합계로 각 요소를 나누어 열의 합이 1이 되도록 정규화한다.
    # 즉, 각 페이지 j가 다른 페이지로 아웃링크를 분배하는 확률을 나타낸다.
    # 단, 아웃링크가 없는 노드(dangling nodes)는 처리하지 않는다.
    # 각 노드 j의 아웃링크 수
    out_degree = M.sum(axis=0)
    # 아웃링크가 0인 노드 처리 (0으로 나누는 것을 방지)
    out_degree[out_degree == 0] = 1 
    # 정규화된 행렬 P (P[i, j] = M[i, j] / out_degree[j])
    P = M / out_degree
    # 2. PageRank 벡터 초기화: 모든 페이지에 균등한 점수 할당
    # r0 = [1/N, 1/N, ..., 1/N]
    r = np.ones(N) / N
    # 3. PageRank 계산 반복
    for i in range(max_iter):
        r_prev = r.copy()
        # PageRank 기본 공식: r_new = d * P * r_prev + (1-d)/N * 1
        # P * r_prev: 현재 PageRank에 따른 링크를 통한 중요도 전파
        # (1-d)/N * 1: 댐핑 팩터 (텔레포트 확률)
        # PageRank의 핵심 반복 공식
        # P @ r_prev는 행렬-벡터 곱 (Numpy의 @ 연산자)
        r = d * (P @ r_prev) + (1 - d) / N
        # 4. 수렴 확인 (L1 Norm 사용)
        # abs(r - r_prev).sum()는 벡터 r과 r_prev의 차이의 절댓값 합계이다.
        if np.abs(r - r_prev).sum() < tol:
            print(f"PageRank가 {i+1}번째 반복에서 수렴했다.")
            return r

    print(f"PageRank가 최대 반복 횟수({max_iter}) 내에 수렴하지 않았다.")
    return r
# 예제 그래프의 인접 행렬 M을 정의한다. 
# M[i, j] = 1은 페이지 j에서 페이지 i로의 링크가 있음을 의미한다.
# 노드: A(0), B(1), C(2), D(3)
# A -> B
# B -> C
# C -> A, B
# D -> A
#     A  B  C  D  <- 'From' Page j
#   ---------------
# A | 0  0  1  1
# B | 1  0  1  0
# C | 0  1  0  0
# D | 0  0  0  0
# ^ 'To' Page i

M_matrix = np.array([
    [0, 0, 1, 1],  # A
    [1, 0, 1, 0],  # B
    [0, 1, 0, 0],  # C
    [0, 0, 0, 0]   # D (이 예제에서는 아웃링크가 없는 노드)
])

# PageRank 계산
pr_scores = pagerank(M_matrix, d=0.85)
# 결과 출력
page_names = ['A', 'B', 'C', 'D']
results = {name: score for name, score in zip(page_names, pr_scores)}
print("\n--- PageRank 점수 결과 ---")
for page, score in sorted(results.items(), key=lambda item: item[1], reverse=True):
    print(f"페이지 {page}: {score:.4f}")
print("\n(모든 PageRank 점수의 합: {:.4f})".format(pr_scores.sum()))    

In [ ]:
import numpy as np

def calculate_stationary_distribution(P):
    """
    전이 행렬 P의 전치 행렬 P^T와 고유값 1을 이용하여 정상 상태 분포를 계산한다.

    Args:
        P (np.ndarray): 전이 확률 행렬 (P의 행 합계는 1이어야 함).

    Returns:
        np.ndarray: 정상 상태 분포 벡터 (원소 합계가 1).
    """
    # 1. 전이 행렬의 전치 행렬 P_T를 구한다.
    P_T = P.T
    
    # 2. 고유값과 고유 벡터를 계산한다. (Eigenvalue problem: P_T * v = lambda * v)
    #    - w: 고유값 (Eigenvalues)
    #    - v: 고유 벡터 (Eigenvectors)
    w, v = np.linalg.eig(P_T)
    
    # 3. 고유값 1에 해당하는 고유 벡터를 찾는다.
    #    (마르코프 행렬의 특성상 고유값 1이 반드시 존재한다.)
    stationary_index = np.isclose(w, 1.0).argmax()
    
    # 4. 고유값 1에 해당하는 고유 벡터를 추출한다.
    #    (v의 열 벡터가 고유 벡터이다.)
    stationary_vector = v[:, stationary_index]
    
    # 5. 모든 원소가 양수(확률)가 되도록 실수부만 취하고 정규화한다.
    #    (계산 오차로 인해 작은 허수부가 남을 수 있음)
    stationary_vector = stationary_vector.real
    
    # 6. 벡터의 합이 1이 되도록 정규화한다. (정상 상태 분포 조건)
    stationary_distribution = stationary_vector / stationary_vector.sum()
    
    # 7. (선택 사항) 행 벡터 형태로 반환한다.
    return stationary_distribution.flatten()

# --- 예제 문제 실행 ---

# 전이 행렬 P (NumPy 배열로 정의)
P_matrix = np.array([
    [0.8, 0.1, 0.1], # A에서 A, B, C로의 전이 확률
    [0.2, 0.7, 0.1], # B에서 A, B, C로의 전이 확률
    [0.1, 0.3, 0.6]  # C에서 A, B, C로의 전이 확률
])

print("--- 입력 전이 행렬 P ---")
print(P_matrix)
print("-" * 25)

# 정상 상태 분포 계산
pi = calculate_stationary_distribution(P_matrix)

print("--- 정상 상태 분포 π ---")
print(f"π = {pi}")
print(f"π_A = {pi[0]:.4f}")
print(f"π_B = {pi[1]:.4f}")
print(f"π_C = {pi[2]:.4f}")

# 확인: 모든 원소의 합이 1인지 확인
print(f"\n확인: 성분 합계 = {pi.sum():.10f}")

# 확인: π * P = π 가 만족되는지 확인
check_pi_P = pi @ P_matrix
print(f"확인: π * P = {check_pi_P}")
print(f"확인: (π * P) - π = {check_pi_P - pi}")

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse.linalg import svds # 일반 SVD보다 빠름 (상위 k개만 계산)

def generalized_svd_recommender():
    # 1. 가상의 쇼핑몰 데이터 생성 (규모 확대: 15명 유저 x 10개 상품)
    # 상품 카테고리: [전자기기 4개, 소설책 3개, 화장품 3개]
    products = [
        '아이패드', '갤럭시폰', '에어팟', '기계식키보드', # Tech
        '해리포터', '반지의제왕', '코스모스',          # Books
        '수분크림', '립스틱', '썬크림'               # Beauty
    ]
    
    # 유저 성향 설정 (랜덤성을 부여하되, 특정 패턴을 심음)
    # User 0~4: Tech 덕후 / User 5~9: 독서광 / User 10~14: 뷰티 관심
    # 0점은 '구매 안 함'을 의미
    raw_data = []
    np.random.seed(42)
    
    for i in range(15):
        row = np.zeros(10)
        if i < 5: # Tech 덕후
            row[0:4] = np.random.randint(4, 6, 4) # 4~5점
            row[4:] = np.random.randint(0, 3, 6)  # 나머지는 낮거나 안 삼
        elif i < 10: # 독서광
            row[4:7] = np.random.randint(4, 6, 3)
            row[[0,1,2,3,7,8,9]] = np.random.randint(0, 3, 7)
        else: # 뷰티
            row[7:] = np.random.randint(4, 6, 3)
            row[:7] = np.random.randint(0, 3, 7)
            
        # 데이터를 듬성듬성하게 만들기 (Sparsity 부여)
        # 임의로 30%의 데이터를 0으로 만듦 (결측치 시뮬레이션)
        mask = np.random.choice([0, 1], size=10, p=[0.3, 0.7])
        row = row * mask
        raw_data.append(row)

    # 데이터프레임 변환
    df_ratings = pd.DataFrame(raw_data, columns=products)
    
    print(f"--- 1. 입력 데이터 (Shape: {df_ratings.shape}) ---")
    print("일부 데이터 미리보기:")
    print(df_ratings.iloc[[0, 5, 10], :]) # 각 그룹의 대표 유저 1명씩 출력

    # ==========================================================
    # 2. 전처리: 사용자 평균 평점 제거 (Normalization)
    # ==========================================================
    # SVD는 0을 '싫음'으로 인식할 위험이 있다.
    # 평균을 빼주면, 0은 '평균 이하'가 아닌 '중립'에 가까운 의미가 된다.
    R = df_ratings.values
    user_ratings_mean = np.mean(R, axis=1)
    R_demeaned = R - user_ratings_mean.reshape(-1, 1)

    # ==========================================================
    # 3. Truncated SVD 수행 (scipy.sparse.linalg.svds)
    # ==========================================================
    # k=3 (잠재 요인을 3개로 압축 -> 아마도 Tech, Book, Beauty 3개가 될 것임)
    k = 3
    U, sigma, Vt = svds(R_demeaned, k=k)

    # svds는 sigma를 대각행렬이 아닌 배열로 준다. 대각행렬로 변환 필요
    Sigma = np.diag(sigma)
    
    # ==========================================================
    # 4. 평점 예측 (행렬 복원)
    # ==========================================================
    # 다시 평균을 더해줘야 원본 스케일(0~5점)로 돌아온다.
    all_user_predicted_ratings = np.dot(np.dot(U, Sigma), Vt) + user_ratings_mean.reshape(-1, 1)
    
    df_preds = pd.DataFrame(all_user_predicted_ratings, columns=df_ratings.columns)

    # ==========================================================
    # 5. 추천 기능 구현
    # ==========================================================
    def recommend_items(user_id, original_df, preds_df, num_recommendations=2):
        # 1. 사용자가 이미 산 제품 제외 (원본이 0이 아닌 것)
        user_row = original_df.iloc[user_id]
        sorted_user_predictions = preds_df.iloc[user_id].sort_values(ascending=False)
        
        # 2. 추천 리스트 생성
        recommendations = []
        for item, rating in sorted_user_predictions.items():
            if user_row[item] == 0: # 안 산 물건 중에서
                recommendations.append((item, rating))
                if len(recommendations) >= num_recommendations:
                    break
        
        return recommendations

    # 결과 확인
    print("\n--- 2. 추천 결과 시뮬레이션 ---")
    
    # Case A: Tech 덕후지만 '에어팟'을 아직 안 산 유저 (ID: 0)
    # 데이터 생성 시 랜덤으로 0이 되었을 확률이 높음
    target_user = 0
    recs = recommend_items(target_user, df_ratings, df_preds)
    print(f"\n[User {target_user} (Tech 성향)]")
    print(f"이미 구매함: {df_ratings.columns[df_ratings.iloc[target_user] > 0].tolist()}")
    print(f"추천 상품: {recs}")

    # Case B: 독서광 유저 (ID: 5)
    target_user = 5
    recs = recommend_items(target_user, df_ratings, df_preds)
    print(f"\n[User {target_user} (Book 성향)]")
    print(f"이미 구매함: {df_ratings.columns[df_ratings.iloc[target_user] > 0].tolist()}")
    print(f"추천 상품: {recs}")
    
    # 6. 잠재 요인(Latent Factor) 해석
    # Vt 행렬의 각 행은 우리가 설정한 k개의 잠재 요인을 의미한다.
    print(f"\n--- 3. 잠재 요인(Latent Factor) 해석 ---")
    df_vt = pd.DataFrame(Vt, columns=products, index=[f'Factor {i+1}' for i in range(k)])
    print(df_vt)

generalized_svd_recommender()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD

def svd_nlp_demo_fixed():
    # 1. 문서 데이터
    docs = [
        "Human machine interface for lab abc computer applications",
        "A survey of user opinion of computer system response time",
        "The EPS user interface management system",
        "System and human system engineering testing of EPS",
        "Relation of user perceived response time to error measurement",
        "The generation of random binary unordered trees",
        "The intersection graph of paths in trees",
        "Graph minors IV Widths of trees and well quasi ordering",
        "Graph minors A survey",
        "Music is the art of arranging sounds in time",
        "Elements of music are rhythm and melody and harmony",
        "The violin is a wooden chordophone string instrument",
        "Piano is a musical instrument played using a keyboard"
    ]
    # 2. 단어-문서 행렬 생성
    # min_df=1: 문서에 1번이라도 나오면 단어로 인정
    vectorizer = CountVectorizer(stop_words='english', min_df=1)
    X = vectorizer.fit_transform(docs)
    # 3. SVD 수행
    svd = TruncatedSVD(n_components=2, random_state=42)
    result = svd.fit_transform(X.T)
    # 4. 시각화 (에러 수정 부분)
    plt.figure(figsize=(10, 8))
    words = vectorizer.get_feature_names_out()
    # 전체 단어 점 찍기
    plt.scatter(result[:, 0], result[:, 1], edgecolors='k', c='lightblue', s=50)
    # 표시할 단어 목록
    target_words = ['computer', 'system', 'interface', 'graph', 'trees', 
                    'music', 'sound', 'art', 'violin', 'piano']
    print("--- 단어 위치 찾기 결과 ---")
    for word in target_words:
        # [수정됨] 먼저 인덱스 목록을 찾음
        indices = np.where(words == word)[0]
        # [안전장치] 해당 단어가 존재할 때만 텍스트 표시
        if len(indices) > 0:
            idx = indices[0]
            plt.text(result[idx, 0], result[idx, 1], word, 
                     fontsize=12, fontweight='bold', color='red')
        else:
            print(f"주의: 단어 '{word}'는 사전에 없어 제외되었다.")
    plt.title("LSA(SVD) Word embedding: Music vs Computer")
    plt.grid(True)
    plt.show()

svd_nlp_demo_fixed()

In [ ]:
import numpy as np

# 1. 목표 공분산 행렬 정의 (3x3)
Sigma = np.array([
    [1.0, 0.8, 0.2],
    [0.8, 2.0, -0.5],
    [0.2, -0.5, 1.5]
])

# 샘플 수
N_samples = 100000

# 2. 촐레스키 분해
try:
    L = np.linalg.cholesky(Sigma)
    print("--- 촐레스키 인자 L (3x3) ---")
    print(L)
except np.linalg.LinAlgError:
    print("오류: 입력 행렬이 양의 정부호가 아니다. 촐레스키 분해 불가.")
    exit()

# 3. 독립적인 표준 정규 난수 생성 (3차원 벡터 N개)
# Z: (3, N_samples) 행렬
Z = np.random.normal(0, 1, size=(3, N_samples))

# 4. 촐레스키 인자를 이용한 변환 (X = L @ Z)
# L: (3, 3) @ Z: (3, N) -> X: (3, N)
X = L @ Z

# X1, X2, X3 변수 분리
X1 = X[0, :]
X2 = X[1, :]
X3 = X[2, :]

# 5. 생성된 변수의 통계량 확인
print("\n--- 생성된 변수의 통계량 ---")
print(f"분산 X1: {np.var(X1):.4f} (목표: 1.0)")
print(f"분산 X2: {np.var(X2):.4f} (목표: 2.0)")
print(f"분산 X3: {np.var(X3):.4f} (목표: 1.5)")

# 공분산 행렬 계산
# np.cov는 (3, N) 형태의 입력에 대해 공분산 행렬을 반환
cov_matrix_result = np.cov(X)

print("\n--- 생성된 공분산 행렬 ---")
print(cov_matrix_result.round(4))

# 공분산 값 비교 (X2와 X3의 음의 상관 관계 확인)
generated_cov_23 = cov_matrix_result[1, 2]
print(f"\n**X2와 X3의 공분산: {generated_cov_23:.4f} (목표: -0.5)**")

In [ ]:
import numpy as np

# 1. 목표 공분산 행렬 (Correlation Matrix) 정의
# (분산이 1이므로 공분산과 상관계수가 같음)
Sigma = np.array([
    [1.0, 0.7],
    [0.7, 1.0]
])

# 샘플 수
N_samples = 100000

# 2. 촐레스키 분해 (Sigma = L @ L.T)
try:
    L = np.linalg.cholesky(Sigma)
    print("--- 촐레스키 인자 L ---")
    print(L)
except np.linalg.LinAlgError:
    print("오류: 행렬이 양의 정부호가 아니다. 촐레스키 분해 불가.")
    exit()

# 3. 독립적인 표준 정규 난수 생성 (2차원 벡터 N개)
# z_i ~ N(0, 1), i.i.d.
Z = np.random.normal(0, 1, size=(2, N_samples))

# 4. 촐레스키 인자를 이용한 변환 (X = L @ Z)
# L: (2, 2) @ Z: (2, N) -> X: (2, N)
X = L @ Z

# X1, X2 변수 분리
X1 = X[0, :]
X2 = X[1, :]

# 5. 생성된 변수의 통계량 확인
print("\n--- 생성된 변수의 통계량 ---")
print(f"평균 X1: {np.mean(X1):.4f} (목표: 0)")
print(f"평균 X2: {np.mean(X2):.4f} (목표: 0)")
print(f"분산 X1: {np.var(X1):.4f} (목표: 1.0)")
print(f"분산 X2: {np.var(X2):.4f} (목표: 1.0)")

# 상관계수 계산
# np.corrcoef는 (2, N) 형태의 입력에 대해 상관 행렬을 반환
correlation_matrix_result = np.corrcoef(X)
generated_corr = correlation_matrix_result[0, 1]

print(f"**생성된 X1, X2의 상관계수: {generated_corr:.4f} (목표: 0.7)**")

In [ ]:
import numpy as np

# 3x3 정방 행렬 정의
A = np.array([[12, -51, 4], 
              [6, 167, -68], 
              [-4, 24, -41]])

print("--- 입력 행렬 A ---")
print(A)

# QR 분해 수행: Q, R = np.linalg.qr(A)
Q, R = np.linalg.qr(A)

print("\n--- Q 행렬 (직교 행렬) ---")
print(Q)

print("\n--- R 행렬 (상삼각 행렬) ---")
print(R)

# A = Q @ R 검증
A_reconstructed = Q @ R
print("\n--- Q @ R 재구성 결과 ---")
print(A_reconstructed)

# 원래 행렬 A와 재구성된 행렬이 같은지 확인
print("\n--- A == Q @ R 확인 (오차 허용 범위 내) ---")
print(np.allclose(A, A_reconstructed))

# Q의 전치 행렬 Q_T
Q_T = Q.T

# Q^T @ Q 계산
QT_Q = Q_T @ Q

print("\n--- Q^T @ Q 결과 ---")
# 결과가 단위 행렬 I (주대각선은 1, 나머지는 0)과 거의 같은지 확인한다.
print(QT_Q)

# 단위 행렬과 같은지 검증
is_orthogonal = np.allclose(QT_Q, np.eye(A.shape[0]))
print(f"\nQ는 직교 행렬인가? (Q^T Q = I): {is_orthogonal}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 데이터 정의
x_data = np.array([1, 2, 3, 4])
y_data = np.array([2.1, 3.9, 6.2, 8.1])

# 2. 행렬 A (설계 행렬) 구성
# 선형 모델: y = m*x + c
# A 행렬은 첫 번째 열이 x_data, 두 번째 열이 상수 1로 구성되어야 한다.
# A = [[x1, 1], [x2, 1], ...]
A = np.vstack([x_data, np.ones(len(x_data))]).T

print("--- 설계 행렬 A ---")
print(A)

# 3. 최소 제곱 해 구하기: np.linalg.lstsq(A, b, rcond=None)
# coeffs[0]: 계수 (m, c)
# residuals[1]: 잔차 제곱합
coeffs, residuals, rank, s = np.linalg.lstsq(A, y_data, rcond=None)

# m (기울기) = coeffs[0], c (y-절편) = coeffs[1]
m, c = coeffs

print("\n--- 최소 제곱 결과 ---")
print(f"기울기 (m): {m:.4f}")
print(f"y-절편 (c): {c:.4f}")
print(f"최소 제곱합 (잔차): {residuals[0]:.4f}")

# 4. 근사된 모델 (직선)
y_fit = m * x_data + c

# 5. 시각화 (선택 사항)
plt.figure(figsize=(8, 5))
plt.scatter(x_data, y_data, label='Original data(Observation)', color='red')
plt.plot(x_data, y_fit, label=f'Best fit line (y = {m:.2f}x + {c:.2f})', color='blue')
plt.title('Linear least squares fit')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np

# A, B, C 순서 (행이 발신, 열이 수신)
# A: 0, 1/2, 1/2
# B: 0, 0, 1
# C: 1, 0, 0
M = np.array([
    [0, 0, 1],       # A가 다른 페이지에게 줄 점수 (전치 행렬 M^T의 형태)
    [0.5, 0, 0],     # B가 다른 페이지에게 줄 점수
    [0.5, 1, 0]      # C가 다른 페이지에게 줄 점수
])

# 초기 랭크 벡터 (모두 1/3)
R = np.array([1/3, 1/3, 1/3]) 

# 반복 횟수 설정
num_iterations = 10 

print(f"초기 랭크 R0: {R}")

for i in range(num_iterations):
    # 다음 랭크 R_next = M * R (여기서 M은 이미 M^T의 역할을 하도록 정의됨)
    R_next = M @ R  
    
    # 랭크 업데이트
    R = R_next
    
    if (i + 1) % 2 == 0:
        print(f"R{i+1} (A, B, C): {R_next.round(4)}")

# 결과:
print("\n--- 최종 PageRank 결과 ---")
print(f"A, B, C의 최종 랭크: {R.round(4)}")

In [ ]:
import numpy as np
import warnings

# 경고 메시지 (NumPy의 FutureWarning)를 무시한다.
warnings.filterwarnings('ignore', category=FutureWarning)

# 전이 행렬 M 정의
# (발신 페이지가 열, 수신 페이지가 행의 역할)
# M의 열 벡터 합은 1이어야 하지만, 제공된 행렬을 그대로 사용한다.
# A: 0, 1/2, 1/2 -> A가 C에게 1.0 (제공된 주석과 다름)
# B: 0, 0, 1
# C: 1, 0, 0
# M 행렬 해석:
# M[i, j] = j가 i에게 주는 점수 (j가 발신, i가 수신)
#          A(j=0)  B(j=1)  C(j=2)
M = np.array([
    [0, 0.5, 0.5],     # A (i=0)가 받는 점수
    [0, 0, 1],         # B (i=1)가 받는 점수
    [1, 0.5, 0]        # C (i=2)가 받는 점수
])

# 초기 랭크 벡터 (모두 1/N, N=3)
R = np.array([1/3, 1/3, 1/3])  

# 반복 횟수 설정
num_iterations = 10 

print(f"---  PageRank 계산 시작 ---")
print(f"초기 랭크 R0 (A, B, C): {R.round(4)}")

# R_history = [R.copy()] # 랭크 변화를 추적하고 싶다면 사용

# PageRank 반복 계산: R_next = M @ R
for i in range(num_iterations):
    R_next = M @ R  
    
    # 랭크 업데이트
    R = R_next
    
    # 2회 반복마다 출력
    if (i + 1) % 2 == 0:
        print(f"R{i+1} (A, B, C): {R_next.round(4)}")

# 결과 출력
print("\n---  최종 PageRank 결과 ---")
print(f"A, B, C의 최종 랭크: {R.round(4)}")
print(f"총 합 (검증): {R.sum().round(4)}") # 총합은 1에 가까워야 한다.

In [ ]:
import numpy as np
from numpy.polynomial import Polynomial
def solve_polynomial_fitting_modern(x_data, y_data, degree=4):
    """
    NumPy의 최신 Polynomial API를 사용한 피팅 및 최적점 탐색 함수
    """
    # 1. 최신 API를 이용한 피팅
    # Polynomial.fit은 내부적으로 x 데이터를 [-1, 1] 구간으로 맵핑(domain windowing)하여
    # 수치적 오차를 획기적으로 줄인다.
    p_obj = Polynomial.fit(x_data, y_data, degree)
    # 2. 미분 및 근(Roots) 찾기 (f'(x) = 0)
    p_deriv = p_obj.deriv()
    roots = p_deriv.roots() # 메소드 호출 방식
    # 3. 실수 근만 필터링 및 범위 제한
    # 복소수 허수부가 거의 0인 경우를 고려해 np.isreal 사용
    real_roots = roots[np.isreal(roots)].real
    x_min = np.min(x_data)
    x_max = np.max(x_data)
    # 데이터 범위(Domain) 내에 있는 근만 후보로 선정
    candidates = [r for r in real_roots if x_min <= r <= x_max]
    # 4. 극소점 판별 (2계 미분 테스트)
    p_deriv2 = p_obj.deriv(2)
    local_minima = []
    for c in candidates:
        # f''(c) > 0 이면 아래로 볼록(Convex) -> 극소점
        if p_deriv2(c) > 0:
            local_minima.append((c, p_obj(c)))
    # 5. 최적값 결정 로직
    if local_minima:
        # 찾은 극소점들 중 가장 작은 함숫값을 가진 지점 선택
        # min 함수는 튜플의 두 번째 요소(함숫값)를 기준으로 최소를 찾음
        best_x, min_val = min(local_minima, key=lambda item: item[1])
        found_local_min = True
    else:
        # 극소점을 찾지 못한 경우 (단조 증가/감소 등), 데이터 구간 내 최소값으로 대체
        # (기존 코드의 의도를 유지)
        min_idx = np.argmin(p_obj(x_data))
        best_x = x_data[min_idx]
        min_val = p_obj(best_x)
        found_local_min = False
    return p_obj, best_x, min_val, found_local_min

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

# 1. 가상의 데이터 생성 (가우시안 피크 + 노이즈)
x = np.linspace(0, 20, 100)
true_signal = np.exp(-(x - 10)**2 / 2) # 뾰족한 피크
noise = np.random.normal(0, 0.1, size=len(x))
y_noisy = true_signal + noise
# 2. Savitzky-Golay 필터 적용
# window=11, polyorder=3 (3차 곡선으로 근사)
y_savgol = savgol_filter(y_noisy, window_length=11, polyorder=3)
# 3. 비교군: 단순 이동 평균 (Moving Average)
# 같은 윈도우 크기(11)로 평균
y_moving_avg = np.convolve(y_noisy, np.ones(11)/11, mode='same')
# 4. 시각화
plt.figure(figsize=(10, 6))
plt.plot(x, y_noisy, '.', color='lightgray', label='Noisy Data')
plt.plot(x, true_signal, 'k--', label='True Signal', alpha=0.5)
# 이동 평균: 피크가 뭉개짐(높이가 낮아짐)
plt.plot(x, y_moving_avg, 'b-', label='Moving Average (Window 11)')
# SavGol: 피크 높이를 잘 따라감
plt.xlabel('x', fontsize=18)
plt.ylabel('y', fontsize=18)
plt.plot(x, y_savgol, 'r-', linewidth=2, label='Savitzky-Golay (Window 11, Poly 3)')
plt.legend()
plt.title("Why Savitzky-Golay is better for peaks")
plt.grid(True)
plt.savefig('sv.png')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

def analyze_derivatives():
    # 1. 가상의 실험 데이터 생성 (가우시안 피크 + 노이즈)
    # 시간 간격 (dt)가 중요하다. 미분값의 크기(Scale)를 결정하기 때문이다.
    dt = 0.1 
    t = np.arange(0, 20, dt)
    # 신호: 10초 지점에서 피크가 있는 가우시안 함수
    true_signal = 10 * np.exp(-(t - 10)**2 / 2)
    noise = np.random.normal(0, 0.5, size=len(t))
    y_raw = true_signal + noise
    # ---------------------------------------------------------
    # 2. Savitzky-Golay 필터 적용 (0차, 1차, 2차 미분)
    # ---------------------------------------------------------
    window = 15    # 창 크기 (홀수)
    poly = 3       # 다항식 차수 (3차)
    # delta=dt 옵션: x축 간격을 반영하여 실제 물리량 단위의 미분값을 구해준다.
    # (이 옵션을 안 쓰면 단순히 인덱스 기준 변화량이 나온다)
    # A. 스무딩 (deriv=0)
    y_smooth = savgol_filter(y_raw, window, poly, deriv=0)
    # B. 1차 미분 (deriv=1) -> 속도 / 기울기
    y_d1 = savgol_filter(y_raw, window, poly, deriv=1, delta=dt)
    # C. 2차 미분 (deriv=2) -> 가속도 / 곡률
    y_d2 = savgol_filter(y_raw, window, poly, deriv=2, delta=dt)
    # ---------------------------------------------------------
    # 3. 결과 시각화 (3단 그래프)
    # ---------------------------------------------------------
    fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)
    # [첫 번째 칸] 원본 vs 스무딩
    axes[0].plot(t, y_raw, '.', color='lightgray', label='Raw Data')
    axes[0].plot(t, y_smooth, 'r-', linewidth=2, label='Smoothed (SavGol)')
    axes[0].set_title('1. Original Signal & Smoothing')
    axes[0].set_ylabel('Amplitude')
    axes[0].legend(loc='upper right')
    axes[0].grid(True)
    # [두 번째 칸] 1차 미분 (Velocity)
    axes[1].plot(t, y_d1, 'g-', label='1st Derivative (Slope)')
    axes[1].axhline(0, color='black', linestyle='--', alpha=0.5) # 0점 기준선
    axes[1].set_title('2. First Derivative (Peak Detection)')
    axes[1].set_ylabel('Slope (dy/dt)')
    axes[1].grid(True)
    # 피크 위치 표시 (1차 미분이 0이 되는 지점)
    # 부호가 양수(+)에서 음수(-)로 바뀌는 지점이 피크이다.
    zero_crossing = t[np.where(np.diff(np.sign(y_d1)))[0]]
    if len(zero_crossing) > 0:
        peak_time = zero_crossing[0]
        axes[1].axvline(peak_time, color='blue', linestyle=':', label=f'Peak at {peak_time:.1f}s')
        axes[1].legend()
    # [세 번째 칸] 2차 미분 (Acceleration)
    axes[2].plot(t, y_d2, 'purple', label='2nd Derivative (Curvature)')
    axes[2].axhline(0, color='black', linestyle='--', alpha=0.5)
    axes[2].set_title('3. Second Derivative (Inflection & Width)')
    axes[2].set_ylabel('Curvature (d²y/dt²)')
    axes[2].set_xlabel('Time (s)')
    axes[2].grid(True)
    axes[2].legend()
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    analyze_derivatives()

In [ ]:
import numpy as np

# 1. 목표 공분산 행렬 (Correlation Matrix) 정의
# (분산이 1이므로 공분산과 상관계수가 같음)
Sigma = np.array([
    [1.0, 0.7],
    [0.7, 1.0]
])

# 샘플 수
N_samples = 100000

# 2. 촐레스키 분해 (Sigma = L @ L.T)
try:
    L = np.linalg.cholesky(Sigma)
    print("--- 촐레스키 인자 L ---")
    print(L)
except np.linalg.LinAlgError:
    print("오류: 행렬이 양의 정부호가 아니다. 촐레스키 분해 불가.")
    exit()

# 3. 독립적인 표준 정규 난수 생성 (2차원 벡터 N개)
# z_i ~ N(0, 1), i.i.d.
Z = np.random.normal(0, 1, size=(2, N_samples))

# 4. 촐레스키 인자를 이용한 변환 (X = L @ Z)
# L: (2, 2) @ Z: (2, N) -> X: (2, N)
X = L @ Z

# X1, X2 변수 분리
X1 = X[0, :]
X2 = X[1, :]

# 5. 생성된 변수의 통계량 확인
print("\n--- 생성된 변수의 통계량 ---")
print(f"평균 X1: {np.mean(X1):.4f} (목표: 0)")
print(f"평균 X2: {np.mean(X2):.4f} (목표: 0)")
print(f"분산 X1: {np.var(X1):.4f} (목표: 1.0)")
print(f"분산 X2: {np.var(X2):.4f} (목표: 1.0)")

# 상관계수 계산
# np.corrcoef는 (2, N) 형태의 입력에 대해 상관 행렬을 반환
correlation_matrix_result = np.corrcoef(X)
generated_corr = correlation_matrix_result[0, 1]

print(f"**생성된 X1, X2의 상관계수: {generated_corr:.4f} (목표: 0.7)**")

In [ ]:
import numpy as np

# 1. 목표 공분산 행렬 정의 (3x3)
Sigma = np.array([
    [1.0, 0.8, 0.2],
    [0.8, 2.0, -0.5],
    [0.2, -0.5, 1.5]
])

# 샘플 수
N_samples = 100000

# 2. 촐레스키 분해
try:
    L = np.linalg.cholesky(Sigma)
    print("--- 촐레스키 인자 L (3x3) ---")
    print(L)
except np.linalg.LinAlgError:
    print("오류: 입력 행렬이 양의 정부호가 아니다. 촐레스키 분해 불가.")
    exit()

# 3. 독립적인 표준 정규 난수 생성 (3차원 벡터 N개)
# Z: (3, N_samples) 행렬
Z = np.random.normal(0, 1, size=(3, N_samples))

# 4. 촐레스키 인자를 이용한 변환 (X = L @ Z)
# L: (3, 3) @ Z: (3, N) -> X: (3, N)
X = L @ Z

# X1, X2, X3 변수 분리
X1 = X[0, :]
X2 = X[1, :]
X3 = X[2, :]

# 5. 생성된 변수의 통계량 확인
print("\n--- 생성된 변수의 통계량 ---")
print(f"분산 X1: {np.var(X1):.4f} (목표: 1.0)")
print(f"분산 X2: {np.var(X2):.4f} (목표: 2.0)")
print(f"분산 X3: {np.var(X3):.4f} (목표: 1.5)")

# 공분산 행렬 계산
# np.cov는 (3, N) 형태의 입력에 대해 공분산 행렬을 반환
cov_matrix_result = np.cov(X)

print("\n--- 생성된 공분산 행렬 ---")
print(cov_matrix_result.round(4))

# 공분산 값 비교 (X2와 X3의 음의 상관 관계 확인)
generated_cov_23 = cov_matrix_result[1, 2]
print(f"\n**X2와 X3의 공분산: {generated_cov_23:.4f} (목표: -0.5)**")

In [ ]:
import numpy as np

## 1. 목적함수 정의
def schaffer_n7(x):
    """
    Schaffer Function N. 7 (2D)
    최소값: 0.0 (x = [0, 0] 에서)
    """
    if len(x) != 2:
        raise ValueError("Input vector must be 2-dimensional")
    x1, x2 = x
    # 내부 항: x1^2 + x2^2
    r = np.sqrt(x1**2 + x2**2)
    # Schaffer N. 7 공식
    term1 = r**(1/4)
    term2 = 0.5 * np.sin(50.0 * r**(1/10))**2 + 0.5
    return term1 * term2

## 2. CMA-ES 클래스 구현 (수정 완료)
class CMAES:
    def __init__(self, objective_function, x_start, sigma_start, population_size=None):
        # 기본 설정
        self.objective_function = objective_function
        self.N = len(x_start)  # 차원
        self.x_mean = np.array(x_start)  # 평균 벡터
        self.sigma = sigma_start  # 단계 크기 (Step size)
        self.generation = 0 # 세대 카운터 초기화
        # 모집단 크기 (lambda_) 및 우수 해의 수 (mu) 설정
        self.lambda_ = population_size if population_size is not None else int(4 + np.floor(3 * np.log(self.N)))
        self.mu = self.lambda_ // 2
        # 가중치 설정
        weights = np.log(self.mu + 0.5) - np.log(np.arange(1, self.mu + 1))
        self.weights = weights / np.sum(weights)
        self.mu_eff = 1 / np.sum(self.weights**2)
        # 공분산 행렬 (C) 초기화
        self.C = np.eye(self.N)
        # 진화 경로 (Evolution Paths) 초기화
        self.p_c = np.zeros(self.N)  # 공분산 경로
        self.p_sigma = np.zeros(self.N)  # 단계 크기 제어 경로
        # 학습률 및 상수 설정
        self.c_c = (4 + self.mu_eff / self.N) / (self.N + 4 + 2 * self.mu_eff / self.N)  
        self.c_sigma = (self.mu_eff + 2) / (self.N + self.mu_eff + 5)
        self.c_1 = 2 / ((self.N + 1.3)**2 + self.mu_eff)
        self.c_mu = min(1 - self.c_1, 2 * (self.mu_eff - 2 + 1 / self.mu_eff) / ((self.N + 2)**2 + 2 * self.mu_eff / 2))
        self.d_sigma = 1 + 2 * max(0, np.sqrt((self.mu_eff - 1) / (self.N + 1)) - 1) + self.c_sigma
    def sample_population(self):
        """
        C 행렬의 고유 분해(B D^2 B^T)를 이용한 샘플링.
        """
        # C 행렬의 고유 분해 수행
        try:
            # D_squared: 고유값 (Eigenvalues), B: 고유벡터 (Eigenvectors)
            D_squared, B = np.linalg.eigh(self.C)
            # 수치 안정성 확보: 음수 고유값 방지
            D_squared[D_squared < 0] = 0
            D = np.sqrt(D_squared)
        except np.linalg.LinAlgError:
            # 공분산 재설정 (Resetting)
            print(f"Warning (Gen {self.generation}): C is not positive definite. Resetting to identity.")
            self.C = np.eye(self.N)
            D_squared, B = np.linalg.eigh(self.C)
            D = np.sqrt(D_squared)

        population = []
        for _ in range(self.lambda_):
            # 표준 정규 분포에서 샘플링 (z)
            z = np.random.randn(self.N)
            # C 행렬을 이용한 변환 (y = B D z)
            y = B @ (D * z)
            # 최종 샘플: x_mean + sigma * y
            x = self.x_mean + self.sigma * y
            population.append((x, self.objective_function(x), y))
        return population

    def update(self, population):
        """
        평균, 진화 경로, 공분산 행렬, 단계 크기를 업데이트한다.
        """
        # 1. 정렬 및 우수 해 선택
        population.sort(key=lambda item: item[1])
        sorted_population = population[:self.lambda_]
        # 우수 해의 평균 계산
        x_old = self.x_mean.copy()
        y_w = np.zeros(self.N) # 가중치 적용된 변화량 합계
        # 2. 평균 업데이트에 사용할 y_w 계산
        for i in range(self.mu):
            x_k, _, y_k = sorted_population[i]
            y_w += self.weights[i] * y_k
        self.x_mean = x_old + self.sigma * y_w
        # ************** 수정된 부분: B와 C_inv_sqrt 재계산 **************
        # 3. C의 역 제곱근 (C^{-1/2}) 계산을 위해 C를 다시 분해
        try:
            D_squared, B = np.linalg.eigh(self.C)
            D_squared[D_squared < 0] = 1e-10 # 작은 양수로 대체하여 안정성 확보
            D_inv = np.diag(1.0 / np.sqrt(D_squared))
            # C_inv_sqrt = B D^{-1} B^T
            C_inv_sqrt = B @ D_inv @ B.T
        except np.linalg.LinAlgError:
             # 실패 시 단위 행렬 사용
            C_inv_sqrt = np.eye(self.N)
        # *************************************************************
        # 4. 단계 크기 제어 경로 (p_sigma) 업데이트
        self.p_sigma = (1 - self.c_sigma) * self.p_sigma + \
                       np.sqrt(self.c_sigma * (2 - self.c_sigma) * self.mu_eff) * C_inv_sqrt @ y_w
        # 5. 단계 크기 (sigma) 업데이트
        E_norm = np.sqrt(self.N) * (1 - 1 / (4 * self.N) + 1 / (21 * self.N**2)) # 정규화 상수
        self.sigma = self.sigma * np.exp((self.c_sigma / self.d_sigma) * (np.linalg.norm(self.p_sigma) / E_norm - 1))
        # 6. 공분산 경로 (p_c) 업데이트
        # H_sigma (Heaviside function): 단계 크기 제어 경로의 기여 여부 결정
        norm_p_sigma = np.linalg.norm(self.p_sigma)
        threshold = np.sqrt(1 + 1 / self.N) * E_norm
        h_sigma = 1
        tmp = np.sqrt(1 - (1 - self.c_sigma)**(2 * self.generation))
        if tmp < 1e-12:
            tmp=1e-12
        if norm_p_sigma /tmp > threshold :
            h_sigma= 0
        
        self.p_c = (1 - self.c_c) * self.p_c + \
                   h_sigma * np.sqrt(self.c_c * (2 - self.c_c) * self.mu_eff) * y_w

        # 7. 공분산 행렬 (C) 업데이트
        # Rank-one update
        # p_c[:, np.newaxis] @ p_c[np.newaxis, :]는 outer product (p_c p_c^T)
        C_rank_one = self.c_1 * (self.p_c[:, np.newaxis] @ self.p_c[np.newaxis, :])
        
        # Rank-mu update
        C_rank_mu = np.zeros((self.N, self.N))
        for i in range(self.mu):
            y_i = sorted_population[i][2]
            # y_i[:, np.newaxis] @ y_i[np.newaxis, :]는 outer product (y_i y_i^T)
            C_rank_mu += self.weights[i] * (y_i[:, np.newaxis] @ y_i[np.newaxis, :])
        C_rank_mu = self.c_mu * C_rank_mu

        # 전체 업데이트
        self.C = (1 - self.c_1 - self.c_mu) * self.C + C_rank_one + C_rank_mu
        
        # 대칭성 강제
        self.C = (self.C + self.C.T) / 2
        
        self.generation += 1

    def minimize(self, max_generations=1000, tol=1e-8):
        best_f = float('inf')
        print("--- CMA-ES 최적화 시작 ---")
        for gen in range(1, max_generations + 1):
            # 1. 새로운 모집단 샘플링
            population = self.sample_population()
            # 2. 업데이트
            self.update(population)
            # 3. 결과 기록
            current_best_x, current_best_f, _ = population[0] 
            if current_best_f < best_f:
                best_f = current_best_f
                best_x = current_best_x
            if gen % 50 == 0:
                print(f"Gen {gen}: Best f(x) = {best_f:.8e}, sigma = {self.sigma:.2e}, x_mean = {self.x_mean}")
            # 4. 수렴 조건 확인
            if self.sigma < tol:
                print(f"Convergence reached: Step size (sigma) below tolerance {tol}.")
                break  
        print("\n--- 최적화 완료 ---")
        print(f"최소값 (f(x)): {best_f:.8e}")
        print(f"최적 해 (x): {best_x}")
        return best_x, best_f


## 3. 프로그램 실행
if __name__ == "__main__":
    # 초기 설정
    x_start = np.array([1.1, -1.0]) # 시작점
    sigma_start = 1.1               # 초기 단계 크기
    # CMA-ES 인스턴스 생성 및 실행
    cmaes_optimizer = CMAES(schaffer_n7, x_start, sigma_start)
    cmaes_optimizer.minimize(max_generations=1000, tol=1e-12)

In [ ]:
import numpy as np

# 3x2 행렬 A 정의 (정방 행렬이 아님)
A = np.array([[1, 1], 
              [0, 1], 
              [1, 0]])

print("--- 입력 행렬 A (3x2) ---")
print(A)

# np.linalg.svd() 함수를 사용하여 SVD 수행
# U: 좌측 특이 벡터 행렬
# s: 특잇값 (Sigma 행렬의 대각 성분)
# Vt: 우측 특이 벡터 행렬의 전치 (V^T)
U, s, Vt = np.linalg.svd(A)

print("\n--- U 행렬 (좌측 특이 벡터, 3x3) ---")
print(U)

print("\n--- 특잇값 s (2개) ---")
print(s)

print("\n--- Vt 행렬 (우측 특이 벡터의 전치, 2x2) ---")
print(Vt)

# A = U * Sigma * Vt 검증을 위한 Sigma 행렬 재구성
# Sigma는 A와 같은 크기인 3x2 행렬로 만들어야 한다.
Sigma = np.zeros(A.shape)
# 특잇값 s를 Sigma의 주대각선에 배치
Sigma[:A.shape[1], :A.shape[1]] = np.diag(s)

print("\n--- 재구성된 Sigma 행렬 (3x2) ---")
print(Sigma)

# U * Sigma * Vt 계산 및 검증
A_reconstructed = U @ Sigma @ Vt

print("\n--- U * Sigma * Vt 재구성 결과 ---")
print(A_reconstructed)

# 원래 행렬 A와 재구성된 행렬이 같은지 확인
print("\n--- 재구성된 행렬이 원본과 같은가? (오차 허용 범위 내) ---")
print(np.allclose(A, A_reconstructed))

In [ ]:
import numpy as np
# 2x2 정방 행렬 정의
A = np.array([[4, 2], 
              [1, 3]])
# np.linalg.eig() 함수를 사용하여 고윳값과 고유벡터 계산
# w: 고윳값(eigenvalues), v: 고유벡터(eigenvectors)
w, v = np.linalg.eig(A)
print("--- 입력 행렬 A ---")
print(A)
print("\n--- 고윳값 (Eigenvalues) ---")
print(w)
print("\n--- 고유벡터 (Eigenvectors) ---")
# 각 열(column)이 해당 고윳값에 대응하는 고유벡터이다.
# w[0] = 5.0 에 대응하는 고유벡터는 v[:, 0] = [0.8944..., 0.4472...] 이다.
# w[1] = 2.0 에 대응하는 고유벡터는 v[:, 1] = [-0.7071..., 0.7071...] 이다.
print(v)

# 첫 번째 고윳값과 고유벡터 추출
lambda_1 = w[0]
v_1 = v[:, 0]
print("--- 첫 번째 고윳값과 고유벡터 ---")
print(f"고윳값 (λ1): {lambda_1}")
print(f"고유벡터 (v1):\n{v_1}")
# A * v1 계산 (Av)
Av_1 = A @ v_1 
# lambda_1 * v1 계산 (λv)
lambda_v_1 = lambda_1 * v_1
print("\n--- Av1 계산 결과 ---")
print(Av_1)
print("\n--- λ1*v1 계산 결과 ---")
print(lambda_v_1)
# 두 결과가 거의 같은지 확인
print("\n--- Av1 == λ1*v1 확인 (오차 허용 범위 내) ---")
print(np.allclose(Av_1, lambda_v_1))

In [ ]:
def vibration_demo():
    print("\n=== 3. Structural vibration (Eigenvalue = Frequency^2) ===")
    
    # 간단한 2자유도 스프링-질량 시스템 (건물 2층 모델)
    # 운동방정식에서 유도된 시스템 행렬 A
    # A = M^(-1) * K 
    k = 1000 # 스프링 상수 (강성)
    m = 10   # 질량
    
    # 행렬 A (시스템의 역학적 특성)
    A = np.array([
        [2*k/m, -k/m],
        [-k/m, k/m]
    ])
    
    # 고유값 계산
    eigenvalues, modes = np.linalg.eig(A)
    
    # 고유진동수 (omega) = sqrt(lambda)
    frequencies = np.sqrt(eigenvalues)
    
    print(f"시스템 행렬 A:\n{A}")
    print("-" * 30)
    for i, freq in enumerate(frequencies):
        print(f"Mode {i+1}:")
        print(f"  - 고유값 (Lambda): {eigenvalues[i]:.2f}")
        print(f"  - 고유 진동수 (Frequency): {freq:.2f} rad/s")
        print(f"  - 진동 모양 (Eigenvector): {modes[:, i]}")
        
    print("\n-> 이 건물을 지을 때, 지진파나 바람의 주파수가")
    print(f"   {frequencies[0]:.2f} 또는 {frequencies[1]:.2f} rad/s 가 되지 않도록 피해야 한다.")

vibration_demo()

In [ ]:
import matplotlib.pyplot as plt

def pca_eigen_demo():
    print("\n=== 2. PCA (principal component analysis) ===")
    
    # 1. 데이터 생성 (우상향하는 타원형 데이터)
    np.random.seed(0)
    mean = [0, 0]
    cov = [[3, 2], [2, 3]] # 공분산 행렬 (x, y가 양의 상관관계)
    x, y = np.random.multivariate_normal(mean, cov, 500).T
    
    # 데이터를 행렬로 결합
    X = np.stack((x, y), axis=0)
    
    # 2. 공분산 행렬 계산
    covariance_matrix = np.cov(X)
    
    # 3. 고유값, 고유벡터 계산
    eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
    print("고유값 (분산의 크기):", eig_vals)
    print("고유벡터 (주성분 방향):\n", eig_vecs)
    
    # 4. 시각화
    plt.figure(figsize=(6, 6))
    plt.scatter(x, y, alpha=0.2)
    
    # 고유벡터 그리기 (중심에서 시작하는 화살표)
    origin = [0, 0]
    for i in range(len(eig_vals)):
        # 고유값의 크기만큼 화살표 길이를 조절해 시각화
        # (시각적 편의를 위해 sqrt를 취해 표준편차 스케일로 그림)
        vec_len = np.sqrt(eig_vals[i]) * 2 
        vec = eig_vecs[:, i] * vec_len
        
        plt.arrow(origin[0], origin[1], vec[0], vec[1], 
                  head_width=0.3, head_length=0.3, fc='r', ec='r', lw=2)
        plt.text(vec[0], vec[1], f"Lambda={eig_vals[i]:.1f}", color='red', fontsize=12)
        
    plt.title("Eigenvalues represent variance")
    plt.grid(True)
    plt.axis('equal')
    plt.show()

pca_eigen_demo()

In [ ]:
import numpy as np

def pagerank_demo():
    print("=== 1. Google PageRank (Billion Dollar Eigenvector) ===")
    
    # 4개의 웹페이지(A, B, C, D)가 서로를 링크하는 상황
    # 열(Column)의 합이 1이 되도록 만든 확률 전이 행렬 (Markov Matrix)
    # A->B, A->C 로 링크를 검 (A열: 0, 0.5, 0.5, 0)
    M = np.array([
        [0.0, 0.0, 1.0, 0.5], # A로 들어오는 링크 (C->A, D->A)
        [0.5, 0.0, 0.0, 0.0], # B로 들어오는 링크 (A->B)
        [0.5, 0.0, 0.0, 0.5], # C로 들어오는 링크 (A->C, D->C)
        [0.0, 1.0, 0.0, 0.0]  # D로 들어오는 링크 (B->D)
    ])
    
    # 고유값 분해
    eigenvalues, eigenvectors = np.linalg.eig(M)
    
    # 고유값이 1인 것(가장 큰 고유값)을 찾음
    # (부동소수점 오차를 고려해 1에 가장 가까운 값 찾기)
    idx = np.argmax(np.abs(eigenvalues))
    largest_eigenvalue = eigenvalues[idx]
    pagerank_vector = np.real(eigenvectors[:, idx])
    
    # 확률의 합이 1이 되도록 정규화
    pagerank_vector = pagerank_vector / np.sum(pagerank_vector)
    
    print(f"최대 고유값: {largest_eigenvalue:.4f}")
    print("페이지 중요도 (PageRank):")
    pages = ['A', 'B', 'C', 'D']
    for p, rank in zip(pages, pagerank_vector):
        print(f"  Page {p}: {rank:.4f}")
        
    print("-> Page A와 C가 가장 중요도가 높게 나왔다.")

pagerank_demo()

In [ ]:
import heapq

class Node:
    def __init__(self, parent=None, position=None):
        self.parent = parent
        self.position = position
        
        self.g = 0  # 시작점부터 현재까지 비용
        self.h = 0  # 현재부터 목표까지 추정 비용 (Heuristic)
        self.f = 0  # f = g + h

    def __eq__(self, other):
        return self.position == other.position
    
    # priority queue에서 비교를 위한 연산자 정의
    def __lt__(self, other):
        return self.f < other.f

def heuristic(a, b):
    """
    휴리스틱 함수: 맨해튼 거리 (Manhattan Distance)
    격자 이동(상하좌우)만 가능할 때 유용함.
    대각선 이동이 가능하다면 유클리드 거리(Euclidean)를 사용하세요.
    """
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def astar_search(maze, start, end):
    """
    maze: 0은 길, 1은 장애물인 2차원 리스트
    start: 시작 좌표 (x, y)
    end: 목표 좌표 (x, y)
    """
    # 1. 시작 노드와 끝 노드 초기화
    start_node = Node(None, start)
    end_node = Node(None, end)
    # 2. 열린 목록(탐색할 노드)과 닫힌 목록(이미 방문한 노드)
    open_list = []
    closed_list = []
    # 시작 노드를 open_list에 추가
    heapq.heappush(open_list, start_node)
    # 3. 메인 루프
    while open_list:
        # f값이 가장 작은 노드를 꺼냄 (heapq가 자동으로 정렬해줌)
        current_node = heapq.heappop(open_list)
        closed_list.append(current_node)
        # 목표 도달 시 경로 추적 (Backtracking)
        if current_node == end_node:
            path = []
            current = current_node
            while current is not None:
                path.append(current.position)
                current = current.parent
            return path[::-1] # 역순으로 뒤집어서 반환
        # 인접 노드(상하좌우) 탐색
        children = []
        # (0, -1), (0, 1), (-1, 0), (1, 0) : 상하좌우
        for new_position in [(0, -1), (0, 1), (-1, 0), (1, 0)]: 
            # 노드 위치 업데이트
            node_position = (current_node.position[0] + new_position[0], 
                             current_node.position[1] + new_position[1])
            # 1. 지도 범위 확인
            if (node_position[0] > (len(maze) - 1) or 
                node_position[0] < 0 or 
                node_position[1] > (len(maze[len(maze)-1]) - 1) or 
                node_position[1] < 0):
                continue
            # 2. 장애물 확인 (1이면 못감)
            if maze[node_position[0]][node_position[1]] != 0:
                continue
            # 새 노드 생성
            new_node = Node(current_node, node_position)
            children.append(new_node)
        # 자식 노드들 처리
        for child in children:
            # 이미 닫힌 목록에 있으면 스킵
            if child in closed_list:
                continue
            # g, h, f 값 계산
            child.g = current_node.g + 1 # 이동 비용 1로 가정
            child.h = heuristic(child.position, end_node.position)
            child.f = child.g + child.h
            # 열린 목록에 있고, 더 비싼 경로라면 스킵 (최적화)
            # (Note: 간단한 구현을 위해 단순히 리스트 확인. 
            #  엄밀하게는 open_list에 있는 동일 노드와 g값을 비교해야 함)
            if len([open_node for open_node in open_list if child == open_node and child.g > open_node.g]) > 0:
                continue
            heapq.heappush(open_list, child)
    return None # 경로 없음

# ==========================================
# 실행 예제
# ==========================================
def run_example():
    # 0: 이동 가능, 1: 장애물
    grid_map = [
        [0, 0, 0, 0, 1, 0, 0],
        [0, 1, 1, 0, 1, 0, 0],
        [0, 1, 0, 0, 1, 0, 0], # 중간에 갇힌 구역
        [0, 0, 0, 1, 1, 0, 0], # 돌아서 가야 함
        [0, 1, 0, 0, 0, 0, 0]
    ]
    start_pos = (0, 0)
    end_pos = (4, 6)
    print(f"Start: {start_pos}, End: {end_pos}")
    path = astar_search(grid_map, start_pos, end_pos)
    if path:
        print(f"최단 경로 발견! (총 {len(path)} 단계)")
        print(path)
        
        # 시각화
        print("\n--- Map visualization ---")
        for x in range(len(grid_map)):
            line = ""
            for y in range(len(grid_map[0])):
                if (x, y) == start_pos:
                    line += "S " # Start
                elif (x, y) == end_pos:
                    line += "E " # End
                elif (x, y) in path:
                    line += "* " # Path
                elif grid_map[x][y] == 1:
                    line += "X " # Wall
                else:
                    line += ". " # Road
            print(line)
    else:
        print("경로를 찾을 수 없다.")

run_example()

In [ ]:
import numpy as np
from scipy.linalg import hessenberg
import pandas as pd

def hessenberg_demo_fixed():
    np.random.seed(42)
    np.set_printoptions(precision=3, suppress=True)

    # 1. 임의의 5x5 행렬 생성
    A = np.random.rand(5, 5)
    print("--- 1. 원본 행렬 A ---")
    print(A)

    # 2. 헤센베르크 분해
    H, Q = hessenberg(A, calc_q=True)

    print("\n--- 2. 헤센베르크 행렬 H ---")
    print(H)
    
    # 3. 시각화 (에러 수정 부분)
    df_h = pd.DataFrame(H)
    
    print("\n[구조 시각화 (빈칸은 0)]")
    # 구버전 Pandas 호환을 위해 applymap 사용
    # applymap: DataFrame의 모든 요소에 함수를 적용함
    print(df_h.applymap(lambda x: f"{x:.2f}" if abs(x) > 1e-10 else ""))

    # 4. 검증
    A_reconstructed = Q @ H @ Q.T
    is_same = np.allclose(A, A_reconstructed)
    print(f"\n--- 3. 검증: A == QHQ^T ? -> {is_same}")
    
hessenberg_demo_fixed()

In [ ]:
import numpy as np
from numpy.linalg import norm, solve

# 1. Hermitian 행렬 A 정의 (실수 대칭 행렬)
A = np.array([[2.0, 1.0], 
              [1.0, 3.0]])

# 2. 초기 추정값 설정
alpha = 4.0          # 목표 고윳값 근처의 값
x_k = np.array([1.0, 1.0]) # 초기 고유벡터 추정값
x_k = x_k / norm(x_k)    # 정규화

# 3. 행렬 B = A - alpha*I 정의 (선형 시스템을 풀 행렬)
B = A - alpha * np.identity(A.shape[0])

# 4. 반복 계산 설정
max_iterations = 10
tolerance = 1e-6
lambda_old = 0.0

print(f"초기 추정값 alpha = {alpha}")
print(f"행렬 B = A - {alpha}I:\n{B}\n")

for k in range(max_iterations):
    # 5. 선형 방정식 풀기: B * y_{k+1} = x_k
    # solve(B, x_k)는 B의 역행렬을 직접 계산하지 않고 B y = x_k를 푼다.
    y_k_plus_1 = solve(B, x_k)

    # 6. 고유값 mu 계산 (B^-1의 고윳값)
    # Rayleigh 몫을 사용: mu = x_k.T @ y_k_plus_1 / x_k.T @ x_k (norm(x_k)=1이므로 분모 생략)
    mu_k_plus_1 = np.dot(x_k, y_k_plus_1)

    # 7. 원래 행렬 A의 고윳값 lambda 계산
    lambda_k_plus_1 = alpha + 1.0 / mu_k_plus_1

    # 8. 고유벡터 정규화
    x_k_plus_1 = y_k_plus_1 / norm(y_k_plus_1)

    # 9. 수렴 조건 확인
    if abs(lambda_k_plus_1 - lambda_old) < tolerance:
        break
    
    # 다음 반복을 위해 업데이트
    x_k = x_k_plus_1
    lambda_old = lambda_k_plus_1
    
    print(f"--- 반복 {k+1} ---")
    print(f"mu_k+1 (B^-1의 고윳값): {mu_k_plus_1:.6f}")
    print(f"lambda_k+1 (A의 고윳값): {lambda_k_plus_1:.6f}")
    print(f"x_k+1 (고유벡터): {x_k_plus_1}")

print("\n--- 결과 ---")
print(f"반복 횟수: {k+1}")
print(f"가장 가까운 고윳값 (lambda): {lambda_k_plus_1:.6f}")
print(f"대응하는 고유벡터 (x): {x_k_plus_1}")

# 검증 (NumPy 내장 함수 사용)
eigenvalues, eigenvectors = np.linalg.eigh(A)
print("\n--- NumPy 결과 (참값) ---")
print(f"고윳값: {eigenvalues}")
# alpha=4에 가장 가까운 고윳값은 3.618034 임

In [ ]:
import numpy as np
import time

def rayleigh_quotient_iteration_hermitian(A, x0, max_iter=10, tolerance=1e-10):
    # 초기 벡터 정규화
    x_k = x0 / np.linalg.norm(x0)
    # 레일리 몫 계산
    lambda_k = (np.conj(x_k).T @ A @ x_k).real
    # print(f"초기 고윳값 근사치 (λ_0): {lambda_k:.6f}") # 대형 행렬에서는 출력 생략
    for k in range(max_iter):
        lambda_prev = lambda_k
        # Shift된 시스템 풀기: (A - λ_k * I) * z = x_k
        B = A - lambda_k * np.eye(A.shape[0])
        try:
            # 선형 시스템 해결: 대형 행렬의 경우 이 부분이 가장 많은 계산을 차지한다.
            z_k_plus_1 = np.linalg.solve(B, x_k)
        except np.linalg.LinAlgError:
            # print(f"경고: {k+1}번째 반복에서 특이 행렬 발생.")
            break
        # 새로운 고유벡터 근사치 정규화
        x_k_plus_1 = z_k_plus_1 / np.linalg.norm(z_k_plus_1)
        # 새로운 레일리 몫 계산
        lambda_k_plus_1 = (np.conj(x_k_plus_1).T @ A @ x_k_plus_1).real
        # 수렴 확인
        eigenvalue_diff = np.abs(lambda_k_plus_1 - lambda_prev)
        # print(f"반복 {k+1}: λ = {lambda_k_plus_1:.6f}, |Δλ| = {eigenvalue_diff:.2e}") # 대형 행렬에서는 출력 생략
        if eigenvalue_diff < tolerance:
            return lambda_k_plus_1, x_k_plus_1, k + 1
        lambda_k = lambda_k_plus_1
        x_k = x_k_plus_1
    return lambda_k, x_k, max_iter

def main():
    # 행렬 크기 설정 (제법 큰 경우)
    N = 100
    print(f" 행렬 크기: {N} x {N}")
    print("-" * 30)

    # 1. Hermitian 행렬 생성
    # 무작위 복소수 행렬 C를 생성하고 C + C^H 를 계산하여 Hermitian 행렬 A를 만든다.
    # Hermitian 행렬은 A = A^H 이며 고윳값은 실수이다.
    C = np.random.rand(N, N) + 1j * np.random.rand(N, N)
    A = C + np.conj(C).T

    # 2. 초기 벡터 생성
    # 복소수 초기 벡터 (랜덤)
    x0 = np.random.rand(N) + 1j * np.random.rand(N)
    
    # RQI 시작 시간 측정
    start_time_rqi = time.time()
    
    # 3. RQI 실행
    # (일반적으로 RQI는 A의 가장 큰 고유값이나 가장 가까운 고유값으로 수렴)
    final_lambda, final_x, iterations = rayleigh_quotient_iteration_hermitian(A, x0, max_iter=20)
    
    end_time_rqi = time.time()

    # 4. NumPy 검증 및 비교
    start_time_numpy = time.time()
    w, v = np.linalg.eigh(A)
    end_time_numpy = time.time()

    # 5. 결과 출력
    print("\n[RQI 계산 결과]")
    print(f"계산된 고윳값 (λ): {final_lambda:.6f}")
    print(f"반복 횟수: {iterations}")
    print(f"계산 시간: {end_time_rqi - start_time_rqi:.4f} 초")
    
    # RQI 결과가 NumPy 결과 중 어떤 고윳값에 가까운지 확인
    min_diff_index = np.argmin(np.abs(w - final_lambda))
    print("\n[NumPy 검증]")
    print(f"NumPy 고윳값 중 RQI 결과와 가장 가까운 값 (w[{min_diff_index}]): {w[min_diff_index]:.6f}")
    
    # 정확도 검증: A*x와 lambda*x의 차이
    Ax = A @ final_x
    lambda_x = final_lambda * final_x
    residual_norm = np.linalg.norm(Ax - lambda_x)
    print(f"잔차 노름 (||Ax - λx||): {residual_norm:.2e}")
    
    print(f"NumPy 전체 고윳값 계산 시간: {end_time_numpy - start_time_numpy:.4f} 초")

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np

def rayleigh_quotient_iteration_hermitian(A, x0, max_iter=10, tolerance=1e-10):
    """
    Hermitian 행렬을 위한 Rayleigh Quotient Iteration (RQI) 함수.
    A: Hermitian 행렬 (A = A^H).
    x0: 복소수 초기 벡터 (complex-valued vector).
    """
    # 초기 벡터 정규화
    x_k = x0 / np.linalg.norm(x0)
    # 레일리 몫 계산: (x^H A x) / (x^H x). 정규화했으므로 분모는 1.
    # np.conj(x_k).T는 켤레 전치 x^H이다.
    lambda_k = np.conj(x_k).T @ A @ x_k
    # Hermitian 행렬의 고윳값은 실수여야 하므로 실수부만 취한다.
    lambda_k = lambda_k.real
    print(f"초기 고윳값 근사치 (λ_0): {lambda_k:.6f}")

    for k in range(max_iter):
        
        lambda_prev = lambda_k
        
        # Shift된 시스템 풀기: (A - λ_k * I) * z = x_k
        # λ_k는 실수이고, I는 항등 행렬이다.
        B = A - lambda_k * np.eye(A.shape[0])
        
        # 선형 시스템 Bz = x_k 풀기
        try:
            # np.linalg.solve는 복소수 방정식을 처리할 수 있다.
            z_k_plus_1 = np.linalg.solve(B, x_k)
        except np.linalg.LinAlgError:
            print(f"경고: {k+1}번째 반복에서 행렬이 특이 행렬이 되어 해를 구할 수 없다.")
            break

        # 새로운 고유벡터 근사치 정규화
        x_k_plus_1 = z_k_plus_1 / np.linalg.norm(z_k_plus_1)
        
        # 새로운 레일리 몫 계산
        lambda_k_plus_1_complex = np.conj(x_k_plus_1).T @ A @ x_k_plus_1
        # Hermitian 고윳값은 실수이므로 실수부만 취한다.
        lambda_k_plus_1 = lambda_k_plus_1_complex.real
        
        # 수렴 확인
        eigenvalue_diff = np.abs(lambda_k_plus_1 - lambda_prev)
        
        print(f"반복 {k+1}: λ = {lambda_k_plus_1:.6f}, |Δλ| = {eigenvalue_diff:.2e}")
        
        if eigenvalue_diff < tolerance:
            print(f"\n {k+1}번째 반복에서 수렴 (오차 허용치 미만).")
            return lambda_k_plus_1, x_k_plus_1, k + 1
        
        # 업데이트
        lambda_k = lambda_k_plus_1
        x_k = x_k_plus_1
        
    print(f"\n 최대 반복 횟수 ({max_iter}) 도달.")
    return lambda_k, x_k, max_iter
# Hermitian 행렬 A 정의 (A^H = A)
A = np.array([
    [2, 3 + 4j],
    [3 - 4j, 2]
])
# 초기 벡터 x0 (복소수 벡터)
x0 = np.array([1.0 + 0j, 0.5 + 0.5j]) 
# RQI 실행
final_lambda, final_x, iterations = rayleigh_quotient_iteration_hermitian(A, x0, max_iter=5)
# 결과 출력
print("\n--- RQI 결과 ---")
print(f"계산된 고윳값 (λ): {final_lambda:.6f}")
print(f"계산된 고유벡터 (x): {final_x}")
# NumPy의 고유값/고유벡터 함수로 검증
w, v = np.linalg.eigh(A) # eigh는 Hermitian/대칭 행렬 전용 함수
print("\n--- NumPy 검증 ---")
print(f"NumPy 고윳값: {w}")
print(f"NumPy 고유벡터 (열 벡터):\n{v}")    

In [ ]:
import numpy as np

def jacobi_method(A, b, max_iter=100, tol=1e-6):
    """
    Jacobi 반복법으로 Ax = b를 푸는 함수.

    Args:
        A (np.array): 계수 행렬
        b (np.array): 상수 벡터
        max_iter (int): 최대 반복 횟수
        tol (float): 수렴 허용 오차 (잔차 노름)

    Returns:
        np.array: 근사 해 벡터 x
        int: 실제 반복 횟수
        list: 각 반복에서의 잔차 노름 리스트
    """
    n = len(b)
    x = np.zeros(n)  # 초기 추측값 (모두 0으로 설정)
    residuals = []

    # 행렬 A를 D (대각 행렬)와 R (나머지 행렬)로 분리
    D = np.diag(np.diag(A))
    R = A - D

    # D의 역행렬을 계산한다. (대각 성분이 0인 경우를 주의해야 한다.)
    try:
        D_inv = np.linalg.inv(D)
    except np.linalg.LinAlgError:
        print("Error: Diagonal element is zero (D is singular). Jacobi method may not be applicable.")
        return x, 0, residuals

    for k in range(max_iter):
        x_new = np.dot(D_inv, (b - np.dot(R, x)))

        # 잔차 계산 및 수렴 확인
        residual_vec = np.dot(A, x_new) - b
        residual_norm = np.linalg.norm(residual_vec)
        residuals.append(residual_norm)

        if residual_norm < tol:
            print(f"Converged after {k+1} iterations.")
            return x_new, k + 1, residuals

        # 해 업데이트
        x = x_new

    print(f"Maximum iterations ({max_iter}) reached. Residual norm: {residual_norm:.2e}")
    return x, max_iter, residuals

# --- 예제 시스템 정의 ---
# Ax = b
# A는 대각 우세(Diagonally Dominant) 행렬일 때 Jacobi 방법이 수렴할 가능성이 높다.
# A = [[10, -1, 2, 0],
#      [-1, 11, -1, 3],
#      [2, -1, 10, -1],
#      [0, 3, -1, 8]]
# b = [6, 25, -11, 15]
A = np.array([
    [10.0, -1.0, 2.0, 0.0],
    [-1.0, 11.0, -1.0, 3.0],
    [2.0, -1.0, 10.0, -1.0],
    [0.0, 3.0, -1.0, 8.0]
])

b = np.array([6.0, 25.0, -11.0, 15.0])

# --- Jacobi 반복법 실행 ---
x_solution, iterations, residual_history = jacobi_method(A, b)

# 결과 출력
print("\n--- 결과 ---")
print("근사 해 x:")
print(x_solution)
print(f"총 반복 횟수: {iterations}")

# numpy의 직접 해법과 비교 (정확한 해)
x_exact = np.linalg.solve(A, b)
print("\n정확한 해 (np.linalg.solve):")
print(x_exact)

# 오차 확인
error = np.linalg.norm(x_solution - x_exact)
print(f"\n정확한 해와의 L2 노름 오차: {error:.2e}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 문제 설정 (정답이 [1, 2, 3]인 문제)
A = np.array([[4.0, 1.0, 1.0],
              [1.0, 5.0, 2.0],
              [1.0, 2.0, 4.0]])

b = np.array([9.0, 17.0, 17.0])
actual_solution = np.array([1.0, 2.0, 3.0])
# 초기 추정값 (0, 0, 0)
x0 = np.zeros(len(b))
# ---------------------------------------------------------
# 1. 야코비 방법 (Jacobi Method)
# 특징: x_new 계산 시 오직 x_old 값만 사용함 (병렬 처리 가능)
# ---------------------------------------------------------
def jacobi(A, b, x0, tol=1e-6, max_iter=100):
    x = x0.copy()
    n = len(b)
    history = [] # 오차 기록용
    for k in range(max_iter):
        x_new = np.zeros_like(x)
        for i in range(n):
            # i번째 행에서 대각성분(A[i,i])을 제외한 나머지 항들의 합 계산
            s = sum(A[i, j] * x[j] for j in range(n) if j != i)
            x_new[i] = (b[i] - s) / A[i, i]
        # 수렴 여부 확인 (L2 Norm)
        error = np.linalg.norm(x_new - x)
        history.append(error)
        if error < tol:
            return x_new, k+1, history
        x = x_new
    return x, max_iter, history

# ---------------------------------------------------------
# 2. 가우스-자이델 방법 (Gauss-Seidel Method)
# 특징: 갱신된 값을 즉시 다음 계산에 반영함 (야코비보다 수렴 빠름)
# ---------------------------------------------------------
def gauss_seidel(A, b, x0, tol=1e-6, max_iter=100):
    x = x0.copy()
    n = len(b)
    history = []
    for k in range(max_iter):
        x_old = x.copy()
        for i in range(n):
            # i번째 행 계산 시, 이미 갱신된 x[0]~x[i-1]을 사용하게 됨
            s = sum(A[i, j] * x[j] for j in range(n) if j != i)
            x[i] = (b[i] - s) / A[i, i]
        error = np.linalg.norm(x - x_old)
        history.append(error)
        if error < tol:
            return x, k+1, history
    return x, max_iter, history

# ---------------------------------------------------------
# 3. SOR 방법 (Successive Over-Relaxation)
# 특징: 가우스-자이델 결과에 가중치(omega)를 두어 더 멀리 점프함
# ---------------------------------------------------------
def sor(A, b, x0, omega, tol=1e-6, max_iter=100):
    x = x0.copy()
    n = len(b)
    history = []
    for k in range(max_iter):
        x_old_iter = x.copy()
        for i in range(n):
            # 먼저 가우스-자이델로 임시 값(sigma) 계산
            s = sum(A[i, j] * x[j] for j in range(n) if j != i)
            x_gs = (b[i] - s) / A[i, i]
            # SOR 공식: (1-w)*이전값 + w*GS값
            x[i] = (1 - omega) * x[i] + omega * x_gs
        error = np.linalg.norm(x - x_old_iter)
        history.append(error)
        
        if error < tol:
            return x, k+1, history
            
    return x, max_iter, history

# ---------------------------------------------------------
# 실행 및 결과 비교
# ---------------------------------------------------------
tol = 1e-12
sol_j, iter_j, hist_j = jacobi(A, b, x0, tol)
sol_g, iter_g, hist_g = gauss_seidel(A, b, x0, tol)
sol_s, iter_s, hist_s = sor(A, b, x0, omega=1.1, tol=tol) # w=1.1 사용

print(f"Goal: {actual_solution}")
print("-" * 50)
print(f"1. Jacobi       : {np.round(sol_j, 8)} | Iterations: {iter_j}")
print(f"2. Gauss-Seidel : {np.round(sol_g, 8)} | Iterations: {iter_g}")
astring=r'3. SOR $\omega=1.1$   :'+ f"{np.round(sol_s,8)}"+f" | Iterations: {iter_s}"
print(astring)

# 수렴 속도 그래프 그리기
plt.figure(figsize=(10, 6))
plt.plot(hist_j, 'o', label=f'Jacobi ({iter_j} iter)', linestyle='--')
plt.plot(hist_g, '+', label=f'Gauss-Seidel ({iter_g} iter)', linestyle='-.')
astring=r'SOR $\omega=1.1$ ' + f'({iter_s} iter)' 
plt.plot(hist_s, '*', label=astring, linewidth=2, linestyle='--')
plt.yscale('log') # 로그 스케일로 봐야 수렴 속도 차이가 잘 보임
plt.xlabel('Iteration', fontsize=18)
plt.ylabel('Error (Log scale)', fontsize=18)
plt.title('Convergence Comparison: Jacobi vs GS vs SOR')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.show()

In [ ]:
import numpy as np

def jacobi_method(A, b, max_iter=100, tol=1e-6):
    """
    Jacobi 반복법으로 Ax = b를 푸는 함수.

    Args:
        A (np.array): 계수 행렬
        b (np.array): 상수 벡터
        max_iter (int): 최대 반복 횟수
        tol (float): 수렴 허용 오차 (잔차 노름)

    Returns:
        np.array: 근사 해 벡터 x
        int: 실제 반복 횟수
        list: 각 반복에서의 잔차 노름 리스트
    """
    n = len(b)
    x = np.zeros(n)  # 초기 추측값 (모두 0으로 설정)
    residuals = []
    # 행렬 A를 D (대각 행렬)와 R (나머지 행렬)로 분리
    D = np.diag(np.diag(A))
    R = A - D
    # D의 역행렬을 계산한다. (대각 성분이 0인 경우를 주의해야 한다.)
    try:
        D_inv = np.linalg.inv(D)
    except np.linalg.LinAlgError:
        print("Error: Diagonal element is zero (D is singular). Jacobi method may not be applicable.")
        return x, 0, residuals
    for k in range(max_iter):
        x_new = np.dot(D_inv, (b - np.dot(R, x)))
        # 잔차 계산 및 수렴 확인
        residual_vec = np.dot(A, x_new) - b
        residual_norm = np.linalg.norm(residual_vec)
        residuals.append(residual_norm)
        if residual_norm < tol:
            print(f"Converged after {k+1} iterations.")
            return x_new, k + 1, residuals
        # 해 업데이트
        x = x_new
    print(f"Maximum iterations ({max_iter}) reached. Residual norm: {residual_norm:.2e}")
    return x, max_iter, residuals

# --- 예제 시스템 정의 ---
# Ax = b
# A는 대각 우세(Diagonally Dominant) 행렬일 때 Jacobi 방법이 수렴할 가능성이 높다.
# A = [[10, -1, 2, 0],
#      [-1, 11, -1, 3],
#      [2, -1, 10, -1],
#      [0, 3, -1, 8]]
# b = [6, 25, -11, 15]
A = np.array([
    [10.0, -1.0, 2.0, 0.0],
    [-1.0, 11.0, -1.0, 3.0],
    [2.0, -1.0, 10.0, -1.0],
    [0.0, 3.0, -1.0, 8.0]
])
b = np.array([6.0, 25.0, -11.0, 15.0])

# --- Jacobi 반복법 실행 ---
x_solution, iterations, residual_history = jacobi_method(A, b)
# 결과 출력
print("\n--- 결과 ---")
print("근사 해 x:")
print(x_solution)
print(f"총 반복 횟수: {iterations}")
# numpy의 직접 해법과 비교 (정확한 해)
x_exact = np.linalg.solve(A, b)
print("\n정확한 해 (np.linalg.solve):")
print(x_exact)
# 오차 확인
error = np.linalg.norm(x_solution - x_exact)
print(f"\n정확한 해와의 L2 노름 오차: {error:.2e}")

In [ ]:
import numpy as np
from scipy.sparse.linalg import LinearOperator, cg

# 1. 행렬-벡터 곱을 정의하는 함수 (오퍼레이터 A의 역할)
def matvec_laplacian(v):
    """
    3차원 이산 라플라시안 오퍼레이터의 행렬-벡터 곱 A*v를 계산하는 함수.
    (예시를 위해 간단한 3x3 대각선 행렬처럼 동작하도록 정의)
    """
    # 실제 라플라시안 행렬은 매우 크고 복잡하지만, 여기서는 개념을 위해 간단히 정의
    # A = [[4, -1, 0],
    #      [-1, 4, -1],
    #      [0, -1, 4]] 처럼 동작하도록 정의
    n = len(v)
    Av = np.zeros_like(v)
    # 대각 성분 (4 * v[i])
    Av = 4 * v
    # 비대각 성분 (-1 * v[i-1] 및 -1 * v[i+1])
    if n > 1:
        Av[0] -= v[1]
        Av[n-1] -= v[n-2]
    if n > 2:
        Av[1:n-1] -= v[0:n-2] # v[i-1]
        Av[1:n-1] -= v[2:n]   # v[i+1]
    return Av

# 2. LinearOperator 생성
N = 100 # 벡터의 크기 (행렬 A의 크기는 N x N)
A_op = LinearOperator((N, N), matvec=matvec_laplacian, dtype=float)
# 3. 상수 벡터 b 생성
# 임의의 벡터 b를 생성
b = np.random.rand(N)
# 4. 공액 기울기법(Conjugate Gradient, CG)으로 해 x 계산
# cg 함수는 A 대신 LinearOperator를 인수로 받을 수 있다.
# tol: 수렴 허용 오차
x_cg, info = cg(A_op, b, rtol=1e-8)
# 5. 결과 확인
if info == 0:
    print(f" CG Method: Converged successfully in {info} iterations.")
    # 잔차 노름 확인: ||Ax - b||
    residual_norm = np.linalg.norm(A_op.dot(x_cg) - b)
    print(f"최종 잔차 노름: {residual_norm:.2e}")
else:
    print(f" CG Method: Did not converge. info = {info}")

# LinearOperator의 기본 속성 확인
print(f"\nLinearOperator의 차원: {A_op.shape}")

In [ ]:
import numpy as np

# --- 1. 일반 행렬-벡터 곱셈 (MVM): 검증 기준 (O(N^2)) ---
def naive_mvm(V_mat, psi_vec):
    """
    일반적인 행렬-벡터 곱셈 (V * psi)을 수행한다. O(N^2)
    """
    return V_mat @ psi_vec

# --- 2. FFT 기반 합성곱 (Convolution): 가속화 연산 (O(N log N)) ---
def fft_convolution_mvm(V_q, psi_vec):
    """
    FFT를 이용한 순환 합성곱을 통해 행렬-벡터 곱셈을 수행한다.
    """
    # 1. FFT 변환 (V_q와 psi_vec 모두)
    FV = np.fft.fft(V_q)
    Fpsi = np.fft.fft(psi_vec)
    
    # 2. 주파수 영역에서 곱셈 (점별 곱셈)
    Fphi = FV * Fpsi
    
    # 3. 역 FFT (IFFT)로 G 공간으로 복원
    phi_fft = np.fft.ifft(Fphi)
    
    return phi_fft

# --- 3. 문제 설정 및 실행 ---
N = 8  # 벡터/행렬의 크기 (차원)

# ψ(G') 벡터 (파동 함수 계수)
np.random.seed(42)
psi_G_prime = (np.random.rand(N) + 1j * np.random.rand(N))

# V(q) 함수 정의 (Potential이 G-G' 차이에만 의존)
def potential_function(q):
    return 1.0 / (1 + np.abs(q))

# 순환 행렬 V 생성
V_matrix = np.zeros((N, N), dtype=complex)
for G in range(N):
    for G_prime in range(N):
        q = (G - G_prime) % N
        if q > N / 2:
            q = q - N
        V_matrix[G, G_prime] = potential_function(q)

# 합성곱에 사용될 V(q) 벡터 (순환 행렬의 첫 행/열)
V_q = V_matrix[0, :]

# 계산 수행
phi_naive = naive_mvm(V_matrix, psi_G_prime)
phi_fft = fft_convolution_mvm(V_q, psi_G_prime)

# --- 4. 검증 및 출력 ---
print("--- FFT 기반 행렬-벡터 곱셈 예제 및 검증 ---")
print(f"차원 (N): {N}")
print("-" * 50)
print(f"일반 MVM 결과 (phi_naive)의 처음 5개 요소:")
print(phi_naive[:5])
print(f"\nFFT 합성곱 결과 (phi_fft)의 처음 5개 요소:")
print(phi_fft[:5])
print("-" * 50)

# NumPy의 allclose를 사용하여 부동 소수점 오차를 고려하여 검증
is_verified = np.allclose(phi_naive, phi_fft)

print(f" 검증 결과: 두 계산 결과가 일치하는가? {is_verified}")
print(f"최대 절대 오차: {np.max(np.abs(phi_naive - phi_fft)):.2e}")

In [ ]:
import numpy as np
from scipy.sparse.linalg import LinearOperator, cg
import matplotlib.pyplot as plt 

# --- 시스템 및 그리드 간격 정의 ---
N = 100  # 전체 격자점 개수
# 1. 그리드 간격 h 계산
L = 1.0  # 공간 길이 (0부터 1까지)
h = L / (N - 1) 

x_left_bc = 0.0
x_right_bc = 0.0
N_internal = N - 2 

# 1. 행렬-벡터 곱 함수 정의 (Ax 연산) - A_old: h^2이 없는 버전
def matvec_laplacian_with_bc(v):
    """
    1D Laplacian operator (A_old: -u_{i-1} + 2u_i - u_{i+1}) 연산 수행.
    이 함수 자체에는 h^2을 곱하지 않고, 우변 b에 h^2을 곱하여 처리한다.
    """
    n = len(v)
    Av = np.zeros_like(v)
    # 중앙 항: 2 * v_i
    Av = 2.0 * v
    # 왼쪽 및 오른쪽 항: -v_{i-1} 및 -v_{i+1}
    if n > 0:
        Av[:-1] -= v[1:] 
        Av[1:] -= v[:-1]
    # 경계값 기여 처리 (BCs Contribution)
    # A_{int} * x_{int} = h^2 * f - A_{mixed} * x_{BC}
    # 여기서 Av는 A_{int} * x_{int} 이므로, A_{mixed} * x_{BC}를 더해야 함.
    if n > 0:
        Av[0] -= x_left_bc    # A_{mixed}는 -1로 정의됨
        Av[n - 1] -= x_right_bc
    return Av

# 2. LinearOperator 생성 
A_op = LinearOperator((N_internal, N_internal), matvec=matvec_laplacian_with_bc, dtype=float)
# 3. 우변 벡터 b (내부 노드만 해당) 생성 및 h^2 적용
# f(t) = 1.0 (참고: Ax = b 형식으로 쓰기 위해 우변 f(t) = 1.0으로 수정. 
#            해석해 u_exact(t) = 0.5t^2 - 0.5t는 -u''=1의 해이다.)
f = 1.0 * np.ones(N) 
f_internal = f[1:N-1]
# 2. **핵심 수정**: 우변 벡터 b에 h^2을 곱한다.
b_internal = -h**2 * f_internal  # b = h^2 * f
# 4. 공액 기울기법(CG)으로 해 x_internal 계산
initial_guess = 0.5 * np.ones(N_internal)
print(f"Solving with h={h:.4f} and h^2 in b...")
x_internal, info = cg(A_op, b_internal, rtol=1e-8, x0=initial_guess)
# 5. 최종 해 벡터 구성 및 시각화 (이전과 동일)
if info == 0:
    print(f" CG method: Converged successfully. iterations: {info}")
    # ... (x_solution 구성)
    x_solution = np.zeros(N)
    x_solution[0] = x_left_bc
    x_solution[-1] = x_right_bc
    x_solution[1:-1] = x_internal
    
    # ... (시각화)
    x_coords = np.linspace(0, 1, N)
    # Exact solution for -u''(t) = 1, u(0)=u(1)=0
    u_exact = 0.5 * x_coords**2 - 0.5 * x_coords 
    
    plt.figure(figsize=(10, 6))
    plt.plot(x_coords, u_exact, 'b-', linewidth=3, label='Exact solution', alpha=0.7)
    plt.plot(x_coords, x_solution, 'ro', markersize=4, label='Iterative solution(CG)', alpha=0.8)
    plt.title(f"Solution with grid spacing $h={h:.4f}$", fontsize=16)
    plt.xlabel("Spatial coordinate (t)", fontsize=14)
    plt.ylabel("Solution value ($u(t)$)", fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()

else:
    print(f" CG Method: Did not converge. info = {info}")

In [ ]:
import numpy as np
from scipy.sparse.linalg import LinearOperator, cg
import matplotlib.pyplot as plt

# --------------------------------------------------------------------------
# --- 경계 조건 및 시스템 정의 ---
# --------------------------------------------------------------------------
L = 1.0              # 도메인 길이 (추가)
N = 100              # 전체 격자점 개수
x_left_bc = 0.0      # 좌측 Dirichlet 경계 조건 u(0)
x_right_bc = 0.0     # 우측 Dirichlet 경계 조건 u(L)
N_internal = N - 2   # 내부 격자점 개수
h = L / (N - 1)      # 격자 간격

# Fornberg 4차 계수 (중앙 차분, 5점 스텐실)
# u''(i) * h^2 ~ (1/12) * [-1, 16, -30, 16, -1]
# 이 계수들은 행렬 A의 내부 계수가 된다.
C_4 = np.array([-1.0, 16.0, -30.0, 16.0, -1.0]) / 12.0

# 1. 고정밀도 행렬-벡터 곱 함수 정의 (4차 + 2차 혼합 스텐실)
def matvec_high_order_laplacian(v):
    """
    4차 중앙 차분 스텐실을 사용하는 행렬-벡터 곱 함수.
    경계 근처 노드(v[0], v[1], v[n-2], v[n-1])는 2차 스텐실을 사용한다.
    v: 내부 노드 해 벡터 (u_1, ..., u_{N-2}), 크기 n = N_internal
    """
    n = len(v)
    Av = np.zeros_like(v)

    # Note: 경계 조건 (u_0=0, u_{N-1}=0)은 우변 벡터 b에 포함되어야 하며,
    # 여기서는 순수한 행렬-벡터 곱셈 A*v만 수행한다.

    # i=1 (v[0]): 2차 스텐실 (u_0 - 2u_1 + u_2). u_0=0
    if n >= 1:
        Av[0] = -2.0 * v[0] + 1.0 * v[1]

    # i=2 (v[1]): 2차 스텐실 (u_1 - 2u_2 + u_3)
    if n >= 2:
        Av[1] = 1.0 * v[0] - 2.0 * v[1] + 1.0 * v[2]

    # Core 4th-order stencil (v[2] up to v[n-3])
    # 이 영역에서 4차 정확도가 달성된다.
    for i in range(2, n - 2):
        # Stencil for Av[i] is centered around v[i] using [v[i-2]...v[i+2]]
        Av[i] = (C_4[0] * v[i-2] + C_4[1] * v[i-1] + C_4[2] * v[i] +
                 C_4[3] * v[i+1] + C_4[4] * v[i+2])

    # i=n-2 (v[n-3]): 2차 스텐실 (u_{N-4} - 2u_{N-3} + u_{N-2})
    if n >= 3:
        Av[n-2] = 1.0 * v[n-3] - 2.0 * v[n-2] + 1.0 * v[n-1]

    # i=n-1 (v[n-2]): 2차 스텐실 (u_{N-3} - 2u_{N-2} + u_{N-1}). u_{N-1}=0
    if n >= 1:
        Av[n-1] = 1.0 * v[n-2] - 2.0 * v[n-1]
        
    return Av

# 2. LinearOperator 생성 (고차 오퍼레이터 사용)
A_op = LinearOperator((N_internal, N_internal), matvec=matvec_high_order_laplacian, dtype=float)
# 3. 우변 벡터 b (내부 노드만 해당) 생성
# 미분 방정식: u''(t) = f(t). 여기서 f(t) = 1.0으로 가정한다.
# 이산화 시스템: A * x_internal = h^2 * f_internal
f_val = 1.0 # f(t) = 1.0
f = h**2 * f_val * np.ones(N) 
b_internal = f[1:N-1] # 내부 노드의 우변 벡터
# 4. 공액 기울기법(CG)으로 해 x_internal 계산
print(f"Solving linear system of size {N_internal}x{N_internal} using High-Order FD...")
initial_guess = np.zeros(len(b_internal))
# 초기 추측 생성
for i in range(len(b_internal)):
    initial_guess[i] = np.random.random() - 0.5
# 높은 정밀도를 위해 rtol을 강화하고 maxiter를 증가시킨다.
x_internal, info = cg(A_op, b_internal, rtol=1e-12, maxiter=1000, x0=initial_guess)
# 5. 최종 해 벡터 구성
if info == 0:
    print(f" CG method: converged successfully.")
    x_solution = np.zeros(N)
    x_solution[0] = x_left_bc
    x_solution[-1] = x_right_bc
    x_solution[1:-1] = x_internal
    residual_norm = np.linalg.norm(A_op.dot(x_internal) - b_internal)
    print("\n--- 결과 요약 (고차 스텐실 적용) ---")
    print(f"사용된 격자점 수 N: {N}")
    print(f"총 반복 횟수: {info}")
    print(f"최종 잔차 노름: {residual_norm:.2e}")
    # ----------------------------------------------------
    ##  해 시각화 및 해석해 비교
    # ----------------------------------------------------
    # 1. 공간 좌표 생성 (0부터 L까지 N개의 점)
    x_coords = np.linspace(0, L, N)
    # 2. 해석해 (Exact Solution) 계산: u''(t) = 1.0, u(0)=0, u(1)=0
    # u(t) = 0.5 * t^2 - 0.5 * t
    u_exact = 0.5 * x_coords**2 - 0.5 * x_coords
    # 3. 그래프 그리기
    plt.figure(figsize=(10, 6))
    # 해석해 (파란색 실선)
    plt.plot(x_coords, u_exact, 'b-', linewidth=3, label='Exact solution $u(t) = 0.5t^2 - 0.5t$', alpha=0.7)
    # 반복법으로 구한 근사 해 (빨간색 점)
    plt.plot(x_coords, x_solution, 'ro', markersize=4, label='4th-order iterative solution(CG)', alpha=0.8)
    # 경계 조건 강조
    plt.plot(x_coords[[0, -1]], x_solution[[0, -1]], 'ko', markersize=6, label='Boundary points')
    plt.title(f"Comparison of high-order FD solution vs. Exact solution ($N={N}$)", fontsize=16)
    plt.xlabel("Spatial coordinate (t)", fontsize=14)
    plt.ylabel("Solution value ($u(t)$)", fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()
else:
    print(f" CG method: Did not converge. info = {info}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# ---------------------------------------------------------
# 1. 문제 생성기 (N x N 크기)
# ---------------------------------------------------------
def create_large_problem(n):
    np.random.seed(42) # 재현성을 위해 시드 고정
    
    # 1. 랜덤 행렬 생성
    A = np.random.rand(n, n)
    
    # 2. 대각 지배적 행렬로 만들기 (수렴 보장 필수 조건)
    # 각 행의 절대값 합보다 대각 성분을 더 크게 만듦
    # 대각 성분에 n을 더해주면 무조건 대각 지배적이 됨
    row_sums = np.sum(np.abs(A), axis=1)
    np.fill_diagonal(A, row_sums + 1.0) 
    
    # 3. 정답(True Solution) 설정: [1, 1, ..., 1]
    x_true = np.ones(n)
    
    # 4. b 벡터 역산출 (b = A * x_true)
    b = np.dot(A, x_true)
    
    return A, b, x_true

# ---------------------------------------------------------
# 2. 솔버 함수들 (이전과 동일 로직, 대규모 처리에 맞게 최적화)
# ---------------------------------------------------------
def solve_jacobi(A, b, tol=1e-6, max_iter=1000):
    n = len(b)
    x = np.zeros(n) # 초기값 0
    history = []
    
    # 대각 성분과 그 외 성분을 미리 분리 (속도 향상)
    D = np.diag(A)
    R = A - np.diag(D) # Remainder matrix
    
    start_time = time.time()
    for k in range(max_iter):
        # Jacobi의 벡터화 연산: x_new = (b - (A-D)x) / D
        x_new = (b - np.dot(R, x)) / D
        
        error = np.linalg.norm(x_new - x)
        history.append(error)
        
        if error < tol:
            return x_new, k+1, history, time.time() - start_time
        x = x_new
        
    return x, max_iter, history, time.time() - start_time

def solve_gauss_seidel(A, b, tol=1e-6, max_iter=1000):
    n = len(b)
    x = np.zeros(n)
    history = []
    
    start_time = time.time()
    for k in range(max_iter):
        x_old = x.copy()
        
        # GS는 순차적 업데이트라 벡터화가 어려워 루프 사용
        for i in range(n):
            # s = sum(A[i,j] * x[j]) for j != i
            # 이를 최적화: (A[i,:] dot x) - A[i,i]*x[i]
            row_dot = np.dot(A[i, :], x)
            s = row_dot - A[i, i] * x[i]
            x[i] = (b[i] - s) / A[i, i]
            
        error = np.linalg.norm(x - x_old)
        history.append(error)
        
        if error < tol:
            return x, k+1, history, time.time() - start_time
            
    return x, max_iter, history, time.time() - start_time

def solve_sor(A, b, omega, tol=1e-6, max_iter=1000):
    n = len(b)
    x = np.zeros(n)
    history = []
    
    start_time = time.time()
    for k in range(max_iter):
        x_old_iter = x.copy()
        
        for i in range(n):
            row_dot = np.dot(A[i, :], x)
            s = row_dot - A[i, i] * x[i]
            
            x_gs = (b[i] - s) / A[i, i]
            x[i] = (1 - omega) * x[i] + omega * x_gs
            
        error = np.linalg.norm(x - x_old_iter)
        history.append(error)
        
        if error < tol:
            return x, k+1, history, time.time() - start_time
            
    return x, max_iter, history, time.time() - start_time

# ---------------------------------------------------------
# 3. 메인 실행 (N=100)
# ---------------------------------------------------------
N = 100
print(f"Generating {N}x{N} System...")
A, b, x_true = create_large_problem(N)

print("Solving...")
tol = 1e-6

# Jacobi
x_j, iter_j, hist_j, t_j = solve_jacobi(A, b, tol=tol)

# Gauss-Seidel
x_g, iter_g, hist_g, t_g = solve_gauss_seidel(A, b, tol=tol)

# SOR (omega=1.2) - 난수 행렬에서는 과하면 발산하므로 보수적으로 설정
omega = 1.2
x_s, iter_s, hist_s, t_s = solve_sor(A, b, omega=omega, tol=tol)

# ---------------------------------------------------------
# 4. 결과 출력 및 시각화
# ---------------------------------------------------------
print("-" * 60)
print(f"{'Method':<15} | {'Iterations':<10} | {'Time (sec)':<10} | {'Max Error'}")
print("-" * 60)
print(f"{'Jacobi':<15} | {iter_j:<10} | {t_j:.4f}     | {np.max(np.abs(x_j - x_true)):.2e}")
print(f"{'Gauss-Seidel':<15} | {iter_g:<10} | {t_g:.4f}     | {np.max(np.abs(x_g - x_true)):.2e}")
print(f"{'SOR (w=1.2)':<15} | {iter_s:<10} | {t_s:.4f}     | {np.max(np.abs(x_s - x_true)):.2e}")
print("-" * 60)

plt.figure(figsize=(10, 6))
plt.plot(hist_j, label=f'Jacobi ({iter_j} iter)', linestyle=':', linewidth=2)
plt.plot(hist_g, label=f'Gauss-Seidel ({iter_g} iter)', linestyle='--', linewidth=2)
astring=r'SOR $\omega=1.2$ ' + f'({iter_s} iter)' 
plt.plot(hist_s, '*', label=astring, linewidth=2, linestyle='--')
#plt.plot(hist_s, label=f'SOR w={omega} ({iter_s} iter)', linestyle='-', linewidth=2)

plt.yscale('log')
plt.xlabel('Iteration')
plt.ylabel('Error (Log scale)')
plt.title(f'Convergence Speed on {N}x{N} Matrix')
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.4)
plt.show()

In [ ]:
import numpy as np
import time

def generate_spd_matrix(n):
    """
    Conjugate Gradient 수렴을 보장하고, 다른 반복법의 수렴성을 높이기 위해
    대칭 양의 정부호(Symmetric Positive Definite) 행렬을 생성한다.
    동시에 대각 지배적(Diagonally Dominant) 성질을 강화한다.
    """
    np.random.seed(42)  # 재현 가능성을 위해 시드 고정
    A = np.random.rand(n, n)
    A = 0.5 * (A + A.T)  # 대칭 행렬로 변환
    # 대각 성분을 크게 만들어 양의 정부호 및 대각 지배적 성질 강화
    A = A + n * np.eye(n) 
    return A

# 1. Jacobi Method
def jacobi(A, b, x0, tol=1e-10, max_iter=1000):
    n = len(b)
    x = x0.copy()
    x_new = np.zeros_like(x)
    
    for k in range(max_iter):
        for i in range(n):
            s = sum(A[i][j] * x[j] for j in range(n) if j != i)
            x_new[i] = (b[i] - s) / A[i][i]
        
        # 수렴 판정 (L2 norm)
        if np.linalg.norm(x_new - x) < tol:
            return x_new, k + 1
        x = x_new.copy()
        
    return x, max_iter

# 2. Gauss-Seidel Method
def gauss_seidel(A, b, x0, tol=1e-10, max_iter=1000):
    n = len(b)
    x = x0.copy()
    
    for k in range(max_iter):
        x_old = x.copy()
        for i in range(n):
            s1 = sum(A[i][j] * x[j] for j in range(i))      # 이미 업데이트된 값
            s2 = sum(A[i][j] * x_old[j] for j in range(i + 1, n)) # 이전 값
            x[i] = (b[i] - s1 - s2) / A[i][i]
            
        if np.linalg.norm(x - x_old) < tol:
            return x, k + 1
            
    return x, max_iter

# 3. SOR (Successive Over-Relaxation)
def sor(A, b, x0, omega, tol=1e-10, max_iter=1000):
    n = len(b)
    x = x0.copy()
    
    for k in range(max_iter):
        x_old = x.copy()
        for i in range(n):
            s1 = sum(A[i][j] * x[j] for j in range(i))
            s2 = sum(A[i][j] * x_old[j] for j in range(i + 1, n))
            # Gauss-Seidel 값 계산
            x_gs = (b[i] - s1 - s2) / A[i][i]
            # 가중치 적용
            x[i] = (1 - omega) * x_old[i] + omega * x_gs
            
        if np.linalg.norm(x - x_old) < tol:
            return x, k + 1
            
    return x, max_iter

# 4. Conjugate Gradient Method
def conjugate_gradient(A, b, x0, tol=1e-10, max_iter=1000):
    x = x0.copy()
    r = b - A @ x  # 초기 잔차 (Residual)
    p = r.copy()   # 초기 검색 방향
    rs_old = r @ r # 잔차의 내적
    
    for k in range(max_iter):
        Ap = A @ p
        alpha = rs_old / (p @ Ap) # 스텝 사이즈
        x = x + alpha * p
        r = r - alpha * Ap
        
        rs_new = r @ r
        if np.sqrt(rs_new) < tol:
            return x, k + 1
            
        p = r + (rs_new / rs_old) * p # 새로운 검색 방향
        rs_old = rs_new
        
    return x, max_iter

# --- 실행 및 테스트 ---

# 문제 설정
N = 10  # 행렬 크기 (10x10)
A = generate_spd_matrix(N)
x_true = np.ones(N)  # 정답을 모두 1로 설정
b = A @ x_true       # Ax = b 계산하여 b 생성

# 초기 추정값 (모두 0으로 시작)
x0 = np.zeros(N)

print(f"시스템 크기: {N}x{N}")
print(f"목표 정답 (상위 3개): {x_true[:3]} ...")
print("-" * 60)
print(f"{'Method':<20} | {'Iterations':<10} | {'Error (L2 Norm)':<15}")
print("-" * 60)

# 1. Jacobi
sol_jac, iter_jac = jacobi(A, b, x0)
err_jac = np.linalg.norm(sol_jac - x_true)
print(f"{'Jacobi':<20} | {iter_jac:<10} | {err_jac:.2e}")

# 2. Gauss-Seidel
sol_gs, iter_gs = gauss_seidel(A, b, x0)
err_gs = np.linalg.norm(sol_gs - x_true)
print(f"{'Gauss-Seidel':<20} | {iter_gs:<10} | {err_gs:.2e}")

# 3. SOR (omega = 1.1)
omega = 1.1
sol_sor, iter_sor = sor(A, b, x0, omega)
err_sor = np.linalg.norm(sol_sor - x_true)
print(f"{'SOR (w=1.1)':<20} | {iter_sor:<10} | {err_sor:.2e}")

# 4. Conjugate Gradient
sol_cg, iter_cg = conjugate_gradient(A, b, x0)
err_cg = np.linalg.norm(sol_cg - x_true)
print(f"{'Conjugate Gradient':<20} | {iter_cg:<10} | {err_cg:.2e}")
print("-" * 60)

# 결과 검증
print("\n[CG Method 결과값 확인 (상위 5개)]")
print(np.round(sol_cg[:5], 5))

In [ ]:
import numpy as np
import time

def setup_large_problem(n):
    """
    N x N 크기의 대칭 양의 정부호(SPD) 행렬을 생성한다.
    수렴성을 위해 대각 성분을 강화한다.
    """
    print(f"[{n}x{n}] 행렬 생성 중... (메모리 할당)", end="\r")
    np.random.seed(42)
    # 1. 임의의 행렬 생성
    A = np.random.rand(n, n)
    # 2. 대칭 행렬로 변환 (A + A.T)
    A = 0.5 * (A + A.T)
    # 3. 대각 지배력 강화 (Jacobi, GS 수렴 보장 및 CG 안정성)
    # 모든 행의 절대값 합보다 대각 성분을 조금 더 크게 설정
    row_sums = np.sum(np.abs(A), axis=1)
    np.fill_diagonal(A, row_sums + 1.0)
    # 4. 정답(x_true) 설정 및 b 계산
    x_true = np.random.rand(n)
    b = A @ x_true
    print(f"[{n}x{n}] 시스템 설정 완료.             ")
    return A, b, x_true

# 1. Jacobi (Vectorized) - 루프 제거로 고속화
def jacobi_vectorized(A, b, x0, tol=1e-8, max_iter=2000):
    start_time = time.time()
    n = len(b)
    x = x0.copy()
    D = np.diag(A)             # 대각 성분
    R = A - np.diag(D)         # 나머지 (L + U)
    for k in range(max_iter):
        # x_new = (b - (L+U)x) / D
        x_new = (b - R @ x) / D
        if np.linalg.norm(x_new - x) < tol:
            return x_new, k + 1, time.time() - start_time
        x = x_new
    return x, max_iter, time.time() - start_time

# 2. Gauss-Seidel (Row-Optimized)
# GS는 앞의 계산 결과에 의존하므로 완전 벡터화는 불가능하지만, 내적(dot)으로 가속
def gauss_seidel_fast(A, b, x0, tol=1e-8, max_iter=2000):
    start_time = time.time()
    n = len(b)
    x = x0.copy()
    for k in range(max_iter):
        x_old_norm = np.linalg.norm(x)
        for i in range(n):
            # A[i, :i] @ x[:i] : 이미 업데이트된 값들 (New)
            # A[i, i+1:] @ x[i+1:] : 아직 업데이트 안 된 값들 (Old)
            sigma = A[i, :i] @ x[:i] + A[i, i+1:] @ x[i+1:]
            x[i] = (b[i] - sigma) / A[i, i]
        if np.linalg.norm(x - x_old_norm) < tol: # 약식 수렴 체크 (속도 위해)
             # 정확한 체크를 위해 한 번 더 잔차 확인
             if np.linalg.norm(b - A @ x) < tol:
                return x, k + 1, time.time() - start_time
    return x, max_iter, time.time() - start_time

# 3. SOR (Row-Optimized)
def sor_fast(A, b, x0, omega, tol=1e-8, max_iter=2000):
    start_time = time.time()
    n = len(b)
    x = x0.copy()
    for k in range(max_iter):
        x_old_copy = x.copy()
        for i in range(n):
            sigma = A[i, :i] @ x[:i] + A[i, i+1:] @ x[i+1:]
            sigma_x = (b[i] - sigma) / A[i, i]
            x[i] = (1 - omega) * x[i] + omega * sigma_x
        if np.linalg.norm(x - x_old_copy) < tol:
            return x, k + 1, time.time() - start_time  
    return x, max_iter, time.time() - start_time

# 4. Conjugate Gradient (Vectorized) - 가장 빠름
def conjugate_gradient(A, b, x0, tol=1e-8, max_iter=2000):
    start_time = time.time()
    x = x0.copy()
    r = b - A @ x
    p = r.copy()
    rs_old = r @ r
    for k in range(max_iter):
        Ap = A @ p
        alpha = rs_old / (p @ Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        rs_new = r @ r
        if np.sqrt(rs_new) < tol:
            return x, k + 1, time.time() - start_time
        p = r + (rs_new / rs_old) * p
        rs_old = rs_new
    return x, max_iter, time.time() - start_time

# --- 메인 실행부 ---

# 1. 문제 설정 (N=1000)
N = 1000
A, b, x_true = setup_large_problem(N)
x0 = np.zeros(N)
print(f"{'Method':<25} | {'Iter':<6} | {'Time (sec)':<10} | {'Error (L2)':<12}")
print("-" * 65)

# 2. Jacobi
sol_jac, iter_jac, t_jac = jacobi_vectorized(A, b, x0)
err_jac = np.linalg.norm(sol_jac - x_true)
print(f"{'Jacobi (Vectorized)':<25} | {iter_jac:<6} | {t_jac:.4f}     | {err_jac:.2e}")
# 3. Conjugate Gradient (순서 변경: 성능 비교를 위해 먼저 출력)
sol_cg, iter_cg, t_cg = conjugate_gradient(A, b, x0)
err_cg = np.linalg.norm(sol_cg - x_true)
print(f"{'Conjugate Gradient':<25} | {iter_cg:<6} | {t_cg:.4f}     | {err_cg:.2e}")
# 4. SOR (Omega=1.1)
# 주의: Python에서 순차 루프(GS/SOR)는 N=1000일 때 매우 느릴 수 있다.
# 비교를 위해 실행하지만 시간이 좀 걸린다.
omega = 1.2
sol_sor, iter_sor, t_sor = sor_fast(A, b, x0, omega)
err_sor = np.linalg.norm(sol_sor - x_true)
print(f"{'SOR (w=1.2, Optimized)':<25} | {iter_sor:<6} | {t_sor:.4f}     | {err_sor:.2e}")
# 5. Gauss-Seidel
sol_gs, iter_gs, t_gs = gauss_seidel_fast(A, b, x0)
err_gs = np.linalg.norm(sol_gs - x_true)
print(f"{'Gauss-Seidel':<25} | {iter_gs:<6} | {t_gs:.4f}     | {err_gs:.2e}")

print("-" * 65)
print("Tip: CG와 Jacobi는 벡터 연산(행렬 곱)을 사용하여 빠르지만,\nGS와 SOR은 알고리즘 특성상 순차 처리가 필요하여 Python에서는 상대적으로 느리다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

def setup_ill_conditioned_problem(n):
    """
    일부러 조건수가 나쁜(Ill-conditioned) 행렬을 생성한다.
    대각 지배력을 약화시켜 타원형 등고선을 길게 만든다.
    """
    np.random.seed(1) # 시드 고정
    A = np.random.rand(n, n)
    A = A @ A.T # 대칭 양의 정부호(SPD) 보장
    
    # 고유값의 스펙트럼을 넓혀 조건수를 악화시킴
    # (일반적인 랜덤 행렬보다 수렴이 훨씬 오래 걸리게 설정)
    D = np.diag(np.linspace(1, 1000, n)) # 고유값이 1에서 1000까지 분포
    # A와 유사한 구조를 갖지만 고유값을 조작
    Q, _ = np.linalg.qr(np.random.rand(n, n))
    A = Q @ D @ Q.T
    
    x_true = np.random.rand(n)
    b = A @ x_true
    return A, b, x_true

# 1. Standard CG
def conjugate_gradient(A, b, x0, x_true, max_iter=1000, tol=1e-8):
    x = x0.copy()
    r = b - A @ x
    p = r.copy()
    rs_old = r @ r
    
    history = []
    
    for k in range(max_iter):
        history.append(np.linalg.norm(x - x_true))
        
        if np.sqrt(rs_old) < tol:
            break
            
        Ap = A @ p
        alpha = rs_old / (p @ Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        
        rs_new = r @ r
        p = r + (rs_new / rs_old) * p
        rs_old = rs_new
        
    return history

# 2. Preconditioned CG (PCG) - Jacobi Preconditioner
def preconditioned_cg(A, b, x0, x_true, max_iter=1000, tol=1e-8):
    x = x0.copy()
    r = b - A @ x
    
    # Preconditioner M 설정 (Jacobi: A의 대각 성분)
    M_diag = np.diag(A) 
    
    # z0 = M^(-1)r0  =>  M * z0 = r0 (요소별 나눗셈)
    z = r / M_diag 
    p = z.copy()
    
    # r^T z (PCG에서는 이것을 추적함)
    rz_old = r @ z 
    
    history = []
    
    for k in range(max_iter):
        history.append(np.linalg.norm(x - x_true))
        
        if np.linalg.norm(r) < tol:
            break
            
        Ap = A @ p
        alpha = rz_old / (p @ Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        
        # --- PCG의 핵심 단계 ---
        # M z_{k+1} = r_{k+1} 풀기 (Jacobi라 나눗셈으로 해결)
        z = r / M_diag 
        
        rz_new = r @ z
        beta = rz_new / rz_old # 베타 공식이 rz 비율로 변경됨
        p = z + beta * p
        
        rz_old = rz_new
        
    return history

# --- 실행 및 비교 ---
N = 100
A, b, x_true = setup_ill_conditioned_problem(N)
x0 = np.zeros(N)

# 조건수 확인
cond_number = np.linalg.cond(A)
print(f"행렬 조건수(Condition Number): {cond_number:.2f}")
print("조건수가 클수록 문제는 어렵다 (Ill-conditioned).")

hist_cg = conjugate_gradient(A, b, x0, x_true, max_iter=200)
hist_pcg = preconditioned_cg(A, b, x0, x_true, max_iter=200)

# 시각화
plt.figure(figsize=(10, 6))
plt.plot(hist_cg, label='Standard CG', linestyle='--', linewidth=2)
plt.plot(hist_pcg, label='Preconditioned CG (Jacobi)', linestyle='-', linewidth=2)
plt.yscale('log')
plt.xlabel('Iterations')
plt.ylabel('Error (L2 Norm)')
plt.title(f'Standard CG vs Preconditioned CG (N={N})')
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

In [ ]:
import numpy as np

def thomas_algorithm(a, b, c, d):
    """
    Thomas 알고리즘(TDMA)을 사용하여 삼중 대각 시스템을 푸는 함수.
    
    a: 하 대각선 벡터 (N-1 길이) - a[0]는 사용되지 않음
    b: 주 대각선 벡터 (N 길이)
    c: 상 대각선 벡터 (N-1 길이) - c[N-1]는 0
    d: 우변 벡터 (N 길이)
    
    return: 해 벡터 x (N 길이)
    """
    N = len(d)
    
    # 1. 전진 소거 단계 (Forward Elimination)
    # P와 Q 벡터 초기화
    # p_i = -c_i / gamma_i
    # q_i = (d_i - a_i * q_{i-1}) / gamma_i
    p = np.zeros(N)
    q = np.zeros(N)
    
    # i = 0 (첫 번째 방정식)
    # 주의: Python 인덱스 0부터 시작
    p[0] = -c[0] / b[0]
    q[0] = d[0] / b[0]
    
    # i = 1부터 N-1까지 반복
    for i in range(1, N):
        # gamma_i 계산
        gamma_i = b[i] + a[i] * p[i-1]
        
        # p_i와 q_i 계산
        # c 벡터의 마지막 요소는 0이므로, c[N-1] 사용 시 문제가 되지 않음.
        # N-1일 때는 c[N-1] = 0이 되어 p[N-1] = 0이 됨.
        p[i] = -c[i] / gamma_i
        q[i] = (d[i] - a[i] * q[i-1]) / gamma_i
    
    # 2. 역대입 단계 (Back Substitution)
    x = np.zeros(N)
    
    # 마지막 해 x_N (Python 인덱스 N-1)
    x[N-1] = q[N-1]
    
    # i = N-2부터 0까지 역순으로 계산
    for i in range(N - 2, -1, -1):
        # x_i = p_i * x_{i+1} + q_i
        x[i] = p[i] * x[i+1] + q[i]
        
    return x

# --- 예제 실행 ---
# N=5
# a[0]은 사용되지 않으므로 0으로 설정
a_vec = np.array([0., 1., 1., 1., 1.]) 
b_vec = np.array([2., 2., 2., 2., 2.])
# c[4]는 0이어야 함
c_vec = np.array([1., 1., 1., 1., 0.])
d_vec = np.array([4., 6., 6., 6., 5.])

solution_x = thomas_algorithm(a_vec, b_vec, c_vec, d_vec)

print("--- Thomas 알고리즘 실행 결과 ---")
print(f"하 대각선 (a):\n{a_vec}")
print(f"주 대각선 (b):\n{b_vec}")
print(f"상 대각선 (c):\n{c_vec}")
print(f"우변 벡터 (d):\n{d_vec}")
print("\n**해 벡터 (x):**")
print(solution_x.round(4))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def thomas_solver(a, b, c, d):
    """
    토마스 알고리즘을 사용하여 삼중 대각 선형 시스템을 푼다.
    """
    n = len(d)
    cp = np.zeros(n)
    dp = np.zeros(n)
    x = np.zeros(n)
    # Forward Elimination (전방 소거)
    # 첫 번째 방정식 처리
    cp[0] = c[0] / b[0]
    dp[0] = d[0] / b[0]
    # 나머지 방정식 처리
    for i in range(1, n):
        denom = b[i] - a[i] * cp[i-1]
        if i < n-1: # 마지막 방정식에서는 c[i]를 계산할 필요 없음
            cp[i] = c[i] / denom
        dp[i] = (d[i] - a[i] * dp[i-1]) / denom
    # Backward Substitution (후방 대입)
    x[n-1] = dp[n-1]
    for i in range(n-2, -1, -1):
        x[i] = dp[i] - cp[i] * x[i+1]
    return x

# --- 경계값 문제 설정 및 토마스 알고리즘 적용 ---
# 문제 설정
L = 1.0  # 핀의 길이 (m)
n_internal_nodes = 3 # 내부 노드 수
total_nodes = n_internal_nodes + 2 # 총 노드 수 (경계 포함)
# 노드 간격
dx = L / (total_nodes - 1)
print(f"노드 간격 (dx): {dx:.2f}m")
# 미분 방정식 계수
# d^2T/dx^2 - T = 0  -> T_i-1 - (2 + dx^2)T_i + T_i+1 = 0
main_diag_val = -(2 + dx**2) # -2.0625
# 경계 조건
T_left = 100 # 왼쪽 끝 온도 (°C)
T_right = 200 # 오른쪽 끝 온도 (°C)
# 토마스 알고리즘 입력 벡터 구성 (내부 노드만 고려)
# 내부 노드는 T1, T2, T3 (총 3개)
# a, b, c 벡터의 길이는 n_internal_nodes 여야 한다.
a = [1] * n_internal_nodes # 하단 대각 (첫 번째 a[0]은 실제로는 사용되지 않음, dummy)
b = [main_diag_val] * n_internal_nodes # 중앙 대각
c = [1] * n_internal_nodes # 상단 대각 (마지막 c[n-1]은 실제로는 사용되지 않음, dummy)
# d 벡터 (우변) 구성
d = [0] * n_internal_nodes
d[0] -= T_left # 첫 번째 내부 노드에 왼쪽 경계 조건 영향
d[n_internal_nodes-1] -= T_right # 마지막 내부 노드에 오른쪽 경계 조건 영향
# 토마스 알고리즘으로 내부 온도 계산
internal_temperatures = thomas_solver(a, b, c, d)
# 전체 온도 분포 (경계 조건 포함)
# x축 좌표 생성
x_coords = np.linspace(0, L, total_nodes)
full_temperatures = np.concatenate(([T_left], internal_temperatures, [T_right]))
print("\n--- 계산된 온도 분포 ---")
for i, temp in enumerate(full_temperatures):
    print(f"x = {x_coords[i]:.2f}m: T = {temp:.2f}°C")
# --- Matplotlib을 이용한 시각화 ---
plt.figure(figsize=(10, 6)) # 그래프 크기 설정
plt.plot(x_coords, full_temperatures, marker='o', linestyle='-', color='b', 
         label='Numerical solution(Thomas algorithm)')
plt.title('Temperature distribution along the fin')
plt.xlabel('Position (m)')
plt.ylabel('Temperature (°C)')
plt.grid(True) # 격자선 표시
plt.legend() # 범례 표시
plt.show()

# --- 노드 수 증가 시도 ---
print("\n--- 노드 수 증가 (n_internal_nodes = 100) ---")
n_internal_nodes_large = 100
total_nodes_large = n_internal_nodes_large + 2
dx_large = L / (total_nodes_large - 1)
main_diag_val_large = -(2 + dx_large**2)
a_large = [1] * n_internal_nodes_large
b_large = [main_diag_val_large] * n_internal_nodes_large
c_large = [1] * n_internal_nodes_large
d_large = [0] * n_internal_nodes_large
d_large[0] -= T_left
d_large[n_internal_nodes_large-1] -= T_right
internal_temperatures_large = thomas_solver(a_large, b_large, c_large, d_large)
x_coords_large = np.linspace(0, L, total_nodes_large)
full_temperatures_large = np.concatenate(([T_left], internal_temperatures_large, [T_right]))
plt.figure(figsize=(10, 6))
plt.plot(x_coords_large, full_temperatures_large, linestyle='-', color='r', 
         label=f'Numerical solution(N={n_internal_nodes_large})')
plt.title('Temperature distribution along the fin (More nodes)')
plt.xlabel('Position (m)')
plt.ylabel('Temperature (°C)')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import numpy as np

def jacobi_method(A, b, max_iter=100, tol=1e-8):
    """
    Jacobi 반복법으로 Ax = b를 푸는 함수.
    Args:
        A (np.array): 계수 행렬
        b (np.array): 상수 벡터
        max_iter (int): 최대 반복 횟수
        tol (float): 수렴 허용 오차 (잔차 노름)
    Returns:
        np.array: 근사 해 벡터 x
        int: 실제 반복 횟수
        list: 각 반복에서의 잔차 노름 리스트
    """
    n = len(b)
    x = np.zeros(n)  # 초기 추측값 (모두 0으로 설정)
    residuals = []

    # 행렬 A를 D (대각 행렬)와 R (나머지 행렬)로 분리
    D = np.diag(np.diag(A))
    R = A - D

    # D의 역행렬을 계산한다. (대각 성분이 0인 경우를 주의해야 한다.)
    try:
        D_inv = np.linalg.inv(D)
    except np.linalg.LinAlgError:
        print("Error: Diagonal element is zero (D is singular). Jacobi method may not be applicable.")
        return x, 0, residuals

    for k in range(max_iter):
        x_new = np.dot(D_inv, (b - np.dot(R, x)))

        # 잔차 계산 및 수렴 확인
        residual_vec = np.dot(A, x_new) - b
        residual_norm = np.linalg.norm(residual_vec)
        residuals.append(residual_norm)

        if residual_norm < tol:
            print(f"Converged after {k+1} iterations.")
            return x_new, k + 1, residuals

        # 해 업데이트
        x = x_new

    print(f"Maximum iterations ({max_iter}) reached. Residual norm: {residual_norm:.2e}")
    return x, max_iter, residuals

# --- 예제 시스템 정의 ---
# Ax = b
# A는 대각 우세(Diagonally Dominant) 행렬일 때 Jacobi 방법이 수렴할 가능성이 높다.
# A = [[10, -1, 2, 0],
#      [-1, 11, -1, 3],
#      [2, -1, 10, -1],
#      [0, 3, -1, 8]]
# b = [6, 25, -11, 15]
A = np.array([
    [10.0, -1.0, 2.0, 0.0],
    [-1.0, 11.0, -1.0, 3.0],
    [2.0, -1.0, 10.0, -1.0],
    [0.0, 3.0, -1.0, 8.0]
])

b = np.array([6.0, 25.0, -11.0, 15.0])

# --- Jacobi 반복법 실행 ---
x_solution, iterations, residual_history = jacobi_method(A, b)

# 결과 출력
print("\n--- 결과 ---")
print("근사 해 x:")
print(x_solution)
print(f"총 반복 횟수: {iterations}")

# numpy의 직접 해법과 비교 (정확한 해)
x_exact = np.linalg.solve(A, b)
print("\n정확한 해 (np.linalg.solve):")
print(x_exact)

# 오차 확인
error = np.linalg.norm(x_solution - x_exact)
print(f"\n정확한 해와의 L2 노름 오차: {error:.2e}")

import numpy as np
from scipy.sparse.linalg import LinearOperator, cg

# 1. 행렬-벡터 곱을 정의하는 함수 (오퍼레이터 A의 역할)
def matvec_laplacian(v):
    """
    3차원 이산 라플라시안 오퍼레이터의 행렬-벡터 곱 A*v를 계산하는 함수.
    (예시를 위해 간단한 3x3 대각선 행렬처럼 동작하도록 정의)
    """
    # 실제 라플라시안 행렬은 매우 크고 복잡하지만, 여기서는 개념을 위해 간단히 정의
    # A = [[4, -1, 0],
    #      [-1, 4, -1],
    #      [0, -1, 4]] 처럼 동작하도록 정의
    
    n = len(v)
    Av = np.zeros_like(v)
    
    # 대각 성분 (4 * v[i])
    Av = 4 * v
    
    # 비대각 성분 (-1 * v[i-1] 및 -1 * v[i+1])
    if n > 1:
        Av[0] -= v[1]
        Av[n-1] -= v[n-2]
    
    if n > 2:
        Av[1:n-1] -= v[0:n-2] # v[i-1]
        Av[1:n-1] -= v[2:n]   # v[i+1]
        
    return Av

# 2. LinearOperator 생성
N = 100 # 벡터의 크기 (행렬 A의 크기는 N x N)
A_op = LinearOperator((N, N), matvec=matvec_laplacian, dtype=float)

# 3. 상수 벡터 b 생성
# 임의의 벡터 b를 생성
b = np.random.rand(N)

# 4. 공액 기울기법(Conjugate Gradient, CG)으로 해 x 계산
# cg 함수는 A 대신 LinearOperator를 인수로 받을 수 있다.
# tol: 수렴 허용 오차
x_cg, info = cg(A_op, b, rtol=1e-8)

# 5. 결과 확인
if info == 0:
    print(f" CG Method: Converged successfully in {info} iterations.")
    # 잔차 노름 확인: ||Ax - b||
    residual_norm = np.linalg.norm(A_op.dot(x_cg) - b)
    print(f"최종 잔차 노름: {residual_norm:.2e}")
else:
    print(f" CG Method: Did not converge. info = {info}")

# LinearOperator의 기본 속성 확인
print(f"\nLinearOperator의 차원: {A_op.shape}")

import numpy as np
from scipy.sparse.linalg import LinearOperator, cg
import matplotlib.pyplot as plt 

# --- 시스템 및 그리드 간격 정의 ---
N = 100  # 전체 격자점 개수
# 1. 그리드 간격 h 계산
L = 1.0  # 공간 길이 (0부터 1까지)
h = L / (N - 1) 

x_left_bc = 0.0
x_right_bc = 0.0
N_internal = N - 2 

# 1. 행렬-벡터 곱 함수 정의 (Ax 연산) - A_old: h^2이 없는 버전
def matvec_laplacian_with_bc(v):
    """
    1D Laplacian operator (A_old: -u_{i-1} + 2u_i - u_{i+1}) 연산 수행.
    이 함수 자체에는 h^2을 곱하지 않고, 우변 b에 h^2을 곱하여 처리한다.
    """
    n = len(v)
    Av = np.zeros_like(v)
    # 중앙 항: 2 * v_i
    Av = 2.0 * v
    # 왼쪽 및 오른쪽 항: -v_{i-1} 및 -v_{i+1}
    if n > 0:
        Av[:-1] -= v[1:] 
        Av[1:] -= v[:-1]
    # 경계값 기여 처리 (BCs Contribution)
    # A_{int} * x_{int} = h^2 * f - A_{mixed} * x_{BC}
    # 여기서 Av는 A_{int} * x_{int} 이므로, A_{mixed} * x_{BC}를 더해야 함.
    if n > 0:
        Av[0] -= x_left_bc    # A_{mixed}는 -1로 정의됨
        Av[n - 1] -= x_right_bc
    return Av

# 2. LinearOperator 생성 
A_op = LinearOperator((N_internal, N_internal), matvec=matvec_laplacian_with_bc, dtype=float)
# 3. 우변 벡터 b (내부 노드만 해당) 생성 및 h^2 적용
# f(t) = 1.0 (참고: Ax = b 형식으로 쓰기 위해 우변 f(t) = 1.0으로 수정. 
#            해석해 u_exact(t) = 0.5t^2 - 0.5t는 -u''=1의 해이다.)
f = 1.0 * np.ones(N) 
f_internal = f[1:N-1]
# 2. **핵심 수정**: 우변 벡터 b에 h^2을 곱한다.
b_internal = -h**2 * f_internal  # b = h^2 * f
# 4. 공액 기울기법(CG)으로 해 x_internal 계산
initial_guess = 0.5 * np.ones(N_internal)
print(f"Solving with h={h:.4f} and h^2 in b...")
x_internal, info = cg(A_op, b_internal, rtol=1e-8, x0=initial_guess)
# 5. 최종 해 벡터 구성 및 시각화 (이전과 동일)
if info == 0:
    print(f" CG method: Converged successfully. iterations: {info}")
    # ... (x_solution 구성)
    x_solution = np.zeros(N)
    x_solution[0] = x_left_bc
    x_solution[-1] = x_right_bc
    x_solution[1:-1] = x_internal
    
    # ... (시각화)
    x_coords = np.linspace(0, 1, N)
    # Exact solution for -u''(t) = 1, u(0)=u(1)=0
    u_exact = 0.5 * x_coords**2 - 0.5 * x_coords 
    
    plt.figure(figsize=(10, 6))
    plt.plot(x_coords, u_exact, 'b-', linewidth=3, label='Exact solution', alpha=0.7)
    plt.plot(x_coords, x_solution, 'ro', markersize=4, label='Iterative solution(CG)', alpha=0.8)
    plt.title(f"Solution with Grid Spacing $h={h:.4f}$", fontsize=16)
    plt.xlabel("Spatial coordinate (t)", fontsize=14)
    plt.ylabel("Solution value ($u(t)$)", fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()

else:
    print(f" CG Method: Did not converge. info = {info}")

import numpy as np
from scipy.sparse.linalg import LinearOperator, cg
import matplotlib.pyplot as plt

# --------------------------------------------------------------------------
# --- 경계 조건 및 시스템 정의 ---
# --------------------------------------------------------------------------
L = 1.0              # 도메인 길이 (추가)
N = 100              # 전체 격자점 개수
x_left_bc = 0.0      # 좌측 Dirichlet 경계 조건 u(0)
x_right_bc = 0.0     # 우측 Dirichlet 경계 조건 u(L)
N_internal = N - 2   # 내부 격자점 개수
h = L / (N - 1)      # 격자 간격

# Fornberg 4차 계수 (중앙 차분, 5점 스텐실)
# u''(i) * h^2 ~ (1/12) * [-1, 16, -30, 16, -1]
# 이 계수들은 행렬 A의 내부 계수가 된다.
C_4 = np.array([-1.0, 16.0, -30.0, 16.0, -1.0]) / 12.0

# 1. 고정밀도 행렬-벡터 곱 함수 정의 (4차 + 2차 혼합 스텐실)
def matvec_high_order_laplacian(v):
    """
    4차 중앙 차분 스텐실을 사용하는 행렬-벡터 곱 함수.
    경계 근처 노드(v[0], v[1], v[n-2], v[n-1])는 2차 스텐실을 사용한다.
    v: 내부 노드 해 벡터 (u_1, ..., u_{N-2}), 크기 n = N_internal
    """
    n = len(v)
    Av = np.zeros_like(v)

    # Note: 경계 조건 (u_0=0, u_{N-1}=0)은 우변 벡터 b에 포함되어야 하며,
    # 여기서는 순수한 행렬-벡터 곱셈 A*v만 수행한다.

    # i=1 (v[0]): 2차 스텐실 (u_0 - 2u_1 + u_2). u_0=0
    if n >= 1:
        Av[0] = -2.0 * v[0] + 1.0 * v[1]

    # i=2 (v[1]): 2차 스텐실 (u_1 - 2u_2 + u_3)
    if n >= 2:
        Av[1] = 1.0 * v[0] - 2.0 * v[1] + 1.0 * v[2]

    # Core 4th-order stencil (v[2] up to v[n-3])
    # 이 영역에서 4차 정확도가 달성된다.
    for i in range(2, n - 2):
        # Stencil for Av[i] is centered around v[i] using [v[i-2]...v[i+2]]
        Av[i] = (C_4[0] * v[i-2] + C_4[1] * v[i-1] + C_4[2] * v[i] +
                 C_4[3] * v[i+1] + C_4[4] * v[i+2])

    # i=n-2 (v[n-3]): 2차 스텐실 (u_{N-4} - 2u_{N-3} + u_{N-2})
    if n >= 3:
        Av[n-2] = 1.0 * v[n-3] - 2.0 * v[n-2] + 1.0 * v[n-1]

    # i=n-1 (v[n-2]): 2차 스텐실 (u_{N-3} - 2u_{N-2} + u_{N-1}). u_{N-1}=0
    if n >= 1:
        Av[n-1] = 1.0 * v[n-2] - 2.0 * v[n-1]
        
    return Av

# 2. LinearOperator 생성 (고차 오퍼레이터 사용)
A_op = LinearOperator((N_internal, N_internal), matvec=matvec_high_order_laplacian, dtype=float)
# 3. 우변 벡터 b (내부 노드만 해당) 생성
# 미분 방정식: u''(t) = f(t). 여기서 f(t) = 1.0으로 가정한다.
# 이산화 시스템: A * x_internal = h^2 * f_internal
f_val = 1.0 # f(t) = 1.0
f = h**2 * f_val * np.ones(N) 
b_internal = f[1:N-1] # 내부 노드의 우변 벡터
# 4. 공액 기울기법(CG)으로 해 x_internal 계산
print(f"Solving linear system of size {N_internal}x{N_internal} using High-Order FD...")
initial_guess = np.zeros(len(b_internal))
# 초기 추측 생성
for i in range(len(b_internal)):
    initial_guess[i] = np.random.random() - 0.5
# 높은 정밀도를 위해 rtol을 강화하고 maxiter를 증가시킨다.
x_internal, info = cg(A_op, b_internal, rtol=1e-12, maxiter=1000, x0=initial_guess)
# 5. 최종 해 벡터 구성
if info == 0:
    print(f" CG method: converged successfully.")
    x_solution = np.zeros(N)
    x_solution[0] = x_left_bc
    x_solution[-1] = x_right_bc
    x_solution[1:-1] = x_internal
    residual_norm = np.linalg.norm(A_op.dot(x_internal) - b_internal)
    print("\n--- 결과 요약 (고차 스텐실 적용) ---")
    print(f"사용된 격자점 수 N: {N}")
    print(f"총 반복 횟수: {info}")
    print(f"최종 잔차 노름: {residual_norm:.2e}")
    # ----------------------------------------------------
    ##  해 시각화 및 해석해 비교
    # ----------------------------------------------------
    # 1. 공간 좌표 생성 (0부터 L까지 N개의 점)
    x_coords = np.linspace(0, L, N)
    # 2. 해석해 (Exact Solution) 계산: u''(t) = 1.0, u(0)=0, u(1)=0
    # u(t) = 0.5 * t^2 - 0.5 * t
    u_exact = 0.5 * x_coords**2 - 0.5 * x_coords
    # 3. 그래프 그리기
    plt.figure(figsize=(10, 6))
    # 해석해 (파란색 실선)
    plt.plot(x_coords, u_exact, 'b-', linewidth=3, label='Exact solution $u(t) = 0.5t^2 - 0.5t$', alpha=0.7)
    # 반복법으로 구한 근사 해 (빨간색 점)
    plt.plot(x_coords, x_solution, 'ro', markersize=4, label='4th-order iterative solution(CG)', alpha=0.8)
    # 경계 조건 강조
    plt.plot(x_coords[[0, -1]], x_solution[[0, -1]], 'ko', markersize=6, label='Boundary points')
    plt.title(f"Comparison of high-order FD solution vs. Exact solution ($N={N}$)", fontsize=16)
    plt.xlabel("Spatial coordinate (t)", fontsize=14)
    plt.ylabel("Solution value ($u(t)$)", fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()
else:
    print(f" CG method: Did not converge. info = {info}")

In [ ]:
import numpy as np

def jacobi(A, b, x0, tol=1e-10, max_iter=100):
    n = len(A)
    x = x0.copy()
    for k in range(max_iter):
        x_new = np.zeros_like(x)
        for i in range(n):
            # x_i^{(k+1)}를 계산
            x_new[i] = (b[i] - np.dot(A[i, :], x) + A[i, i] * x[i]) / A[i, i]
            # 대각선 원소는 제외하고 나머지 항들만 사용하여 업데이트
            x_new[i] += (b[i] - np.sum(A[i, :] * x) + A[i, i] * x[i]) / A[i, i]
        
        # 수렴 조건 체크 (변화량이 작으면 종료)
        if np.linalg.norm(x_new - x, ord=np.inf) < tol:
            print(f"Converged in {k+1} iterations")
            return x_new
        x = x_new
    return x

# 행렬 A와 벡터 b 정의
A = np.array([[3, -1, 1], [2, 4, -1], [1, -1, 3]], dtype=float)
b = np.array([1, -2, 3], dtype=float)

# 초기 추정값 x0 설정
x0 = np.zeros_like(b)

# Jacobi 방법 호출
x_solution = jacobi(A, b, x0)
print("해:", x_solution)

In [ ]:
import numpy as np

def gauss_seidel(A, b, x0, tol=1e-10, max_iter=100):
    n = len(A)
    x = x0.copy()
    for k in range(max_iter):
        x_new = np.copy(x)
        for i in range(n):
            # Gauss-Seidel 업데이트: 이전에 계산된 값을 사용
            sum1 = np.dot(A[i, :i], x_new[:i])
            sum2 = np.dot(A[i, i+1:], x[i+1:])
            x_new[i] = (b[i] - sum1 - sum2) / A[i, i]
        
        # 수렴 조건 체크 (변화량이 작으면 종료)
        if np.linalg.norm(x_new - x, ord=np.inf) < tol:
            print(f"Converged in {k+1} iterations")
            return x_new
        x = x_new
    return x

# 행렬 A와 벡터 b 정의
A = np.array([[3, -1, 1], [2, 4, -1], [1, -1, 3]], dtype=float)
b = np.array([1, -2, 3], dtype=float)

# 초기 추정값 x0 설정
x0 = np.zeros_like(b)

# Gauss-Seidel 방법 호출
x_solution = gauss_seidel(A, b, x0)
print("해:", x_solution)

In [ ]:
import numpy as np

def rayleigh_quotient_iteration(A, x0, max_iter=10, tolerance=1e-10):
    """
    Rayleigh Quotient Iteration (RQI)을 사용하여 행렬 A의 고윳값과 고유벡터를 계산한다.

    Args:
        A (np.array): 고윳값을 계산할 대칭 행렬 (n x n).
        x0 (np.array): 초기 고유벡터 근사치 (길이 n 벡터).
        max_iter (int): 최대 반복 횟수.
        tolerance (float): 수렴 판단을 위한 허용 오차.

    Returns:
        tuple: (최종 고윳값 근사치, 최종 고유벡터 근사치, 반복 횟수)
    """
    
    # 1. 초기 벡터 정규화
    x_k = x0 / np.linalg.norm(x0)
    
    # 2. 초기 레일리 몫 (고윳값 근사치) 계산
    # R(A, x) = (x^T A x) / (x^T x). 정규화했으므로 분모는 1이다.
    lambda_k = x_k.T @ A @ x_k
    
    print(f"초기 고윳값 근사치 (λ_0): {lambda_k:.6f}")

    for k in range(max_iter):
        
        # 이전 고윳값 및 고유벡터 저장
        lambda_prev = lambda_k
        x_prev = x_k
        
        # 3. Shift된 시스템 풀기: (A - λ_k * I) * z = x_k
        # z = (A - λ_k * I)^-1 * x_k 를 계산한다.
        # np.eye(A.shape[0])은 A와 같은 크기의 단위 행렬 I를 만든다.
        B = A - lambda_k * np.eye(A.shape[0])
        
        # 선형 시스템 Bz = x_k 풀기
        try:
            # np.linalg.solve는 역행렬을 직접 계산하는 것보다 빠르고 수치적으로 안정적이다.
            z_k_plus_1 = np.linalg.solve(B, x_k)
        except np.linalg.LinAlgError:
            print(f"경고: {k+1}번째 반복에서 행렬이 특이 행렬이 되어 해를 구할 수 없다. (고윳값에 매우 근접)")
            break

        # 4. 새로운 고유벡터 근사치 정규화
        x_k_plus_1 = z_k_plus_1 / np.linalg.norm(z_k_plus_1)
        
        # 5. 새로운 레일리 몫 계산 (고윳값 근사치)
        lambda_k_plus_1 = x_k_plus_1.T @ A @ x_k_plus_1
        
        # 6. 수렴 확인
        eigenvalue_diff = np.abs(lambda_k_plus_1 - lambda_prev)
        
        print(f"반복 {k+1}: λ = {lambda_k_plus_1:.6f}, |Δλ| = {eigenvalue_diff:.2e}")
        
        if eigenvalue_diff < tolerance:
            print(f"\n {k+1}번째 반복에서 수렴 (오차 허용치 미만).")
            return lambda_k_plus_1, x_k_plus_1, k + 1
        
        # 업데이트
        lambda_k = lambda_k_plus_1
        x_k = x_k_plus_1
        
    print(f"\n 최대 반복 횟수 ({max_iter}) 도달.")
    return lambda_k, x_k, max_iter

# 대칭 행렬 A 정의
A = np.array([
    [2, 1],
    [1, 2]
])

# 초기 벡터 x0 (임의의 근사치)
x0 = np.array([1.0, 0.0]) 

# RQI 실행
final_lambda, final_x, iterations = rayleigh_quotient_iteration(A, x0, max_iter=5)

# 결과 출력
print("--- 결과 ---")
print(f"계산된 고윳값 (λ): {final_lambda:.6f}")
print(f"계산된 고유벡터 (x): {final_x}")

# NumPy의 고유값/고유벡터 함수로 검증
w, v = np.linalg.eig(A)
print("\n--- NumPy 검증 ---")
print(f"NumPy 고윳값: {w}")
print(f"NumPy 고유벡터:\n{v}")    

In [ ]:
import numpy as np
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

def solve_schrodinger_lanczos():
    # ---------------------------------------------------------
    # 1. 격자 설정 (공간 이산화)
    # ---------------------------------------------------------
    N = 2000  # 격자점 개수 (Matrix 크기: 2000 x 2000)
    L = 10.0  # 공간 범위 -L ~ L
    dx = (2 * L) / (N - 1)
    x = np.linspace(-L, L, N)

    # ---------------------------------------------------------
    # 2. 포텐셜 정의 V(x)
    # ---------------------------------------------------------
    # 예: 조화 진동자 (Harmonic Oscillator) V(x) = 0.5 * x^2
    # 이론적 정답 에너지: 0.5, 1.5, 2.5, ...
    V = 0.5 * x**2 
    # (옵션) 비조화 항을 추가하여 문제를 어렵게 만들 수도 있음
    # V = 0.5 * x**2 + 0.1 * x**4 

    # ---------------------------------------------------------
    # 3. Matrix-Free Hamiltonian 연산자 정의 (H = T + V)
    # ---------------------------------------------------------
    # H * psi = -0.5 * d^2/dx^2 * psi + V * psi
    def hamiltonian_operator(psi):
        # (1) 운동 에너지 (Kinetic): -0.5 * 2차 미분 (Central Difference)
        # d^2/dx^2 approx (psi[i+1] - 2psi[i] + psi[i-1]) / dx^2
        T_psi = np.zeros_like(psi)
        T_psi[1:-1] = -0.5 * (psi[2:] - 2*psi[1:-1] + psi[0:-2]) / (dx**2)
        
        # (2) 위치 에너지 (Potential): V * psi
        V_psi = V * psi
        
        return T_psi + V_psi

    # LinearOperator 객체 생성
    H_op = spla.LinearOperator((N, N), matvec=hamiltonian_operator)

    # ---------------------------------------------------------
    # 4. Lanczos 반복법 실행 (eigsh)
    # ---------------------------------------------------------
    print(f"Matrix Size: {N}x{N}")
    print("Lanczos 알고리즘으로 하위 5개 고윳값 계산 중...")

    # eigsh: Eigenvalues of Sparse Hermitian (대칭 행렬용 Lanczos 래퍼)
    # k=5: 찾고 싶은 고윳값 개수
    # which='SA': Smallest Algebraic (가장 작은 값부터 찾음 -> 바닥 상태)
    # tol: 수렴 오차
    evals, evecs = spla.eigsh(H_op, k=5, which='SA', tol=1e-8)

    # ---------------------------------------------------------
    # 5. 결과 분석 및 시각화
    # ---------------------------------------------------------
    print("\n[계산된 에너지 준위 (Eigenvalues)]")
    for i, e in enumerate(evals):
        print(f"State {i} (E_{i}): {e:.6f} (이론값: {0.5 + i:.1f})")

    # 파동함수(고유벡터) 그리기
    plt.figure(figsize=(10, 6))
    
    # 포텐셜 그리기 (배경)
    plt.plot(x, V, 'k--', linewidth=1, alpha=0.5, label='Potential V(x)')
    
    # 고유함수 그리기 (에너지 준위에 맞춰서 y축 이동)
    scale = 1.0 # 파동함수 크기 조절용
    for i in range(5):
        # 확률 밀도 |psi|^2 대신 파동함수 psi 자체를 그림 (위상 확인용)
        # y축 위치를 에너지값(evals[i])에 더해서 층층이 그림
        plt.plot(x, evals[i] + scale * evecs[:, i], label=f'State {i}')
        plt.axhline(evals[i], color='gray', linestyle=':', alpha=0.3)

    plt.title("Iterative Diagonalization via Lanczos Method (Quantum Harmonic Oscillator)")
    plt.xlabel("Position x")
    plt.ylabel("Energy")
    plt.ylim(0, 5) # 보고 싶은 에너지 범위
    plt.xlim(-6, 6)
    plt.legend(loc='upper right')
    plt.show()

solve_schrodinger_lanczos()

In [ ]:
import numpy as np
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

def solve_high_precision_lanczos():
    # ---------------------------------------------------------
    # 1. 격자 설정
    # ---------------------------------------------------------
    N = 1000  # 격자점 개수 (고차 미분을 쓰면 N이 적어도 정확함)
    L = 10.0  # 공간 범위 -L ~ L
    dx = (2 * L) / (N - 1)
    x = np.linspace(-L, L, N)

    # ---------------------------------------------------------
    # 2. 포텐셜 정의 V(x)
    # ---------------------------------------------------------
    # 조화 진동자: 이론적 에너지는 0.5, 1.5, 2.5 ...
    V = 0.5 * x**2 

    # ---------------------------------------------------------
    # 3. 고정밀 Matrix-Free Hamiltonian (5-point Stencil)
    # ---------------------------------------------------------
    # H * psi = -0.5 * d^2/dx^2 * psi + V * psi
    def hamiltonian_operator_5point(psi):
        # 2계 미분 배열 초기화
        d2_psi = np.zeros_like(psi)
        
        # [5-point Central Difference Formula]
        # 오차율: O(dx^4) - 매우 정밀함
        # 경계조건(Boundary Condition): 양 끝 2칸은 0으로 가정 (Dirichlet)
        # psi[2:-2] 영역에 대해 계산 수행
        
        # 수식: (-psi[i+2] + 16*psi[i+1] - 30*psi[i] + 16*psi[i-1] - psi[i-2]) / 12dx^2
        
        term_p2 = psi[4:]      # psi(i+2)
        term_p1 = psi[3:-1]    # psi(i+1)
        term_c  = psi[2:-2]    # psi(i)
        term_m1 = psi[1:-3]    # psi(i-1)
        term_m2 = psi[0:-4]    # psi(i-2)
        
        d2_psi[2:-2] = (-term_p2 + 16*term_p1 - 30*term_c + 16*term_m1 - term_m2) / (12 * dx**2)
        
        # 운동 에너지 (-1/2 * d^2/dx^2)
        T_psi = -0.5 * d2_psi
        
        # 위치 에너지 (V * psi)
        V_psi = V * psi
        
        return T_psi + V_psi

    # LinearOperator 객체 생성
    H_op = spla.LinearOperator((N, N), matvec=hamiltonian_operator_5point)

    # ---------------------------------------------------------
    # 4. Lanczos 반복법 실행
    # ---------------------------------------------------------
    print(f"시스템 크기: {N}x{N} (5-point High Precision Stencil 적용)")
    print("Lanczos 알고리즘 실행 중...")

    # 바닥 상태부터 5개 찾기
    evals, evecs = spla.eigsh(H_op, k=5, which='SA', tol=1e-10)

    # ---------------------------------------------------------
    # 5. 결과 출력 (정밀도 확인)
    # ---------------------------------------------------------
    print("\n[에너지 준위 결과 비교]")
    print(f"{'State':<10} | {'Calculated':<12} | {'Theory':<10} | {'Error':<12}")
    print("-" * 50)
    
    for i, e in enumerate(evals):
        theory = 0.5 + i
        error = abs(e - theory)
        print(f"State {i:<4} | {e:.8f}   | {theory:.1f}        | {error:.2e}")

    # ---------------------------------------------------------
    # 6. 시각화
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6))
    plt.plot(x, V, 'k--', linewidth=1, alpha=0.3, label='Potential V(x)')
    
    scale = 0.8
    for i in range(5):
        # 파동함수의 부호를 맞춰서 그리기 (가끔 뒤집혀서 나올 수 있음)
        psi = evecs[:, i]
        if psi[np.argmax(np.abs(psi))] < 0: 
            psi = -psi
            
        plt.plot(x, evals[i] + scale * psi, label=f'State {i} (E={evals[i]:.4f})')
        plt.axhline(evals[i], color='gray', linestyle=':', alpha=0.3)

    plt.title("High-Precision Quantum Solver (5-point Stencil)")
    plt.xlabel("Position x")
    plt.ylabel("Energy")
    plt.ylim(0, 5.5)
    plt.xlim(-6, 6)
    plt.legend()
    plt.show()

solve_high_precision_lanczos()

In [ ]:
import numpy as np
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

def solve_ultra_precision_lanczos_7point():
    # ---------------------------------------------------------
    # 1. 격자 설정
    # ---------------------------------------------------------
    N = 1000  # 격자점 개수
    L = 10.0  # 공간 범위
    dx = (2 * L) / (N - 1)
    x = np.linspace(-L, L, N)

    # ---------------------------------------------------------
    # 2. 포텐셜 V(x)
    # ---------------------------------------------------------
    V = 0.5 * x**2 

    # ---------------------------------------------------------
    # 3. 7-point Stencil Hamiltonian Operator
    # ---------------------------------------------------------
    def hamiltonian_operator_7point(psi):
        d2_psi = np.zeros_like(psi)
        
        # [7-point Central Difference Formula]
        # Accuracy: O(dx^6)
        # 분자 계수: 2, -27, 270, -490, 270, -27, 2
        # 분모: 180 * dx^2
        
        # 양 끝 3칸(총 6칸)은 경계조건으로 0 처리하고
        # [3:-3] 구간에 대해서만 계산한다.
        
        # 슬라이싱을 이용한 벡터화 연산 (Vectorized Slicing)
        # 인덱스 기준: i를 중앙(term_c)으로 둠
        
        term_p3 = psi[6:]      # psi(i+3)
        term_p2 = psi[5:-1]    # psi(i+2)
        term_p1 = psi[4:-2]    # psi(i+1)
        term_c  = psi[3:-3]    # psi(i)
        term_m1 = psi[2:-4]    # psi(i-1)
        term_m2 = psi[1:-5]    # psi(i-2)
        term_m3 = psi[0:-6]    # psi(i-3)
        
        numerator = (
             2.0 * term_p3 
            - 27.0 * term_p2 
            + 270.0 * term_p1 
            - 490.0 * term_c 
            + 270.0 * term_m1 
            - 27.0 * term_m2 
            + 2.0 * term_m3
        )
        
        d2_psi[3:-3] = numerator / (180.0 * dx**2)
        
        # Hamiltonian H = -1/2 * d^2/dx^2 + V
        return -0.5 * d2_psi + V * psi

    # LinearOperator 생성
    H_op = spla.LinearOperator((N, N), matvec=hamiltonian_operator_7point)

    # ---------------------------------------------------------
    # 4. Lanczos 실행
    # ---------------------------------------------------------
    print(f"시스템 크기: {N}x{N} (7-point Ultra Precision Stencil)")
    
    # 매우 정밀하므로 tol 값을 더 낮춰서 극한의 정밀도 테스트
    evals, evecs = spla.eigsh(H_op, k=5, which='SA', tol=1e-13)

    # ---------------------------------------------------------
    # 5. 결과 비교
    # ---------------------------------------------------------
    print("\n[에너지 준위 결과 (7-point Stencil)]")
    print(f"{'State':<6} | {'Calculated':<14} | {'Theory':<10} | {'Error':<12}")
    print("-" * 50)
    
    for i, e in enumerate(evals):
        theory = 0.5 + i
        error = abs(e - theory)
        # 오차가 매우 작으므로 e 표기법 사용
        print(f"{i:<6} | {e:.10f}     | {theory:.1f}        | {error:.2e}")

    # ---------------------------------------------------------
    # 6. 시각화
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6))
    plt.plot(x, V, 'k--', linewidth=1, alpha=0.3, label='Potential V(x)')
    
    scale = 0.8
    for i in range(5):
        psi = evecs[:, i]
        if psi[np.argmax(np.abs(psi))] < 0: psi = -psi # 위상 정렬
        plt.plot(x, evals[i] + scale * psi, label=f'n={i}')
        plt.axhline(evals[i], color='gray', linestyle=':', alpha=0.3)

    plt.title("7-point Stencil Quantum Solver (O(dx^6))")
    plt.xlabel("Position x")
    plt.ylabel("Energy")
    plt.ylim(0, 5.5)
    plt.xlim(-6, 6)
    plt.legend()
    plt.show()

solve_ultra_precision_lanczos_7point()

In [ ]:
import numpy as np
import scipy.sparse as sparse
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

def solve_exact_7point_lanczos():
    # ---------------------------------------------------------
    # 1. 격자 및 파라미터 설정
    # ---------------------------------------------------------
    N = 1000       # 격자점 개수
    L = 10.0       # 공간 범위
    dx = (2 * L) / (N - 1)
    x = np.linspace(-L, L, N)

    # ---------------------------------------------------------
    # 2. 희소 행렬(Sparse Matrix) 생성
    # ---------------------------------------------------------
    # 7-point Central Difference Coefficients
    # 분모: 180 * dx^2
    denom = 180.0 * dx**2
    
    # 계수들 (Coefficients)
    c0 = -490.0 / denom  # 중심 (i)
    c1 =  270.0 / denom  # i±1
    c2 =  -27.0 / denom  # i±2
    c3 =    2.0 / denom  # i±3
    
    # 대각선 데이터 생성 (모든 행에 대해 동일한 값)
    # 1D 배열을 만들어서 대각 행렬 생성기에 전달
    ones = np.ones(N)
    
    # 운동 에너지 행렬 T = -0.5 * d^2/dx^2
    # 계수에 -0.5를 곱해서 T 행렬의 성분을 만듦
    T_diags = [
        -0.5 * c3 * ones,  # offset -3
        -0.5 * c2 * ones,  # offset -2
        -0.5 * c1 * ones,  # offset -1
        -0.5 * c0 * ones,  # offset  0 (Main Diagonal)
        -0.5 * c1 * ones,  # offset +1
        -0.5 * c2 * ones,  # offset +2
        -0.5 * c3 * ones   # offset +3
    ]
    offsets = [-3, -2, -1, 0, 1, 2, 3]
    
    # 희소 행렬 조립 (Compressed Sparse Row format)
    T = sparse.diags(T_diags, offsets, shape=(N, N), format='csr')
    
    # 위치 에너지 행렬 V (대각 행렬)
    V_vec = 0.5 * x**2
    V = sparse.diags([V_vec], [0], shape=(N, N), format='csr')
    
    # 전체 Hamiltonian H = T + V
    H = T + V

    # ---------------------------------------------------------
    # 3. 고윳값 계산 (Shift-Invert Mode) - 핵심 변경 사항
    # ---------------------------------------------------------
    print(f"시스템 크기: {N}x{N}")
    print("Shift-Invert 모드로 정밀 계산 중...")

    # sigma=0.0: 0.0 근처의 해를 찾겠다는 뜻
    # which='LM': (H - sigma)^(-1)의 Largest Magnitude를 찾음
    #             => 즉, H의 0에 가장 가까운 값을 찾음 (매우 안정적)
    k_target = 5
    evals, evecs = spla.eigsh(H, k=k_target, sigma=0.0, which='LM', tol=1e-13)

    # ---------------------------------------------------------
    # 4. 결과 분석
    # ---------------------------------------------------------
    print("\n[에너지 준위 결과 (Shift-Invert 7-point)]")
    print(f"{'State':<6} | {'Calculated':<16} | {'Theory':<10} | {'Error':<12}")
    print("-" * 55)
    
    for i, e in enumerate(evals):
        theory = 0.5 + i
        error = abs(e - theory)
        print(f"{i:<6} | {e:.12f}     | {theory:.1f}        | {error:.2e}")

    # ---------------------------------------------------------
    # 5. 시각화
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6))
    plt.plot(x, V_vec, 'k--', linewidth=1, alpha=0.3, label='Potential V(x)')
    
    scale = 0.8
    for i in range(k_target):
        psi = evecs[:, i]
        # 파동함수 정규화 및 위상 정렬
        psi = psi / np.linalg.norm(psi) * np.sqrt(N) # 시각화를 위해 크기 조정
        if psi[np.argmax(np.abs(psi))] < 0: psi = -psi 
            
        plt.plot(x, evals[i] + scale * psi * 0.2, label=f'n={i}')
        plt.axhline(evals[i], color='gray', linestyle=':', alpha=0.3)

    plt.title("Shift-Invert Lanczos Solver (Correct & Stable)")
    plt.xlabel("Position x")
    plt.ylabel("Energy")
    plt.ylim(0, 5.0)
    plt.xlim(-6, 6)
    plt.legend()
    plt.show()

solve_exact_7point_lanczos()

In [ ]:
import numpy as np
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

def solve_matrix_free_shift_invert():
    # ---------------------------------------------------------
    # 1. 격자 설정
    # ---------------------------------------------------------
    N = 1000
    L = 10.0
    dx = (2 * L) / (N - 1)
    x = np.linspace(-L, L, N)
    
    # 7-point 계수 미리 계산
    denom = 180.0 * dx**2
    # Hamiltonian 계수 (-0.5 * d^2/dx^2)
    # 7-point 공식의 계수에 -0.5를 곱하고 dx^2로 나눈 것
    coeff_0 = -0.5 * (-490.0) / denom
    coeff_1 = -0.5 * ( 270.0) / denom
    coeff_2 = -0.5 * ( -27.0) / denom
    coeff_3 = -0.5 * (   2.0) / denom
    
    V = 0.5 * x**2 # 포텐셜

    # ---------------------------------------------------------
    # 2. 순방향 연산자 (H * v) 정의
    # ---------------------------------------------------------
    # GMRES가 (H - sigma*I)x = b 를 풀기 위해 필요함
    def apply_hamiltonian(psi):
        # 7-point stencil (Vectorized) using numpy slicing
        # Boundary condition: 0
        res = np.zeros_like(psi)
        
        # Central
        res[3:-3] = coeff_0 * psi[3:-3]
        
        # Neighbors
        res[3:-3] += coeff_1 * (psi[4:-2] + psi[2:-4]) # i+1, i-1
        res[3:-3] += coeff_2 * (psi[5:-1] + psi[1:-5]) # i+2, i-2
        res[3:-3] += coeff_3 * (psi[6:]   + psi[0:-6]) # i+3, i-3
        
        # Add Potential
        return res + V * psi

    # ---------------------------------------------------------
    # 3. 역방향 연산자 ((H - sigma*I)^-1 * v) 정의 - 핵심!
    # ---------------------------------------------------------
    # Lanczos(eigsh)가 호출할 "가짜 역행렬 함수"
    
    sigma = 0.0 # 타겟 에너지 (Shift 값)
    
    # (H - sigma * I) 연산자 생성 (GMRES용)
    def apply_shifted_H(v):
        return apply_hamiltonian(v) - sigma * v
        
    Op_shifted = spla.LinearOperator((N, N), matvec=apply_shifted_H)
    
    # 역행렬 연산자 함수
    # 입력 v를 받아서, (H - sigma*I)x = v 의 해 x를 구해서 반환
    def apply_inverse_op(v):
        # Inner Loop: GMRES로 선형 방정식 풀이
        # tol: Inner loop는 Outer loop보다 더 정밀해야 함 (보통 1e-10 이하)
        x, info = spla.gmres(Op_shifted, v, rtol=1e-12, atol=1e-12)
        if info != 0:
            print("Warning: Inner GMRES did not converge")
        return x

    # 이것이 eigsh에 들어갈 "LinearOperator"이다.
    # eigsh 입장에서는 이것이 그냥 행렬 A처럼 보인다.
    Op_inv = spla.LinearOperator((N, N), matvec=apply_inverse_op)

    # ---------------------------------------------------------
    # 4. 고윳값 계산 (Outer Loop)
    # ---------------------------------------------------------
    print(f"시스템 크기: {N}x{N} (Matrix-Free + Shift-Invert)")
    print("Outer Loop(Lanczos)가 Inner Loop(GMRES)를 호출한다. (시간 소요됨)")
    
    # 우리는 (H - sigma)^(-1)의 고윳값을 구한다.
    # 이 값들은 1 / (E - sigma) 형태이다.
    # 따라서 절대값이 가장 큰 놈('LM')이 E가 sigma에 가장 가까운 놈이다.
    
    k_target = 5
    evals_inv, evecs = spla.eigsh(Op_inv, k=k_target, which='LM', tol=1e-9)
    
    # 고윳값 변환: lambda_inv = 1 / (E - sigma)  =>  E = 1/lambda_inv + sigma
    evals = (1.0 / evals_inv) + sigma
    
    # 정렬 (작은 에너지 순서대로)
    idx = np.argsort(evals)
    evals = evals[idx]
    evecs = evecs[:, idx]

    # ---------------------------------------------------------
    # 5. 결과 출력
    # ---------------------------------------------------------
    print("\n[Matrix-Free Shift-Invert 결과]")
    print(f"{'State':<6} | {'Calculated':<14} | {'Theory':<10} | {'Error':<12}")
    print("-" * 55)
    
    for i, e in enumerate(evals):
        theory = 0.5 + i
        error = abs(e - theory)
        print(f"{i:<6} | {e:.10f}     | {theory:.1f}        | {error:.2e}")

    # 시각화 (생략 가능하나 확인용)
    plt.figure(figsize=(8, 5))
    plt.plot(x, V, 'k--', alpha=0.3)
    for i in range(k_target):
        psi = evecs[:, i]
        if psi[np.argmax(np.abs(psi))] < 0: psi = -psi
        # 정규화
        psi = psi / np.linalg.norm(psi) * 4.0 
        plt.plot(x, evals[i] + psi, label=f'n={i}')
    plt.ylim(0, 5)
    plt.legend()
    plt.show()

solve_matrix_free_shift_invert()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def arnoldi_iteration(A, b, m):
    """
    Arnoldi Iteration
    A: NxN 행렬 (일반/비대칭)
    b: 초기 시작 벡터
    m: Krylov 부분공간의 차원 (반복 횟수)
    
    Returns:
    Q: (N x m+1) 정규 직교 기저 행렬 (Krylov Basis)
    H: (m+1 x m) 상부 헤센버그 행렬
    """
    N = A.shape[0]
    
    # 1. 결과 담을 행렬 초기화
    Q = np.zeros((N, m + 1))
    H = np.zeros((m + 1, m))
    
    # 2. 초기 벡터 정규화
    Q[:, 0] = b / np.linalg.norm(b)
    
    # 3. 반복 (Krylov subspace 확장)
    for k in range(m):
        # (1) 다음 후보 벡터 생성: v = A * q_k
        v = A @ Q[:, k]
        
        # (2) 직교화 (Gram-Schmidt): 이전 모든 기저들에 대해 수행
        for j in range(k + 1):
            # 내적 계산 H[j, k] = q_j^T * v
            H[j, k] = np.dot(Q[:, j], v)
            # 성분 제거 (Projection)
            v = v - H[j, k] * Q[:, j]
            
        # (3) 정규화 (새로운 축 생성)
        H[k + 1, k] = np.linalg.norm(v)
        
        # Breakdown 체크 (운 좋게 정확한 해를 찾은 경우 등)
        if H[k + 1, k] < 1e-12: 
            print("Lucky breakdown!")
            break
            
        Q[:, k + 1] = v / H[k + 1, k]
        
    return Q, H

# ---------------------------------------------------------
# 실행 및 검증
# ---------------------------------------------------------
np.random.seed(42)
N = 100
m = 20 # 축소할 차원

# 비대칭 행렬 생성
A = np.random.rand(N, N)
b = np.random.rand(N)

# 아놀디 실행
Q, H = arnoldi_iteration(A, b, m)

# ---------------------------------------------------------
# 결과 시각화
# ---------------------------------------------------------
# 축소된 행렬 H (m x m 부분만 떼어내서 고윳값 계산)
H_m = H[:m, :] 

# 고윳값 비교
eig_A = np.linalg.eigvals(A)       # 전체 (참값)
eig_H = np.linalg.eigvals(H_m)     # 근사값

plt.figure(figsize=(10, 5))

# (1) 행렬 H의 구조 (상부 헤센버그 형태 확인)
plt.subplot(1, 2, 1)
plt.imshow(np.abs(H_m) > 1e-6, cmap='Greys', interpolation='nearest')
plt.title(f"Structure of Hessenberg Matrix H ({m}x{m})")
plt.grid(False)

# (2) 고윳값 분포 (복소 평면)
plt.subplot(1, 2, 2)
plt.scatter(eig_A.real, eig_A.imag, c='gray', alpha=0.3, label='Original Eigenvalues')
plt.scatter(eig_H.real, eig_H.imag, c='red', marker='x', s=50, label='Ritz Values (Arnoldi)')
plt.title("Eigenvalue Approximation")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def lanczos_iteration(A, b, m):
    """
    Lanczos Iteration
    A: NxN 행렬 (대칭 행렬이어야 함)
    b: 초기 시작 벡터
    m: 반복 횟수
    
    Returns:
    Q: (N x m) 정규 직교 기저 행렬
    T: (m x m) 삼중 대각 행렬
    """
    N = A.shape[0]
    
    # 1. 초기화
    Q = np.zeros((N, m))
    alpha = np.zeros(m) # 대각 성분
    beta = np.zeros(m)  # 부대각 성분 (beta[0]은 사용 안함)
    
    # 초기 벡터 정규화
    v = b / np.linalg.norm(b)
    Q[:, 0] = v
    
    # 2. 반복
    for k in range(m - 1):
        # 현재 기저 벡터 q_k
        w = A @ Q[:, k]
        
        # (1) 알파 계산 (대각 성분): alpha_k = q_k^T A q_k
        alpha[k] = np.dot(Q[:, k], w)
        
        # (2) 직교화: w = A*q_k - alpha_k*q_k - beta_{k-1}*q_{k-1}
        # (직전 2개 항만 빼주면 됨 -> 3항 점화식)
        w = w - alpha[k] * Q[:, k]
        if k > 0:
            w = w - beta[k] * Q[:, k-1]
            
        # (3) 베타 계산 (부대각 성분): 다음 벡터의 norm
        beta[k+1] = np.linalg.norm(w)
        
        if beta[k+1] < 1e-12: break
            
        # (4) 다음 기저 벡터 저장
        Q[:, k+1] = w / beta[k+1]
        
    # 마지막 alpha 처리
    w = A @ Q[:, m-1]
    alpha[m-1] = np.dot(Q[:, m-1], w)
    
    # 3. 삼중 대각 행렬 T 조립
    # Main diagonal: alpha
    # Off diagonal: beta (1부터 끝까지)
    T = np.diag(alpha) + np.diag(beta[1:], k=1) + np.diag(beta[1:], k=-1)
    
    return Q, T

# ---------------------------------------------------------
# 실행 및 검증
# ---------------------------------------------------------
np.random.seed(42)
N = 100
m = 20

# 대칭 행렬 생성 (Symmetric Matrix)
# A = B + B.T 형태로 만들면 무조건 대칭임
B = np.random.randn(N, N)
A = B + B.T 
b = np.random.rand(N)

# 란초스 실행
Q, T = lanczos_iteration(A, b, m)

# ---------------------------------------------------------
# 결과 시각화
# ---------------------------------------------------------
eig_A = np.linalg.eigvalsh(A) # 전체 (참값) - 대칭행렬용 함수 사용
eig_T = np.linalg.eigvalsh(T) # 근사값

plt.figure(figsize=(10, 5))

# (1) 행렬 T의 구조 (삼중 대각 형태 확인)
plt.subplot(1, 2, 1)
# 0이 아닌 부분만 점 찍기 (Spy plot)
plt.spy(T, markersize=5, color='black')
plt.title(f"Structure of Tridiagonal Matrix T ({m}x{m})")
plt.grid(True)

# (2) 고윳값 분포 (실수축 위의 점)
plt.subplot(1, 2, 2)
# 보기 좋게 y축을 0과 1로 나누어 표현
plt.plot(eig_A, np.zeros_like(eig_A), 'ko', alpha=0.2, label='Original Eigenvalues', markersize=8)
plt.plot(eig_T, np.zeros_like(eig_T) + 0.1, 'rx', markeredgewidth=2, label='Ritz Values (Lanczos)', markersize=8)
plt.title("Eigenvalue Approximation (Real Axis)")
plt.yticks([])
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ortho_group # 랜덤 직교 행렬 생성용

# =========================================================
# 1. 테스트용 행렬 생성 함수 (정답 고윳값 심어넣기)
# =========================================================
def create_test_matrix(eigenvalues, symmetric=False):
    """
    주어진 eigenvalues를 가지는 Dense 행렬 A를 생성한다.
    symmetric=True이면 대칭 행렬(Lanczos용), False면 일반 행렬(Arnoldi용)을 만든다.
    원리: A = Q * diag(eigenvalues) * Q.T
    """
    N = len(eigenvalues)
    
    # 1. 랜덤한 직교 행렬(Rotation Matrix) Q 생성
    # (행렬을 섞어버리는 역할)
    Q = ortho_group.rvs(dim=N)
    
    # 2. 대각 행렬 Lambda 생성
    Lambda = np.diag(eigenvalues)
    
    # 3. 유사 변환 (Similarity Transformation)
    # A와 Lambda는 고윳값이 같다.
    if symmetric:
        # 대칭 행렬: A = Q D Q^T
        A = Q @ Lambda @ Q.T
    else:
        # 비대칭 행렬을 만들기 위해 Q를 살짝 찌그러뜨리거나 다른 역행렬 사용
        # 여기서는 간단히 Q D Q^T를 하되, 노이즈를 섞어 비대칭성을 부여하거나
        # 복소수 고윳값을 위해 블록 대각 행렬을 써야 하지만,
        # 편의상 A = Q D Q^-1 형태로 구현
        A = Q @ Lambda @ Q.T
        
        # 강제로 비대칭성을 부여하기 위해 상삼각 성분에 노이즈 추가
        # (주의: 이렇게 하면 고윳값이 약간 변하지만, Arnoldi 테스트용으로는 충분함)
        tri_u = np.triu(np.random.rand(N, N), k=1) * 0.5
        A = A + tri_u 
        
        # 비대칭 행렬의 진짜 고윳값 다시 계산 (노이즈 때문에 변했으므로)
        return A, np.linalg.eigvals(A)

    return A, eigenvalues

# =========================================================
# 2. 아놀디 알고리즘 (Arnoldi Method)
# =========================================================
def run_arnoldi(A, m):
    """
    일반 행렬용 Krylov subspace 방법
    Return: 리츠 값(Approximated Eigenvalues)
    """
    N = A.shape[0]
    b = np.random.rand(N) # 시작 벡터
    Q = np.zeros((N, m + 1))
    H = np.zeros((m + 1, m))
    
    Q[:, 0] = b / np.linalg.norm(b)
    
    for k in range(m):
        v = A @ Q[:, k]
        for j in range(k + 1):
            H[j, k] = np.dot(Q[:, j], v)
            v = v - H[j, k] * Q[:, j]
            
        H[k + 1, k] = np.linalg.norm(v)
        if H[k + 1, k] < 1e-12: break
        Q[:, k + 1] = v / H[k + 1, k]
        
    # 상부 헤센버그 행렬 H_m (m x m)의 고윳값 계산
    H_m = H[:m, :m]
    ritz_values = np.linalg.eigvals(H_m)
    return ritz_values

# =========================================================
# 3. 란초스 알고리즘 (Lanczos Method)
# =========================================================
def run_lanczos(A, m):
    """
    대칭 행렬용 Krylov subspace 방법
    Return: 리츠 값(Approximated Eigenvalues)
    """
    N = A.shape[0]
    b = np.random.rand(N)
    Q = np.zeros((N, m))
    alpha = np.zeros(m)
    beta = np.zeros(m)
    
    v = b / np.linalg.norm(b)
    Q[:, 0] = v
    
    for k in range(m - 1):
        w = A @ Q[:, k]
        alpha[k] = np.dot(Q[:, k], w)
        
        w = w - alpha[k] * Q[:, k]
        if k > 0:
            w = w - beta[k] * Q[:, k-1]
            
        beta[k+1] = np.linalg.norm(w)
        if beta[k+1] < 1e-12: break
        Q[:, k+1] = w / beta[k+1]
        
    # 마지막 alpha 처리
    w = A @ Q[:, m-1]
    alpha[m-1] = np.dot(Q[:, m-1], w)
    
    # 삼중 대각 행렬 T의 고윳값 계산
    # eigh_tridiagonal은 대칭 삼중대각 전용이라 매우 빠름
    # d: diagonal(alpha), e: off-diagonal(beta[1:])
    from scipy.linalg import eigh_tridiagonal
    ritz_values = eigh_tridiagonal(alpha, beta[1:])
    return ritz_values[0] # 고윳값만 리턴

# =========================================================
# 4. 메인 실행 및 시각화
# =========================================================
def compare_algorithms():
    np.random.seed(0)
    
    # 설정
    N = 100         # 전체 행렬 크기
    m = 15          # Krylov 부분공간 크기 (15번만 반복)
    
    # -----------------------------------------------------
    # Case A: 대칭 행렬 (Lanczos 테스트)
    # -----------------------------------------------------
    # 우리가 원하는 정답 고윳값 (스펙트럼)
    # 0~1 사이에 촘촘하게 있고, 10.0과 -5.0에 떨어진 값(Outlier) 배치
    # Krylov 방법은 이렇게 멀리 떨어진 값을 가장 먼저 찾는다.
    true_evals_sym = np.linspace(0, 1, N-2)
    true_evals_sym = np.append(true_evals_sym, [10.0, -5.0]) # Outliers added
    
    A_sym, _ = create_test_matrix(true_evals_sym, symmetric=True)
    approx_evals_sym = run_lanczos(A_sym, m)
    
    # -----------------------------------------------------
    # Case B: 비대칭 행렬 (Arnoldi 테스트)
    # -----------------------------------------------------
    # 랜덤 고윳값을 가진 비대칭 행렬 생성
    # (함수 내부에서 노이즈를 추가해 비대칭으로 만들고 실제 고윳값 리턴)
    dummy_evals = np.linspace(0, 10, N) 
    A_gen, true_evals_gen = create_test_matrix(dummy_evals, symmetric=False)
    approx_evals_gen = run_arnoldi(A_gen, m)
    
    # -----------------------------------------------------
    # 시각화
    # -----------------------------------------------------
    plt.figure(figsize=(12, 6))
    
    # (1) Lanczos 결과 (실수축)
    plt.subplot(1, 2, 1)
    # 정답 그리기 (검은선)
    for e in true_evals_sym:
        plt.axhline(e, color='black', alpha=0.3, linewidth=1)
    plt.plot(np.zeros_like(true_evals_sym), true_evals_sym, 'ko', label='True Eigenvalues')
    
    # 근사값 그리기 (빨간 X)
    plt.plot(np.zeros_like(approx_evals_sym), approx_evals_sym, 'rx', markersize=10, markeredgewidth=2, label=f'Lanczos (m={m})')
    
    plt.title("Lanczos Convergence (Symmetric)")
    plt.ylabel("Eigenvalue")
    plt.xticks([])
    plt.legend()
    # 설명: 멀리 떨어진 10.0과 -5.0은 정확히 맞춤
    
    # (2) Arnoldi 결과 (복소평면)
    plt.subplot(1, 2, 2)
    plt.scatter(true_evals_gen.real, true_evals_gen.imag, c='black', alpha=0.3, label='True Eigenvalues')
    plt.scatter(approx_evals_gen.real, approx_evals_gen.imag, c='red', marker='x', s=80, linewidth=2, label=f'Arnoldi (m={m})')
    
    plt.title("Arnoldi Convergence (Non-Symmetric)")
    plt.xlabel("Real Part")
    plt.ylabel("Imaginary Part")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    compare_algorithms()

In [ ]:
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

def solve_block_davidson():
    # ---------------------------------------------------------
    # 1. 문제 설정
    # ---------------------------------------------------------
    N = 500
    L = 8.0
    dx = (2 * L) / (N - 1)
    x = np.linspace(-L, L, N)
    V_pot = 0.5 * x**2 
    diag_kinetic = 1.0 / (dx**2)
    D = diag_kinetic + V_pot
    # ---------------------------------------------------------
    # 2. Hamiltonian
    # ---------------------------------------------------------
    def apply_hamiltonian(Psi_block):
        T_psi = np.zeros_like(Psi_block)
        T_psi[1:-1, :] = -0.5 * (Psi_block[2:, :] - 2*Psi_block[1:-1, :] + Psi_block[:-2, :]) / dx**2
        V_psi = V_pot[:, None] * Psi_block
        return T_psi + V_psi

    # ---------------------------------------------------------
    # 3. Solver
    # ---------------------------------------------------------
    print("Block Davidson solver 시작...")
    n_roots = 5        
    block_size = 10    
    max_subspace = 40  
    tol = 1e-6         
    max_iter = 300
    V = np.random.rand(N, block_size)
    V, _ = la.qr(V, mode='economic')
    final_evals = None
    final_evecs = None
    # [수정 1] 비상시 사용할 최신 근사해를 담을 변수
    last_iter_best_vecs = None 
    for iteration in range(max_iter):
        AV = apply_hamiltonian(V)
        T = V.T @ AV
        evals_sub, evecs_sub = la.eigh(T)
        current_evals = evals_sub[:n_roots]
        # [수정 2] V가 변하기(Restart) 전에 현재의 베스트 해를 미리 계산해둠
        # V(N x k) @ evecs(k x n_roots) -> (N x n_roots) 매칭 보장됨
        last_iter_best_vecs = V @ evecs_sub[:, :n_roots]
        residuals = []
        converged_count = 0
        correction_vectors = []
        for i in range(block_size): 
            lambda_i = evals_sub[i]
            # 여기서는 i번째 벡터 하나만 계산
            x_i = V @ evecs_sub[:, i] 
            Av_i = AV @ evecs_sub[:, i]
            r = Av_i - lambda_i * x_i
            norm_r = np.linalg.norm(r)
            if i < n_roots and norm_r < tol:
                converged_count += 1
            denom = D - lambda_i
            denom = np.where(np.abs(denom) < 1e-6, 1e-6, denom)
            delta = r / denom
            correction_vectors.append(delta)
        print(f"Iter {iteration+1}: Converged {converged_count}/{n_roots}, E_0 = {current_evals[0]:.6f}")
        if converged_count >= n_roots:
            print(">>> 모든 목표 고윳값 수렴 완료!")
            final_evals = current_evals
            # 수렴했으니 미리 계산해둔 값을 확정
            final_evecs = last_iter_best_vecs 
            break
        new_vecs = np.array(correction_vectors).T 
        combined = np.hstack([V, new_vecs])
        Q_combined, _ = la.qr(combined, mode='economic')
        # Restart Logic (여기서 V의 크기가 변할 수 있음)
        if Q_combined.shape[1] > max_subspace:
            # print("--- Restarting Subspace ---")
            best_ritz_vecs = V @ evecs_sub[:, :block_size]
            V, _ = la.qr(best_ritz_vecs, mode='economic')
        else:
            V = Q_combined
    # [수정 3] 반복문 종료 후 처리
    # V가 Restart로 인해 크기가 변했더라도, last_iter_best_vecs는 안전함
    if final_evals is None:
        print(">>> 주의: 최대 반복 횟수 도달. 현재 근사값을 출력한다.")
        final_evals = evals_sub[:n_roots]
        final_evecs = last_iter_best_vecs # 안전하게 미리 저장된 벡터 사용
    # ---------------------------------------------------------
    # 4. 결과 출력
    # ---------------------------------------------------------
    print("\n[Block Davidson 결과]")
    for i, e in enumerate(final_evals):
        theory = 0.5 + i
        print(f"State {i}: {e:.6f} (Theory: {theory:.1f})")
    plt.figure(figsize=(8, 5))
    plt.plot(x, V_pot, 'k--', alpha=0.3)
    for i in range(n_roots):
        psi = final_evecs[:, i]
        if psi[N//2] < 0: psi = -psi
        psi = psi / np.linalg.norm(psi) * 3.0
        plt.plot(x, final_evals[i] + psi, label=f'n={i}')
    plt.title("Block Davidson solution")
    plt.xlim(-5.0,5.0)
    plt.ylim(0.0, 6.0)
    plt.xlabel("x(atomic unit)", fontsize=18)
    plt.ylabel("Energy(atomic unit)", fontsize=18)
    plt.show()

solve_block_davidson()

In [ ]:
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
import time

def solve_3d_harmonic_oscillator_complete():
    # ---------------------------------------------------------
    # 1. 3D 격자 설정
    # ---------------------------------------------------------
    # N=40 -> 전체 행렬 크기 64,000 x 64,000
    N = 40  
    L = 5.0 
    dx = (2 * L) / (N - 1)
    # 1D 좌표
    x_1d = np.linspace(-L, L, N)
    # 3D Grid (Indexing='ij'는 행렬 방식 순서)
    X, Y, Z = np.meshgrid(x_1d, x_1d, x_1d, indexing='ij')
    # 3D Potential: V = 0.5 * (x^2 + y^2 + z^2)
    V_pot_3d = 0.5 * (X**2 + Y**2 + Z**2)
    # Preconditioner 계산을 위해 1차원으로 펼침
    V_pot_flat = V_pot_3d.ravel()
    # 전체 차원 수
    dim = N**3
    print(f"System Dimension: {N}^3 = {dim}")
    # ---------------------------------------------------------
    # 2. 7-point Stencil 계수 설정 (고정밀)
    # ---------------------------------------------------------
    # f''(x) approx (2f(i+3) - 27f(i+2) + ... ) / 180h^2
    denom = 180.0 * (dx**2)
    factor = -0.5 / denom # Hamiltonian Kinetic term (-0.5 * laplacian)
    c3 = factor * 2.0
    c2 = factor * -27.0
    c1 = factor * 270.0
    c0 = factor * -490.0
    # 3D Laplacian의 대각 성분 (x, y, z 방향 모두 고려)
    # D_ii = 3 * (T_ii_1d) + V_ii
    diag_kinetic = 3.0 * c0
    D_flat = diag_kinetic + V_pot_flat
    # ---------------------------------------------------------
    # 3. Matrix-Free 3D Hamiltonian Function
    # ---------------------------------------------------------
    def apply_hamiltonian_3d(Psi_block):
        # Psi_block: (N^3, block_size) -> 2D array
        n_vecs = Psi_block.shape[1]
        # 3D 연산을 위해 (N, N, N, k) 형태로 뷰(View) 변환
        psi_3d = Psi_block.reshape((N, N, N, n_vecs))
        # 결과 배열
        H_psi_3d = np.zeros_like(psi_3d)
        # [Helper] 특정 축(axis) 방향으로 7-point stencil 적용
        def add_kinetic_along_axis(axis_idx):
            # 슬라이싱 헬퍼: axis_idx 방향으로만 [start:end] 적용
            def s(start, end):
                slices = [slice(None)] * 4 # (x, y, z, block)
                slices[axis_idx] = slice(start, end)
                return tuple(slices)
            # Central term
            H_psi_3d[s(3,-3)] += c0 * psi_3d[s(3,-3)]
            # Neighbors (+/- 1)
            H_psi_3d[s(3,-3)] += c1 * (psi_3d[s(4,-2)] + psi_3d[s(2,-4)])
            # Neighbors (+/- 2)
            H_psi_3d[s(3,-3)] += c2 * (psi_3d[s(5,-1)] + psi_3d[s(1,-5)])
            # Neighbors (+/- 3)
            H_psi_3d[s(3,-3)] += c3 * (psi_3d[s(6,None)] + psi_3d[s(0,-6)])
        # x, y, z 축 각각 적용
        add_kinetic_along_axis(0)
        add_kinetic_along_axis(1)
        add_kinetic_along_axis(2)
        # Potential Energy 추가 (Broadcasting)
        H_psi_3d += V_pot_3d[:, :, :, None] * psi_3d
        # 다시 1D로 펴서 반환
        return H_psi_3d.reshape((dim, n_vecs))
    # ---------------------------------------------------------
    # 4. Block Davidson Solver Execution
    # ---------------------------------------------------------
    print("Block Davidson (3D, 7-point High Precision) 시작...")
    start_time = time.time()
    n_roots = 6        # 바닥상태(1) + 1차 여기(3) + 2차 여기 일부(2)
    block_size = 12    # 블록 크기
    max_subspace = 60  # 최대 부분공간 크기
    tol = 1e-6         # 수렴 조건
    max_iter = 300     # 최대 반복 횟수
    # 초기화
    V = np.random.rand(dim, block_size)
    V, _ = la.qr(V, mode='economic')
    # 안전장치용 변수 초기화
    final_evals = None
    for iteration in range(max_iter):
        # (1) Subspace Projection
        AV = apply_hamiltonian_3d(V)
        T = V.T @ AV
        # (2) Diagonalization of Subspace Matrix
        evals_sub, evecs_sub = la.eigh(T)
        # [중요 수정] 현재 스텝의 근사값을 매번 저장 (오류 방지)
        current_evals = evals_sub[:n_roots]
        final_evals = current_evals 
        # (3) Residual Check & Correction Vector Generation
        correction_vectors = []
        converged_count = 0
        for i in range(block_size):
            lam = evals_sub[i]
            # r = AV*y - lam*V*y (이미 계산된 행렬 활용)
            r = AV @ evecs_sub[:, i] - lam * (V @ evecs_sub[:, i])
            norm_r = np.linalg.norm(r)
            # 수렴 체크
            if i < n_roots and norm_r < tol:
                converged_count += 1
            # Davidson Preconditioning: delta = r / (D - lambda)
            denom = D_flat - lam
            # 0으로 나누기 방지 (특이점 처리)
            denom = np.where(np.abs(denom) < 1e-5, 1e-5, denom)
            delta = r / denom
            correction_vectors.append(delta)
        # 진행상황 출력
        if iteration % 5 == 0:
            print(f"Iter {iteration}: Converged {converged_count}/{n_roots}, Ground E = {current_evals[0]:.6f}")
        # (4) Convergence Exit
        if converged_count >= n_roots:
            print(">>> 모든 목표 상태 수렴 완료!")
            break
        # (5) Expansion (New Vectors Addition)
        new_vecs = np.array(correction_vectors).T
        V_next = np.hstack([V, new_vecs])
        # 직교화 (QR)
        Q, _ = la.qr(V_next, mode='economic')
        # (6) Restart Logic
        if Q.shape[1] > max_subspace:
            # 부분공간이 너무 커지면 중요한 벡터(best Ritz vectors)만 남기고 축소
            # print("--- Restarting Subspace ---")
            best_vecs = V @ evecs_sub[:, :block_size]
            V, _ = la.qr(best_vecs, mode='economic')
        else:
            V = Q
    # ---------------------------------------------------------
    # 5. 결과 분석 및 출력
    # ---------------------------------------------------------
    end_time = time.time()
    print(f"\nTotal Time: {end_time - start_time:.2f} sec")
    print("\n[3D Harmonic Oscillator Energy Levels (7-point Stencil)]")
    print(f"{'State':<6} | {'Calculated':<10} | {'Theory':<10} | {'Note'}")
    print("-" * 55)
    # 이론값: E = nx + ny + nz + 1.5
    # 0,0,0 -> 1.5
    # 1,0,0 (3개) -> 2.5
    # 2,0,0 or 1,1,0 (6개) -> 3.5
    theory_levels = [1.5, 2.5, 2.5, 2.5, 3.5, 3.5] 
    for i in range(n_roots):
        val = final_evals[i]
        # n_roots가 이론값 리스트보다 길 경우 대비
        th = theory_levels[i] if i < len(theory_levels) else 0.0
        # 축퇴 여부 주석
        note = ""
        if abs(val - 1.5) < 0.1: note = "Ground"
        elif abs(val - 2.5) < 0.1: note = "1st Excited (Degenerate x3)"
        elif abs(val - 3.5) < 0.1: note = "2nd Excited"
        print(f"{i:<6} | {val:.6f}   | {th:.1f}        | {note}")
solve_3d_harmonic_oscillator_complete()

In [ ]:
import numpy as np

def rayleigh_quotient_iteration_hermitian(A, x0, max_iter=10, tolerance=1e-10):
    """
    Hermitian 행렬을 위한 Rayleigh Quotient Iteration (RQI) 함수.
    A: Hermitian 행렬 (A = A^H).
    x0: 복소수 초기 벡터 (complex-valued vector).
    """
    # 초기 벡터 정규화
    x_k = x0 / np.linalg.norm(x0)
    # 레일리 몫 계산: (x^H A x) / (x^H x). 정규화했으므로 분모는 1.
    # np.conj(x_k).T는 켤레 전치 x^H이다.
    lambda_k = np.conj(x_k).T @ A @ x_k
    # Hermitian 행렬의 고윳값은 실수여야 하므로 실수부만 취한다.
    lambda_k = lambda_k.real
    print(f"초기 고윳값 근사치 (λ_0): {lambda_k:.6f}")

    for k in range(max_iter):
        
        lambda_prev = lambda_k
        
        # Shift된 시스템 풀기: (A - λ_k * I) * z = x_k
        # λ_k는 실수이고, I는 항등 행렬이다.
        B = A - lambda_k * np.eye(A.shape[0])
        
        # 선형 시스템 Bz = x_k 풀기
        try:
            # np.linalg.solve는 복소수 방정식을 처리할 수 있다.
            z_k_plus_1 = np.linalg.solve(B, x_k)
        except np.linalg.LinAlgError:
            print(f"경고: {k+1}번째 반복에서 행렬이 특이 행렬이 되어 해를 구할 수 없다.")
            break

        # 새로운 고유벡터 근사치 정규화
        x_k_plus_1 = z_k_plus_1 / np.linalg.norm(z_k_plus_1)
        
        # 새로운 레일리 몫 계산
        lambda_k_plus_1_complex = np.conj(x_k_plus_1).T @ A @ x_k_plus_1
        # Hermitian 고윳값은 실수이므로 실수부만 취한다.
        lambda_k_plus_1 = lambda_k_plus_1_complex.real
        
        # 수렴 확인
        eigenvalue_diff = np.abs(lambda_k_plus_1 - lambda_prev)
        
        print(f"반복 {k+1}: λ = {lambda_k_plus_1:.6f}, |Δλ| = {eigenvalue_diff:.2e}")
        
        if eigenvalue_diff < tolerance:
            print(f"\n {k+1}번째 반복에서 수렴 (오차 허용치 미만).")
            return lambda_k_plus_1, x_k_plus_1, k + 1
        
        # 업데이트
        lambda_k = lambda_k_plus_1
        x_k = x_k_plus_1
        
    print(f"\n 최대 반복 횟수 ({max_iter}) 도달.")
    return lambda_k, x_k, max_iter
# Hermitian 행렬 A 정의 (A^H = A)
A = np.array([
    [2, 3 + 4j],
    [3 - 4j, 2]
])
# 초기 벡터 x0 (복소수 벡터)
x0 = np.array([1.0 + 0j, 0.5 + 0.5j]) 
# RQI 실행
final_lambda, final_x, iterations = rayleigh_quotient_iteration_hermitian(A, x0, max_iter=5)
# 결과 출력
print("\n--- RQI 결과 ---")
print(f"계산된 고윳값 (λ): {final_lambda:.6f}")
print(f"계산된 고유벡터 (x): {final_x}")
# NumPy의 고유값/고유벡터 함수로 검증
w, v = np.linalg.eigh(A) # eigh는 Hermitian/대칭 행렬 전용 함수
print("\n--- NumPy 검증 ---")
print(f"NumPy 고윳값: {w}")
print(f"NumPy 고유벡터 (열 벡터):\n{v}")    

In [ ]:
import numpy as np
import time

def rayleigh_quotient_iteration_hermitian(A, x0, max_iter=10, tolerance=1e-10):
    # 초기 벡터 정규화
    x_k = x0 / np.linalg.norm(x0)
    # 레일리 몫 계산
    lambda_k = (np.conj(x_k).T @ A @ x_k).real
    # print(f"초기 고윳값 근사치 (λ_0): {lambda_k:.6f}") # 대형 행렬에서는 출력 생략
    for k in range(max_iter):
        lambda_prev = lambda_k
        # Shift된 시스템 풀기: (A - λ_k * I) * z = x_k
        B = A - lambda_k * np.eye(A.shape[0])
        try:
            # 선형 시스템 해결: 대형 행렬의 경우 이 부분이 가장 많은 계산을 차지한다.
            z_k_plus_1 = np.linalg.solve(B, x_k)
        except np.linalg.LinAlgError:
            # print(f"경고: {k+1}번째 반복에서 특이 행렬 발생.")
            break
        # 새로운 고유벡터 근사치 정규화
        x_k_plus_1 = z_k_plus_1 / np.linalg.norm(z_k_plus_1)
        # 새로운 레일리 몫 계산
        lambda_k_plus_1 = (np.conj(x_k_plus_1).T @ A @ x_k_plus_1).real
        # 수렴 확인
        eigenvalue_diff = np.abs(lambda_k_plus_1 - lambda_prev)
        # print(f"반복 {k+1}: λ = {lambda_k_plus_1:.6f}, |Δλ| = {eigenvalue_diff:.2e}") # 대형 행렬에서는 출력 생략
        if eigenvalue_diff < tolerance:
            return lambda_k_plus_1, x_k_plus_1, k + 1
        lambda_k = lambda_k_plus_1
        x_k = x_k_plus_1
    return lambda_k, x_k, max_iter

def main():
    # 행렬 크기 설정 (제법 큰 경우)
    N = 100
    print(f" 행렬 크기: {N} x {N}")
    print("-" * 30)

    # 1. Hermitian 행렬 생성
    # 무작위 복소수 행렬 C를 생성하고 C + C^H 를 계산하여 Hermitian 행렬 A를 만든다.
    # Hermitian 행렬은 A = A^H 이며 고윳값은 실수이다.
    C = np.random.rand(N, N) + 1j * np.random.rand(N, N)
    A = C + np.conj(C).T

    # 2. 초기 벡터 생성
    # 복소수 초기 벡터 (랜덤)
    x0 = np.random.rand(N) + 1j * np.random.rand(N)
    
    # RQI 시작 시간 측정
    start_time_rqi = time.time()
    
    # 3. RQI 실행
    # (일반적으로 RQI는 A의 가장 큰 고유값이나 가장 가까운 고유값으로 수렴)
    final_lambda, final_x, iterations = rayleigh_quotient_iteration_hermitian(A, x0, max_iter=20)
    
    end_time_rqi = time.time()

    # 4. NumPy 검증 및 비교
    start_time_numpy = time.time()
    w, v = np.linalg.eigh(A)
    end_time_numpy = time.time()

    # 5. 결과 출력
    print("\n[RQI 계산 결과]")
    print(f"계산된 고윳값 (λ): {final_lambda:.6f}")
    print(f"반복 횟수: {iterations}")
    print(f"계산 시간: {end_time_rqi - start_time_rqi:.4f} 초")
    
    # RQI 결과가 NumPy 결과 중 어떤 고윳값에 가까운지 확인
    min_diff_index = np.argmin(np.abs(w - final_lambda))
    print("\n[NumPy 검증]")
    print(f"NumPy 고윳값 중 RQI 결과와 가장 가까운 값 (w[{min_diff_index}]): {w[min_diff_index]:.6f}")
    
    # 정확도 검증: A*x와 lambda*x의 차이
    Ax = A @ final_x
    lambda_x = final_lambda * final_x
    residual_norm = np.linalg.norm(Ax - lambda_x)
    print(f"잔차 노름 (||Ax - λx||): {residual_norm:.2e}")
    
    print(f"NumPy 전체 고윳값 계산 시간: {end_time_numpy - start_time_numpy:.4f} 초")

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import time

def rayleigh_quotient_iteration_hermitian(A, x0, max_iter=10, tolerance=1e-10):
    """
    Hermitian 행렬을 위한 Rayleigh Quotient Iteration (RQI) 함수.
    """
    x_k = x0 / np.linalg.norm(x0)
    lambda_k = (np.conj(x_k).T @ A @ x_k).real
    
    for k in range(max_iter):
        lambda_prev = lambda_k
        B = A - lambda_k * np.eye(A.shape[0])
        
        try:
            z_k_plus_1 = np.linalg.solve(B, x_k)
        except np.linalg.LinAlgError:
            break

        x_k_plus_1 = z_k_plus_1 / np.linalg.norm(z_k_plus_1)
        lambda_k_plus_1 = (np.conj(x_k_plus_1).T @ A @ x_k_plus_1).real
        
        eigenvalue_diff = np.abs(lambda_k_plus_1 - lambda_prev)
        
        if eigenvalue_diff < tolerance:
            return lambda_k_plus_1, x_k_plus_1, k + 1
        
        lambda_k = lambda_k_plus_1
        x_k = x_k_plus_1
        
    return lambda_k, x_k, max_iter

def main():
    # 행렬 크기 설정
    N = 100
    print(f" 행렬 크기: {N} x {N}")
    print("-" * 30)

    # 1. Hermitian 행렬 생성
    C = np.random.rand(N, N) + 1j * np.random.rand(N, N)
    A = C + np.conj(C).T

    # 2. 초기 벡터 생성
    x0 = np.random.rand(N) + 1j * np.random.rand(N)
    
    start_time_rqi = time.time()
    
    # 3. RQI 실행
    final_lambda, final_x, iterations = rayleigh_quotient_iteration_hermitian(A, x0, max_iter=20)
    
    end_time_rqi = time.time()

    # 4. NumPy 검증 (모든 고윳값 계산)
    start_time_numpy = time.time()
    w, v = np.linalg.eigh(A)
    end_time_numpy = time.time()

    # 5. 결과 출력
    print("\n[RQI 계산 결과]")
    print(f"계산된 고윳값 (λ): {final_lambda:.6f}")
    print(f"반복 횟수: {iterations}")
    print(f"계산 시간: {end_time_rqi - start_time_rqi:.4f} 초")
    
    # RQI 결과가 NumPy 결과 중 어떤 고윳값에 가까운지 확인
    min_diff_index = np.argmin(np.abs(w - final_lambda))
    print("\n[NumPy 검증 및 전체 스펙트럼]")
    print(f"NumPy 고윳값 중 RQI 결과와 가장 가까운 값 (w[{min_diff_index}]): {w[min_diff_index]:.6f}")
    
    # 잔차 노름 검증
    residual_norm = np.linalg.norm(A @ final_x - final_lambda * final_x)
    print(f"잔차 노름 (||Ax - λx||): {residual_norm:.2e}")
    
    print("-" * 30)
    # **추가된 부분: 모든 고윳값 출력**
    print(f"NumPy로 계산된 **모든 고윳값 (총 {N}개)**:")
    # 모든 고윳값을 소수점 4자리까지 출력 (보기 쉽게 정렬하여 출력)
    print(np.array2string(w, precision=4, separator=', ', suppress_small=True))
    print(f"(참고: 고윳값의 범위는 약 {w.min():.2f} 에서 {w.max():.2f} 이다.)")
    print("-" * 30)
    
    print(f"NumPy 전체 고윳값 계산 시간: {end_time_numpy - start_time_numpy:.4f} 초")

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import time

# RQI 함수는 이전과 동일하다 (내부에서 정규화를 수행함)
def rayleigh_quotient_iteration_hermitian(A, x0, max_iter=10, tolerance=1e-10):
    """
    Hermitian 행렬을 위한 Rayleigh Quotient Iteration (RQI) 함수.
    """
    x_k = x0 / np.linalg.norm(x0)
    lambda_k = (np.conj(x_k).T @ A @ x_k).real
    
    for k in range(max_iter):
        lambda_prev = lambda_k
        B = A - lambda_k * np.eye(A.shape[0])
        
        try:
            z_k_plus_1 = np.linalg.solve(B, x_k)
        except np.linalg.LinAlgError:
            break

        # 정규화 부분 (Orthonormality 조건 중 Normalization을 만족시킴)
        x_k_plus_1 = z_k_plus_1 / np.linalg.norm(z_k_plus_1)
        
        lambda_k_plus_1 = (np.conj(x_k_plus_1).T @ A @ x_k_plus_1).real
        eigenvalue_diff = np.abs(lambda_k_plus_1 - lambda_prev)
        
        if eigenvalue_diff < tolerance:
            return lambda_k_plus_1, x_k_plus_1, k + 1
        
        lambda_k = lambda_k_plus_1
        x_k = x_k_plus_1
        
    return lambda_k, x_k, max_iter

def main():
    # 행렬 크기 설정
    N = 100
    print(f" 행렬 크기: {N} x {N}")
    print("-" * 30)

    # 1. Hermitian 행렬 생성
    C = np.random.rand(N, N) + 1j * np.random.rand(N, N)
    A = C + np.conj(C).T

    # 2. 초기 벡터 생성
    x0 = np.random.rand(N) + 1j * np.random.rand(N)
    
    start_time_rqi = time.time()
    
    # 3. RQI 실행
    final_lambda, final_x, iterations = rayleigh_quotient_iteration_hermitian(A, x0, max_iter=20)
    
    end_time_rqi = time.time()

    # 4. NumPy 검증 (모든 고윳값 계산)
    start_time_numpy = time.time()
    w, v = np.linalg.eigh(A)
    end_time_numpy = time.time()

    # 5. 결과 출력
    print("\n[RQI 계산 결과]")
    print(f"계산된 고윳값 (λ): {final_lambda:.6f}")
    print(f"반복 횟수: {iterations}")
    print(f"계산 시간: {end_time_rqi - start_time_rqi:.4f} 초")
    
    # RQI 결과가 NumPy 결과 중 어떤 고윳값에 가까운지 확인
    min_diff_index = np.argmin(np.abs(w - final_lambda))
    print("\n[NumPy 검증 및 전체 스펙트럼]")
    print(f"NumPy 고윳값 중 RQI 결과와 가장 가까운 값 (w[{min_diff_index}]): {w[min_diff_index]:.6f}")
    
    # 잔차 노름 검증
    residual_norm = np.linalg.norm(A @ final_x - final_lambda * final_x)
    print(f"잔차 노름 (||Ax - λx||): {residual_norm:.2e}")
    
    # **추가된 부분: Orthonormality (정규화) 조건 체크**
    norm_x = np.linalg.norm(final_x)
    # L2 노름이 1과 얼마나 가까운지 확인
    orthonormal_check = np.abs(norm_x - 1.0)
    
    print("\n[직교 정규성 (Orthonormality) 검사]")
    print(f"계산된 고유벡터의 L2 노름 (||x||): {norm_x:.10f}")
    
    if orthonormal_check < 1e-9:
        print(" 정규화 조건 만족: ||x||는 1에 매우 근접한다.")
    else:
        print(" 정규화 조건 불만족.")
    
    print("-" * 30)
    # 모든 고윳값 출력
    print(f"NumPy로 계산된 **모든 고윳값 (총 {N}개)**:")
    print(np.array2string(w, precision=4, separator=', ', suppress_small=True))
    print(f"(참고: 고윳값의 범위는 약 {w.min():.2f} 에서 {w.max():.2f} 이다.)")
    print("-" * 30)
    
    print(f"NumPy 전체 고윳값 계산 시간: {end_time_numpy - start_time_numpy:.4f} 초")

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 초기 설정
L = 50  # 그리드 크기 (50x50)
T = np.zeros((L, L)) # 온도 행렬 초기화 (모두 0)

# 경계 조건 설정 (Boundary Conditions)
# 위쪽 경계 (y=L-1) 온도를 100으로 설정
T[L-1, :] = 100.0
# 왼쪽, 오른쪽, 아래쪽 경계는 0으로 유지

# 2. 가우스-자이델 반복법 (평형 상태에 도달할 때까지 반복)
max_iterations = 1000
for iteration in range(max_iterations):
    T_old = T.copy()

    # 내부 그리드 포인트에 대해 반복
    for i in range(1, L - 1):
        for j in range(1, L - 1):
            # 라플라스 방정식의 유한 차분 근사:
            # T_new[i, j] = 1/4 * (T[i+1, j] + T[i-1, j] + T[i, j+1] + T[i, j-1])
            T[i, j] = 0.25 * (T[i+1, j] + T[i-1, j] + T[i, j+1] + T[i, j-1])
            
    # 수렴 확인 (옵션: 오차 변화가 작아지면 중단)
    error = np.max(np.abs(T - T_old))
    if error < 1e-4:
        break

# 3. 히트맵 시각화
plt.figure(figsize=(8, 6), dpi=150)

# 'imshow'를 사용하여 행렬 데이터를 색상 맵으로 변환
# 'inferno'는 대비가 강하여 표지 그림으로 시각적인 임팩트가 좋다.
plt.imshow(T, cmap='inferno', origin='lower')

# 컬러바 추가 (온도 스케일 표시)
cbar = plt.colorbar(label='Temperature distribution (T)')
cbar.set_label('Temperature)', fontsize=12)

# 축 레이블 제거 (수치 해석적 느낌 강조)
plt.xticks([])
plt.yticks([])

plt.title(f'Heatmap of Laplace equation solution (Iterations: {iteration+1})', fontsize=15)
plt.show()

## Chapter 5 미분방정식    파이썬을 활용한 수치해석(Numerical Analysis with Python) 이인호 (북스힐, 2026)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def rk4_solver(f, t0, y0, t_end, h):
    """
    룽게-쿠타 4차(RK4) 방법을 사용하여 상미분방정식을 푸는 함수
    Args:
    f (function): dy/dt = f(t, y) 형태의 미분방정식 함수
    t0 (float): 초기 t 값
    y0 (float): 초기 y 값
    t_end (float): 적분 종료 t 값
    h (float): 스텝 크기 (step size)
    Returns:
    tuple: t 값 배열과 y 값 배열 (t_values, y_values)
    """
    t_values = [t0]
    y_values = [y0]
    t = t0
    y = y0
    while t < t_end:
        # 마지막 스텝 크기가 t_end를 넘지 않도록 조정
        if t + h > t_end:
            h = t_end - t
        # RK4 계수 계산
        # k1은 t에서의 기울기 (오일러 방법의 기울기와 같음)
        k1 = f(t, y)
        # k2는 중간점 (t + h/2, y + k1*h/2)에서의 기울기
        k2 = f(t + 0.5 * h, y + 0.5 * h * k1)
        # k3는 또 다른 중간점 (t + h/2, y + k2*h/2)에서의 기울기
        k3 = f(t + 0.5 * h, y + 0.5 * h * k2)
        # k4는 끝점 (t + h, y + k3*h)에서의 기울기
        k4 = f(t + h, y + h * k3)
        # 다음 y 값 계산 (가중 평균 기울기를 사용)
        y_next = y + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
        # t와 y 업데이트
        t_next = t + h
        t = t_next
        y = y_next
        t_values.append(t)
        y_values.append(y)
    return np.array(t_values), np.array(y_values)
# --- 예제 상미분방정식 정의 ---
def func_f(t, y):
    """ 
    미분방정식: dy/dt = t - y
    """
    return t - y
# --- 초기 조건 및 설정 ---
t0 = 0.0          # 초기 t
y0 = 1.0          # 초기 y (y(0) = 1)
t_end = 5.0       # 적분 종료 t
h = 0.1           # 스텝 크기
# --- RK4 솔버 실행 ---
t_rk4, y_rk4 = rk4_solver(func_f, t0, y0, t_end, h)
# --- 정확한 해 (참고용) ---
def exact_solution(t):
    """
    미분방정식 dy/dt = t - y의 정확한 해: y(t) = 2*exp(-t) + t - 1
    """
    return 2 * np.exp(-t) + t - 1
t_exact = np.linspace(t0, t_end, 100)
y_exact = exact_solution(t_exact)
# --- 결과 시각화 ---
plt.figure(figsize=(10, 6))
plt.plot(t_exact, y_exact, label='Exact solution', color='blue', linewidth=2)
plt.plot(t_rk4, y_rk4, 'o', label=f'RK4 solution (h={h})', color='red', markersize=4)
plt.title(f'RK4 solution for $dy/dt = t - y$')
plt.xlabel('t', fontsize=16)
plt.ylabel('y(t)', fontsize=16)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def simulation_comparison_with_verlet():
    # 1. 설정 (Configuration)
    dt = 0.1          # 시간 간격
    steps = 1000      # 총 시뮬레이션 횟수
    # 초기 조건
    x0, v0 = 1.0, 0.0
    # 데이터 저장용 배열
    # [1] 일반 오일러 (Standard Euler)
    x_euler, v_euler, E_euler = np.zeros(steps), np.zeros(steps), np.zeros(steps)
    # [2] 심플렉틱 오일러 (Symplectic Euler)
    x_sym, v_sym, E_sym = np.zeros(steps), np.zeros(steps), np.zeros(steps)
    # [3] 속도 벨렛 (Velocity Verlet) - NEW!
    x_ver, v_ver, E_ver = np.zeros(steps), np.zeros(steps), np.zeros(steps)
    # 초기값 세팅
    x_euler[0] = x_sym[0] = x_ver[0] = x0
    v_euler[0] = v_sym[0] = v_ver[0] = v0
    initial_energy = 0.5 * v0**2 + 0.5 * x0**2
    E_euler[0] = E_sym[0] = E_ver[0] = initial_energy
    # 가속도 함수 (F = -kx, m=1 이라 가정하면 a = -x)
    def get_acc(x):
        return -x
    # 2. 시뮬레이션 루프
    for i in range(steps - 1):
	    # --- A. 일반 오일러 (Standard Euler) ---
	    a_curr = get_acc(x_euler[i])
	    x_euler[i+1] = x_euler[i] + v_euler[i] * dt
	    v_euler[i+1] = v_euler[i] + a_curr * dt
	    E_euler[i+1] = 0.5 * v_euler[i+1]**2 + 0.5 * x_euler[i+1]**2
	    # --- B. 심플렉틱 오일러 (Symplectic Euler) ---
	    a_curr_sym = get_acc(x_sym[i])
	    v_sym[i+1] = v_sym[i] + a_curr_sym * dt      # 속도 먼저 갱신
	    x_sym[i+1] = x_sym[i] + v_sym[i+1] * dt      # 갱신된 속도로 위치 갱신
	    E_sym[i+1] = 0.5 * v_sym[i+1]**2 + 0.5 * x_sym[i+1]**2
	    # --- C. 속도 벨렛 (Velocity Verlet) ---
	    # 1단계: 속도를 절반(0.5 dt)만큼 업데이트 (Kick)
	    a_curr_ver = get_acc(x_ver[i])
	    v_half = v_ver[i] + 0.5 * a_curr_ver * dt
    	# 2단계: 위치를 완전한 한 스텝(dt)만큼 업데이트 (Drift)
	    x_ver[i+1] = x_ver[i] + v_half * dt
	    # 3단계: 새로운 위치에서의 가속도 계산
	    a_next_ver = get_acc(x_ver[i+1])
    	# 4단계: 남은 속도 절반(0.5 dt)을 업데이트 (Kick)
	    v_ver[i+1] = v_half + 0.5 * a_next_ver * dt
	    # 에너지 계산
	    E_ver[i+1] = 0.5 * v_ver[i+1]**2 + 0.5 * x_ver[i+1]**2
    # 3. 결과 시각화
    plt.figure(figsize=(14, 6))
    # 그래프 1: 위상 공간 (Phase Space)
    plt.subplot(1, 2, 1)
    plt.plot(x_euler, v_euler, 'r--', label='Standard Euler(Explodes)', alpha=0.5)
    plt.plot(x_sym, v_sym, 'b-', label='Symplectic Euler(Stable)', alpha=0.6)
    plt.plot(x_ver,v_ver,'g-',label='Velocity Verlet(Stable & Accurate)',linewidth=2)
    plt.title('Phase Space (Orbit)')
    plt.xlabel('Position (x)')
    plt.ylabel('Velocity (v)')
    plt.legend()
    plt.grid(True)
    plt.axis('equal')
    # 그래프 2: 에너지 오차 (초기 에너지와의 차이)
    plt.subplot(1, 2, 2)
    # 로그 스케일이 아니면 일반 오일러 때문에 다른 그래프가 안 보임. 오차만 확대해서 본다.
    plt.plot(E_euler - initial_energy, 'r--', label='Standard Euler Error')
    plt.plot(E_sym - initial_energy, 'b-', label='Symp. Euler Error')
    plt.plot(E_ver - initial_energy, 'g-', label='Verlet Error')
    plt.title('Energy Error (E - E0)')
    plt.xlabel('Steps')
    plt.ylabel('Energy Drift')
    plt.legend()
    plt.grid(True)
    plt.ylim(-0.2, 0.2) # 에너지 오차를 자세히 보기 위해 y축 제한
    plt.tight_layout()
    plt.savefig('symplectic.png')
    plt.show()
if __name__ == "__main__":
    simulation_comparison_with_verlet()

In [ ]:
import numpy as np
# 1. 미분방정식 정의
def f(t, y):
    """dy/dt = -2*t*y"""
    # y는 항상 1차원 배열이라고 가정하고 계산
    return -2.0 * t * y
# 2. 수정된 중점 방법 (Modified Midpoint Method)
def modified_midpoint(f, t0, y0_array, H, n):
    """
    구간 H에 대해 n개의 부분 단계를 사용하여 해를 근사하는 함수.
    """
    h = H / n
    t = t0
    y = y0_array.copy()
    # (1) 초기 단계 (Euler Step)
    y_old = y.copy()
    y = y_old + h * f(t, y_old)
    # (2) 재귀 단계 (Midpoint Steps)
    for i in range(1, n):
        t += h
        y_next = y_old + 2.0 * h * f(t, y)
        y_old = y
        y = y_next
    # (3) 마지막 단계 (Smoothing Step)
    t += h # t = t0 + H
    T_n = 0.5 * (y + y_old + h * f(t, y))
    return T_n
# 3. Richardson/Neville 다항식 외삽법
def neville_extrapolation(T_values, n_steps):
    """
    T_values (배열 리스트)를 사용하여 다항식 외삽법을 적용한다.
    T_values는 NumPy 배열의 리스트이다.
    """
    m = len(T_values)
    # T는 외삽법 테이블. 모든 요소를 0으로 초기화하고, T_values를 복사
    # T[i, j]는 (m, m, len(y)) 차원의 3차원 배열이 된다.
    T = np.zeros((m, m, T_values[0].size))
    # 0차 (기본 T_n 값 복사)
    for i in range(m):
        T[i, 0] = T_values[i]
    # 1차 이상의 외삽법
    for j in range(1, m): # 외삽 차수 (j = 1, 2, ...)
        for i in range(j, m): # 데이터 포인트 (i = j, j+1, ...)
            # (h_{i-j} / h_i)^2 = (n_i / n_{i-j})^2
            ratio_sq = (n_steps[i] / n_steps[i-j])**2
            # Neville/Richardson 다항식 외삽 공식
            # T_{i,j}=T_{i,j-1}+(T_{i,j-1}-T_{i-1,j-1})/((h_{i-j}/h_i)^2-1)
            # **주의**: T[i-1,j-1] 대신 T[i-1,j]가 들어가야 함(Neville 테이블 규칙)
            # T[i, j] = T[i, j-1] + (T[i, j-1] - T[i-1, j-1])/(ratio_sq - 1.0)
            # Bulirsch-Stoer의 Rational Function Extrapolation과 유사하게 
            # 일반적인 Neville 다항식 보간 공식을 사용한다.:
            T[i, j] = T[i, j-1] + (T[i, j-1] - T[i-1, j-1]) / (ratio_sq - 1.0)
            # NOTE: 이 공식은 T[i-1, j-1]을 사용하므로 인덱싱 오류를 피할 수 있다.
    # 가장 정확한 추정값은 오른쪽 아래 값 (T[m-1, m-1]은 배열)
    return T[m-1, m-1], T
    # 4. Bulirsch-Stoer 메인 루틴
def bulirsch_stoer_step(f, t0, y0_array, H):
    # n의 시퀀스
    n_sequence = [2, 4, 6, 8, 12, 16] 
    T_values = []
    # 1. 수정된 중점 방법으로 T_n 값들을 계산
    for n in n_sequence:
        T_n = modified_midpoint(f, t0, y0_array, H, n)
        T_values.append(T_n)
    # 2. 외삽법 적용
    final_result, T_table = neville_extrapolation(T_values, n_sequence)
    return final_result, T_table, n_sequence
# 5. 실행
t_start = 0.0
t_end = 1.0
H_total = t_end - t_start
# 초기 조건을 **반드시** 1차원 NumPy 배열로 정의
y_initial = np.array([1.0]) 
# Bulirsch-Stoer 계산
y_final_bs, T_table, n_sequence = bulirsch_stoer_step(f, t_start, y_initial, H_total)	
# 해석적 해
y_analytical = np.array([np.exp(-(t_end**2))])
print(f"---  Bulirsch-Stoer 알고리즘 (t={t_start}에서 t={t_end}까지) ---")
print(f"**해석적 해 y({t_end})**: {y_analytical[0]:.12f}")
print(f"**Bulirsch-Stoer 해 y({t_end})**: {y_final_bs[0]:.12f}")
print(f"**절대 오차**: {np.abs(y_final_bs[0] - y_analytical[0]):.2e}")
print("\n---  외삽법 테이블 (T[i, j]) ---")
# T_table은 (m, m, 1) 차원의 3차원 배열이다.
for i, n in enumerate(n_sequence):
	row = f"n={n:2}: "
	for j in range(i + 1):
	    # T_table[i, j]는 NumPy 배열이므로 [0]으로 스칼라 값을 추출
	    row += f"{T_table[i, j, 0]:.12f}  "
	print(row)

In [ ]:
import numpy as np
import time
# --- ODE 정의 ---
def f(t, y):
    """테스트 ODE: y' = y"""
    return y
# --- 정확한 해 ---
def true_solution(t):
    """정확한 해: y(t) = e^t"""
    return np.exp(t)
# --- RK4 함수 (ABM 초기값 계산용) ---
def rk4_step(f, t_n, y_n, h):
    k1 = h * f(t_n, y_n)
    k2 = h * f(t_n + 0.5 * h, y_n + 0.5 * k1)
    k3 = h * f(t_n + 0.5 * h, y_n + 0.5 * k2)
    k4 = h * f(t_n + h, y_n + k3)
    return y_n + (k1 + 2*k2 + 2*k3 + k4) / 6
# --- 1. 아담스-배쉬포스-몰튼 4차 (ABM4) ---
def adams_bashforth_moulton_4th_order(f, t_start, y_start, t_end, N):
    h = (t_end - t_start) / N
    t = np.linspace(t_start, t_end, N + 1)
    y = np.zeros(N + 1)
    y[0] = y_start
    # RK4로 초기 3개 값 계산
    for i in range(3):
        y[i+1] = rk4_step(f, t[i], y[i], h)
    f_values = [f(t[i], y[i]) for i in range(4)]
    for i in range(3, N):
        # 예측 (AB4)
        y_star = y[i] + h / 24 * (
            55 * f_values[3] - 59 * f_values[2] + 
            37 * f_values[1] -  9 * f_values[0]
        )
        f_star_next = f(t[i+1], y_star)
        # 수정 (AM4)
        y[i+1] = y[i] + h / 24 * (
             9 * f_star_next + 19 * f_values[3] - 
             5 * f_values[2] +  1 * f_values[1] 
        )
        # f_values 업데이트
        f_values.pop(0) 
        f_values.append(f(t[i+1], y[i+1]))
    return y[-1]
# --- 2. 룽게-쿠타 4차 (RK4) ---
def runge_kutta_4th_order(f, t_start, y_start, t_end, N):
    h = (t_end - t_start) / N
    y = y_start
    t = t_start
    for _ in range(N):
        k1 = h * f(t, y)
        k2 = h * f(t + 0.5 * h, y + 0.5 * k1)
        k3 = h * f(t + 0.5 * h, y + 0.5 * k2)
        k4 = h * f(t + h, y + k3)
        y = y + (k1 + 2*k2 + 2*k3 + k4) / 6
        t = t + h
    return y
# --- 3. 오일러 방법 (Euler's Method) ---
def euler_method(f, t_start, y_start, t_end, N):
    h = (t_end - t_start) / N
    y = y_start
    t = t_start
    for _ in range(N):
        y = y + h * f(t, y)
        t = t + h
    return y
# --- 4. 수정된 오일러 방법 (Modified Euler / Midpoint Method) ---
def modified_euler_method(f, t_start, y_start, t_end, N):
    h = (t_end - t_start) / N
    y = y_start
    t = t_start
    for _ in range(N):
        k1 = f(t, y)
        k2 = f(t + 0.5 * h, y + 0.5 * h * k1)
        y = y + h * k2
        t = t + h
    return y
# --- 5. 호인 방법 (Heun's Method / Improved Euler) ---
def heuns_method(f, t_start, y_start, t_end, N):
    h = (t_end - t_start) / N
    y = y_start
    t = t_start
    for _ in range(N):
        f_n = f(t, y)
        y_star = y + h * f_n
        f_star_next = f(t + h, y_star)
        y = y + 0.5 * h * (f_n + f_star_next)
        t = t + h
    return y
# --- 비교할 함수 리스트 ---
methods = {
	"Euler (1차)": (euler_method, 1),
	"Mod. Euler (2차)": (modified_euler_method, 2),
	"Heun (2차)": (heuns_method, 2),
	"RK4 (4차)": (runge_kutta_4th_order, 4),
	"ABM4 (4차)": (adams_bashforth_moulton_4th_order, 4),
    }
# --- 문제 설정 ---
T_START = 0.0
Y_START = 1.0
T_END = 1.0
N_STEPS = 1000  # 적분 스텝 수 (정확도와 계산 시간을 동시에 비교하기 위함)
TRUE_VALUE = true_solution(T_END)
results = []
print(f"##  ODE 수치적분 방법 성능 비교 (N={N_STEPS} 스텝)")
print("-" * 60)
print(f"| {'방법':<15} | {'차수':<3} | {'최종 해':<15} | {'절대 오차':<15} | {'계산 시간 (s)':<15} |")
print("|" + "-"*17 + "|" + "-"*5 + "|" + "-"*17 + "|" + "-"*17 + "|" + "-"*17 + "|")
for name, (method_func, order) in methods.items():
    start_time = time.perf_counter()
    # 적분 실행
    final_y = method_func(f, T_START, Y_START, T_END, N_STEPS)
    end_time = time.perf_counter()
    # 결과 계산
    error = np.abs(final_y - TRUE_VALUE)
    elapsed_time = end_time - start_time
    results.append({
 	    "name": name,
	    "order": order,
	    "final_y": final_y,
	    "error": error,
	    "time": elapsed_time
    })
    # 결과 출력
    print(f"| {name:<15} | {order:<3} | {final_y:.10f} | {error:.2e} | {elapsed_time:.6f} |")
print("-" * 60)
print(f"| {'정확한 해':<15} | {'-':<3} | {TRUE_VALUE:.10f} | {'-':<15} | {'-':<15} |")
print("-" * 60)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# ---------------------------------------------------------
# 1. 문제 정의
# ---------------------------------------------------------
def f(t, y):
	""" 도함수 y' = f(t, y) """
	return -y + t + 1
def exact_solution(t):
	""" 실제 정답 y(t) """
	return t + np.exp(-t)
# ---------------------------------------------------------
# 2. RK4 (초기 시동용)
# ---------------------------------------------------------
def rk4_step(t, y, h):
	k1 = h * f(t, y)
	k2 = h * f(t + 0.5*h, y + 0.5*k1)
	k3 = h * f(t + 0.5*h, y + 0.5*k2)
	k4 = h * f(t + h, y + k3)
	return y + (k1 + 2*k2 + 2*k3 + k4) / 6.0
# ---------------------------------------------------------
# 3. AB-AM Predictor-Corrector Solver
# ---------------------------------------------------------
def solve_ab_am_4th_order(t0, y0, h, num_steps):
	# 시간 배열 생성
	t = np.linspace(t0, t0 + num_steps*h, num_steps + 1)
	y = np.zeros(num_steps + 1)
	y[0] = y0
	# [Step 1] RK4로 초기 3스텝(y1, y2, y3) 계산 (시동 걸기)
	print(">>> 초기 3스텝은 RK4로 시동을 건다.")
	for i in range(3):
	    y[i+1] = rk4_step(t[i], y[i], h)
	# [Step 2] AB-AM 루프 시작 (i는 3부터 시작)
	# AB4 예측자 계수: [55, -59, 37, -9] / 24
	# AM4 수정자 계수: [9, 19, -5, 1] / 24
	print(">>> AB-AM 4차 예측-수정 알고리즘 가동 중...")
	for i in range(3, num_steps):
	    # 미리 계산된 기울기들 (History)
	    f_n   = f(t[i],   y[i])
	    f_n1  = f(t[i-1], y[i-1])
	    f_n2  = f(t[i-2], y[i-2])
	    f_n3  = f(t[i-3], y[i-3])
	    # -----------------------------------------------
	    # (A) Predictor (Adams-Bashforth 4-step)
	    # -----------------------------------------------
	    y_pred = y[i] + (h/24.0) * (55*f_n - 59*f_n1 + 37*f_n2 - 9*f_n3)
	    # -----------------------------------------------
	    # (B) Evaluator
	    # -----------------------------------------------
	    # 예측된 미래 시점(t[i+1])에서의 기울기 계산
	    f_next_pred = f(t[i+1], y_pred)
	    # -----------------------------------------------
	    # (C) Corrector (Adams-Moulton 3-step implicit)
	    # -----------------------------------------------
	    # AM4 공식: y_{n+1} = y_n + h/24 * (9*f_{n+1} + 19*f_n - 5*f_{n-1} + f_{n-2})
	    y[i+1] = y[i] + (h/24.0) * (9*f_next_pred + 19*f_n - 5*f_n1 + f_n2)
	return t, y
# ---------------------------------------------------------
# 4. 실행 및 결과 비교
# ---------------------------------------------------------
# 설정: 0초부터 10초까지, 간격 h=0.1
t0 = 0.0
y0 = 1.0 # y(0) = 0 + e^0 = 1
h = 0.2  # 스텝 사이즈 (일부러 오차를 보기 위해 조금 크게 설정)
steps = 50
# 계산 수행
t_vals, y_approx = solve_ab_am_4th_order(t0, y0, h, steps)
# 실제 정답 계산
y_exact = exact_solution(t_vals)
# 오차 계산
error = np.abs(y_exact - y_approx)
# ---------------------------------------------------------
# 5. 시각화
# ---------------------------------------------------------
plt.figure(figsize=(12, 5))
# (1) 해 궤적 비교
plt.subplot(1, 2, 1)
plt.plot(t_vals, y_exact, 'k-', linewidth=2, label='Exact solution')
plt.plot(t_vals, y_approx, 'r--', marker='o', markersize=4, label='AB-AM (4th order)')
plt.title(f"Predictor-Corrector solution (h={h})")
plt.xlabel("Time t")
plt.ylabel("y(t)")
plt.legend()
plt.grid(True)
# (2) 오차 그래프
plt.subplot(1, 2, 2)
plt.plot(t_vals, error, 'b.-')
plt.title("Absolute error")
plt.xlabel("Time t")
plt.ylabel("|Exact - Approx|")
plt.yscale('log') # 로그 스케일로 오차 확인
plt.grid(True)
plt.tight_layout()
plt.show()
# ---------------------------------------------------------
# 결과 텍스트 출력
# ---------------------------------------------------------
print("\n" + "="*40)
print(f"최종 시간 t = {t_vals[-1]:.1f}")
print(f"실제값 : {y_exact[-1]:.8f}")
print(f"근삿값 : {y_approx[-1]:.8f}")
print(f"오  차 : {error[-1]:.4e}")
print("="*40)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rk4_solver(f, t0, y0, t_end, h):
    """
    룽게-쿠타 4차(RK4) 방법을 사용하여 상미분방정식을 푸는 함수

    Args:
        f (function): dy/dt = f(t, y) 형태의 미분 방정식 함수
        t0 (float): 초기 t 값
        y0 (float): 초기 y 값
        t_end (float): 적분 종료 t 값
        h (float): 스텝 크기 (step size)

    Returns:
        tuple: t 값 배열과 y 값 배열 (t_values, y_values)
    """
    t_values = [t0]
    y_values = [y0]
    t = t0
    y = y0

    while t < t_end:
        # 마지막 스텝 크기가 t_end를 넘지 않도록 조정
        if t + h > t_end:
            h = t_end - t

        # RK4 계수 계산
        # k1은 t에서의 기울기 (오일러 방법의 기울기와 같음)
        k1 = f(t, y)
        
        # k2는 중간점 (t + h/2, y + k1*h/2)에서의 기울기
        k2 = f(t + 0.5 * h, y + 0.5 * h * k1)
        
        # k3는 또 다른 중간점 (t + h/2, y + k2*h/2)에서의 기울기
        k3 = f(t + 0.5 * h, y + 0.5 * h * k2)
        
        # k4는 끝점 (t + h, y + k3*h)에서의 기울기
        k4 = f(t + h, y + h * k3)

        # 다음 y 값 계산 (가중 평균 기울기를 사용)
        y_next = y + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
        
        # t와 y 업데이트
        t_next = t + h
        t = t_next
        y = y_next
        
        t_values.append(t)
        y_values.append(y)

    return np.array(t_values), np.array(y_values)

# --- 예제 상미분방정식 정의 ---
def func_f(t, y):
    """
    미분 방정식: dy/dt = t - y
    """
    return t - y

# --- 초기 조건 및 설정 ---
t0 = 0.0          # 초기 t
y0 = 1.0          # 초기 y (y(0) = 1)
t_end = 5.0       # 적분 종료 t
h = 0.1           # 스텝 크기

# --- RK4 솔버 실행 ---
t_rk4, y_rk4 = rk4_solver(func_f, t0, y0, t_end, h)

# --- 정확한 해 (참고용) ---
def exact_solution(t):
    """
    미분 방정식 dy/dt = t - y의 정확한 해: y(t) = 2*exp(-t) + t - 1
    """
    return 2 * np.exp(-t) + t - 1

t_exact = np.linspace(t0, t_end, 100)
y_exact = exact_solution(t_exact)

# --- 결과 시각화 ---
plt.figure(figsize=(10, 6))
plt.plot(t_exact, y_exact, label='Exact solution', color='blue', linewidth=2)
plt.plot(t_rk4, y_rk4, 'o', label=f'RK4 solution (h={h})', color='red', markersize=4)

plt.title(f'RK4 Solution for $dy/dt = t - y$')
plt.xlabel('t')
plt.ylabel('y(t)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. 물리 시스템 설정 (단진동: F = -kx) ---
class HarmonicOscillator:
    def __init__(self, k=1.0, m=1.0):
        self.k = k
        self.m = m

    def get_accel(self, x):
        """a = F/m = -kx/m"""
        return -(self.k / self.m) * x

    def get_energy(self, x, v):
        """Total Energy = 0.5mv^2 + 0.5kx^2"""
        return 0.5 * self.m * v**2 + 0.5 * self.k * x**2

# --- 2. RK4 Solver ---
def solve_rk4(system, x0, v0, dt, t_max):
    t_values = [0]
    x_values = [x0]
    v_values = [v0]
    e_values = [system.get_energy(x0, v0)]
    
    x = x0
    v = v0
    t = 0
    
    while t < t_max:
        # 상태 벡터 state = [x, v]
        # 변화율 함수 f(state) -> [v, a]
        
        # k1
        k1_x = v
        k1_v = system.get_accel(x)
        
        # k2
        k2_x = v + k1_v * (dt / 2)
        k2_v = system.get_accel(x + k1_x * (dt / 2))
        
        # k3
        k3_x = v + k2_v * (dt / 2)
        k3_v = system.get_accel(x + k2_x * (dt / 2))
        
        # k4
        k4_x = v + k3_v * dt
        k4_v = system.get_accel(x + k3_x * dt)
        
        # Update
        x_new = x + (dt / 6) * (k1_x + 2*k2_x + 2*k3_x + k4_x)
        v_new = v + (dt / 6) * (k1_v + 2*k2_v + 2*k3_v + k4_v)
        
        x, v = x_new, v_new
        t += dt
        
        t_values.append(t)
        x_values.append(x)
        v_values.append(v)
        e_values.append(system.get_energy(x, v))
        
    return t_values, x_values, e_values

# --- 3. Velocity Verlet Solver ---
def solve_verlet(system, x0, v0, dt, t_max):
    t_values = [0]
    x_values = [x0]
    v_values = [v0]
    e_values = [system.get_energy(x0, v0)]
    
    x = x0
    v = v0
    # Verlet은 시작할 때 가속도가 필요함
    a = system.get_accel(x)
    t = 0
    
    while t < t_max:
        # 1. 위치 반 스텝 전진 (Half-step update는 아니지만 순서상 위치 먼저 확정)
        x_new = x + v * dt + 0.5 * a * (dt**2)
        
        # 2. 속도 반 스텝 전진 (v_half)
        v_half = v + 0.5 * a * dt
        
        # 3. 새로운 위치에서의 가속도 갱신
        a_new = system.get_accel(x_new)
        
        # 4. 속도 나머지 반 스텝 전진
        v_new = v_half + 0.5 * a_new * dt
        
        x, v, a = x_new, v_new, a_new
        t += dt
        
        t_values.append(t)
        x_values.append(x)
        v_values.append(v)
        e_values.append(system.get_energy(x, v))
        
    return t_values, x_values, e_values

# --- 4. 메인 실행 및 비교 ---

# 설정
system = HarmonicOscillator(k=1.0, m=1.0)
x0, v0 = 1.0, 0.0  # 초기 위치 1, 정지 상태에서 출발
dt = 0.15          # 시간 간격 (오차를 확인하기 위해 적당히 설정)
t_max = 100.0       # 시뮬레이션 시간

# 시뮬레이션 수행
t_rk4, x_rk4, e_rk4 = solve_rk4(system, x0, v0, dt, t_max)
t_ver, x_ver, e_ver = solve_verlet(system, x0, v0, dt, t_max)

# 시각화
plt.figure(figsize=(12, 8))

# 그래프 1: 위치 비교 (Trajectory)
plt.subplot(2, 1, 1)
plt.plot(t_rk4, x_rk4, label='RK4', linewidth=2, alpha=0.7)
plt.plot(t_ver, x_ver, label='Velocity Verlet', linestyle='--', color='red')
plt.title(f'Position Comparison (dt={dt})')
plt.ylabel('Position (x)')
plt.legend()
plt.grid(True)

# 그래프 2: 에너지 비교 (Energy Conservation)
plt.subplot(2, 1, 2)
# 기준 에너지 (초기 에너지)
E0 = system.get_energy(x0, v0)
plt.axhline(y=E0, color='black', linestyle=':', label='Theoretical energy')

plt.plot(t_rk4, e_rk4, label='RK4 energy')
plt.plot(t_ver, e_ver, label='Verlet energy', color='red')

plt.title('Total energy stability comparison')
plt.xlabel('Time (t)')
plt.ylabel('Total energy (E)')
plt.legend()
plt.grid(True)
# 에너지 변화를 확대해서 보기 위해 y축 범위 조정 (선택 사항)
# plt.ylim(E0 * 0.99, E0 * 1.01)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulation_comparison():
    # 1. 설정 (Configuration)
    dt = 0.1          # 시간 간격 (크게 잡아야 오차를 빨리 볼 수 있음)
    steps = 1000      # 총 시뮬레이션 횟수
    
    # 초기 조건 (위치=1, 속도=0)
    x0, v0 = 1.0, 0.0
    
    # 데이터 저장을 위한 배열 생성
    # [일반 오일러용]
    t_euler = np.zeros(steps)
    x_euler = np.zeros(steps)
    v_euler = np.zeros(steps)
    E_euler = np.zeros(steps) # 에너지
    
    # [심플렉틱 오일러용]
    t_sym = np.zeros(steps)
    x_sym = np.zeros(steps)
    v_sym = np.zeros(steps)
    E_sym = np.zeros(steps)   # 에너지

    # 초기값 세팅
    x_euler[0] = x_sym[0] = x0
    v_euler[0] = v_sym[0] = v0
    E_euler[0] = E_sym[0] = 0.5 * v0**2 + 0.5 * x0**2

    # 2. 시뮬레이션 루프
    for i in range(steps - 1):
        t_euler[i+1] = t_euler[i] + dt
        t_sym[i+1]   = t_sym[i] + dt
        
        # --- A. 일반 오일러 (Standard Euler) ---
        # 위치와 속도를 '현재' 값을 기준으로 동시에 업데이트
        # x_new = x_old + v_old * dt
        # v_new = v_old + a_old * dt
        a_current = -x_euler[i] # 가속도 (F = -kx, k=1, m=1)
        x_euler[i+1] = x_euler[i] + v_euler[i] * dt
        v_euler[i+1] = v_euler[i] + a_current * dt
        
        # 에너지 계산 (E = 0.5v^2 + 0.5x^2)
        E_euler[i+1] = 0.5 * v_euler[i+1]**2 + 0.5 * x_euler[i+1]**2

        # --- B. 심플렉틱 오일러 (Symplectic Euler) ---
        # 속도를 먼저 업데이트하고, 그 '새로운 속도'로 위치를 업데이트
        # v_new = v_old + a_old * dt
        # x_new = x_old + v_new * dt  <-- 핵심 차이!
        a_current_sym = -x_sym[i]
        v_sym[i+1] = v_sym[i] + a_current_sym * dt      # 속도 먼저 갱신
        x_sym[i+1] = x_sym[i] + v_sym[i+1] * dt         # 갱신된 속도 사용
        
        # 에너지 계산
        E_sym[i+1] = 0.5 * v_sym[i+1]**2 + 0.5 * x_sym[i+1]**2

    # 3. 결과 시각화 (Plotting)
    plt.figure(figsize=(12, 5))

    # 그래프 1: 위상 공간 (Phase Space) - 궤도 비교
    plt.subplot(1, 2, 1)
    plt.plot(x_euler, v_euler, label='Standard euler (Drift)', color='red', alpha=0.6)
    plt.plot(x_sym, v_sym, label='Symplectic euler (Stable)', color='blue')
    plt.title('Phase space trajectory (Orbit)')
    plt.xlabel('Position (x)')
    plt.ylabel('Velocity (v)')
    plt.legend()
    plt.grid(True)
    plt.axis('equal') # 원형 궤도를 찌그러지지 않게

    # 그래프 2: 에너지 변화 비교
    plt.subplot(1, 2, 2)
    plt.plot(t_euler, E_euler, label='Standard euler', color='red')
    plt.plot(t_sym, E_sym, label='Symplectic euler', color='blue')
    plt.title('Total energy over time')
    plt.xlabel('Time')
    plt.ylabel('Energy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    simulation_comparison()

In [ ]:
import numpy as np

# 1. 미분 방정식 정의
def f(t, y):
    """dy/dt = -2*t*y"""
    # y는 항상 1차원 배열이라고 가정하고 계산
    return -2.0 * t * y

# 2. 수정된 중점 방법 (Modified Midpoint Method)
def modified_midpoint(f, t0, y0_array, H, n):
    """
    구간 H에 대해 n개의 부분 단계를 사용하여 해를 근사하는 함수.
    """
    h = H / n
    t = t0
    y = y0_array.copy()
    # (1) 초기 단계 (Euler Step)
    y_old = y.copy()
    y = y_old + h * f(t, y_old)
    # (2) 재귀 단계 (Midpoint Steps)
    for i in range(1, n):
        t += h
        y_next = y_old + 2.0 * h * f(t, y)
        y_old = y
        y = y_next
    # (3) 마지막 단계 (Smoothing Step)
    t += h # t = t0 + H
    T_n = 0.5 * (y + y_old + h * f(t, y))
    return T_n

# 3. Richardson/Neville 다항식 보외법
def neville_extrapolation(T_values, n_steps):
    """
    T_values (배열 리스트)를 사용하여 다항식 외삽법을 적용한다.
    T_values는 NumPy 배열의 리스트이다.
    """
    m = len(T_values)
    # T는 외삽법 테이블. 모든 요소를 0으로 초기화하고, T_values를 복사
    # T[i, j]는 (m, m, len(y)) 차원의 3차원 배열이 된다.
    T = np.zeros((m, m, T_values[0].size))
    # 0차 (기본 T_n 값 복사)
    for i in range(m):
        T[i, 0] = T_values[i]
    # 1차 이상의 보외법
    for j in range(1, m): # 외삽 차수 (j = 1, 2, ...)
        for i in range(j, m): # 데이터 포인트 (i = j, j+1, ...) 
            # (h_{i-j} / h_i)^2 = (n_i / n_{i-j})^2
            ratio_sq = (n_steps[i] / n_steps[i-j])**2
            # Neville/Richardson 다항식 외삽 공식
            # T_{i, j} = T_{i, j-1} + (T_{i, j-1} - T_{i-1, j-1}) / ( (h_{i-j}/h_i)^2 - 1 )
            # **주의**: T[i-1, j-1] 대신 T[i-1, j]가 들어가야 함 (Neville 테이블 규칙)
            # T[i, j] = T[i, j-1] + (T[i, j-1] - T[i-1, j-1]) / (ratio_sq - 1.0)
            # Bulirsch-Stoer의 Rational Function Extrapolation과 유사하게 
            # 일반적인 Neville 다항식 보간 공식을 사용하겠다:
            T[i, j] = T[i, j-1] + (T[i, j-1] - T[i-1, j-1]) / (ratio_sq - 1.0)
            # NOTE: 이 공식은 T[i-1, j-1]을 사용하므로 인덱싱 오류를 피할 수 있다.
            
    # 가장 정확한 추정값은 오른쪽 아래 값 (T[m-1, m-1]은 배열)
    return T[m-1, m-1], T

# 4. Bulirsch-Stoer 메인 루틴
def bulirsch_stoer_step(f, t0, y0_array, H):
    # n의 시퀀스
    n_sequence = [2, 4, 6, 8, 12, 16] 
    T_values = []
    # 1. 수정된 중점 방법으로 T_n 값들을 계산
    for n in n_sequence:
        T_n = modified_midpoint(f, t0, y0_array, H, n)
        T_values.append(T_n)
    # 2. 외삽법 적용
    final_result, T_table = neville_extrapolation(T_values, n_sequence)
    return final_result, T_table, n_sequence

# 5. 실행
t_start = 0.0
t_end = 1.0
H_total = t_end - t_start
# 초기 조건을 **반드시** 1차원 NumPy 배열로 정의
y_initial = np.array([1.0]) 
# Bulirsch-Stoer 계산
y_final_bs, T_table, n_sequence = bulirsch_stoer_step(f, t_start, y_initial, H_total)
# 해석적 해
y_analytical = np.array([np.exp(-(t_end**2))])
print(f"---  Bulirsch-Stoer 알고리즘 (t={t_start}에서 t={t_end}까지) ---")
print(f"**해석적 해 y({t_end})**: {y_analytical[0]:.12f}")
print(f"**Bulirsch-Stoer 해 y({t_end})**: {y_final_bs[0]:.12f}")
print(f"**절대 오차**: {np.abs(y_final_bs[0] - y_analytical[0]):.2e}")
print("\n--- 외삽법 테이블 (T[i, j]) ---")
# T_table은 (m, m, 1) 차원의 3차원 배열이다.
for i, n in enumerate(n_sequence):
    row = f"n={n:2}: "
    for j in range(i + 1):
        # T_table[i, j]는 NumPy 배열이므로 [0]으로 스칼라 값을 추출
        row += f"{T_table[i, j, 0]:.12f}  "
    print(row)

In [ ]:
import numpy as np

# 1. ODE 정의: y' = f(t, y)
# 예제: y' = y
def f(t, y):
    return y

# 2. 수정된 중점 공식 (Modified Midpoint Rule, MM)
# MM은 특정 시간 간격 h 내에서 'n'개의 하위 단계를 사용하여 적분한다.
def modified_midpoint(f, t0, y0, H, n):
    """
    f: ODE 함수 (y' = f(t, y))
    t0, y0: 초기 조건
    H: 총 적분 시간 간격 (h = H/n)
    n: MM 단계의 하위 스텝 수
    """
    h = H / n  # 하위 스텝 크기
    t = t0
    y = y0
    
    # 첫 번째 스텝: Euler 방식으로 시작
    y1 = y0 + h * f(t, y0)
    t += h

    # n-1 스텝: 중점 공식 반복
    for i in range(1, n):
        t_next = t + h
        # 중점 공식: y_k+1 = y_k-1 + 2 * h * f(t_k, y_k)
        y_next = y0 + 2 * h * f(t, y1) 
        
        y0 = y1 # 이전 스텝의 y
        y1 = y_next # 현재 스텝의 y
        t = t_next

    # 마지막 스텝: 평균을 사용하여 정확도 향상
    # y_final = 0.5 * (y_penultimate + y_ultimate) + 0.5 * h * f(t_ultimate, y_ultimate)
    # y1은 y_ultimate, y0은 y_penultimate
    y_final = 0.5 * (y0 + y1) + 0.5 * h * f(t, y1) 
    
    return y_final

# 3. Bulirsch-Stoer 외삽(Extrapolation) 함수
def bulirsch_stoer_extrapolate(results, n_values):
    """
    MM 결과 배열과 해당 n 값 배열을 사용하여 외삽 테이블을 생성한다.
    """
    N = len(results)
    T = np.zeros((N, N)) # 외삽 테이블
    
    for i in range(N):
        T[i, 0] = results[i] # 첫 번째 열은 MM 결과
        
    for j in range(1, N):
        for i in range(N - j):
            # Q = (n_i+1^2 / n_i^2)
            Q = (n_values[i+j] / n_values[i])**2
            
            # 리처드슨 외삽 공식: T(i, j) = T(i+1, j-1) + [T(i+1, j-1) - T(i, j-1)] / (Q - 1)
            T[i, j] = T[i+1, j-1] + (T[i+1, j-1] - T[i, j-1]) / (Q - 1)
            
    # 외삽된 최종 결과 (테이블의 첫 행 마지막 열)
    return T[0, N-1], T

# 초기 조건 및 적분 범위 설정
t_start = 0.0
y_start = 1.0
t_end = 1.0
H = t_end - t_start # 전체 스텝 크기

# MM에 사용할 하위 스텝 수 (n 값은 증가하는 수열, 일반적으로 2의 배수 사용)
n_values = [2, 4, 6, 8, 10]
# n_values = [2, 4, 8, 16, 32] # 2의 거듭제곱이 더 일반적이지만, 이해를 위해 간단한 수열 사용

mm_results = []

# (1) 다양한 n 값으로 수정된 중점 공식(MM) 실행
print(f"##  MM 적분 결과 (H={H}, f(t,y)=y)")
for n in n_values:
    y_mm = modified_midpoint(f, t_start, y_start, H, n)
    mm_results.append(y_mm)
    print(f"n={n}: y({t_end}) ≈ {y_mm:.10f}")
    
# (2) 리처드슨 외삽법 적용
final_result, table = bulirsch_stoer_extrapolate(mm_results, n_values)

# (3) 정확한 해와의 비교
true_solution = np.exp(t_end)

print("\n---")
print(f"##  Bulirsch-Stoer 외삽 최종 결과")
print(f"외삽된 최종 해: y({t_end}) ≈ {final_result:.10f}")
print(f"정확한 해 (e^1): {true_solution:.10f}")
print(f"오차: |{final_result - true_solution:.2e}|")

In [ ]:
import numpy as np
from scipy.interpolate import BarycentricInterpolator

# 1. 미분 방정식 정의: dy/dt = f(t, y)
def f(t, y):
    """
    예제 ODE: y' = -y
    """
    return -y

# 2. 수정된 중점 방법 (Modified Midpoint Method)
def modified_midpoint(f, t0, y0, H, n):
    """
    구간 [t0, t0 + H]에 대해 n개의 보조 단계로 수정된 중점법을 적용한다.
    H: 전체 단계 크기
    n: 보조 단계의 수
    h: 보조 단계 크기 (H / n)
    """
    h = H / n
    y = np.copy(y0)
    
    # 첫 번째 단계는 오일러법
    y_next = y + h * f(t0, y)
    
    # 이후의 단계는 중점법
    for i in range(1, n):
        t = t0 + i * h
        
        # 중점법 단계: y_{i+1} = y_{i-1} + 2h * f(t, y_i)
        y_temp = y + 2 * h * f(t, y_next)
        y = y_next
        y_next = y_temp
        
    # 마지막 단계: (y_{n} + y_{n-1} + h * f(t_n, y_n)) / 2
    # 여기서 y는 y_{n-1}, y_next는 y_n 이다.
    t_final = t0 + H
    y_final = 0.5 * (y + y_next + h * f(t_final, y_next))
    
    return y_final

# 3. 불리르슈-슈토어 알고리즘의 보외(Extrapolation) 부분
def bulirsch_stoer_step(f, t0, y0, H, n_values):
    """
    여러 보조 단계 수(n)로 수정된 중점법을 실행하고 외삽한다.
    n_values: 사용할 보조 단계의 수 리스트 (짝수여야 함)
    """
    
    # (H/n)^2 값의 리스트
    h_squared_values = [] 
    # n에 따른 수정된 중점법 결과 y_n(H) 값의 리스트
    y_n_H_values = []   
    
    print(f"--- 보조 단계 수에 따른 결과 --- (H={H:.2f})")
    
    for n in n_values:
        if n % 2 != 0:
            print(f"경고: n={n}은 짝수가 아니다. 건너뛴다.")
            continue
            
        y_H = modified_midpoint(f, t0, y0, H, n)
        
        h = H / n
        h_squared = h**2
        
        h_squared_values.append(h_squared)
        y_n_H_values.append(y_H)
        
        print(f"n={n}, 보조 단계 크기 h={h:.4f}, (h/H)^2 = {h_squared:.6f}, 결과 y_n(H)={y_H:.8f}")

    # 리처드슨 보외법 (유리 함수 보외 대신 Barycentric Interpolator 사용)
    # Barycentric Interpolator는 주어진 데이터를 통과하는 다항식을 생성한다.
    # 불리르슈-슈토어는 유리 함수 보외를 사용하지만, 여기서는 NumPy 기반의 보간법을 사용한다.
    interpolator = BarycentricInterpolator(h_squared_values, y_n_H_values)
    
    # h^2 = 0 인 지점으로 보외 (가장 정확한 추정값)
    y_extrapolated = interpolator(0.0)
    
    return y_extrapolated, y_n_H_values[-1] # 보외값과 가장 작은 h에서의 결과

# --- 실행 ---
t_start = 0.0
y_start = 1.0  # 초기 조건 y(0) = 1
H_step = 0.5   # 전체 적분 단계 크기 H
n_test_values = [2, 4, 6, 8] # 테스트할 보조 단계의 수

print(f"### 불리르슈-슈토어 기반 수치 적분 ###")
print(f"ODE: y' = -y, 초기 조건 y({t_start}) = {y_start}")
print(f"정확한 해: y({t_start + H_step}) = {y_start * np.exp(-H_step):.8f}\n")

y_extrap, y_last_h = bulirsch_stoer_step(f, t_start, y_start, H_step, n_test_values)

# --- 결과 출력 ---
print("\n--- 최종 결과 비교 ---")
print(f"정확한 해 y({t_start + H_step:.2f}):    {y_start * np.exp(-H_step):.8f}")
print(f"가장 작은 h에서의 중점법 결과: {y_last_h:.8f}")
print(f"**외삽를 통한 추정값:** **{y_extrap:.8f}**")

error_h = abs(y_start * np.exp(-H_step) - y_last_h)
error_extrap = abs(y_start * np.exp(-H_step) - y_extrap)

print(f"\n중점법 오차: {error_h:.2e}")
print(f"외삽법 오차: {error_extrap:.2e} (외삽를 통해 오차가 크게 감소했음을 확인)")

In [ ]:
import numpy as np

# 1. 미분 방정식 정의
def f(t, y):
    """dy/dt = -2*t*y"""
    # y는 항상 1차원 배열이라고 가정하고 계산
    return -2.0 * t * y

# 2. 수정된 중점 방법 (Modified Midpoint Method)
def modified_midpoint(f, t0, y0_array, H, n):
    """
    구간 H에 대해 n개의 부분 단계를 사용하여 해를 근사하는 함수.
    """
    h = H / n
    t = t0
    y = y0_array.copy()

    # (1) 초기 단계 (Euler Step)
    y_old = y.copy()
    y = y_old + h * f(t, y_old)

    # (2) 재귀 단계 (Midpoint Steps)
    for i in range(1, n):
        t += h
        y_next = y_old + 2.0 * h * f(t, y)
        y_old = y
        y = y_next

    # (3) 마지막 단계 (Smoothing Step)
    t += h # t = t0 + H
    T_n = 0.5 * (y + y_old + h * f(t, y))
    
    return T_n

# 3. Richardson/Neville 다항식 보외법
def neville_extrapolation(T_values, n_steps):
    """
    T_values (배열 리스트)를 사용하여 다항식 외삽법을 적용한다.
    T_values는 NumPy 배열의 리스트이다.
    """
    m = len(T_values)
    # T는 외삽법 테이블. 모든 요소를 0으로 초기화하고, T_values를 복사
    # T[i, j]는 (m, m, len(y)) 차원의 3차원 배열이 된다.
    T = np.zeros((m, m, T_values[0].size))
    
    # 0차 (기본 T_n 값 복사)
    for i in range(m):
        T[i, 0] = T_values[i]
    
    # 1차 이상의 보외법
    for j in range(1, m): # 외삽 차수 (j = 1, 2, ...)
        for i in range(j, m): # 데이터 포인트 (i = j, j+1, ...)
            
            # (h_{i-j} / h_i)^2 = (n_i / n_{i-j})^2
            ratio_sq = (n_steps[i] / n_steps[i-j])**2
            
            # Neville/Richardson 다항식 외삽 공식
            # T_{i, j} = T_{i, j-1} + (T_{i, j-1} - T_{i-1, j-1}) / ( (h_{i-j}/h_i)^2 - 1 )
            
            # **주의**: T[i-1, j-1] 대신 T[i-1, j]가 들어가야 함 (Neville 테이블 규칙)
            # T[i, j] = T[i, j-1] + (T[i, j-1] - T[i-1, j-1]) / (ratio_sq - 1.0)
            
            # Bulirsch-Stoer의 Rational Function Extrapolation과 유사하게 
            # 일반적인 Neville 다항식 보간 공식을 사용하겠다:
            T[i, j] = T[i, j-1] + (T[i, j-1] - T[i-1, j-1]) / (ratio_sq - 1.0)

            # NOTE: 이 공식은 T[i-1, j-1]을 사용하므로 인덱싱 오류를 피할 수 있다.
            
    # 가장 정확한 추정값은 오른쪽 아래 값 (T[m-1, m-1]은 배열)
    return T[m-1, m-1], T

# 4. Bulirsch-Stoer 메인 루틴
def bulirsch_stoer_step(f, t0, y0_array, H):
    # n의 시퀀스
    n_sequence = [2, 4, 6, 8, 12, 16] 
    
    T_values = []
    
    # 1. 수정된 중점 방법으로 T_n 값들을 계산
    for n in n_sequence:
        T_n = modified_midpoint(f, t0, y0_array, H, n)
        T_values.append(T_n)
    
    # 2. 외삽법 적용
    final_result, T_table = neville_extrapolation(T_values, n_sequence)
    
    return final_result, T_table, n_sequence

# 5. 실행
t_start = 0.0
t_end = 1.0
H_total = t_end - t_start
# 초기 조건을 **반드시** 1차원 NumPy 배열로 정의
y_initial = np.array([1.0]) 

# Bulirsch-Stoer 계산
y_final_bs, T_table, n_sequence = bulirsch_stoer_step(f, t_start, y_initial, H_total)

# 해석적 해
y_analytical = np.array([np.exp(-(t_end**2))])

print(f"---  Bulirsch-Stoer 알고리즘 (t={t_start}에서 t={t_end}까지) ---")
print(f"**해석적 해 y({t_end})**: {y_analytical[0]:.12f}")
print(f"**Bulirsch-Stoer 해 y({t_end})**: {y_final_bs[0]:.12f}")
print(f"**절대 오차**: {np.abs(y_final_bs[0] - y_analytical[0]):.2e}")

print("\n--- 외삽법 테이블 (T[i, j]) ---")
# T_table은 (m, m, 1) 차원의 3차원 배열이다.
for i, n in enumerate(n_sequence):
    row = f"n={n:2}: "
    for j in range(i + 1):
        # T_table[i, j]는 NumPy 배열이므로 [0]으로 스칼라 값을 추출
        row += f"{T_table[i, j, 0]:.12f}  "
    print(row)

In [ ]:
import numpy as np
import time

# --- ODE 정의 ---
def f(t, y):
    """테스트 ODE: y' = y"""
    return y

# --- 정확한 해 ---
def true_solution(t):
    """정확한 해: y(t) = e^t"""
    return np.exp(t)

# --- RK4 함수 (ABM 초기값 계산용) ---
def rk4_step(f, t_n, y_n, h):
    k1 = h * f(t_n, y_n)
    k2 = h * f(t_n + 0.5 * h, y_n + 0.5 * k1)
    k3 = h * f(t_n + 0.5 * h, y_n + 0.5 * k2)
    k4 = h * f(t_n + h, y_n + k3)
    return y_n + (k1 + 2*k2 + 2*k3 + k4) / 6

# --- 1. 아담스-배쉬포스-몰튼 4차 (ABM4) ---
def adams_bashforth_moulton_4th_order(f, t_start, y_start, t_end, N):
    h = (t_end - t_start) / N
    t = np.linspace(t_start, t_end, N + 1)
    y = np.zeros(N + 1)
    y[0] = y_start
    
    # RK4로 초기 3개 값 계산
    for i in range(3):
        y[i+1] = rk4_step(f, t[i], y[i], h)
    
    f_values = [f(t[i], y[i]) for i in range(4)]

    for i in range(3, N):
        # 예측 (AB4)
        y_star = y[i] + h / 24 * (
            55 * f_values[3] - 59 * f_values[2] + 
            37 * f_values[1] -  9 * f_values[0]
        )
        f_star_next = f(t[i+1], y_star)

        # 수정 (AM4)
        y[i+1] = y[i] + h / 24 * (
             9 * f_star_next + 19 * f_values[3] - 
             5 * f_values[2] +  1 * f_values[1] 
        )
        
        # f_values 업데이트
        f_values.pop(0) 
        f_values.append(f(t[i+1], y[i+1]))
        
    return y[-1]

# --- 2. 룽게-쿠타 4차 (RK4) ---
def runge_kutta_4th_order(f, t_start, y_start, t_end, N):
    h = (t_end - t_start) / N
    y = y_start
    t = t_start
    
    for _ in range(N):
        k1 = h * f(t, y)
        k2 = h * f(t + 0.5 * h, y + 0.5 * k1)
        k3 = h * f(t + 0.5 * h, y + 0.5 * k2)
        k4 = h * f(t + h, y + k3)
        y = y + (k1 + 2*k2 + 2*k3 + k4) / 6
        t = t + h
        
    return y

# --- 3. 오일러 방법 (Euler's Method) ---
def euler_method(f, t_start, y_start, t_end, N):
    h = (t_end - t_start) / N
    y = y_start
    t = t_start
    
    for _ in range(N):
        y = y + h * f(t, y)
        t = t + h
        
    return y

# --- 4. 수정된 오일러 방법 (Modified Euler / Midpoint Method) ---
def modified_euler_method(f, t_start, y_start, t_end, N):
    h = (t_end - t_start) / N
    y = y_start
    t = t_start
    
    for _ in range(N):
        k1 = f(t, y)
        k2 = f(t + 0.5 * h, y + 0.5 * h * k1)
        y = y + h * k2
        t = t + h
        
    return y

# --- 5. 호인 방법 (Heun's Method / Improved Euler) ---
def heuns_method(f, t_start, y_start, t_end, N):
    h = (t_end - t_start) / N
    y = y_start
    t = t_start
    
    for _ in range(N):
        f_n = f(t, y)
        y_star = y + h * f_n
        f_star_next = f(t + h, y_star)
        y = y + 0.5 * h * (f_n + f_star_next)
        t = t + h
        
    return y

# --- 비교할 함수 리스트 ---
methods = {
    "Euler (1차)": (euler_method, 1),
    "Mod. Euler (2차)": (modified_euler_method, 2),
    "Heun (2차)": (heuns_method, 2),
    "RK4 (4차)": (runge_kutta_4th_order, 4),
    "ABM4 (4차)": (adams_bashforth_moulton_4th_order, 4),
}

# --- 문제 설정 ---
T_START = 0.0
Y_START = 1.0
T_END = 1.0
N_STEPS = 1000  # 적분 스텝 수 (정확도와 계산 시간을 동시에 비교하기 위함)
TRUE_VALUE = true_solution(T_END)

results = []
print(f"##  ODE 수치 적분 방법 성능 비교 (N={N_STEPS} 스텝)")
print("-" * 60)
print(f"| {'방법':<15} | {'차수':<3} | {'최종 해':<15} | {'절대 오차':<15} | {'계산 시간 (s)':<15} |")
print("|" + "-"*17 + "|" + "-"*5 + "|" + "-"*17 + "|" + "-"*17 + "|" + "-"*17 + "|")

for name, (method_func, order) in methods.items():
    start_time = time.perf_counter()
    
    # 적분 실행
    final_y = method_func(f, T_START, Y_START, T_END, N_STEPS)
    
    end_time = time.perf_counter()
    
    # 결과 계산
    error = np.abs(final_y - TRUE_VALUE)
    elapsed_time = end_time - start_time
    
    results.append({
        "name": name,
        "order": order,
        "final_y": final_y,
        "error": error,
        "time": elapsed_time
    })
    
    # 결과 출력
    print(f"| {name:<15} | {order:<3} | {final_y:.10f} | {error:.2e} | {elapsed_time:.6f} |")

print("-" * 60)
print(f"| {'정확한 해':<15} | {'-':<3} | {TRUE_VALUE:.10f} | {'-':<15} | {'-':<15} |")
print("-" * 60)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. 문제 정의
# ---------------------------------------------------------
def f(t, y):
    """ 도함수 y' = f(t, y) """
    return -y + t + 1
def exact_solution(t):
    """ 실제 정답 y(t) """
    return t + np.exp(-t)
# ---------------------------------------------------------
# 2. RK4 (초기 시동용)
# ---------------------------------------------------------
def rk4_step(t, y, h):
    k1 = h * f(t, y)
    k2 = h * f(t + 0.5*h, y + 0.5*k1)
    k3 = h * f(t + 0.5*h, y + 0.5*k2)
    k4 = h * f(t + h, y + k3)
    return y + (k1 + 2*k2 + 2*k3 + k4) / 6.0
# ---------------------------------------------------------
# 3. AB-AM Predictor-Corrector Solver
# ---------------------------------------------------------
def solve_ab_am_4th_order(t0, y0, h, num_steps):
    # 시간 배열 생성
    t = np.linspace(t0, t0 + num_steps*h, num_steps + 1)
    y = np.zeros(num_steps + 1)
    y[0] = y0
    # [Step 1] RK4로 초기 3스텝(y1, y2, y3) 계산 (시동 걸기)
    print(">>> 초기 3스텝은 RK4로 시동을 건다.")
    for i in range(3):
        y[i+1] = rk4_step(t[i], y[i], h)
    # [Step 2] AB-AM 루프 시작 (i는 3부터 시작)
    # AB4 예측자 계수: [55, -59, 37, -9] / 24
    # AM4 수정자 계수: [9, 19, -5, 1] / 24
    print(">>> AB-AM 4차 예측-수정 알고리즘 가동 중...")
    for i in range(3, num_steps):
        # 미리 계산된 기울기들 (History)
        f_n   = f(t[i],   y[i])
        f_n1  = f(t[i-1], y[i-1])
        f_n2  = f(t[i-2], y[i-2])
        f_n3  = f(t[i-3], y[i-3])
        # -----------------------------------------------
        # (A) Predictor (Adams-Bashforth 4-step)
        # -----------------------------------------------
        y_pred = y[i] + (h/24.0) * (55*f_n - 59*f_n1 + 37*f_n2 - 9*f_n3)
        # -----------------------------------------------
        # (B) Evaluator
        # -----------------------------------------------
        # 예측된 미래 시점(t[i+1])에서의 기울기 계산
        f_next_pred = f(t[i+1], y_pred)
        # -----------------------------------------------
        # (C) Corrector (Adams-Moulton 3-step implicit)
        # -----------------------------------------------
        # AM4 공식: y_{n+1} = y_n + h/24 * (9*f_{n+1} + 19*f_n - 5*f_{n-1} + f_{n-2})
        y[i+1] = y[i] + (h/24.0) * (9*f_next_pred + 19*f_n - 5*f_n1 + f_n2)
    return t, y
# ---------------------------------------------------------
# 4. 실행 및 결과 비교
# ---------------------------------------------------------
# 설정: 0초부터 10초까지, 간격 h=0.1
t0 = 0.0
y0 = 1.0 # y(0) = 0 + e^0 = 1
h = 0.2  # 스텝 사이즈 (일부러 오차를 보기 위해 조금 크게 설정)
steps = 50
# 계산 수행
t_vals, y_approx = solve_ab_am_4th_order(t0, y0, h, steps)
# 실제 정답 계산
y_exact = exact_solution(t_vals)
# 오차 계산
error = np.abs(y_exact - y_approx)
# ---------------------------------------------------------
# 5. 시각화
# ---------------------------------------------------------
plt.figure(figsize=(12, 5))
# (1) 해 궤적 비교
plt.subplot(1, 2, 1)
plt.plot(t_vals, y_exact, 'k-', linewidth=2, label='Exact solution')
plt.plot(t_vals, y_approx, 'r--', marker='o', markersize=4, label='AB-AM (4th order)')
plt.title(f"Predictor-Corrector solution (h={h})")
plt.xlabel("t", fontsize=18)
plt.ylabel("y(t)", fontsize=18)
plt.legend()
plt.grid(True)
# (2) 오차 그래프
plt.subplot(1, 2, 2)
plt.plot(t_vals, error, 'b.-')
plt.title("Absolute error")
plt.xlabel("t", fontsize=18)
plt.ylabel("|Exact - Approx|", fontsize=18)
plt.yscale('log') # 로그 스케일로 오차 확인
plt.grid(True)
plt.tight_layout()
plt.savefig('rk4pc.png')
plt.show()
# ---------------------------------------------------------
# 결과 텍스트 출력
# ---------------------------------------------------------
print("\n" + "="*40)
print(f"최종 시간 t = {t_vals[-1]:.1f}")
print(f"실제값 : {y_exact[-1]:.8f}")
print(f"근사값 : {y_approx[-1]:.8f}")
print(f"오  차 : {error[-1]:.4e}")
print("="*40)

In [ ]:
import numpy as np
import scipy.sparse as sparse
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
def solve_3d_poisson_fdm_verified():
    print("3D Poisson Solver (FDM) with Verification 시작...")
    # ---------------------------------------------------------
    # 1. 문제 설정
    # ---------------------------------------------------------
    N = 30  # 격자 해상도 (30^3 = 27,000 DOFs)
    L = 1.0
    h = L / (N - 1)
    # 격자 생성
    x = np.linspace(0, L, N)
    y = np.linspace(0, L, N)
    z = np.linspace(0, L, N)
    # indexing='ij': 행렬 좌표계 (i, j, k) 순서 유지
    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
    # ---------------------------------------------------------
    # 2. 정답(Exact Solution) 미리 계산
    # ---------------------------------------------------------
    # 정답 가정: u = sin(pi*x)sin(pi*y)sin(pi*z)
    u_exact = np.sin(np.pi * X) * np.sin(np.pi * Y) * np.sin(np.pi * Z)
    # ---------------------------------------------------------
    # 3. 희소 행렬(Laplacian) 조립
    # ---------------------------------------------------------
    # 1D Laplacian Operator (Tridiagonal: 1, -2, 1)
    e = np.ones(N)
    diags = [e, -2*e, e]
    offsets = [-1, 0, 1]
    D1 = sparse.spdiags(diags, offsets, N, N)
    I1 = sparse.eye(N)
    # 3D Laplacian via Kronecker Sum
    # L3 = D_xx + D_yy + D_zz
    L3 = sparse.kron(sparse.kron(D1, I1), I1) + \
         sparse.kron(sparse.kron(I1, D1), I1) + \
         sparse.kron(sparse.kron(I1, I1), D1)
    L3 = L3 / h**2 # 스케일링
    # ---------------------------------------------------------
    # 4. 우변 벡터 및 경계 조건 설정
    # ---------------------------------------------------------
    # 방정식: -Laplacian u = f
    # u_exact를 대입해서 f를 구함: f = 3 * pi^2 * u
    f = 3 * (np.pi**2) * u_exact
    b = -f.flatten() # L3 * u = -f
    # 경계 조건 (Dirichlet u=0)
    # 경계 노드 인덱스 추출
    indices = np.arange(N**3).reshape((N, N, N))
    mask_boundary = np.zeros((N, N, N), dtype=bool)
    mask_boundary[0,:,:] = True; mask_boundary[-1,:,:] = True
    mask_boundary[:,0,:] = True; mask_boundary[:,-1,:] = True
    mask_boundary[:,:,0] = True; mask_boundary[:,:,-1] = True
    boundary_idx = indices[mask_boundary].flatten()
    # 행렬 수정 (Zero-out rows and 1 on diagonal)
    L3 = L3.tolil()
    for idx in boundary_idx:
        L3[idx, :] = 0
        L3[idx, idx] = 1
        b[idx] = 0 # 경계값 0
    L3 = L3.tocsr()
    # ---------------------------------------------------------
    # 5. 선형 시스템 풀이
    # ---------------------------------------------------------
    print(f"선형 시스템 풀이 중... (Unknowns: {N**3})")
    u_vec = spla.spsolve(L3, b)
    u_numer = u_vec.reshape((N, N, N))
    # ---------------------------------------------------------
    # 6. 정확도 검증 (Verification) - 추가된 부분
    # ---------------------------------------------------------
    # 오차 행렬 계산
    error_field = np.abs(u_numer - u_exact)
    # 1) 최대 오차 (L-infinity Norm)
    max_error = np.max(error_field)
    # 2) 평균 제곱근 오차 (Discrete L2 Norm)
    # RMS Error = sqrt( sum(error^2) / N_points )
    l2_error = np.sqrt(np.mean(error_field**2))
    print("-" * 40)
    print(f"검증 결과 (Grid N={N})")
    print("-" * 40)
    print(f"Max Error (L_inf) : {max_error:.6e}")
    print(f"RMS Error (L_2)   : {l2_error:.6e}")
    print("-" * 40)
    # ---------------------------------------------------------
    # 7. 결과 시각화 (비교)
    # ---------------------------------------------------------
    z_slice = N // 2
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    # (1) 수치해 (Numerical Solution)
    im1 = axes[0].contourf(X[:,:,z_slice], Y[:,:,z_slice], u_numer[:,:,z_slice], 
         levels=50, cmap='viridis')
    axes[0].set_title(f"Numerical Solution (FDM)")
    plt.colorbar(im1, ax=axes[0])
    # (2) 정답 (Exact Solution)
    im2 = axes[1].contourf(X[:,:,z_slice], Y[:,:,z_slice], u_exact[:,:,z_slice], 
        levels=50, cmap='viridis')
    axes[1].set_title(f"Exact Solution")
    plt.colorbar(im2, ax=axes[1])
    # (3) 오차 (Absolute Error)
    im3 = axes[2].contourf(X[:,:,z_slice], Y[:,:,z_slice], error_field[:,:,z_slice], 
        levels=50, cmap='inferno')
    axes[2].set_title(f"Absolute Error |Num - Exact|")
    plt.colorbar(im3, ax=axes[2])
    for ax in axes:
        ax.set_xlabel("x")
        ax.set_ylabel("y")
    plt.suptitle(f"3D Poisson equation verif. (Slice at z={z[z_slice]:.2f})",fontsize=16)
    plt.tight_layout()
    plt.show()
if __name__ == "__main__":
    solve_3d_poisson_fdm_verified()

## Chapter 6 편미분방정식(PDE)    파이썬을 활용한 수치해석(Numerical Analysis with Python) 이인호 (북스힐, 2026)

In [ ]:
import numpy as np
import scipy.sparse as sparse
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

def solve_3d_poisson_fdm_verified():
    print("3D Poisson Solver (FDM) with verification 시작...")
    # ---------------------------------------------------------
    # 1. 문제 설정
    # ---------------------------------------------------------
    N = 30  # 격자 해상도 (30^3 = 27,000 DOFs)
    L = 1.0
    h = L / (N - 1)
    # 격자 생성
    x = np.linspace(0, L, N)
    y = np.linspace(0, L, N)
    z = np.linspace(0, L, N)
    # indexing='ij': 행렬 좌표계 (i, j, k) 순서 유지
    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
    # ---------------------------------------------------------
    # 2. 정답(Exact Solution) 미리 계산
    # ---------------------------------------------------------
    # 정답 가정: u = sin(pi*x)sin(pi*y)sin(pi*z)
    u_exact = np.sin(np.pi * X) * np.sin(np.pi * Y) * np.sin(np.pi * Z)
    # ---------------------------------------------------------
    # 3. 희소 행렬(Laplacian) 조립
    # ---------------------------------------------------------
    # 1D Laplacian Operator (Tridiagonal: 1, -2, 1)
    e = np.ones(N)
    diags = [e, -2*e, e]
    offsets = [-1, 0, 1]
    D1 = sparse.spdiags(diags, offsets, N, N)
    I1 = sparse.eye(N)
    # 3D Laplacian via Kronecker Sum
    # L3 = D_xx + D_yy + D_zz
    L3 = sparse.kron(sparse.kron(D1, I1), I1) + \
         sparse.kron(sparse.kron(I1, D1), I1) + \
         sparse.kron(sparse.kron(I1, I1), D1)
    L3 = L3 / h**2 # 스케일링
    # ---------------------------------------------------------
    # 4. 우변 벡터 및 경계 조건 설정
    # ---------------------------------------------------------
    # 방정식: -Laplacian u = f
    # u_exact를 대입해서 f를 구함: f = 3 * pi^2 * u
    f = 3 * (np.pi**2) * u_exact
    b = -f.flatten() # L3 * u = -f
    # 경계 조건 (Dirichlet u=0)
    # 경계 노드 인덱스 추출
    indices = np.arange(N**3).reshape((N, N, N))
    mask_boundary = np.zeros((N, N, N), dtype=bool)
    mask_boundary[0,:,:] = True; mask_boundary[-1,:,:] = True
    mask_boundary[:,0,:] = True; mask_boundary[:,-1,:] = True
    mask_boundary[:,:,0] = True; mask_boundary[:,:,-1] = True
    boundary_idx = indices[mask_boundary].flatten()
    # 행렬 수정 (Zero-out rows and 1 on diagonal)
    L3 = L3.tolil()
    for idx in boundary_idx:
        L3[idx, :] = 0
        L3[idx, idx] = 1
        b[idx] = 0 # 경계값 0
    L3 = L3.tocsr()
    # ---------------------------------------------------------
    # 5. 선형 시스템 풀이
    # ---------------------------------------------------------
    print(f"선형 시스템 풀이 중... (Unknowns: {N**3})")
    u_vec = spla.spsolve(L3, b)
    u_numer = u_vec.reshape((N, N, N))
    # ---------------------------------------------------------
    # 6. 정확도 검증 (Verification) - 추가된 부분
    # ---------------------------------------------------------
    # 오차 행렬 계산
    error_field = np.abs(u_numer - u_exact)
    # 1) 최대 오차 (L-infinity Norm)
    max_error = np.max(error_field)
    # 2) 평균 제곱근 오차 (Discrete L2 Norm)
    # RMS Error = sqrt( sum(error^2) / N_points )
    l2_error = np.sqrt(np.mean(error_field**2))
    print("-" * 40)
    print(f"검증 결과 (Grid N={N})")
    print("-" * 40)
    print(f"Max Error (L_inf) : {max_error:.6e}")
    print(f"RMS Error (L_2)   : {l2_error:.6e}")
    print("-" * 40)
    # ---------------------------------------------------------
    # 7. 결과 시각화 (비교)
    # ---------------------------------------------------------
    z_slice = N // 2
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    # (1) 수치해 (Numerical Solution)
    im1 = axes[0].contourf(X[:,:,z_slice], Y[:,:,z_slice], u_numer[:,:,z_slice], 
                           levels=50, cmap='viridis')
    axes[0].set_title(f"Numerical solution (FDM)")
    plt.colorbar(im1, ax=axes[0])
    # (2) 정답 (Exact Solution)
    im2 = axes[1].contourf(X[:,:,z_slice], Y[:,:,z_slice], u_exact[:,:,z_slice], 
                           levels=50, cmap='viridis')
    axes[1].set_title(f"Exact solution")
    plt.colorbar(im2, ax=axes[1])
    # (3) 오차 (Absolute Error)
    im3 = axes[2].contourf(X[:,:,z_slice], Y[:,:,z_slice], error_field[:,:,z_slice], 
                           levels=50, cmap='inferno')
    axes[2].set_title(f"Absolute Error |Num - Exact|")
    plt.colorbar(im3, ax=axes[2])
    for ax in axes:
        ax.set_xlabel("x")
        ax.set_ylabel("y")
    plt.suptitle(f"3D Poisson equation verification (Slice at z={z[z_slice]:.2f})", fontsize=16)
    plt.tight_layout()
    plt.show()
if __name__ == "__main__":
    solve_3d_poisson_fdm_verified()

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
def solve_3d_harmonic_oscillator_corrected():
    print("Corrected 3D FEM solver 시작...")
    # 1. 설정 (이산화)
    # 영역을 조금 더 넓혔다 (경계 효과 방지)
    L = 4.0         
    N = 65          # 격자 수 
    N = 45          # 격자 수 
    x = np.linspace(-L, L, N)
    h = x[1] - x[0] # 요소 크기 (dx)
    # 2. 1차원 FEM 행렬 생성
    e = np.ones(N)
    # Stiffness (K1): [-1, 2, -1] / h
    # (약한 형식 적분 결과: 1/h 스케일)
    K1 = sp.spdiags([-e, 2*e, -e], [-1, 0, 1], N, N) / h
    # Mass (M1): [1, 4, 1] * h / 6 (Simpson's rule 가중치와 유사)
    # (약한 형식 적분 결과: h 스케일)
    M1 = sp.spdiags([e, 4*e, e], [-1, 0, 1], N, N) * (h / 6)
    # 3. 3차원 행렬 확장 (Kronecker Product)
    # Identity 대신 Mass Matrix를 사용해야 정확한 FEM이 된다.
    # 전체 질량 행렬 M_3d = Mx (x) My (x) Mz
    # 스케일: h * h * h = h^3
    M_3d = sp.kron(sp.kron(M1, M1), M1)
    # 전체 강성 행렬 K_3d (Laplacian)
    # K_3d = Kx(x)My(x)Mz + ...
    # 스케일: (1/h) * h * h = h
    K_x = sp.kron(sp.kron(K1, M1), M1)
    K_y = sp.kron(sp.kron(M1, K1), M1)
    K_z = sp.kron(sp.kron(M1, M1), K1)
    K_3d = K_x + K_y + K_z
    # 4. 퍼텐셜 에너지 행렬 (핵심 수정 부분!)
    # V(x,y,z) 값을 구함
    X, Y, Z = np.meshgrid(x, x, x, indexing='ij')
    V_vals = 0.5 * (X**2 + Y**2 + Z**2)
    V_vec = V_vals.flatten()
    V_mat = sp.diags(V_vec * (h**3))
    # 최종 Hamiltonian (좌변 행렬 A)
    # A = 0.5 * Stiffness + Potential_Integrated
    H_sys = 0.5 * K_3d + V_mat
    # 5. 경계 조건 (Slicing)
    mask = np.zeros((N, N, N), dtype=bool)
    mask[1:-1, 1:-1, 1:-1] = True
    inner_indices = np.where(mask.flatten())[0]
    # 내부 행렬만 추출
    H_inner = H_sys.tocsr()[inner_indices, :][:, inner_indices]
    M_inner = M_3d.tocsr()[inner_indices, :][:, inner_indices]
    # 6. 고유값 풀이 (Generalized Eigenvalue Problem)
    print(f"고유값 계산 중... (Matrix size: {H_inner.shape})")
    # shift-invert mode (sigma=1.5 근처 탐색)
    vals, vecs = spla.eigsh(H_inner, k=5, M=M_inner, sigma=1.4, which='LM')
    # 7. 결과 출력
    print("\n[Calculated eigenvalues vs Theory]")
    print(f"{'State':<6} | {'Calc (FEM)':<12} | {'Theory':<12} | {'Error (%)'}")
    print("-" * 50)
    theoretical_levels = [1.5, 2.5, 2.5, 2.5, 3.5] # 0, (1,0,0), (0,1,0), (0,0,1), ...
    # eigsh 결과는 정렬되지 않을 수 있으므로 정렬
    idx = vals.argsort()
    vals = vals[idx]
    vecs = vecs[:, idx]
    for i in range(5):
        calc = vals[i]
        theo = theoretical_levels[i]
        error = abs(calc - theo) / theo * 100
        print(f"{i:<6} | {calc:.6f}     | {theo:.1f}          | {error:.4f}%")
    # 시각화 (Ground State)
    u_final = np.zeros(N**3)
    u_final[inner_indices] = vecs[:, 0]
    u_3d = u_final.reshape((N, N, N))
    mid = N // 2
    plt.figure(figsize=(8, 6))
    plt.contourf(X[:,:,mid], Y[:,:,mid], u_3d[:,:,mid], levels=50, cmap='inferno')
    plt.colorbar(label='Wavefunction $\psi$')
    plt.title(f"Corrected 3D FEM results (E0 = {vals[0]:.4f})")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.show()
if __name__ == "__main__":
    solve_3d_harmonic_oscillator_corrected()

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

def solve_matrix_free_3d_fem_harmonic():
    print("Matrix-free (Tensor product) 3D FEM Solver 시작...")
    # ---------------------------------------------------------
    # 1. 문제 설정
    # ---------------------------------------------------------
    # 메모리 걱정 없이 해상도를 높일 수 있다.
    N = 40          # 한 축의 격자 수 (전체 자유도 64,000)
    L = 4.0         # 영역 [-4, 4]
    x = np.linspace(-L, L, N)
    h = x[1] - x[0] # 요소 크기
    # ---------------------------------------------------------
    # 2. 1차원 소형 행렬 준비 (N x N)
    # ---------------------------------------------------------
    # 이 작은 행렬들만 메모리에 저장한다.
    e = np.ones(N)
    # 1D Stiffness (K1): 1/h 스케일
    K1 = sp.spdiags([-e, 2*e, -e], [-1, 0, 1], N, N) / h
    # 1D Mass (M1): h 스케일 (Simpson's rule 유사 가중치)
    M1 = sp.spdiags([e, 4*e, e], [-1, 0, 1], N, N) * (h / 6.0)
    # ---------------------------------------------------------
    # 3. Matrix-Free 연산 함수 (핵심!)
    # ---------------------------------------------------------
    # 입력 벡터 u (1D array)를 받아 3D 연산을 수행하고 다시 1D로 반환
    
    def apply_tensor_op_3d(u_vec, op_x, op_y, op_z):
        """
        Compute (op_x (kron) op_y (kron) op_z) * u
        행렬 조립 없이 차원별로 순차적으로 연산 적용
        """
        # 1. 3차원 텐서로 변환 (x, y, z)
        U = u_vec.reshape((N, N, N))
        # 2. z축 연산 적용 (Rightmost operator first)
        # U의 마지막 축(axis 2)에 대해 op_z 곱셈
        # 행렬 곱셈을 위해 (N*N, N)으로 reshape 했다가 복구
        tmp = op_z @ U.reshape(-1, N).T 
        U_z = tmp.T.reshape(N, N, N)
        # 3. y축 연산 적용
        # 축을 바꿔서(transpose) 연산하고 다시 돌려놓음 (swapaxes)
        U_swapped = U_z.transpose(0, 2, 1) # (x, z, y)
        tmp = op_y @ U_swapped.reshape(-1, N).T
        U_y = tmp.T.reshape(N, N, N).transpose(0, 2, 1) # 복구
        # 4. x축 연산 적용
        # (y, z, x) 순서로 보냄
        U_swapped = U_y.transpose(1, 2, 0)
        tmp = op_x @ U_swapped.reshape(-1, N).T
        U_x = tmp.T.reshape(N, N, N).transpose(2, 0, 1) # 복구
        return U_x.ravel()

    # (A) 전체 질량 연산자 M * u
    def matvec_M(u_vec):
        # M = Mx (x) My (x) Mz
        return apply_tensor_op_3d(u_vec, M1, M1, M1)

    # (B) 전체 강성(Laplacian) 연산자 K * u
    def matvec_K(u_vec):
        # K = KMM + MKM + MMK
        term1 = apply_tensor_op_3d(u_vec, K1, M1, M1) # d2/dx2
        term2 = apply_tensor_op_3d(u_vec, M1, K1, M1) # d2/dy2
        term3 = apply_tensor_op_3d(u_vec, M1, M1, K1) # d2/dz2
        return term1 + term2 + term3

    # (C) 포텐셜 에너지 V * u (Lumped Mass Approximation)
    X, Y, Z = np.meshgrid(x, x, x, indexing='ij')
    V_vals = 0.5 * (X**2 + Y**2 + Z**2)
    V_flat = V_vals.ravel()
    # 적분 가중치 (h^3) 포함
    V_scale = h**3 
    
    def matvec_V(u_vec):
        # Diagonal Matrix Free: 그냥 요소별 곱셈(Element-wise multiply)
        return (V_flat * u_vec) * V_scale

    # (D) 최종 Hamiltonian 연산자 H * u
    def matvec_H(u_vec):
        # H = 0.5 * K + V
        return 0.5 * matvec_K(u_vec) + matvec_V(u_vec)

    # ---------------------------------------------------------
    # 4. LinearOperator 정의 및 경계 조건
    # ---------------------------------------------------------
    # 경계 조건(u=0)을 Operator 내부에서 처리하는 것은 복잡하므로,
    # 여기서는 간단히 '내부 노드'만 계산하는 방식을 쓰지 않고
    # 전체를 계산하되, 경계값의 영향이 미미해지는 큰 박스 L=4.0을 사용한다.
    # (엄밀한 처리를 위해서는 Projection Matrix P를 앞뒤로 곱해야 함 P^T H P)
    dim = N**3
    H_op = spla.LinearOperator((dim, dim), matvec=matvec_H)
    M_op = spla.LinearOperator((dim, dim), matvec=matvec_M)
    # ---------------------------------------------------------
    # 5. 고윳값 풀이 (Matrix-Free Eigensolver)
    # ---------------------------------------------------------
    print(f"고윳값 계산 중... (자유도: {dim:,})")
    print("Matrix-free 방식은 Shift-invert를 쓰기 어려워 수렴이 조금 느릴 수 있다.")
    # 'SA': Smallest Algebraic eigenvalues (가장 작은 값 찾기)
    # Shift-Invert(sigma)를 쓰지 않는 이유는 (H - sigma M)^-1 연산을 
    # Matrix-Free로 구현하려면 내부에 또다시 반복법(CG)을 써야 해서 매우 느려지기 때문이다.
    vals, vecs = spla.eigsh(H_op, k=5, M=M_op, which='SA', tol=1e-4)
    # ---------------------------------------------------------
    # 6. 결과 출력
    # ---------------------------------------------------------
    print("\n[Matrix-free FEM 결과]")
    print(f"{'State':<6} | {'Calculated':<12} | {'Theory':<12} | {'Error (%)'}")
    print("-" * 50)
    theoretical_levels = [1.5, 2.5, 2.5, 2.5, 3.5]
    # 정렬
    idx = vals.argsort()
    vals = vals[idx]
    vecs = vecs[:, idx]
    for i in range(5):
        error = abs(vals[i] - theoretical_levels[i]) / theoretical_levels[i] * 100
        print(f"{i:<6} | {vals[i]:.6f}     | {theoretical_levels[i]:.1f}          | {error:.4f}%")
    # 시각화 (중앙 단면)
    u_3d = vecs[:, 0].reshape((N, N, N))
    mid = N // 2
    plt.figure(figsize=(8, 6))
    plt.contourf(X[:,:,mid], Y[:,:,mid], u_3d[:,:,mid], levels=50, cmap='inferno')
    plt.colorbar(label='Wavefunction')
    plt.title(f"Matrix-free FEM (N={N}^3)\nGround state energy: {vals[0]:.4f}")
    plt.show()
if __name__ == "__main__":
    solve_matrix_free_3d_fem_harmonic()

In [ ]:
import numpy as np
from scipy.sparse import diags, kron, identity
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math
# === 상수 설정 ===
N = 71          # 각 차원 격자점 수 (N^3 격자) - 정확도 향상을 위해 41 사용
L = 20.0        # 계산 영역의 길이 ([-L/2, L/2])
h = L / (N - 1) # 격자 간격
omega = 1.0     # 조화 진동자 상수 (omega_x = omega_y = omega_z = 1)
# Fornberg 공식 설정
N_points = 7    # 7점 공식 사용 (O(h^6) 정확도)
M = (N_points - 1) // 2 # 중앙점으로부터의 오프셋
k_deriv = 2     # 2차 도함수
def calculate_fd_weights():
    """
    7점 중앙 차분 공식의 2차 도함수 계수 (h=1 기준)를 반환한다.
    """
    # 7점 공식 계수: [-1/180, 2/45, -1/5, -49/18, -1/5, 2/45, -1/180]
    # NOTE: 부동 소수점 정밀도를 위해 유리수 형태로 입력한다.
    weights = np.array([
    1.0/90.0, 
    -3.0/20.0, 
    3.0/2.0, 
    -49.0/18.0, # 중앙점 계수
    3.0/2.0, 
    -3.0/20.0, 
    1.0/90.0
    ])
    return weights
weights_7_point_h1 = calculate_fd_weights()
# h 팩터 적용: w_i / h^2
weights_7_point = weights_7_point_h1 / h**2 
offsets = np.arange(-M, M + 1) # [-3, -2, -1, 0, 1, 2, 3]
# === 1차원 운동 에너지 행렬 (T_1D) 구성 ===
# T_1D 행렬의 데이터 리스트 (대각선 별로)를 생성한다.
# 각 요소는: -1/2 * (w_i / h^2)
T_1D_data = [
(-0.5 * weights_7_point[i]) * np.ones(N - abs(offsets[i])) 
for i in range(len(offsets))
]
# 희소 행렬 (CSR 형식) 생성
T_1D = diags(T_1D_data, offsets, shape=(N, N), format='csr')
# === 3차원 운동 에너지 행렬 (T_3D) ===
I_N = identity(N, format='csr')
# Kronecker 곱을 이용한 3차원 확장
# T_x 항
T_x = kron(kron(T_1D, I_N), I_N, format='csr')
# T_y 항
T_y = kron(kron(I_N, T_1D), I_N, format='csr')
# T_z 항
T_z = kron(kron(I_N, I_N), T_1D, format='csr')
T_3D = T_x + T_y + T_z
# === 3차원 퍼텐셜 에너지 행렬 (V_3D) ===
x = np.linspace(-L/2, L/2, N)
y = np.linspace(-L/2, L/2, N)
z = np.linspace(-L/2, L/2, N)
# 3D 격자점의 퍼텐셜 값 V(x, y, z) 계산 (V = 1/2 * omega^2 * r^2)
X, Y, Z = np.meshgrid(x, y, z, indexing='ij') 
V_vector = 0.5 * omega**2 * (X**2 + Y**2 + Z**2)
V_vector = V_vector.ravel()
# 퍼텐셜 행렬은 대각 행렬
V_3D = diags(V_vector, 0, format='csr')
# === 전체 해밀토니안 행렬 ===
H = T_3D + V_3D
print(f"3차원 해밀토니안 H 구성 완료. 크기: {H.shape[0]} x {H.shape[1]}")
# === 고유값 문제 풀이 ===
k_states = 5 
# eigsh: 희소 행렬의 고유값 계산 (SA: Smallest Algebraic magnitude)
# N=71은 행렬 크기가 커서 계산 시간이 오래 걸릴 수 있다.
energies, wavefunctions = eigsh(H, k=k_states, which='SA') 
# === 결과 출력 ===
print("\n--- 결과 (N=71, 7점 공식) ---")
print("고유값 (에너지 준위):")
for i, E in enumerate(energies):
    # 이론값 E_n = n + 1.5, 여기서 n = n_x + n_y + n_z
    n_sum = round(E - 1.5)
    E_theoretical = n_sum + 1.5
    error = E - E_theoretical
    print(f"state {i+1} (n_sum={n_sum}): E_FDM = {E:.8f}, E_th = {E_theoretical:.8f}, 오차 = {error:+.2e}")
# === 파동 함수 시각화 (바닥 상태) ===
# 3D 파동 함수를 (N, N, N) 격자로 재구성
Psi_0_3D = wavefunctions[:, 0].reshape((N, N, N))
# 중앙 단면(z=0)을 시각화
mid_index = N // 2
Psi_0_slice = Psi_0_3D[:, :, mid_index]
X_slice, Y_slice = np.meshgrid(x, x, indexing='ij')
plt.figure(figsize=(8, 6))
# 확률 밀도 함수 시각화 (|Psi|^2)
plt.contourf(X_slice, Y_slice, np.abs(Psi_0_slice)**2, 50, cmap='viridis')
plt.colorbar(label='Probability Density $|\psi_0|^2$')
plt.title(f'3D ground state (z=0 )\nE = {energies[0]:.6f}')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.show()
# 파동 함수 표면 플롯 (선택 사항 - 느릴 수 있음)
# fig = plt.figure(figsize=(10, 8))
# ax = fig.add_subplot(111, projection='3d')
# ax.plot_surface(X_slice, Y_slice, np.abs(Psi_0_slice)**2, cmap='viridis')
# ax.set_title('3D 바닥 상태 파동 함수 표면')
# plt.show()

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

def solve_3d_harmonic_oscillator_corrected():
    print("Corrected 3D FEM Solver 시작...")
    # 1. 설정 (이산화)
    # 영역을 조금 더 넓혔다 (경계 효과 방지)
    L = 4.0         
    N = 45          # 격자 수 
    x = np.linspace(-L, L, N)
    h = x[1] - x[0] # 요소 크기 (dx)
    # 2. 1차원 FEM 행렬 생성
    e = np.ones(N)
    # Stiffness (K1): [-1, 2, -1] / h
    # (약한 형식 적분 결과: 1/h 스케일)
    K1 = sp.spdiags([-e, 2*e, -e], [-1, 0, 1], N, N) / h
    # Mass (M1): [1, 4, 1] * h / 6 (Simpson's rule 가중치와 유사)
    # (약한 형식 적분 결과: h 스케일)
    M1 = sp.spdiags([e, 4*e, e], [-1, 0, 1], N, N) * (h / 6)
    # 3. 3차원 행렬 확장 (Kronecker Product)
    # Identity 대신 Mass Matrix를 사용해야 정확한 FEM이 된다.
    # 전체 질량 행렬 M_3d = Mx (x) My (x) Mz
    # 스케일: h * h * h = h^3
    M_3d = sp.kron(sp.kron(M1, M1), M1)
    # 전체 강성 행렬 K_3d (Laplacian)
    # K_3d = Kx(x)My(x)Mz + ...
    # 스케일: (1/h) * h * h = h
    K_x = sp.kron(sp.kron(K1, M1), M1)
    K_y = sp.kron(sp.kron(M1, K1), M1)
    K_z = sp.kron(sp.kron(M1, M1), K1)
    K_3d = K_x + K_y + K_z
    # 4. 포텐셜 에너지 행렬 (핵심 수정 부분!)
    # V(x,y,z) 값을 구함
    X, Y, Z = np.meshgrid(x, x, x, indexing='ij')
    V_vals = 0.5 * (X**2 + Y**2 + Z**2)
    V_vec = V_vals.flatten()
    V_mat = sp.diags(V_vec * (h**3))
    # 최종 Hamiltonian (좌변 행렬 A)
    # A = 0.5 * Stiffness + Potential_Integrated
    H_sys = 0.5 * K_3d + V_mat
    # 5. 경계 조건 (Slicing)
    mask = np.zeros((N, N, N), dtype=bool)
    mask[1:-1, 1:-1, 1:-1] = True
    inner_indices = np.where(mask.flatten())[0]
    # 내부 행렬만 추출
    H_inner = H_sys.tocsr()[inner_indices, :][:, inner_indices]
    M_inner = M_3d.tocsr()[inner_indices, :][:, inner_indices]
    # 6. 고윳값 풀이 (Generalized Eigenvalue Problem)
    print(f"고윳값 계산 중... (Matrix Size: {H_inner.shape})")
    # shift-invert mode (sigma=1.5 근처 탐색)
    vals, vecs = spla.eigsh(H_inner, k=5, M=M_inner, sigma=1.4, which='LM')
    # 7. 결과 출력
    print("\n[Calculated Eigenvalues vs Theory]")
    print(f"{'State':<6} | {'Calc (FEM)':<12} | {'Theory':<12} | {'Error (%)'}")
    print("-" * 50)
    theoretical_levels = [1.5, 2.5, 2.5, 2.5, 3.5] # 0, (1,0,0), (0,1,0), (0,0,1), ...
    # eigsh 결과는 정렬되지 않을 수 있으므로 정렬
    idx = vals.argsort()
    vals = vals[idx]
    vecs = vecs[:, idx]
    for i in range(5):
        calc = vals[i]
        theo = theoretical_levels[i]
        error = abs(calc - theo) / theo * 100
        print(f"{i:<6} | {calc:.6f}     | {theo:.1f}          | {error:.4f}%")
    # 시각화 (Ground State)
    u_final = np.zeros(N**3)
    u_final[inner_indices] = vecs[:, 0]
    u_3d = u_final.reshape((N, N, N))
    mid = N // 2
    plt.figure(figsize=(8, 6))
    plt.contourf(X[:,:,mid], Y[:,:,mid], u_3d[:,:,mid], levels=50, cmap='inferno')
    plt.colorbar(label='Wavefunction $\psi$')
    plt.title(f"Corrected 3D FEM Results (E0 = {vals[0]:.4f})")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.show()
if __name__ == "__main__":
    solve_3d_harmonic_oscillator_corrected()

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

def solve_matrix_free_3d_fem_harmonic():
    print("Matrix-free (Tensor product) 3D FEM Solver 시작...")
    # ---------------------------------------------------------
    # 1. 문제 설정
    # ---------------------------------------------------------
    # 메모리 걱정 없이 해상도를 높일 수 있다.
    N = 40          # 한 축의 격자 수 (전체 자유도 64,000)
    L = 4.0         # 영역 [-4, 4]
    x = np.linspace(-L, L, N)
    h = x[1] - x[0] # 요소 크기
    # ---------------------------------------------------------
    # 2. 1차원 소형 행렬 준비 (N x N)
    # ---------------------------------------------------------
    # 이 작은 행렬들만 메모리에 저장한다.
    e = np.ones(N)
    # 1D Stiffness (K1): 1/h 스케일
    K1 = sp.spdiags([-e, 2*e, -e], [-1, 0, 1], N, N) / h
    # 1D Mass (M1): h 스케일 (Simpson's rule 유사 가중치)
    M1 = sp.spdiags([e, 4*e, e], [-1, 0, 1], N, N) * (h / 6.0)
    # ---------------------------------------------------------
    # 3. Matrix-Free 연산 함수 (핵심!)
    # ---------------------------------------------------------
    # 입력 벡터 u (1D array)를 받아 3D 연산을 수행하고 다시 1D로 반환
    
    def apply_tensor_op_3d(u_vec, op_x, op_y, op_z):
        """
        Compute (op_x (kron) op_y (kron) op_z) * u
        행렬 조립 없이 차원별로 순차적으로 연산 적용
        """
        # 1. 3차원 텐서로 변환 (x, y, z)
        U = u_vec.reshape((N, N, N))
        # 2. z축 연산 적용 (Rightmost operator first)
        # U의 마지막 축(axis 2)에 대해 op_z 곱셈
        # 행렬 곱셈을 위해 (N*N, N)으로 reshape 했다가 복구
        tmp = op_z @ U.reshape(-1, N).T 
        U_z = tmp.T.reshape(N, N, N)
        # 3. y축 연산 적용
        # 축을 바꿔서(transpose) 연산하고 다시 돌려놓음 (swapaxes)
        U_swapped = U_z.transpose(0, 2, 1) # (x, z, y)
        tmp = op_y @ U_swapped.reshape(-1, N).T
        U_y = tmp.T.reshape(N, N, N).transpose(0, 2, 1) # 복구
        # 4. x축 연산 적용
        # (y, z, x) 순서로 보냄
        U_swapped = U_y.transpose(1, 2, 0)
        tmp = op_x @ U_swapped.reshape(-1, N).T
        U_x = tmp.T.reshape(N, N, N).transpose(2, 0, 1) # 복구
        return U_x.ravel()

    # (A) 전체 질량 연산자 M * u
    def matvec_M(u_vec):
        # M = Mx (x) My (x) Mz
        return apply_tensor_op_3d(u_vec, M1, M1, M1)

    # (B) 전체 강성(Laplacian) 연산자 K * u
    def matvec_K(u_vec):
        # K = KMM + MKM + MMK
        term1 = apply_tensor_op_3d(u_vec, K1, M1, M1) # d2/dx2
        term2 = apply_tensor_op_3d(u_vec, M1, K1, M1) # d2/dy2
        term3 = apply_tensor_op_3d(u_vec, M1, M1, K1) # d2/dz2
        return term1 + term2 + term3

    # (C) 포텐셜 에너지 V * u (Lumped Mass Approximation)
    X, Y, Z = np.meshgrid(x, x, x, indexing='ij')
    V_vals = 0.5 * (X**2 + Y**2 + Z**2)
    V_flat = V_vals.ravel()
    # 적분 가중치 (h^3) 포함
    V_scale = h**3 
    
    def matvec_V(u_vec):
        # Diagonal Matrix Free: 그냥 요소별 곱셈(Element-wise multiply)
        return (V_flat * u_vec) * V_scale

    # (D) 최종 Hamiltonian 연산자 H * u
    def matvec_H(u_vec):
        # H = 0.5 * K + V
        return 0.5 * matvec_K(u_vec) + matvec_V(u_vec)

    # ---------------------------------------------------------
    # 4. LinearOperator 정의 및 경계 조건
    # ---------------------------------------------------------
    # 경계 조건(u=0)을 Operator 내부에서 처리하는 것은 복잡하므로,
    # 여기서는 간단히 '내부 노드'만 계산하는 방식을 쓰지 않고
    # 전체를 계산하되, 경계값의 영향이 미미해지는 큰 박스 L=4.0을 사용한다.
    # (엄밀한 처리를 위해서는 Projection Matrix P를 앞뒤로 곱해야 함 P^T H P)
    dim = N**3
    H_op = spla.LinearOperator((dim, dim), matvec=matvec_H)
    M_op = spla.LinearOperator((dim, dim), matvec=matvec_M)
    # ---------------------------------------------------------
    # 5. 고윳값 풀이 (Matrix-Free Eigensolver)
    # ---------------------------------------------------------
    print(f"고윳값 계산 중... (자유도: {dim:,})")
    print("Matrix-free 방식은 Shift-invert를 쓰기 어려워 수렴이 조금 느릴 수 있다.")
    # 'SA': Smallest Algebraic eigenvalues (가장 작은 값 찾기)
    # Shift-Invert(sigma)를 쓰지 않는 이유는 (H - sigma M)^-1 연산을 
    # Matrix-Free로 구현하려면 내부에 또다시 반복법(CG)을 써야 해서 매우 느려지기 때문이다.
    vals, vecs = spla.eigsh(H_op, k=5, M=M_op, which='SA', tol=1e-4)
    # ---------------------------------------------------------
    # 6. 결과 출력
    # ---------------------------------------------------------
    print("\n[Matrix-free FEM 결과]")
    print(f"{'State':<6} | {'Calculated':<12} | {'Theory':<12} | {'Error (%)'}")
    print("-" * 50)
    theoretical_levels = [1.5, 2.5, 2.5, 2.5, 3.5]
    # 정렬
    idx = vals.argsort()
    vals = vals[idx]
    vecs = vecs[:, idx]
    for i in range(5):
        error = abs(vals[i] - theoretical_levels[i]) / theoretical_levels[i] * 100
        print(f"{i:<6} | {vals[i]:.6f}     | {theoretical_levels[i]:.1f}          | {error:.4f}%")
    # 시각화 (중앙 단면)
    u_3d = vecs[:, 0].reshape((N, N, N))
    mid = N // 2
    plt.figure(figsize=(8, 6))
    plt.contourf(X[:,:,mid], Y[:,:,mid], u_3d[:,:,mid], levels=50, cmap='inferno')
    plt.colorbar(label='Wavefunction')
    plt.title(f"Matrix-free FEM (N={N}^3)\nGround state energy: {vals[0]:.4f}")
    plt.show()
if __name__ == "__main__":
    solve_matrix_free_3d_fem_harmonic()

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
def solve_3d_harmonic_oscillator_corrected():
    print("Corrected 3D FEM Solver 시작...")
    # 1. 설정 (이산화)
    # 영역을 조금 더 넓혔다 (경계 효과 방지)
    L = 4.0         
    N = 65          # 격자 수
    N = 45
    x = np.linspace(-L, L, N)
    h = x[1] - x[0] # 요소 크기 (dx)
    # 2. 1차원 FEM 행렬 생성
    e = np.ones(N)
    # Stiffness (K1): [-1, 2, -1] / h
    # (약한 형식 적분 결과: 1/h 스케일)
    K1 = sp.spdiags([-e, 2*e, -e], [-1, 0, 1], N, N) / h
    # Mass (M1): [1, 4, 1] * h / 6 (Simpson's rule 가중치와 유사)
    # (약한 형식 적분 결과: h 스케일)
    M1 = sp.spdiags([e, 4*e, e], [-1, 0, 1], N, N) * (h / 6)
    # 3. 3차원 행렬 확장 (Kronecker Product)
    # Identity 대신 Mass Matrix를 사용해야 정확한 FEM이 된다.
    # 전체 질량 행렬 M_3d = Mx (x) My (x) Mz
    # 스케일: h * h * h = h^3
    M_3d = sp.kron(sp.kron(M1, M1), M1)
    # 전체 강성 행렬 K_3d (Laplacian)
    # K_3d = Kx(x)My(x)Mz + ...
    # 스케일: (1/h) * h * h = h
    K_x = sp.kron(sp.kron(K1, M1), M1)
    K_y = sp.kron(sp.kron(M1, K1), M1)
    K_z = sp.kron(sp.kron(M1, M1), K1)
    K_3d = K_x + K_y + K_z
    # 4. 퍼텐셜 에너지 행렬 (핵심 수정 부분!)
    # V(x,y,z) 값을 구함
    X, Y, Z = np.meshgrid(x, x, x, indexing='ij')
    V_vals = 0.5 * (X**2 + Y**2 + Z**2)
    V_vec = V_vals.flatten()
    V_mat = sp.diags(V_vec * (h**3))
    # 최종 Hamiltonian (좌변 행렬 A)
    # A = 0.5 * Stiffness + Potential_Integrated
    H_sys = 0.5 * K_3d + V_mat
    # 5. 경계 조건 (Slicing)
    mask = np.zeros((N, N, N), dtype=bool)
    mask[1:-1, 1:-1, 1:-1] = True
    inner_indices = np.where(mask.flatten())[0]
    # 내부 행렬만 추출
    H_inner = H_sys.tocsr()[inner_indices, :][:, inner_indices]
    M_inner = M_3d.tocsr()[inner_indices, :][:, inner_indices]
    # 6. 고유값 풀이 (Generalized Eigenvalue Problem)
    print(f"고유값 계산 중... (Matrix Size: {H_inner.shape})")
    # shift-invert mode (sigma=1.5 근처 탐색)
    vals, vecs = spla.eigsh(H_inner, k=5, M=M_inner, sigma=1.4, which='LM')
    # 7. 결과 출력
    print("\n[Calculated Eigenvalues vs Theory]")
    print(f"{'State':<6} | {'Calc (FEM)':<12} | {'Theory':<12} | {'Error (%)'}")
    print("-" * 50)
    theoretical_levels = [1.5, 2.5, 2.5, 2.5, 3.5] # 0, (1,0,0), (0,1,0), (0,0,1), ...
    # eigsh 결과는 정렬되지 않을 수 있으므로 정렬
    idx = vals.argsort()
    vals = vals[idx]
    vecs = vecs[:, idx]
    for i in range(5):
        calc = vals[i]
        theo = theoretical_levels[i]
        error = abs(calc - theo) / theo * 100
        print(f"{i:<6} | {calc:.6f}     | {theo:.1f}          | {error:.4f}%")
    # 시각화 (Ground State)
    u_final = np.zeros(N**3)
    u_final[inner_indices] = vecs[:, 0]
    u_3d = u_final.reshape((N, N, N))
    mid = N // 2
    plt.figure(figsize=(8, 6))
    plt.contourf(X[:,:,mid], Y[:,:,mid], u_3d[:,:,mid], levels=50, cmap='inferno')
    plt.colorbar(label='Wavefunction $\psi$')
    plt.title(f"Corrected 3D FEM Results (E0 = {vals[0]:.4f})")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.show()
if __name__ == "__main__":
    solve_3d_harmonic_oscillator_corrected()

In [ ]:
import numpy as np
from scipy.sparse import diags, kron, identity
from scipy.sparse.linalg import eigsh
from scipy.optimize import approx_fprime # SciPy의 FDM 기능을 사용하여 계수를 얻는 대체 방법
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# === 상수 설정 ===
# N: 각 차원 격자점 수 (N^3 격자)
N = 71 
# L: 계산 영역의 길이 ([-L/2, L/2])
L = 10.0
# 격자 간격
h = L / (N - 1)
# 조화 진동자 상수 (omega_x = omega_y = omega_z = 1)
omega = 1.0

print(f"격자점 수: {N}^3 = {N**3}")
print(f"격자 간격 h: {h:.4f}")

# === Fornberg 계수 (고정밀 2차 도함수) ===
# 5점 중앙 차분 공식 (정확도 O(h^4))의 계수:
# c_{-2}, c_{-1}, c_0, c_1, c_2
# 2차 도함수 d^2/dx^2에 대한 계수는 다음과 같다.
# 출처: Fornberg 논문 또는 Taylor 전개
c_weights = np.array([-1./12, 4./3, -5./2, 4./3, -1./12]) * (1 / h**2)

# === 1차원 운동 에너지 행렬 (T_1D) 구성 ===
# 대각선 요소의 오프셋 (k = 0: 중앙, k = 1: 윗대각선, k = -1: 아랫대각선 등)
offsets = np.array([-2, -1, 0, 1, 2]) 
# -1/2 팩터 적용 (운동 에너지 항: -1/2 * (d^2/dx^2))
T_1D_data = [(-0.5 * c_weights[i]) * np.ones(N - abs(offsets[i])) for i in range(len(offsets))]

# 희소 행렬 (CSR 형식) 생성
T_1D = diags(T_1D_data, offsets, shape=(N, N), format='csr')

print("\n1D 운동 에너지 행렬 T_1D 구성 완료.")

# === 3차원 운동 에너지 행렬 (T_3D) ===
I_N = identity(N, format='csr')

# T_x 항
T_x = kron(kron(T_1D, I_N), I_N, format='csr')
# T_y 항
T_y = kron(kron(I_N, T_1D), I_N, format='csr')
# T_z 항
T_z = kron(kron(I_N, I_N), T_1D, format='csr')

T_3D = T_x + T_y + T_z

# === 3차원 퍼텐셜 에너지 행렬 (V_3D) ===
# 1D 좌표 생성
x = np.linspace(-L/2, L/2, N)
y = np.linspace(-L/2, L/2, N)
z = np.linspace(-L/2, L/2, N)

# 3D 격자점의 퍼텐셜 값 V(x, y, z) 계산 (V = 1/2 * omega^2 * (x^2 + y^2 + z^2))
# np.meshgrid는 (N, N, N) 배열을 반환하며, np.ravel()로 1D 벡터화
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
V_vector = 0.5 * omega**2 * (X**2 + Y**2 + Z**2)
V_vector = V_vector.ravel()

# 퍼텐셜 행렬은 대각 행렬
V_3D = diags(V_vector, 0, format='csr')

# === 해밀토니안 행렬 ===
H = T_3D + V_3D

print("3차원 해밀토니안 H 구성 완료.")

# === 고유값 문제 풀이 ===
# eigsh: 희소 행렬의 고유값과 고유 벡터를 계산 (가장 작은 k개의 값을 찾음)
k_states = 5 
energies, wavefunctions = eigsh(H, k=k_states, which='SA') # SA: Smallest Magnitude (대부분 Lowest Energy)

# === 결과 출력 ===
print("\n--- 결과 ---")
print("고유값 (에너지 준위):")
for i, E in enumerate(energies):
    # 이론값 E_n = n + 1.5, 여기서 n = n_x + n_y + n_z
    n_sum = round(E - 1.5)
    E_theoretical = n_sum + 1.5
    print(f"state {i+1} (n_x+n_y+n_z={n_sum}): E_FDM = {E:.6f}, E_th = {E_theoretical:.6f}")

# === 파동 함수 시각화 (바닥 상태) ===
# 3D 파동 함수를 (N, N, N) 격자로 재구성
Psi_0_3D = wavefunctions[:, 0].reshape((N, N, N))

# 중앙 단면(z=0)을 시각화
mid_index = N // 2
Psi_0_slice = Psi_0_3D[:, :, mid_index]
X_slice, Y_slice = np.meshgrid(x, y, indexing='ij')

plt.figure(figsize=(8, 6))
plt.contourf(X_slice, Y_slice, np.abs(Psi_0_slice)**2, 50, cmap='viridis')
plt.colorbar(label='Probability density $|\psi_0|^2$')
plt.title(f'3D harmonic oscillator ground state (z=0)\nE = {energies[0]:.4f}')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.show()

In [ ]:
import numpy as np
from scipy.sparse import diags, kron, identity
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math

# === 상수 설정 ===
N = 71          # 각 차원 격자점 수 (N^3 격자) - 정확도 향상을 위해 41 사용
L = 20.0        # 계산 영역의 길이 ([-L/2, L/2])
h = L / (N - 1) # 격자 간격
omega = 1.0     # 조화 진동자 상수 (omega_x = omega_y = omega_z = 1)

# Fornberg 공식 설정
N_points = 7    # 7점 공식 사용 (O(h^6) 정확도)
M = (N_points - 1) // 2 # 중앙점으로부터의 오프셋
k_deriv = 2     # 2차 도함수

def calculate_fd_weights():
    """
    7점 중앙 차분 공식의 2차 도함수 계수 (h=1 기준)를 반환한다.
    """
    # 7점 공식 계수: [-1/180, 2/45, -1/5, -49/18, -1/5, 2/45, -1/180]
    # NOTE: 부동 소수점 정밀도를 위해 유리수 형태로 입력한다.
    weights = np.array([
        1.0/90.0, 
        -3.0/20.0, 
        3.0/2.0, 
        -49.0/18.0, # 중앙점 계수
        3.0/2.0, 
        -3.0/20.0, 
        1.0/90.0
    ])
    return weights

weights_7_point_h1 = calculate_fd_weights()
# h 팩터 적용: w_i / h^2
weights_7_point = weights_7_point_h1 / h**2 
offsets = np.arange(-M, M + 1) # [-3, -2, -1, 0, 1, 2, 3]

# === 1차원 운동 에너지 행렬 (T_1D) 구성 ===

# T_1D 행렬의 데이터 리스트 (대각선 별로)를 생성한다.
# 각 요소는: -1/2 * (w_i / h^2)
T_1D_data = [
    (-0.5 * weights_7_point[i]) * np.ones(N - abs(offsets[i])) 
    for i in range(len(offsets))
]

# 희소 행렬 (CSR 형식) 생성
T_1D = diags(T_1D_data, offsets, shape=(N, N), format='csr')

# === 3차원 운동 에너지 행렬 (T_3D) ===
I_N = identity(N, format='csr')

# Kronecker 곱을 이용한 3차원 확장
# T_x 항
T_x = kron(kron(T_1D, I_N), I_N, format='csr')
# T_y 항
T_y = kron(kron(I_N, T_1D), I_N, format='csr')
# T_z 항
T_z = kron(kron(I_N, I_N), T_1D, format='csr')

T_3D = T_x + T_y + T_z


# === 3차원 퍼텐셜 에너지 행렬 (V_3D) ===
x = np.linspace(-L/2, L/2, N)
y = np.linspace(-L/2, L/2, N)
z = np.linspace(-L/2, L/2, N)
# 3D 격자점의 퍼텐셜 값 V(x, y, z) 계산 (V = 1/2 * omega^2 * r^2)
X, Y, Z = np.meshgrid(x, y, z, indexing='ij') 
V_vector = 0.5 * omega**2 * (X**2 + Y**2 + Z**2)
V_vector = V_vector.ravel()

# 퍼텐셜 행렬은 대각 행렬
V_3D = diags(V_vector, 0, format='csr')

# === 전체 해밀토니안 행렬 ===
H = T_3D + V_3D

print(f"3차원 해밀토니안 H 구성 완료. 크기: {H.shape[0]} x {H.shape[1]}")


# === 고유값 문제 풀이 ===
k_states = 5 
# eigsh: 희소 행렬의 고유값 계산 (SA: Smallest Algebraic magnitude)
# N=41은 행렬 크기가 커서 계산 시간이 오래 걸릴 수 있다.
energies, wavefunctions = eigsh(H, k=k_states, which='SA') 

# === 결과 출력 ===
print("\n--- 결과 (N=71, 7점 공식) ---")
print("고유값 (에너지 준위):")
for i, E in enumerate(energies):
    # 이론값 E_n = n + 1.5, 여기서 n = n_x + n_y + n_z
    n_sum = round(E - 1.5)
    E_theoretical = n_sum + 1.5
    error = E - E_theoretical
    print(f"state {i+1} (n_sum={n_sum}): E_FDM = {E:.8f}, E_th = {E_theoretical:.8f}, 오차 = {error:+.2e}")

# === 파동 함수 시각화 (바닥 상태) ===
# 3D 파동 함수를 (N, N, N) 격자로 재구성
Psi_0_3D = wavefunctions[:, 0].reshape((N, N, N))

# 중앙 단면(z=0)을 시각화
mid_index = N // 2
Psi_0_slice = Psi_0_3D[:, :, mid_index]
X_slice, Y_slice = np.meshgrid(x, x, indexing='ij')

plt.figure(figsize=(8, 6))
# 확률 밀도 함수 시각화 (|Psi|^2)
plt.contourf(X_slice, Y_slice, np.abs(Psi_0_slice)**2, 50, cmap='viridis')
plt.colorbar(label='Probability density $|\psi_0|^2$')
plt.title(f'3D ground state (z=0 )\nE = {energies[0]:.6f}')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.show()

# 파동 함수 표면 플롯 (선택 사항 - 느릴 수 있음)
# fig = plt.figure(figsize=(10, 8))
# ax = fig.add_subplot(111, projection='3d')
# ax.plot_surface(X_slice, Y_slice, np.abs(Psi_0_slice)**2, cmap='viridis')
# ax.set_title('3D 바닥 상태 파동 함수 표면')
# plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# matplotlib의 경고 메시지를 무시할 경우:
# import warnings
# warnings.filterwarnings("ignore", category=UserWarning)

def fft_poisson_solver_3d(N=64):
    """
    3차원 푸아송 방정식 (∇²φ = f)을 고속 푸리에 변환(FFT)을 사용하여 푼다.
    주기적 경계 조건을 가정하며, 분석적 해와 비교하여 정확도를 확인한다.
    """
    
    # --- 1. 도메인 및 격자 설정 ---
    L = 2 * np.pi  # 도메인 크기 [0, 2*pi]
    h = L / N      # 격자 간격

    # 1차원 공간 배열 생성
    x = np.linspace(0, L, N, endpoint=False)
    y = np.linspace(0, L, N, endpoint=False)
    z = np.linspace(0, L, N, endpoint=False)

    # 3차원 격자 생성
    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

    # --- 2. 분석적 해 및 소스 항 f 계산 ---
    # 분석적 해 (정답): phi_exact = sin(x) * cos(2y) * sin(3z)
    phi_exact = np.sin(X) * np.cos(2*Y) * np.sin(3*Z)

    # 소스 항 f = ∇²(phi_exact) = -(1² + 2² + 3²) * phi_exact = -14 * phi_exact
    f = -14 * phi_exact 

    # --- 3. FFT를 이용한 해법 ---

    # 3D FFT 수행 (f -> f_hat)
    f_hat = np.fft.fftn(f)

    # 파수(Wave number) 배열 생성 
    # k_array는 2 * pi * j / L (j는 FFT 주파수 인덱스)
    k_array = 2 * np.pi * np.fft.fftfreq(N, d=L/N) 

    # 3차원 파수 배열 Kx, Ky, Kz 생성
    Kx, Ky, Kz = np.meshgrid(k_array, k_array, k_array, indexing='ij')

    # 라플라스 연산자의 파수 공간 표현: K_squared = -(kx^2 + ky^2 + kz^2)
    K_squared = -(Kx**2 + Ky**2 + Kz**2)

    # --- 4. 파수 공간에서 해 계산 (f_hat -> phi_hat) ---
    # phi_hat = f_hat / K_squared
    
    # K_squared = 0 인 경우 (k=0 모드) 처리
    # 주기적 경계 조건에서 해가 존재하려면 f의 평균이 0이어야 한다. (현재 f는 평균이 0)
    # k=0 모드는 0으로 나누는 것을 방지하기 위해 임의의 값 설정 후, phi_hat[0, 0, 0]을 0으로 강제한다.
    # 이는 해의 평균값을 0으로 설정하는 것과 같다.
    K_squared[0, 0, 0] = 1.0 
    
    phi_hat = f_hat / K_squared
    phi_hat[0, 0, 0] = 0.0 

    # --- 5. 역 FFT 수행 (phi_hat -> phi_numerical) ---
    # ifftn의 결과는 복소수이므로 실수부(.real)만 취한다.
    phi_numerical = np.fft.ifftn(phi_hat).real

    # --- 6. 결과 비교 및 오차 계산 ---
    # L2 상대 오차
    error = np.linalg.norm(phi_numerical - phi_exact) / np.linalg.norm(phi_exact)

    print(f"--- 3D FFT Poisson Solver Result ---")
    print(f"격자점 수 (N x N x N): {N} x {N} x {N}")
    print(f"L2 상대 오차: {error:.2e}")
    print(f"------------------------------------")
    
    # --- 7. 결과 시각화 (개선된 옵션) ---
    slice_index = 0  # Z축의 첫 번째 평면 (z=0)
    
    # Phi의 값 범위 설정 (sin 함수 기반이므로 대략 -1.0 ~ 1.0)
    v_max_phi = np.max(phi_exact) 
    v_min_phi = np.min(phi_exact) 

    # 오차의 값 범위 설정 (오차는 float 정밀도 수준)
    v_max_error = np.max(np.abs(phi_numerical - phi_exact))

    plt.figure(figsize=(15, 5)) # 전체 그림 크기 확대

    # 1. 분석적 해 (phi_exact)
    plt.subplot(131)
    im1 = plt.imshow(
        phi_exact[:, :, slice_index], 
        origin='lower', 
        extent=[0, L, 0, L],
        cmap='seismic',       # 양수/음수 값을 잘 보여주는 다이버지 색상 맵
        vmin=v_min_phi,       # 통일된 최소값
        vmax=v_max_phi        # 통일된 최대값
    )
    plt.title(r'Analytical $\phi$', fontsize=14)
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.colorbar(im1, fraction=0.046, pad=0.04)

    # 2. 수치 해 (phi_numerical)
    plt.subplot(132)
    im2 = plt.imshow(
        phi_numerical[:, :, slice_index], 
        origin='lower', 
        extent=[0, L, 0, L],
        cmap='seismic',       # 분석적 해와 동일한 설정 사용
        vmin=v_min_phi,
        vmax=v_max_phi
    )
    plt.title(r'Numerical $\phi$ (FFT)', fontsize=14)
    plt.xlabel('X')
    plt.colorbar(im2, fraction=0.046, pad=0.04)

    # 3. 절대 오차 (Absolute Error)
    plt.subplot(133)
    im3 = plt.imshow(
        np.abs(phi_numerical[:, :, slice_index] - phi_exact[:, :, slice_index]), 
        origin='lower', 
        extent=[0, L, 0, L],
        cmap='magma',        # 오차를 강조하는 순차적 색상 맵
        vmin=0,
        # 최대 오차보다 작은 값으로 설정하여 작은 오차의 미세한 패턴을 강조
        vmax=v_max_error * 1.5 
    )
    plt.title('Absolute Error', fontsize=14)
    plt.xlabel('X')
    # 과학적 표기법으로 컬러바 포맷 지정
    plt.colorbar(im3, fraction=0.046, pad=0.04, format='%.1e') 
    
    plt.suptitle(f'FFT Poisson Solver (Z-Slice at $z={x[slice_index]:.2f}$)', fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) 
    plt.show()

if __name__ == '__main__':
    # N 값을 변경하여 격자 해상도를 조절할 수 있다.
    fft_poisson_solver_3d(N=64)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --- 1. 도메인 및 격자 설정 ---
L = 2 * np.pi  # 도메인 크기 [0, 2*pi]
N = 64         # 격자점 수 (각 축에 대해)
h = L / N      # 격자 간격

# 1차원 공간 배열 생성
x = np.linspace(0, L, N, endpoint=False)
y = np.linspace(0, L, N, endpoint=False)
z = np.linspace(0, L, N, endpoint=False)

# 3차원 격자 생성
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

# --- 2. 분석적 해 및 소스 항 f 계산 ---
# 분석적 해 (정답)
phi_exact = np.sin(X) * np.cos(2*Y) * np.sin(3*Z)

# 소스 항 f = Nabla^2(phi_exact)
f = -14 * phi_exact 

# --- 3. FFT를 이용한 해법 ---

# 3D FFT 수행 (f -> f_hat)
f_hat = np.fft.fftn(f)

# 파수(Wave number) 배열 생성 (FFT 주파수 배열)
# np.fft.fftfreq는 k_i = 2*pi*j/(N*h)를 계산해야 하지만,
# h가 이미 L/N이므로, L = 2*pi인 경우 k_i = j/h * 2pi/L * L = 2pi*j/L
k_array = 2 * np.pi * np.fft.fftfreq(N, d=h) 

# 3차원 파수 배열 Kx, Ky, Kz 생성
Kx, Ky, Kz = np.meshgrid(k_array, k_array, k_array, indexing='ij')

# 라플라스 연산자의 파수 공간 표현: K^2 = -(kx^2 + ky^2 + kz^2)
K_squared = -(Kx**2 + Ky**2 + Kz**2)

# --- 4. 파수 공간에서 해 계산 (f_hat -> phi_hat) ---
# phi_hat = f_hat / K^2
phi_hat = np.zeros_like(f_hat, dtype=complex)

# K_squared = 0 인 경우 (k=0 모드) 처리
# 포아송 방정식의 해는 상수를 더해도 동일하므로, k=0 모드는 0으로 설정하거나
# 소스 항 f의 평균이 0인지 확인해야 한다. (주기적 경계 조건에서 해가 존재하기 위한 조건)
# 현재 f는 평균이 0이다 (사인/코사인 함수의 곱).
K_squared[0, 0, 0] = 1.0 # 0으로 나누는 것을 방지하기 위해 임의의 값 설정
phi_hat = f_hat / K_squared
phi_hat[0, 0, 0] = 0.0 # k=0 모드는 0으로 강제 (평균 포텐셜은 0으로 설정)

# --- 5. 역 FFT 수행 (phi_hat -> phi_numerical) ---
phi_numerical = np.fft.ifftn(phi_hat).real

# --- 6. 결과 비교 및 오차 계산 ---
# 수치 해와 분석적 해의 차이 (L2 노름 오차)
error = np.linalg.norm(phi_numerical - phi_exact) / np.linalg.norm(phi_exact)

print(f"격자점 수 (N): {N}")
print(f"L2 상대 오차: {error:.2e}")

# --- 7. 결과 시각화 (선택적) ---
# 시각화를 위해 z=0 평면 (k=0)에서의 2D 단면을 사용
slice_index = 0  # Z축의 첫 번째 평면
plt.figure(figsize=(12, 4))

plt.subplot(131)
plt.imshow(phi_exact[:, :, slice_index], origin='lower', extent=[0, L, 0, L])
plt.title(r'Analytical $\phi$')
plt.colorbar()

plt.subplot(132)
plt.imshow(phi_numerical[:, :, slice_index], origin='lower', extent=[0, L, 0, L])
plt.title(r'Numerical $\phi$ (FFT)')
plt.colorbar()

plt.subplot(133)
plt.imshow(np.abs(phi_numerical[:, :, slice_index] - phi_exact[:, :, slice_index]), origin='lower', extent=[0, L, 0, L])
plt.title('Absolute error')
plt.colorbar()
plt.suptitle(r'FFT Poisson solver $\phi(x, y, z=0)$ Slice')
plt.show()

In [ ]:
import numpy as np
import scipy.sparse as sparse
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

def solve_3d_poisson_fdm_verified():
    print("3D Poisson Solver (FDM) with Verification 시작...")
    # ---------------------------------------------------------
    # 1. 문제 설정
    # ---------------------------------------------------------
    N = 30  # 격자 해상도 (30^3 = 27,000 DOFs)
    L = 1.0
    h = L / (N - 1)
    # 격자 생성
    x = np.linspace(0, L, N)
    y = np.linspace(0, L, N)
    z = np.linspace(0, L, N)
    # indexing='ij': 행렬 좌표계 (i, j, k) 순서 유지
    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
    # ---------------------------------------------------------
    # 2. 정답(Exact Solution) 미리 계산
    # ---------------------------------------------------------
    # 정답 가정: u = sin(pi*x)sin(pi*y)sin(pi*z)
    u_exact = np.sin(np.pi * X) * np.sin(np.pi * Y) * np.sin(np.pi * Z)
    # ---------------------------------------------------------
    # 3. 희소 행렬(Laplacian) 조립
    # ---------------------------------------------------------
    # 1D Laplacian Operator (Tridiagonal: 1, -2, 1)
    e = np.ones(N)
    diags = [e, -2*e, e]
    offsets = [-1, 0, 1]
    D1 = sparse.spdiags(diags, offsets, N, N)
    I1 = sparse.eye(N)
    # 3D Laplacian via Kronecker Sum
    # L3 = D_xx + D_yy + D_zz
    L3 = sparse.kron(sparse.kron(D1, I1), I1) + \
         sparse.kron(sparse.kron(I1, D1), I1) + \
         sparse.kron(sparse.kron(I1, I1), D1)
    L3 = L3 / h**2 # 스케일링
    # ---------------------------------------------------------
    # 4. 우변 벡터 및 경계 조건 설정
    # ---------------------------------------------------------
    # 방정식: -Laplacian u = f
    # u_exact를 대입해서 f를 구함: f = 3 * pi^2 * u
    f = 3 * (np.pi**2) * u_exact
    b = -f.flatten() # L3 * u = -f
    # 경계 조건 (Dirichlet u=0)
    # 경계 노드 인덱스 추출
    indices = np.arange(N**3).reshape((N, N, N))
    mask_boundary = np.zeros((N, N, N), dtype=bool)
    mask_boundary[0,:,:] = True; mask_boundary[-1,:,:] = True
    mask_boundary[:,0,:] = True; mask_boundary[:,-1,:] = True
    mask_boundary[:,:,0] = True; mask_boundary[:,:,-1] = True
    boundary_idx = indices[mask_boundary].flatten()
    # 행렬 수정 (Zero-out rows and 1 on diagonal)
    L3 = L3.tolil()
    for idx in boundary_idx:
        L3[idx, :] = 0
        L3[idx, idx] = 1
        b[idx] = 0 # 경계값 0
    L3 = L3.tocsr()
    # ---------------------------------------------------------
    # 5. 선형 시스템 풀이
    # ---------------------------------------------------------
    print(f"선형 시스템 풀이 중... (Unknowns: {N**3})")
    u_vec = spla.spsolve(L3, b)
    u_numer = u_vec.reshape((N, N, N))
    # ---------------------------------------------------------
    # 6. 정확도 검증 (Verification) - 추가된 부분
    # ---------------------------------------------------------
    # 오차 행렬 계산
    error_field = np.abs(u_numer - u_exact)
    # 1) 최대 오차 (L-infinity Norm)
    max_error = np.max(error_field)
    # 2) 평균 제곱근 오차 (Discrete L2 Norm)
    # RMS Error = sqrt( sum(error^2) / N_points )
    l2_error = np.sqrt(np.mean(error_field**2))
    print("-" * 40)
    print(f"검증 결과 (Grid N={N})")
    print("-" * 40)
    print(f"Max Error (L_inf) : {max_error:.6e}")
    print(f"RMS Error (L_2)   : {l2_error:.6e}")
    print("-" * 40)
    # ---------------------------------------------------------
    # 7. 결과 시각화 (비교)
    # ---------------------------------------------------------
    z_slice = N // 2
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    # (1) 수치해 (Numerical Solution)
    im1 = axes[0].contourf(X[:,:,z_slice], Y[:,:,z_slice], u_numer[:,:,z_slice], 
                           levels=50, cmap='viridis')
    axes[0].set_title(f"Numerical Solution (FDM)")
    plt.colorbar(im1, ax=axes[0])
    # (2) 정답 (Exact Solution)
    im2 = axes[1].contourf(X[:,:,z_slice], Y[:,:,z_slice], u_exact[:,:,z_slice], 
                           levels=50, cmap='viridis')
    axes[1].set_title(f"Exact Solution")
    plt.colorbar(im2, ax=axes[1])
    # (3) 오차 (Absolute Error)
    im3 = axes[2].contourf(X[:,:,z_slice], Y[:,:,z_slice], error_field[:,:,z_slice], 
                           levels=50, cmap='inferno')
    axes[2].set_title(f"Absolute Error |Num - Exact|")
    plt.colorbar(im3, ax=axes[2])
    for ax in axes:
        ax.set_xlabel("x")
        ax.set_ylabel("y")
    plt.suptitle(f"3D Poisson equation verification (Slice at z={z[z_slice]:.2f})", fontsize=16)
    plt.tight_layout()
    plt.show()
if __name__ == "__main__":
    solve_3d_poisson_fdm_verified()

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

def solve_3d_harmonic_oscillator_corrected():
    print("Corrected 3D FEM Solver 시작...")
    # 1. 설정 (이산화)
    # 영역을 조금 더 넓혔다 (경계 효과 방지)
    L = 4.0         
    N = 45          # 격자 수 
    x = np.linspace(-L, L, N)
    h = x[1] - x[0] # 요소 크기 (dx)
    # 2. 1차원 FEM 행렬 생성
    e = np.ones(N)
    # Stiffness (K1): [-1, 2, -1] / h
    # (약한 형식 적분 결과: 1/h 스케일)
    K1 = sp.spdiags([-e, 2*e, -e], [-1, 0, 1], N, N) / h
    # Mass (M1): [1, 4, 1] * h / 6 (Simpson's rule 가중치와 유사)
    # (약한 형식 적분 결과: h 스케일)
    M1 = sp.spdiags([e, 4*e, e], [-1, 0, 1], N, N) * (h / 6)
    # 3. 3차원 행렬 확장 (Kronecker Product)
    # Identity 대신 Mass Matrix를 사용해야 정확한 FEM이 된다.
    # 전체 질량 행렬 M_3d = Mx (x) My (x) Mz
    # 스케일: h * h * h = h^3
    M_3d = sp.kron(sp.kron(M1, M1), M1)
    # 전체 강성 행렬 K_3d (Laplacian)
    # K_3d = Kx(x)My(x)Mz + ...
    # 스케일: (1/h) * h * h = h
    K_x = sp.kron(sp.kron(K1, M1), M1)
    K_y = sp.kron(sp.kron(M1, K1), M1)
    K_z = sp.kron(sp.kron(M1, M1), K1)
    K_3d = K_x + K_y + K_z
    # 4. 포텐셜 에너지 행렬 (핵심 수정 부분!)
    # V(x,y,z) 값을 구함
    X, Y, Z = np.meshgrid(x, x, x, indexing='ij')
    V_vals = 0.5 * (X**2 + Y**2 + Z**2)
    V_vec = V_vals.flatten()
    V_mat = sp.diags(V_vec * (h**3))
    # 최종 Hamiltonian (좌변 행렬 A)
    # A = 0.5 * Stiffness + Potential_Integrated
    H_sys = 0.5 * K_3d + V_mat
    # 5. 경계 조건 (Slicing)
    mask = np.zeros((N, N, N), dtype=bool)
    mask[1:-1, 1:-1, 1:-1] = True
    inner_indices = np.where(mask.flatten())[0]
    # 내부 행렬만 추출
    H_inner = H_sys.tocsr()[inner_indices, :][:, inner_indices]
    M_inner = M_3d.tocsr()[inner_indices, :][:, inner_indices]
    # 6. 고윳값 풀이 (Generalized Eigenvalue Problem)
    print(f"고윳값 계산 중... (Matrix Size: {H_inner.shape})")
    # shift-invert mode (sigma=1.5 근처 탐색)
    vals, vecs = spla.eigsh(H_inner, k=5, M=M_inner, sigma=1.4, which='LM')
    # 7. 결과 출력
    print("\n[Calculated Eigenvalues vs Theory]")
    print(f"{'State':<6} | {'Calc (FEM)':<12} | {'Theory':<12} | {'Error (%)'}")
    print("-" * 50)
    theoretical_levels = [1.5, 2.5, 2.5, 2.5, 3.5] # 0, (1,0,0), (0,1,0), (0,0,1), ...
    # eigsh 결과는 정렬되지 않을 수 있으므로 정렬
    idx = vals.argsort()
    vals = vals[idx]
    vecs = vecs[:, idx]
    for i in range(5):
        calc = vals[i]
        theo = theoretical_levels[i]
        error = abs(calc - theo) / theo * 100
        print(f"{i:<6} | {calc:.6f}     | {theo:.1f}          | {error:.4f}%")
    # 시각화 (Ground State)
    u_final = np.zeros(N**3)
    u_final[inner_indices] = vecs[:, 0]
    u_3d = u_final.reshape((N, N, N))
    mid = N // 2
    plt.figure(figsize=(8, 6))
    plt.contourf(X[:,:,mid], Y[:,:,mid], u_3d[:,:,mid], levels=50, cmap='inferno')
    plt.colorbar(label='Wavefunction $\psi$')
    plt.title(f"Corrected 3D FEM Results (E0 = {vals[0]:.4f})")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.show()
if __name__ == "__main__":
    solve_3d_harmonic_oscillator_corrected()

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

def solve_matrix_free_3d_fem_harmonic():
    print("Matrix-Free (Tensor Product) 3D FEM Solver 시작...")
    # ---------------------------------------------------------
    # 1. 문제 설정
    # ---------------------------------------------------------
    # 메모리 걱정 없이 해상도를 높일 수 있다.
    N = 40          # 한 축의 격자 수 (전체 자유도 64,000)
    L = 4.0         # 영역 [-4, 4]
    x = np.linspace(-L, L, N)
    h = x[1] - x[0] # 요소 크기
    # ---------------------------------------------------------
    # 2. 1차원 소형 행렬 준비 (N x N)
    # ---------------------------------------------------------
    # 이 작은 행렬들만 메모리에 저장한다.
    e = np.ones(N)
    # 1D Stiffness (K1): 1/h 스케일
    K1 = sp.spdiags([-e, 2*e, -e], [-1, 0, 1], N, N) / h
    # 1D Mass (M1): h 스케일 (Simpson's rule 유사 가중치)
    M1 = sp.spdiags([e, 4*e, e], [-1, 0, 1], N, N) * (h / 6.0)
    # ---------------------------------------------------------
    # 3. Matrix-Free 연산 함수 (핵심!)
    # ---------------------------------------------------------
    # 입력 벡터 u (1D array)를 받아 3D 연산을 수행하고 다시 1D로 반환
    
    def apply_tensor_op_3d(u_vec, op_x, op_y, op_z):
        """
        Compute (op_x (kron) op_y (kron) op_z) * u
        행렬 조립 없이 차원별로 순차적으로 연산 적용
        """
        # 1. 3차원 텐서로 변환 (x, y, z)
        U = u_vec.reshape((N, N, N))
        # 2. z축 연산 적용 (Rightmost operator first)
        # U의 마지막 축(axis 2)에 대해 op_z 곱셈
        # 행렬 곱셈을 위해 (N*N, N)으로 reshape 했다가 복구
        tmp = op_z @ U.reshape(-1, N).T 
        U_z = tmp.T.reshape(N, N, N)
        # 3. y축 연산 적용
        # 축을 바꿔서(transpose) 연산하고 다시 돌려놓음 (swapaxes)
        U_swapped = U_z.transpose(0, 2, 1) # (x, z, y)
        tmp = op_y @ U_swapped.reshape(-1, N).T
        U_y = tmp.T.reshape(N, N, N).transpose(0, 2, 1) # 복구
        # 4. x축 연산 적용
        # (y, z, x) 순서로 보냄
        U_swapped = U_y.transpose(1, 2, 0)
        tmp = op_x @ U_swapped.reshape(-1, N).T
        U_x = tmp.T.reshape(N, N, N).transpose(2, 0, 1) # 복구
        return U_x.ravel()

    # (A) 전체 질량 연산자 M * u
    def matvec_M(u_vec):
        # M = Mx (x) My (x) Mz
        return apply_tensor_op_3d(u_vec, M1, M1, M1)

    # (B) 전체 강성(Laplacian) 연산자 K * u
    def matvec_K(u_vec):
        # K = KMM + MKM + MMK
        term1 = apply_tensor_op_3d(u_vec, K1, M1, M1) # d2/dx2
        term2 = apply_tensor_op_3d(u_vec, M1, K1, M1) # d2/dy2
        term3 = apply_tensor_op_3d(u_vec, M1, M1, K1) # d2/dz2
        return term1 + term2 + term3

    # (C) 포텐셜 에너지 V * u (Lumped Mass Approximation)
    X, Y, Z = np.meshgrid(x, x, x, indexing='ij')
    V_vals = 0.5 * (X**2 + Y**2 + Z**2)
    V_flat = V_vals.ravel()
    # 적분 가중치 (h^3) 포함
    V_scale = h**3 
    
    def matvec_V(u_vec):
        # Diagonal Matrix Free: 그냥 요소별 곱셈(Element-wise multiply)
        return (V_flat * u_vec) * V_scale

    # (D) 최종 Hamiltonian 연산자 H * u
    def matvec_H(u_vec):
        # H = 0.5 * K + V
        return 0.5 * matvec_K(u_vec) + matvec_V(u_vec)

    # ---------------------------------------------------------
    # 4. LinearOperator 정의 및 경계 조건
    # ---------------------------------------------------------
    # 경계 조건(u=0)을 Operator 내부에서 처리하는 것은 복잡하므로,
    # 여기서는 간단히 '내부 노드'만 계산하는 방식을 쓰지 않고
    # 전체를 계산하되, 경계값의 영향이 미미해지는 큰 박스 L=4.0을 사용한다.
    # (엄밀한 처리를 위해서는 Projection Matrix P를 앞뒤로 곱해야 함 P^T H P)
    dim = N**3
    H_op = spla.LinearOperator((dim, dim), matvec=matvec_H)
    M_op = spla.LinearOperator((dim, dim), matvec=matvec_M)
    # ---------------------------------------------------------
    # 5. 고윳값 풀이 (Matrix-Free Eigensolver)
    # ---------------------------------------------------------
    print(f"고윳값 계산 중... (자유도: {dim:,})")
    print("Matrix-Free 방식은 Shift-Invert를 쓰기 어려워 수렴이 조금 느릴 수 있다.")
    # 'SA': Smallest Algebraic eigenvalues (가장 작은 값 찾기)
    # Shift-Invert(sigma)를 쓰지 않는 이유는 (H - sigma M)^-1 연산을 
    # Matrix-Free로 구현하려면 내부에 또다시 반복법(CG)을 써야 해서 매우 느려지기 때문이다.
    vals, vecs = spla.eigsh(H_op, k=5, M=M_op, which='SA', tol=1e-4)
    # ---------------------------------------------------------
    # 6. 결과 출력
    # ---------------------------------------------------------
    print("\n[Matrix-Free FEM 결과]")
    print(f"{'State':<6} | {'Calculated':<12} | {'Theory':<12} | {'Error (%)'}")
    print("-" * 50)
    theoretical_levels = [1.5, 2.5, 2.5, 2.5, 3.5]
    # 정렬
    idx = vals.argsort()
    vals = vals[idx]
    vecs = vecs[:, idx]
    for i in range(5):
        error = abs(vals[i] - theoretical_levels[i]) / theoretical_levels[i] * 100
        print(f"{i:<6} | {vals[i]:.6f}     | {theoretical_levels[i]:.1f}          | {error:.4f}%")
    # 시각화 (중앙 단면)
    u_3d = vecs[:, 0].reshape((N, N, N))
    mid = N // 2
    plt.figure(figsize=(8, 6))
    plt.contourf(X[:,:,mid], Y[:,:,mid], u_3d[:,:,mid], levels=50, cmap='inferno')
    plt.colorbar(label='Wavefunction')
    plt.title(f"Matrix-Free FEM (N={N}^3)\nGround State Energy: {vals[0]:.4f}")
    plt.show()
if __name__ == "__main__":
    solve_matrix_free_3d_fem_harmonic()

In [ ]:
import numpy as np
from scipy.sparse import diags, kron, identity
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math
# === 상수 설정 ===
N = 71          # 각 차원 격자점 수 (N^3 격자) - 정확도 향상을 위해 41 사용
L = 20.0        # 계산 영역의 길이 ([-L/2, L/2])
h = L / (N - 1) # 격자 간격
omega = 1.0     # 조화 진동자 상수 (omega_x = omega_y = omega_z = 1)
# Fornberg 공식 설정
N_points = 7    # 7점 공식 사용 (O(h^6) 정확도)
M = (N_points - 1) // 2 # 중앙점으로부터의 오프셋
k_deriv = 2     # 2차 도함수
def calculate_fd_weights():
    """
    7점 중앙 차분 공식의 2차 도함수 계수 (h=1 기준)를 반환한다.
    """
    # 7점 공식 계수: [-1/180, 2/45, -1/5, -49/18, -1/5, 2/45, -1/180]
    # NOTE: 부동 소수점 정밀도를 위해 유리수 형태로 입력한다.
    weights = np.array([
    1.0/90.0, 
    -3.0/20.0, 
    3.0/2.0, 
    -49.0/18.0, # 중앙점 계수
    3.0/2.0, 
    -3.0/20.0, 
    1.0/90.0
    ])
    return weights
weights_7_point_h1 = calculate_fd_weights()
# h 팩터 적용: w_i / h^2
weights_7_point = weights_7_point_h1 / h**2 
offsets = np.arange(-M, M + 1) # [-3, -2, -1, 0, 1, 2, 3]
# === 1차원 운동 에너지 행렬 (T_1D) 구성 ===
# T_1D 행렬의 데이터 리스트 (대각선 별로)를 생성한다.
# 각 요소는: -1/2 * (w_i / h^2)
T_1D_data = [
(-0.5 * weights_7_point[i]) * np.ones(N - abs(offsets[i])) 
for i in range(len(offsets))
]
# 희소 행렬 (CSR 형식) 생성
T_1D = diags(T_1D_data, offsets, shape=(N, N), format='csr')
# === 3차원 운동 에너지 행렬 (T_3D) ===
I_N = identity(N, format='csr')
# Kronecker 곱을 이용한 3차원 확장
# T_x 항
T_x = kron(kron(T_1D, I_N), I_N, format='csr')
# T_y 항
T_y = kron(kron(I_N, T_1D), I_N, format='csr')
# T_z 항
T_z = kron(kron(I_N, I_N), T_1D, format='csr')
T_3D = T_x + T_y + T_z
# === 3차원 퍼텐셜 에너지 행렬 (V_3D) ===
x = np.linspace(-L/2, L/2, N)
y = np.linspace(-L/2, L/2, N)
z = np.linspace(-L/2, L/2, N)
# 3D 격자점의 퍼텐셜 값 V(x, y, z) 계산 (V = 1/2 * omega^2 * r^2)
X, Y, Z = np.meshgrid(x, y, z, indexing='ij') 
V_vector = 0.5 * omega**2 * (X**2 + Y**2 + Z**2)
V_vector = V_vector.ravel()
# 퍼텐셜 행렬은 대각 행렬
V_3D = diags(V_vector, 0, format='csr')
# === 전체 해밀토니안 행렬 ===
H = T_3D + V_3D
print(f"3차원 해밀토니안 H 구성 완료. 크기: {H.shape[0]} x {H.shape[1]}")
# === 고유값 문제 풀이 ===
k_states = 5 
# eigsh: 희소 행렬의 고유값 계산 (SA: Smallest Algebraic magnitude)
# N=71은 행렬 크기가 커서 계산 시간이 오래 걸릴 수 있다.
energies, wavefunctions = eigsh(H, k=k_states, which='SA') 
# === 결과 출력 ===
print("\n--- 결과 (N=71, 7점 공식) ---")
print("고유값 (에너지 준위):")
for i, E in enumerate(energies):
    # 이론값 E_n = n + 1.5, 여기서 n = n_x + n_y + n_z
    n_sum = round(E - 1.5)
    E_theoretical = n_sum + 1.5
    error = E - E_theoretical
    print(f"state {i+1} (n_sum={n_sum}): E_FDM = {E:.8f}, E_th = {E_theoretical:.8f}, 오차 = {error:+.2e}")
# === 파동 함수 시각화 (바닥 상태) ===
# 3D 파동 함수를 (N, N, N) 격자로 재구성
Psi_0_3D = wavefunctions[:, 0].reshape((N, N, N))
# 중앙 단면(z=0)을 시각화
mid_index = N // 2
Psi_0_slice = Psi_0_3D[:, :, mid_index]
X_slice, Y_slice = np.meshgrid(x, x, indexing='ij')
plt.figure(figsize=(8, 6))
# 확률 밀도 함수 시각화 (|Psi|^2)
plt.contourf(X_slice, Y_slice, np.abs(Psi_0_slice)**2, 50, cmap='viridis')
plt.colorbar(label='Probability Density $|\psi_0|^2$')
plt.title(f'3D ground state (z=0 )\nE = {energies[0]:.6f}')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.show()
# 파동 함수 표면 플롯 (선택 사항 - 느릴 수 있음)
# fig = plt.figure(figsize=(10, 8))
# ax = fig.add_subplot(111, projection='3d')
# ax.plot_surface(X_slice, Y_slice, np.abs(Psi_0_slice)**2, cmap='viridis')
# ax.set_title('3D 바닥 상태 파동 함수 표면')
# plt.show()

## Chapter 7 적분   파이썬을 활용한 수치해석(Numerical Analysis with Python) 이인호 (북스힐, 2026)

In [ ]:
import math
def f(x):
	"""적분할 함수: x^3"""
	return x**3
def simpsons_rule(a, b, f):
	"""단일 구간 [a, b]에 대한 심슨 규칙"""
	h = (b - a) / 2
	c = (a + b) / 2
	# I = (h/3) * [f(a) + 4f(c) + f(b)]
	return (h / 3) * (f(a) + 4 * f(c) + f(b))
def adaptive_simpson(a, b, f, tol, whole_integral=None):
	"""
	재귀적 적응형 심슨 구적법	
	a, b: 구간 경계
	f: 함수
	tol: 목표 오차 (Tolerance)
	whole_integral: 상위 단계에서 계산된 넓은 구간의 적분값 (재사용)
	"""
	# 1. 넓은 구간의 적분값을 계산하거나 재사용
	if whole_integral is None:
	    # 처음 호출 시 전체 구간에 대해 심슨 규칙 적용
	    whole_integral = simpsons_rule(a, b, f)
	# 2. 구간을 두 개로 분할하고 각각의 적분값을 계산
	c = (a + b) / 2
	left_integral = simpsons_rule(a, c, f)
	right_integral = simpsons_rule(c, b, f)
	# 두 하위 구간의 합
	half_integral = left_integral + right_integral
	# 3. 오차 추정 (Error Estimation)
	# 심슨 규칙 기반의 오차 추정 공식: E ~ 1/15 * |I_half - I_whole|
	error_estimate = abs(half_integral - whole_integral) / 15
	# 4. 종료 조건 확인
	if error_estimate < tol:
    	# 오차가 목표치보다 작으면 계산 종료 후 합산된 값 반환
	    # (오차 추정값을 더하여 더 정확한 값을 반환하는 방식도 사용되지만, 
	    # 여기서는 간단히 합산값 반환)
	    return half_integral
	else:
	    # 오차가 목표치보다 크면 구간을 재귀적으로 분할
	    # 목표 오차를 두 하위 구간에 균등하게 분배
	    tol_half = tol / 2
	    # 왼쪽 하위 구간에 대해 재귀 호출
	    int_left = adaptive_simpson(a, c, f, tol_half, left_integral)
	    # 오른쪽 하위 구간에 대해 재귀 호출
	    int_right = adaptive_simpson(c, b, f, tol_half, right_integral)
	    # 두 재귀 호출 결과 합산
	    return int_left + int_right
# --- 실행 부분 ---
a = 0.0
b = 1.0
tolerance = 1e-4  # 목표 오차 (0.0001)
# 적응형 구적법 실행
result = adaptive_simpson(a, b, f, tolerance)
# 결과 출력
print(f"적분 구간: [{a}, {b}]")
print(f"함수: f(x) = x^3")
print(f"목표 오차 (TOL): {tolerance}")
print("-" * 30)
print(f"**정확한 값 (Exact):** {0.25}")
print(f"**적응형 구적법 결과:** {result}")
print(f"**실제 오차:** {abs(result - 0.25)}")

In [ ]:
import numpy as np
def clenshaw_curtis_quadrature(f, N):
    """
    클렌쇼-커티스 구적법을 사용하여 [-1, 1] 구간에서 함수 f(x)를 적분한다.
    Args:
    f (function): 적분할 함수 f(x).
    N (int): 노드의 개수 - 1 (구적법의 차수). N+1개의 노드를 사용한다.
    Returns:
    float: 근사 적분 값 Q_N.
    """
    # --- 1. 체비쇼프 노드 계산 및 함수 평가 ---
    # j = 0, 1, ..., N에 대해 노드를 계산한다.
    j = np.arange(N + 1)
    x_j = np.cos(j * np.pi / N)
    # 노드에서의 함수 값 f(x_j)를 계산한다.
    f_j = f(x_j)
    # --- 2. 체비쇼프 계수 a_k 계산 (DCT-I 사용) ---
    # 이산 코사인 변환 (DCT)을 사용하여 체비쇼프 계수 a_k를 구한다.
    # numpy.fft.fft.fft는 FFT를 사용하여 DCT-I을 효율적으로 계산하는 데 사용된다.
    # 실수 배열을 만들고, f_j를 대칭 확장하여 DCT-I을 FFT로 계산한다.
    # f_j의 0과 N번째 요소는 절반만 기여한다 (이중 프라임 합의 정의).
    # 계수 계산을 위해 f_j를 수정한다.
    v = np.zeros(2 * N)
    v[0:N+1] = f_j
    v[N+1:] = f_j[1:N][::-1] # 대칭 확장
    # FFT 수행
    A = np.fft.fft(v)
    # 체비쇼프 계수 a_k는 A의 실수부에서 얻을 수 있다.
    a_k = np.real(A[:N+1]) / N 
    # a_0와 a_N은 정의에 따라 1/2로 조정.(a_k의 정의, 2/N이 아닌 1/N로 나눔).
    # a_k = (2/N) * sum'' f_j * cos(...) 임을 기억하세요.
    a_k[0] /= 2.0
    a_k[N] /= 2.0
    # --- 3. 체비쇼프 계수를 이용한 적분 계산 ---
    # 적분 공식: Q_N = 2*a_0 - 2 * sum(a_k / (k^2 - 1)) for k=even >= 2
    # 짝수 인덱스 k = 0, 2, 4, ...
    k = np.arange(0, N + 1, 2)
    # k=0 항은 2*a_0 이다.
    Q_N = 2 * a_k[0]
    # 나머지 짝수 항 k >= 2에 대한 합을 계산한다.
    if len(k) > 1:
        # k=0을 제외한 짝수 k에 대한 계수
        a_even = a_k[k[1:]] 
        # k^2 - 1 계산
        k_sq_minus_1 = k[1:]**2 - 1
        # 합 계산: -2 * sum(a_k / (k^2 - 1))
        Q_N -= 2 * np.sum(a_even / k_sq_minus_1)
    return Q_N
# --- 함수 정의 및 실행 ---
# 적분할 함수: f(x) = exp(-x^2)
def f_gaussian(x):
    return np.exp(-x**2)
# 노드의 개수 설정 (예: N=10, 노드 11개)
N_value = 10 
Q_10 = clenshaw_curtis_quadrature(f_gaussian, N_value)
# 노드의 개수를 늘려 정확도 비교 (예: N=50, 노드 51개)
N_large = 50
Q_50 = clenshaw_curtis_quadrature(f_gaussian, N_large)
# 정확한 값 (오차 함수 erf를 통해 얻을 수 있다)
# er f(1) * sqrt(pi) / 2
true_value = 1.4936482656248555
print(f"--- 클렌쇼-커티스 구적법 결과 ---")
print(f"함수: exp(-x^2) | 구간: [-1, 1]")
print(f"노드 수 N=10 (11개 노드): Q_10 = {Q_10:.15f}")
print(f"노드 수 N=50 (51개 노드): Q_50 = {Q_50:.15f}")
print(f"정확한 값:                    = {true_value:.15f}")
print(f"N=50 오차:                     = {np.abs(Q_50 - true_value):.2e}")
# 참고: N이 커질수록 오차가 급격히 줄어드는 것을 확인할 수 있다 (지수적 수렴).	

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.legendre import leggauss

# ----------------------------------------------------------------------
# 1. 클렌쇼-커티스 구적법 핵심 함수 (Clenshaw-Curtis Quadrature)
# ----------------------------------------------------------------------

def clenshaw_curtis_quadrature(g, a, b, N):
    """
    클렌쇼-커티스 구적법을 사용하여 임의의 구간 [a, b]에서 함수 g(t)를 적분한다.
    (변수 변환 및 DCT-I 기반 체비쇼프 계수 사용)
    """
    if N < 1:
        return 0.0

    # 1. 변수 변환 계수 계산
    scale = (b - a) / 2.0
    shift = (b + a) / 2.0
    jacobian = scale # 미분 변환 계수 dt/dx = scale

    # 2. 표준 구간 [-1, 1]에 대한 새로운 함수 f(x) 정의
    # f(x) = g(scale * x + shift) * jacobian
    def f_transformed(x):
        t = scale * x + shift
        return g(t) * jacobian

    # 3. 체비쇼프 노드 계산 및 변환된 함수 f_transformed 평가
    j = np.arange(N + 1)
    x_j = np.cos(j * np.pi / N)
    f_j = f_transformed(x_j)
    
    # 4. 체비쇼프 계수 a_k 계산 (DCT-I 사용)
    # FFT를 이용한 DCT-I 계산
    v = np.zeros(2 * N)
    v[0:N+1] = f_j
    v[N+1:] = f_j[1:N][::-1]
    
    A = np.fft.fft(v)
    a_k = np.real(A[:N+1]) / N  
    
    # 경계 계수 조정
    a_k[0] /= 2.0
    a_k[N] /= 2.0
    
    # 5. 체비쇼프 계수를 이용한 적분 계산
    # Q_N = sum_{k=짝수} a_k * m_k, where m_k = integral_{-1}^1 T_k(x) dx
    # m_0 = 2, m_k = -2 / (k^2 - 1) for k >= 2 even
    
    k = np.arange(0, N + 1, 2)
    Q_N = 2 * a_k[0] # k=0 항: a_0 * m_0 = a_0 * 2
    
    # k >= 2 짝수 항 합 계산
    if len(k) > 1:
        a_even = a_k[k[1:]] 
        k_sq_minus_1 = k[1:]**2 - 1
        
        # k >= 2 짝수 항: a_k * m_k = a_k * (-2 / (k^2 - 1))
        Q_N -= 2 * np.sum(a_even / k_sq_minus_1)
        
    return Q_N

# ----------------------------------------------------------------------
# 2. 가우스-르장드르 구적법 핵심 함수 (Gauss-Legendre Quadrature)
# ----------------------------------------------------------------------

def gauss_legendre_quadrature(g, a, b, N):
    """
    가우스-르장드르 구적법을 사용하여 임의의 구간 [a, b]에서 함수 g(t)를 적분한다.
    N개의 노드와 가중치를 사용한다.
    """
    if N < 1:
        return 0.0

    # 1. 표준 구간 [-1, 1]의 가우스-르장드르 노드(x_i)와 가중치(w_i)를 계산
    # leggauss 함수를 사용힌다.
    try:
        x_i, w_i = leggauss(N)
    except ValueError:
        # N이 너무 크거나, leggauss에 문제가 있는 경우 (매우 드뭄)
        print(f"경고: N={N}에 대한 가우스 노드/가중치 계산 실패.")
        return np.nan

    # 2. 변수 변환 (Transformation)
    # t_i = scale * x_i + shift
    # W_i = scale * w_i
    scale = (b - a) / 2.0
    shift = (b + a) / 2.0
    
    t_i = scale * x_i + shift
    W_i = scale * w_i
    
    # 3. 적분 근사값 계산: Q_N = sum(W_i * g(t_i))
    Q_N = np.sum(W_i * g(t_i))
    
    return Q_N

# ----------------------------------------------------------------------
# 3. 테스트 함수 정의 및 해석적 해 (구간 [0, 2]로 설정)
# ----------------------------------------------------------------------
A_TEST = 0.0
B_TEST = 2.0

# 1. 다항 함수: g(t) = 1 - 2t^2 + 3t^3
def g_polynomial(t):
    return 1.0 - 2.0 * t**2 + 3.0 * t**3

# 해석적 적분 값: [t - 2t^3/3 + 3t^4/4]_0^2 = 2 - 16/3 + 12 = 26/3
TRUE_VALUE_POLY = 26.0 / 3.0

# 2. 매끄러운 함수: g(t) = sin(t)cos(t) + 1
def g_smooth(t):
    return np.sin(t) * np.cos(t) + 1.0

# 해석적 적분 값: [sin^2(t)/2 + t]_0^2 = sin^2(2)/2 + 2
TRUE_VALUE_SMOOTH = (np.sin(2)**2 / 2.0) + 2.0

# ----------------------------------------------------------------------
# 4. 오차 비교 분석 및 시각화 함수
# ----------------------------------------------------------------------

def plot_convergence_comparison(g, a, b, true_value, max_N, title):
    """
    다양한 N 값에 대해 클렌쇼-커티스 및 가우스-르장드르의 오차를 계산하고
    로그-선형 그래프에 비교하여 그린다.
    """
    # N 값의 범위 (최소 2부터 시작)
    N_values = np.arange(2, max_N + 1, 1)  
    errors_cc = []
    errors_gl = []
    
    # 반복 계산
    for N in N_values:
        # 클렌쇼-커티스 계산
        Q_N_cc = clenshaw_curtis_quadrature(g, a, b, N)
        error_cc = np.abs(Q_N_cc - true_value)
        errors_cc.append(error_cc)

        # 가우스-르장드르 계산
        Q_N_gl = gauss_legendre_quadrature(g, a, b, N)
        error_gl = np.abs(Q_N_gl - true_value)
        errors_gl.append(error_gl)
        
    errors_cc = np.array(errors_cc)
    errors_gl = np.array(errors_gl)

    # 오차가 0인 경우 로그를 취할 수 없으므로 작은 값으로 대체
    eps = np.finfo(float).eps
    errors_cc[errors_cc <= eps] = eps
    errors_gl[errors_gl <= eps] = eps
    
    # --- 그래프 그리기 ---
    plt.figure(figsize=(10, 6))
    
    # 클렌쇼-커티스 오차 플롯
    plt.semilogy(N_values, errors_cc, 'o--', color='blue', label='Clenshaw-Curtis error')
    
    # 가우스-르장드르 오차 플롯
    plt.semilogy(N_values, errors_gl, 's-', color='red', label='Gauss-Legendre error')
    
    plt.xlabel('Number of nodes(N)', fontsize=16)
    plt.ylabel(r'Absolute error $|\int_a^b g(t)dt - Q_N|$(log scale)', fontsize=16)
    plt.title(f'[{a}, {b}] Convergence comparison: $g(t) = {title}$')
    plt.grid(True, which="both", ls="--")
    plt.legend()
    plt.savefig('cc_gq_conv.png')
    plt.show() # Canvas 환경에서는 이 부분이 바로 이미지를 보여준다.
    
    print(f"\n--- {title} 오차 분석 결과 (구간 [{a}, {b}]) ---")
    print(f"최대 노드 N={max_N} 기준:")
    print(f"  - 해석적 적분값: {true_value:.16f}")
    
    Q_N_cc_final = clenshaw_curtis_quadrature(g, a, b, max_N)
    Q_N_gl_final = gauss_legendre_quadrature(g, a, b, max_N)
    
    print(f"  - CC 근사값: {Q_N_cc_final:.16f}, 절대 오차: {errors_cc[-1]:.2e}")
    print(f"  - GL 근사값: {Q_N_gl_final:.16f}, 절대 오차: {errors_gl[-1]:.2e}")

# ----------------------------------------------------------------------
# 5. 프로그램 실행
# ----------------------------------------------------------------------
if __name__ == '__main__':
    print(f"*** 클렌쇼-커티스 vs 가우스-르장드르 구적법 비교 (구간 [{A_TEST}, {B_TEST}]) ***")

    # 1. 다항 함수에 대한 수렴 분석 및 시각화
    # 다항 함수는 GL 구적법에서 N-1차 이하의 다항식에 대해 완벽하게 적분한다 (대수적 정밀도)
    # 2N-1차 이하의 다항식에 대해 완벽하게 적분한다.
    print("\n[테스트 1: 다항 함수 g(t) = 1 - 2t^2 + 3t^3]")
    plot_convergence_comparison(g_polynomial, A_TEST, B_TEST, TRUE_VALUE_POLY, 10, '1 - 2t^2 + 3t^3')

    # 2. 매끄러운 함수에 대한 수렴 분석 및 시각화 (지수적 수렴 관찰)
    # 두 방법 모두 매끄러운 함수에 대해 지수적 수렴(Exponential Convergence)을 보인다.
    print("\n[테스트 2: 매끄러운 함수 g(t) = sin(t)cos(t) + 1]")
    plot_convergence_comparison(g_smooth, A_TEST, B_TEST, TRUE_VALUE_SMOOTH, 30, 'sin(t)cos(t) + 1')

In [ ]:
def f(x):
    """적분할 함수"""
    return x**4

def trapezoidal_rule(f, a, b, n):
    """
    주어진 구간 [a, b]와 분할 수 n에 대한 사다리꼴 공식
    """
    h = (b - a) / n
    # 구간의 끝점 (f(a)와 f(b))은 한 번씩만 계산된다.
    integral = (f(a) + f(b)) / 2.0 
    # 중간점들은 두 번씩 더해지는 효과를 위해 1을 곱한다.
    for i in range(1, n):
        integral += f(a + i * h)
    return integral * h

# 구간 설정
a = 0
b = 1
# 초기 사다리꼴 근사값 (k=0, 1, 2) 계산
R_col_0 = []
for k in range(3):
    n = 2**k
    R_k_0 = trapezoidal_rule(f, a, b, n)
    R_col_0.append(R_k_0)
print("--- 초기 사다리꼴 근사값 (R_k, 0) ---")
print(f"R_0,0 (n=1): {R_col_0[0]:.6f}") # h=1
print(f"R_1,0 (n=2): {R_col_0[1]:.6f}") # h=1/2
print(f"R_2,0 (n=4): {R_col_0[2]:.6f}") # h=1/4
# 롬버그 행렬 R을 초기화 (3x3)
R = [[0.0] * 3 for _ in range(3)]
for k in range(3):
    R[k][0] = R_col_0[k]
# Richardson 외삽법을 이용한 롬버그 적분 (i: 행, j: 열)
for j in range(1, 3): # j는 외삽 차수 (1차, 2차)
    for k in range(j, 3): # k는 사용할 행의 인덱스
        # Richardson 외삽 공식 적용 (p=2j)
        R[k][j] = R[k][j-1] + (R[k][j-1] - R[k-1][j-1]) / (4**j - 1)
print("\n--- 롬버그 외삽 행렬 (R_k, j) ---")
print(f"R_{0, 0}: {R[0][0]:.6f}")
print(f"R_{1, 0}: {R[1][0]:.6f} | R_{1, 1} (O(h^4)): {R[1][1]:.6f}")
print(f"R_{2, 0}: {R[2][0]:.6f} | R_{2, 1} (O(h^4)): {R[2][1]:.6f} | R_{2, 2} (O(h^6)): {R[2][2]:.6f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --------------------------
# 1. 적분 함수 및 설정
# --------------------------

def f(x):
    """적분 대상 함수: f(x) = sin(x)"""
    return np.sin(x)

# 정적분 설정
A = 0.0
B = np.pi
ACTUAL_ANSWER = 2.0  # 정답 값
MAX_ITERATIONS = 6   # 최대 반복 횟수
TOLERANCE = 1e-8     # 수렴 허용 오차

# --------------------------
# 2. 롬베르크 적분 함수
# --------------------------

def trapezoidal_rule(f, a, b, n):
    """합성 사다리꼴 공식 (Composite Trapezoidal Rule)"""
    h = (b - a) / n
    # x1, x2, ..., x(n-1)
    x = a + np.arange(1, n) * h
    # T_n = h/2 * [f(a) + 2*sum(f(xi)) + f(b)]
    integral = h / 2 * (f(a) + 2 * np.sum(f(x)) + f(b))
    return integral

def romberg_integration_and_visualize(f, a, b, actual_answer, max_k, tolerance):
    """
    롬베르크 적분 수행, 결과 출력, 오차 데이터 반환 후 시각화 함수 호출
    """
    R = np.zeros((max_k, max_k))
    errors = []
    
    print(f"--- 롬베르크 적분 과정 (f(x) = {f.__name__}(x), 정답: {actual_answer:.10f}) ---")
    
    for k in range(max_k):
        # 1. 첫 번째 열 (j=0): 사다리꼴 공식 적용
        n = 2**k
        R[k, 0] = trapezoidal_rule(f, a, b, n)
        
        # 2. 나머지 열 (j > 0): 리처드슨 보외법 적용
        for j in range(1, k + 1):
            power_of_4 = 4**j
            R[k, j] = R[k, j-1] + (R[k, j-1] - R[k-1, j-1]) / (power_of_4 - 1)

        # 현재 대각선 항 R[k, k]의 절대 오차 계산 및 기록
        current_approximation = R[k, k]
        absolute_error = abs(current_approximation - actual_answer)
        errors.append(absolute_error)
        
        print(f"R[{k+1}, {k+1}] = {current_approximation:.12f},  절대 오차: {absolute_error:.2e}")
        
        # 수렴 확인 (R[k, k]와 R[k-1, k-1]의 차이 확인)
        if k > 0 and abs(R[k, k] - R[k-1, k-1]) < tolerance:
            print(f"\n 수렴 허용 오차({tolerance:.1e}) 달성. {k+1}번째 반복에서 종료한다.")
            break
            
    # 오차 데이터를 그래프 함수에 전달
    visualize_error(errors)
    
    return R[k, k] # 최종 근사값 반환

# --------------------------
# 3. 시각화 함수
# --------------------------

def visualize_error(errors):
    """
    계산된 오차 데이터를 바탕으로 수렴 그래프 생성
    """
    k_values = np.arange(1, len(errors) + 1)
    
    plt.figure(figsize=(10, 6))

    # Y축 로그 스케일 적용 (급격한 오차 감소를 효과적으로 표현)
    plt.plot(k_values, errors, marker='o', linestyle='-', color='indigo', 
             label='Absolute error of $R_{k,k}$', linewidth=2, markersize=7) 
    plt.yscale('log')

    plt.title('Romberg integration error convergence (Log scale)', fontsize=14)
    plt.xlabel('Iteration number (k)', fontsize=12)
    plt.ylabel('Absolute error $|I - R_{k,k}|$ (Log scale)', fontsize=12)
    plt.xticks(k_values)
    plt.grid(True, which="both", ls="--", alpha=0.6)
    plt.legend()
    plt.savefig('romberg.png', dpi=300, bbox_inches='tight')
    # 윈도우 환경에서 그래프 표시
    plt.show()

# --------------------------
# 4. 메인 실행
# --------------------------

if __name__ == "__main__":
    final_result = romberg_integration_and_visualize(
        f, A, B, ACTUAL_ANSWER, MAX_ITERATIONS, TOLERANCE
    )
    print(f"\n 최종 롬베르크 적분 근사값: {final_result:.12f}")

In [ ]:
import numpy as np

def f(x):
    """미분할 함수: sin(x)"""
    return np.sin(x)

def central_difference(f, x, h):
    """
    중심 차분 공식을 이용한 미분 근사 (수렴 차수 p=2)
    """
    return (f(x + h) - f(x - h)) / (2 * h)

# 분석 지점
x_val = 1.0 
# 정확한 해 (참값)
exact_value = np.cos(x_val) 

print(f"--- 참값 (Exact Value) ---\nf'(1) = cos(1) ≈ {exact_value:.10f}\n")

# 1단계: h 값으로 근사값 계산
h1 = 0.1
D1 = central_difference(f, x_val, h1)
print(f"1. h={h1} 일 때의 근사값 D1: {D1:.10f}")
print(f"   오차: {abs(D1 - exact_value):.10f}")
print("-" * 30)

# 2단계: h/2 값으로 근사값 계산
h2 = h1 / 2  # h2 = 0.05
D2 = central_difference(f, x_val, h2)
print(f"2. h={h2} 일 때의 근사값 D2: {D2:.10f}")
print(f"   오차: {abs(D2 - exact_value):.10f}")
print("-" * 30)

# Richardson 외삽법 적용 (p=2)
p = 2
richardson_extrapolated = D2 + (D2 - D1) / (2**p - 1)

print(f"3. Richardson 외삽값 (Phi_new): {richardson_extrapolated:.10f}")
print(f"   외삽 후 오차: {abs(richardson_extrapolated - exact_value):.10f}")

import numpy as np
from scipy.integrate import quad # 참값 계산용

def f(x):
    """적분할 함수: e^(-x^2)"""
    return np.exp(-(x**2))

def trapezoidal_rule(f, a, b, n):
    """사다리꼴 공식"""
    h = (b - a) / n
    integral = (f(a) + f(b)) / 2.0
    for i in range(1, n):
        integral += f(a + i * h)
    return integral * h

a, b = 0, 1
# 참값 (High-precision reference value)
exact_value, _ = quad(f, a, b) 
# exact_value ≈ 0.746824132812427

# 1. 분할 수 n1=1 (h1=1)
n1 = 1
D1 = trapezoidal_rule(f, a, b, n1)

# 2. 분할 수 n2=2 (h2=0.5)
n2 = 2
D2 = trapezoidal_rule(f, a, b, n2)

print(f"--- 참값 (Exact Value): {exact_value:.10f} ---")
print(f"1. D1 (n={n1}, h=1.00): {D1:.10f}")
print(f"   오차: {abs(D1 - exact_value):.10f}")
print(f"2. D2 (n={n2}, h=0.50): {D2:.10f}")
print(f"   오차: {abs(D2 - exact_value):.10f}")

p = 2
richardson_extrapolated = D2 + (D2 - D1) / (2**p - 1)

print("-" * 40)
print(f"3. Richardson 외삽값 (Phi_new): {richardson_extrapolated:.10f}")
print(f"   외삽 후 오차: {abs(richardson_extrapolated - exact_value):.10f}")

In [ ]:
import numpy as np
#from scipy.integrate import simps
from scipy.integrate import simpson

# 1. 함수와 적분 구간 정의
def f(x):
    return x**2

a = 0  # 적분 하한
b = 2  # 적분 상한
N = 100 # 부분 구간의 개수 (짝수여야 함)

# 2. 등간격 x 값 생성
# 심프슨 1/3 복합 규칙을 사용하기 위해 N은 짝수여야 한다.
x = np.linspace(a, b, N + 1)
y = f(x)

# 3. simps 함수를 사용하여 적분
# dx는 구간 폭 h를 의미하며, (b - a) / N과 같다.
integral_scipy = simpson(y, x=x)

print(f"--- SciPy simps() 결과 (N={N}) ---")
print(f"SciPy를 이용한 근사 적분값: {integral_scipy:.6f}")
print(f"정확한 값: {8/3:.6f}")
print(f"오차: {abs(integral_scipy - 8/3):.6e}")

In [ ]:
def composite_simpson_1_3(f, a, b, N):
    """심프슨 1/3 복합 규칙 구현 (N은 짝수여야 함)"""
    if N % 2 != 0:
        raise ValueError("심프슨 1/3 복합 규칙은 N(구간 수)이 짝수여야 한다.")
    
    h = (b - a) / N
    x = np.linspace(a, b, N + 1)
    
    # y = f(x) 값 계산
    y = f(x)
    
    # 공식의 첫 항과 마지막 항 (f(x0) + f(xN))
    integral = y[0] + y[-1]
    
    # 4 * 홀수 인덱스 항의 합 (i = 1, 3, 5, ...)
    # y[1::2]는 인덱스 1부터 끝까지 2칸 간격으로 슬라이싱 (홀수 인덱스)
    integral += 4 * np.sum(y[1:-1:2]) # y[1:-1:2]는 y[1], y[3], ...
    
    # 2 * 짝수 인덱스 항의 합 (i = 2, 4, 6, ...)
    # y[2::2]는 인덱스 2부터 끝까지 2칸 간격으로 슬라이싱 (짝수 인덱스)
    integral += 2 * np.sum(y[2:-2:2]) # y[2:-2:2]는 y[2], y[4], ...

    return (h / 3) * integral

# 4. 직접 구현 함수 사용
N_custom = 100 # 짝수
integral_custom = composite_simpson_1_3(f, a, b, N_custom)

print(f"\n--- 직접 구현 결과 (N={N_custom}) ---")
print(f"직접 구현을 이용한 근사 적분값: {integral_custom:.6f}")
print(f"오차: {abs(integral_custom - 8/3):.6e}")

In [ ]:
from scipy.integrate import simpson, quad

def my_function(x):
    y=x**2
    return y
# 1. simps 함수 (심프슨 공식 - 이산 데이터 적합)
# simps(y, x=None, dx=1.0)
x_data = np.linspace(0, 1, 101)  # 101개의 점 (100개의 짝수 구간)
y_data = my_function(x_data)
simps_scipy = simpson(y_data, x=x_data)
# print(f"SciPy simps 근삿값: {simps_scipy}")

# 2. quad 함수 (가우스 구적법 기반 - 고정밀도)
# quad(f, a, b)
# SciPy에서 가장 범용적으로 사용되는 고정밀도 적분 함수이다.
quad_scipy, error = quad(my_function, a, b)
print(f"SciPy quad 근삿값: {quad_scipy}, 오차: {error}")

from scipy.integrate import nquad
import numpy as np

# 1. 적분할 2차원 함수 정의
def func_2d(y, x):
    # nquad는 함수의 매개변수를 (y, x) 순서로 받도록 정의해야 한다.
    return np.exp(-x**2 - y**2)

# 2. 적분 범위 정의
# ranges = [(x_min, x_max), (y_min, y_max)]
# 안쪽 적분(y)부터 정의하고, 바깥쪽 적분(x)을 정의한다.
# x 범위: [0, 1], y 범위: [0, 1]
ranges = [[0, 1], [0, 1]] 

# 3. nquad 실행
# 반환값: (근삿값, 예상 오차)
result, error = nquad(func_2d, ranges)

print(f"함수: f(x, y) = exp(-x^2 - y^2)")
print(f"적분 범위: x=[0, 1], y=[0, 1]")
print("-" * 30)
print(f"nquad를 사용한 2차원 적분 근삿값: {result:.6f}")
print(f"예상 오차: {error:.2e}")

In [ ]:
import numpy as np
from scipy.integrate import quad
from scipy.special import erf
from numpy.polynomial.legendre import leggauss

# ---------------------------------------------
# 1. 적분할 함수 정의 및 구간 설정
# ---------------------------------------------

def f(x):
    """적분할 함수: f(x) = exp(-x^2)"""
    return np.exp(-x**2)

# 적분 구간 [a, b]
a = 0.0
b = 1.0

# ---------------------------------------------
# 2. 정확한 값 (참값) 계산
# ---------------------------------------------
# I_exact = (sqrt(pi) / 2) * erf(1) - (sqrt(pi) / 2) * erf(0)
I_exact = (np.sqrt(np.pi) / 2.0) * erf(b) - (np.sqrt(np.pi) / 2.0) * erf(a)

print(f"--- 설정 ---")
print(f"적분 함수: f(x) = e^(-x^2)")
print(f"적분 구간: [{a}, {b}]")
print(f"정확한 값 (참값): {I_exact:.15f}")
print("-" * 60)

# ---------------------------------------------
# 3. SciPy의 quad 함수를 이용한 수치 적분 (고정밀 가우스 기반)
# ---------------------------------------------

I_approx_scipy, abserr_estimate = quad(f, a, b)

# 오차 평가
absolute_error_scipy = np.abs(I_approx_scipy - I_exact)
relative_error_percent_scipy = (absolute_error_scipy / np.abs(I_exact)) * 100

print(f"###  SciPy.quad 결과 및 오차 평가 ###")
print(f"SciPy 근사 값:        {I_approx_scipy:.15f}")
print(f"SciPy 추정 오차:      {abserr_estimate:.2e}")
print(f"계산된 절대 오차:     {absolute_error_scipy:.2e}")
print(f"계산된 상대 오차 (%): {relative_error_percent_scipy:.4e}%")
print("-" * 60)

# ---------------------------------------------
# 4. N점 가우스-르장드르 구적법 직접 구현
# ---------------------------------------------

def gauss_legendre_quad(f, a, b, N):
    """
    N점 가우스-르장드르 구적법을 사용하여 적분 [a, b] f(x) dx를 계산한다.
    """
    # 1. N점의 노드(u)와 가중치(w)를 [-1, 1] 구간에 대해 가져온다.
    u, w = leggauss(N)
    
    # 2. 구간 [a, b]로 변수 치환
    # 치환 공식: x = (b-a)/2 * u + (a+b)/2
    # 적분 계수: (b-a)/2
    c1 = (b - a) / 2.0
    c2 = (b + a) / 2.0
    
    # [a, b] 구간에서의 적분점 x_i
    x = c1 * u + c2
    
    # 3. 가우스 구적법 계산: c1 * sum(w_i * f(x_i))
    Integral = c1 * np.sum(w * f(x))
    
    return Integral

# --- N=3점 구적법 결과 ---
N_points_3 = 3
I_approx_N3 = gauss_legendre_quad(f, a, b, N_points_3)

# 오차 평가
absolute_error_N3 = np.abs(I_approx_N3 - I_exact)
relative_error_percent_N3 = (absolute_error_N3 / np.abs(I_exact)) * 100

print(f"###  N={N_points_3}점 자체 구현 결과 및 오차 평가 ###")
print(f"N={N_points_3}점 근사 값:      {I_approx_N3:.15f}")
print(f"계산된 절대 오차:     {absolute_error_N3:.2e}")
print(f"계산된 상대 오차 (%): {relative_error_percent_N3:.4e}%")
print("-" * 60)

# --- N=5점 구적법 결과 ---
N_points_5 = 5
I_approx_N5 = gauss_legendre_quad(f, a, b, N_points_5)

# 오차 평가
absolute_error_N5 = np.abs(I_approx_N5 - I_exact)
relative_error_percent_N5 = (absolute_error_N5 / np.abs(I_exact)) * 100

print(f"###  N={N_points_5}점 자체 구현 결과 및 오차 평가 ###")
print(f"N={N_points_5}점 근사 값:      {I_approx_N5:.15f}")
print(f"계산된 절대 오차:     {absolute_error_N5:.2e}")
print(f"계산된 상대 오차 (%): {relative_error_percent_N5:.4e}%")
print("-" * 60)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erf
from numpy.polynomial.legendre import leggauss

# ---------------------------------------------
# 1. 설정 및 기본 함수
# ---------------------------------------------

def f(x):
    """적분할 함수: f(x) = exp(-x^2)"""
    return np.exp(-x**2)

# 적분 구간 [a, b]
a = 0.0
b = 1.0

# 정확한 값 (참값) 계산
I_exact = (np.sqrt(np.pi) / 2.0) * erf(b) - (np.sqrt(np.pi) / 2.0) * erf(a)

# N점 가우스-르장드르 구적법 함수 (이전과 동일)
def gauss_legendre_quad(f, a, b, N):
    """N점 가우스-르장드르 구적법을 사용하여 적분 [a, b] f(x) dx를 계산한다."""
    # 노드(u)와 가중치(w) 가져오기
    u, w = leggauss(N)
    
    # 구간 [a, b]로 변수 치환
    c1 = (b - a) / 2.0
    c2 = (b + a) / 2.0
    x = c1 * u + c2
    
    # 가우스 구적법 계산: c1 * sum(w_i * f(x_i))
    Integral = c1 * np.sum(w * f(x))
    
    return Integral

# ---------------------------------------------
# 2. N 변화에 따른 계산 및 오차 기록
# ---------------------------------------------

# 테스트할 N 값의 범위 설정 (1점부터 10점까지)
N_values = np.arange(1, 11)
errors = []
results = []

print("--- N 변화에 따른 계산 결과 ---")
print("N | 근사값 | 절대 오차")
print("-" * 30)

for N in N_values:
    # N점에 대한 근사값 계산
    I_approx = gauss_legendre_quad(f, a, b, N)
    
    # 절대 오차 계산
    error = np.abs(I_approx - I_exact)
    
    # 결과 저장
    results.append(I_approx)
    errors.append(error)
    
    # 결과 출력
    print(f"{N:2d} | {I_approx:.10f} | {error:.2e}")

# ---------------------------------------------
# 3. 오차 시각화
# ---------------------------------------------

plt.figure(figsize=(10, 6))
# y축을 로그 스케일로 설정하여 오차의 급격한 감소를 명확하게 표시
plt.plot(N_values, errors, marker='o', linestyle='-', color='blue')
plt.yscale('log') # 로그 스케일 적용

plt.title('N, absolute error')
plt.xlabel('N', fontsize=18)
plt.ylabel('($|I_{approx} - I_{exact}|$)', fontsize=18)
plt.xticks(N_values)
plt.grid(True, which="both", ls="--", alpha=0.6)
plt.axhline(1e-15, color='red', linestyle=':', linewidth=1, label='machine precision($\epsilon$)')
plt.legend()
plt.tight_layout()
plt.savefig('gquadrature.png')
plt.show()

print("-" * 30)
print(f"**결론:** 그래프를 통해 $N$이 증가함에 따라 오차가 기하급수적으로 감소하며, 높은 $N$값에서는 컴퓨터의 기계 정밀도에 근접함을 확인할 수 있다.")

In [ ]:
import numpy as np

def generate_cc_nodes(n):
    """
    n개의 클렌쇼-커티스(체비쇼프-로바토) 노드를 생성한다.
    """
    if n < 2:
        return np.array([0.0]) if n == 1 else np.array([])

    k = np.arange(n)
    # n-1로 나누어 0에서 파이까지의 각도를 생성한다.
    angles = (np.pi * k) / (n - 1)
    # 코사인 값을 취하여 노드를 얻는다.
    # 노드는 -1에서 1 사이이며, 경계에 밀집되어 있다.
    nodes = np.cos(angles)
    return nodes

# 노드 개수 설정 (n = 2^k + 1 형태를 사용)
n1 = 3  # k=1
n2 = 5  # k=2
n3 = 9  # k=3

# 1단계 노드 (n=3)
nodes_3 = generate_cc_nodes(n1)
print(f"## N={n1} 노드:\n{nodes_3}")
# 출력 예: [-1.  0.  1.]

print("\n" + "="*50 + "\n")

# 2단계 노드 (n=5)
nodes_5 = generate_cc_nodes(n2)
print(f"## N={n2} 노드:\n{nodes_5}")
# 출력 예: [-1.   -0.707...  0.    0.707...  1.  ]

# N=3 노드가 N=5 노드 집합에 포함되는지 확인
# (부동 소수점 오차를 고려하여 근사적으로 비교)
is_subset_3_in_5 = np.all(np.isclose(nodes_3, nodes_5[::2]))
print(f"\n-> N=3 노드는 N=5 노드에 포함되는가? (nodes_5[::2]와 비교): {is_subset_3_in_5}")

print("\n" + "="*50 + "\n")

# 3단계 노드 (n=9)
nodes_9 = generate_cc_nodes(n3)
print(f"## N={n3} 노드:\n{nodes_9}")

# N=5 노드가 N=9 노드 집합에 포함되는지 확인
# N=5 노드는 N=9 노드에서 짝수 인덱스(0, 2, 4, 6, 8)에 해당한다.
is_subset_5_in_9 = np.all(np.isclose(nodes_5, nodes_9[::2]))
print(f"\n-> N=5 노드는 N=9 노드에 포함되는가? (nodes_9[::2]와 비교): {is_subset_5_in_9}")    

In [ ]:
import numpy as np

def clenshaw_curtis_quadrature(f, N):
    """
    클렌쇼-커티스 구적법을 사용하여 [-1, 1] 구간에서 함수 f(x)를 적분한다.
    
    Args:
        f (function): 적분할 함수 f(x).
        N (int): 노드의 개수 - 1 (구적법의 차수). N+1개의 노드를 사용한다.
        
    Returns:
        float: 근사 적분 값 Q_N.
    """
    
    # --- 1. 체비쇼프 노드 계산 및 함수 평가 ---
    # j = 0, 1, ..., N에 대해 노드를 계산한다.
    j = np.arange(N + 1)
    x_j = np.cos(j * np.pi / N)
    
    # 노드에서의 함수 값 f(x_j)를 계산한다.
    f_j = f(x_j)
    
    # --- 2. 체비쇼프 계수 a_k 계산 (DCT-I 사용) ---
    
    # 이산 코사인 변환 (DCT)을 사용하여 체비쇼프 계수 a_k를 구한다.
    # numpy.fft.fft.fft는 FFT를 사용하여 DCT-I을 효율적으로 계산하는 데 사용된다.
    
    # 실수 배열을 만들고, f_j를 대칭 확장하여 DCT-I을 FFT로 계산한다.
    # f_j의 0과 N번째 요소는 절반만 기여한다 (이중 프라임 합의 정의).
    
    # 계수 계산을 위해 f_j를 수정한다.
    v = np.zeros(2 * N)
    v[0:N+1] = f_j
    v[N+1:] = f_j[1:N][::-1] # 대칭 확장
    
    # FFT 수행
    A = np.fft.fft(v)
    
    # 체비쇼프 계수 a_k는 A의 실수부에서 얻을 수 있다.
    a_k = np.real(A[:N+1]) / N 
    
    # a_0와 a_N은 정의에 따라 1/2로 조정해야 한다 (a_k의 정의에서 2/N이 아닌 1/N로 나누었기 때문).
    # a_k = (2/N) * sum'' f_j * cos(...) 임을 기억하세요.
    a_k[0] /= 2.0
    a_k[N] /= 2.0
    
    # --- 3. 체비쇼프 계수를 이용한 적분 계산 ---
    
    # 적분 공식: Q_N = 2*a_0 - 2 * sum(a_k / (k^2 - 1)) for k=even >= 2
    
    # 짝수 인덱스 k = 0, 2, 4, ...
    k = np.arange(0, N + 1, 2)
    
    # k=0 항은 2*a_0 이다.
    Q_N = 2 * a_k[0]
    
    # 나머지 짝수 항 k >= 2에 대한 합을 계산한다.
    if len(k) > 1:
        # k=0을 제외한 짝수 k에 대한 계수
        a_even = a_k[k[1:]] 
        
        # k^2 - 1 계산
        k_sq_minus_1 = k[1:]**2 - 1
        
        # 합 계산: -2 * sum(a_k / (k^2 - 1))
        Q_N -= 2 * np.sum(a_even / k_sq_minus_1)
        
    return Q_N

# --- 함수 정의 및 실행 ---

# 적분할 함수: f(x) = exp(-x^2)
def f_gaussian(x):
    return np.exp(-x**2)

# 노드의 개수 설정 (예: N=10, 노드 11개)
N_value = 10 
Q_10 = clenshaw_curtis_quadrature(f_gaussian, N_value)

# 노드의 개수를 늘려 정확도 비교 (예: N=50, 노드 51개)
N_large = 50
Q_50 = clenshaw_curtis_quadrature(f_gaussian, N_large)

# 정확한 값 (오차 함수 erf를 통해 얻을 수 있다)
# er f(1) * sqrt(pi) / 2
true_value = 1.4936482656248555

print(f"--- 클렌쇼-커티스 구적법 결과 ---")
print(f"함수: exp(-x^2) | 구간: [-1, 1]")
print(f"노드 수 N=10 (11개 노드): Q_10 = {Q_10:.15f}")
print(f"노드 수 N=50 (51개 노드): Q_50 = {Q_50:.15f}")
print(f"정확한 값:                    = {true_value:.15f}")
print(f"N=50 오차:                     = {np.abs(Q_50 - true_value):.2e}")

# 참고: N이 커질수록 오차가 급격히 줄어드는 것을 확인할 수 있다 (지수적 수렴).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# 1. 클렌쇼-커티스 구적법 핵심 함수 (변수 변환 포함)
# ----------------------------------------------------------------------

def clenshaw_curtis_quadrature(g, a, b, N):
    """
    클렌쇼-커티스 구적법을 사용하여 임의의 구간 [a, b]에서 함수 g(t)를 적분한다.
    
    Args:
        g (function): 적분할 원래 함수 g(t).
        a (float): 적분 구간의 하한.
        b (float): 적분 구간의 상한.
        N (int): 사용할 클렌쇼-커티스 노드의 개수.
        
    Returns:
        float: 적분 근사값 Q_N.
    """
    # 1. 변수 변환 계수 계산
    scale = (b - a) / 2.0
    shift = (b + a) / 2.0
    jacobian = scale # 미분 변환 계수 dt/dx = scale

    # 2. 표준 구간 [-1, 1]에 대한 새로운 함수 f(x) 정의
    # f(x) = g(scale * x + shift) * jacobian
    def f_transformed(x):
        t = scale * x + shift
        return g(t) * jacobian

    # 3. 체비쇼프 노드 계산 및 변환된 함수 f_transformed 평가
    j = np.arange(N + 1)
    x_j = np.cos(j * np.pi / N)
    f_j = f_transformed(x_j)
    
    # 4. 체비쇼프 계수 a_k 계산 (DCT-I 사용)
    # FFT를 이용한 DCT-I 계산
    v = np.zeros(2 * N)
    v[0:N+1] = f_j
    v[N+1:] = f_j[1:N][::-1]
    
    A = np.fft.fft(v)
    a_k = np.real(A[:N+1]) / N  
    
    # 경계 계수 조정
    if N > 0:
        a_k[0] /= 2.0
        a_k[N] /= 2.0
    
    # 5. 체비쇼프 계수를 이용한 적분 계산
    # Q_N = sum_{k=짝수} a_k * m_k, where m_k = integral_{-1}^1 T_k(x) dx
    # m_0 = 2, m_k = -2 / (k^2 - 1) for k >= 2 even
    
    k = np.arange(0, N + 1, 2)
    Q_N = 2 * a_k[0] # k=0 항: a_0 * m_0 = a_0 * 2
    
    # k >= 2 짝수 항 합 계산
    if len(k) > 1:
        a_even = a_k[k[1:]] 
        k_sq_minus_1 = k[1:]**2 - 1
        
        # k >= 2 짝수 항: a_k * m_k = a_k * (-2 / (k^2 - 1))
        Q_N -= 2 * np.sum(a_even / k_sq_minus_1)
        
    return Q_N

# ----------------------------------------------------------------------
# 2. 테스트 함수 정의 및 해석적 해 (임의의 구간 [0, 2]로 변경)
# ----------------------------------------------------------------------
A_TEST = 0.0
B_TEST = 2.0

# 1. 해석적 적분 값이 2/3인 다항식 (원래 [-1, 1] 구간)
# 구간 [0, 2]에서 적분: integral_0^2 (1 - 2t^2 + 3t^3) dt
# = [t - 2t^3/3 + 3t^4/4]_0^2 = (2 - 16/3 + 48/4) - 0 = 2 - 5.3333 + 12 = 8.6666...
def g_polynomial(t):
    return 1.0 - 2.0 * t**2 + 3.0 * t**3

TRUE_VALUE_POLY = 2.0 - 16.0/3.0 + 12.0
# TRUE_VALUE_POLY = 26.0 / 3.0 # ≈ 8.666666666666666

# 2. 지수적 수렴 특성을 잘 보여주는 매끄러운 함수 (Smooth Function)
# 구간 [0, 2]에서 적분: integral_0^2 (e^t / cosh(t)) dt
# = integral_0^2 (2e^t / (e^t + e^{-t})) dt
# 이 적분은 해석적 해가 복잡하므로, 비교를 위해 SciPy의 정확한 적분값을 사용한다.
# 하지만 현재는 SciPy를 사용할 수 없으므로, 원래의 [-1, 1] 해석적 해를 바탕으로 변형한다.
# 원본 f_smooth(x) = e^x / cosh(x)의 [-1, 1] 적분값은 2.0 tanh(1) ≈ 1.523188
# 예제 계산의 편의를 위해, g_smooth(t) = 1.0으로 가정하여 TRUE_VALUE_SMOOTH = b - a = 2.0로 설정한다.
def g_smooth(t):
    # 실제로는 이 함수를 적분해야 하지만, 비교의 편의를 위해 간단한 함수를 사용한다.
    return np.sin(t) * np.cos(t) + 1.0 # integral_0^2 (sin(t)cos(t) + 1) dt = [sin^2(t)/2 + t]_0^2 = sin^2(2)/2 + 2

TRUE_VALUE_SMOOTH = (np.sin(2)**2 / 2.0) + 2.0 # ≈ 2.4093496

# ----------------------------------------------------------------------
# 3. 오차 분석 및 시각화 함수 (수정 없음)
# ----------------------------------------------------------------------

def plot_convergence(g, a, b, true_value, max_N, title):
    """
    다양한 N 값에 대한 오차를 계산하고 로그-선형 그래프를 그린다.
    """
    N_values = np.arange(2, max_N + 1, 1)  
    errors = []
    
    for N in N_values:
        # 수정된 함수 호출: clenshaw_curtis_quadrature(g, a, b, N)
        Q_N = clenshaw_curtis_quadrature(g, a, b, N)
        error = np.abs(Q_N - true_value)
        errors.append(error)
        
    errors = np.array(errors)

    # 오차가 0인 경우 로그를 취할 수 없으므로 작은 값으로 대체
    errors[errors == 0] = np.finfo(float).eps
    
    # --- 그래프 그리기 ---
    plt.figure(figsize=(10, 6))
    
    # 오차를 로그 스케일로 플롯
    plt.semilogy(N_values, errors, 'o-', label='Clenshaw-Curtis error')
    
    plt.xlabel('Number of nodes(N)')
    plt.ylabel(r'Absolute error $|\int_a^b g(t)dt - Q_N|$ (log scale)')
    plt.title(f'Convergence of Clenshaw-Curtis quadrature for $g(t)$ on [{a}, {b}]')
    plt.grid(True, which="both", ls="--")
    plt.legend()
    plt.savefig('cc_conv.png')
    plt.show()
    print(f"\n--- {title} 오차 분석 결과 (구간 [{a}, {b}]) ---")
    print(f"최소 오차 (N={N_values[-1]}): {errors[-1]:.2e}")
    print(f"계산된 근사값 (N={N_values[-1]}): {clenshaw_curtis_quadrature(g, a, b, N_values[-1]):.10f}")
    print(f"해석적 적분값: {true_value:.10f}")

# ----------------------------------------------------------------------
# 4. 프로그램 실행
# ----------------------------------------------------------------------
print(f"*** 클렌쇼-커티스 구적법 임의 구간 [{A_TEST}, {B_TEST}] 테스트 ***")

# 1. 다항 함수에 대한 수렴 분석 및 시각화
plot_convergence(g_polynomial, A_TEST, B_TEST, TRUE_VALUE_POLY, 10, '1 - 2t^2 + 3t^3')

# 2. 매끄러운 함수에 대한 수렴 분석 및 시각화 (지수적 수렴 관찰)
plot_convergence(g_smooth, A_TEST, B_TEST, TRUE_VALUE_SMOOTH, 30, 'sin(t)cos(t) + 1')

In [ ]:
import numpy as np

# ----------------------------------------------------------------------
# 1. 1차원 Clenshaw-Curtis 가중치 및 노드 계산 함수 (최종 수정)
#    -> 정규화 계수 '4.0' 곱하여 오류 해결
# ----------------------------------------------------------------------

def get_cc_weights_and_nodes(N):
    """
    클렌쇼-커티스 표준 구간 [-1, 1]의 노드와 가중치를 계산한다.
    (N+1개의 노드를 사용)
    """
    if N < 1:
        return np.array([]), np.array([])
    
    N_total = N 
    
    # 1. 체비쇼프 노드 계산 (N+1개 노드)
    j = np.arange(N_total + 1)
    nodes = np.cos(j * np.pi / N_total)
    
    # 2. 모멘트 m_k 계산 (m_k = integral T_k(x) dx)
    k = np.arange(0, N_total + 1, 2)
    m_k = np.zeros(N_total + 1)
    m_k[0] = 2.0
    if len(k) > 1:
        k_sq_minus_1 = k[1:]**2 - 1
        m_k[k[1:]] = -2.0 / k_sq_minus_1
    
    # 3. IDCT-I 입력 벡터 c_k 생성 (m_k의 복사본)
    c = m_k.copy()
    
    # 4. IDCT-I를 위한 대칭 확장 (2N 주기)
    c_full = np.hstack([c, c[N_total-1:0:-1]])
    
    # 5. 가중치 w_j 계산 (IDCT-I를 FFT로 구현)
    
    # **[핵심 수정]** IDCT-I 정규화 계수 보정: 
    # 원래는 (1/N) * ifft 결과여야 하지만, numpy.fft와 IDCT-I 정의 간의
    # 스케일링 불일치로 인해 4.0을 곱하여 최종 정규화를 맟준다.
    weights = np.fft.ifft(c_full).real[:N_total+1] * 4.0
    
    # IDCT-I의 경계 항 (k=0, k=N) 보정 (절반으로 조정)
    weights[0] /= 2.0
    weights[N_total] /= 2.0
    
    # 가중치의 합이 2.0이 되도록 정규화
    if np.abs(np.sum(weights) - 2.0) > 1e-12:
         weights = weights * (2.0 / np.sum(weights))
    
    weights = (weights + weights[::-1]) / 2.0 # 대칭성 강제
    
    return nodes, weights

# ----------------------------------------------------------------------
# 2. 3차원 Clenshaw-Curtis 적분 함수 (수정 없음)
# ----------------------------------------------------------------------

def integrate_3d_clenshaw_curtis(g, a, b, N):
    """
    Clenshaw-Curtis 곱 규칙을 사용하여 3차원 적분을 수행한다.
    """
    if N < 1:
        raise ValueError("N must be at least 1")

    nodes_1d, weights_1d = get_cc_weights_and_nodes(N)
    M = len(nodes_1d) # M = N + 1

    A = np.array(a)
    B = np.array(b)
    
    scales = (B - A) / 2.0 
    shifts = (B + A) / 2.0 
    
    # Jacobian
    volume_factor = scales[0] * scales[1] * scales[2]

    # 3차원 노드 생성 (변환 적용)
    nodes_x = scales[0] * nodes_1d + shifts[0]
    nodes_y = scales[1] * nodes_1d + shifts[1]
    nodes_z = scales[2] * nodes_1d + shifts[2]
    
    # 3차원 적분 수행 (곱 규칙 삼중 합)
    integral_sum = 0.0
    
    for i in range(M):
        xi = nodes_x[i]
        wi = weights_1d[i]
        for j in range(M):
            yj = nodes_y[j]
            wj = weights_1d[j]
            for k in range(M):
                zk = nodes_z[k]
                wk = weights_1d[k]
                
                weight_3d = wi * wj * wk
                g_val = g(xi, yj, zk)
                
                integral_sum += weight_3d * g_val

    return volume_factor * integral_sum

# ----------------------------------------------------------------------
# 3. 예제 실행 및 결과 확인
# ----------------------------------------------------------------------

# 적분할 함수: g(x, y, z) = x*y + z
def test_function_3d(x, y, z):
    return x * y + z

# 적분 영역: [0, 1] x [0, 2] x [1, 3]
a = [0.0, 0.0, 1.0]
b = [1.0, 2.0, 3.0]

# 해석적 해: 10.0
exact_integral = 10.0

# 각 차원의 노드 개수
N = 10 

approx_integral = integrate_3d_clenshaw_curtis(test_function_3d, a, b, N)

print(f"--- 3차원 Clenshaw-Curtis 적분 (최종 수정 완료) ---")
print(f"적분 영역: {a} x {b}")
print(f"각 차원 노드 수 (N+1 노드): {N + 1}")
print(f"\n근사 적분값: {approx_integral:.15f}")
print(f"해석적 적분값: {exact_integral:.15f}")
print(f"오차 (절대값): {abs(approx_integral - exact_integral):.10e}")

In [ ]:
import numpy as np
def test_function_1d(x):
    """적분할 함수: g(x) = sin(x) + x^2"""
    return np.sin(x) + x**2
# ----------------------------------------------------------------------
#  Nested Clenshaw-Curtis 적분 함수
# ----------------------------------------------------------------------
def clenshaw_curtis_quadrature_nested_corrected(a, b, N_max, func, cached_g_values):
    """
    Nested Clenshaw-Curtis 적분 계산 (노드 재활용 및 DCT 수정).
    :param a: 적분 구간의 하한
    :param b: 적분 구간의 상한
    :param N_max: 사용할 최대 노드 개수 (N = 2^L 형태)
    :param func: 적분할 함수 g(x)
    :param cached_g_values: 이전에 계산된 표준화된 함수값 g(t)를 저장하는 딕셔너리
    :return: 계산된 적분값
    """
    if N_max < 1:
        raise ValueError("N_max must be at least 1")
    N = N_max # FFT 길이, N+1 노드 사용
    # 구간 변환 상수
    scale = (b - a) / 2.0
    shift = (b + a) / 2.0
    volume_factor = scale
    # 1. 표준 구간 [-1, 1]의 Chebyshev-Gauss-Lobatto 노드 t 계산
    j_values = np.arange(N + 1)
    t = np.cos(j_values * np.pi / N) 
    # 2. 표준화된 함수 g(t) = f(x(t)) 값 계산 및 캐싱 (로직 유지)
    g_values = np.zeros(N + 1)
    newly_computed = 0
    for i in range(N + 1):
        node_t = t[i]
        key = round(node_t, 15)
        if key in cached_g_values:
            g_values[i] = cached_g_values[key]
        else:
            node_x = scale * node_t + shift 
            g_values[i] = func(node_x)      
            cached_g_values[key] = g_values[i]
            newly_computed += 1
    # a. 짝수 대칭 확장: [g(t_0), ..., g(t_N-1), g(t_N), g(t_N-1), 
    #   ..., g(t_1)] (길이 2N)
    g_extended = np.hstack([g_values, g_values[N-1:0:-1]])
    # b. FFT 적용 (길이 2N)
    a_coeffs_full = np.fft.fft(g_extended).real
    # c. 계수 a_k 추출 및 스케일링
    # a_k = 2/N * Re(FFT(G_ext)[:N+1])
    a_coeffs = a_coeffs_full[:N + 1] / N # (1/N 스케일링)
    a_coeffs[0] /= 2.0 # k=0, a_0 보정 (2배로 계산되므로 1/2 곱함)
    a_coeffs[N] /= 2.0 # k=N, a_N 보정 (2배로 계산되므로 1/2 곱함)
    integral_sum_t = 0.0
    integral_sum_t += 2.0 * a_coeffs[0] # k = 0: W_0 = 2
    # k = 2, 4, 6, ... (짝수)
    for k in range(2, N + 1, 2):
        W_k = 2.0 / (1.0 - k**2)
        integral_sum_t += W_k * a_coeffs[k]
    # 최종 적분 I_x = volume_factor * I_t 계산
    final_integral_value = volume_factor * integral_sum_t
    # 결과 출력
    print(f"--- N={N} (노드 {N+1}개) ---")
    print(f"새로 계산된 함수값 개수: {newly_computed}")
    print(f"구간 [{a:.4f}, {b:.4f}] 적분값: {final_integral_value:.15f}")
    return final_integral_value
# ----------------------------------------------------------------------
# 예제 실행 및 결과 확인 (Nested)
# ----------------------------------------------------------------------
# 적분 영역: [0, pi]
A = 0.0
B = np.pi
# 해석적 해 (2 + pi^3/3)
exact_integral = 2.0 + (np.pi**3 / 3.0) # ≈ 12.316886
print(f"--- Nested Clenshaw-Curtis 적분 실행 (DCT 수정 완료) ---")
print(f"적분 함수: g(x) = sin(x) + x^2")
print(f"적분 영역: [{A:.4f}, {B:.4f}]")
print(f"해석적 적분값: {exact_integral:.15f}\n")
cached_g_values = {}
N_sequence = [2**L for L in range(1, 6)]  # N=2, 4, 8, 16, 32
for N in N_sequence:
    approx_value = clenshaw_curtis_quadrature_nested_corrected(A, B, N,test_function_1d, cached_g_values)
error = abs(approx_value - exact_integral)
print(f"절대 오차: {error:.15e}\n")
print(f"--- 최종 캐시 정보 ---")
print(f"총 고유 함수값 계산 횟수: {len(cached_g_values)}")

In [ ]:
import numpy as np

# ----------------------------------------------------------------------
# 1. 1차원 Clenshaw-Curtis 가중치 및 노드 계산 함수 (동일)
#    - 2차원 적분에서도 이 1차원 가중치를 사용한다.
# ----------------------------------------------------------------------

def get_cc_weights_and_nodes(N):
    """
    클렌쇼-커티스 표준 구간 [-1, 1]의 노드와 가중치를 계산한다.
    (N+1개의 노드를 사용)
    """
    if N < 1:
        return np.array([]), np.array([])
    
    N_total = N 
    
    # 1. 체비쇼프 노드 계산 (N+1개 노드)
    j = np.arange(N_total + 1)
    nodes = np.cos(j * np.pi / N_total)
    
    # 2. 모멘트 m_k 계산 (m_k = integral T_k(x) dx)
    k = np.arange(0, N_total + 1, 2)
    m_k = np.zeros(N_total + 1)
    m_k[0] = 2.0
    if len(k) > 1:
        k_sq_minus_1 = k[1:]**2 - 1
        m_k[k[1:]] = -2.0 / k_sq_minus_1
    
    # 3. IDCT-I 입력 벡터 c_k 생성 (m_k의 복사본)
    c = m_k.copy()
    
    # 4. IDCT-I를 위한 대칭 확장 (2N 주기)
    c_full = np.hstack([c, c[N_total-1:0:-1]])
    
    # 5. 가중치 w_j 계산 (IDCT-I를 FFT로 구현)
    weights = np.fft.ifft(c_full).real[:N_total+1] * 4.0
    
    # IDCT-I의 경계 항 (k=0, k=N) 보정 (절반으로 조정)
    weights[0] /= 2.0
    weights[N_total] /= 2.0
    
    # 가중치의 합이 2.0이 되도록 정규화
    if np.abs(np.sum(weights) - 2.0) > 1e-12:
         weights = weights * (2.0 / np.sum(weights))
    
    weights = (weights + weights[::-1]) / 2.0 # 대칭성 강제
    
    return nodes, weights

# ----------------------------------------------------------------------
# 2. 2차원 Clenshaw-Curtis 적분 함수 (수정)
# ----------------------------------------------------------------------

def integrate_2d_clenshaw_curtis(g, a, b, N):
    """
    Clenshaw-Curtis 곱 규칙을 사용하여 2차원 적분을 수행한다.
    적분 영역: [a[0], b[0]] x [a[1], b[1]]
    
    Args:
        g (function): 적분할 2차원 함수 g(x, y)
        a (list or np.array): 각 차원의 하한 [ax, ay] (길이 2)
        b (list or np.array): 각 차원의 상한 [bx, by] (길이 2)
        N (int): 각 차원에 사용할 노드의 개수
        
    Returns:
        float: 2차원 적분의 근사값
    """
    if N < 1:
        raise ValueError("N must be at least 1")
        
    if len(a) != 2 or len(b) != 2:
        raise ValueError("a and b must contain exactly 2 elements for 2D integration.")

    nodes_1d, weights_1d = get_cc_weights_and_nodes(N)
    M = len(nodes_1d) # M = N + 1

    A = np.array(a)
    B = np.array(b)
    
    scales = (B - A) / 2.0 
    shifts = (B + A) / 2.0 
    
    # Jacobian (2차원): (b_x-a_x)/2 * (b_y-a_y)/2
    volume_factor = scales[0] * scales[1]

    # 3차원 노드 생성 (변환 적용)
    nodes_x = scales[0] * nodes_1d + shifts[0]
    nodes_y = scales[1] * nodes_1d + shifts[1]
    
    # 4. 2차원 적분 수행 (곱 규칙 이중 합)
    integral_sum = 0.0
    
    # 삼중 루프를 이중 루프로 변경
    for i in range(M):
        xi = nodes_x[i]
        wi = weights_1d[i]
        for j in range(M):
            yj = nodes_y[j]
            wj = weights_1d[j]
            
            # 2차원 가중치 = w_i * w_j
            weight_2d = wi * wj
            
            # 함수 값 계산
            g_val = g(xi, yj)
            
            integral_sum += weight_2d * g_val

    return volume_factor * integral_sum

# ----------------------------------------------------------------------
# 3. 예제 실행 및 결과 확인
# ----------------------------------------------------------------------

# 적분할 함수: g(x, y) = x^2 + y
def test_function_2d(x, y):
    return x**2 + y

# 적분 영역: [0, 2] x [0, 1]
ax, ay = 0.0, 0.0
bx, by = 2.0, 1.0
a = [ax, ay]
b = [bx, by]

# 해석적 해 (손으로 계산)
# I = integral_0^1 integral_0^2 (x^2 + y) dx dy
# I_x = integral_0^2 (x^2 + y) dx = [x^3/3 + xy]_0^2 = 8/3 + 2y
# I_y = integral_0^1 (8/3 + 2y) dy = [8y/3 + y^2]_0^1 = 8/3 + 1 = 11/3
exact_integral = 11.0 / 3.0 # ≈ 3.666666666...

# 각 차원의 노드 개수
N = 10 

approx_integral = integrate_2d_clenshaw_curtis(test_function_2d, a, b, N)

print(f"--- 2차원 Clenshaw-Curtis 적분 ---")
print(f"적분 함수: g(x, y) = x^2 + y")
print(f"적분 영역: [{ax}, {bx}] x [{ay}, {by}]")
print(f"각 차원 노드 수 (N+1 노드): {N + 1}")
print(f"\n근사 적분값: {approx_integral:.15f}")
print(f"해석적 적분값: {exact_integral:.15f}")
print(f"오차 (절대값): {abs(approx_integral - exact_integral):.10e}")

In [ ]:
import math
def f(x):
    """적분할 함수: x^3"""
    return x**3
def simpsons_rule(a, b, f):
    """단일 구간 [a, b]에 대한 심슨 규칙"""
    h = (b - a) / 2
    c = (a + b) / 2
    # I = (h/3) * [f(a) + 4f(c) + f(b)]
    return (h / 3) * (f(a) + 4 * f(c) + f(b))
def adaptive_simpson(a, b, f, tol, whole_integral=None):
    """
    재귀적 적응형 심슨 구적법
    
    a, b: 구간 경계
    f: 함수
    tol: 목표 오차 (Tolerance)
    whole_integral: 상위 단계에서 계산된 넓은 구간의 적분값 (재사용)
    """
    # 1. 넓은 구간의 적분값을 계산하거나 재사용
    if whole_integral is None:
        # 처음 호출 시 전체 구간에 대해 심슨 규칙 적용
        whole_integral = simpsons_rule(a, b, f)
    # 2. 구간을 두 개로 분할하고 각각의 적분값을 계산
    c = (a + b) / 2
    left_integral = simpsons_rule(a, c, f)
    right_integral = simpsons_rule(c, b, f)
    # 두 하위 구간의 합
    half_integral = left_integral + right_integral
    # 3. 오차 추정 (Error Estimation)
    # 심슨 규칙 기반의 오차 추정 공식: E ~ 1/15 * |I_half - I_whole|
    error_estimate = abs(half_integral - whole_integral) / 15
    # 4. 종료 조건 확인
    if error_estimate < tol:
        # 오차가 목표치보다 작으면 계산 종료 후 합산된 값 반환
        # (오차 추정값을 더하여 더 정확한 값을 반환하는 방식도 사용되지만, 여기서는 간단히 합산값 반환)
        return half_integral
    else:
        # 오차가 목표치보다 크면 구간을 재귀적으로 분할
        # 목표 오차를 두 하위 구간에 균등하게 분배
        tol_half = tol / 2
        # 왼쪽 하위 구간에 대해 재귀 호출
        int_left = adaptive_simpson(a, c, f, tol_half, left_integral)
        # 오른쪽 하위 구간에 대해 재귀 호출
        int_right = adaptive_simpson(c, b, f, tol_half, right_integral)
        # 두 재귀 호출 결과 합산
        return int_left + int_right
# --- 실행 부분 ---
a = 0.0
b = 1.0
tolerance = 1e-4  # 목표 오차 (0.0001)
# 적응형 구적법 실행
result = adaptive_simpson(a, b, f, tolerance)
# 결과 출력
print(f"적분 구간: [{a}, {b}]")
print(f"함수: f(x) = x^3")
print(f"목표 오차 (TOL): {tolerance}")
print("-" * 30)
print(f"**정확한 값 (Exact):** {0.25}")
print(f"**적응형 구적법 결과:** {result}")
print(f"**실제 오차:** {abs(result - 0.25)}")

In [ ]:
import math

# 재귀 깊이를 추적하기 위한 전역 변수 (선택 사항)
recursion_depth = 0

def f(x):
    """새로운 적분할 함수: sqrt(x). x=0에서 미분 불가능하지만, 심슨 규칙은 작동함."""
    return math.sqrt(x)

def simpsons_rule(a, b, f):
    """단일 구간 [a, b]에 대한 심슨 규칙"""
    h = (b - a) / 2
    c = (a + b) / 2
    return (h / 3) * (f(a) + 4 * f(c) + f(b))

def adaptive_simpson(a, b, f, tol, whole_integral=None, level=0):
    """재귀적 적응형 심슨 구적법 (출력 기능 포함)"""
    indent = "  " * level
    
    # x=0에서 함수값이 NaN이 되는 것을 방지하기 위한 안전 장치 (sqrt(x)의 경우)
    if a < 1e-15:
         a = 0.0 # 작은 음수 방지

    # 1. 넓은 구간의 적분값을 계산하거나 재사용
    if whole_integral is None:
        whole_integral = simpsons_rule(a, b, f)
        
    # 2. 구간을 두 개로 분할하고 각각의 적분값을 계산
    c = (a + b) / 2
    left_integral = simpsons_rule(a, c, f)
    right_integral = simpsons_rule(c, b, f)
    
    half_integral = left_integral + right_integral
    
    # 3. 오차 추정 (Error Estimation)
    # E ~ 1/15 * |I_half - I_whole|
    error_estimate = abs(half_integral - whole_integral) / 15
    
    # 4. 출력: 현재 구간의 오차 정보
    print(f"{indent} Level {level}: 구간 [{a:.6f}, {b:.6f}] (폭: {b-a:.6f})")
    print(f"{indent}   - 추정 오차: {error_estimate:.10e}")
    print(f"{indent}   - 목표 오차: {tol:.10e}")
    
    # 5. 종료 조건 확인
    if error_estimate < tol:
        # 오차가 목표치보다 작으면 계산 종료
        print(f"{indent}    **통과:** 오차 기준 만족 -> 값 반환")
        return half_integral
    else:
        # 오차가 목표치보다 크면 구간을 재귀적으로 분할
        print(f"{indent}    **실패:** 오차 기준 초과 -> 하위 구간 분할")
        
        # 목표 오차를 두 하위 구간에 균등하게 분배
        tol_half = tol / 2
        
        # 왼쪽 하위 구간에 대해 재귀 호출 (x=0 근처: 변화가 심함)
        int_left = adaptive_simpson(a, c, f, tol_half, left_integral, level + 1)
        
        # 오른쪽 하위 구간에 대해 재귀 호출 (x=1 근처: 비교적 평탄함)
        int_right = adaptive_simpson(c, b, f, tol_half, right_integral, level + 1)
        
        # 두 재귀 호출 결과 합산
        return int_left + int_right

# --- 실행 부분 ---
a = 0.0
b = 1.0
tolerance = 1e-6  # 목표 오차 (0.0001)

print(f"--- 적분 함수: f(x) = sqrt(x), 목표 오차: {tolerance:.1e} ---")
result = adaptive_simpson(a, b, f, tolerance)
print("-----------------------------------")

# 결과 출력
exact_value = 2/3
print(f"**최종 결과 (Approximation):** {result:.12f}")
print(f"**정확한 값 (Exact, 2/3):** {exact_value:.12f}")
print(f"**실제 오차:** {abs(result - exact_value):.12e}")

In [ ]:
import math

# 재귀 깊이를 추적하기 위한 전역 변수 (선택 사항)
recursion_depth = 0

def f(x):
    """적분할 함수: x^3"""
    return x**3

def simpsons_rule(a, b, f):
    """단일 구간 [a, b]에 대한 심슨 규칙"""
    h = (b - a) / 2
    c = (a + b) / 2
    return (h / 3) * (f(a) + 4 * f(c) + f(b))

def adaptive_simpson(a, b, f, tol, whole_integral=None, level=0):
    """
    재귀적 적응형 심슨 구적법 (출력 기능 추가)
    
    level: 현재 재귀 깊이
    """
    indent = "  " * level
    
    # 1. 넓은 구간의 적분값을 계산하거나 재사용
    if whole_integral is None:
        whole_integral = simpsons_rule(a, b, f)
        
    # 2. 구간을 두 개로 분할하고 각각의 적분값을 계산
    c = (a + b) / 2
    left_integral = simpsons_rule(a, c, f)
    right_integral = simpsons_rule(c, b, f)
    
    half_integral = left_integral + right_integral
    
    # 3. 오차 추정 (Error Estimation)
    # E ~ 1/15 * |I_half - I_whole|
    error_estimate = abs(half_integral - whole_integral) / 15
    
    # 4. 출력: 현재 구간의 오차 정보
    print(f"{indent} Level {level}: 구간 [{a:.4f}, {b:.4f}] (폭: {b-a:.4f})")
    print(f"{indent}   - 추정 오차: {error_estimate:.10e}")
    print(f"{indent}   - 목표 오차: {tol:.10e}")
    
    # 5. 종료 조건 확인
    if error_estimate < tol:
        # 오차가 목표치보다 작으면 계산 종료
        print(f"{indent}    **통과:** 오차 기준 만족 -> 값 반환")
        return half_integral
    else:
        # 오차가 목표치보다 크면 구간을 재귀적으로 분할
        print(f"{indent}    **실패:** 오차 기준 초과 -> 하위 구간 분할")
        
        # 목표 오차를 두 하위 구간에 균등하게 분배
        tol_half = tol / 2
        
        # 왼쪽 하위 구간에 대해 재귀 호출
        int_left = adaptive_simpson(a, c, f, tol_half, left_integral, level + 1)
        
        # 오른쪽 하위 구간에 대해 재귀 호출
        int_right = adaptive_simpson(c, b, f, tol_half, right_integral, level + 1)
        
        # 두 재귀 호출 결과 합산
        return int_left + int_right

# --- 실행 부분 ---
a = 0.0
b = 1.0
tolerance = 1e-4  # 목표 오차 (0.0001)

print("--- 적응형 구적법 재귀 과정 출력 ---")
result = adaptive_simpson(a, b, f, tolerance)
print("-----------------------------------")

# 결과 출력
print(f"**최종 결과:** {result:.16f}")
print(f"**정확한 값:** {0.25}")
print(f"**실제 오차:** {abs(result - 0.25):.16e}")

In [ ]:
import numpy as np

def clenshaw_curtis_quadrature(f, N):
    """
    클렌쇼-커티스 구적법을 사용하여 [-1, 1] 구간에서 함수 f(x)를 적분한다.
    
    Args:
        f (function): 적분할 함수 f(x).
        N (int): 노드의 개수 - 1 (구적법의 차수). N+1개의 노드를 사용한다.
        
    Returns:
        float: 근사 적분 값 Q_N.
    """
    
    # --- 1. 체비쇼프 노드 계산 및 함수 평가 ---
    # j = 0, 1, ..., N에 대해 노드를 계산한다.
    j = np.arange(N + 1)
    x_j = np.cos(j * np.pi / N)
    
    # 노드에서의 함수 값 f(x_j)를 계산한다.
    f_j = f(x_j)
    
    # --- 2. 체비쇼프 계수 a_k 계산 (DCT-I 사용) ---
    
    # 계수 계산을 위해 f_j를 수정한다.
    v = np.zeros(2 * N)
    v[0:N+1] = f_j
    v[N+1:] = f_j[1:N][::-1] # 대칭 확장
    
    # FFT 수행
    A = np.fft.fft(v)
    
    # 체비쇼프 계수 a_k는 A의 실수부에서 얻을 수 있다.
    a_k = np.real(A[:N+1]) / N  
    
    # a_0와 a_N은 정의에 따라 1/2로 조정한다.
    a_k[0] /= 2.0
    a_k[N] /= 2.0
    
    # --- 3. 체비쇼프 계수를 이용한 적분 계산 ---
    
    # 적분 공식: Q_N = 2*a_0 - 2 * sum(a_k / (k^2 - 1)) for k=even >= 2
    
    # 짝수 인덱스 k = 0, 2, 4, ...
    k = np.arange(0, N + 1, 2)
    
    # k=0 항은 2*a_0 이다.
    Q_N = 2 * a_k[0]
    
    # 나머지 짝수 항 k >= 2에 대한 합을 계산한다.
    if len(k) > 1:
        # k=0을 제외한 짝수 k에 대한 계수
        a_even = a_k[k[1:]] 
        
        # k^2 - 1 계산
        k_sq_minus_1 = k[1:]**2 - 1
        
        # 합 계산: -2 * sum(a_k / (k^2 - 1))
        Q_N -= 2 * np.sum(a_even / k_sq_minus_1)
        
    return Q_N

# --- 함수 정의 및 실행 ---

# 적분할 함수: 해석적 적분 값이 2/3로 알려진 다항식
def f_polynomial(x):
    return 1.0 - 2.0 * x**2 + 3.0 * x**3

# 정확한 값 (해석적 적분 결과)
TRUE_VALUE = 2.0 / 3.0

# 노드의 개수 설정 (N=4는 다항식 차수 3보다 크다)
N_value = 4 
Q_4 = clenshaw_curtis_quadrature(f_polynomial, N_value)

# 노드의 개수를 늘려 비교 (N=10)
N_large = 10
Q_10 = clenshaw_curtis_quadrature(f_polynomial, N_large)

print(f"--- 클렌쇼-커티스 구적법 결과 (다항 함수) ---")
print(f"함수: f(x) = 1 - 2x^2 + 3x^3 | 구간: [-1, 1]")
print(f"해석적 적분 값 (True Value): = {TRUE_VALUE:.15f}")
print("---")

# 다항식의 차수가 3이므로, N >= 3이면 근사값이 해석적인 값과 거의 일치해야 한다.
print(f"노드 수 N=4 (5개 노드): Q_4 = {Q_4:.15f}")
print(f"N=4 오차:                   = {np.abs(Q_4 - TRUE_VALUE):.2e}")
print("---")
print(f"노드 수 N=10 (11개 노드): Q_10 = {Q_10:.15f}")
print(f"N=10 오차:                  = {np.abs(Q_10 - TRUE_VALUE):.2e}")

In [ ]:
import numpy as np

# ----------------------------------------------------------------------
# 1. 적분할 함수 정의
# ----------------------------------------------------------------------
def test_function_1d(x):
    """적분할 함수: g(x) = sin(x) + x^2"""
    return np.sin(x) + x**2

# ----------------------------------------------------------------------
# 2. Nested Clenshaw-Curtis 적분 함수 (수정된 DCT 구현)
# ----------------------------------------------------------------------

def clenshaw_curtis_quadrature_nested_corrected(a, b, N_max, func, cached_g_values):
    """
    Nested Clenshaw-Curtis 적분 계산 (노드 재활용 및 DCT 수정).
    
    :param a: 적분 구간의 하한
    :param b: 적분 구간의 상한
    :param N_max: 사용할 최대 노드 개수 (N = 2^L 형태)
    :param func: 적분할 함수 g(x)
    :param cached_g_values: 이전에 계산된 표준화된 함수값 g(t)를 저장하는 딕셔너리
    :return: 계산된 적분값
    """
    if N_max < 1:
        raise ValueError("N_max must be at least 1")
    
    N = N_max # FFT 길이, N+1 노드 사용
    
    # 구간 변환 상수
    scale = (b - a) / 2.0
    shift = (b + a) / 2.0
    volume_factor = scale
    
    # 1. 표준 구간 [-1, 1]의 Chebyshev-Gauss-Lobatto 노드 t 계산
    j_values = np.arange(N + 1)
    t = np.cos(j_values * np.pi / N) 
    
    # 2. 표준화된 함수 g(t) = f(x(t)) 값 계산 및 캐싱 (로직 유지)
    g_values = np.zeros(N + 1)
    newly_computed = 0
    
    for i in range(N + 1):
        node_t = t[i]
        key = round(node_t, 15)
        
        if key in cached_g_values:
            g_values[i] = cached_g_values[key]
        else:
            node_x = scale * node_t + shift 
            g_values[i] = func(node_x)      
            cached_g_values[key] = g_values[i]
            newly_computed += 1
            
    # 3.  수정된 체비쇼프 계수 a_k 계산 (Type-I DCT)
    # G = [g(t_0), g(t_1), ..., g(t_N)]
    # a_k = DCT-I(G)
    
    # a. 짝수 대칭 확장: [g(t_0), ..., g(t_N-1), g(t_N), g(t_N-1), ..., g(t_1)] (길이 2N)
    g_extended = np.hstack([g_values, g_values[N-1:0:-1]])
    
    # b. FFT 적용 (길이 2N)
    a_coeffs_full = np.fft.fft(g_extended).real
    
    # c. 계수 a_k 추출 및 스케일링
    # a_k = 2/N * Re(FFT(G_ext)[:N+1])
    a_coeffs = a_coeffs_full[:N + 1] / N # (1/N 스케일링)
    a_coeffs[0] /= 2.0 # k=0, a_0 보정 (2배로 계산되므로 1/2 곱함)
    a_coeffs[N] /= 2.0 # k=N, a_N 보정 (2배로 계산되므로 1/2 곱함)
    
    # 4. 적분 가중치 (W_k)를 이용한 표준 구간 적분 I_t 계산
    integral_sum_t = 0.0
    integral_sum_t += 2.0 * a_coeffs[0] # k = 0: W_0 = 2
    
    # k = 2, 4, 6, ... (짝수)
    for k in range(2, N + 1, 2):
        W_k = 2.0 / (1.0 - k**2)
        integral_sum_t += W_k * a_coeffs[k]
    
    # 5. 최종 적분 I_x = volume_factor * I_t 계산
    final_integral_value = volume_factor * integral_sum_t

    # 결과 출력
    print(f"--- N={N} (노드 {N+1}개) ---")
    print(f"새로 계산된 함수값 개수: {newly_computed}")
    print(f"구간 [{a:.4f}, {b:.4f}] 적분값: {final_integral_value:.15f}")
    
    return final_integral_value

# ----------------------------------------------------------------------
# 3. 예제 실행 및 결과 확인 (Nested)
# ----------------------------------------------------------------------

if __name__ == "__main__":
    # 적분 영역: [0, pi]
    A = 0.0
    B = np.pi
    
    # 해석적 해 (2 + pi^3/3)
    exact_integral = 2.0 + (np.pi**3 / 3.0) # ≈ 12.316886
    
    print(f"--- Nested Clenshaw-Curtis 적분 실행 (DCT 수정 완료) ---")
    print(f"적분 함수: g(x) = sin(x) + x^2")
    print(f"적분 영역: [{A:.4f}, {B:.4f}]")
    print(f"해석적 적분값: {exact_integral:.15f}\n")
    
    cached_g_values = {}
    N_sequence = [2**L for L in range(1, 6)]  # N=2, 4, 8, 16, 32

    for N in N_sequence:
        approx_value = clenshaw_curtis_quadrature_nested_corrected(A, B, N, test_function_1d, cached_g_values)
        error = abs(approx_value - exact_integral)
        print(f"절대 오차: {error:.15e}\n")
        
    print(f"--- 최종 캐시 정보 ---")
    print(f"총 고유 함수값 계산 횟수: {len(cached_g_values)}")

In [ ]:
import numpy as np

# ----------------------------------------------------------------------
# 1. 1차원 Clenshaw-Curtis 가중치 및 노드 계산 함수 (동일)
# ----------------------------------------------------------------------

def get_cc_weights_and_nodes(N):
    """
    클렌쇼-커티스 표준 구간 [-1, 1]의 노드와 가중치를 계산한다.
    (N+1개의 노드를 사용)
    """
    if N < 1:
        return np.array([]), np.array([])
    
    N_total = N 
    
    # 1. 체비쇼프 노드 계산 (N+1개 노드)
    j = np.arange(N_total + 1)
    nodes = np.cos(j * np.pi / N_total)
    
    # 2. 모멘트 m_k 계산 (m_k = integral T_k(x) dx)
    k = np.arange(0, N_total + 1, 2)
    m_k = np.zeros(N_total + 1)
    m_k[0] = 2.0
    if len(k) > 1:
        k_sq_minus_1 = k[1:]**2 - 1
        m_k[k[1:]] = -2.0 / k_sq_minus_1
    
    # 3. IDCT-I 입력 벡터 c_k 생성 (m_k의 복사본)
    c = m_k.copy()
    
    # 4. IDCT-I를 위한 대칭 확장 (2N 주기)
    c_full = np.hstack([c, c[N_total-1:0:-1]])
    
    # 5. 가중치 w_j 계산 (IDCT-I를 FFT로 구현)
    weights = np.fft.ifft(c_full).real[:N_total+1] * 4.0
    
    # IDCT-I의 경계 항 (k=0, k=N) 보정 (절반으로 조정)
    weights[0] /= 2.0
    weights[N_total] /= 2.0
    
    # 가중치의 합이 2.0이 되도록 정규화
    if np.abs(np.sum(weights) - 2.0) > 1e-12:
         weights = weights * (2.0 / np.sum(weights))
    
    weights = (weights + weights[::-1]) / 2.0 # 대칭성 강제
    
    return nodes, weights

# ----------------------------------------------------------------------
# 2. 1차원 Clenshaw-Curtis 적분 함수 (수정)
# ----------------------------------------------------------------------

def integrate_1d_clenshaw_curtis(g, a, b, N):
    """
    Clenshaw-Curtis 적분 규칙을 사용하여 1차원 구간 [a, b]에서 함수 g(x)를 적분한다.
    
    Args:
        g (function): 적분할 1차원 함수 g(x)
        a (float or list): 적분 구간의 하한 ax
        b (float or list): 적분 구간의 상한 bx
        N (int): 사용할 노드의 개수
        
    Returns:
        float: 1차원 적분의 근사값
    """
    if N < 1:
        raise ValueError("N must be at least 1")
    
    # 입력이 리스트 형태인 경우 첫 번째 요소만 사용 (1차원)
    if isinstance(a, list):
        ax = a[0]
        bx = b[0]
    else:
        ax = a
        bx = b

    nodes_1d, weights_1d = get_cc_weights_and_nodes(N)
    M = len(nodes_1d) # M = N + 1

    # 변환 계수
    scale = (bx - ax) / 2.0 
    shift = (bx + ax) / 2.0 
    
    # Jacobian (1차원): (b_x-a_x)/2
    volume_factor = scale

    # 1차원 노드 생성 (변환 적용)
    nodes_x = scale * nodes_1d + shift
    
    # 4. 1차원 적분 수행 (단일 합)
    integral_sum = 0.0
    
    # 이중 루프를 단일 루프로 변경
    for i in range(M):
        xi = nodes_x[i]
        wi = weights_1d[i]
        
        # 1차원 가중치 = w_i
        weight_1d = wi
        
        # 함수 값 계산
        g_val = g(xi)
        
        integral_sum += weight_1d * g_val

    return volume_factor * integral_sum

# ----------------------------------------------------------------------
# 3. 예제 실행 및 결과 확인
# ----------------------------------------------------------------------

# 적분할 함수: g(x) = sin(x) + x^2
def test_function_1d(x):
    return np.sin(x) + x**2

# 적분 영역: [0, pi]
ax = 0.0
bx = np.pi
a = ax # 단일 float 값으로 전달
b = bx

# 해석적 해 (손으로 계산)
# I = integral_0^pi (sin(x) + x^2) dx
# I = [-cos(x) + x^3/3]_0^pi
# I = (-cos(pi) + pi^3/3) - (-cos(0) + 0)
# I = (-(-1) + pi^3/3) - (-1) = 1 + pi^3/3 + 1 = 2 + pi^3/3
exact_integral = 2.0 + (np.pi**3 / 3.0) # ≈ 12.316886

# 노드 개수
N = 10 

approx_integral = integrate_1d_clenshaw_curtis(test_function_1d, a, b, N)

print(f"--- 1차원 Clenshaw-Curtis 적분 ---")
print(f"적분 함수: g(x) = sin(x) + x^2")
print(f"적분 영역: [{ax:.4f}, {bx:.4f}] (pi ≈ 3.1416)")
print(f"노드 수 (N+1 노드): {N + 1}")
print(f"\n근사 적분값: {approx_integral:.15f}")
print(f"해석적 적분값: {exact_integral:.15f}")
print(f"오차 (절대값): {abs(approx_integral - exact_integral):.10e}")

In [ ]:
import numpy as np

def clenshaw_curtis_quadrature(f, N):
    """
    클렌쇼-커티스 구적법을 사용하여 [-1, 1] 구간에서 함수 f(x)를 적분한다.
    
    Args:
        f (function): 적분할 함수 f(x).
        N (int): 노드의 개수 - 1 (구적법의 차수). N+1개의 노드를 사용한다.
        
    Returns:
        float: 근사 적분 값 Q_N.
    """
    
    # --- 1. 체비쇼프 노드 계산 및 함수 평가 ---
    # j = 0, 1, ..., N에 대해 노드를 계산한다.
    j = np.arange(N + 1)
    x_j = np.cos(j * np.pi / N)
    
    # 노드에서의 함수 값 f(x_j)를 계산한다.
    f_j = f(x_j)
    
    # --- 2. 체비쇼프 계수 a_k 계산 (DCT-I 사용) ---
    
    # 이산 코사인 변환 (DCT)을 사용하여 체비쇼프 계수 a_k를 구한다.
    # numpy.fft.fft.fft는 FFT를 사용하여 DCT-I을 효율적으로 계산하는 데 사용된다.
    
    # 실수 배열을 만들고, f_j를 대칭 확장하여 DCT-I을 FFT로 계산한다.
    # f_j의 0과 N번째 요소는 절반만 기여한다 (이중 프라임 합의 정의).
    
    # 계수 계산을 위해 f_j를 수정한다.
    v = np.zeros(2 * N)
    v[0:N+1] = f_j
    v[N+1:] = f_j[1:N][::-1] # 대칭 확장
    
    # FFT 수행
    A = np.fft.fft(v)
    
    # 체비쇼프 계수 a_k는 A의 실수부에서 얻을 수 있다.
    a_k = np.real(A[:N+1]) / N 
    
    # a_0와 a_N은 정의에 따라 1/2로 조정해야 한다 (a_k의 정의에서 2/N이 아닌 1/N로 나누었기 때문).
    # a_k = (2/N) * sum'' f_j * cos(...) 임을 기억하세요.
    a_k[0] /= 2.0
    a_k[N] /= 2.0
    
    # --- 3. 체비쇼프 계수를 이용한 적분 계산 ---
    
    # 적분 공식: Q_N = 2*a_0 - 2 * sum(a_k / (k^2 - 1)) for k=even >= 2
    
    # 짝수 인덱스 k = 0, 2, 4, ...
    k = np.arange(0, N + 1, 2)
    
    # k=0 항은 2*a_0 이다.
    Q_N = 2 * a_k[0]
    
    # 나머지 짝수 항 k >= 2에 대한 합을 계산한다.
    if len(k) > 1:
        # k=0을 제외한 짝수 k에 대한 계수
        a_even = a_k[k[1:]] 
        
        # k^2 - 1 계산
        k_sq_minus_1 = k[1:]**2 - 1
        
        # 합 계산: -2 * sum(a_k / (k^2 - 1))
        Q_N -= 2 * np.sum(a_even / k_sq_minus_1)
        
    return Q_N

# --- 함수 정의 및 실행 ---

# 적분할 함수: f(x) = exp(-x^2)
def f_gaussian(x):
    return np.exp(-x**2)

# 노드의 개수 설정 (예: N=10, 노드 11개)
N_value = 10 
Q_10 = clenshaw_curtis_quadrature(f_gaussian, N_value)

# 노드의 개수를 늘려 정확도 비교 (예: N=50, 노드 51개)
N_large = 50
Q_50 = clenshaw_curtis_quadrature(f_gaussian, N_large)

# 정확한 값 (오차 함수 erf를 통해 얻을 수 있다)
# er f(1) * sqrt(pi) / 2
true_value = 1.4936482656248555

print(f"--- 클렌쇼-커티스 구적법 결과 ---")
print(f"함수: exp(-x^2) | 구간: [-1, 1]")
print(f"노드 수 N=10 (11개 노드): Q_10 = {Q_10:.15f}")
print(f"노드 수 N=50 (51개 노드): Q_50 = {Q_50:.15f}")
print(f"정확한 값:                    = {true_value:.15f}")
print(f"N=50 오차:                     = {np.abs(Q_50 - true_value):.2e}")

# 참고: N이 커질수록 오차가 급격히 줄어드는 것을 확인할 수 있다 (지수적 수렴).

import numpy as np

def clenshaw_curtis_quadrature(f, N):
    """
    클렌쇼-커티스 구적법을 사용하여 [-1, 1] 구간에서 함수 f(x)를 적분한다.
    
    Args:
        f (function): 적분할 함수 f(x).
        N (int): 노드의 개수 - 1 (구적법의 차수). N+1개의 노드를 사용한다.
        
    Returns:
        float: 근사 적분 값 Q_N.
    """
    
    # --- 1. 체비쇼프 노드 계산 및 함수 평가 ---
    # j = 0, 1, ..., N에 대해 노드를 계산한다.
    j = np.arange(N + 1)
    x_j = np.cos(j * np.pi / N)
    
    # 노드에서의 함수 값 f(x_j)를 계산한다.
    f_j = f(x_j)
    
    # --- 2. 체비쇼프 계수 a_k 계산 (DCT-I 사용) ---
    
    # 계수 계산을 위해 f_j를 수정한다.
    v = np.zeros(2 * N)
    v[0:N+1] = f_j
    v[N+1:] = f_j[1:N][::-1] # 대칭 확장
    
    # FFT 수행
    A = np.fft.fft(v)
    
    # 체비쇼프 계수 a_k는 A의 실수부에서 얻을 수 있다.
    a_k = np.real(A[:N+1]) / N  
    
    # a_0와 a_N은 정의에 따라 1/2로 조정한다.
    a_k[0] /= 2.0
    a_k[N] /= 2.0
    
    # --- 3. 체비쇼프 계수를 이용한 적분 계산 ---
    
    # 적분 공식: Q_N = 2*a_0 - 2 * sum(a_k / (k^2 - 1)) for k=even >= 2
    
    # 짝수 인덱스 k = 0, 2, 4, ...
    k = np.arange(0, N + 1, 2)
    
    # k=0 항은 2*a_0 이다.
    Q_N = 2 * a_k[0]
    
    # 나머지 짝수 항 k >= 2에 대한 합을 계산한다.
    if len(k) > 1:
        # k=0을 제외한 짝수 k에 대한 계수
        a_even = a_k[k[1:]] 
        
        # k^2 - 1 계산
        k_sq_minus_1 = k[1:]**2 - 1
        
        # 합 계산: -2 * sum(a_k / (k^2 - 1))
        Q_N -= 2 * np.sum(a_even / k_sq_minus_1)
        
    return Q_N

# --- 함수 정의 및 실행 ---

# 적분할 함수: 해석적 적분 값이 2/3로 알려진 다항식
def f_polynomial(x):
    return 1.0 - 2.0 * x**2 + 3.0 * x**3

# 정확한 값 (해석적 적분 결과)
TRUE_VALUE = 2.0 / 3.0

# 노드의 개수 설정 (N=4는 다항식 차수 3보다 크다)
N_value = 4 
Q_4 = clenshaw_curtis_quadrature(f_polynomial, N_value)

# 노드의 개수를 늘려 비교 (N=10)
N_large = 10
Q_10 = clenshaw_curtis_quadrature(f_polynomial, N_large)

print(f"--- 클렌쇼-커티스 구적법 결과 (다항 함수) ---")
print(f"함수: f(x) = 1 - 2x^2 + 3x^3 | 구간: [-1, 1]")
print(f"해석적 적분 값 (True value): = {TRUE_VALUE:.15f}")
print("---")

# 다항식의 차수가 3이므로, N >= 3이면 근사값이 해석적인 값과 거의 일치해야 한다.
print(f"노드 수 N=4 (5개 노드): Q_4 = {Q_4:.15f}")
print(f"N=4 오차:                   = {np.abs(Q_4 - TRUE_VALUE):.2e}")
print("---")
print(f"노드 수 N=10 (11개 노드): Q_10 = {Q_10:.15f}")
print(f"N=10 오차:                  = {np.abs(Q_10 - TRUE_VALUE):.2e}")

import numpy as np
import matplotlib.pyplot as plt

# 이전과 동일한 클렌쇼-커티스 구적법 함수
def clenshaw_curtis_quadrature(f, N):
    """
    클렌쇼-커티스 구적법을 사용하여 [-1, 1] 구간에서 함수 f(x)를 적분한다.
    """
    
    # 1. 체비쇼프 노드 계산 및 함수 평가
    j = np.arange(N + 1)
    x_j = np.cos(j * np.pi / N)
    f_j = f(x_j)
    
    # 2. 체비쇼프 계수 a_k 계산 (DCT-I 사용)
    v = np.zeros(2 * N)
    v[0:N+1] = f_j
    v[N+1:] = f_j[1:N][::-1]
    
    A = np.fft.fft(v)
    a_k = np.real(A[:N+1]) / N  
    
    # 경계 계수 조정
    if N > 0:
        a_k[0] /= 2.0
        a_k[N] /= 2.0
    
    # 3. 체비쇼프 계수를 이용한 적분 계산
    k = np.arange(0, N + 1, 2)
    Q_N = 2 * a_k[0]
    
    # k >= 2 짝수 항 합 계산
    if len(k) > 1:
        a_even = a_k[k[1:]] 
        k_sq_minus_1 = k[1:]**2 - 1
        Q_N -= 2 * np.sum(a_even / k_sq_minus_1)
        
    return Q_N

# --- 테스트 함수 정의 ---

# 1. 해석적 적분 값이 2/3인 다항식 (Polynomial)
def f_polynomial(x):
    return 1.0 - 2.0 * x**2 + 3.0 * x**3

TRUE_VALUE_POLY = 2.0 / 3.0

# 2. 지수적 수렴 특성을 잘 보여주는 매끄러운 함수 (Smooth Function)
# 해석적 적분 값: tanh(1) ≈ 0.7615941559557649
def f_smooth(x):
    return np.exp(x) / np.cosh(x)

TRUE_VALUE_SMOOTH = 0.7615941559557649
TRUE_VALUE_SMOOTH = 2.0

# --- 오차 분석 및 시각화 함수 ---

def plot_convergence(f, true_value, max_N, title):
    """
    다양한 N 값에 대한 오차를 계산하고 로그-선형 그래프를 그린다.
    """
    # 테스트할 N 값 리스트 (2^k 형태가 일반적이며, 여기서는 N=2부터 max_N까지 테스트)
    N_values = np.arange(2, max_N + 1, 1) 
    
    # 오차를 저장할 리스트
    errors = []
    
    for N in N_values:
        Q_N = clenshaw_curtis_quadrature(f, N)
        error = np.abs(Q_N - true_value)
        errors.append(error)
        
    errors = np.array(errors)

    # 오차가 0인 경우 로그를 취할 수 없으므로 작은 값으로 대체
    # 다항 함수의 경우 N >= 차수 에서 오차가 0에 매우 가까워짐
    errors[errors == 0] = np.finfo(float).eps
    
    # --- 그래프 그리기 ---
    plt.figure(figsize=(10, 6))
    
    # 오차를 로그 스케일로 플롯
    plt.semilogy(N_values, errors, 'o-', label='Clenshaw-Curtis error')
    
    plt.xlabel('Number of nodes N', fontsize=18)
    plt.ylabel(r'Absolute error $|\int f(x)dx - Q_N|$ (Log scale)', fontsize=18)
    plt.title(f'Convergence of Clenshaw-Curtis quadrature for $f(x) = {title}$')
    plt.grid(True, which="both", ls="--")
    plt.legend()
    plt.show()
    print(f"\n--- {title} 오차 분석 결과 ---")
    print(f"최소 오차 (N={N_values[-1]}): {errors[-1]:.2e}")

# --- 프로그램 실행 ---

# 1. 다항 함수에 대한 수렴 분석 및 시각화
plot_convergence(f_polynomial, TRUE_VALUE_POLY, 10, '1 - 2x^2 + 3x^3')

# 2. 매끄러운 함수에 대한 수렴 분석 및 시각화 (지수적 수렴 관찰)
# N_max를 늘리면 오차가 더욱 급격하게 감소하는 것을 볼 수 있다.
plot_convergence(f_smooth, TRUE_VALUE_SMOOTH, 30, 'e^x / cosh(x)')

In [ ]:
import numpy as np
from scipy.integrate import quad # 참값 계산용

def f(x):
    """적분할 함수: e^(-x^2)"""
    return np.exp(-(x**2))

def trapezoidal_rule(f, a, b, n):
    """사다리꼴 공식"""
    h = (b - a) / n
    integral = (f(a) + f(b)) / 2.0
    for i in range(1, n):
        integral += f(a + i * h)
    return integral * h

a, b = 0, 1
# 참값 (High-precision reference value)
exact_value, _ = quad(f, a, b) 
# exact_value ≈ 0.746824132812427

# 1. 분할 수 n1=1 (h1=1)
n1 = 1
D1 = trapezoidal_rule(f, a, b, n1)

# 2. 분할 수 n2=2 (h2=0.5)
n2 = 2
D2 = trapezoidal_rule(f, a, b, n2)

print(f"--- 참값 (Exact value): {exact_value:.10f} ---")
print(f"1. D1 (n={n1}, h=1.00): {D1:.10f}")
print(f"   오차: {abs(D1 - exact_value):.10f}")
print(f"2. D2 (n={n2}, h=0.50): {D2:.10f}")
print(f"   오차: {abs(D2 - exact_value):.10f}")

p = 2
richardson_extrapolated = D2 + (D2 - D1) / (2**p - 1)

print("-" * 40)
print(f"3. Richardson 외삽값 (Phi_new): {richardson_extrapolated:.10f}")
print(f"   외삽 후 오차: {abs(richardson_extrapolated - exact_value):.10f}")

In [ ]:
import numpy as np

# 적분 대상 함수
def f(x):
    """Integrand: f(x) = sin(x)"""
    return np.sin(x)

#  정답 값 (Actual Answer)
ACTUAL_ANSWER = 2.0

def trapezoidal_rule(f, a, b, n):
    """합성 사다리꼴 공식"""
    h = (b - a) / n
    # np.arange는 끝점 b를 포함하지 않으므로, f(a)와 f(b)를 따로 처리한다.
    x = a + np.arange(1, n) * h  
    integral = h / 2 * (f(a) + 2 * np.sum(f(x)) + f(b))
    return integral

def romberg_integration_with_error(f, a, b, actual_answer, max_k=10, tolerance=1e-8):
    """
    롬베르크 적분과 정답 대비 오차 출력
    """
    R = np.zeros((max_k, max_k))
    print("--- 롬베르크 적분 과정 (f(x) = sin(x), 정답: 2.0) ---")
    
    for k in range(max_k):
        # 1. 첫 번째 열 (j=0): 사다리꼴 공식
        n = 2**k
        R[k, 0] = trapezoidal_rule(f, a, b, n)
        
        # 2. 나머지 열 (j > 0): 리처드슨 보외법
        for j in range(1, k + 1):
            power_of_4 = 4**j
            R[k, j] = R[k, j-1] + (R[k, j-1] - R[k-1, j-1]) / (power_of_4 - 1)

        # 3. 결과 출력 및 수렴 확인 (매번 대각선 항 R[k, k]의 오차 확인)
        current_approximation = R[k, k]
        absolute_error = abs(current_approximation - actual_answer)
        
        print(f"R[{k+1}, {k+1}] = {current_approximation:.12f},  절대 오차: {absolute_error:.2e}")
        
        if k > 0 and abs(R[k, k] - R[k-1, k-1]) < tolerance:
            print(f"\n 수렴 허용 오차({tolerance:.1e}) 달성. {k+1}번째 반복에서 종료한다.")
            return R[k, k], R[:k+1, :k+1]

    return R[max_k-1, max_k-1], R # 최대 반복 횟수 도달 시

# 적분 설정 및 실행
a = 0.0
b = np.pi
max_iterations = 6
tol = 1e-8

final_result, romberg_table = romberg_integration_with_error(f, a, b, ACTUAL_ANSWER, max_iterations, tol)

import numpy as np
import matplotlib.pyplot as plt

# 이전 롬베르크 적분 실행 결과 (R[k, k]에 대한 절대 오차)
# k=0 (R11): 2.00e+00
# k=1 (R22): 9.44e-02
# k=2 (R33): 1.43e-03
# k=3 (R44): 1.69e-06
# k=4 (R55): 3.20e-11

# k는 반복 횟수 (테이블 행 번호 - 1)
k_values = np.array([0, 1, 2, 3, 4]) 
# 각 R[k+1, k+1]의 절대 오차 값
errors = np.array([2.0, 9.44e-2, 1.43e-3, 1.69e-6, 3.20e-11]) 

# Matplotlib를 이용한 시각화
plt.figure(figsize=(10, 6))

# 오차 변화를 명확히 보기 위해 Y축을 로그 스케일로 설정
plt.plot(k_values + 1, errors, marker='o', linestyle='-', color='indigo', label='Absolute error of $R_{k,k}$') 
plt.yscale('log') # 로그 스케일 적용

plt.title('Romberg integration error convergence (for $\int_{0}^{\pi} \sin(x) dx$)', fontsize=14)
plt.xlabel('Iteration Number(k)', fontsize=12)
plt.ylabel('Absolute error (Log scale)', fontsize=12)
plt.xticks(k_values + 1) # x축 눈금을 1, 2, 3, 4, 5로 설정
plt.grid(True, which="both", ls="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import math
def f(x):
    """적분할 함수: x^3"""
    return x**3
def simpsons_rule(a, b, f):
    """단일 구간 [a, b]에 대한 심슨 규칙"""
    h = (b - a) / 2
    c = (a + b) / 2
    # I = (h/3) * [f(a) + 4f(c) + f(b)]
    return (h / 3) * (f(a) + 4 * f(c) + f(b))
def adaptive_simpson(a, b, f, tol, whole_integral=None):
    """
    재귀적 적응형 심슨 구적법
    
    a, b: 구간 경계
    f: 함수
    tol: 목표 오차 (Tolerance)
    whole_integral: 상위 단계에서 계산된 넓은 구간의 적분값 (재사용)
    """
    # 1. 넓은 구간의 적분값을 계산하거나 재사용
    if whole_integral is None:
        # 처음 호출 시 전체 구간에 대해 심슨 규칙 적용
        whole_integral = simpsons_rule(a, b, f)
    # 2. 구간을 두 개로 분할하고 각각의 적분값을 계산
    c = (a + b) / 2
    left_integral = simpsons_rule(a, c, f)
    right_integral = simpsons_rule(c, b, f)
    # 두 하위 구간의 합
    half_integral = left_integral + right_integral
    # 3. 오차 추정 (Error Estimation)
    # 심슨 규칙 기반의 오차 추정 공식: E ~ 1/15 * |I_half - I_whole|
    error_estimate = abs(half_integral - whole_integral) / 15
    # 4. 종료 조건 확인
    if error_estimate < tol:
        # 오차가 목표치보다 작으면 계산 종료 후 합산된 값 반환
        # (오차 추정값을 더하여 더 정확한 값을 반환하는 방식도 사용되지만, 여기서는 간단히 합산값 반환)
        return half_integral
    else:
        # 오차가 목표치보다 크면 구간을 재귀적으로 분할
        # 목표 오차를 두 하위 구간에 균등하게 분배
        tol_half = tol / 2
        # 왼쪽 하위 구간에 대해 재귀 호출
        int_left = adaptive_simpson(a, c, f, tol_half, left_integral)
        # 오른쪽 하위 구간에 대해 재귀 호출
        int_right = adaptive_simpson(c, b, f, tol_half, right_integral)
        # 두 재귀 호출 결과 합산
        return int_left + int_right
# --- 실행 부분 ---
a = 0.0
b = 1.0
tolerance = 1e-4  # 목표 오차 (0.0001)
# 적응형 구적법 실행
result = adaptive_simpson(a, b, f, tolerance)
# 결과 출력
print(f"적분 구간: [{a}, {b}]")
print(f"함수: f(x) = x^3")
print(f"목표 오차 (TOL): {tolerance}")
print("-" * 30)
print(f"**정확한 값 (Exact):** {0.25}")
print(f"**적응형 구적법 결과:** {result}")
print(f"**실제 오차:** {abs(result - 0.25)}")

import math

# 재귀 깊이를 추적하기 위한 전역 변수 (선택 사항)
recursion_depth = 0

def f(x):
    """적분할 함수: x^3"""
    return x**3

def simpsons_rule(a, b, f):
    """단일 구간 [a, b]에 대한 심슨 규칙"""
    h = (b - a) / 2
    c = (a + b) / 2
    return (h / 3) * (f(a) + 4 * f(c) + f(b))

def adaptive_simpson(a, b, f, tol, whole_integral=None, level=0):
    """
    재귀적 적응형 심슨 구적법 (출력 기능 추가)
    
    level: 현재 재귀 깊이
    """
    indent = "  " * level
    
    # 1. 넓은 구간의 적분값을 계산하거나 재사용
    if whole_integral is None:
        whole_integral = simpsons_rule(a, b, f)
        
    # 2. 구간을 두 개로 분할하고 각각의 적분값을 계산
    c = (a + b) / 2
    left_integral = simpsons_rule(a, c, f)
    right_integral = simpsons_rule(c, b, f)
    
    half_integral = left_integral + right_integral
    
    # 3. 오차 추정 (Error Estimation)
    # E ~ 1/15 * |I_half - I_whole|
    error_estimate = abs(half_integral - whole_integral) / 15
    
    # 4. 출력: 현재 구간의 오차 정보
    print(f"{indent} Level {level}: 구간 [{a:.4f}, {b:.4f}] (폭: {b-a:.4f})")
    print(f"{indent}   - 추정 오차: {error_estimate:.10e}")
    print(f"{indent}   - 목표 오차: {tol:.10e}")
    
    # 5. 종료 조건 확인
    if error_estimate < tol:
        # 오차가 목표치보다 작으면 계산 종료
        print(f"{indent}    **통과:** 오차 기준 만족 -> 값 반환")
        return half_integral
    else:
        # 오차가 목표치보다 크면 구간을 재귀적으로 분할
        print(f"{indent}    **실패:** 오차 기준 초과 -> 하위 구간 분할")
        
        # 목표 오차를 두 하위 구간에 균등하게 분배
        tol_half = tol / 2
        
        # 왼쪽 하위 구간에 대해 재귀 호출
        int_left = adaptive_simpson(a, c, f, tol_half, left_integral, level + 1)
        
        # 오른쪽 하위 구간에 대해 재귀 호출
        int_right = adaptive_simpson(c, b, f, tol_half, right_integral, level + 1)
        
        # 두 재귀 호출 결과 합산
        return int_left + int_right

# --- 실행 부분 ---
a = 0.0
b = 1.0
tolerance = 1e-4  # 목표 오차 (0.0001)

print("--- 적응형 구적법 재귀 과정 출력 ---")
result = adaptive_simpson(a, b, f, tolerance)
print("-----------------------------------")

# 결과 출력
print(f"**최종 결과:** {result:.16f}")
print(f"**정확한 값:** {0.25}")
print(f"**실제 오차:** {abs(result - 0.25):.16e}")

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np

# 함수 값을 평가한 모든 점들을 기록할 리스트
evaluated_points = set() # 중복 방지를 위해 set 사용

def f(x):
    """새로운 적분할 함수: sqrt(x)"""
    # 함수 호출 시점마다 점 기록
    evaluated_points.add(x)
    return math.sqrt(x)

def simpsons_rule(a, b, f):
    """단일 구간 [a, b]에 대한 심슨 규칙"""
    # 심슨 규칙은 a, (a+b)/2, b 세 점에서 함수를 평가함
    c = (a + b) / 2
    
    # 여기서 f(a), f(c), f(b)를 호출하며 evaluated_points에 추가됨
    return (b - a) / 6 * (f(a) + 4 * f(c) + f(b)) 
    # (b-a)/6 * [f(a) + 4f(c) + f(b)] = (h/3) * [f(a) + 4f(c) + f(b)] where h = (b-a)/2

def adaptive_simpson(a, b, f, tol, whole_integral=None, level=0):
    """재귀적 적응형 심슨 구적법 (출력 및 점 기록 기능 포함)"""
    indent = "  " * level
    
    # x=0에서 함수값이 NaN이 되는 것을 방지하기 위한 안전 장치 (sqrt(x)의 경우)
    # math.sqrt(0.0)은 0.0을 반환하므로 특별히 필요 없을 수도 있지만, 부동소수점 오차 방지
    if a < 1e-15:
         a = 0.0 

    # 1. 넓은 구간의 적분값을 계산하거나 재사용
    if whole_integral is None:
        whole_integral = simpsons_rule(a, b, f)
        
    # 2. 구간을 두 개로 분할하고 각각의 적분값을 계산
    c = (a + b) / 2
    left_integral = simpsons_rule(a, c, f)
    right_integral = simpsons_rule(c, b, f)
    
    half_integral = left_integral + right_integral
    
    # 3. 오차 추정 (Error Estimation)
    error_estimate = abs(half_integral - whole_integral) / 15
    
    # 4. 출력 (선택 사항, 상세 로그를 보고 싶을 때 주석 해제)
    # print(f"{indent} Level {level}: 구간 [{a:.6f}, {b:.6f}] (폭: {b-a:.6f})")
    # print(f"{indent}   - 추정 오차: {error_estimate:.10e}")
    # print(f"{indent}   - 목표 오차: {tol:.10e}")
    
    # 5. 종료 조건 확인
    if error_estimate < tol:
        # print(f"{indent}    **통과:** 오차 기준 만족 -> 값 반환")
        return half_integral
    else:
        # print(f"{indent}    **실패:** 오차 기준 초과 -> 하위 구간 분할")
        
        # 목표 오차를 두 하위 구간에 균등하게 분배
        tol_half = tol / 2
        
        int_left = adaptive_simpson(a, c, f, tol_half, left_integral, level + 1)
        int_right = adaptive_simpson(c, b, f, tol_half, right_integral, level + 1)
        
        return int_left + int_right

# --- 실행 부분 ---
a = 0.0
b = 1.0
tolerance = 1e-4  # 목표 오차 (0.0001)

# evaluated_points 리셋 (여러 번 실행할 경우 대비)
evaluated_points.clear() 

print(f"--- f(x) = sqrt(x), target error: {tolerance:.1e} ---")
result = adaptive_simpson(a, b, f, tolerance)
print("-----------------------------------")

# 결과 출력
exact_value = 2/3
print(f"**최종 결과 (Approximation):** {result:.12f}")
print(f"**정확한 값 (Exact, 2/3):** {exact_value:.12f}")
print(f"**실제 오차:** {abs(result - exact_value):.12e}")
print(f"**동원된 점의 총 개수:** {len(evaluated_points)}개")

# --- 시각화 ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={'height_ratios': [3, 1]})

# 1. 함수 그래프 및 동원된 점들
x_vals_plot = np.linspace(a, b, 500)
y_vals_plot = np.array([f(x) for x in x_vals_plot]) # f(x) 호출 시점에도 evaluated_points에 추가됨

ax1.plot(x_vals_plot, y_vals_plot, label='$f(x) = \sqrt{x}$', color='blue')

# 동원된 점들을 정렬
sorted_points = sorted(list(evaluated_points))
# 해당 점들에서의 함수 값 계산
y_evaluated = [f(x) for x in sorted_points] # f(x) 호출 시점에도 evaluated_points에 추가됨

# 기록된 점들을 플롯
ax1.scatter(sorted_points, y_evaluated, color='red', s=20, zorder=5, label='points')
ax1.set_title(f'$f(x) = \sqrt{{x}}$ and adaptive ($TOL = {tolerance:.1e}$)')
ax1.set_xlabel('x')
ax1.set_ylabel('f(x)')
ax1.legend()
ax1.grid(True)

# 2. 동원된 점들의 밀집도 히스토그램 또는 분포
# 점들의 x-좌표만 가져와서 막대 그래프로 분포 확인
ax2.hist(list(evaluated_points), bins=50, range=(a,b), color='green', alpha=0.7)
ax2.set_title('point distribution')
ax2.set_xlabel('x')
ax2.set_ylabel('number of points')
ax2.grid(True)

plt.tight_layout() # 서브플롯 간격 자동 조절
plt.show()

In [ ]:
import numpy as np

def monte_carlo_integration(f, a, b, N, dim=1):
    """
    몬테카를로 방법을 사용하여 다차원 정적분을 근사한다.

    Parameters:
    f (function): 적분할 함수 (입력으로 array를 받음)
    a (float): 구간 하한
    b (float): 구간 상한
    N (int): 샘플(난수)의 개수
    dim (int): 적분의 차원

    Returns:
    float: 몬테카를로 적분 근삿값
    """
    
    # 1. 난수 샘플 생성 (균일 분포)
    # dim 차원의 난수 N개를 [a, b] 구간에서 생성한다.
    # shape: (N, dim)
    samples = np.random.uniform(low=a, high=b, size=(N, dim))
    
    # 2. 난수를 함수에 대입하여 함숫값 계산
    # *samples.T는 함수가 f(x)나 f(x, y)를 받도록 데이터를 transpose하여 언팩하는 과정이다.
    function_values = f(*samples.T)
    
    # 3. 적분 구간의 체적(Volume) 계산 (다차원)
    volume = (b - a) ** dim
    
    # 4. 몬테카를로 공식 적용: Volume * (함숫값들의 평균)
    approx_integral = volume * np.mean(function_values)
    
    return approx_integral

# --- 2차원 예제 실행 ---
def func_sum(x, y):
    # 몬테카를로 함수는 언팩된 x, y 벡터를 받는다.
    return x + y

# 변수 설정
a_val, b_val = 0, 1  # 적분 구간 [0, 1]
N_samples = 100000   # 샘플 개수 (많을수록 정확함)
dim_val = 2          # 2차원 적분

mc_result = monte_carlo_integration(func_sum, a_val, b_val, N_samples, dim_val)

print(f"\n함수: f(x, y) = x + y")
print(f"적분 범위: x=[0, 1], y=[0, 1]")
print(f"샘플 개수 (N): {N_samples}")
print("-" * 30)
print(f"몬테카를로 적분 근삿값: {mc_result:.4f}")
print(f"정확한 값: 1.0000")

import numpy as np
from scipy.stats import norm, expon

# --- 1. 표준 정규 분포 함수 (우리가 적분하려는 함수 f(x)) ---
# x >= 3 인 영역의 확률 (p(x)의 꼬리 부분)
def p_x(x):
    return norm.pdf(x, loc=0, scale=1) # 표준 정규 분포

# 참 값 (SciPy의 CDF를 사용)
true_value = 1.0 - norm.cdf(3)
print(f"참 값 (P(X >= 3)): {true_value:.8f}")
print("-" * 50)

# --- 2. 샘플링 설정 ---
N = 100000 # 샘플 개수

# 일반 몬테카를로 적분 (균일 분포 샘플링)
def standard_mc(f, a, b, N):
    x_samples = np.random.uniform(a, b, N)
    
    # 균일 분포의 확률 밀도 함수 (p_samples)
    p_samples = 1.0 / (b - a) 
    
    # 몬테카를로 추정: (b-a) * 평균(f(x))
    # 또는: 평균(f(x) / p_samples) * p_samples * (b-a) = 평균(f(x) / p_samples) 
    # 하지만 여기서는 적분 구간 [a, b]로 샘플링하므로 Volume * Mean(f(x))로 계산
    estimate = (b - a) * np.mean(f(x_samples))
    return estimate

mc_uniform_estimate = standard_mc(p_x, a=3, b=10, N=N)
print(f"1. 표준 몬테카를로 (균일 분포) 근삿값: {mc_uniform_estimate:.8f}")

# 중요도 샘플링 (지수 분포 샘플링)
def importance_sampling_mc(f_ratio, g_sampler, g_pdf, N):
    
    # 1. 샘플링 분포 g(x)에서 N개의 샘플 추출
    # 지수 분포를 사용하여 x >= 3 영역에 샘플을 집중
    g_samples = g_sampler(size=N)
    
    # 2. 샘플 x_i가 적분 구간 (여기서는 [3, 무한대]) 내에 있는 경우만 필터링
    # g(x)의 지원(support)이 f(x)의 지원을 포함해야 하지만, 여기서는 근사를 위해 필터링
    valid_samples = g_samples[g_samples >= 3]
    
    if len(valid_samples) == 0:
        return 0
    
    # 3. 중요도 비율 (Weight) 계산: w(x) = f(x) / g(x)
    # f(x): 표준 정규 분포의 밀도 (p_x)
    # g(x): 선택한 샘플링 분포의 밀도 (g_pdf)
    f_val = p_x(valid_samples) 
    g_val = g_pdf(valid_samples)
    
    # 가중 평균 계산: np.mean(f(x) / g(x))
    # 참고: 지수 분포는 적분 구간이 [3, inf]가 아니라 [0, inf]이므로,
    # g_pdf를 사용할 때 g(x)의 정규화 상수를 고려해야 하지만, 여기서는 샘플링된 값으로 근사한다.
    # 지수 분포의 정규화 상수는 1/(rate)
    
    weights = f_val / g_val
    
    # 적분값 추정: 평균(w(x))
    # 지수 분포의 면적이 1이므로, 이 추정은 P(X>=3)로 바로 수렴한다.
    estimate = np.mean(weights)
    return estimate

# 지수 분포 (rate=1.0)를 샘플링 분포 g(x)로 사용
# (x >= 0 영역에 집중되며, x=3 이후의 꼬리 부분을 더 잘 포착하도록 선택)
g_sampler = lambda size: expon.rvs(loc=0, scale=1/0.1, size=size) # rate=0.5
g_pdf = lambda x: expon.pdf(x, loc=0, scale=1/0.1)

mc_importance_estimate = importance_sampling_mc(p_x, g_sampler, g_pdf, N=N)
print(f"2. 중요도 샘플링 (지수 분포) 근삿값: {mc_importance_estimate:.8f}")

import numpy as np
from scipy.stats import norm, expon

# --- 함수 정의 (이전 예제와 동일) ---
def p_x(x):
    return norm.pdf(x, loc=0, scale=1) # 표준 정규 분포 PDF

# 지수 분포 (g(x) 샘플링 분포) 설정
# rate=0.5, scale=2.0
g_sampler = lambda size: expon.rvs(loc=0, scale=1/0.05, size=size) 
g_pdf = lambda x: expon.pdf(x, loc=0, scale=1/0.05)

# --- 중요도 샘플링 함수 (분산 계산 추가) ---
def importance_sampling_mc_with_variance(f_pdf, g_sampler, g_pdf, N):
    
    # 1. 샘플링 분포 g(x)에서 N개의 샘플 추출
    g_samples = g_sampler(size=N)
    
    # 2. 적분 구간 [3, inf] 내 샘플만 필터링
    valid_samples = g_samples[g_samples >= 3]
    
    if len(valid_samples) == 0:
        return 0, 0, 0
    
    # 3. 중요도 비율 (Weight) 계산: w_i = f(x_i) / g(x_i)
    f_val = f_pdf(valid_samples) 
    g_val = g_pdf(valid_samples)
    
    # 가중치 벡터 w
    weights = f_val / g_val
    
    # --- 적분값 추정 ---
    estimate = np.mean(weights)
    
    # --- 분산 및 표준 오차 계산 ---
    
    # 4. 가중치 w의 표본 분산 (V) 계산
    # np.var(weights)는 1/N * sum((w_i - mean(w))^2)을 계산한다.
    variance_of_weights = np.var(weights, ddof=1) # ddof=1: 표본 분산 (N-1)
    
    # 5. 추정량 I_hat의 분산 계산: Var(I_hat) = V / N
    # 샘플의 총 개수 N을 사용하여 추정량의 분산을 계산한다.
    # 주의: 여기서 N은 '유효 샘플 개수' (len(valid_samples))가 아닌
    # 초기에 생성한 총 샘플 개수 (N)를 사용하거나,
    # 필터링된 샘플 개수 len(valid_samples)를 사용하여 근사한다.
    # 보수적인 근사를 위해 필터링된 샘플 개수를 사용한다.
    n_eff = len(valid_samples)
    variance_of_estimate = variance_of_weights / n_eff
    
    # 6. 표준 오차 (Standard Error, SE): 분산의 제곱근
    standard_error = np.sqrt(variance_of_estimate)
    
    return estimate, variance_of_estimate, standard_error

# --- 실행 ---
N_samples = 1000000 
true_value = 1.0 - norm.cdf(3)

estimate, variance, se = importance_sampling_mc_with_variance(p_x, g_sampler, g_pdf, N=N_samples)

print(f"참 값 (P(X >= 3)): {true_value:.8f}")
print(f"총 샘플 개수 (N): {N_samples}")
print("-" * 50)
print(f"**중요도 샘플링 근삿값:** {estimate:.8f}")
print(f"**추정값의 분산:** {variance:.2e}")
print(f"**추정값의 표준 오차 (SE):** {se:.2e}")

import numpy as np

def f(x, y, z):
    """적분 함수: f(x, y, z) = x^2 + y^2 + z^2"""
    return x**2 + y**2 + z**2

# 샘플 수
N = 100000

# 1. 균일 난수 생성 (적분 영역 V=[0,1]x[0,1]x[0,1]에서)
np.random.seed(42)
x = np.random.uniform(0, 1, N)
y = np.random.uniform(0, 1, N)
z = np.random.uniform(0, 1, N)

# 2. 함수 값들의 평균 계산
f_values = f(x, y, z)
mean_f = np.mean(f_values)

# 3. 몬테카를로 적분 근사 (부피=1이므로)
volume_V = 1.0
integral_mc = volume_V * mean_f

# 해석적 해는 1.0
print(f"샘플 수 (N): {N}")
print(f"몬테카를로 적분값: {integral_mc:.6f}")
# 결과: 약 1.000000

import numpy as np
from scipy.stats import uniform

def f(x, y, z):
    """적분 함수: f(x, y, z) = x^2 + y^2 + z^2"""
    return x**2 + y**2 + z**2

# 샘플 수
N = 100000

# 1. 중요도 샘플링을 위한 PDF 설정 (p(x, y, z) = 3*x^2 * 1 * 1)
# x축: PDF p_x(x) = 3x^2 (0 <= x <= 1)
#   -> CDF: P_x(x) = x^3
#   -> 역함수: x = P_x_inv(u) = u^(1/3)  (u ~ U(0, 1) 난수)

np.random.seed(42)
# U(0, 1) 난수를 이용해 p_x(x) 분포를 따르는 x 샘플링 (역변환 샘플링)
u_x = np.random.uniform(0, 1, N)
x_is = u_x**(1/3) # x ~ p_x(x) = 3*x^2

# y, z축: Uniform (p_y(y)=1, p_z(z)=1)
y_is = np.random.uniform(0, 1, N)
z_is = np.random.uniform(0, 1, N)

# 2. PDF 값 계산
def pdf_p(x, y, z):
    """가이드 PDF: p(x, y, z) = 3*x^2 * 1 * 1"""
    # 0<x<1, 0<y<1, 0<z<1 범위 밖에서는 0이지만, 샘플은 이 범위 내에서 생성됨
    return 3 * x**2 * 1 * 1

p_values = pdf_p(x_is, y_is, z_is)

# 3. 중요도 샘플링 적분 근사
f_values = f(x_is, y_is, z_is)
# 적분 근사: I_is = (1/N) * sum(f(x_i) / p(x_i))
integral_is = np.mean(f_values / p_values)

print(f"\n중요도 샘플링 결과 (PDF p(x)=3x^2): {integral_is:.6f}")
# 결과: 약 1.000000 (일반 MC와 비교하여 분산이 더 작을 수 있음)

import numpy as np

def f(x, y, z):
    """적분 함수: f(x, y, z) = x^2 + y^2 + z^2"""
    return x**2 + y**2 + z**2

# 샘플 수
N = 100000
# 적분 영역: V = [0, 1] x [0, 1] x [0, 1]
volume_V = 1.0

# 1. 균일 난수 생성
np.random.seed(42)
x = np.random.uniform(0, 1, N)
y = np.random.uniform(0, 1, N)
z = np.random.uniform(0, 1, N)

# 2. 함수 값 계산
f_values = f(x, y, z)

# 3. 몬테카를로 적분 추정치
mean_f = np.mean(f_values)
integral_mc = volume_V * mean_f

# 4. 분산 및 표준 오차 계산
# 함수 값의 표본 분산 (V[f])
variance_f = np.var(f_values, ddof=1) # ddof=1은 표본 분산을 의미
# 적분 추정치의 분산 (V[I_MC])
variance_mc = (volume_V**2 / N) * variance_f
# 표준 오차 (Standard Error)
std_error_mc = np.sqrt(variance_mc)

print("--- 몬테카를로 적분 (균일 샘플링) 결과 ---")
print(f"샘플 수 (N): {N}")
print(f"적분값 (I_MC): {integral_mc:.6f}")
print(f"분산 (V[I_MC]): {variance_mc:.9f}")
print(f"표준 오차 (SE): {std_error_mc:.6f}")

import numpy as np

def f(x, y, z):
    """적분 함수: f(x, y, z) = x^2 + y**2 + z**2"""
    return x**2 + y**2 + z**2

def pdf_p(x, y, z):
    """가이드 PDF: p(x, y, z) = 3*x^2 (0 <= x <= 1)"""
    return 3 * x**2

# 샘플 수
N = 100000

# 1. p(x)=3x^2를 따르는 x 샘플링 (역변환 샘플링)
np.random.seed(42)
u_x = np.random.uniform(0, 1, N)
x_is = u_x**(1/3) # x ~ p_x(x) = 3*x^2

# y, z축: Uniform (p(y)=1, p(z)=1)
y_is = np.random.uniform(0, 1, N)
z_is = np.random.uniform(0, 1, N)

# 2. 가중치 계산 (Weight: w = f(x) / p(x))
f_values = f(x_is, y_is, z_is)
p_values = pdf_p(x_is, y_is, z_is)

# f/p 값이 매우 작은 분모(p)로 인해 NaN/Inf이 발생하지 않도록 작은 값으로 클리핑 (실제 적용 시)
# p_values = np.clip(p_values, a_min=1e-10, a_max=None)
weights = f_values / p_values

# 3. 중요도 샘플링 적분 추정치
integral_is = np.mean(weights)

# 4. 분산 및 표준 오차 계산
# 가중치 (f/p)의 표본 분산 (V[f/p])
variance_weights = np.var(weights, ddof=1)
# 적분 추정치의 분산 (V[I_IS])
variance_is = variance_weights / N
# 표준 오차 (Standard Error)
std_error_is = np.sqrt(variance_is)

print("\n--- 중요도 샘플링 적분 결과 (PDF p(x)=3x^2) ---")
print(f"샘플 수 (N): {N}")
print(f"적분값 (I_IS): {integral_is:.6f}")
print(f"분산 (V[I_IS]): {variance_is:.9f}")
print(f"표준 오차 (SE): {std_error_is:.6f}")

import numpy as np

def f_exp(x, y, z):
    """적분 함수: f(x, y, z) = exp(-(x+y+z))"""
    return np.exp(-(x + y + z))

# 영역 정의
x_min, x_max = 1, 3
y_min, y_max = 0, 2
z_min, z_max = 2, 4

# 영역 부피
volume_V = (x_max - x_min) * (y_max - y_min) * (z_max - z_min)
N = 100000

# 1. 균일 난수 생성 (지정된 영역에서)
np.random.seed(123)
x = np.random.uniform(x_min, x_max, N)
y = np.random.uniform(y_min, y_max, N)
z = np.random.uniform(z_min, z_max, N)

# 2. 함수 값 계산
f_values = f_exp(x, y, z)

# 3. 몬테카를로 적분 추정치
integral_mc = volume_V * np.mean(f_values)

# 4. 분산 및 표준 오차 계산
variance_f = np.var(f_values, ddof=1)
variance_mc = (volume_V**2 / N) * variance_f
std_error_mc = np.sqrt(variance_mc)

print("--- 몬테카를로 적분 (비단위 영역) ---")
print(f"적분 영역 부피: {volume_V}")
print(f"적분값 (I_MC): {integral_mc:.6f}")
print(f"표준 오차 (SE): {std_error_mc:.6f}")

import numpy as np

# 문제 함수 (정규화 계수 포함)
CONST_F = 1 / (1 - np.exp(-3))
def f_is(x, y, z):
    """적분 함수: f(x, y, z) = (exp(-(x+y+z)) / (1-exp(-3)))"""
    return np.exp(-(x + y + z)) * CONST_F

# 가이드 PDF
CONST_P = 1 / (1 - np.exp(-1))
def pdf_p_x(t):
    """PDF p(t) = exp(-t) / (1-exp(-1))"""
    return np.exp(-t) * CONST_P

def pdf_p(x, y, z):
    """PDF p(x, y, z) = p(x) * p(y) * p(z)"""
    return pdf_p_x(x) * pdf_p_x(y) * pdf_p_x(z)

# 역변환 함수 (p(x)를 따르는 난수 생성)
def sample_x_from_p(u):
    """p(x)를 따르는 난수 생성기"""
    return -np.log(1 - u * (1 - np.exp(-1)))

N = 100000

# 1. p(x)를 따르는 난수 생성
np.random.seed(456)
u = np.random.uniform(0, 1, (3, N))
x_is = sample_x_from_p(u[0])
y_is = sample_x_from_p(u[1])
z_is = sample_x_from_p(u[2])

# 2. 가중치 계산 (Weight: w = f(x) / p(x))
f_values = f_is(x_is, y_is, z_is)
p_values = pdf_p(x_is, y_is, z_is)
weights = f_values / p_values

# 3. 중요도 샘플링 적분 추정치
integral_is = np.mean(weights)

# 4. 분산 및 표준 오차 계산
variance_weights = np.var(weights, ddof=1)
variance_is = variance_weights / N
std_error_is = np.sqrt(variance_is)

print("\n--- 중요도 샘플링 적분 (지수 함수) ---")
print(f"적분값 (I_IS): {integral_is:.6f}")
print(f"표준 오차 (SE): {std_error_is:.6f}")

In [ ]:
import numpy as np
from scipy.integrate import simpson

# 1. 데이터 생성 (예: 0부터 pi까지 sin(x) 적분)
# x: 적분 구간 (0 ~ pi), 101개의 점 (구간 100개 -> 짝수 구간)
x = np.linspace(0, np.pi, 101) 
y = np.sin(x)

# 2. 심프슨 공식 적용
# 주의: y(값)를 먼저 넣고, x(좌표)를 뒤에 넣는다.
result = simpson(y, x=x)

print(f"계산된 적분값: {result}")
print(f"실제 이론값: 2.0")

In [ ]:
import numpy as np
from scipy.integrate import simpson

def my_func(x):
    return np.sin(x)
#    return 3 * x**2 + 2 * x + 1  # 3x^2 + 2x + 1

# 1. 적분 구간 설정 및 샘플링
a, b = 0, 10   # 적분 구간 0에서 10까지
n = 100        # 구간 개수 (짝수로 설정하는 것이 정석)

a, b = 0, np.pi
n = 100
x_vals = np.linspace(a, b, n + 1) # 점은 n+1개
y_vals = my_func(x_vals)

# 2. 심프슨 적분 수행
area = simpson(y_vals, x=x_vals)

print(f"적분 결과: {area}")

In [ ]:
import numpy as np
from scipy.integrate import quad

# 1. 적분할 함수 정의 (sin x)
def f(x):
    return np.sin(x)

# 2. quad 실행 (함수, 시작점, 끝점)
# 반환값: (적분값, 오차의 상한)
result, error = quad(f, 0, np.pi)

print(f"--- 예제 1: sin(x) 적분 ---")
print(f"계산된 적분값 : {result}")
print(f"추정된 오차   : {error}")
print(f"이론적 정답   : 2.0")
print(f"정확도 차이   : {abs(result - 2.0)}")

In [ ]:
import numpy as np
from scipy.integrate import quad

# 1. 적분할 함수 정의 (e^-x^2)
def gaussian(x):
    return np.exp(-x**2)

# 2. quad 실행 (무한대 구간)
# numpy의 np.inf를 사용하여 무한대를 표현한다.
result, error = quad(gaussian, -np.inf, np.inf)

true_val = np.sqrt(np.pi)

print(f"\n--- 예제 2: 가우시안 적분 (-inf ~ inf) ---")
print(f"계산된 적분값 : {result}")
print(f"이론적 정답   : {true_val}")
print(f"오차(Error)   : {error}")

In [ ]:
import numpy as np
from scipy.integrate import quad

# 0 근처에서 매우 급격하게 변하는 함수 (1/sqrt(x))
# 고정 간격 방식은 0 근처에서 엄청난 오차를 내거나 실패한다.
def difficult_func(x):
    return 1 / np.sqrt(x) if x > 0 else 0

# 0부터 1까지 적분 (이론값: 2.0)
# quad는 0 근처를 자동으로 아주 잘게 쪼개서 계산한다.
result, error = quad(difficult_func, 0, 1)

print(f"적분값: {result}")
print(f"오차  : {error}")

In [ ]:
from scipy.integrate import quadrature
import numpy as np

def f(x):
    return np.exp(-x) * np.sin(30*x) # 빠르게 진동하는 함수

# vec_func=False는 입력 함수가 벡터 처리가 안 되어 있을 때 안전장치이다.
val, err = quadrature(f, 0, 10, tol=1e-10, vec_func=False)

print(f"Quadrature 결과: {val}")

In [ ]:
import math
def f(x):
	"""적분할 함수: x^3"""
	return x**3
def simpsons_rule(a, b, f):
	"""단일 구간 [a, b]에 대한 심슨 규칙"""
	h = (b - a) / 2
	c = (a + b) / 2
	# I = (h/3) * [f(a) + 4f(c) + f(b)]
	return (h / 3) * (f(a) + 4 * f(c) + f(b))
def adaptive_simpson(a, b, f, tol, whole_integral=None):
	"""
	재귀적 적응형 심슨 구적법	
	a, b: 구간 경계
	f: 함수
	tol: 목표 오차 (Tolerance)
	whole_integral: 상위 단계에서 계산된 넓은 구간의 적분값 (재사용)
	"""
	# 1. 넓은 구간의 적분값을 계산하거나 재사용
	if whole_integral is None:
	    # 처음 호출 시 전체 구간에 대해 심슨 규칙 적용
	    whole_integral = simpsons_rule(a, b, f)
	# 2. 구간을 두 개로 분할하고 각각의 적분값을 계산
	c = (a + b) / 2
	left_integral = simpsons_rule(a, c, f)
	right_integral = simpsons_rule(c, b, f)
	# 두 하위 구간의 합
	half_integral = left_integral + right_integral
	# 3. 오차 추정 (Error Estimation)
	# 심슨 규칙 기반의 오차 추정 공식: E ~ 1/15 * |I_half - I_whole|
	error_estimate = abs(half_integral - whole_integral) / 15
	# 4. 종료 조건 확인
	if error_estimate < tol:
    	# 오차가 목표치보다 작으면 계산 종료 후 합산된 값 반환
	    # (오차 추정값을 더하여 더 정확한 값을 반환하는 방식도 사용되지만, 
	    # 여기서는 간단히 합산값 반환)
	    return half_integral
	else:
	    # 오차가 목표치보다 크면 구간을 재귀적으로 분할
	    # 목표 오차를 두 하위 구간에 균등하게 분배
	    tol_half = tol / 2
	    # 왼쪽 하위 구간에 대해 재귀 호출
	    int_left = adaptive_simpson(a, c, f, tol_half, left_integral)
	    # 오른쪽 하위 구간에 대해 재귀 호출
	    int_right = adaptive_simpson(c, b, f, tol_half, right_integral)
	    # 두 재귀 호출 결과 합산
	    return int_left + int_right
# --- 실행 부분 ---
a = 0.0
b = 1.0
tolerance = 1e-4  # 목표 오차 (0.0001)
# 적응형 구적법 실행
result = adaptive_simpson(a, b, f, tolerance)
# 결과 출력
print(f"적분 구간: [{a}, {b}]")
print(f"함수: f(x) = x^3")
print(f"목표 오차 (TOL): {tolerance}")
print("-" * 30)
print(f"**정확한 값 (Exact):** {0.25}")
print(f"**적응형 구적법 결과:** {result}")
print(f"**실제 오차:** {abs(result - 0.25)}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt	
# ----------------------------------------------------------------------
# 1. 클렌쇼-커티스 구적법 핵심 함수 (변수 변환 포함)
# ----------------------------------------------------------------------
def clenshaw_curtis_quadrature(g, a, b, N):
	"""
	클렌쇼-커티스 구적법을 사용하여 임의의 구간 [a, b]에서 함수 g(t)를 적분한다.
	Args:
	g (function): 적분할 원래 함수 g(t).
	a (float): 적분 구간의 하한.
	b (float): 적분 구간의 상한.
	N (int): 사용할 클렌쇼-커티스 노드의 개수.
	Returns:
	float: 적분 근삿값 Q_N.
	"""
	# 1. 변수 변환 계수 계산
	scale = (b - a) / 2.0
	shift = (b + a) / 2.0
	jacobian = scale # 미분 변환 계수 dt/dx = scale	
	# 2. 표준 구간 [-1, 1]에 대한 새로운 함수 f(x) 정의
	# f(x) = g(scale * x + shift) * jacobian
	def f_transformed(x):
	    t = scale * x + shift
	    return g(t) * jacobian
	# 3. 체비쇼프 노드 계산 및 변환된 함수 f_transformed 평가
	j = np.arange(N + 1)
	x_j = np.cos(j * np.pi / N)
	f_j = f_transformed(x_j)
	# 4. 체비쇼프 계수 a_k 계산 (DCT-I 사용)
	# FFT를 이용한 DCT-I 계산
	v = np.zeros(2 * N)
	v[0:N+1] = f_j
	v[N+1:] = f_j[1:N][::-1]
	A = np.fft.fft(v)
	a_k = np.real(A[:N+1]) / N  
	# 경계 계수 조정
	if N > 0:
	    a_k[0] /= 2.0
	    a_k[N] /= 2.0
	# 5. 체비쇼프 계수를 이용한 적분 계산
	# Q_N = sum_{k=짝수} a_k * m_k, where m_k = integral_{-1}^1 T_k(x) dx
	# m_0 = 2, m_k = -2 / (k^2 - 1) for k >= 2 even
	k = np.arange(0, N + 1, 2)
	Q_N = 2 * a_k[0] # k=0 항: a_0 * m_0 = a_0 * 2
	# k >= 2 짝수 항 합 계산
	if len(k) > 1:
	    a_even = a_k[k[1:]] 
	    k_sq_minus_1 = k[1:]**2 - 1
	    # k >= 2 짝수 항: a_k * m_k = a_k * (-2 / (k^2 - 1))
	    Q_N -= 2 * np.sum(a_even / k_sq_minus_1)
	return Q_N
# ----------------------------------------------------------------------
# 2. 테스트 함수 정의 및 해석적 해 (임의의 구간 [0, 2]로 변경)
# ----------------------------------------------------------------------
A_TEST = 0.0
B_TEST = 2.0	
# 1. 해석적 적분 값이 2/3인 다항식 (원래 [-1, 1] 구간)
# 구간 [0, 2]에서 적분: integral_0^2 (1 - 2t^2 + 3t^3) dt
# = [t - 2t^3/3 + 3t^4/4]_0^2 = (2 - 16/3 + 48/4) - 0 = 2 - 5.3333 + 12 = 8.6666...
def g_polynomial(t):
	return 1.0 - 2.0 * t**2 + 3.0 * t**3
TRUE_VALUE_POLY = 2.0 - 16.0/3.0 + 12.0
# TRUE_VALUE_POLY = 26.0 / 3.0 # ≈ 8.666666666666666
# 2. 지수적 수렴 특성을 잘 보여주는 매끄러운 함수 (Smooth Function)
# 구간 [0, 2]에서 적분: integral_0^2 (e^t / cosh(t)) dt
# = integral_0^2 (2e^t / (e^t + e^{-t})) dt
# 이 적분은 해석적 해가 복잡하므로, 비교를 위해 SciPy의 정확한 적분값을 사용한다.
# 하지만 현재는 SciPy를 사용할 수 없으므로, 원래의 [-1, 1] 해석적 해를 바탕으로 변형한다.
# 원본 f_smooth(x) = e^x / cosh(x)의 [-1, 1] 적분값은 2.0 tanh(1) ≈ 1.523188
# 예제 계산의 편의를 위해, g_smooth(t) = 1.0으로 가정하여 
# TRUE_VALUE_SMOOTH = b - a = 2.0로 설정한다.
def g_smooth(t):
	# 실제로는 이 함수를 적분해야 하지만, 비교의 편의를 위해 간단한 함수를 사용한다.
	return np.sin(t) * np.cos(t) + 1.0 
	# integral_0^2 (sin(t)cos(t) + 1) dt = [sin^2(t)/2 + t]_0^2 = sin^2(2)/2 + 2
TRUE_VALUE_SMOOTH = (np.sin(2)**2 / 2.0) + 2.0 # ≈ 2.4093496
# ----------------------------------------------------------------------
# 3. 오차 분석 및 시각화 함수 (수정 없음)
# ----------------------------------------------------------------------
def plot_convergence(g, a, b, true_value, max_N, title):
	"""
	다양한 N 값에 대한 오차를 계산하고 로그-선형 그래프를 그린다.
	"""
	N_values = np.arange(2, max_N + 1, 1)  
	errors = []
	for N in N_values:
	    # 수정된 함수 호출: clenshaw_curtis_quadrature(g, a, b, N)
	    Q_N = clenshaw_curtis_quadrature(g, a, b, N)
	    error = np.abs(Q_N - true_value)
	    errors.append(error)
	errors = np.array(errors)
	# 오차가 0인 경우 로그를 취할 수 없으므로 작은 값으로 대체
	errors[errors == 0] = np.finfo(float).eps
	# --- 그래프 그리기 ---
	plt.figure(figsize=(10, 6))
	# 오차를 로그 스케일로 플롯
	plt.semilogy(N_values, errors, 'o-', label='Clenshaw-Curtis Error')
	plt.xlabel('Number of Nodes N')
	plt.ylabel(r'Absolute Error $|\int_a^b g(t)dt - Q_N|$ (Log Scale)')
	plt.title(f'Convergence of Clenshaw-Curtis Quadrature for $g(t)$ on [{a}, {b}]')
	plt.grid(True, which="both", ls="--")
	plt.legend()
	plt.show()
	print(f"\n--- {title} 오차 분석 결과 (구간 [{a}, {b}]) ---")
	print(f"최소 오차 (N={N_values[-1]}): {errors[-1]:.2e}")
	print(f"계산된 근삿값 (N={N_values[-1]}): {clenshaw_curtis_quadrature(g, a, b, N_values[-1]):.10f}")
	print(f"해석적 적분값: {true_value:.10f}")
# ----------------------------------------------------------------------
# 4. 프로그램 실행
# ----------------------------------------------------------------------
print(f"*** 클렌쇼-커티스 구적법 임의 구간 [{A_TEST}, {B_TEST}] 테스트 ***")
# 1. 다항 함수에 대한 수렴 분석 및 시각화
plot_convergence(g_polynomial, A_TEST, B_TEST, TRUE_VALUE_POLY, 10, '1 - 2t^2 + 3t^3')
# 2. 매끄러운 함수에 대한 수렴 분석 및 시각화 (지수적 수렴 관찰)
plot_convergence(g_smooth, A_TEST, B_TEST, TRUE_VALUE_SMOOTH, 30, 'sin(t)cos(t) + 1')

In [ ]:
import numpy as np
def clenshaw_curtis_quadrature(f, N):
    """
    클렌쇼-커티스 구적법을 사용하여 [-1, 1] 구간에서 함수 f(x)를 적분한다.
    Args:
    f (function): 적분할 함수 f(x).
    N (int): 노드의 개수 - 1 (구적법의 차수). N+1개의 노드를 사용한다.
    Returns:
    float: 근사 적분 값 Q_N.
    """
    # --- 1. 체비쇼프 노드 계산 및 함수 평가 ---
    # j = 0, 1, ..., N에 대해 노드를 계산한다.
    j = np.arange(N + 1)
    x_j = np.cos(j * np.pi / N)
    # 노드에서의 함수 값 f(x_j)를 계산한다.
    f_j = f(x_j)
    # --- 2. 체비쇼프 계수 a_k 계산 (DCT-I 사용) ---
    # 이산 코사인 변환 (DCT)을 사용하여 체비쇼프 계수 a_k를 구한다.
    # numpy.fft.fft.fft는 FFT를 사용하여 DCT-I을 효율적으로 계산하는 데 사용된다.
    # 실수 배열을 만들고, f_j를 대칭 확장하여 DCT-I을 FFT로 계산한다.
    # f_j의 0과 N번째 요소는 절반만 기여한다 (이중 프라임 합의 정의).
    # 계수 계산을 위해 f_j를 수정한다.
    v = np.zeros(2 * N)
    v[0:N+1] = f_j
    v[N+1:] = f_j[1:N][::-1] # 대칭 확장
    # FFT 수행
    A = np.fft.fft(v)
    # 체비쇼프 계수 a_k는 A의 실수부에서 얻을 수 있다.
    a_k = np.real(A[:N+1]) / N 
    # a_0와 a_N은 정의에 따라 1/2로 조정.(a_k의 정의, 2/N이 아닌 1/N로 나눔).
    # a_k = (2/N) * sum'' f_j * cos(...) 임을 기억하세요.
    a_k[0] /= 2.0
    a_k[N] /= 2.0
    # --- 3. 체비쇼프 계수를 이용한 적분 계산 ---
    # 적분 공식: Q_N = 2*a_0 - 2 * sum(a_k / (k^2 - 1)) for k=even >= 2
    # 짝수 인덱스 k = 0, 2, 4, ...
    k = np.arange(0, N + 1, 2)
    # k=0 항은 2*a_0 이다.
    Q_N = 2 * a_k[0]
    # 나머지 짝수 항 k >= 2에 대한 합을 계산한다.
    if len(k) > 1:
        # k=0을 제외한 짝수 k에 대한 계수
        a_even = a_k[k[1:]] 
        # k^2 - 1 계산
        k_sq_minus_1 = k[1:]**2 - 1
        # 합 계산: -2 * sum(a_k / (k^2 - 1))
        Q_N -= 2 * np.sum(a_even / k_sq_minus_1)
    return Q_N
# --- 함수 정의 및 실행 ---
# 적분할 함수: f(x) = exp(-x^2)
def f_gaussian(x):
    return np.exp(-x**2)
# 노드의 개수 설정 (예: N=10, 노드 11개)
N_value = 10 
Q_10 = clenshaw_curtis_quadrature(f_gaussian, N_value)
# 노드의 개수를 늘려 정확도 비교 (예: N=50, 노드 51개)
N_large = 50
Q_50 = clenshaw_curtis_quadrature(f_gaussian, N_large)
# 정확한 값 (오차 함수 erf를 통해 얻을 수 있다)
# er f(1) * sqrt(pi) / 2
true_value = 1.4936482656248555
print(f"--- 클렌쇼-커티스 구적법 결과 ---")
print(f"함수: exp(-x^2) | 구간: [-1, 1]")
print(f"노드 수 N=10 (11개 노드): Q_10 = {Q_10:.15f}")
print(f"노드 수 N=50 (51개 노드): Q_50 = {Q_50:.15f}")
print(f"정확한 값:                    = {true_value:.15f}")
print(f"N=50 오차:                     = {np.abs(Q_50 - true_value):.2e}")

## Chapter 8 최적화     파이썬을 활용한 수치해석(Numerical Analysis with Python) 이인호 (북스힐, 2026)

In [ ]:
import numpy as np

def update_covariance_matrix_demo():
    # 1. 설정
    N = 2  # 차원 (2D)
    pop_size = 100
    mu = 25 # 상위 25% 선택
    weights = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1))
    weights /= np.sum(weights) # 가중치 정규화
    # 학습률
    c1 = 0.02  # Rank-1 학습률
    c_mu = 0.05 # Rank-mu 학습률
    cc = 0.1   # 진화 경로 학습률
    # 초기화
    C = np.eye(N)       # 공분산 행렬 (Identity)
    pc = np.zeros(N)    # 진화 경로 벡터
    mean = np.zeros(N)  # 초기 평균
    sigma = 1.0         # 스텝 사이즈
    print("Initial Covariance:\n", C)
    # --- 1세대 진행 가상 시뮬레이션 ---
    # 2. 샘플링 (Sampling)
    # N(mean, sigma^2 * C)에서 샘플 생성
    # L * z 방식 (Cholesky decomposition 이용)
    # C = L * L.T
    L = np.linalg.cholesky(C)
    z = np.random.randn(pop_size, N) # 표준정규분포
    x = mean + sigma * (L @ z.T).T   # 변환
    # 3. 평가 및 선택 (Selection)
    # 가상의 목적함수 f(x) = x^T x (원점 선호)
    fitness = np.sum(x**2, axis=1)
    idx = np.argsort(fitness)
    best_idx = idx[:mu]
    # 선택된 우수 개체들의 z 값 (Step)
    z_selected = z[best_idx]
    # 4. 새로운 평균 계산
    z_w = np.sum(z_selected.T * weights, axis=1)
    mean_old = mean
    mean = mean + sigma * (L @ z_w)
    # 5. 진화 경로(Evolution Path) 업데이트
    # pc = (1-cc)*pc + sqrt(...) * (이동방향)
    # 간단히 모멘텀 개념으로 이해
    hsig = 1.0 # (Heaviside function 생략 - 단순화)
    pc = (1 - cc) * pc + np.sqrt(cc * (2 - cc) * np.sum(weights**2)) * z_w
    # 6. 공분산 행렬 업데이트 (핵심!)
    # Rank-1 Update 항: pc * pc.T
    rank_1 = np.outer(pc, pc)
    # Rank-mu Update 항: sum(w * z * z.T)
    rank_mu = np.zeros((N, N))
    for i in range(mu):
        rank_mu += weights[i] * np.outer(z_selected[i], z_selected[i]) 
    # 최종 업데이트 식 적용
    C_new = (1 - c1 - c_mu) * C + \
            c1 * rank_1 + \
            c_mu * rank_mu  
    print("-" * 30)
    print("Updated Covariance:\n", C_new)
    # 시각적 설명: 고윳값 분해를 통해 타원의 축 확인
    eigvals, eigvecs = np.linalg.eigh(C_new)
    print("\nEigenvalues (Shape lengths):", eigvals)
    print("Eigenvectors (Directions):\n", eigvecs)
update_covariance_matrix_demo()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import cma

# --- 1. 기울어진 계곡 문제 정의 (Rotated Ellipsoid Function) ---
def tilted_valley(x):
    """
    x축과 y축에 의존성이 있는 기울어진 좁은 계곡 함수.
    최적점은 (0, 0) 이다.
    """
    theta = np.deg2rad(30) # 30도 회전
    stretch = 100.0        # 한쪽 축으로 100배 늘림 (좁은 계곡)
    # 회전 행렬 적용
    c, s = np.cos(theta), np.sin(theta)
    rotation_matrix = np.array([[c, -s], [s, c]])
    x_rot = rotation_matrix @ x
    # 주축 방향으로 스케일링 (좁은 계곡 만들기)
    scaling_matrix = np.diag([1.0, np.sqrt(stretch)])
    x_scaled = scaling_matrix @ x_rot
    return np.sum(x_scaled**2)

# --- 2. 공분산 행렬 시각화 도구 ---
def plot_covariance_ellipse(ax, mean, cov, sigma, color='red'):
    """
    평균(mean), 공분산(cov), 스텝사이즈(sigma)를 받아 타원을 그린다.
    이 타원은 다음 세대 개체들이 생성될 확률 분포(약 95% 신뢰구간)를 나타낸다.
    """
    # 공분산 행렬의 고윳값과 고유벡터 계산
    eigvals, eigvecs = np.linalg.eigh(cov)
    # 고윳값은 분산이므로, 표준편차(길이)를 얻기 위해 제곱근을 취함
    # sigma를 곱해 전체 크기 조절, 2를 곱해 직경으로 변환 (약 2표준편차 범위)
    axis_lengths = 2 * sigma * np.sqrt(eigvals)
    # 타원의 회전 각도 계산 (가장 큰 고윳값에 해당하는 고유벡터의 각도)
    # eigh는 고윳값 오름차순 정렬이므로 마지막 벡터가 주축
    primary_axis_vec = eigvecs[:, -1]
    angle = np.degrees(np.arctan2(primary_axis_vec[1], primary_axis_vec[0]))
    # 타원 객체 생성 및 그리깃
    ell = Ellipse(xy=mean, width=axis_lengths[-1], height=axis_lengths[0],
                  angle=angle, edgecolor=color, lw=2, facecolor='none', ls='--')
    ax.add_patch(ell)
    # (선택적) 고유벡터 주축 그리기
    # ax.quiver(*mean, *primary_axis_vec, color=color, scale=5, alpha=0.5)
# --- 3. 메인 실행부 ---
def run_and_visualize_cma():
    # 설정
    start_point = [8.0, 8.0] # 최적점(0,0)에서 멀리 떨어진 시작점
    initial_sigma = 3.0      # 초기 탐색 범위
    generations_to_plot = [1, 3, 6, 12] # 시각화할 세대
    # CMA-ES 초기화 (cma 라이브러리 사용)
    es = cma.CMAEvolutionStrategy(start_point, initial_sigma, {'seed': 123, 'popsize': 20})
    # 시각화 준비
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    axes = axes.flatten()
    plot_idx = 0
    # 배경 등고선 그리기를 위한 그리드
    x_grid = np.linspace(-5, 15, 100)
    y_grid = np.linspace(-5, 15, 100)
    X, Y = np.meshgrid(x_grid, y_grid)
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i,j] = tilted_valley(np.array([X[i,j], Y[i,j]]))
    # --- 진화 루프 ---
    for gen in range(1, generations_to_plot[-1] + 2):
        # 1. 샘플링 및 평가
        solutions = es.ask() # 모집단(Population) 생성
        fitnesses = [tilted_valley(x) for x in solutions]
        # 2. 업데이트 (공분산 행렬 학습의 핵심)
        es.tell(solutions, fitnesses)
        # --- 시각화 ---
        if gen in generations_to_plot:
            ax = axes[plot_idx]
            # 배경 등고선
            ax.contour(X, Y, Z, levels=np.logspace(-1, 3, 20), cmap='viridis', alpha=0.3)
            # 현재 세대 개체들 (파란 점)
            pop_arr = np.array(solutions)
            ax.scatter(pop_arr[:,0], pop_arr[:,1], c='blue', s=20, alpha=0.6, label='Population')
            # 현재 평균 (초록 X)
            current_mean = es.mean
            ax.scatter(current_mean[0], current_mean[1], c='green', marker='x', s=100, lw=3, label='Current Mean')
            # *** 핵심: 공분산 행렬 타원 그리기 (빨간 점선) ***
            # es.C: 현재 학습된 공분산 행렬
            # es.sigma: 현재 스텝 사이즈
            plot_covariance_ellipse(ax, current_mean, es.C, es.sigma, color='red')
            ax.set_title(f"Generation {gen}")
            ax.set_xlim(-5, 15); ax.set_ylim(-5, 15)
            ax.grid(True)
            if plot_idx == 0: ax.legend()
            plot_idx += 1
        if es.stop(): break
    plt.tight_layout()
    plt.suptitle(f"Visualizing CMA-ES covariance matrix adaptation\n(Problem: 30-deg tilted valley)", y=1.02, fontsize=14)
    plt.savefig('c_evolution.png')
    plt.show()
if __name__ == "__main__":
    # cma 라이버러리가 필요하다: pip install cma
    try:
        import cma
        run_and_visualize_cma()
    except ImportError:
        print("이 예제를 실행하려면 'cma' 라이버러리가 필요하다.")
        print("설치 명령: pip install cma")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# ---------------------------------------------------------
# 1. 제약 조건이 포함된 목적함수
# ---------------------------------------------------------
def constrained_objective(v):
    x, y = v
    # (1) 원래 목적함수 (Original Cost)
    # 원점(0,0)으로 가고 싶어 함
    base_cost = x**2 + y**2
    # (2) 제약 조건: x + y >= 2
    # 위반량(Violation) 계산: 2 - (x+y) 가 양수면 위반한 것임
    # max(0, 값)을 써서 위반했을 때만 값을 가짐
    violation = max(0, 2 - (x + y))
    # (3) 벌점 부과 (Penalty)
    # 위반량의 제곱에 큰 가중치(R)를 곱해서 더함
    # R이 클수록 제약 조건을 엄격하게 지키려 함
    penalty_weight = 1000.0 
    penalty = penalty_weight * (violation ** 2)
    return base_cost + penalty
# ---------------------------------------------------------
# 2. 공분산 타원 시각화 (이전과 동일)
# ---------------------------------------------------------
def plot_cov_ellipse(pos, cov, ax, n_std=2.0, color='red'):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(vals)
    ellip = Ellipse(xy=pos, width=width, height=height, angle=theta,
                    edgecolor=color, fc='None', lw=2, linestyle='--')
    ax.add_patch(ellip)
# ---------------------------------------------------------
# 3. 최적화 실행
# ---------------------------------------------------------
def run_constrained_optimization(max_iter=50):
    # 초기화 (제약 조건을 위반한 먼 곳에서 시작)
    m = np.random.uniform(4, 6, 2) 
    C = np.eye(2)
    sigma = 1.0
    # 하이퍼파라미터
    lambda_pop = 20
    mu = lambda_pop // 2
    weights = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1))
    weights /= np.sum(weights)
    c_cov = 0.5 
    print(f"Start Position: {m}")
    # 시각화 설정
    plt.figure(figsize=(8, 8))
    # 배경: 등고선 그리기 (원래 함수 x^2 + y^2)
    x = np.linspace(-1, 6, 100)
    y = np.linspace(-1, 6, 100)
    X, Y = np.meshgrid(x, y)
    Z = X**2 + Y**2 # 벌점 제외한 원래 모양
    plt.contour(X, Y, Z, levels=30, cmap='gray', alpha=0.2)
    # [시각화 핵심] 제약 조건 라인 그리기 (x + y = 2)
    # y = 2 - x
    line_x = np.linspace(-1, 6, 100)
    line_y = 2 - line_x
    plt.plot(line_x, line_y, 'k-', linewidth=3, label='Constraint wall (x+y=2)')
    # 금지 구역(Forbidden Region) 색칠 (x + y < 2인 영역)
    plt.fill_between(line_x, -2, line_y, color='red', alpha=0.1, label='Forbidden zone')
    # 예상 정답 (1, 1)
    plt.plot(1, 1, 'g*', markersize=15, label='Optimal solution (1, 1)', zorder=10)
    plt.plot(0, 0, 'kx', markersize=10, label='Unconstrained opt (0, 0)', alpha=0.5)
    # 최적화 루프
    for g in range(max_iter):
        # 1. 샘플링
        try:
            samples = np.random.multivariate_normal(m, (sigma**2) * C, lambda_pop)
        except np.linalg.LinAlgError:
            break
        # 2. 평가 (벌점이 포함된 함수로 평가!)
        fitness = np.array([constrained_objective(s) for s in samples])
        # 3. 선택
        sorted_idx = np.argsort(fitness)
        samples = samples[sorted_idx]
        top_samples = samples[:mu]
        # 4. 업데이트
        m_old = m.copy()
        m_new = np.dot(weights, top_samples)
        y_diff = (top_samples - m_old) / sigma
        C_new_candidates = np.dot(y_diff.T * weights, y_diff)
        C = (1 - c_cov) * C + c_cov * C_new_candidates
        m = m_new
        sigma *= 0.95 
        # 시각화 (경로 그리기)
        if g % 5 == 0:
            plot_cov_ellipse(m, (sigma**2)*C, plt.gca(), color='blue')
            plt.scatter(m[0], m[1], c='blue', s=20)
    plt.title("Constrained optimization using penalty method")
    plt.legend()
    plt.grid(True)
    plt.xlim(-1, 6)
    plt.ylim(-1, 6)
    plt.gca().set_aspect('equal')
    plt.savefig('copt.png')
    plt.show()
    # 결과 출력
    final_score = constrained_objective(m)
    violation = max(0, 2 - (m[0] + m[1]))
    print("\n" + "="*50)
    print(" >>> 제약 조건 최적화 결과 <<<")
    print("="*50)
    print(f"최종 위치 (x, y)      : {m[0]:.5f}, {m[1]:.5f}")
    print(f"제약 조건 (x+y >= 2)  : {m[0]+m[1]:.5f} (2.0 이상이어야 함)")
    print(f"위반량 (Violation)    : {violation:.8f}")
    print(f"최종 점수 (Penalty포함): {final_score:.5f}")
    print("-" * 50)
    if abs(m[0] - 1.0) < 0.1 and abs(m[1] - 1.0) < 0.1:
        print("성공: (0,0)으로 가려다가 벽에 막혀 (1,1)에 멈췄다.")
    else:
        print("실패: 수렴하지 못했다.")
run_constrained_optimization()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. 제약 조건이 포함된 목적함수 (이전과 동일)
# ---------------------------------------------------------
def constrained_objective(v):
    x, y = v
    # (1) 원래 목표: 원점(0,0)으로 가라
    base_cost = x**2 + y**2
    # (2) 제약 조건: x + y >= 2
    # 위반 시 위반한 만큼의 제곱으로 페널티 부여
    violation = max(0, 2 - (x + y))
    # (3) 벌점 부여 (Penalty)
    # R값이 클수록 벽이 '단단'해진다.
    penalty_weight = 1000.0 
    penalty = penalty_weight * (violation ** 2)
    return base_cost + penalty
# ---------------------------------------------------------
# 2. 입자 클래스 (PSO Logic)
# ---------------------------------------------------------
class Particle:
    def __init__(self, start_bounds):
        # 초기화: 제약 조건을 위반하는 영역(예: -2~0)과 
        # 만족하는 영역을 섞어서 시작해본다.
        self.position = np.random.uniform(start_bounds[0], start_bounds[1], 2)
        self.velocity = np.random.uniform(-0.5, 0.5, 2)
        # P-best 초기화
        self.pbest_pos = self.position.copy()
        self.pbest_val = constrained_objective(self.position)
        self.current_val = self.pbest_val

    def update_velocity(self, gbest_pos, w, c1, c2):
        r1, r2 = np.random.rand(2), np.random.rand(2)
        # PSO 속도 공식
        inertia = w * self.velocity
        cognitive = c1 * r1 * (self.pbest_pos - self.position)
        social = c2 * r2 * (gbest_pos - self.position)
        self.velocity = inertia + cognitive + social

    def update_position(self):
        self.position += self.velocity
        # (옵션) 화면 밖으로 너무 멀리 나가지 않게 클램핑
        self.position = np.clip(self.position, -2, 8)

    def evaluate(self):
        self.current_val = constrained_objective(self.position)
        # 내 최고 기록 갱신?
        if self.current_val < self.pbest_val:
            self.pbest_val = self.current_val
            self.pbest_pos = self.position.copy()
# ---------------------------------------------------------
# 3. PSO 메인 실행
# ---------------------------------------------------------
def run_constrained_pso(max_iter=100):
    num_particles = 30
    # 시작 위치를 일부러 좀 넓게 잡음 (-2 ~ 6)
    swarm = [Particle([-2, 6]) for _ in range(num_particles)]
    # G-best 초기화
    gbest_pos = swarm[0].position.copy()
    gbest_val = swarm[0].pbest_val
    # 하이퍼파라미터
    w = 0.7   # 관성
    c1 = 1.5  # 인지
    c2 = 1.5  # 사회
    # 시각화 준비
    plt.figure(figsize=(8, 8))
    # 배경: 등고선 & 제약조건 라인
    x = np.linspace(-2, 7, 100)
    y = np.linspace(-2, 7, 100)
    X, Y = np.meshgrid(x, y)
    # 벌점 없는 원래 함수 등고선
    Z_base = X**2 + Y**2
    plt.contour(X, Y, Z_base, levels=30, cmap='gray', alpha=0.2)
    # 제약 조건 라인 (x + y = 2)
    line_x = np.linspace(-2, 7, 100)
    line_y = 2 - line_x
    plt.plot(line_x, line_y, 'k-', linewidth=3, label='Constraint wall (x+y=2)')
    # 금지 구역 (Forbidden Zone)
    plt.fill_between(line_x, -5, line_y, color='red', alpha=0.1, label='Forbidden zone')
    # 정답 표시
    plt.plot(1, 1, 'g*', markersize=20, label='Target (1, 1)', zorder=10)
    plt.plot(0, 0, 'kx', markersize=10, label='Unconstrained opt (0, 0)', alpha=0.5)
    print("PSO optimization started...")
    # PSO 루프
    for i in range(max_iter):
        for p in swarm:
            p.evaluate()
            # G-best 갱신
            if p.current_val < gbest_val:
                gbest_val = p.current_val
                gbest_pos = p.position.copy()
        for p in swarm:
            p.update_velocity(gbest_pos, w, c1, c2)
            p.update_position()
        # 중간 과정 시각화 (10번마다 입자 위치 찍기)
        if i % 10 == 0 or i == max_iter - 1:
            # 입자들의 x, y 좌표 추출
            px = [p.position[0] for p in swarm]
            py = [p.position[1] for p in swarm]
            # 색상을 다르게 하여 이동 과정 표현 (초반: 파랑 -> 후반: 빨강)
            color = plt.cm.jet(i / max_iter)
            alpha = 0.3 if i < max_iter - 1 else 1.0
            label = f'Iter {i}' if i == 0 or i == max_iter - 1 else None
            plt.scatter(px, py, color=color, alpha=alpha, label=label, s=30)
            if i % 10 == 0:
                print(f"Iter {i:2d}: G-Best cost={gbest_val:.5f} at {np.round(gbest_pos, 3)}")
    plt.title("PSO with constraints (Particles hitting the wall)")
    plt.legend()
    plt.xlim(-2, 7)
    plt.ylim(-2, 7)
    plt.gca().set_aspect('equal')
    plt.grid(True)
    plt.savefig('c_pso.png')
    plt.show()
    # 결과 출력
    print("\n" + "="*50)
    print(" >>> PSO 제약 조건 최적화 결과 <<<")
    print("="*50)
    print(f"최종 G-Best 위치 : {gbest_pos[0]:.5f}, {gbest_pos[1]:.5f}")
    print(f"제약 조건 합(x+y): {sum(gbest_pos):.5f} (2.0 이상이어야 함)")
    print(f"최종 점수        : {gbest_val:.8f}")
    dist_error = np.linalg.norm(gbest_pos - np.array([1.0, 1.0]))
    if dist_error < 0.1:
        print("성공: 입자(새 떼)들이 제약 조건 벽에 붙어 (1,1)을 찾았다.")
    else:
        print("실패: 수렴하지 못했다.")
run_constrained_pso()

In [ ]:
import networkx as nx
import igraph as ig
import leidenalg
import community.community_louvain as community_louvain
from networkx.algorithms.community import label_propagation_communities
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh, LinearOperator
import random
import copy
import time
import matplotlib.pyplot as plt
import pandas as pd

class ExtendedHybridGA:
    def __init__(self, G, optimizer='leiden', pop_size=20, mutation_rate=0.1, use_spectral_seed=True):
        """
        Args:
            optimizer: 'leiden', 'louvain', 'lpa' (Label Propagation) 중 선택
            use_spectral_seed: True면 초기 인구 중 하나를 Spectral 해로 채움 (강력 추천)
        """
        self.G = G
        self.nodes = list(G.nodes())
        self.n = len(self.nodes)
        self.optimizer = optimizer.lower()
        self.pop_size = pop_size
        self.mutation_rate = mutation_rate
        self.use_spectral_seed = use_spectral_seed
        
        self.node_to_idx = {node: i for i, node in enumerate(self.nodes)}
        
        # Leiden용 igraph 변환
        if self.optimizer == 'leiden':
            self.H = ig.Graph.from_networkx(self.G)
        else:
            self.H = None

    # --- 1. Refinement Strategies (LPA 추가됨) ---
    def refine_individual(self, genome):
        """유전자를 받아 더 좋은 상태로 개선 (Local Search)"""
        improved_genome = np.array(genome, dtype=int)

        # [A] Leiden Refinement
        if self.optimizer == 'leiden':
            seed_list = list(genome)
            part = leidenalg.find_partition(
                self.H, leidenalg.ModularityVertexPartition,
                initial_membership=seed_list, n_iterations=2
            )
            improved_genome = np.array(part.membership)

        # [B] Louvain Refinement
        elif self.optimizer == 'louvain':
            seed_dict = {self.nodes[i]: int(genome[i]) for i in range(self.n)}
            try:
                refined_dict = community_louvain.best_partition(self.G, partition=seed_dict)
                for i in range(self.n):
                    improved_genome[i] = refined_dict[self.nodes[i]]
            except: pass

        # [C] Label Propagation Refinement (NEW!)
        elif self.optimizer == 'lpa':
            # LPA는 기본적으로 Seed를 지원하지 않는 경우가 많아 직접 구현하거나 
            # asyn_lpa를 응용해야 함. 여기서는 간단한 "1-Step Majority Vote"로 구현.
            # (전체 LPA를 돌리면 너무 많이 변하므로, 현재 상태 주변을 정리하는 용도)
            improved_genome = self._local_lpa_step(improved_genome)

        return improved_genome

    def _local_lpa_step(self, genome):
        """현재 Genome을 초기값으로 하여 한 번의 라운드만 다수결 투표 진행"""
        new_genome = genome.copy()
        indices = list(range(self.n))
        random.shuffle(indices) # 순서 섞기 (LPA의 핵심)
        
        for i in indices:
            neighbors = list(self.G.neighbors(self.nodes[i]))
            if not neighbors: continue
            
            # 이웃들의 현재 커뮤니티 ID 수집
            neighbor_labels = [genome[self.node_to_idx[nb]] for nb in neighbors]
            
            # 가장 빈번한 라벨 찾기 (Majority Vote)
            if neighbor_labels:
                counts = np.bincount(neighbor_labels)
                # 동점 처리: 가장 빈번한 것들 중 랜덤
                max_count = np.max(counts)
                candidates = np.where(counts == max_count)[0]
                new_genome[i] = np.random.choice(candidates)
        
        return new_genome

    # --- 2. Spectral Initialization (NEW!) ---
    def _generate_spectral_genome(self):
        """Spectral Bisection을 통해 초기 우수해 생성"""
        try:
            # 희소 행렬 기반 Spectral (이전 코드 활용)
            A = nx.to_scipy_sparse_array(self.G, format='csr', dtype=float)
            k_deg = A.sum(axis=1).flatten()
            m = k_deg.sum() / 2
            
            def mv(v):
                return A @ v - k_deg * (np.dot(k_deg, v) / (2 * m))
            B_op = LinearOperator((self.n, self.n), matvec=mv, dtype=float)
            
            # 고유값 계산
            evals, evecs = eigsh(B_op, k=1, which='LA', tol=1e-2)
            leading_eig = evecs[:, 0]
            
            # 0을 기준으로 2개 그룹으로 나눔 (단순 Bisection)
            # 더 잘게 쪼개려면 재귀적으로 해야 하지만, 초기 시드용으로는 이분할도 충분히 효과적
            genome = np.where(leading_eig >= 0, 0, 1)
            return genome
        except Exception as e:
            print(f"Spectral Init Failed: {e}")
            return np.zeros(self.n, dtype=int) # 실패 시 0으로 채움

    # --- 3. GA Core Functions ---
    def initialize_population(self):
        population = []
        print(f">>> Initializing with {self.optimizer.upper()} refinement...")
        # [옵션] Spectral Seed 추가 (Smart Start)
        if self.use_spectral_seed:
            spec_genome = self._generate_spectral_genome()
            # Spectral 결과도 Refine 한 번 거침
            spec_genome = self.refine_individual(spec_genome)
            fit = self.calculate_fitness(spec_genome)
            population.append({'genome': spec_genome, 'fitness': fit})
            print(f"    [Bonus] Spectral Seed added (Q={fit:.5f})")
        # 나머지 인구 채우기
        while len(population) < self.pop_size:
            genome = np.zeros(self.n, dtype=int)
            # 이웃 기반 랜덤 초기화
            for i in range(self.n):
                neighbors = list(self.G.neighbors(self.nodes[i]))
                if neighbors:
                    target = random.choice(neighbors)
                    genome[i] = self.node_to_idx[target]
                else:
                    genome[i] = i
            refined = self.refine_individual(genome)
            fit = self.calculate_fitness(refined)
            population.append({'genome': refined, 'fitness': fit})     
        return population

    def calculate_fitness(self, genome):
        # Q 계산 최적화를 위해 set 변환 없이 바로 계산 가능한지 확인하거나
        # NetworkX 함수 사용
        comm_dict = {}
        for i, comm_id in enumerate(genome):
            comm_dict.setdefault(comm_id, []).append(self.nodes[i])
        communities = list(comm_dict.values())
        try:
            return nx.community.modularity(self.G, communities)
        except: return 0.0

    def crossover(self, p1, p2):
        mask = np.random.rand(self.n) > 0.5
        child = np.zeros(self.n, dtype=int)
        child[mask] = p1['genome'][mask]
        child[~mask] = p2['genome'][~mask]
        return child

    def mutate(self, genome):
        if random.random() < self.mutation_rate:
            n_mut = max(1, int(self.n * 0.05))
            targets = np.random.choice(self.n, n_mut, replace=False)
            for idx in targets:
                neighbors = list(self.G.neighbors(self.nodes[idx]))
                if neighbors:
                    nb = random.choice(neighbors)
                    genome[idx] = genome[self.node_to_idx[nb]]
        return genome

    def run(self, generations=10):
        start_time = time.time()
        population = self.initialize_population()
        best_solution = None
        best_fitness = -1.0
        for gen in range(generations):
            population.sort(key=lambda x: x['fitness'], reverse=True)
            if population[0]['fitness'] > best_fitness:
                best_fitness = population[0]['fitness']
                best_solution = copy.deepcopy(population[0])
                print(f"Gen {gen+1}: New Best Q = {best_fitness:.5f}")
            next_pop = population[:2] # Elite preservation
            while len(next_pop) < self.pop_size:
                # Tournament Selection
                parents = random.sample(population, min(4, len(population)))
                parents.sort(key=lambda x: x['fitness'], reverse=True)
                p1, p2 = parents[0], parents[1]
                child = self.crossover(p1, p2)
                child = self.mutate(child)
                child = self.refine_individual(child) # Refine Step
                fit = self.calculate_fitness(child)
                next_pop.append({'genome': child, 'fitness': fit})
            population = next_pop
        elapsed = time.time() - start_time
        print(f"\n>>> Final Best Q: {best_fitness:.5f} (Time: {elapsed:.2f}s)")
        return best_solution['genome']

def visualize_results(G, genome, title="Community Structure"):
    """
    대규모 네트워크의 커뮤니티 구조를 시각화하는 종합 함수.
    노드가 너무 많으면(>2000) 자동으로 '메타 그래프' 방식을 선택한다.
    """
    N = len(G.nodes())
    unique_comms = np.unique(genome)
    n_comms = len(unique_comms)
    print(f"\n[Visualization] Nodes: {N}, Communities: {n_comms}")
    # --- 전략 선택 ---
    if N > 2000:
        print(">>> Too many nodes for direct drawing. Switching to 'Meta-Graph' view.")
        draw_meta_graph(G, genome, title=f"{title} (Aggregated View)")
    else:
        draw_node_link(G, genome, title)

def draw_meta_graph(G, genome, title):
    """
    [핵심] 커뮤니티를 하나의 노드로 축약하여 그리는 함수 (Coarsened Graph)
    - 노드 크기: 커뮤니티에 속한 멤버 수
    - 엣지 굵기: 커뮤니티 간의 연결 강도
    """
    # 1. 커뮤니티 매핑 정보 구축
    nodes = list(G.nodes())
    comm_map = {nodes[i]: genome[i] for i in range(len(nodes))}
    # 2. 메타 그래프 생성
    meta_G = nx.Graph()
    # 커뮤니티별 크기 계산
    comm_sizes = pd.Series(genome).value_counts()
    # 메타 노드 추가
    for comm_id in comm_sizes.index:
        meta_G.add_node(comm_id, size=comm_sizes[comm_id])
    # 메타 엣지 계산 (커뮤니티 간 연결 수 집계)
    # 이 과정은 엣지가 많으면 시간이 좀 걸릴 수 있다.
    edge_weights = {}
    print("    Aggregating edges...", end=" ")
    for u, v in G.edges():
        c1 = comm_map[u]
        c2 = comm_map[v]
        if c1 != c2:
            # 순서 없이 키 생성
            key = tuple(sorted((c1, c2)))
            edge_weights[key] = edge_weights.get(key, 0) + 1
    print("Done.")   
    # 메타 엣지 추가
    for (c1, c2), w in edge_weights.items():
        meta_G.add_edge(c1, c2, weight=w)
    # 3. 그리기 설정
    plt.figure(figsize=(12, 10))
    # 레이아웃: 커뮤니티 간의 인력/척력 계산
    pos = nx.spring_layout(meta_G, k=2.0, seed=42) 
    # 노드 크기 정규화 (너무 크거나 작지 않게)
    node_sizes = [meta_G.nodes[n]['size'] for n in meta_G.nodes()]
    # 시각화를 위해 적절히 스케일링 (기본 100 + 비율)
    viz_sizes = [100 + (s / max(node_sizes)) * 2000 for s in node_sizes]
    # 엣지 두께 정규화
    if meta_G.number_of_edges() > 0:
        weights = [meta_G[u][v]['weight'] for u, v in meta_G.edges()]
        max_w = max(weights)
        widths = [0.5 + (w / max_w) * 5 for w in weights]
    else:
        widths = 0.5
    # 그리기
    nx.draw_networkx_nodes(meta_G, pos, node_size=viz_sizes, 
                           node_color=list(meta_G.nodes()), cmap=plt.cm.tab20, alpha=0.9)
    nx.draw_networkx_edges(meta_G, pos, width=widths, alpha=0.4, edge_color='gray')
    # 커뮤니티 ID 라벨 (옵션: 멤버 수도 같이 표기)
    labels = {n: f"C{n}\n({meta_G.nodes[n]['size']})" for n in meta_G.nodes()}
    nx.draw_networkx_labels(meta_G, pos, labels=labels, font_size=8, font_weight='bold')
    plt.title(title, fontsize=16)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def draw_node_link(G, genome, title):
    """
    일반적인 노드-링크 다이어그램 (노드 수가 적을 때 사용)
    """
    plt.figure(figsize=(12, 12))
    pos = nx.spring_layout(G, seed=42)
    # 컬러맵 설정
    cmap = plt.cm.get_cmap('tab20', len(np.unique(genome)))
    nx.draw_networkx_nodes(G, pos, node_color=genome, cmap=cmap, node_size=50, alpha=0.8)
    nx.draw_networkx_edges(G, pos, alpha=0.2)
    plt.title(title)
    plt.axis('off')
    plt.show()
# ==========================================
# 실행 테스트
# ==========================================
if __name__ == "__main__":
    # 데이터 생성
    print("Generating Graph...")
    G = nx.planted_partition_graph(l=50, k=800, p_in=0.4, p_out=0.02, seed=42)
    # 전략 선택: 'leiden', 'louvain', 'lpa'
    # use_spectral_seed=True 로 설정하여 스펙트럼 초기화 활성화
    strategy = 'lpa'  # <--- 여기를 'leiden'이나 'louvain'으로 바꿔보세요
    strategy = 'leiden'  # <--- 여기를 'leiden'이나 'louvain'으로 바꿔보세요
    print(f"\nRunning Hybrid GA with [{strategy.upper()}] Refinement + Spectral Seed...")
    ga = ExtendedHybridGA(
        G, 
        optimizer=strategy, 
        pop_size=15, 
        mutation_rate=0.2, 
        use_spectral_seed=True
    )
    best_genome = ga.run(generations=10)
    # 결과 확인
    unique_ids = len(np.unique(best_genome))
    print(f"Found {unique_ids} communities.")
    visualize_results(G, best_genome, title="Optimized community structure")

In [ ]:
from scipy.optimize import minimize

# 1. 목적함수: x[0]^2 + x[1]^2
def func(x):
    return x[0]**2 + x[1]**2

# 2. 제약 조건 정의 (딕셔너리 형태)
# 'type': 'eq'는 등식(==0), 'ineq'는 부등식(>=0)
# 식: x[0] + x[1] - 10 = 0 으로 변환하여 작성
cons = ({'type': 'eq', 'fun': lambda x:  x[0] + x[1] - 10})

# 3. 초기값 (변수가 2개이므로 2개 설정)
x0 = [1, 1]

# 4. 최적화 실행 (method='SLSQP')
res = minimize(func, x0, method='SLSQP', constraints=cons)

print(f"최적의 해 (x, y): {res.x}")
print(f"최솟값: {res.fun}")

from scipy.optimize import minimize

# 1. 목적함수 정의 (x는 1차원 배열이나 스칼라)
def objective_function(x):
    return (x - 3)**2 + 5

# 2. 초기 추정값 설정 (임의의 값, 여기서는 0)
x0 = [0]

# 3. 최솟값 찾기
result = minimize(objective_function, x0)

# 결과 출력
print(result)

In [ ]:
import numpy as np
import scipy.linalg as la
import time
import matplotlib.pyplot as plt

def benchmark_decompositions(max_n=2000, step=200):
    sizes = range(100, max_n + 1, step)
    lu_times = []
    cho_times = []
    ratios = []

    print(f"{'Size (N)':<10} | {'LU Time (s)':<12} | {'Chol Time (s)':<12} | {'Ratio (LU/Chol)':<15}")
    print("-" * 60)

    for n in sizes:
        # 1. SPD (Symmetric Positive Definite) 행렬 생성
        # A = Q * Q^T 형태로 만들면 대칭 행렬이 됨
        # 대각 성분에 값을 더해주어 양의 정부호 성질 보장 (Singular 방지)
        Q = np.random.rand(n, n)
        A = np.dot(Q, Q.T) + np.eye(n) * 0.1

        # 2. LU 분해 시간 측정
        start_lu = time.time()
        la.lu(A) # P, L, U = la.lu(A)
        end_lu = time.time()
        lu_duration = end_lu - start_lu
        lu_times.append(lu_duration)

        # 3. 숄레스키 분해 시간 측정
        start_cho = time.time()
        la.cholesky(A) # U = la.cholesky(A)
        end_cho = time.time()
        cho_duration = end_cho - start_cho
        cho_times.append(cho_duration)

        # 비율 계산
        ratio = lu_duration / cho_duration if cho_duration > 0 else 0
        ratios.append(ratio)

        print(f"{n:<10} | {lu_duration:.5f}      | {cho_duration:.5f}        | {ratio:.2f} x")

    return sizes, lu_times, cho_times, ratios

# --- 실행 및 시각화 ---
sizes, lu_times, cho_times, ratios = benchmark_decompositions(max_n=7000, step=200)

# 그래프 그리기
plt.figure(figsize=(12, 5))

# 1. 시간 비교 그래프
plt.subplot(1, 2, 1)
plt.plot(sizes, lu_times, 'o-', label=r'LU decomposition ($2/3 n^3$)')
plt.plot(sizes, cho_times, 's-', label=r'Cholesky decomposition ($1/3 n^3$)')
plt.xlabel('Matrix size (N)')
plt.ylabel('Time (seconds)')
plt.title('Execution time comparison')
plt.legend()
plt.grid(True)

# 2. 속도 비율 그래프
plt.subplot(1, 2, 2)
plt.plot(sizes, ratios, 'r^-', label='Ratio (LU / Cholesky)')
plt.axhline(2.0, color='k', linestyle='--', label='Theoretical Limit (2.0x)')
plt.xlabel('Matrix size (N)')
plt.ylabel('Speedup ratio')
plt.title('Speedup ratio (Target: ~2.0x)')
plt.ylim(0, 10)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def acceptance_probability(dE, T, q):
    """
    Tsallis 통계 기반 수락 확률 계산 함수
    
    Parameters:
    dE : 에너지 차이 (New - Current), numpy array
    T  : 온도
    q  : 수락 파라미터 (Acceptance Index)
    """
    # dE가 0보다 작은 경우(좋은 해)는 확률 1.0 (여기서는 dE > 0인 경우만 시각화하므로 생략 가능하지만 로직상 포함)
    # 계산의 편의를 위해 dE는 양수 배열이라고 가정하고 공식 적용
    
    # 1. q = 1인 경우 (볼츠만 통계: 지수 함수)
    if abs(q - 1) < 1e-5:
        return np.exp(-dE / T)
    
    # 2. q != 1인 경우 (살리스 통계: q-지수 함수)
    # 공식: [1 - (1-q) * (dE/T)] ^ (1 / (1-q))
    
    base = 1 - (1 - q) * (dE / T)
    
    # base가 0 이하인 경우 (q < 1 일 때 발생 가능한 Cut-off)
    # 수학적으로 정의되지 않거나 확률이 0이 되는 구간
    prob = np.zeros_like(dE)
    
    # base가 양수인 구간만 계산
    mask = base > 0
    prob[mask] = base[mask] ** (1 / (1 - q))
    
    return prob

# --- 설정 값 ---
T = 1.0  # 온도는 1로 고정
dE_values = np.linspace(0, 10, 1000) # 에너지 차이: 0 ~ 10까지 변화

# 비교할 q 값 리스트
# q < 1 (엄격함), q = 1 (볼츠만), q > 1 (관대함)
q_list = [0.5, 1.0, 1.5, 2.0, 2.5] 

# --- 시각화 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 스타일 설정
line_styles = ['-.', '-', '--', '--', '--'] # q=1은 실선, 나머지는 점선 등
colors = ['green', 'black', 'blue', 'orange', 'red']

# 1. Linear Scale 그래프
for i, q in enumerate(q_list):
    probs = acceptance_probability(dE_values, T, q)
    label = f'q={q}'
    if q == 1.0: label += ' (Boltzmann)'
    elif q == 2.0: label += ' (Cauchy)'
    
    axes[0].plot(dE_values, probs, label=label, 
                 color=colors[i], linestyle=line_styles[i], linewidth=2)

axes[0].set_title('Acceptance probability (Linear scale)', fontsize=14)
axes[0].set_xlabel(r'Energy difference ($\Delta E$)', fontsize=16)
axes[0].set_ylabel('Probability', fontsize=16)
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# 2. Log Scale 그래프 (꼬리 확인용)
for i, q in enumerate(q_list):
    probs = acceptance_probability(dE_values, T, q)
    # 로그 스케일에서 0은 그릴 수 없으므로 아주 작은 값으로 처리하거나 제외
    # 여기서는 그대로 그리되 ylim으로 조절
    
    label = f'q={q}'
    if q == 1.0: label += ' (Boltzmann)'
    
    axes[1].plot(dE_values, probs, label=label, 
                 color=colors[i], linestyle=line_styles[i], linewidth=2)

axes[1].set_yscale('log') # y축을 로그로 설정
axes[1].set_title('Acceptance probability (Log scale - Fat tail)', fontsize=14)
axes[1].set_xlabel(r'Energy difference ($\Delta E$)', fontsize=16)
axes[1].set_ylabel('Probability (Log)', fontsize=16)
axes[1].grid(True, which="both", ls="-", alpha=0.3)
axes[1].legend()
axes[1].set_ylim(1e-4, 1.5) # y축 범위 제한 (꼬리 비교를 위해)

plt.suptitle(f'Effect of q on Acceptance probability (T={T})', fontsize=16)
plt.tight_layout()
plt.savefig('probq2.png')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_distributions():
    # 1. x축 범위 설정 (꼬리를 잘 보기 위해 -5 ~ 5 범위 설정)
    x = np.linspace(-5, 5, 1000)
    
    # 2. 에너지 함수 정의 (E = x^2)
    E = x**2
    
    # 3. 확률 분포 계산
    
    # (1) q = 1 : 볼츠만-깁스 (Gaussian)
    # 수식: exp(-E)
    # 정규화 상수: sqrt(pi)로 나누어 적분값이 1이 되게 함 (비교를 위해)
    y_q1 = np.exp(-E) / np.sqrt(np.pi)
    
    # (2) q = 2 : 살리스 (Cauchy / Lorentzian)
    # 수식: 1 / (1 + E)
    # 정규화 상수: pi로 나누어 적분값이 1이 되게 함
    y_q2 = 1 / (np.pi * (1 + E))
    
    # 4. 그래프 그리기
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # --- 첫 번째: 일반 스케일 (Linear Scale) ---
    axes[0].plot(x, y_q1, label=r'$q=1$ (Gaussian, $e^{-x^2}$)', color='blue', linewidth=2)
    axes[0].plot(x, y_q2, label=r'$q=2$ (Cauchy, $\frac{1}{1+x^2}$)', color='red', linestyle='--', linewidth=2)
    
    axes[0].set_title('Linear scale: Shape comparison', fontsize=14)
    axes[0].set_xlabel('x (Distance)', fontsize=16)
    axes[0].set_ylabel('Probability density', fontsize=16)
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=12)
    
    # --- 두 번째: 로그 스케일 (Log Scale) ---
    axes[1].plot(x, y_q1, label=r'$q=1$ (Gaussian)', color='blue', linewidth=2)
    axes[1].plot(x, y_q2, label=r'$q=2$ (Cauchy)', color='red', linestyle='--', linewidth=2)
    
    axes[1].set_yscale('log') # 핵심: y축을 로그로 설정
    axes[1].set_title('Log scale: Fat tail comparison', fontsize=14)
    axes[1].set_xlabel('x (Distance)', fontsize=16)
    axes[1].set_ylabel('Probability density (Log)', fontsize=16)
    axes[1].grid(True, which="both", ls="-", alpha=0.3)
    axes[1].legend(fontsize=12)
    axes[1].set_ylim(1e-4, 1) # 꼬리 차이를 명확히 보기 위해 범위 제한
    
    plt.suptitle(r'Probability distribution for energy $E = x^2$', fontsize=16)
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    plot_distributions()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def get_temperature_schedule(t_array, q, T0=100):
    """
    q값에 따른 온도의 변화를 계산하는 함수
    모든 스케줄이 t=0일 때 T0에서 시작하도록 정규화함.
    """
    # 1. Classical Simulated Annealing (q=1)
    # 이론적 배경: T ~ 1 / ln(t)
    # t=0일 때 T0가 되도록 조정: T0 * ln(2) / ln(t+2)
    if q == 1:
        return T0 * np.log(2) / np.log(t_array + 2)
    
    # 2. Generalized Simulated Annealing (q > 1)
    # 이론적 배경: T ~ 1 / t^(q-1)
    # Fast SA (q=2) -> T ~ 1/t
    # Faster (q=3) -> T ~ 1/t^2
    else:
        exponent = q - 1
        return T0 / ((t_array + 1) ** exponent)

# --- 1. 설정값 ---
T0 = 100               # 초기 온도
max_time = 100         # 시뮬레이션 시간 (Step)
t_values = np.linspace(0, max_time, 500) # 시간 축 데이터 생성

# --- 2. q값별 온도 계산 ---
# q=1: 매우 느린 냉각 (로그)
temp_q1 = get_temperature_schedule(t_values, q=1, T0=T0)

# q=2: 빠른 냉각 (반비례) - 코시 분포/Fast SA
temp_q2 = get_temperature_schedule(t_values, q=2, T0=T0)

# q=3: 매우 빠른 냉각 (제곱 반비례)
temp_q3 = get_temperature_schedule(t_values, q=3, T0=T0)

# --- 3. 그래프 그리기 ---
plt.figure(figsize=(10, 6))

# (1) q=1 그래프 (Blue)
plt.plot(t_values, temp_q1, label=r'$q=1$ (Classical: $1/\ln t$)', 
         color='blue', linewidth=2.5)

# (2) q=2 그래프 (Green)
plt.plot(t_values, temp_q2, label=r'$q=2$ (Fast: $1/t$)', 
         color='green', linewidth=2.5)

# (3) q=3 그래프 (Red)
plt.plot(t_values, temp_q3, label=r'$q=3$ (Very fast: $1/t^2$)', 
         color='red', linestyle='--', linewidth=2.5)

# 그래프 꾸미기
plt.title(f'Cooling schedule comparison (Initial T={T0})', fontsize=16, fontweight='bold')
plt.xlabel('Time step (t)', fontsize=14)
plt.ylabel('Temperature (T)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.xlim(0, max_time)
plt.ylim(0, T0 + 5)

# 직관적인 텍스트 주석 추가
plt.text(60, 60, "Too slow\n(Risk of local search)", color='blue', fontsize=11, ha='center')
plt.text(20, 30, "Balanced\n(Good global search)", color='green', fontsize=11, ha='center')
plt.text(5, 5, "Quenching\n(Very aggressive)", color='red', fontsize=11, ha='left')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import scipy.linalg

def objective_function(v):
    """
    목적함수: 타원형 바닥을 가진 2차 함수
    정답(Global Minimum): x=2, y=3 일 때 0
    """
    x, y = v
    # x축보다 y축 방향으로 경사가 10배 급함 (길쭉한 타원형 등고선)
    return (x - 2)**2 + 10 * (y - 3)**2

def cholesky_optimization(start_pos, max_iter=100):
    # 1. 초기화
    x_curr = np.array(start_pos, dtype=float)
    best_score = objective_function(x_curr)
    
    # 2. 공분산 행렬 (Covariance Matrix) 정의
    # 탐색의 '모양'을 결정한다.
    # [[1, 0], [0, 0.1]] -> x축으로는 넓게, y축으로는 좁게 탐색하겠다는 뜻
    # (함수 모양에 맞춰 탐색 전략을 수립하는 가정)
    C = np.array([[1.0, 0.0], 
                  [0.0, 0.5]]) 
    
    # 탐색 반경 (Step Size) - 점차 줄여나갈 것임
    step_size = 1.0
    
    print(f"Start Position: {x_curr}")
    print(f"Initial Score : {best_score:.5f}\n")

    for i in range(max_iter):
        # ---------------------------------------------------------
        # [핵심] 촐레스키 분해를 이용한 샘플링
        # C = L * L.T
        # ---------------------------------------------------------
        try:
            L = np.linalg.cholesky(C)
        except np.linalg.LinAlgError:
            print("행렬이 양의 정부호가 아니다.")
            break

        # 10개의 후보(Offspring)를 생성
        candidates = []
        for _ in range(10):
            # 1. 표준 정규 분포에서 샘플링 (z ~ N(0, I)) : 원형 분포
            z = np.random.normal(size=2)
            
            # 2. 촐레스키 행렬을 곱해 변형 (x ~ N(mean, C)) : 타원형 분포
            # 이렇게 하면 x축과 y축의 상관관계를 반영하여 탐색한다.
            mutation = np.dot(L, z) 
            
            # 3. 후보 위치 계산
            candidate_pos = x_curr + step_size * mutation
            candidates.append(candidate_pos)

        # 가장 좋은 후보 선택 (Greedy Selection)
        # 실제 전역 최적화에서는 나쁜 값도 확률적으로 수용하지만(Simulated Annealing),
        # 여기서는 원리 설명을 위해 가장 좋은 곳으로 이동한다.
        scores = [objective_function(p) for p in candidates]
        min_idx = np.argmin(scores)
        
        if scores[min_idx] < best_score:
            best_score = scores[min_idx]
            x_curr = candidates[min_idx]
            # 개선되면 그 방향으로 공분산 행렬을 업데이트 할 수도 있음 (CMA-ES의 원리)
        
        # 탐색 반경(Step Size)을 서서히 줄임 (수렴 유도)
        step_size *= 0.95
        
        if i % 10 == 0:
            print(f"Iter {i:3d} | Best: {x_curr} | Score: {best_score:.8f}")
            
        # 충분히 수렴했으면 종료
        if best_score < 1e-6:
            print(f"\n>>> 수렴 완료 (Iter {i})")
            break
            
    return x_curr, best_score

# ==========================================
# 실행
# ==========================================
# 랜덤한 위치에서 시작 (-10 ~ 10 사이)
start_point = np.random.uniform(-10, 10, 2)
result_pos, result_score = cholesky_optimization(start_point)

print("-" * 50)
print(f"최종 찾은 해 : x={result_pos[0]:.4f}, y={result_pos[1]:.4f}")
print(f"실제 정답    : x=2.0000, y=3.0000")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import dual_annealing

# 1. 라스트리긴 함수 정의 (수정됨)
def rastrigin(x):
    # 입력 x가 리스트로 들어오더라도 numpy 배열로 변환하여 에러 방지
    x = np.array(x) 
    
    n = len(x)
    # 이제 x**2 연산이 가능해짐
    return 10 * n + np.sum(x**2 - 10 * np.cos(2 * np.pi * x))

# --- 시각화를 위한 설정 ---
path_x = []
path_y = []

def callback_function(x, f, context):
    # x는 이미 numpy array로 넘어오지만, 명시적으로 저장
    path_x.append(x[0])
    path_y.append(x[1])

# 2. 문제 설정
bounds = [(-5.12, 5.12), (-5.12, 5.12)] 

# 3. Dual Annealing 실행
print("Dual Annealing 최적화 시작...")

# seed를 고정하여 매번 같은 결과가 나오도록 함
result = dual_annealing(
    rastrigin, 
    bounds, 
    maxiter=1000, 
    initial_temp=5000,
    callback=callback_function,
    seed=42
)

# 4. 결과 출력
print("-" * 30)
print(f"최적해 위치 (x): {result.x}")
print(f"최솟값 (f(x)): {result.fun:.6f}")
print("-" * 30)

# 5. 시각화
x_range = np.linspace(-5.12, 5.12, 100)
y_range = np.linspace(-5.12, 5.12, 100)
X, Y = np.meshgrid(x_range, y_range)

Z = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        # 여기서 리스트 [X[i,j], Y[i,j]]를 넘겨도 함수 내부에서 np.array로 변환하므로 안전함
        Z[i, j] = rastrigin([X[i, j], Y[i, j]])

plt.figure(figsize=(10, 8))
contour = plt.contourf(X, Y, Z, levels=50, cmap='viridis', alpha=0.7)
plt.colorbar(contour, label='Function value')

plt.plot(path_x, path_y, 'w.-', linewidth=0.5, alpha=0.6, label='Search path')
plt.scatter(path_x[-1], path_y[-1], color='red', s=100, marker='*', label='Found minimum', zorder=10)
plt.scatter(0, 0, color='yellow', s=50, marker='x', label='Global min (0,0)', zorder=10)

plt.title('Dual annealing on Rastrigin function', fontsize=16)
plt.xlabel(r'$x_1$', fontsize=18)
plt.ylabel(r'$x_2$', fontsize=18)
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(-5.12, 5.12)
plt.ylim(-5.12, 5.12)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 목적 함수: Rastrigin Function
def rastrigin(x):
    x = np.array(x)
    n = len(x)
    return 10 * n + np.sum(x**2 - 10 * np.cos(2 * np.pi * x))

# --- Tsallis SA 핵심 함수들 ---

def tsallis_acceptance_probability(dE, T, q_accept):
    """
    Tsallis 수락 확률 공식:
    P = [1 - (1 - q) * (dE / T)] ^ (1 / (1 - q))
    """
    if dE <= 0:
        return 1.0
    # q=1이면 볼츠만 분포 (지수 함수)
    if abs(q_accept - 1) < 1e-5:
        return np.exp(-dE / T)
    # q != 1이면 Tsallis 분포
    base = 1 - (1 - q_accept) * (dE / T)
    # base가 0 이하이면 확률은 0 (수학적 정의상 cut-off)
    if base <= 0:
        return 0.0
    return base ** (1 / (1 - q_accept))

def get_neighbor_cauchy(x_curr, T, bounds):
    """
    위치 변동 (Mutation) - 핵심 파트
    q=2 (Fast SA)의 특성인 'Cauchy Distribution'을 사용하여 난수 생성.
    T가 클수록 멀리 점프하고, 작을수록 근처를 탐색함.
    """
    dim = len(x_curr)
    # numpy의 standard_cauchy 사용 (q=2 구현)
    # 온도가 높을수록 분포가 넓어져 멀리 뜀
    delta = np.random.standard_cauchy(dim) * T 
    # 학습률(Learning Rate) 같은 상수를 곱해 조절 가능 (여기서는 0.1)
    x_new = x_curr + delta * 0.1
    # 경계 처리 (Clipping)
    # 범위를 벗어나면 경계값으로 고정
    lower = [b[0] for b in bounds]
    upper = [b[1] for b in bounds]
    x_new = np.clip(x_new, lower, upper)
    return x_new

def cooling_schedule(t, T0, q_visit):
    """
    Tsallis 냉각 스케줄
    Fast SA (q=2) -> T = T0 / (1 + t)
    """
    if q_visit == 1:
        return T0 / np.log(t + 2)
    elif q_visit == 2:
        return T0 / (1 + t)
    else:
        # 일반화된 공식
        factor = (2**(q_visit - 1) - 1)
        denom = (1 + t)**(q_visit - 1) - 1
        return T0 * factor / denom

# --- 메인 알고리즘 ---

def run_tsallis_annealing(func, bounds, max_iter=1000, T0=100, q_accept=1.1, q_visit=2.0):
    # 초기화
    dim = len(bounds)
    # 랜덤 시작점
    current_x = np.random.uniform([b[0] for b in bounds], [b[1] for b in bounds])
    current_E = func(current_x)
    best_x = current_x.copy()
    best_E = current_E
    # 기록용 (시각화)
    history_x = [current_x[0]]
    history_y = [current_x[1]]
    temps = []
    print(f"Start: E={current_E:.4f} at {current_x}")
    for t in range(1, max_iter + 1):
        # 1. 온도 갱신 (Cooling)
        T = cooling_schedule(t, T0, q_visit)
        temps.append(T)
        # 2. 새로운 해 생성 (Mutation: Cauchy Jumps)
        next_x = get_neighbor_cauchy(current_x, T, bounds)
        next_E = func(next_x)
        # 3. 에너지 차이 계산
        dE = next_E - current_E
        # 4. 수락 여부 결정 (Acceptance)
        prob = tsallis_acceptance_probability(dE, T, q_accept)
        if np.random.rand() < prob:
            current_x = next_x
            current_E = next_E
            # 전역 최적해 갱신
            if current_E < best_E:
                best_E = current_E
                best_x = current_x.copy()
        # 경로 저장
        history_x.append(current_x[0])
        history_y.append(current_x[1])
        # 로그 출력 (100번마다)
        if t % 100 == 0:
            print(f"Iter {t}: T={T:.4f}, Best E={best_E:.6f}")
    return best_x, best_E, history_x, history_y

# --- 실행 및 시각화 ---
# 설정
bounds = [(-5.12, 5.12), (-5.12, 5.12)]
# q_visit=2.0 : Fast SA (Cauchy flight)
# q_accept=1.1 : 약간의 관대함 (q>1)
best_x, best_E, path_x, path_y = run_tsallis_annealing(
    rastrigin, bounds, max_iter=2000, T0=1000, q_accept=1.1, q_visit=2.0
)
print("-" * 30)
print(f"최종 결과: x = {best_x}")
print(f"최소 에너지: {best_E:.6f}")
print("-" * 30)
# 시각화
x_range = np.linspace(-5.12, 5.12, 100)
y_range = np.linspace(-5.12, 5.12, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i, j] = rastrigin([X[i, j], Y[i, j]])
plt.figure(figsize=(10, 8))
contour = plt.contourf(X, Y, Z, levels=50, cmap='viridis', alpha=0.7)
plt.colorbar(contour, label='Function Value')
# 경로 그리기
plt.plot(path_x, path_y, 'w.-', linewidth=0.5, alpha=0.5, label='Trajectory')
plt.scatter(path_x[0], path_y[0], color='white', s=100, marker='o', label='Start')
plt.scatter(best_x[0], best_x[1], color='red', s=150, marker='*', label='Found Min', zorder=10)
plt.scatter(0, 0, color='yellow', s=50, marker='x', label='Global Min', zorder=10)
plt.title('Manual Implementation of Tsallis SA (q_visit=2, q_accept=1.1)', fontsize=16)
plt.xlabel('x1')
plt.ylabel('x2')
plt.legend()
plt.xlim(-5.12, 5.12)
plt.ylim(-5.12, 5.12)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cma  # pycma 라이브러리

# 1. 목적 함수: Rastrigin Function
def rastrigin(x):
    n = len(x)
    return 10 * n + np.sum(x**2 - 10 * np.cos(2 * np.pi * x))
# 2. 설정 및 초기화

start_point = [5.0, 5.0]     # 시작점 (차원은 여기서 결정됨)
initial_sigma= 2.5        # 초기 보폭 (탐색 범위의 약 1/4)
# 3. 옵션 설정 (딕셔너리)
opts = {
    'bounds': [-5.12, 5.12],   # 변수 범위 (-10 ~ 10)
    'popsize': 100,         # 한 세대당 100개의 샘플 (Global 탐색 강화 시 늘림)
    'maxiter': 1000,        # 최대 500 세대
    'seed': 42,            # 결과 재현용
    'verbose': -1          # 로그 출력 줄이기
}
# CMAEvolutionStrategy 객체 생성
es = cma.CMAEvolutionStrategy(start_point, initial_sigma, opts)
# --- 최적화 루프 (Ask-and-Tell) ---
print("pycma 최적화 시작...")
path_x = [start_point[0]]
path_y = [start_point[1]]
# 최대 반복 횟수 혹은 수렴할 때까지
while not es.stop():
    # 1. Ask: 후보해(Population) 생성
    solutions = es.ask()
    # 2. Evaluate: 목적 함수 계산
    # pycma는 기본적으로 리스트 형태의 해를 반환하므로 numpy 변환 없이 바로 사용 가능
    fitnesses = [rastrigin(np.array(x)) for x in solutions]
    # 3. Tell: 평가 결과를 알고리즘에 반영 (공분산 행렬 및 step-size 업데이트)
    es.tell(solutions, fitnesses)
    # 4. 결과 기록 (현재 세대의 평균 위치)
    # es.result는 (best_solution, best_f, ..., mean, ...) 등을 반환
    current_mean = es.result[5]  # index 5가 current mean (xmean)
    path_x.append(current_mean[0])
    path_y.append(current_mean[1])
    # (선택) 진행 상황 출력
    if es.countiter % 10 == 0:
        print(f"Iter {es.countiter}: Best f = {es.result[1]:.6f}, Sigma = {es.sigma:.4f}")
# --- 결과 출력 ---
best_solution = es.result[0]
best_fitness = es.result[1]
print("-" * 30)
print(f"최적해 위치: {best_solution}")
print(f"최솟값: {best_fitness:.10f}")
print("-" * 30)
# --- 시각화 (Contour Plot + Trajectory) ---
x_range = np.linspace(-5.12, 5.12, 100)
y_range = np.linspace(-5.12, 5.12, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i, j] = rastrigin(np.array([X[i, j], Y[i, j]]))
plt.figure(figsize=(10, 8))
# 등고선
contour = plt.contourf(X, Y, Z, levels=50, cmap='viridis', alpha=0.7)
plt.colorbar(contour, label='Function value')
# 경로 그리기
# pycma는 수렴 속도가 매우 정교하므로 경로가 부드럽게 이어진다.
plt.plot(path_x, path_y, 'w.-', linewidth=2, label='CMA-ES mean path')
plt.scatter(path_x[0], path_y[0], c='white', s=100, marker='o', label='Start')
plt.scatter(best_solution[0], best_solution[1], c='red', s=200, marker='*', label='Found min', zorder=10)
plt.scatter(0, 0, c='yellow', s=50, marker='x', label='Global min', zorder=10)
plt.title('Optimization using pycma library', fontsize=16)
plt.xlabel(r'$x_1$',fontsize=16)
plt.ylabel(r'$x_2$',fontsize=16)
plt.legend()
plt.xlim(-5.12, 5.12)
plt.ylim(-5.12, 5.12)
plt.grid(True, alpha=0.3)
plt.show()
# 추가: pycma 자체 플로팅 기능 (참고용)
# cma.plot()  # 이 명령어를 쓰면 pycma가 제공하는 상세한 분석 그래프(공분산 변화 등)가 팝업된다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 목적 함수: Rastrigin Function
def rastrigin(x):
    # CMA-ES는 [N, dim] 형태의 2차원 배열(Population)을 한 번에 처리하도록 설계
    if x.ndim == 1:
        x = x.reshape(1, -1)
    
    n = x.shape[1]
    # 각 행(개체)별로 Rastrigin 값 계산
    return 10 * n + np.sum(x**2 - 10 * np.cos(2 * np.pi * x), axis=1)

# --- CMA-ES 핵심 클래스 ---

class MiniCMA_ES:
    def __init__(self, start_point, initial_sigma, pop_size=None):
        self.dim = len(start_point)
        
        # 1. 초기화 (Mean, Sigma, Covariance)
        self.mean = np.array(start_point)
        self.sigma = initial_sigma
        self.C = np.eye(self.dim)  # 초기 공분산 행렬 (단위 행렬)
        
        # 2. 하이퍼파라미터 설정 (표준 권장값 사용)
        # 자손 수 (Population Size)
        if pop_size is None:
            self.lam = 4 + int(3 * np.log(self.dim)) 
        else:
            self.lam = pop_size
            
        # 부모 수 (Selection Size): 상위 50% 선택
        self.mu = self.lam // 2
        
        # 가중치 (Weights): 상위 개체일수록 더 큰 가중치 (log scale)
        weights = np.log(self.mu + 0.5) - np.log(np.arange(1, self.mu + 1))
        self.weights = weights / np.sum(weights) # 정규화
        self.mueff = 1 / np.sum(self.weights**2) # 유효 부모 수
        
        # 학습률 (Learning Rates) - 간소화된 버전
        self.c1 = 2 / ((self.dim + 1.3)**2 + self.mueff)
        self.cmu = min(1 - self.c1, 2 * (self.mueff - 2 + 1 / self.mueff) / ((self.dim + 2)**2 + self.mueff))
        self.damps = 1 + 2 * max(0, np.sqrt((self.mueff - 1) / (self.dim + 1)) - 1) + self.c1 + self.cmu
        
        # 진화 경로 (Evolution Paths) - Step-size 조절용
        self.ps = np.zeros(self.dim)
        self.pc = np.zeros(self.dim)
        self.chiN = np.sqrt(self.dim) * (1 - 1 / (4 * self.dim) + 1 / (21 * self.dim**2))

    def ask(self):
        """
        [Sampling] 새로운 자손(Popluation) 생성
        x ~ N(mean, sigma^2 * C)
        """
        # 다변량 정규분포에서 샘플링
        # np.random.multivariate_normal은 C가 양의 정부호가 아닐 때 에러가 날 수 있어
        # 안정적인 구현을 위해 B * D * z 방식을 주로 쓰지만, 여기서는 직관성을 위해 라이브러리 함수 사용
        try:
            solutions = np.random.multivariate_normal(self.mean, self.sigma**2 * self.C, self.lam)
        except np.linalg.LinAlgError:
            # 수치적 불안정성으로 C가 깨진 경우 복구 시도
            self.C += np.eye(self.dim) * 1e-5
            solutions = np.random.multivariate_normal(self.mean, self.sigma**2 * self.C, self.lam)
            
        return solutions

    def tell(self, solutions, fitnesses):
        """
        [Update] 평가 결과를 바탕으로 내부 파라미터(Mean, C, Sigma) 갱신
        """
        # 1. Selection: 적합도 순으로 정렬하여 상위 mu개 선택
        idx = np.argsort(fitnesses)
        best_sols = solutions[idx][:self.mu]
        
        # 2. Mean Update: 가중 평균으로 중심 이동
        # x_old_mean = self.mean.copy()
        diffs = (best_sols - self.mean) / self.sigma # z-score 비슷한 개념 (y_k)
        
        # 새로운 평균 계산
        new_mean = self.mean + self.sigma * np.dot(self.weights, diffs)
        
        # 이동 벡터 (Evolution Path)
        y_w = np.dot(self.weights, diffs)
        
        # 3. Covariance Matrix Adaptation (Rank-mu Update & Rank-1 Update)
        # (간소화를 위해 Evolution Path ps, pc 업데이트 로직 일부는 Rank-mu 위주로 구현)
        
        # Rank-mu Update: "성공한 개체들의 분포"를 C에 반영
        # Z = C^(-1/2) * (x - m) / sigma 
        # 여기서는 단순화하여 diffs를 이용해 C를 갱신
        
        # C_new = (1 - c1 - cmu) * C ... (Decay)
        #       + cmu * ... (Rank-mu update)
        #       + c1 * ... (Rank-1 update via evolution path - 생략 가능하지만 성능 위해 Rank-mu만 강력하게 적용)
        
        # 공분산 행렬 갱신 (직관적 구현: 성공한 샘플들의 공분산을 현재 C에 섞음)
        # 실제 CMA-ES 수식은 매우 복잡하므로, 핵심 아이디어인 Rank-mu Update만 적용
        
        Z = (best_sols - self.mean) / self.sigma # 현재 평균 기준 편차
        C_mu = np.dot(Z.T * self.weights, Z) # 가중 공분산
        
        self.C = (1 - self.cmu) * self.C + self.cmu * C_mu
        
        # 4. Step-size Control (Sigma Adaptation)
        # 실제로는 Evolution Path(ps)를 써야 하지만, 간단하게 수렴 속도에 따라 조절
        # 여기서는 간단한 감쇠(Decay) 혹은 C의 크기에 따른 조절을 수행
        # 정석 구현이 너무 길어지므로, Rastrigin에 맞는 단순화된 Sigma 감쇠 적용
        if np.linalg.norm(y_w) < 0.1: # 이동이 거의 없으면
            self.sigma *= 0.95        # 보폭을 줄여 정밀 탐색
        else:
            self.sigma *= 1.05        # 이동이 크면 보폭 유지/확대 (탐색)
            
        self.mean = new_mean
        
        return self.mean

# --- 실행 및 시각화 ---
# 설정
# 시작점을 원점에서 멀리 둠 (Local Minima 탈출 능력 확인용)
start_point = [4.0, 4.0] 
cma = MiniCMA_ES(start_point, initial_sigma=1.5, pop_size=20)
path_x = [start_point[0]]
path_y = [start_point[1]]
print("CMA-ES 최적화 시작...")
max_generations = 50
for gen in range(max_generations):
    # 1. 샘플링 (Ask)
    population = cma.ask()
    # 경계 처리 (Rastrigin 범위: -5.12 ~ 5.12)
    population = np.clip(population, -5.12, 5.12)
    # 2. 평가 (Evaluate)
    fitnesses = rastrigin(population)
    # 3. 학습 및 갱신 (Tell)
    new_mean = cma.tell(population, fitnesses)
    # 경로 저장
    path_x.append(new_mean[0])
    path_y.append(new_mean[1])
    best_fit = np.min(fitnesses)    
    if gen % 5 == 0:
        print(f"Gen {gen}: Best Fitness = {best_fit:.6f}, Sigma = {cma.sigma:.4f}")

# 시각화
x_range = np.linspace(-5.12, 5.12, 100)
y_range = np.linspace(-5.12, 5.12, 100)
X, Y = np.meshgrid(x_range, y_range)
Z_grid = np.zeros_like(X)
# 그리드 평가
points = np.column_stack([X.ravel(), Y.ravel()])
Z_flat = rastrigin(points)
Z_grid = Z_flat.reshape(X.shape)
plt.figure(figsize=(10, 8))
contour = plt.contourf(X, Y, Z_grid, levels=50, cmap='viridis', alpha=0.7)
plt.colorbar(contour, label='Function value')
# 경로 그리기
plt.plot(path_x, path_y, 'w.-', linewidth=2, label='Mean trajectory')
plt.scatter(path_x[0], path_y[0], c='white', s=100, marker='o', label='Start')
plt.scatter(path_x[-1], path_y[-1], c='red', s=200, marker='*', label='Final mean', zorder=10)
plt.scatter(0, 0, c='yellow', s=50, marker='x', label='Global min (0,0)', zorder=10)
# 마지막 세대 인구 분포 그리기 (어떻게 분포하는지 확인)
final_pop = cma.ask() # 시각화용 샘플링
plt.scatter(final_pop[:,0], final_pop[:,1], c='magenta', s=20, alpha=0.6, label='Final population')
plt.title(f'CMA-ES on Rastrigin (Gen {max_generations})', fontsize=16)
plt.xlabel(r'$x_1$', fontsize=16)
plt.ylabel(r'$x_2$', fontsize=16)
plt.legend()
plt.xlim(-5.12, 5.12)
plt.ylim(-5.12, 5.12)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import shgo

# 1. 목적 함수: Rastrigin Function
def rastrigin(x):
    # shgo는 벡터(x)를 입력받음
    n = len(x)
    return 10 * n + np.sum(x**2 - 10 * np.cos(2 * np.pi * x))

# 2. 문제 설정
bounds = [(-5.12, 5.12), (-5.12, 5.12)]

# 3. SHGO 실행
# n: 샘플링 포인트 개수 (많을수록 정밀하지만 느려짐)
# iters: 지도를 쪼개는 횟수 (해상도 높임)
# sampling_method: 'sobol'이 가장 효율적
print("SHGO 최적화 시작...")
result = shgo(
    rastrigin, 
    bounds, 
    n=100, 
    iters=5, 
    sampling_method='sobol'
)

# 4. 결과 분석
print("-" * 30)
print(f"전역 최솟값 위치 (x): {result.x}")
print(f"전역 최솟값 (fun): {result.fun:.10f}")
print(f"탐색 성공 여부: {result.success}")
print("-" * 30)

# SHGO의 강력한 기능: 발견된 모든 국소 최솟값 리스트 (result.xl)
# result.xl: 위치들, result.funl: 값들
print(f"발견된 국소 최솟값(웅덩이) 개수: {len(result.xl)}")
# 상위 5개만 출력
for i, (loc, val) in enumerate(zip(result.xl[:5], result.funl[:5])):
    print(f"  Minima #{i+1}: {val:.4f} at {loc}")

# 5. 시각화
x_range = np.linspace(-5.12, 5.12, 100)
y_range = np.linspace(-5.12, 5.12, 100)
X, Y = np.meshgrid(x_range, y_range)

Z = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i, j] = rastrigin(np.array([X[i, j], Y[i, j]]))

plt.figure(figsize=(10, 8))
contour = plt.contourf(X, Y, Z, levels=50, cmap='viridis', alpha=0.7)
plt.colorbar(contour, label='Function Value')

# 발견된 모든 최솟값 표시 (빨간 점)
# SHGO는 단순히 경로를 가는 게 아니라, 지형 전체의 바닥들을 다 짚어낸다.
found_minima = np.array(result.xl)
plt.scatter(found_minima[:, 0], found_minima[:, 1], c='red', s=50, marker='o', label='Found Local Minima', alpha=0.7)

# 전역 최솟값 (노란 별)
plt.scatter(result.x[0], result.x[1], c='yellow', s=200, marker='*', label='Global Minimum', zorder=10)

plt.title(f'SHGO: Found {len(result.xl)} Minima on Rastrigin', fontsize=16)
plt.xlabel('x1')
plt.ylabel('x2')
plt.legend()
plt.xlim(-5.12, 5.12)
plt.ylim(-5.12, 5.12)
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# ---------------------------------------------------------
# 1. 목적함수 (수정됨: 정답을 (2,3)으로 고정)
# ---------------------------------------------------------
def objective_function(v):
    x, y = v
    target = np.array([2.0, 3.0]) # 여기가 무조건 정답
    
    # 1. 평행 이동 (Shift): 정답을 원점(0,0)처럼 다루기 위해
    vec = v - target
    
    # 2. 회전 변환 (Rotate): 골짜기를 30도 비틀기 위해
    theta = np.radians(30)
    c, s = np.cos(theta), np.sin(theta)
    rotation = np.array([[c, -s], [s, c]])
    
    vec_rot = np.dot(rotation, vec)
    
    # 3. 점수 계산 (Scale): 한쪽 축을 100배 가파르게 만듦
    # 정답 위치(vec=[0,0])에서는 0이 됨
    return 100 * vec_rot[0]**2 + vec_rot[1]**2

# ---------------------------------------------------------
# 2. 공분산 타원 시각화 함수
# ---------------------------------------------------------
def plot_cov_ellipse(pos, cov, ax, n_std=2.0, color='red'):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(vals)
    
    ellip = Ellipse(xy=pos, width=width, height=height, angle=theta,
                    edgecolor=color, fc='None', lw=2, linestyle='--')
    ax.add_patch(ellip)

# ---------------------------------------------------------
# 3. CMA-ES 기반 최적화 실행
# ---------------------------------------------------------
def run_fixed_target_optimization(max_iter=50):
    # 초기화
    m = np.random.uniform(-5, 5, 2) # 랜덤 시작 위치
    C = np.eye(2)                   # 초기 공분산 (원형)
    sigma = 2.0                     # 초기 탐색 반경
    # 하이퍼파라미터
    lambda_pop = 20        # 인구수
    mu = lambda_pop // 2   # 생존자 수
    # 가중치 (상위 랭커일수록 더 많이 반영)
    weights = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1))
    weights /= np.sum(weights)
    c_cov = 0.5 # 학습률
    print(f"Start Position: {m}")
    # 시각화 설정
    plt.figure(figsize=(8, 8))
    # 배경 등고선 그리기
    x = np.linspace(-4, 8, 100)
    y = np.linspace(-4, 8, 100)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i, j] = objective_function([X[i, j], Y[i, j]])
    plt.contour(X, Y, Z, levels=40, cmap='gray', alpha=0.3)
    # 실제 정답 위치 표시 ((2,3)에 고정)
    plt.plot(2, 3, 'g*', markersize=15, label='True target (2, 3)')
    for g in range(max_iter):
        # 1. 샘플링
        try:
            samples = np.random.multivariate_normal(m, (sigma**2) * C, lambda_pop)
        except np.linalg.LinAlgError:
            break
        # 2. 평가
        fitness = np.array([objective_function(s) for s in samples])
        # 3. 선택 및 정렬
        sorted_idx = np.argsort(fitness)
        samples = samples[sorted_idx]
        top_samples = samples[:mu]
        # 4. 업데이트 (평균 및 공분산)
        m_old = m.copy()
        m_new = np.dot(weights, top_samples)
        y_diff = (top_samples - m_old) / sigma
        C_new_candidates = np.dot(y_diff.T * weights, y_diff)
        C = (1 - c_cov) * C + c_cov * C_new_candidates
        m = m_new
        sigma *= 0.95 # 수렴 속도 조절
        # 시각화 (5회마다)
        if g % 10 == 0 or g == max_iter - 1:
            plot_cov_ellipse(m, (sigma**2)*C, plt.gca(), color='blue' if g < max_iter-1 else 'red')
            plt.scatter(m[0], m[1], c='blue', s=10, alpha=0.5)
    plt.title("Optimization targeting exactly (2, 3)")
    plt.legend()
    plt.grid(True)
    plt.axis('equal')
    plt.show()

    # ==========================================
    # 결과 출력
    # ==========================================
    final_score = objective_function(m)
    
    print("\n" + "="*50)
    print(" >>> 최적화 최종 결과 <<<")
    print("="*50)
    print(f"최종 찾은 위치 (x, y) : {m[0]:.5f}, {m[1]:.5f}")
    print(f"실제 정답 위치 (x, y) : 2.00000, 3.00000")
    print(f"오차 거리 (Distance)  : {np.linalg.norm(m - np.array([2,3])):.8f}")
    print(f"최종 목적함수 값      : {final_score:.8f}")
    print("-" * 50)
    print("최종 공분산 행렬 C:")
    print(np.round(C, 3))

run_fixed_target_optimization()

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# ---------------------------------------------------------
# 1. 제약 조건이 포함된 목적함수
# ---------------------------------------------------------
def constrained_objective(v):
    x, y = v
    # (1) 원래 목적함수 (Original Cost)
    # 원점(0,0)으로 가고 싶어 함
    base_cost = x**2 + y**2
    # (2) 제약 조건: x + y >= 2
    # 위반량(Violation) 계산: 2 - (x+y) 가 양수면 위반한 것임
    # max(0, 값)을 써서 위반했을 때만 값을 가짐
    violation = max(0, 2 - (x + y))
    # (3) 벌점 부과 (Penalty)
    # 위반량의 제곱에 큰 가중치(R)를 곱해서 더함
    # R이 클수록 제약 조건을 엄격하게 지키려 함
    penalty_weight = 1000.0 
    penalty = penalty_weight * (violation ** 2)
    return base_cost + penalty
# ---------------------------------------------------------
# 2. 공분산 타원 시각화 (이전과 동일)
# ---------------------------------------------------------
def plot_cov_ellipse(pos, cov, ax, n_std=2.0, color='red'):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(vals)
    ellip = Ellipse(xy=pos, width=width, height=height, angle=theta,
                    edgecolor=color, fc='None', lw=2, linestyle='--')
    ax.add_patch(ellip)
# ---------------------------------------------------------
# 3. 최적화 실행
# ---------------------------------------------------------
def run_constrained_optimization(max_iter=50):
    # 초기화 (제약 조건을 위반한 먼 곳에서 시작)
    m = np.random.uniform(4, 6, 2) 
    C = np.eye(2)
    sigma = 1.0
    # 하이퍼파라미터
    lambda_pop = 20
    mu = lambda_pop // 2
    weights = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1))
    weights /= np.sum(weights)
    c_cov = 0.5 
    print(f"Start Position: {m}")
    # 시각화 설정
    plt.figure(figsize=(8, 8))
    # 배경: 등고선 그리기 (원래 함수 x^2 + y^2)
    x = np.linspace(-1, 6, 100)
    y = np.linspace(-1, 6, 100)
    X, Y = np.meshgrid(x, y)
    Z = X**2 + Y**2 # 벌점 제외한 원래 모양
    plt.contour(X, Y, Z, levels=30, cmap='gray', alpha=0.2)
    # [시각화 핵심] 제약 조건 라인 그리기 (x + y = 2)
    # y = 2 - x
    line_x = np.linspace(-1, 6, 100)
    line_y = 2 - line_x
    plt.plot(line_x, line_y, 'k-', linewidth=3, label='Constraint wall (x+y=2)')
    # 금지 구역(Forbidden Region) 색칠 (x + y < 2인 영역)
    plt.fill_between(line_x, -2, line_y, color='red', alpha=0.1, label='Forbidden zone')
    # 예상 정답 (1, 1)
    plt.plot(1, 1, 'g*', markersize=15, label='Optimal solution (1, 1)', zorder=10)
    plt.plot(0, 0, 'kx', markersize=10, label='Unconstrained opt (0, 0)', alpha=0.5)
    # 최적화 루프
    for g in range(max_iter):
        # 1. 샘플링
        try:
            samples = np.random.multivariate_normal(m, (sigma**2) * C, lambda_pop)
        except np.linalg.LinAlgError:
            break
        # 2. 평가 (벌점이 포함된 함수로 평가!)
        fitness = np.array([constrained_objective(s) for s in samples])
        # 3. 선택
        sorted_idx = np.argsort(fitness)
        samples = samples[sorted_idx]
        top_samples = samples[:mu]
        # 4. 업데이트
        m_old = m.copy()
        m_new = np.dot(weights, top_samples)
        y_diff = (top_samples - m_old) / sigma
        C_new_candidates = np.dot(y_diff.T * weights, y_diff)
        C = (1 - c_cov) * C + c_cov * C_new_candidates
        m = m_new
        sigma *= 0.95 
        # 시각화 (경로 그리기)
        if g % 5 == 0:
            plot_cov_ellipse(m, (sigma**2)*C, plt.gca(), color='blue')
            plt.scatter(m[0], m[1], c='blue', s=20)
    plt.title("Constrained optimization using penalty method")
    plt.legend()
    plt.grid(True)
    plt.xlim(-1, 6)
    plt.ylim(-1, 6)
    plt.gca().set_aspect('equal')
    plt.savefig('copt.png')
    plt.show()
    # 결과 출력
    final_score = constrained_objective(m)
    violation = max(0, 2 - (m[0] + m[1]))
    print("\n" + "="*50)
    print(" >>> 제약 조건 최적화 결과 <<<")
    print("="*50)
    print(f"최종 위치 (x, y)      : {m[0]:.5f}, {m[1]:.5f}")
    print(f"제약 조건 (x+y >= 2)  : {m[0]+m[1]:.5f} (2.0 이상이어야 함)")
    print(f"위반량 (Violation)    : {violation:.8f}")
    print(f"최종 점수 (Penalty포함): {final_score:.5f}")
    print("-" * 50)
    if abs(m[0] - 1.0) < 0.1 and abs(m[1] - 1.0) < 0.1:
        print("성공: (0,0)으로 가려다가 벽에 막혀 (1,1)에 멈췄다.")
    else:
        print("실패: 수렴하지 못했다.")
run_constrained_optimization()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. 제약 조건이 포함된 목적함수 (이전과 동일)
# ---------------------------------------------------------
def constrained_objective(v):
    x, y = v
    # (1) 원래 목표: 원점(0,0)으로 가라
    base_cost = x**2 + y**2
    # (2) 제약 조건: x + y >= 2
    # 위반 시 위반한 만큼의 제곱으로 페널티 부여
    violation = max(0, 2 - (x + y))
    # (3) 벌점 부여 (Penalty)
    # R값이 클수록 벽이 '단단'해진다.
    penalty_weight = 1000.0 
    penalty = penalty_weight * (violation ** 2)
    return base_cost + penalty
# ---------------------------------------------------------
# 2. 입자 클래스 (PSO Logic)
# ---------------------------------------------------------
class Particle:
    def __init__(self, start_bounds):
        # 초기화: 제약 조건을 위반하는 영역(예: -2~0)과 
        # 만족하는 영역을 섞어서 시작해본다.
        self.position = np.random.uniform(start_bounds[0], start_bounds[1], 2)
        self.velocity = np.random.uniform(-0.5, 0.5, 2)
        # P-best 초기화
        self.pbest_pos = self.position.copy()
        self.pbest_val = constrained_objective(self.position)
        self.current_val = self.pbest_val

    def update_velocity(self, gbest_pos, w, c1, c2):
        r1, r2 = np.random.rand(2), np.random.rand(2)
        # PSO 속도 공식
        inertia = w * self.velocity
        cognitive = c1 * r1 * (self.pbest_pos - self.position)
        social = c2 * r2 * (gbest_pos - self.position)
        self.velocity = inertia + cognitive + social

    def update_position(self):
        self.position += self.velocity
        # (옵션) 화면 밖으로 너무 멀리 나가지 않게 클램핑
        self.position = np.clip(self.position, -2, 8)

    def evaluate(self):
        self.current_val = constrained_objective(self.position)
        # 내 최고 기록 갱신?
        if self.current_val < self.pbest_val:
            self.pbest_val = self.current_val
            self.pbest_pos = self.position.copy()
# ---------------------------------------------------------
# 3. PSO 메인 실행
# ---------------------------------------------------------
def run_constrained_pso(max_iter=100):
    num_particles = 30
    # 시작 위치를 일부러 좀 넓게 잡음 (-2 ~ 6)
    swarm = [Particle([-2, 6]) for _ in range(num_particles)]
    # G-best 초기화
    gbest_pos = swarm[0].position.copy()
    gbest_val = swarm[0].pbest_val
    # 하이퍼파라미터
    w = 0.7   # 관성
    c1 = 1.5  # 인지
    c2 = 1.5  # 사회
    # 시각화 준비
    plt.figure(figsize=(8, 8))
    # 배경: 등고선 & 제약조건 라인
    x = np.linspace(-2, 7, 100)
    y = np.linspace(-2, 7, 100)
    X, Y = np.meshgrid(x, y)
    # 벌점 없는 원래 함수 등고선
    Z_base = X**2 + Y**2
    plt.contour(X, Y, Z_base, levels=30, cmap='gray', alpha=0.2)
    # 제약 조건 라인 (x + y = 2)
    line_x = np.linspace(-2, 7, 100)
    line_y = 2 - line_x
    plt.plot(line_x, line_y, 'k-', linewidth=3, label='Constraint wall (x+y=2)')
    # 금지 구역 (Forbidden Zone)
    plt.fill_between(line_x, -5, line_y, color='red', alpha=0.1, label='Forbidden zone')
    # 정답 표시
    plt.plot(1, 1, 'g*', markersize=20, label='Target (1, 1)', zorder=10)
    plt.plot(0, 0, 'kx', markersize=10, label='Unconstrained opt (0, 0)', alpha=0.5)
    print("PSO optimization started...")
    # PSO 루프
    for i in range(max_iter):
        for p in swarm:
            p.evaluate()
            # G-best 갱신
            if p.current_val < gbest_val:
                gbest_val = p.current_val
                gbest_pos = p.position.copy()
        for p in swarm:
            p.update_velocity(gbest_pos, w, c1, c2)
            p.update_position()
        # 중간 과정 시각화 (10번마다 입자 위치 찍기)
        if i % 10 == 0 or i == max_iter - 1:
            # 입자들의 x, y 좌표 추출
            px = [p.position[0] for p in swarm]
            py = [p.position[1] for p in swarm]
            # 색상을 다르게 하여 이동 과정 표현 (초반: 파랑 -> 후반: 빨강)
            color = plt.cm.jet(i / max_iter)
            alpha = 0.3 if i < max_iter - 1 else 1.0
            label = f'Iter {i}' if i == 0 or i == max_iter - 1 else None
            plt.scatter(px, py, color=color, alpha=alpha, label=label, s=30)
            if i % 10 == 0:
                print(f"Iter {i:2d}: G-Best cost={gbest_val:.5f} at {np.round(gbest_pos, 3)}")
    plt.title("PSO with constraints (Particles hitting the wall)")
    plt.legend()
    plt.xlim(-2, 7)
    plt.ylim(-2, 7)
    plt.gca().set_aspect('equal')
    plt.grid(True)
    plt.savefig('c_pso.png')
    plt.show()
    # 결과 출력
    print("\n" + "="*50)
    print(" >>> PSO 제약 조건 최적화 결과 <<<")
    print("="*50)
    print(f"최종 G-Best 위치 : {gbest_pos[0]:.5f}, {gbest_pos[1]:.5f}")
    print(f"제약 조건 합(x+y): {sum(gbest_pos):.5f} (2.0 이상이어야 함)")
    print(f"최종 점수        : {gbest_val:.8f}")
    dist_error = np.linalg.norm(gbest_pos - np.array([1.0, 1.0]))
    if dist_error < 0.1:
        print("성공: 입자(새 떼)들이 제약 조건 벽에 붙어 (1,1)을 찾았다.")
    else:
        print("실패: 수렴하지 못했다.")
run_constrained_pso()

import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. 목적함수 (Objective Function) - 이전과 동일
# ---------------------------------------------------------
def objective_function(pos):
    x, y = pos
    # 정답: (2, 3)일 때 0
    return (x - 2)**2 + 10 * (y - 3)**2

# ---------------------------------------------------------
# 2. 입자 클래스 (Particle Class)
# ---------------------------------------------------------
class Particle:
    def __init__(self, bounds):
        # 위치와 속도를 무작위 초기화
        self.position = np.random.uniform(bounds[0], bounds[1], 2)
        self.velocity = np.random.uniform(-1, 1, 2)
        # 개인 최고 기록 (P-best)
        self.pbest_position = self.position.copy()
        self.pbest_value = objective_function(self.position)
        # 현재 값
        self.current_value = self.pbest_value

    def update_velocity(self, gbest_position, w, c1, c2):
        """속도 업데이트 (핵심 알고리즘)"""
        r1 = np.random.rand(2) # 랜덤 변수 1
        r2 = np.random.rand(2) # 랜덤 변수 2
        # 1. 관성 (Inertia): 하던 대로 가려는 성질
        inertia = w * self.velocity
        # 2. 인지 (Cognitive): 나의 성공 경험으로 가려는 성질
        cognitive = c1 * r1 * (self.pbest_position - self.position)
        # 3. 사회 (Social): 리더(전역 최적)를 따라가려는 성질
        social = c2 * r2 * (gbest_position - self.position)
        self.velocity = inertia + cognitive + social

    def update_position(self, bounds):
        """위치 업데이트"""
        self.position += self.velocity
        # (선택사항) 탐색 범위를 벗어나지 않게 제한
        # self.position = np.clip(self.position, bounds[0], bounds[1])

    def evaluate(self):
        """적합도 평가 및 P-best 업데이트"""
        self.current_value = objective_function(self.position)
        # 내가 경험한 최고의 위치 갱신
        if self.current_value < self.pbest_value:
            self.pbest_value = self.current_value
            self.pbest_position = self.position.copy()

# ---------------------------------------------------------
# 3. PSO 알고리즘 메인
# ---------------------------------------------------------
def run_pso(num_particles=30, max_iter=50):
    # 탐색 범위 (-10 ~ 10)
    bounds = [-10, 10]
    # 하이퍼파라미터
    w = 0.7    # 관성 가중치 (0.5~0.9 추천)
    c1 = 1.5   # 개인 가중치
    c2 = 1.5   # 사회 가중치
    # 입자 군집 생성
    swarm = [Particle(bounds) for _ in range(num_particles)]
    # 전역 최고 기록 (G-best) 초기화
    gbest_position = swarm[0].position.copy()
    gbest_value = swarm[0].pbest_value
    history_gbest = [] # 그래프용 기록
    print(f"PSO 시작 (입자 수: {num_particles})")
    print(f"목표: x=2, y=3 찾기\n")
    for i in range(max_iter):
        # 모든 입자에 대해 루프
        for particle in swarm:
            # 1. 평가 및 P-best 갱신
            particle.evaluate()
            # 2. G-best 갱신 (팀 전체 최고 기록 확인)
            if particle.current_value < gbest_value:
                gbest_value = particle.current_value
                gbest_position = particle.position.copy()
        # 3. 이동 (속도 및 위치 업데이트) - G-best 정보를 바탕으로
        for particle in swarm:
            particle.update_velocity(gbest_position, w, c1, c2)
            particle.update_position(bounds)
        history_gbest.append(gbest_value)
        if i % 10 == 0:
            print(f"Iter {i:3d} | G-Best: {gbest_value:.8f} at {gbest_position}")
    print("-" * 50)
    print(f"최종 결과: x={gbest_position[0]:.4f}, y={gbest_position[1]:.4f}")
    print(f"실제 정답: x=2.0000, y=3.0000")
    return swarm, history_gbest, gbest_position

# ==========================================
# 실행 및 시각화
# ==========================================
final_swarm, history, final_gbest = run_pso()
# 결과 그래프
plt.figure(figsize=(12, 5))
# 1. 수렴 그래프
plt.subplot(1, 2, 1)
plt.plot(history, 'b-')
plt.title("Convergence (Fitness value)")
plt.xlabel("Iteration")
plt.ylabel("Objective function value")
plt.yscale('log') # 로그 스케일로 보면 수렴이 잘 보임
plt.grid(True)
# 2. 입자 위치 시각화 (최종 상태)
plt.subplot(1, 2, 2)
# 등고선 그리기
x_range = np.linspace(-5, 10, 100)
y_range = np.linspace(-5, 10, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = (X - 2)**2 + 10 * (Y - 3)**2
plt.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.5)
# 입자들 찍기
for p in final_swarm:
    plt.scatter(p.position[0], p.position[1], c='blue', alpha=0.6, s=30)
# 최종 정답(G-best) 찍기
plt.scatter(final_gbest[0], final_gbest[1], c='red', marker='*', s=200, label='Global best')
plt.scatter(2, 3, c='green', marker='x', s=100, label='True solution')
plt.title("Final particle positions")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import cma
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. 두 개의 상충하는 목적함수 정의
# ---------------------------------------------------------
def f1(x):
    """목표 1: 원점(0,0)과의 거리 제곱 (최소화)"""
    return x[0]**2 + x[1]**2

def f2(x):
    """목표 2: 점(2,2)와의 거리 제곱 (최소화)"""
    return (x[0] - 2)**2 + (x[1] - 2)**2

# ---------------------------------------------------------
# 2. 가중치 합 목적함수 (Wrapper)
# ---------------------------------------------------------
def weighted_objective(x, weight):
    """
    weight가 1에 가까우면 f1을 중시
    weight가 0에 가까우면 f2를 중시
    """
    return weight * f1(x) + (1 - weight) * f2(x)

# ---------------------------------------------------------
# 3. 메인 실행 루프 (가중치를 바꿔가며 반복 실행)
# ---------------------------------------------------------
def run_multiobjective_cma():
    # 결과를 저장할 리스트
    pareto_f1 = []
    pareto_f2 = []
    solutions = []
    # 가중치를 0.0에서 1.0까지 20단계로 쪼개서 실행
    weights = np.linspace(0, 1, 21)
    print("다목적 최적화 시작 (Scanning Pareto front)...")
    for w in weights:
        # CMA-ES 실행 (cma.fmin 활용)
        # args=(w,)를 통해 weighted_objective 함수에 가중치 전달
        # 시작점은 무난하게 [1, 1]에서 시작
        res = cma.fmin(weighted_objective, [1.0, 1.0], 0.5, 
                       args=(w,), options={'verbose': -9}) # 로그 끔
        best_x = res[0]
        # 찾은 해(best_x)의 f1값과 f2값을 따로 계산해서 저장
        val_f1 = f1(best_x)
        val_f2 = f2(best_x)
        pareto_f1.append(val_f1)
        pareto_f2.append(val_f2)
        solutions.append(best_x)
        print(f"Weight {w:.2f} -> Solution: {np.round(best_x, 2)}")
    # ---------------------------------------------------------
    # 4. 시각화
    # ---------------------------------------------------------
    plt.figure(figsize=(12, 5))
    # (1) 결정 공간 (Decision Space): 실제 x, y 좌표
    plt.subplot(1, 2, 1)
    s_x = [s[0] for s in solutions]
    s_y = [s[1] for s in solutions]
    plt.plot(s_x, s_y, 'bo-', label='Pareto set')
    plt.plot(0, 0, 'rx', markersize=10, label='Target 1 (0,0)')
    plt.plot(2, 2, 'gx', markersize=10, label='Target 2 (2,2)')
    plt.title("Decision space (x vs y)")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.legend()
    plt.grid(True)
    # (2) 목적함수 공간 (Objective Space): f1 vs f2 (파레토 프론트)
    plt.subplot(1, 2, 2)
    plt.plot(pareto_f1, pareto_f2, 'ro-', linewidth=2)
    plt.title("Objective space (Pareto front)")
    plt.xlabel("f1 Value (Minimize)")
    plt.ylabel("f2 Value (Minimize)")
    plt.grid(True)

    plt.tight_layout()
    plt.show()

run_multiobjective_cma()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. 가상의 데이터 생성 (두 가지 목표를 가진 500개의 해)
# ---------------------------------------------------------
np.random.seed(42)
n_samples = 500

# 목표 1 (f1): 0.1 ~ 1.0 사이의 값 (작을수록 좋음)
f1 = np.random.uniform(0.1, 1.0, n_samples)

# 목표 2 (f2): f1과 반비례 관계를 가지도록 설정 (상충 관계 구현)
# 이론적인 완벽한 선(f2 = 0.1 / f1) 위에 노이즈를 더해 실제 데이터처럼 만듦
# 노이즈는 항상 양수로 더해서 이론적 한계선보다 위쪽에 위치하게 함
noise = np.random.exponential(0.15, n_samples)
f2 = (0.1 / f1) + noise

# (N, 2) 형태의 행렬로 합침
points = np.column_stack((f1, f2))

# ---------------------------------------------------------
# 2. 파레토 필터링 함수 (지배당하지 않은 해 찾기)
# ---------------------------------------------------------
def get_pareto_mask_2d_minimize(costs):
    """
    2차원 최소화 문제에서 파레토 최적해의 마스크(True/False)를 반환
    효율적인 알고리즘 사용 (O(N log N))
    """
    n_points = costs.shape[0]
    is_pareto = np.ones(n_points, dtype=bool) # 일단 모두 True로 시작

    # 1. 첫 번째 목표(f1)를 기준으로 오름차순 정렬
    # 정렬된 인덱스를 가져옴
    sorted_indices = np.argsort(costs[:, 0])
    
    # 2. 정렬된 순서대로 탐색하며 f2의 최소값을 갱신
    current_min_f2 = float('inf') # 무한대로 초기화

    for i in range(n_points):
        original_idx = sorted_indices[i]
        current_f2 = costs[original_idx, 1]

        # f1은 이미 정렬되어 점점 커지고 있음.
        # 만약 현재 점의 f2가 지금까지 본 최소 f2보다 작다면?
        # -> 이 점은 f1은 좀 커도 f2가 압도적으로 좋으므로 '파레토 최적해'임.
        if current_f2 < current_min_f2:
            current_min_f2 = current_f2 # 최소 f2 기록 갱신
            # is_pareto[original_idx]는 True로 유지
        else:
            # 만약 현재 점의 f2가 기존 최소값보다 크거나 같다면?
            # -> 이 점보다 f1도 작고 f2도 작은 점이 이미 앞서 존재했다는 뜻.
            # -> 즉, 지배당함 (탈락)
            is_pareto[original_idx] = False
            
    return is_pareto

# ---------------------------------------------------------
# 3. 필터링 적용 및 데이터 분리
# ---------------------------------------------------------
mask = get_pareto_mask_2d_minimize(points)

pareto_points = points[mask]      # 최적해 (프론트)
dominated_points = points[~mask]  # 지배당한 해 (열등한 해)

# 선을 예쁘게 잇기 위해 파레토 점들을 f1 기준으로 정렬
pareto_points = pareto_points[np.argsort(pareto_points[:, 0])]

# ---------------------------------------------------------
# 4. 시각화 (핵심 부분)
# ---------------------------------------------------------
plt.figure(figsize=(10, 7))

# (1) 열등한 해들 (회색 점)
plt.scatter(dominated_points[:, 0], dominated_points[:, 1], 
            c='gray', alpha=0.4, s=20, label='Dominated solutions (Inferior)')

# (2) 파레토 프론트 (빨간 점과 선)
plt.plot(pareto_points[:, 0], pareto_points[:, 1], 'r-', linewidth=2, alpha=0.8)
plt.scatter(pareto_points[:, 0], pareto_points[:, 1], 
            c='red', s=60, zorder=5, label='Pareto front (Non-dominated)')

# 그래프 꾸미기
plt.title("Visualizing the Pareto front (Minimization problem)")
plt.xlabel("Objective 1 (Cost)  Minimize", fontsize=18)
plt.ylabel("Objective 2 (Time)  Minimize", fontsize=18)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# 설명 화살표 추가
plt.arrow(0.6, 0.8, -0.1, -0.1, head_width=0.03, head_length=0.05, fc='k', ec='k')
plt.text(0.62, 0.82, "Better direction\n(Lower cost & Lower time)", fontsize=11)

# 한계선 영역 표시 (불가능한 영역)
fill_x = np.linspace(0.1, 1.0, 100)
fill_y = 0.1 / fill_x
plt.fill_between(fill_x, 0, fill_y, color='blue', alpha=0.1)
plt.text(0.2, 0.1, "Physically impossible region", color='blue', fontsize=12, fontweight='bold')

plt.xlim(0, 1.1)
plt.ylim(0, 1.2)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def generate_car_designs(num_samples=1000):
    """
    무작위 자동차 설계 생성
    입력 변수:
      - engine_size (cc): 1.0 ~ 5.0 리터
      - aero_cost ($): 공기역학 투자 비용 (10 ~ 100)
    """
    np.random.seed(42)
    engine_size = np.random.uniform(1.0, 5.0, num_samples)
    aero_cost = np.random.uniform(10, 100, num_samples)
    # 목적함수 1: 최고 속도 (최대화 목표)
    # 엔진이 클수록(+), 공기역학 비용을 많이 쓸수록(+) 빨라짐
    # (약간의 랜덤 노이즈 추가)
    speed = (engine_size * 40) + (aero_cost * 0.5) + np.random.normal(0, 5, num_samples)
    # 목적함수 2: 연비 (최대화 목표)
    # 엔진이 클수록(-) 나빠짐. 공기역학 비용을 쓰면(+) 조금 좋아짐
    efficiency = (20 / engine_size) + (aero_cost * 0.05) + np.random.normal(0, 1, num_samples)
    return speed, efficiency

def find_pareto_frontier(costs_1, costs_2):
    """
    파레토 프론트(비지배 해 집합)를 찾는 함수
    두 목적함수 모두 '최대화(Maximize)'가 목표라고 가정
    """
    population_size = len(costs_1)
    # 파레토 여부를 저장할 마스크 (True면 파레토 최적해)
    is_pareto = np.ones(population_size, dtype=bool)
    for i in range(population_size):
        for j in range(population_size):
            if i == j: continue
            # [핵심 로직] j가 i를 지배하는가?
            # 조건: j가 i보다 두 지표 모두 크거나 같으면서, 적어도 하나는 더 커야 함
            if (costs_1[j] >= costs_1[i] and costs_2[j] >= costs_2[i]) and \
               (costs_1[j] > costs_1[i] or costs_2[j] > costs_2[i]):
                # j가 더 우월하므로 i는 탈락(지배당함)
                is_pareto[i] = False
                break
    return is_pareto
# ---------------------------------------------------------
# 실행 및 시각화
# ---------------------------------------------------------
# 1. 1000개의 무작위 설계안 생성
speeds, efficiencies = generate_car_designs(1000)
# 2. 파레토 프론트 찾기
pareto_mask = find_pareto_frontier(speeds, efficiencies)
# 파레토 해만 추출
pareto_speeds = speeds[pareto_mask]
pareto_effs = efficiencies[pareto_mask]
# 시각화를 위해 순서대로 정렬 (선을 잇기 위함)
sorted_indices = np.argsort(pareto_speeds)
pareto_speeds = pareto_speeds[sorted_indices]
pareto_effs = pareto_effs[sorted_indices]
# 3. 그래프 그리기
plt.figure(figsize=(10, 6))
# (1) 모든 설계안 (회색 점)
plt.scatter(speeds, efficiencies, c='gray', alpha=0.3, label='All designs (Dominated)')
# (2) 파레토 프론트 (빨간 점 & 선)
plt.plot(pareto_speeds, pareto_effs, 'r-', linewidth=2, alpha=0.8) # 선 연결
plt.scatter(pareto_speeds, pareto_effs, c='red', s=50, label='Pareto front (Non-dominated)')
plt.title("Multi-objective optimization: Speed vs Efficiency")
plt.xlabel("Max speed (km/h), Maximize", fontsize=18)
plt.ylabel("Fuel efficiency (km/l), Maximize", fontsize=18)
plt.legend()
plt.grid(True)
# 화살표로 설명 추가
plt.arrow(100, 10, 20, 2, head_width=1, head_length=3, fc='k', ec='k')
plt.text(105, 13, "Better direction", fontsize=12, fontweight='bold')
plt.savefig('pareto_front.png')
plt.show()
print(f"전체 설계안 수: {len(speeds)}")
print(f"파레토 최적해 수: {len(pareto_speeds)}")
print("\n[추천 설계안 예시]")
print(f"1. 속도 중시형 : 속도 {pareto_speeds[-1]:.1f} km/h, 연비 {pareto_effs[-1]:.1f} km/l")
print(f"2. 연비 중시형 : 속도 {pareto_speeds[0]:.1f} km/h, 연비 {pareto_effs[0]:.1f} km/l")
print(f"3. 밸런스형    : 속도 {pareto_speeds[len(pareto_speeds)//2]:.1f} km/h, 연비 {pareto_effs[len(pareto_speeds)//2]:.1f} km/l")

In [ ]:
import numpy as np

# 1. 예시 데이터 생성 (샘플 수: 5, 변수: 2개 - 키, 몸무게)
# 상관관계를 명확히 하기 위해 데이터를 조금 더 늘렸다.
# [키, 몸무게]
data = np.array([
    [170.1, 60.5],
    [180.2, 80.1],
    [160.5, 50.3],
    [175.0, 70.2],
    [165.3, 55.4]
])

print("--- 1. 공분산 행렬 계산 ---")
# rowvar=False: 각 열(Column)이 변수임을 명시
cov_matrix = np.cov(data, rowvar=False)

print("Covariance Matrix (C):")
print(cov_matrix)
print(f"Shape: {cov_matrix.shape}")
print("-" * 30)

print("--- 2. 숄레스키 분해 수행 ---")
# 조건: 공분산 행렬은 대칭이고, 데이터가 충분하면 양의 정부호(Positive Definite)이다.
try:
    # L * L.T = C 가 되는 하삼각 행렬 L을 구함
    L = np.linalg.cholesky(cov_matrix)
    
    print("Cholesky Factor (L) - Lower Triangular:")
    print(L)
    print("-" * 30)
    
    print("--- 3. 검증 (Reconstruction) ---")
    # L과 L의 전치행렬(L.T)을 곱해서 원래 공분산 행렬이 나오는지 확인
    # np.dot(L, L.T)
    reconstructed_cov = np.dot(L, L.T)
    
    print("L * L.T (복원된 행렬):")
    print(reconstructed_cov)
    
    # 원래 행렬과 비교 (거의 0에 가까워야 함)
    diff = np.abs(cov_matrix - reconstructed_cov).max()
    print(f"\n최대 오차(Original - Reconstructed): {diff:.20f}")
    
    if diff < 1e-10:
        print(">> 검증 성공: 완벽하게 복원되었다.")
    else:
        print(">> 검증 실패")

except np.linalg.LinAlgError:
    print("오류: 행렬이 양의 정부호(Positive Definite)가 아니다.")
    print("데이터 샘플 수가 변수 개수보다 적거나, 변수 간 완전한 선형 관계가 있을 수 있다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. 문제 정의: 100차원 비밀번호 맞추기
# ---------------------------------------------------------
N_DIM = 100

# 정답지 (랜덤한 0과 1의 조합)
np.random.seed(42)
TARGET_BINARY = np.random.randint(0, 2, N_DIM)

def objective_function_binary(z_continuous):
    """
    입력: 실수 벡터 (Continuous)
    출력: 정답과 일치하는 비트 수 (높을수록 좋음)
    """
    # [핵심] 실수 -> 이진수 변환 (Thresholding)
    # 0보다 크면 1, 아니면 0
    binary_vector = (z_continuous >= 0).astype(int)
    # 정답과 비교 (일치하는 개수)
    # TARGET_BINARY와 같으면 1점씩
    score = np.sum(binary_vector == TARGET_BINARY)
    return score

# ---------------------------------------------------------
# 2. 촐레스키 기반 최적화 (Binary용 변형)
# ---------------------------------------------------------
def run_binary_cma_es(max_iter=500):
    # 초기화
    # 평균 위치는 0 근처에서 시작 (확률 50:50)
    m = np.zeros(N_DIM) 
    # 공분산 행렬 (100x100 차원)
    # 초기에는 상관관계 없이 독립적으로 탐색
    C = np.eye(N_DIM) 
    # 탐색 범위 (Step Size)
    sigma = 2.0 
    # 하이퍼파라미터
    lambda_pop = 50         # 한 세대당 샘플 수
    mu = lambda_pop // 4    # 상위 25%만 생존
    # 가중치 (로그 스케일)
    weights = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1))
    weights /= np.sum(weights)
    c_cov = 0.1 # 학습률 (차원이 높으므로 좀 낮게 설정)
    print(f"목표: 100개의 비트 패턴을 찾아라!")
    print(f"Target (First 10): {TARGET_BINARY[:10]} ...\n")
    history_best = []
    for g in range(max_iter):
        # -----------------------------------------------------
        # 1. 샘플링 (실수 공간에서 수행)
        # -----------------------------------------------------
        # N=100이므로 공분산 행렬 연산이 조금 무거울 수 있음
        try:
            # 촐레스키 분해 대신, 차원이 높을 때는 svd나 eigh가 안정적일 수 있으나
            # 여기서는 편의상 다변량 정규분포 함수 사용
            samples = np.random.multivariate_normal(m, (sigma**2) * C, lambda_pop)
        except np.linalg.LinAlgError:
            print("행렬 분해 실패 (Numerical Instability)")
            break
        # 2. 평가 (변환 후 평가)
        # 함수 내부에서 실수 -> 이진수 변환이 일어남
        fitness = np.array([objective_function_binary(s) for s in samples])
        # 3. 선택 (최대화 문제이므로 내림차순 정렬)
        sorted_idx = np.argsort(fitness)[::-1] # Descending
        samples = samples[sorted_idx]
        top_samples = samples[:mu]
        best_score = fitness[sorted_idx[0]]
        history_best.append(best_score)
        # 4. 업데이트 (실수 공간의 m과 C를 업데이트)
        # "점수가 좋았던 '실수 좌표'들의 경향성을 학습함"
        m_old = m.copy()
        m_new = np.dot(weights, top_samples)
        # 공분산 업데이트
        y_diff = (top_samples - m_old) / sigma
        # Rank-mu Update
        # (차원이 커서 연산량이 많음. 실제로는 대각 행렬만 쓰는 방식을 쓰기도 함)
        C_new_candidates = np.dot(y_diff.T * weights, y_diff)
        # 공분산 행렬 혼합
        C = (1 - c_cov) * C + c_cov * C_new_candidates
        # 학습 안정화를 위해 대각 성분(분산)에 약간의 노이즈 추가 (정규화)
        # 100차원이라 행렬이 망가지기 쉬워서 넣는 안전장치
        C += np.eye(N_DIM) * 1e-5
        m = m_new
        # 이진 문제는 수렴할수록 sigma를 너무 빨리 줄이면 안됨 (탐색 유지)
        sigma *= 0.98 
        if g % 10 == 0:
            print(f"Iter {g:3d}: Best Score = {best_score}/100")
        if best_score == N_DIM:
            print(f"\n>>> 정답 발견! (Iter {g})")
            break
    # 결과 출력
    final_binary = (m >= 0).astype(int)
    print("\n" + "="*50)
    print(f"최종 예측 (First 10): {final_binary[:10]} ...")
    print(f"실제 정답 (First 10): {TARGET_BINARY[:10]} ...")
    print(f"일치 개수: {np.sum(final_binary == TARGET_BINARY)} / {N_DIM}")
    # 그래프
    plt.plot(history_best)
    plt.title("Optimization progress (100-dim binary)")
    plt.xlabel("Iteration")
    plt.ylabel("Matches (Max 100)")
    plt.grid(True)
    plt.show()
run_binary_cma_es()

import cma
import numpy as np

# 1. 목적함수 정의 (정답: 2, 3)
def objective_function(x):
    return (x[0] - 2)**2 + 10 * (x[1] - 3)**2

# 2. 최적화 실행 (단 한 줄!)
# cma.fmin(함수, 시작위치, 초기표준편차)
res = cma.fmin(objective_function, [0, 0], 0.5)

# 3. 결과 출력
print("\n-------------------------")
print(f"최적해 (Best Solution): {res[0]}")
print(f"최적값 (Best Value):    {res[1]}")

import cma
import numpy as np
import matplotlib.pyplot as plt

def objective_function(x):
    # 타원형 함수 (정답: 2, 3)
    return (x[0] - 2)**2 + 10 * (x[1] - 3)**2

def run_pycma_demo():
    # 1. 설정 (Options)
    # bounds: 탐색 범위 제한 (예: -5 ~ 5 사이)
    # verbose: 로그 출력 레벨
    opts = {
        'bounds': [[-5, -5], [5, 5]], 
        'popsize': 10,
        'verbose': -1  # 출력 최소화
    }
    # 2. ES 객체 생성 (시작점 [0,0], sigma=0.5)
    es = cma.CMAEvolutionStrategy([0, 0], 0.5, opts)
    print("최적화 시작...")
    # 3. 최적화 루프
    while not es.stop():
        # (1) 샘플링 (ask)
        solutions = es.ask()   
        # (2) 평가 (evaluate)
        # 여기서 제약 조건 위반 시 벌점(Penalty)을 추가할 수도 있음
        fitness = [objective_function(s) for s in solutions]
        # (3) 업데이트 (tell)
        es.tell(solutions, fitness)
        # (4) 로그 기록 (시각화용)
        es.logger.add() 
        # 진행 상황 출력
        es.disp()
    # 4. 결과 출력
    result = es.result
    print("\n>>> 최적화 완료")
    print(f"최종 해: {result.xbest}")
    print(f"최종 값: {result.fbest:.8f}")
    # 5. 자동 시각화 (pycma 내장 기능)
    # 수렴 그래프, 스텝 사이즈 변화, 공분산 행렬 고유값 변화 등을 그려줌
    print("그래프를 그린다...")
    cma.plot() 
    plt.show()

run_pycma_demo()

import cma
import numpy as np

# 1. 제약 조건이 포함된 목적함수 정의
def constrained_function(x):
    # (1) 기본 목적함수 (Base Cost)
    # 목표: (0,0)으로 가라 -> x^2 + y^2
    base_cost = x[0]**2 + x[1]**2
    # (2) 제약 조건 계산
    # 조건: x + y >= 1
    # 위반량(violation) = 1 - (x + y) 가 0보다 크면 위반한 것
    # 예: x=0, y=0 이면 violation = 1 (위반)
    # 예: x=1, y=1 이면 violation = -1 (통과 -> max함수로 0 처리)
    violation = max(0, 1 - (x[0] + x[1]))
    # (3) 벌점(Penalty) 추가
    # 위반량이 있을 때만 엄청난 값을 더해줌
    # 제곱을 해주는 이유: 경계선 근처에서 미분 가능하게 만들어 수렴을 돕기 위해
    if violation > 0:
        penalty = 1e6 * (violation**2) # 가중치(1e6)는 충분히 크게
        return base_cost + penalty
    else:
        return base_cost

# 2. CMA-ES 실행
print("최적화 시작...")
# 초기 위치 [2, 2] (안전한 곳에서 시작), 초기 표준편차 0.5
# cma.fmin(함수, 시작위치, sigma)
res = cma.fmin(constrained_function, [2, 2], 0.5, options={'verbose': -1})
# 3. 결과 확인
best_x = res[0]
best_val = res[1]
print("\n" + "="*40)
print(" >>> 결과 분석 <<<")
print("="*40)
print(f"최종 좌표 (x, y)  : {best_x[0]:.5f}, {best_x[1]:.5f}")
print(f"제약 조건 (x+y)   : {sum(best_x):.5f} (1.0 이상이어야 함)")
print(f"최종 목적함수 값  : {best_val:.5f}")
# 검증
if abs(sum(best_x) - 1.0) < 1e-3:
    print("-> 성공! 벽(x+y=1)에 정확히 붙어서 멈췄다.")
else:
    print("-> 실패 또는 제약 조건 위반.")

In [ ]:
!pip install python-louvain

In [ ]:
!pip install leidenalg

In [ ]:
!pip install python-igraph

In [ ]:
import numpy as np

# 다항식의 값을 계산하는 함수
def polynomial_value(coeff, x):
    result = 0
    for i in range(len(coeff)):
        result += coeff[i] * (x ** (len(coeff) - i - 1))
    return result

# 다항식의 도함수 값을 계산하는 함수 (첫 번째 도함수)
def polynomial_derivative(coeff, x):
    result = 0
    for i in range(len(coeff) - 1):
        result += (len(coeff) - i - 1) * coeff[i] * (x ** (len(coeff) - i - 2))
    return result

# 다항식의 두 번째 도함수 값을 계산하는 함수
def polynomial_second_derivative(coeff, x):
    result = 0
    for i in range(len(coeff) - 2):
        result += (len(coeff) - i - 1) * (len(coeff) - i - 2) * coeff[i] * (x ** (len(coeff) - i - 3))
    return result

# Laguerre's method로 근을 찾는 함수
def laguerre_method(coeff, x0, max_iter=100, tol=1e-6):
    n = len(coeff) - 1  # 다항식 차수
    x = x0  # 초기 추정값

    for _ in range(max_iter):
        # 다항식과 도함수 계산
        fx = polynomial_value(coeff, x)
        f_prime = polynomial_derivative(coeff, x)
        f_double_prime = polynomial_second_derivative(coeff, x)

        # G와 H 계산
        G = f_prime / fx
        H = G**2 - f_double_prime / fx

        # Laguerre's method 공식
        denominator = max(G + np.sqrt((n - 1) * (n * H - G**2)), G - np.sqrt((n - 1) * (n * H - G**2)))
        denominator = denominator if denominator != 0 else 1e-10  # 0으로 나누지 않도록 처리
        delta_x = n / denominator

        # 근 업데이트
        x = x - delta_x

        # 수렴 조건
        if abs(delta_x) < tol:
            return x

    return x

# 예시 다항식: x^3 - 6x^2 + 11x - 6
coeff = [1, -6, 11, -6]

# 초기 추정값 설정 (임의로 선택)
initial_guess = 3.1

# Laguerre's method 실행
root = laguerre_method(coeff, initial_guess)
print(f"근: {root}")

# 다항식의 값을 계산하여 정확성 체크
print(f"다항식 값: {polynomial_value(coeff, root)}")

In [ ]:
from scipy.optimize import root

# 예시 다항식: x^3 - 6x^2 + 11x - 6
def poly(x):
    return x**3 - 6*x**2 + 11*x - 6

# 여러 초기 추정값을 사용하여 근을 찾음
initial_guesses = [1, 2, 3]
sol = root(poly, initial_guesses)
print(f"다항식의 근: {sol.x}")

In [ ]:
import numpy as np

# 1. 5차 방정식의 계수를 정의한다.
# 계수는 높은 차수부터 낮은 차수 순서로 배열에 넣는다.
# P(x) = a*x^5 + b*x^4 + c*x^3 + d*x^2 + e*x + f
# [a, b, c, d, e, f]
# 예시: x^5 - 2x^4 - 6x^3 + 8x^2 + 5x - 4 = 0
coefficients = [1, -2, -6, 8, 5, -4]

# 2. numpy.roots 함수를 사용하여 근을 계산한다.
# 이 함수는 수치 해석적 방법을 사용하여 근의 근사치를 반환한다.
roots = np.roots(coefficients)

# 3. 결과 출력
print("### 5차 방정식의 근사 근 (NumPy 이용) ###")
print(f"방정식 계수: {coefficients}")
print(f"총 근의 개수: {len(roots)}개")
print("\n[계산된 5개의 근]")
for i, root in enumerate(roots):
    print(f"근 {i+1}: {root}")

# 근이 실수인지 허수인지 구분하여 출력 (매우 작은 허수부는 무시)
# print("\n[실수/복소수 구분]")
# for root in roots:
#     if np.isclose(root.imag, 0):
#         print(f"실수 근: {root.real:.6f}")
#     else:
#         print(f"복소수 근: {root}")

In [ ]:
import numpy as np

# 3차 방정식의 계수: [a3, a2, a1, a0]
coefficients = [1, -6, 11, -6 ]

# 수치적 근사치 계산
roots = np.roots(coefficients)

print("### 3차 방정식의 근사 근 (NumPy) ###")
print(f"총 근의 개수: {len(roots)}개")
print("\n[계산된 3개의 근]")
for i, root in enumerate(roots):
    print(f"근 {i+1}: {root}")

In [ ]:
import numpy as np

# 6차 방정식의 계수: [a6, a5, a4, a3, a2, a1, a0]
coefficients = [1, -3, 5, -2, -1, 7, 10]

# 수치적 근사치 계산
roots = np.roots(coefficients)

print("### 6차 방정식의 근사 근 (NumPy) ###")
print(f"총 근의 개수: {len(roots)}개")
print("\n[계산된 6개의 근]")
for i, root in enumerate(roots):
    print(f"근 {i+1}: {root}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 적분 대상 함수: f(x) = 1/x
def f(x):
    return 1/x

# 정답 (참값)
TRUE_VALUE = np.log(2)
A = 1.0  # 적분 하한
B = 2.0  # 적분 상한
MAX_ITER = 5 # 최대 5단계 외삽 (행렬 크기 5x5)

## 1. 사다리꼴 공식 근사값 계산 (R_{i, 0})
def trapezoidal_rule(f, a, b, n):
    """
    n은 구간의 개수 (segments)
    h는 구간 폭 (step size)
    """
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    # 사다리꼴 공식: h/2 * (y[0] + 2*y[1] + ... + 2*y[n-1] + y[n])
    integral = h/2 * (y[0] + 2 * np.sum(y[1:-1]) + y[-1])
    return integral

## 2. 롬베르크 외삽행렬 생성 (R_{i, j})
def romberg_integration(f, a, b, max_iter):
    # R 행렬 초기화 (max_iter x max_iter)
    R = np.zeros((max_iter, max_iter))
    
    # 1열 (j=0) 계산: R_{i, 0} = 사다리꼴 공식 결과
    for i in range(max_iter):
        n = 2**i  # 구간 개수: 1, 2, 4, 8, 16, ...
        R[i, 0] = trapezoidal_rule(f, a, b, n)
        
    # 롬베르크 외삽 계산: R_{i, j}
    # R_{i, j} = (4^j * R_{i, j-1} - R_{i-1, j-1}) / (4^j - 1)
    for j in range(1, max_iter):
        for i in range(j, max_iter):
            power_of_4 = 4**j
            R[i, j] = (power_of_4 * R[i, j-1] - R[i-1, j-1]) / (power_of_4 - 1)
            
    return R

## 3. 롬베르크 외삽행렬 출력
R_matrix = romberg_integration(f, A, B, MAX_ITER)

print(" 롬베르크 외삽행렬 (R_{i, j})")
print("-" * 50)
print(f"함수: f(x) = 1/x, 구간: [{A}, {B}], 참값: {TRUE_VALUE:.10f}\n")

# 외삽 행렬을 표 형태로 보기 쉽게 출력
header = [f"j={j} (Order O(h^{2*(j+1)}))" for j in range(MAX_ITER)]
print("i | " + " | ".join(header))
print("-" * 50)
for i in range(MAX_ITER):
    row_str = f"{i} | "
    row_str += " | ".join([f"{R_matrix[i, j]:.10f}" if R_matrix[i, j] != 0 else "" for j in range(i + 1)])
    print(row_str)

print("-" * 50)

## 4. 적분의 수렴성 그래프 (절대 오차)
def plot_convergence(R_matrix, true_value, max_iter):
    # 절대 오차 계산
    Absolute_Error = np.abs(R_matrix - true_value)
    
    plt.figure(figsize=(10, 6))
    
    # 각 외삽 단계 j에 대한 수렴성 그리기
    for j in range(max_iter):
        # 0이 아닌 값들만 플롯 (각 행의 R[i,j] 값)
        errors = Absolute_Error[j:max_iter, j] 
        # R_matrix[i, j]에 해당하는 구간 개수 n=2^i의 역수 (h^2에 비례)
        h_values = [1/(2**i)**2 for i in range(j, max_iter)] 
        
        if errors.size > 0:
            plt.plot(h_values, errors, marker='o', 
                     linestyle='-', label=f'R($h, 2^{j-1}h, \dots$) - Order $O(h^{2*(j+1)})$')
            
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('$\propto h^2$ (Log Scale)')
    plt.ylabel('Absolute Error $|R_{i, j} - I_{true}|$ (Log Scale)')
    plt.title(' Romberg Integration Convergence Plot (Absolute Error)')
    plt.legend(loc='lower left')
    plt.grid(True, which="both", ls="--")
    plt.gca().invert_xaxis() # h가 작아질수록 (정확도가 높아질수록) 오른쪽으로 이동
    plt.show()

plot_convergence(R_matrix, TRUE_VALUE, MAX_ITER)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 상수 및 함수 정의 ---

# 적분 대상 함수: f(x) = 1/x
def f(x):
    return 1/x

# 정답 (참값)
TRUE_VALUE = np.log(2)
A = 1.0  # 적분 하한
B = 2.0  # 적분 상한
MAX_ITER = 5 # 최대 5단계 외삽 (행렬 크기 5x5)

## 1. 사다리꼴 공식 근사값 계산 (R_{i, 0})
def trapezoidal_rule(f, a, b, n):
    """
    n은 구간의 개수 (segments)
    h는 구간 폭 (step size)
    """
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    # 사다리꼴 공식: h/2 * (y[0] + 2*y[1] + ... + 2*y[n-1] + y[n])
    integral = h/2 * (y[0] + 2 * np.sum(y[1:-1]) + y[-1])
    return integral

## 2. 롬베르크 외삽행렬 생성 (R_{i, j})
def romberg_integration(f, a, b, max_iter):
    # R 행렬 초기화 (max_iter x max_iter)
    R = np.zeros((max_iter, max_iter))
    
    # 1열 (j=0) 계산: R_{i, 0} = 사다리꼴 공식 결과
    for i in range(max_iter):
        n = 2**i  # 구간 개수: 1, 2, 4, 8, 16, ...
        R[i, 0] = trapezoidal_rule(f, a, b, n)
        
    # 롬베르크 외삽 계산: R_{i, j}
    # R_{i, j} = (4^j * R_{i, j-1} - R_{i-1, j-1}) / (4^j - 1)
    for j in range(1, max_iter):
        for i in range(j, max_iter):
            power_of_4 = 4**j
            R[i, j] = (power_of_4 * R[i, j-1] - R[i-1, j-1]) / (power_of_4 - 1)
            
    return R

## 3. 적분의 수렴성 그래프 (절대 오차)
def plot_convergence(R_matrix, true_value, max_iter):
    # 절대 오차 계산
    Absolute_Error = np.abs(R_matrix - true_value)
    
    plt.figure(figsize=(10, 6))
    
    # 각 외삽 단계 j에 대한 수렴성 그리기
    for j in range(max_iter):
        errors = Absolute_Error[j:max_iter, j] 
        # h^2에 비례하는 값: (1/n)^2 = (1/2^i)^2
        h_values = [1/(2**i)**2 for i in range(j, max_iter)] 
        
        if errors.size > 0:
            order = 2 * (j + 1)
            # **수정된 legend 라벨**: Raw String (r'')과 지수 표현 {10} 사용
            # Order O(h^{2(j+1)})을 명확히 표시
            legend_label = r'$R_{i, %d}$ ($\mathcal{O}(h^{%d})$)' % (j, order)
            
            plt.plot(h_values, errors, marker='o', 
                     linestyle='-', label=legend_label)
            
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel(r'$\propto h^2$ (Log Scale)')
    plt.ylabel(r'Absolute Error $|R_{i, j} - I_{\text{true}}|$ (Log Scale)')
    plt.title(' Romberg Integration Convergence Plot (Absolute Error)')
    plt.legend(loc='lower left')
    plt.grid(True, which="both", ls="--")
    plt.gca().invert_xaxis() # h가 작아질수록 (정확도가 높아질수록) 오른쪽으로 이동
    plt.savefig('ri_conv.png')
    plt.show()

# --- 프로그램 실행 ---

R_matrix = romberg_integration(f, A, B, MAX_ITER)

## 4. 롬베르크 외삽행렬 출력
print(" 롬베르크 외삽행렬 (R_{i, j})")
print("-" * 60)
print(f"함수: f(x) = 1/x, 구간: [{A}, {B}], 참값: {TRUE_VALUE:.12f}\n")

# 외삽 행렬을 표 형태로 보기 쉽게 출력
header = [f"j={j} (O(h^{2*(j+1)}))" for j in range(MAX_ITER)]
print("i | " + " | ".join(header))
print("-" * 60)
for i in range(MAX_ITER):
    row_str = f"{i} | "
    # 행렬 출력 시 값이 0이 아니거나 대각선 아래에 있는 값만 출력
    row_str += " | ".join([f"{R_matrix[i, j]:.10f}" if j <= i else "" for j in range(MAX_ITER)])
    print(row_str)

print("-" * 60)

## 5. 수렴성 그래프 플로팅
plot_convergence(R_matrix, TRUE_VALUE, MAX_ITER)

In [ ]:
import numpy as np
from math import pi

# ---------------------------
# 1. 문제 설정
# ---------------------------
N = 10000
a = 1
b = N

# 근사할 함수 f(x) = 1/x^2
def f(x):
    return 1.0 / (x**2)

# f(x)의 도함수 (필요한 차수만 정의)
# f'(x) = -2/x^3
def f_prime(x):
    return -2.0 / (x**3)

# f'''(x) = -24/x^5
def f_triple_prime(x):
    return -24.0 / (x**5)

# 베르누이 수
B2 = 1.0 / 6.0
B4 = -1.0 / 30.0

# ---------------------------
# 2. 직접 합 계산 (참값 근사)
# ---------------------------
sum_actual = sum(f(k) for k in range(a, b + 1))

# ---------------------------
# 3. 오일러-매클로린 공식 적용
# ---------------------------

# 1. 적분항: integral(1/x^2 dx) = -1/x
# integral_a_to_b f(x) dx = [-1/x]_a^b = (-1/b) - (-1/a)
integral_term = (1.0 / a) - (1.0 / b) 

# 2. 사다리꼴 오차항: (f(a) + f(b)) / 2
trapezoidal_error = (f(a) + f(b)) / 2.0

# 3. 보정항 (j=1, 2차항)
# (B2 / 2!) * (f'(b) - f'(a))
corr_term_2nd = (B2 / math.factorial(2)) * (f_prime(b) - f_prime(a))

# 4. 보정항 (j=2, 4차항)
# (B4 / 4!) * (f'''(b) - f'''(a))
corr_term_4th = (B4 / math.factorial(4)) * (f_triple_prime(b) - f_triple_prime(a))

# 최종 근삿값
sum_euler_maclaurin = integral_term + trapezoidal_error + corr_term_2nd + corr_term_4th

# ---------------------------
# 4. 결과 출력
# ---------------------------
print(f"--- 오일러-매클로린 공식을 이용한 합 근사 (N={N}) ---")
print(f"함수: f(x) = 1/x^2")
print(f"직접 합 계산 (참값 근사): {sum_actual:.15f}")
print(f"오일러-매클로린 근삿값: {sum_euler_maclaurin:.15f}")
print(f"절대 오차: {abs(sum_actual - sum_euler_maclaurin):.15e}")

# 무한 급수 값 (pi^2/6)
zeta_2 = (pi**2) / 6.0
# (N=1000)까지의 합에 대한 오차 분석이므로 무한 급수값 자체는 직접적인 비교 대상이 아님
# print(f"무한 급수 (zeta(2)) 값: {zeta_2:.15f}")

In [ ]:
import math

def f(x):
    """f(x) = 1/sqrt(x)"""
    return 1.0 / math.sqrt(x)

def f_prime(x):
    """f'(x) = -1/2 * x^(-3/2)"""
    return -0.5 * math.pow(x, -1.5)

def integral_f(N):
    """
    적분 \\int_{1}^{N} (1/sqrt(x)) dx = 2*sqrt(N) - 2
    """
    return 2 * math.sqrt(N) - 2

def euler_maclaurin_approx(N):
    """
    오일러-매클로린 공식을 사용하여 급수를 근사한다.
    (M=1, 즉 2차 도함수 항까지 사용)
    """
    
    # 1. 적분 항
    Integral_Term = integral_f(N)
    
    # 2. 경계값 항 (f(1) + f(N)) / 2
    Boundary_Term = (f(1) + f(N)) / 2.0
    
    # 3. 1차 도함수 항 (베르누이 수 B2/2! = 1/12)
    # B_2 / 2! * [f'(N) - f'(1)]
    Derivative_Term_1 = (1.0 / 12.0) * (f_prime(N) - f_prime(1))
    
    # f'(1) = -0.5 * 1^(-1.5) = -0.5 이므로
    # Derivative_Term_1 = (1.0 / 12.0) * (f_prime(N) - (-0.5))
    
    approximation = Integral_Term + Boundary_Term + Derivative_Term_1
    
    return approximation

def sum_series_exact(N):
    """
    정확한 값을 계산한다. (비교용)
    """
    return sum(1.0 / math.sqrt(k) for k in range(1, N + 1))

# --- 테스트 ---
N_value = 100000

# 정확한 값
exact_sum = sum_series_exact(N_value)
# 오일러-매클로린 근사 값
em_approx = euler_maclaurin_approx(N_value)

# 결과 출력
print(f"--- N={N_value} 에 대한 오일러-매클로린 근사 ---")
print(f"정확한 합계: {exact_sum:.15f}")
print(f"근사 합계: {em_approx:.15f}")

# 오차 계산 및 비교
absolute_error = abs(exact_sum - em_approx)
relative_error = absolute_error / exact_sum * 100

print(f"\n절대 오차: {absolute_error:.2e}")
print(f"상대 오차율: {relative_error:.10f} %")

In [ ]:
import math

def f(x):
    """f(x) = 1/x^2"""
    return 1.0 / (x**2)

def f_prime(x):
    """f'(x) = -2/x^3"""
    return -2.0 / (x**3)

def f_triple_prime(x):
    """f'''(x) = -24/x^5"""
    return -24.0 / (x**5)

def euler_maclaurin_k_squared(N, M=2):
    """
    오일러-매클로린 공식을 사용하여 sum(1/k^2)를 근사한다.
    M=2 (1차 및 3차 도함수 항 포함)
    """
    
    # 1. 적분 항: 1 - 1/N
    integral_term = 1.0 - (1.0 / N)
    
    # 2. 경계값 항: (f(1) + f(N)) / 2
    boundary_term = (f(1) + f(N)) / 2.0
    
    # 3. 1차 도함수 항 (m=1): B2/2! * [f'(N) - f'(1)] = 1/12 * [f'(N) - f'(1)]
    f_prime_N = f_prime(N)
    f_prime_1 = f_prime(1)
    derivative_term_1 = (1.0 / 12.0) * (f_prime_N - f_prime_1)
    
    # 4. 3차 도함수 항 (m=2): B4/4! * [f'''(N) - f'''(1)] = -1/720 * [f'''(N) - f'''(1)]
    if M >= 2:
        f_triple_prime_N = f_triple_prime(N)
        f_triple_prime_1 = f_triple_prime(1)
        derivative_term_2 = (-1.0 / 720.0) * (f_triple_prime_N - f_triple_prime_1)
    else:
        derivative_term_2 = 0.0

    approximation = integral_term + boundary_term + derivative_term_1 + derivative_term_2
    
    return approximation

def sum_series_exact(N):
    """정확한 합계 계산 (비교용)"""
    return sum(f(k) for k in range(1, N + 1))

# --- 테스트 ---
N_value = 100000

# 정확한 값
exact_sum = sum_series_exact(N_value)
# 오일러-매클로린 근사 값 (M=2)
em_approx = euler_maclaurin_k_squared(N_value, M=2)

# 결과 출력
print(f"--- N={N_value} 에 대한 오일러-매클로린 근사 ($\sum 1/k^2$) ---")
print(f"정확한 합계: {exact_sum:.15f}")
print(f"근사 합계: {em_approx:.15f}")

absolute_error = abs(exact_sum - em_approx)
print(f"\n절대 오차: {absolute_error:.2e}")

In [ ]:
def kahan_sum(input_list):
    """
    카한(Kahan) 합산 알고리즘을 사용하여 부동소수점 리스트의 합계를 계산한다.
    누적되는 반올림 오차(rounding error)를 보정한다.
    """
    sum_value = 0.0  # 누적 합계
    c = 0.0          # 오차 보정 값 (손실된 하위 비트)

    for x in input_list:
        # 1. 현재 오차 c를 x에서 빼서 이전 오차를 보정하려고 시도
        y = x - c

        # 2. 보정된 값 y를 합계에 더함
        t = sum_value + y

        # 3. 이번 단계에서 실제로 손실된 오차를 계산하여 c에 저장
        #    (t - sum_value)는 실제로 sum에 더해진 y의 부분
        #    (t - sum_value)를 y에서 빼면 손실된 부분(오차)이 나옴
        c = (t - sum_value) - y

        # 4. 합계 업데이트
        sum_value = t

    return sum_value
# --- 테스트 데이터 준비 ---
# 1. 매우 큰 숫자
large_number = 1000000.0

# 2. 매우 작은 숫자 (이 숫자를 10,000번 더하면 1.0이 되어야 한다.)
small_increment = 0.0001
num_iterations = 10000

# 최종 합계는 1000000.0 + (0.0001 * 10000) = 1000000.0 + 1.0 = 1000001.0
expected_result = 1000001.0

# 합산할 리스트 생성: 큰 숫자 1개 + 작은 숫자 10,000개
test_list = [large_number] + [small_increment] * num_iterations

# --- 결과 계산 및 출력 ---

# 1. 카한 합산 결과
kahan_result = kahan_sum(test_list)
kahan_error = abs(kahan_result - expected_result)

# 2. 파이썬 기본 sum() 함수 결과 (대부분의 환경에서 Kahan과 유사하게 정교함)
standard_sum_result = sum(test_list)
standard_sum_error = abs(standard_sum_result - expected_result)

# 3. 단순 반복 덧셈 결과 (가장 큰 오차 발생 가능성이 높음)
simple_sum = 0.0
for x in test_list:
    simple_sum += x
simple_sum_error = abs(simple_sum - expected_result)


print(f"--- 카한 합산 정확도 비교 ---")
print(f" 목표(기대) 합계: {expected_result}")
print("-" * 30)
print(f"1. 카한 합산 결과:     {kahan_result}")
print(f"   절대 오차:         {kahan_error:.15e}") # 과학적 표기법으로 오차 표시

print("-" * 30)
print(f"2. Python sum() 결과: {standard_sum_result}")
print(f"   절대 오차:         {standard_sum_error:.15e}")

print("-" * 30)
print(f"3. 단순 덧셈 결과:    {simple_sum}")
print(f"   절대 오차:         {simple_sum_error:.15e}")

In [ ]:
import numpy as np
# 1. 근을 찾고자 하는 함수 f(x)
def f(x):
    return x**3 - 6 * x**2 + 11 * x - 6
# 2. 함수 f(x)의 미분 함수 f'(x)
def f_prime(x):
    return 3 * x**2 - 12 * x + 11
def hybrid_newton_bisection(f, f_prime, a, b, tol=1e-6, max_iter=100):
    """
    뉴턴-랩슨법과 이분법을 결합한 하이브리드 해 찾기 함수이다.
    :param f: 함수 f(x)
    :param f_prime: 함수 f'(x)
    :param a, b: 근을 포함하는 초기 구간 (f(a)와 f(b)의 부호가 달라야 함)
    :return: 근사 해
    """
    if f(a) * f(b) >= 0:
        print(" 오류: f(a)와 f(b)의 부호가 달라야 한다.")
        return None
    a_n, b_n = a, b
    x_n = (a + b) / 2 # 초기 추정값은 구간의 중간으로 시작
    print(f"--- 하이브리드 시작 (초기 구간 [{a:.6f}, {b:.6f}], x0={x_n:.6f}) ---")
    for i in range(max_iter):
        f_x = f(x_n)
        # 1. 수렴 조건 확인
        if abs(f_x) < tol or (b_n - a_n) < tol:
            print(f" {i+1}번째 반복에서 수렴. 근: {x_n:.10f}")
            return x_n
        # 2. 뉴턴-랩슨법 시도
        f_prime_x = f_prime(x_n)
        # 미분값이 0에 가깝거나 뉴턴법으로 계산한 x_next가 불안정할 때 이분법으로 전환
        if abs(f_prime_x) < 1e-10:
            is_newton_safe = False
        else:
            x_next_newton = x_n - f_x / f_prime_x
            # 뉴턴법 안전성 검사:
            # A. x_next가 현재 구간 [a_n, b_n] 안에 있는지
            # B. f(x_next)의 값이 f(x_n)보다 작아졌는지 (수렴 방향인지)
            is_in_range = (x_next_newton > a_n) and (x_next_newton < b_n)
            is_better = abs(f(x_next_newton)) < abs(f_x)
  
            is_newton_safe = is_in_range and is_better
        # 3. 다음 단계 결정 및 구간 업데이트
        if is_newton_safe:
            # 뉴턴법 성공: 뉴턴법 결과 사용
            x_next = x_next_newton
            print(f"반복 {i+1} (Newton): x = {x_next:.6f}")
        else:
            # 뉴턴법 실패 또는 불안정: 이분법 강제 실행
            x_next = (a_n + b_n) / 2
            print(f"반복 {i+1} (Bisection): x = {x_next:.6f} [Fallback]")
       # 4. 구간 [a_n, b_n] 업데이트 (이분법 로직 적용)
        f_next = f(x_next)
        if f(a_n) * f_next < 0:
            b_n = x_next # 해는 [a_n, x_next]에 있음
        else:
              a_n = x_next # 해는 [x_next, b_n]에 있음
        x_n = x_next
    print(" 최대 반복 횟수에 도달했다.")
    return x_n
  
# --- 실행 예제 ---
# 근 x=1을 찾기 위해 구간 [0.5, 1.5] 사용 (f(0.5)=-3.375, f(1.5)=-1.125) -> 부호가 같아 실패!
# 근 x=1을 찾기 위해 구간 [0.5, 1.2] 사용 (f(0.5)=-3.375, f(1.2)=0.048) -> 성공!
root_hybrid = hybrid_newton_bisection(f, f_prime, a=0.6, b=1.2)
print(f"\n최종 찾은 근 (하이브리드): **{root_hybrid}**")
root_hybrid = hybrid_newton_bisection(f, f_prime, a=1.2, b=2.2)
print(f"\n최종 찾은 근 (하이브리드): **{root_hybrid}**")
root_hybrid = hybrid_newton_bisection(f, f_prime, a=2.2, b=3.2)
print(f"\n최종 찾은 근 (하이브리드): **{root_hybrid}**")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# matplotlib 설정을 위한 코드 추가 (한글 깨짐 방지 등)
# 폰트가 설치되어 있지 않으면 기본 설정으로 실행된다.
try:
    plt.rcParams['font.family'] = 'Malgun Gothic' # Windows 사용자용
except:
    pass

plt.rcParams['axes.unicode_minus'] = False # 마이너스 폰트 깨짐 방지

# --- 1. Fornberg 가중치 계산 함수 ---
def weights(z, x, nd, m):
    """
    Fornberg의 방법을 사용하여 유한 차분 가중치를 계산한다.
    """
    c1 = 1
    c4 = x[0] - z
    c = np.zeros((nd+1, m+1)) 
    c[0, 0] = 1 
    
    for i in range(1, nd+1):
        mn = min(i, m)
        c2 = 1
        c5 = c4
        c4 = x[i] - z
        
        for j in range(0, i):
            c3 = x[i] - x[j]
            c2 = c2*c3
            
            if j == i-1:
                for k in range(mn, 0, -1):
                    c[i, k] = c1*(k*c[i-1, k-1] - c5*c[i-1, k])/c2
                c[i, 0] = -c1*c5*c[i-1, 0]/c2
            
            for k in range(mn, 0, -1):
                c[j, k] = (c4*c[j, k] - k*c[j, k-1])/c3
            c[j, 0] = c4*c[j, 0]/c3
            
        c1 = c2
    return c

# --- 2. 미분값 계산 함수 ---
def get_yprimes(nleft, m, x, y):
    """
    주어진 함수 y에 대해 x 배열의 모든 점에서 1차 및 2차 미분값을 계산한다.
    """
    npt = len(x)
    y1 = np.zeros(npt)
    y2 = np.zeros(npt)
    
    nright = nleft 
    nd = nleft+nright 
    xsten = np.zeros(nd+1) 
    
    for j in range(npt): 
        z = x[j] 
        tmp = 0.
        tmq = 0.
        
        # 스텐실 포인트 결정 및 경계 조건 처리
        if j-nleft < 0: 
            j0 = 0
            j1 = nd+1
            xsten[0:nd+1] = x[j0:j1]
        elif j-nleft+nd+1 > npt: 
            j1 = npt
            j0 = j1-nd-1
            xsten[0:nd+1] = x[j0:j1]
        else: 
            j0 = j-nleft
            j1 = j0+nd+1
            xsten[0:nd+1] = x[j0:j1]
            
        c = weights(z, xsten, nd, m) 
        
        # 1차, 2차 미분값 계산 (가중치 c[k, 1]과 c[k, 2] 사용)
        for k in range(nd+1):
            tmp = tmp+c[k, 1]*y[j0+k] 
            tmq = tmq+c[k, 2]*y[j0+k] 
            
        y1[j] = tmp
        y2[j] = tmq
        
    return y1, y2

# --- 3. 정밀도 분석 및 출력 함수 (이전 단계에서 제공) ---
def analyze_precision(f, f_prime_exact, f_double_prime_exact, target_x=1.0, 
                      n_points=1001, range_start=0.0, range_end=2.0, max_order=2, 
                      nleft_values=[1, 2, 3, 5, 7, 10]):
    """
    Fornberg 미분법의 정밀도를 분석하고 결과를 출력한다. (테이블 출력)
    """
    # ... (이전 코드의 analyze_precision 내용) ...
    print(f"--- Fornberg 미분 정밀도 분석: f(x) = exp(-x) ---")
    
    x = np.linspace(range_start, range_end, n_points)
    y = f(x)
    
    true_y1_at_x1 = f_prime_exact(target_x)
    true_y2_at_x1 = f_double_prime_exact(target_x)
    
    print(f"대상 x = {target_x}")
    print(f"1차 미분 참값 f'({target_x}) = {true_y1_at_x1:.10f}")
    print(f"2차 미분 참값 f''({target_x}) = {true_y2_at_x1:.10f}")
    print("-" * 50)

    results = []
    max_nleft = (n_points - 1) // 2
    nleft_values = [n for n in nleft_values if n <= max_nleft]
    
    for nleft in nleft_values:
        stencil_size = 2 * nleft + 1 
        
        y1_num, y2_num = get_yprimes(nleft, max_order, x, y)
        
        target_index = np.argmin(np.abs(x - target_x))
        
        num_y1_at_x1 = y1_num[target_index]
        num_y2_at_x1 = y2_num[target_index]
        
        error_y1 = np.abs(num_y1_at_x1 - true_y1_at_x1)
        error_y2 = np.abs(num_y2_at_x1 - true_y2_at_x1)
        
        results.append({
            'n_left': nleft,
            'Stencil Size (2n+1)': stencil_size,
            "1st Derivative Num": f"{num_y1_at_x1:.10f}",
            "1st Derivative Error": f"{error_y1:.2e}",
            "2nd Derivative Num": f"{num_y2_at_x1:.10f}",
            "2nd Derivative Error": f"{error_y2:.2e}"
        })

    df_results = pd.DataFrame(results)
    print(f"계산된 미분값과 정밀도 (절대 오차) - x={target_x} 지점:")
    print(df_results.to_markdown(index=False, numalign="left", stralign="left"))
    print("\n")


# --- 4. 시각화 함수 ---
def plot_results(x, y1_num, y2_num, f_prime_exact, f_double_prime_exact, nleft):
    """
    Fornberg 미분 결과를 시각화한다.
    """
    y1_true = f_prime_exact(x)
    y2_true = f_double_prime_exact(x)
    
    error_y1 = np.abs(y1_num - y1_true)
    error_y2 = np.abs(y2_num - y2_true)

    plt.style.use('seaborn-v0_8-whitegrid')
    
    # --- 첫 번째 그림: 미분값 비교 ---
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    fig.suptitle(f'Fornberg 유한 차분 결과 (n_left={nleft}, 스텐실={2*nleft+1})', fontsize=16)

    # 1차 미분 비교
    axes[0].plot(x, y1_num, 'b-', label='calc. $f\'(x)$')
    axes[0].plot(x, y1_true, 'r--', label='true $f\'(x) = -e^{-x}$')
    axes[0].set_ylabel('1st deri.', fontsize=12)
    axes[0].legend()
    axes[0].set_title('calc. vs. true (1st deri.)', fontsize=14)

    # 2차 미분 비교
    axes[1].plot(x, y2_num, 'g-', label='calc. $f\'\'(x)$')
    axes[1].plot(x, y2_true, 'r--', label='true $f\'\'(x) = e^{-x}$')
    axes[1].set_xlabel('x', fontsize=12)
    axes[1].set_ylabel('2nd deri.', fontsize=12)
    axes[1].legend()
    axes[1].set_title('calc. vs. true (2nd deri)', fontsize=14)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show() # 


    # --- 두 번째 그림: 절대 오차 (로그 스케일) ---
    plt.figure(figsize=(10, 5))
    plt.title(f'absoulute error (log scale) - Fornberg method (n_left={nleft})', fontsize=16)
    
    plt.plot(x, error_y1, 'b-', label='1st deri. error $|f\'_{num} - f\'_{true}|$')
    plt.plot(x, error_y2, 'g-', label='2nd deri. error $|f\'\'_{num} - f\'\'_{true}|$')
    
    plt.yscale('log') 
    plt.xlabel('x', fontsize=12)
    plt.ylabel('absolute error', fontsize=12)
    
    # 기계 정밀도 (Machine Epsilon) 표시
    plt.axhline(y=np.finfo(float).eps, color='k', linestyle=':', 
                label=f'machine accuracy (Machine Epsilon $\\approx 10^{{{np.log10(np.finfo(float).eps):.1f}}}$)')

    plt.ylim(1e-18, 1) 
    plt.legend()
    plt.grid(True, which="both", ls="--")
    plt.tight_layout()
    plt.show() # 


# --- 5. 메인 테스트 및 실행 ---
if __name__ == "__main__":
    
    # 함수 정의: f(x) = exp(-x)
    f = lambda x: np.exp(-x)
    f_prime_exact = lambda x: -np.exp(-x) # f'(x) = -exp(-x)
    f_double_prime_exact = lambda x: np.exp(-x) # f''(x) = exp(-x)

    # 설정값
    N_POINTS = 1001 # 데이터 포인트 수
    RANGE_START = 0.0
    RANGE_END = 2.0
    MAX_ORDER = 2   # 최대 2차 미분까지 계산
    NLEFT_TO_TEST = 5 # 시각화에 사용할 nleft 값 (11점 스텐실)
    
    # 1. 데이터 생성
    x = np.linspace(RANGE_START, RANGE_END, N_POINTS)
    y = f(x)
    
    print("=========================================================")
    print(f"| 테스트 함수: f(x) = exp(-x) | N = {N_POINTS} 포인트 |")
    print("=========================================================")

    # 2. 정밀도 분석 (표 출력)
    analyze_precision(f, f_prime_exact, f_double_prime_exact)
    
    # 3. 미분값 계산 (시각화에 사용할 nleft 값으로 재계산)
    print(f"--- 미분값 계산 (n_left={NLEFT_TO_TEST}) ---")
    y1_num, y2_num = get_yprimes(NLEFT_TO_TEST, MAX_ORDER, x, y)
    print(f"-> {NLEFT_TO_TEST*2+1}점 스텐실을 사용하여 미분값 계산 완료.")

    # 4. 그림 그리기 함수 호출
    print("\n--- 계산 결과 시각화 시작 ---")
    plot_results(x, y1_num, y2_num, f_prime_exact, f_double_prime_exact, NLEFT_TO_TEST)
    print("----------------------------")
    print("시각화가 완료되었다. 두 개의 그래프 창을 확인하세요.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# matplotlib 설정을 위한 코드 추가 (한글 깨짐 방지 등)
# 폰트가 설치되어 있지 않으면 기본 설정으로 실행된다.
try:
    plt.rcParams['font.family'] = 'Malgun Gothic' # Windows 사용자용
except:
    pass

plt.rcParams['axes.unicode_minus'] = False # 마이너스 폰트 깨짐 방지

# --- 1. Fornberg 가중치 계산 함수 ---
def weights(z, x, nd, m):
    """
    Fornberg의 방법을 사용하여 유한 차분 가중치를 계산한다.
    """
    c1 = 1
    c4 = x[0] - z
    c = np.zeros((nd+1, m+1)) 
    c[0, 0] = 1 
    
    for i in range(1, nd+1):
        mn = min(i, m)
        c2 = 1
        c5 = c4
        c4 = x[i] - z
        
        for j in range(0, i):
            c3 = x[i] - x[j]
            c2 = c2*c3
            
            if j == i-1:
                for k in range(mn, 0, -1):
                    c[i, k] = c1*(k*c[i-1, k-1] - c5*c[i-1, k])/c2
                c[i, 0] = -c1*c5*c[i-1, 0]/c2
            
            for k in range(mn, 0, -1):
                c[j, k] = (c4*c[j, k] - k*c[j, k-1])/c3
            c[j, 0] = c4*c[j, 0]/c3
            
        c1 = c2
    return c

# --- 2. 미분값 계산 함수 ---
def get_yprimes(nleft, m, x, y):
    """
    주어진 함수 y에 대해 x 배열의 모든 점에서 1차 및 2차 미분값을 계산한다.
    """
    npt = len(x)
    y1 = np.zeros(npt)
    y2 = np.zeros(npt)
    
    nright = nleft 
    nd = nleft+nright 
    xsten = np.zeros(nd+1) 
    
    for j in range(npt): 
        z = x[j] 
        tmp = 0.
        tmq = 0.
        
        # 스텐실 포인트 결정 및 경계 조건 처리
        if j-nleft < 0: 
            j0 = 0
            j1 = nd+1
            xsten[0:nd+1] = x[j0:j1]
        elif j-nleft+nd+1 > npt: 
            j1 = npt
            j0 = j1-nd-1
            xsten[0:nd+1] = x[j0:j1]
        else: 
            j0 = j-nleft
            j1 = j0+nd+1
            xsten[0:nd+1] = x[j0:j1]
            
        c = weights(z, xsten, nd, m) 
        
        # 1차, 2차 미분값 계산 (가중치 c[k, 1]과 c[k, 2] 사용)
        for k in range(nd+1):
            tmp = tmp+c[k, 1]*y[j0+k] 
            tmq = tmq+c[k, 2]*y[j0+k] 
            
        y1[j] = tmp
        y2[j] = tmq
        
    return y1, y2

# --- 3. 정밀도 분석 및 출력 함수 (이전 단계에서 제공) ---
def analyze_precision(f, f_prime_exact, f_double_prime_exact, target_x=1.0, 
                      n_points=1001, range_start=0.0, range_end=2.0, max_order=2, 
                      nleft_values=[1, 2, 3, 5, 7, 10]):
    """
    Fornberg 미분법의 정밀도를 분석하고 결과를 출력한다. (테이블 출력)
    """
    # ... (이전 코드의 analyze_precision 내용) ...
    print(f"--- Fornberg 미분 정밀도 분석: f(x) = exp(-x) ---")
    
    x = np.linspace(range_start, range_end, n_points)
    y = f(x)
    
    true_y1_at_x1 = f_prime_exact(target_x)
    true_y2_at_x1 = f_double_prime_exact(target_x)
    
    print(f"대상 x = {target_x}")
    print(f"1차 미분 참값 f'({target_x}) = {true_y1_at_x1:.10f}")
    print(f"2차 미분 참값 f''({target_x}) = {true_y2_at_x1:.10f}")
    print("-" * 50)

    results = []
    max_nleft = (n_points - 1) // 2
    nleft_values = [n for n in nleft_values if n <= max_nleft]
    
    for nleft in nleft_values:
        stencil_size = 2 * nleft + 1 
        
        y1_num, y2_num = get_yprimes(nleft, max_order, x, y)
        
        target_index = np.argmin(np.abs(x - target_x))
        
        num_y1_at_x1 = y1_num[target_index]
        num_y2_at_x1 = y2_num[target_index]
        
        error_y1 = np.abs(num_y1_at_x1 - true_y1_at_x1)
        error_y2 = np.abs(num_y2_at_x1 - true_y2_at_x1)
        
        results.append({
            'n_left': nleft,
            'Stencil Size (2n+1)': stencil_size,
            "1st Derivative Num": f"{num_y1_at_x1:.10f}",
            "1st Derivative Error": f"{error_y1:.2e}",
            "2nd Derivative Num": f"{num_y2_at_x1:.10f}",
            "2nd Derivative Error": f"{error_y2:.2e}"
        })

    df_results = pd.DataFrame(results)
    print(f"계산된 미분값과 정밀도 (절대 오차) - x={target_x} 지점:")
    print(df_results.to_markdown(index=False, numalign="left", stralign="left"))
    print("\n")


# --- 4. 시각화 함수 ---
def plot_results(x, y1_num, y2_num, f_prime_exact, f_double_prime_exact, nleft):
    """
    Fornberg 미분 결과를 시각화한다.
    """
    y1_true = f_prime_exact(x)
    y2_true = f_double_prime_exact(x)
    
    error_y1 = np.abs(y1_num - y1_true)
    error_y2 = np.abs(y2_num - y2_true)

    plt.style.use('seaborn-v0_8-whitegrid')
    
    # --- 첫 번째 그림: 미분값 비교 ---
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    fig.suptitle(f'Fornberg 유한 차분 결과 (n_left={nleft}, 스텐실={2*nleft+1})', fontsize=16)

    # 1차 미분 비교
    axes[0].plot(x, y1_num, 'b-', label='calc. $f\'(x)$')
    axes[0].plot(x, y1_true, 'r--', label='true $f\'(x) = -e^{-x}$')
    axes[0].set_ylabel('1st deri.', fontsize=12)
    axes[0].legend()
    axes[0].set_title('calc. vs. true (1st deri.)', fontsize=14)

    # 2차 미분 비교
    axes[1].plot(x, y2_num, 'g-', label='calc. $f\'\'(x)$')
    axes[1].plot(x, y2_true, 'r--', label='true $f\'\'(x) = e^{-x}$')
    axes[1].set_xlabel('x', fontsize=12)
    axes[1].set_ylabel('2nd deri.', fontsize=12)
    axes[1].legend()
    axes[1].set_title('calc. vs. true (2nd deri)', fontsize=14)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show() # 


    # --- 두 번째 그림: 절대 오차 (로그 스케일) ---
    plt.figure(figsize=(10, 5))
    plt.title(f'absoulute error (log scale) - Fornberg method (n_left={nleft})', fontsize=16)
    
    plt.plot(x, error_y1, 'b-', label='1st deri. error $|f\'_{num} - f\'_{true}|$')
    plt.plot(x, error_y2, 'g-', label='2nd deri. error $|f\'\'_{num} - f\'\'_{true}|$')
    
    plt.yscale('log') 
    plt.xlabel('x', fontsize=12)
    plt.ylabel('absolute error', fontsize=12)
    
    # 기계 정밀도 (Machine Epsilon) 표시
    plt.axhline(y=np.finfo(float).eps, color='k', linestyle=':', 
                label=f'machine accuracy (Machine Epsilon $\\approx 10^{{{np.log10(np.finfo(float).eps):.1f}}}$)')

    plt.ylim(1e-18, 1) 
    plt.legend()
    plt.grid(True, which="both", ls="--")
    plt.tight_layout()
    plt.show() # 


# --- 5. 메인 테스트 및 실행 ---
if __name__ == "__main__":
    
    # 함수 정의: f(x) = exp(-x)
    f = lambda x: np.exp(-x)
    f_prime_exact = lambda x: -np.exp(-x) # f'(x) = -exp(-x)
    f_double_prime_exact = lambda x: np.exp(-x) # f''(x) = exp(-x)

    # 설정값
    N_POINTS = 1001 # 데이터 포인트 수
    RANGE_START = 0.0
    RANGE_END = 2.0
    MAX_ORDER = 2   # 최대 2차 미분까지 계산
    NLEFT_TO_TEST = 4 # 시각화에 사용할 nleft 값 (9점 스텐실)
    
    # 1. 데이터 생성
    x = np.linspace(RANGE_START, RANGE_END, N_POINTS)
    y = f(x)
    
    print("=========================================================")
    print(f"| 테스트 함수: f(x) = exp(-x) | N = {N_POINTS} 포인트 |")
    print("=========================================================")

    # 2. 정밀도 분석 (표 출력)
    analyze_precision(f, f_prime_exact, f_double_prime_exact)
    
    # 3. 미분값 계산 (시각화에 사용할 nleft 값으로 재계산)
    print(f"--- 미분값 계산 (n_left={NLEFT_TO_TEST}) ---")
    y1_num, y2_num = get_yprimes(NLEFT_TO_TEST, MAX_ORDER, x, y)
    print(f"-> {NLEFT_TO_TEST*2+1}점 스텐실을 사용하여 미분값 계산 완료.")

    # 4. 그림 그리기 함수 호출
    print("\n--- 계산 결과 시각화 시작 ---")
    plot_results(x, y1_num, y2_num, f_prime_exact, f_double_prime_exact, NLEFT_TO_TEST)
    print("----------------------------")
    print("시각화가 완료되었다. 두 개의 그래프 창을 확인하세요.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter



# --- 1. Fornberg 가중치 계산 함수 ---
def weights(z, x, nd, m):
    """
    Fornberg의 방법을 사용하여 유한 차분 가중치를 계산한다.
    z: 미분할 지점
    x: 스텐실(Stencil) 포인트 배열
    nd: 스텐실 포인트의 수 - 1
    m: 계산할 최대 미분 차수
    """
    c1 = 1
    c4 = x[0] - z
    c = np.zeros((nd+1, m+1)) 
    c[0, 0] = 1 
    
    for i in range(1, nd+1):
        mn = min(i, m)
        c2 = 1
        c5 = c4
        c4 = x[i] - z
        
        for j in range(0, i):
            c3 = x[i] - x[j]
            c2 = c2*c3
            
            if j == i-1:
                for k in range(mn, 0, -1):
                    c[i, k] = c1*(k*c[i-1, k-1] - c5*c[i-1, k])/c2
                c[i, 0] = -c1*c5*c[i-1, 0]/c2
            
            for k in range(mn, 0, -1):
                c[j, k] = (c4*c[j, k] - k*c[j, k-1])/c3
            c[j, 0] = c4*c[j, 0]/c3
            
        c1 = c2
    return c

# --- 2. 미분값 계산 함수 ---
def get_yprimes(nleft, m, x, y):
    """
    주어진 함수 y에 대해 x 배열의 모든 점에서 1차 및 2차 미분값을 계산한다.
    """
    npt = len(x)
    y1 = np.zeros(npt)
    y2 = np.zeros(npt)
    
    nright = nleft 
    nd = nleft+nright 
    xsten = np.zeros(nd+1) 
    
    for j in range(npt): 
        z = x[j] 
        tmp = 0.
        tmq = 0.
        
        # 스텐실 포인트 결정 및 경계 조건 처리
        if j-nleft < 0: 
            j0 = 0
            j1 = nd+1
            xsten[0:nd+1] = x[j0:j1]
        elif j-nleft+nd+1 > npt: 
            j1 = npt
            j0 = j1-nd-1
            xsten[0:nd+1] = x[j0:j1]
        else: 
            j0 = j-nleft
            j1 = j0+nd+1
            xsten[0:nd+1] = x[j0:j1]
            
        c = weights(z, xsten, nd, m) 
        
        # 1차, 2차 미분값 계산
        for k in range(nd+1):
            tmp = tmp+c[k, 1]*y[j0+k] 
            tmq = tmq+c[k, 2]*y[j0+k] 
            
        y1[j] = tmp
        y2[j] = tmq
        
    return y1, y2

# --- 3. 고정 그리드 수렴성 분석 및 시각화 함수 (통합) ---
def analyze_precision_convergence(f, f_prime_exact, f_double_prime_exact, target_x=1.0, 
                                  n_points=101, range_start=0.0, range_end=2.0, max_order=2, 
                                  nleft_values=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]):
    """
    고정된 그리드 간격(h)에서 스텐실 크기(n_left)에 따른 미분 정밀도(오차) 수렴성을 분석하고 시각화한다.
    """
    
    print("=========================================================")
    print(f"| 고정 그리드 수렴성 테스트: f(x) = exp(-x) | N = {n_points} 포인트 |")
    print("=========================================================")
    
    # 1. 고정된 그리드 간격 h 계산
    x = np.linspace(range_start, range_end, n_points)
    y = f(x)
    h = x[1] - x[0]
    
    print(f" 고정된 그리드 간격 h: {h:.4e}")
    
    # 2. 참값 계산
    true_y1_at_x1 = f_prime_exact(target_x)
    true_y2_at_x1 = f_double_prime_exact(target_x)
    
    target_index = np.argmin(np.abs(x - target_x))
    
    print(f"대상 지점 x = {target_x}")
    print(f"1차 미분 참값 f'({target_x}) = {true_y1_at_x1:.10f}")
    print(f"2차 미분 참값 f''({target_x}) = {true_y2_at_x1:.10f}")
    print("-" * 50)

    # 3. 다양한 n_left에 대한 오차 계산
    results = []
    max_nleft = (n_points - 1) // 2
    nleft_valid = [n for n in nleft_values if n <= max_nleft]
    
    error_p1_list = []
    error_p2_list = []

    for nleft in nleft_valid:
        stencil_size = 2 * nleft + 1 
        
        # 미분값 계산
        y1_num, y2_num = get_yprimes(nleft, max_order, x, y)
        
        # x=1.0에서의 오차 추출
        num_y1_at_x1 = y1_num[target_index]
        num_y2_at_x1 = y2_num[target_index]
        
        error_y1 = np.abs(num_y1_at_x1 - true_y1_at_x1)
        error_y2 = np.abs(num_y2_at_x1 - true_y2_at_x1)
        
        error_p1_list.append(error_y1)
        error_p2_list.append(error_y2)

        results.append({
            'n_left': nleft,
            'Stencil Size (2n+1)': stencil_size,
            "1st Derivative Order (2n)": 2*nleft, # 대칭 스텐실의 절단 오차 차수: O(h^(2n))
            "1st Derivative Error": f"{error_y1:.2e}",
            "2nd Derivative Error": f"{error_y2:.2e}"
        })

    # 4. 결과 표 출력
    df_results = pd.DataFrame(results)
    print(f"미분 차수(스텐실 크기)에 따른 오차 변화 - x={target_x} 지점:")
    print(df_results.to_markdown(index=False, numalign="left", stralign="left"))
    
    # 5. 오차 시각화 (로그-선형 플롯)
    plt.figure(figsize=(10, 6))
    plt.title(f'fixed $h$ accuracy (h={h:.1e})', fontsize=16)
    ax = plt.gca() # 현재 축 객체를 가져옴

    n_labels = [str(2*n) for n in nleft_valid] 
    x_positions = np.arange(len(nleft_valid))

    plt.plot(x_positions, error_p1_list, 'o-', color='blue', label='1st deri. error $|f\'_{num} - f\'_{true}|$')
    plt.plot(x_positions, error_p2_list, 's-', color='green', label='2nd deri. error $|f\'\'_{num} - f\'\'_{true}|$')
    formatter = FormatStrFormatter('%.0e') 
    ax.yaxis.set_major_formatter(formatter)

    plt.yscale('log')
    plt.ylim(1e-18, 1e-3) 
    plt.xticks(x_positions, n_labels)
    
    plt.xlabel('truncation error p (2n)', fontsize=18)
    plt.ylabel('absolute error (log scale)', fontsize=18)
    
    log_eps = np.log10(np.finfo(float).eps) # 기계 정밀도의 10을 밑으로 하는 로그 값을 계산 및 정의
    plt.axhline(y=np.finfo(float).eps, color='k', linestyle=':', 
                label=fr'machine accuracy ($\approx 10^{{{log_eps:.1f}}}$)')
    plt.legend()
    plt.grid(True, which="both", ls="--")
    plt.tight_layout()
    plt.show() # 


# --- 4. 메인 실행 블록 ---
if __name__ == "__main__":
    
    # 해석적 함수 정의: f(x) = exp(-x)
    f = lambda x: np.exp(-x)
    f_prime_exact = lambda x: -np.exp(-x)
    f_double_prime_exact = lambda x: np.exp(-x)

    # 분석 설정값
    N_POINTS = 101 # 그리드 간격 h를 고정 (h = 0.02)
    
    # 테스트 실행
    analyze_precision_convergence(f, f_prime_exact, f_double_prime_exact, 
                                  n_points=N_POINTS)

    print("\n 분석 및 시각화가 완료되었다.")
    print("출력된 표와 그래프를 통해 고정된 그리드 간격 h에서 스텐실 크기를 늘릴수록")
    print("미분 오차가 기계 정밀도 한계까지 빠르게 감소하는 것을 확인할 수 있다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 예시 데이터 (X: 입력 변수, Y: 출력 변수)
# 예: X = [1, 2, 3, 4, 5], Y = [1, 2, 1.9, 4.2, 5.1]
X = np.array([1, 2, 3, 4, 5])
Y = np.array([1, 2, 1.9, 4.2, 5.1])
# 입력 데이터 행렬 A 구성 (선형 회귀에서는 1을 추가하여 상수항을 포함)
A = np.vstack([X, np.ones_like(X)]).T
# 최소제곱법을 통한 회귀 계수(beta) 계산
# beta = (A^T * A)^(-1) * A^T * Y
beta = np.linalg.inv(A.T @ A) @ A.T @ Y
# 회귀 직선 방정식
slope, intercept = beta
# 예측 값 계산 (회귀 직선 상의 Y 값)
Y_pred = A @ beta
# 결과 출력
print(f"기울기 (slope): {slope}")
print(f"절편 (intercept): {intercept}")
# 원본 데이터와 회귀 직선 시각화
plt.scatter(X, Y, color='blue', label='Original data')
plt.plot(X, Y_pred, color='red', label='Fitted line')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.show()

In [ ]:
import numpy as np
from numpy.linalg import cholesky

# 1. 최소화할 함수 (목표 함수) 정의
def objective_function(x):
    """
    2차원 쿼드라틱 함수: f(x) = (x1 - 1)^2 + (x2 - 2)^2
    최소값은 (1, 2)에서 0이다.
    """
    return (x[0] - 1.0)**2 + (x[1] - 2.0)**2
def monte_carlo_optimization_with_cholesky(
    func, 
    initial_guess, 
    n_iterations=1000, 
    initial_step_size=0.5, 
    decay_rate=0.99
):
    """
    촐레스키 분해를 사용하여 탐색을 개선하는 몬테카를로 최소화.
    
    :param func: 최소화할 함수
    :param initial_guess: 초기 탐색 지점 (numpy array)
    :param n_iterations: 반복 횟수
    :param initial_step_size: 초기 탐색 반경 (표준편차)
    :param decay_rate: 탐색 반경 감소율
    :return: 최적의 해 x_best, 최소값 f_best
    """
    # 초기 설정
    current_x = np.array(initial_guess)
    current_f = func(current_x)
    best_x = current_x
    best_f = current_f
    dim = len(initial_guess)
    step_size = initial_step_size

    print(f"초기 추정: x={current_x}, f={current_f:.4f}")

    for i in range(n_iterations):
        
        # 1. 공분산 행렬 정의 및 촐레스키 분해
        # 탐색 범위를 정의하는 공분산 행렬 (여기서는 간단히 대각 행렬 사용)
        # 탐색이 진행될수록 step_size가 줄어들어 공분산 행렬의 분산이 작아짐
        covariance_matrix = np.eye(dim) * (step_size**2)
        
        try:
            # L: 촐레스키 분해 결과 (하삼각 행렬)
            L = cholesky(covariance_matrix)
        except np.linalg.LinAlgError:
            # 행렬이 양의 정부호가 아닐 때 (거의 발생하지 않음)
            print("촐레스키 분해 오류. 탐색을 종료한다.")
            break
        
        # 2. 표준 정규 난수 생성
        # 독립적인 표준 정규 난수 벡터 z
        standard_normal_z = np.random.normal(size=dim)
        
        # 3. 상관 난수 (탐색 이동량) 생성
        # Lz는 공분산 행렬 Sigma를 가진 다변수 정규 분포에서의 샘플링에 해당
        # L @ standard_normal_z는 탐색 이동 벡터(delta)가 된다.
        delta_x = L @ standard_normal_z
        
        # 새로운 탐색 지점
        next_x = current_x + delta_x
        next_f = func(next_x)

        # 4. 수락/거절 단계 (단순 개선 수락)
        if next_f < current_f:
            # 개선되면 현재 지점을 업데이트
            current_x = next_x
            current_f = next_f

            # 전체 최소값 업데이트
            if current_f < best_f:
                best_f = current_f
                best_x = current_x

        # 5. 탐색 반경 감소 (Step size decay)
        # 시간이 지남에 따라 탐색 범위를 좁혀 해에 수렴하도록 유도
        step_size *= decay_rate
        
        if (i + 1) % 100 == 0:
            print(f"반복 {i+1}: f_best={best_f:.6f}, x_best={best_x}")

    print("-" * 30)
    print(f" 최종 최소값: f({best_x}) = {best_f:.6f}")
    return best_x, best_f
# 프로그램 실행
initial_x = np.array([-5.0, 5.0]) # 초기 추측 지점
# Cholesky를 활용한 몬테카를로 최적화 실행
optimal_x, minimal_f = monte_carlo_optimization_with_cholesky(
    objective_function, 
    initial_x, 
    n_iterations=2000,
    initial_step_size=1.0, 
    decay_rate=0.999 # 느린 감소율로 수렴 유도
)

# 실제 최적의 해는 (1.0, 2.0)
print(f"실제 최적의 해: (1.0, 2.0)")    

In [ ]:
import numpy as np
from numpy.linalg import cholesky

def objective_function(x):
    return (x[0] - 1.0)**2 + (x[1] - 2.0)**2 +(x[2]-3.0)**2

def monte_carlo_optimization_with_adaptive_cholesky(
    func, 
    initial_guess, 
    n_iterations=5000, 
    n_adapt_samples=500, # 공분산 행렬을 업데이트하는 데 사용할 샘플 수
    initial_step_scale=1.0, 
    decay_rate=0.9999, # 적응 단계에서는 느린 감쇠율 사용
):
    """
    적응적 몬테카를로 방법을 활용하여 공분산 행렬을 학습하는 최적화.
    """
    # 초기 설정
    current_x = np.array(initial_guess)
    current_f = func(current_x)
    best_x = current_x
    best_f = current_f
    dim = len(initial_guess)
    #  적응적 공분산 행렬 초기화 (초기에는 독립적인 분산)
    # 탐색 스케일은 initial_step_scale에 의해 결정된다.
    covariance_matrix = np.eye(dim) * (initial_step_scale**2)
    #  수락된 샘플을 저장할 리스트 (공분산 계산에 사용)
    accepted_samples = [current_x]
    print(f"initial : x={current_x}, f={current_f:.4f}")
    for i in range(n_iterations):
        # 1. 적응 단계: 공분산 행렬 업데이트 (주기적인 학습)
        if (i + 1) % n_adapt_samples == 0 and len(accepted_samples) >= dim:
            # 수락된 샘플들의 공분산 행렬 계산 (np.cov는 분산/공분산 계산에 최적화됨)
            # rowvar=False: 각 열이 하나의 변수를 나타냄
            # 수락된 샘플이 충분할 때만 업데이트한다.
            samples_array = np.array(accepted_samples)
            # 표본 공분산 계산
            new_cov = np.cov(samples_array, rowvar=False)
            # 탐색 스케일 factor를 적용하여 새로운 공분산 행렬 정의
            # decay_rate를 사용하여 시간이 지남에 따라 전체 탐색 범위를 축소
            scale_factor = (decay_rate ** i) 
            covariance_matrix = new_cov * scale_factor
            # 샘플 리스트 초기화 후 현재 최적점으로 재시작 (새로운 탐색 분포 학습)
            accepted_samples = [current_x]
            print(f"--- [ADAPTED] 반복 {i+1}: 공분산 행렬 업데이트 및 스케일 조정 (Factor: {scale_factor:.4f}) ---")
        # 2. 촐레스키 분해
        try:
            L = cholesky(covariance_matrix)
        except np.linalg.LinAlgError:
            # 행렬이 양의 정부호가 아닐 때 (적절한 스케일링이 필요할 수 있음)
            print(f"촐레스키 분해 오류. 현재 공분산 행렬: \n{covariance_matrix}. 탐색을 종료한다.")
            break
        # 3. 상관 난수 (탐색 이동량) 생성
        standard_normal_z = np.random.normal(size=dim)
        delta_x = L @ standard_normal_z
        next_x = current_x + delta_x
        next_f = func(next_x)
        # 4. 수락/거절 단계 (단순 개선 수락)
        if next_f < current_f:
            current_x = next_x
            current_f = next_f
            # 수락된 샘플 리스트에 추가 (공분산 학습을 위해)
            accepted_samples.append(current_x)
            if current_f < best_f:
                best_f = current_f
                best_x = current_x
        if (i + 1) % 500 == 0:
            print(f"반복 {i+1}: f_best={best_f:.6f}, x_best={best_x}")
    print("-" * 30)
    print(f" 최종 최소값: f({best_x}) = {best_f:.6f}")
    return best_x, best_f
# 프로그램 실행
initial_x = np.array([-5.0, 5.0, 6.0]) 
optimal_x, minimal_f = monte_carlo_optimization_with_adaptive_cholesky(
    objective_function, 
    initial_x, 
    n_iterations=50000,
    n_adapt_samples=1000, # 500번의 수락된 샘플마다 공분산 갱신
    initial_step_scale=2.0, 
    decay_rate=0.9999)

In [ ]:
import numpy as np

def f(x):
    """미분할 함수: sin(x)"""
    return np.sin(x)

def central_difference(f, x, h):
    """
    중심 차분 공식을 이용한 미분 근사 (수렴 차수 p=2)
    """
    return (f(x + h) - f(x - h)) / (2 * h)

# 분석 지점
x_val = 1.0 
# 정확한 해 (참값)
exact_value = np.cos(x_val) 

print(f"--- 참값 (Exact Value) ---\nf'(1) = cos(1) ≈ {exact_value:.10f}\n")

# 1단계: h 값으로 근사값 계산
h1 = 0.1
D1 = central_difference(f, x_val, h1)
print(f"1. h={h1} 일 때의 근사값 D1: {D1:.10f}")
print(f"   오차: {abs(D1 - exact_value):.10f}")
print("-" * 30)

# 2단계: h/2 값으로 근사값 계산
h2 = h1 / 2  # h2 = 0.05
D2 = central_difference(f, x_val, h2)
print(f"2. h={h2} 일 때의 근사값 D2: {D2:.10f}")
print(f"   오차: {abs(D2 - exact_value):.10f}")
print("-" * 30)

# 리처드슨 외삽법 적용 (p=2)
p = 2
richardson_extrapolated = D2 + (D2 - D1) / (2**p - 1)

print(f"3. 리처드슨 외삽값 (Phi_new): {richardson_extrapolated:.10f}")
print(f"   외삽 후 오차: {abs(richardson_extrapolated - exact_value):.10f}")	

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp # 시작점 계산을 위해 Runge-Kutta 사용

# ====================================================================
# I. ODE 정의 및 참 해(True Solution)
# ====================================================================

# 1. 미분 방정식: y'(t) = f(t, y)
def f(t, y):
    """ODE: y'(t) = y - t^2 + 1"""
    return y - t**2 + 1

# 2. 참 해 (True Solution)
def true_solution(t, y0):
    """
    ODE y' = y - t^2 + 1의 일반해:
    y(t) = t^2 + 2t + 1 + C * exp(t)
    주어진 초기 조건 y(t0) = y0를 사용하여 C를 구한다.
    """
    t0 = 0 # 초기 시간
    # y0 = t0^2 + 2*t0 + 1 + C * exp(t0)
    # y0 = 1 + C * 1  => C = y0 - 1
    C = y0 - (t0**2 + 2 * t0 + 1)
    
    return t**2 + 2 * t + 1 + C * np.exp(t)

# ====================================================================
# II. Adam-Bashforth-Moulton (ABM4) 알고리즘 구현
# ====================================================================

def abm4_predictor_corrector(f, y0, t_span, h):
    """
    4차 Adam-Bashforth-Moulton 예측-수정자 알고리즘을 사용하여 ODE를 푼다.
    
    Args:
        f (function): 미분 함수 f(t, y)
        y0 (float): 초기 조건 y(t0)
        t_span (tuple): (t_start, t_end)
        h (float): 단계 크기 (step size)
        
    Returns:
        tuple: (시간 배열 t, 근사 해 배열 y)
    """
    t_start, t_end = t_span
    t = np.arange(t_start, t_end + h, h)
    n = len(t)
    y = np.zeros(n)
    y[0] = y0
    
    # 1. 시작점 계산 (Starting Values)
    # ABM4는 4단계 방법이므로, 시작점 y1, y2, y3를 계산해야 한다.
    # 높은 정밀도의 룽게-쿠타 방법(RK45)을 사용하여 이 시작점들을 계산한다.
    print(f"단계 크기 h = {h:.4f}에 대해 RK45를 사용하여 시작점 3개 계산 중...")
    
    # solve_ivp를 사용하여 t_start에서 t[3]까지의 해를 구한다.
    sol = solve_ivp(f, (t_start, t[3]), [y0], method='RK45', t_eval=t[:4])
    y[:4] = sol.y[0]
    
    # f_i = f(t_i, y_i) 값들을 저장하여 재활용
    f_vals = np.zeros(n)
    f_vals[:4] = f(t[:4], y[:4])

    # 2. ABM4 주 루프 (t4부터 t_end까지)
    for i in range(3, n - 1):
        # --- (A) Predictor (Adam-Bashforth 4th order, AB4) ---
        # y_{i+1}^* = y_i + h/24 * [55*f_i - 59*f_{i-1} + 37*f_{i-2} - 9*f_{i-3}]
        
        y_pred = y[i] + h / 24.0 * (
            55.0 * f_vals[i] - 59.0 * f_vals[i-1] + 
            37.0 * f_vals[i-2] - 9.0 * f_vals[i-3]
        )
        
        # f_{i+1}^* = f(t_{i+1}, y_{i+1}^*)
        f_pred = f(t[i+1], y_pred)
        
        # --- (B) Corrector (Adam-Moulton 4th order, AM4) ---
        # y_{i+1} = y_i + h/24 * [9*f_{i+1}^* + 19*f_i - 5*f_{i-1} + 1*f_{i-2}]
        
        y[i+1] = y[i] + h / 24.0 * (
            9.0 * f_pred + 19.0 * f_vals[i] - 
            5.0 * f_vals[i-1] + 1.0 * f_vals[i-2]
        )
        
        # 다음 단계에서 사용하기 위해 f_{i+1} 업데이트
        f_vals[i+1] = f(t[i+1], y[i+1])
        
    return t, y

# ====================================================================
# III. 정밀도 테스트 실행 및 검증
# ====================================================================

# 초기 조건 및 범위
y0 = 0.5
t_span = (0.0, 1.0) 

# 단계 크기 설정 (정밀도 비교를 위해 두 가지 사용)
H_SIZES = [0.1, 0.05]

plt.figure(figsize=(12, 6))

# 1. 참 해 계산 (정밀 비교를 위해 매우 작은 단계 크기 사용)
t_true = np.linspace(t_span[0], t_span[1], 500)
y_true = true_solution(t_true, y0)
plt.plot(t_true, y_true, 'k-', label='True Solution $y(t)$', linewidth=2)

# 2. 각 단계 크기에 대해 ABM4 실행 및 정밀도 검증
for h in H_SIZES:
    t_approx, y_approx = abm4_predictor_corrector(f, y0, t_span, h)
    
    # 3. 오차 계산
    # 근사 해의 각 지점에서의 참 해를 계산
    y_true_at_approx = true_solution(t_approx, y0)
    
    # 절대 오차: |참 해 - 근사 해|
    abs_error = np.abs(y_true_at_approx - y_approx)
    max_error = np.max(abs_error)
    
    # 4. 결과 시각화
    plt.plot(t_approx, y_approx, 'o--', label=f'ABM4 (h={h:.2f}, Max Error: {max_error:.2e})')
    
    print(f"단계 크기 h = {h:.2f}: 최대 절대 오차 = {max_error:.5e}")


# 그래프 설정
plt.title('Adam-Bashforth-Moulton (ABM4) Algorithm Verification')
plt.xlabel('t')
plt.ylabel('y(t)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.show()

# --- 오차 시각화 (선택 사항) ---
plt.figure(figsize=(8, 4))
for h in H_SIZES:
    t_approx, y_approx = abm4_predictor_corrector(f, y0, t_span, h)
    y_true_at_approx = true_solution(t_approx, y0)
    abs_error = np.abs(y_true_at_approx - y_approx)
    plt.plot(t_approx, abs_error, 's-', label=f'Absolute Error (h={h:.2f})', markersize=4)

plt.title('Absolute Error Analysis')
plt.xlabel('t')
plt.ylabel('Error |y_true - y_approx|')
plt.yscale('log') # 오차 비교를 위해 로그 스케일 사용
plt.legend()
plt.grid(True, which="both", linestyle=':', alpha=0.7)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# ====================================================================
# I. ODE 정의 및 참 해 (복잡한 버전)
# ====================================================================

# 초기 조건 설정 (t0=1, y0=2)
T0 = 1.0
Y0 = 2.0
np.seterr(divide='ignore', invalid='ignore') # t=0 나누기 오류 경고 무시 설정

# 1. 미분 방정식: y'(t) = y/t + t*cos(t)
def f_complex(t, y):
    """
    ODE: y'(t) = y/t + t*cos(t). 
    t가 배열일 경우에도 문제 없이 작동하도록 NumPy 배열 연산으로 작성.
    """
    # t가 0에 가까울 때 나누기 오류를 방지하기 위해 where 조건을 사용한다.
    # 단, T0 > 0이므로 이 코드에서는 t가 0에 도달하지 않아야 한다.
    # np.divide는 배열 연산의 where 조건 처리를 지원하여 단일 불리언 오류를 피한다.
    
    # t=0일 경우를 처리하는 안전한 나누기
    term1 = np.divide(y, t, out=np.zeros_like(y, dtype=float), where=t!=0)
    term2 = t * np.cos(t)
    
    return term1 + term2

# 2. 참 해 (True Solution): y(t) = t*sin(t) + C*t
def true_solution_complex(t, t0, y0):
    """
    ODE y'(t) = y/t + t*cos(t)의 참 해
    상수 C = y0/t0 - sin(t0)
    """
    # 초기 조건을 사용하여 상수 C 계산
    C = y0 / t0 - np.sin(t0)
    # NumPy 배열 연산은 if/else를 사용하지 않아 'ambiguous' 오류 발생 위험이 없다.
    return t * np.sin(t) + C * t

# ====================================================================
# II. Adam-Bashforth-Moulton (ABM4) 알고리즘 구현
# ====================================================================

def abm4_predictor_corrector(f, y0, t_span, h):
    """
    4차 Adam-Bashforth-Moulton 예측-수정자 알고리즘을 사용하여 ODE를 푼다.
    """
    t_start, t_end = t_span
    # t_end 포함을 위해 작은 epsilon 추가
    t = np.arange(t_start, t_end + h, h)
    n = len(t)
    y = np.zeros(n)
    y[0] = y0
    
    if n < 4:
        raise ValueError("단계 크기가 너무 커서 4개의 시작점을 확보할 수 없다.")
    
    # 1. 시작점 계산 (RK45 사용)
    # atol, rtol을 매우 작게 설정하여 시작점의 정밀도를 극대화
    sol = solve_ivp(f, (t_start, t[3]), [y0], method='RK45', t_eval=t[:4], atol=1e-12, rtol=1e-12)
    y[:4] = sol.y[0]
    
    # f_i = f(t_i, y_i) 값들을 저장
    f_vals = np.zeros(n)
    f_vals[:4] = f(t[:4], y[:4])

    # 2. ABM4 주 루프 (t4부터 t_end까지)
    for i in range(3, n - 1):
        # --- (A) Predictor (AB4) ---
        y_pred = y[i] + h / 24.0 * (
            55.0 * f_vals[i] - 59.0 * f_vals[i-1] + 
            37.0 * f_vals[i-2] - 9.0 * f_vals[i-3]
        )
        f_pred = f(t[i+1], y_pred)
        
        # --- (B) Corrector (AM4) ---
        y[i+1] = y[i] + h / 24.0 * (
            9.0 * f_pred + 19.0 * f_vals[i] - 
            5.0 * f_vals[i-1] + 1.0 * f_vals[i-2]
        )
        
        # 다음 단계에서 사용하기 위해 f_{i+1} 업데이트
        f_vals[i+1] = f(t[i+1], y[i+1])
        
    return t, y

# ====================================================================
# III. 정밀도 테스트 실행 및 검증
# ====================================================================

# 초기 조건 및 범위
t_span = (T0, 5.0) 

# 단계 크기 설정 (h를 절반으로 줄여 4차 정밀도 확인)
H_SIZES = [0.2, 0.1, 0.05]

plt.figure(figsize=(14, 6))

# 1. 참 해 계산 
t_true = np.linspace(t_span[0], t_span[1], 500)
y_true = true_solution_complex(t_true, T0, Y0)

plt.subplot(1, 2, 1)
plt.plot(t_true, y_true, 'k-', label='True Solution $y(t)$', linewidth=3)
plt.title('Function Comparison (Complex ODE)')
plt.xlabel('t')
plt.ylabel('y(t)')
plt.grid(True, linestyle=':', alpha=0.7)

error_results = {}

# 2. 각 단계 크기에 대해 ABM4 실행 및 정밀도 검증
for h in H_SIZES:
    t_approx, y_approx = abm4_predictor_corrector(f_complex, Y0, t_span, h)
    
    # 3. 오차 계산
    y_true_at_approx = true_solution_complex(t_approx, T0, Y0)
    abs_error = np.abs(y_true_at_approx - y_approx)
    max_error = np.max(abs_error)
    
    error_results[h] = abs_error
    
    # 4. 결과 시각화
    plt.subplot(1, 2, 1)
    plt.plot(t_approx, y_approx, 'o--', label=f'ABM4 (h={h:.2f})', markersize=4 if h == H_SIZES[-1] else 2)
    
    print(f"단계 크기 h = {h:.2f}: 최대 절대 오차 = {max_error:.5e}")

plt.subplot(1, 2, 1)
plt.legend()

# --- 오차 시각화 (정밀도 검증의 핵심) ---
plt.subplot(1, 2, 2)
for h, error in error_results.items():
    t_approx, _ = abm4_predictor_corrector(f_complex, Y0, t_span, h) # t 값 다시 계산
    plt.plot(t_approx, error, 's-', label=f'Error (h={h:.2f})', markersize=3)

plt.title('Absolute Error Analysis (Log Scale)')
plt.xlabel('t')
plt.ylabel('Error $|y_{true} - y_{approx}|$')
plt.yscale('log')
plt.legend()
plt.grid(True, which="both", linestyle=':', alpha=0.7)

# 4차 방법의 수렴 차수 확인 (h가 절반이 될 때 오차가 16배 감소하는지)
h1 = H_SIZES[0]
h2 = H_SIZES[1]
if h1 / h2 == 2:
    max_e1 = np.max(error_results[h1])
    max_e2 = np.max(error_results[h2])
    print("\n--- 수렴 차수 검증 ---")
    print(f"h={h1} 최대 오차: {max_e1:.2e}")
    print(f"h={h2} 최대 오차: {max_e2:.2e}")
    print(f"오차 감소 비율 ($E_1/E_2$): {max_e1/max_e2:.2f} (4차 방법이므로 이론적으로 약 16이어야 함)")


plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 문제 설정: 적분 영역은 [0, 1] x [0, 1] 이다.
def f(x1, x2):
    """적분할 함수 f(x1, x2) = sin(pi*x1) * sin(pi*x2)"""
    return np.sin(np.pi * x1) * np.sin(np.pi * x2)

# 2. 해석적 해 (참값)
# I_true = (2/pi) * (2/pi) = 4 / pi^2
I_true = 4.0 / (np.pi ** 2)
print(f"해석적 해 (참값 I_true): {I_true:.6f}")

# ----------------------------------------------------

##  몬테카를로 적분 함수 정의
def monte_carlo_integration(N):
    """
    2차원 몬테카를로 적분 근사 함수
    :param N: 샘플(표본) 개수
    :return: 근사 적분값 I_N
    """
    # 1. 적분 영역 [0, 1] x [0, 1]에서 N개의 난수 샘플 생성
    # x1_i와 x2_i는 0과 1 사이의 균일 분포를 따른다.
    x1 = np.random.rand(N)
    x2 = np.random.rand(N)

    # 2. 함수값 계산
    f_values = f(x1, x2)

    # 3. 몬테카를로 적분 근사: I_N = (V/N) * sum(f(x_i))
    # 여기서 영역의 부피 V = 1 * 1 = 1 이다.
    I_N = np.mean(f_values) # np.mean(f_values)는 sum(f_values)/N 과 동일하다.
    return I_N, f_values

# ----------------------------------------------------

## 수렴성 분석 및 시각화

# A. 샘플 수 N의 범위를 설정 (로그 스케일)
N_list = np.logspace(1, 8, 20, dtype=int) # 10^1부터 10^7까지 20개의 N 값
estimated_integrals = []
errors = []

print("\n--- 2D MC integration (N) ---")
for N in N_list:
    # 각 N에 대해 몬테카를로 적분 실행
    I_N, f_values = monte_carlo_integration(N)
    estimated_integrals.append(I_N)

    # 오차: |I_N - I_true|
    error = np.abs(I_N - I_true)
    errors.append(error)

    # 각 N에 대한 결과 출력
    print(f"N = {N:9d}, MC 값 = {I_N:.6f}, 오차 = {error:.6e}")

# B. 표준 오차 (Standard Error) 분석
# 몬테카를로 적분의 오차는 1/sqrt(N)에 비례한다.
# 이론적 표준 편차 (상수)를 계산하여 비교선을 그린다.
# Var(f) = (1/V) * Integral(f(x)^2) - I^2
# Integral(f^2) = Integral(sin^2(pi*x1) * sin^2(pi*x2)) dx1 dx2
# Integral(sin^2(pi*x)) dx from 0 to 1 = 1/2
# Integral(f^2) = 1/2 * 1/2 = 0.25
variance_f = 0.25 - I_true**2
C = np.sqrt(variance_f) # 이론적 표준 편차 상수 (SE = C/sqrt(N))
theoretical_errors = C / np.sqrt(N_list)

# ----------------------------------------------------

##  결과 시각화
plt.figure(figsize=(12, 5))

### 1. 적분값 수렴 그래프 (왼쪽)
plt.subplot(1, 2, 1)
plt.plot(N_list, estimated_integrals, 'o-', label='MC $I_N$', markersize=4)
plt.axhline(I_true, color='r', linestyle='--', label=f'$I_{{true}}$ = {I_true:.6f}')
plt.xscale('log')
plt.xlabel('$N$')
plt.ylabel('$I_N$')
#plt.title('몬테카를로 적분값의 수렴성')
plt.legend()
plt.grid(True, which="both", ls="--")

### 2. 오차 수렴 그래프 (오른쪽)
plt.subplot(1, 2, 2)
plt.loglog(N_list, errors, 'o', label='$|I_N - I_{true}|$', markersize=4)
plt.loglog(N_list, theoretical_errors, '--', color='gray', label=r'$\propto 1/\sqrt{N}$')
plt.loglog(N_list, 1/N_list, ':', color='purple', label=r'comparison $\propto 1/N$ (lattice)') # 전통적 방법과의 비교
plt.xscale('log')
plt.yscale('log')
plt.xlabel('$N$')
plt.ylabel('$|\Delta I|$')
#plt.title('오차의 수렴 속도 분석')
plt.legend()
plt.grid(True, which="both", ls="--")

plt.tight_layout()
plt.savefig('mc2.png')
plt.show()

print("\n--- 분석 요약 ---")
print(f"몬테카를로 적분의 오차는 표본 수 N이 증가함에 따라 이론적으로 1/sqrt(N)의 속도로 감소한다. 오른쪽 그래프에서 실제 오차(파란색 점)가 이론적 오차 선(회색 점선)을 따라가는 것을 확인할 수 있다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 문제 설정: 적분 영역은 [0, 1] x [0, 1] x [0, 1] 이다.
def f_3d(x1, x2, x3):
    """적분할 3차원 함수 f(x1, x2, x3) = sin(pi*x1) * sin(pi*x2) * sin(pi*x3)"""
    return np.sin(np.pi * x1) * np.sin(np.pi * x2) * np.sin(np.pi * x3)

# 2. 해석적 해 (참값)
# I_true = (Integral sin(pi*x) dx from 0 to 1)^3
# Integral sin(pi*x) dx from 0 to 1 = 2/pi
I_true_3d = (2.0 / np.pi) ** 3
print(f" 해석적 해 (3차원 참값 I_true): {I_true_3d:.6f}")

# ----------------------------------------------------

##  몬테카를로 적분 함수 정의 (3D)
def monte_carlo_integration_3d(N):
    """
    3차원 몬테카를로 적분 근사 함수
    :param N: 샘플(표본) 개수
    :return: 근사 적분값 I_N, 함수값 배열
    """
    # 1. 적분 영역 [0, 1]^3에서 N개의 난수 샘플 생성
    # x1_i, x2_i, x3_i는 0과 1 사이의 균일 분포를 따른다.
    x1 = np.random.rand(N)
    x2 = np.random.rand(N)
    x3 = np.random.rand(N)

    # 2. 함수값 계산
    f_values = f_3d(x1, x2, x3)

    # 3. 몬테카를로 적분 근사: I_N = (V/N) * sum(f(x_i))
    # 여기서 영역의 부피 V = 1 * 1 * 1 = 1 이다.
    I_N = np.mean(f_values) # sum(f_values)/N 과 동일
    return I_N, f_values

# ----------------------------------------------------

##  수렴성 분석 및 시각화

# A. 샘플 수 N의 범위를 설정
N_list = np.logspace(1, 8, 20, dtype=int) # 10^1부터 10^7까지 20개의 N 값
estimated_integrals = []
errors = []

print("\n--- 3D MC integration (N) ---")
for N in N_list:
    I_N, f_values = monte_carlo_integration_3d(N)
    estimated_integrals.append(I_N)

    # 오차: |I_N - I_true|
    error = np.abs(I_N - I_true_3d)
    errors.append(error)

    # 각 N에 대한 결과 출력
    print(f"N = {N:9d}, MC 값 = {I_N:.6f}, 오차 = {error:.6e}")

# B. 표준 오차 (Standard Error) 분석
# 몬테카를로 적분의 오차는 1/sqrt(N)에 비례한다.
# 이론적 표준 편차 상수 C를 계산하여 비교선을 그린다.
# Integral(f^2) = Integral(sin^2(pi*x1) * sin^2(pi*x2) * sin^2(pi*x3)) dV
# Integral(sin^2(pi*x)) dx from 0 to 1 = 1/2
# Integral(f^2) = 1/2 * 1/2 * 1/2 = 0.125
variance_f = 0.125 - I_true_3d**2
C = np.sqrt(variance_f) # 이론적 표준 편차 상수 (SE = C/sqrt(N))
theoretical_errors = C / np.sqrt(N_list)

# ----------------------------------------------------

## 결과 시각화
plt.figure(figsize=(12, 5))

### 1. 적분값 수렴 그래프 (왼쪽)
plt.subplot(1, 2, 1)
plt.plot(N_list, estimated_integrals, 'o-', label='MC $I_N$', markersize=4)
plt.axhline(I_true_3d, color='r', linestyle='--', label=f'$I_{{true}}$ = {I_true_3d:.6f}')
plt.xscale('log')
plt.xlabel('$N$')
plt.ylabel('$I_N$')
#plt.title('3차원 몬테카를로 적분값의 수렴성')
plt.legend()
plt.grid(True, which="both", ls="--")

### 2. 오차 수렴 그래프 (오른쪽)
plt.subplot(1, 2, 2)
plt.loglog(N_list, errors, 'o', label='$|I_N - I_{true}|$', markersize=4)
plt.loglog(N_list, theoretical_errors, '--', color='gray', label=r'$\propto 1/\sqrt{N}$')
plt.loglog(N_list, 1/N_list**(1/3), ':', color='purple', label=r'reference line $\propto 1/N^{1/3}$ (lattice)') # 3차원 격자 기반 방법과의 비교
plt.xscale('log')
plt.yscale('log')
plt.xlabel('$N$')
plt.ylabel('$|\Delta I|$')
#plt.title('3차원 오차의 수렴 속도 분석')
plt.legend()
plt.grid(True, which="both", ls="--")

plt.tight_layout()
plt.savefig('mc3.png')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import dct

# 1. 룽게 함수 정의
def runge_function(x):
    """룽게 함수 f(x) = 1 / (1 + 25x^2)"""
    return 1.0 / (1.0 + 25.0 * x**2)

# 2. Chebyshev 계수 계산 함수 (DCT 이용)
def compute_chebyshev_coeffs(f, N):
    """
    N차 보간 다항식의 Chebyshev 계수 (c_k)를 계산한다. 
    (N+1개의 Chebyshev 마디 이용)
    """
    # N+1 개의 Chebyshev 마디 생성 (구간 [-1, 1])
    # x_j = cos((2j + 1) * pi / (2(N+1))) for j = 0 to N
    j = np.arange(N + 1)
    # Note: Scipy의 dct는 첫 번째 인덱스(j=0)를 cos((j) * pi / N)으로 정의하므로,
    # 여기서는 고유한 Chebyshev 마디 정의를 사용해야 한다.
    # 실제로 Chebyshev 마디에서 함수 값을 구하고 DCT-I (또는 DCT-III)와 유사한 형태를 사용한다.
    
    # Chebyshev-Gauss-Lobatto 마디 (경계를 포함)를 사용하면 DCT-I을 바로 사용할 수 있으나,
    # 여기서는 표준 Chebyshev 마디 (근)를 사용하고 명시적으로 계수를 계산한다.
    
    # 보간점 (Chebyshev 마디)
    x_nodes = np.cos( (np.pi * (j + 0.5)) / (N + 1) )
    y_nodes = f(x_nodes)
    
    # 계수 c_k 계산
    coeffs = np.zeros(N + 1)
    for k in range(N + 1):
        # 직교성을 이용하여 계수 c_k 계산
        c_k = (2.0 / (N + 1)) * np.sum(y_nodes * np.cos(k * np.pi * (j + 0.5) / (N + 1)))
        
        # c_0는 다른 계수의 절반
        if k == 0:
            c_k /= 2.0
            
        coeffs[k] = c_k
        
    return coeffs

# 3. Chebyshev 다항식 평가 함수
def chebyshev_series_eval(x, coeffs):
    """
    Chebyshev 계수(c_k)를 이용하여 x에서의 다항식 값을 계산한다.
    Clenshaw 알고리즘을 사용하면 효율적이지만, 여기서는 T_k(x)를 직접 계산한다.
    """
    N = len(coeffs) - 1
    T = np.zeros((N + 1, len(x))) # T_k(x) 값을 저장할 배열
    
    # T_0(x) = 1
    T[0, :] = 1.0
    
    if N >= 1:
        # T_1(x) = x
        T[1, :] = x
        
        # 재귀 관계 T_{k+1}(x) = 2x * T_k(x) - T_{k-1}(x) 이용
        for k in range(1, N):
            T[k + 1, :] = 2.0 * x * T[k, :] - T[k - 1, :]
            
    # P_N(x) = sum(c_k * T_k(x))
    P_N = np.dot(coeffs, T)
    return P_N

# ----------------- 4. 실행 및 시각화 -----------------
N = 35 # 보간 차수 (N+1개의 보간점)

# 플롯을 위한 x 축 생성 (촘촘한 간격)
x_plot = np.linspace(-1, 1, 200)
y_exact = runge_function(x_plot)

# Chebyshev 계수 계산
cheby_coeffs = compute_chebyshev_coeffs(runge_function, N)

# 보간 다항식 값 계산
y_interp_cheby = chebyshev_series_eval(x_plot, cheby_coeffs)

# 보간에 사용된 Chebyshev 마디 (Lagrange 방식과 동일)
j = np.arange(N + 1)
x_nodes = np.cos( (np.pi * (j + 0.5)) / (N + 1) )
y_nodes = runge_function(x_nodes)


# 시각화
plt.figure(figsize=(8, 6))
plt.plot(x_plot, y_exact, 'k-', linewidth=2, label='Exact Runge function')
plt.plot(x_nodes, y_nodes, 'o', color='blue', markersize=5, label=f'Chebyshev nodes (N+1={N+1})')
plt.plot(x_plot, y_interp_cheby, 'b--', linewidth=1.5, label=f'Chebyshev series interp. (N={N})')

plt.title(f'Chebyshev series interpolation of Runge function (N={N})')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.ylim(-0.2, 1.2) # y축 범위를 조정하여 결과를 명확히 표시
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def apply_fornberg_diff_general(y, x, order, num_points=5):
    """
    일반화된 Fornberg 미분 함수
    Parameters:
    - y: 미분할 데이터 배열 (값)
    - x: 데이터의 위치 배열 (좌표)
    - order: 미분 차수 (1=속도, 2=가속도/곡률, 3=Jerk/Shear)
    - num_points: 참조할 격자 점의 개수 (홀수 권장: 3, 5, 7, 9...)
      -> 점이 많을수록 이론적 정밀도는 높아지지만(O(h^(N-1))), 계산 비용이 늘어남.
    """
    n = len(x)
    if n < num_points:
        raise ValueError(f"데이터 개수({n})가 참조할 점의 개수({num_points})보다 적습니다.")
    dy = np.zeros_like(y)
    # 윈도우의 '반지름' (중심에서 한쪽으로 몇 칸인지)
    radius = num_points // 2
    for i in range(n):
        # 1. 이상적인 윈도우 설정 (중심 i를 기준으로 대칭)
        # Python slice 문법상 end는 포함되지 않으므로 +1 필요
        # 예: num_points=5 (radius=2) -> start=i-2, end=i+3 (총 5개)
        start = i - radius
        end = start + num_points
        # 2. 경계 처리 (윈도우가 배열 범위를 벗어나지 않게 클램핑)
        # (1) 왼쪽 경계: start가 0보다 작으면 0으로 고정하고, 윈도우 크기만큼 end 설정
        if start < 0:
            start = 0
            end = num_points
        # (2) 오른쪽 경계: end가 n보다 크면 n으로 고정하고, 윈도우 크기만큼 start 뒤로 밀기
        elif end > n:
            end = n
            start = n - num_points
        # 3. 스텐실 추출
        x_stencil = x[start:end]
        y_stencil = y[start:end]
        # 4. Fornberg 가중치 계산
        # x[i] 위치에서 미분하기 위한 가중치를 구함
        weights = fornberg_weights(x[i], x_stencil, order)
        # 5. 내적 (Dot Product) -> 미분값
        dy[i] = np.dot(weights, y_stencil)
    return dy


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def sieve_of_eratosthenes(limit):
    """
    에라토스테네스의 체를 사용하여 limit까지의 소수 여부를 담은 불리언 배열을 반환한다.
    """
    is_prime = np.ones(limit + 1, dtype=bool)
    is_prime[0] = is_prime[1] = False
    for p in range(2, int(np.sqrt(limit)) + 1):
        if is_prime[p]:
            is_prime[p*p : limit+1 : p] = False
    return is_prime

def generate_spiral_coords(max_n):
    """
    1부터 max_n까지 나선형으로 배치될 때의 (x, y) 좌표를 생성한다.
    중심(1)의 좌표는 (0, 0)이다.
    """
    coords = np.zeros((max_n + 1, 2), dtype=int)
    
    # 시작점 (숫자 1)
    x, y = 0, 0
    coords[1] = [x, y]

    # 이동 방향: 오른쪽(1,0) -> 위(0,1) -> 왼쪽(-1,0) -> 아래(0,-1) 순서
    # (dx, dy)를 시계 반대 방향으로 90도 회전시키는 로직: (dx, dy) -> (-dy, dx)
    dx, dy = 1, 0 
    
    step_size = 1 # 현재 방향으로 이동할 걸음 수
    steps_taken = 0 # 현재 방향으로 이동한 걸음 수
    turns = 0 # 방향 전환 횟수

    for n in range(2, max_n + 1):
        x += dx
        y += dy
        coords[n] = [x, y]
        steps_taken += 1

        # 현재 방향으로 정해진 걸음만큼 이동했으면 방향 전환
        if steps_taken == step_size:
            # 방향 회전 (시계 반대 방향)
            dx, dy = -dy, dx
            steps_taken = 0
            turns += 1
            
            # 두 번 방향을 바꿀 때마다 이동할 걸음 수가 1씩 증가함
            # 예: R(1), U(1), L(2), D(2), R(3), U(3)...
            if turns % 2 == 0:
                step_size += 1
                
    return coords

def plot_ulam_spiral(side_length):
    """
    주어진 한 변의 길이를 가지는 정사각형 격자에 울람 나선을 그린다.
    """
    max_num = side_length * side_length
    print(f"Calculating primes up to {max_num}...")
    
    # 1. 소수 판별
    is_prime = sieve_of_eratosthenes(max_num)
    
    print("Generating spiral coordinates...")
    # 2. 나선 좌표 생성
    coords = generate_spiral_coords(max_num)
    
    # 소수와 합성수의 좌표 분리
    prime_x, prime_y = [], []
    comp_x, comp_y = [], []
    
    for n in range(1, max_num + 1):
        cx, cy = coords[n]
        if is_prime[n]:
            prime_x.append(cx)
            prime_y.append(cy)
        else:
            comp_x.append(cx)
            comp_y.append(cy)
            
    print("Plotting...")
    # 3. 시각화
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # 배경 스타일 설정 (검은색 배경에 흰색 점으로 하면 더 극적임)
    plt.style.use('dark_background') 
    fig.patch.set_facecolor('black')
    ax.set_facecolor('black')

    # 합성수 그리기 (나선 구조를 보여주기 위해 희미하게 표시, 원하지 않으면 주석 처리)
    ax.scatter(comp_x, comp_y, s=1, c='gray', alpha=0.2, marker='.', label='Composite')
    
    # 소수 그리기 (강조)
    ax.scatter(prime_x, prime_y, s=15, c='white', marker='o', label='Prime')
    
    # 중심점(1) 표시
    ax.scatter(0, 0, s=50, c='red', marker='x', label='Center (1)')

    # 그래프 꾸미기
    ax.set_aspect('equal') # 정사각형 비율 유지
    ax.axis('off') # 축 숨기기
    ax.set_title(f"Ulam Spiral (Max Number: {max_num})", color='white', fontsize=15)
    
    # 범례 표시 (필요한 경우 주석 해제)
    # ax.legend(loc='upper right')

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    # 정사각형 한 변의 길이 설정 (숫자가 클수록 계산 시간이 걸림)
    # 200 정도면 40,000까지의 수를 표현하며 패턴을 보기에 적당하다.
    SIDE_LENGTH = 200 
    plot_ulam_spiral(SIDE_LENGTH)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def sieve_of_eratosthenes(limit):
    """ 에라토스테네스의 체: 소수 판별 """
    is_prime = np.ones(limit + 1, dtype=bool)
    is_prime[0] = is_prime[1] = False
    for p in range(2, int(np.sqrt(limit)) + 1):
        if is_prime[p]:
            is_prime[p*p : limit+1 : p] = False
    return is_prime

def generate_spiral_coords(max_n):
    """ 나선형 좌표 생성 로직 """
    coords = np.zeros((max_n + 1, 2), dtype=int)
    x, y = 0, 0
    coords[1] = [x, y]

    dx, dy = 1, 0 
    step_size = 1
    steps_taken = 0
    turns = 0

    for n in range(2, max_n + 1):
        x += dx
        y += dy
        coords[n] = [x, y]
        steps_taken += 1

        if steps_taken == step_size:
            dx, dy = -dy, dx # 방향 회전
            steps_taken = 0
            turns += 1
            if turns % 2 == 0:
                step_size += 1
    return coords

def plot_ulam_spiral_white_bg(side_length):
    max_num = side_length * side_length
    print(f"Calculating primes up to {max_num}...")
    is_prime = sieve_of_eratosthenes(max_num)
    
    print("Generating spiral coordinates...")
    coords = generate_spiral_coords(max_num)
    
    prime_x, prime_y = [], []
    comp_x, comp_y = [], []
    
    for n in range(1, max_num + 1):
        cx, cy = coords[n]
        if is_prime[n]:
            prime_x.append(cx)
            prime_y.append(cy)
        else:
            comp_x.append(cx)
            comp_y.append(cy)
            
    print("Plotting...")
    # 기본 스타일 사용 (흰색 배경)
    plt.style.use('default')
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # 1. 합성수 그리기 (배경 패턴)
    # 흰 배경에서는 회색 점이 너무 진하면 지저분해 보일 수 있어 alpha를 낮춤
    ax.scatter(comp_x, comp_y, s=1, c='gray', alpha=0.15, marker='.', zorder=1)
    
    # 2. 소수 그리기 (강조)
    # 색상을 푸른색 계열인 'royalblue'로 설정. s(크기)를 약간 키워 강조.
    ax.scatter(prime_x, prime_y, s=18, c='royalblue', marker='o', zorder=2, label='Prime')
    
    # 3. 중심점(1) 표시
    ax.scatter(0, 0, s=60, c='red', marker='x', zorder=3, label='Center (1)')

    # 그래프 꾸미기
    ax.set_aspect('equal')
    ax.axis('off') # 깔끔하게 축 숨기기
    # 제목 색상은 기본 검정으로 둠
    ax.set_title(f"Ulam Spiral (Max Number: {max_num})", fontsize=15, pad=20)
    
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    # 한 변의 길이 설정
    SIDE_LENGTH = 300 
    plot_ulam_spiral_white_bg(SIDE_LENGTH)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_roots_of_unity():
    # 1. 해 구하기 (기하학적 방법: 회전)
    # 0도, 120도, 240도 (라디안 변환)
    angles = np.array([0, 2*np.pi/3, 4*np.pi/3])
    roots = np.exp(1j * angles)

    # 2. 그래프 설정
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # 3. 단위 원 그리기 (|z|=1)
    circle = plt.Circle((0, 0), 1, color='lightgray', fill=False, linestyle='--', linewidth=1.5, label='Unit Circle (|z|=1)')
    ax.add_artist(circle)
    
    # 4. 정삼각형 그리기 (해들을 연결)
    # 닫힌 도형을 만들기 위해 첫 번째 점을 리스트 끝에 추가
    triangle_points = np.append(roots, roots[0]) 
    ax.plot(triangle_points.real, triangle_points.imag, 'b-', alpha=0.3, label='Equilateral Triangle')

    # 5. 해(Roots) 점 찍기
    # 1 (실수)
    ax.scatter(roots[0].real, roots[0].imag, color='red', s=150, zorder=5, label='Root: 1')
    # omega, omega^2 (허수)
    ax.scatter(roots[1:].real, roots[1:].imag, color='blue', s=150, zorder=5, label=r'Roots: $\omega, \omega^2$')

    # 6. 중심점(Origin) 표현 - 요청하신 부분
    ax.scatter(0, 0, color='black', marker='x', s=200, linewidth=2, zorder=5, label='Center (0,0)')

    # 7. 주석 달기 (좌표 표시)
    offset = 0.1
    for i, root in enumerate(roots):
        label_text = ""
        if i == 0: label_text = "1"
        elif i == 1: label_text = r"$\omega = e^{i2\pi/3}$"
        else: label_text = r"$\omega^2 = e^{i4\pi/3}$"
        
        # 텍스트 위치 약간 조정
        txt_x = root.real + offset if root.real > 0 else root.real - offset*2
        txt_y = root.imag + offset if root.imag > 0 else root.imag - offset
        
        ax.text(txt_x, txt_y, label_text, fontsize=14, fontweight='bold')

    # 8. 축 및 스타일 설정
    ax.axhline(0, color='black', linewidth=1)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_title(r"Roots of $x^3 = 1$ in Complex plane", fontsize=16)
    ax.set_xlabel("Real part")
    ax.set_ylabel("Imaginary part")
    ax.legend(loc='lower right')

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    plot_roots_of_unity()

In [ ]:
import sympy.combinatorics.named_groups as ng

def check_solvability_simulation(group_name, group_obj):
    print(f"--- {group_name} 시뮬레이션 시작 ---")
    
    current_group = group_obj
    step = 0
    
    print(f"Step {step}: 크기(Order) = {current_group.order()}")
    
    # 그룹의 크기가 1이 되거나, 더 이상 줄어들지 않을 때까지 반복
    while current_group.order() > 1:
        # 교환자 부분군(Derived Subgroup) 계산: [G, G]
        next_group = current_group.derived_subgroup()
        
        # 만약 그룹이 더 이상 줄어들지 않으면 (핵이 남음)
        if next_group.order() == current_group.order():
            print(f"Step {step+1}: 크기(Order) = {next_group.order()} -> [붕괴 멈춤!]")
            print(f"결과: {group_name}은 '가해군'이 아니다. (일반 공식 없음)\n")
            return
            
        current_group = next_group
        step += 1
        print(f"Step {step}: 크기(Order) = {current_group.order()}")
        
    print(f"결과: {group_name}은 1까지 붕괴되었다. -> '가해군'이다.(일반 공식 존재)\n")

# 1. S3 (3차 방정식 대칭군)
s3 = ng.SymmetricGroup(3)
check_solvability_simulation("S3", s3)

# 2. S4 (4차 방정식 대칭군)
s4 = ng.SymmetricGroup(4)
check_solvability_simulation("S4", s4)

# 3. S5 (5차 방정식 대칭군)
s5 = ng.SymmetricGroup(5)
check_solvability_simulation("S5", s5)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def durand_kerner(coeffs, max_iter=100, tol=1e-12):
    """
    Durand-Kerner 방법을 사용하여 다항식의 모든 근을 구한다.
    
    Args:
        coeffs: 다항식의 계수 리스트 (내림차순, 예: [1, 0, 2] -> x^2 + 2)
        max_iter: 최대 반복 횟수
        tol: 수렴 허용 오차
        
    Returns:
        roots: 최종 수렴된 근들의 배열
        history: 시각화를 위한 반복 과정 기록 (num_iter x num_roots)
    """
    # 1. 최고차항 계수로 정규화 (모닉 다항식으로 변환)
    p = np.array(coeffs, dtype=complex)
    if p[0] != 1:
        p = p / p[0]
    
    degree = len(p) - 1
    
    # 2. 초기값 설정: 복소평면의 단위 원 위에 고르게 배치
    # (서로 겹치지 않게 하기 위함, 0.4 + 0.9j 같은 오프셋을 주기도 함)
    radius = 0.4 + 0.9j 
    roots = np.array([radius ** (i+1) for i in range(degree)], dtype=complex)
    
    history = [roots.copy()]

    # 3. 반복 수행
    for iter_count in range(max_iter):
        current_roots = roots.copy()
        
        # 각 근에 대해 동시에 업데이트 수행
        for i in range(degree):
            numerator = np.polyval(p, current_roots[i]) # P(z_i)
            
            # 분모: product(z_i - z_j) for j != i
            denominator = 1.0
            for j in range(degree):
                if i != j:
                    denominator *= (current_roots[i] - current_roots[j])
            
            # DK 업데이트 공식 적용
            roots[i] = current_roots[i] - numerator / denominator
            
        history.append(roots.copy())
        
        # 수렴 판정 (변화량이 tol보다 작으면 종료)
        if np.max(np.abs(roots - current_roots)) < tol:
            print(f"Converged in {iter_count + 1} iterations.")
            break
            
    return roots, np.array(history)

# --- 실행 예제 ---

# 다항식: (x-1)(x+1)(x-3)(x^2 + 4) = x^5 - 3x^4 + 3x^3 - 9x^2 - 4x + 12 = 0
# 근: 1, -1, 3, 2i, -2i
coeffs = [1, -3, 3, -9, -4, 12]

final_roots, history = durand_kerner(coeffs)

# 결과 출력
print("\n--- Calculated Roots ---")
for i, root in enumerate(final_roots):
    print(f"Root {i+1}: {root.real:.5f} + {root.imag:.5f}j")

# --- 시각화 (근의 이동 경로) ---
plt.figure(figsize=(8, 8))
colors = plt.cm.rainbow(np.linspace(0, 1, len(final_roots)))

# 1. 이동 경로 그리기
for i in range(len(final_roots)):
    path = history[:, i]
    plt.plot(path.real, path.imag, '.-', color=colors[i], alpha=0.5, label=f'Root {i+1} Trajectory')
    # 시작점 표시
    plt.plot(path[0].real, path[0].imag, 'kx', markersize=8) 
    # 끝점(최종 근) 표시
    plt.plot(path[-1].real, path[-1].imag, 'ko', markersize=8)

# 2. 축 설정
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.grid(True, linestyle='--', alpha=0.7)
plt.title(f"Convergence of Durand-Kerner method(degree {len(coeffs)-1})")
plt.xlabel("Real part", fontsize=18)
plt.ylabel("Imaginary part", fontsize=18)
plt.legend()
plt.savefig('dk5.png')
plt.show()

In [ ]:
import numpy as np

def aberth_ehrlich(coeffs, max_iter=100, tol=1e-12):
    """
    Aberth-Ehrlich 방법을 사용하여 다항식의 근을 구한다.
    중근이나 근접한 근에 대해 DK 방법보다 더 강건하다.
    """
    # 다항식 P(x)와 도함수 P'(x) 설정
    p = np.poly1d(coeffs)
    dp = p.deriv()
    degree = len(coeffs) - 1
    # 초기값 설정 (복소평면 단위 원 위에 분산)
    # 중근 분리를 위해 미세한 랜덤 노이즈를 추가하기도 함
    radius = 0.5 + 0.8j # 회전 반경
    roots = np.array([radius**i for i in range(degree)], dtype=complex)
    for iteration in range(max_iter):
        current_roots = roots.copy()
        updates = np.zeros_like(roots)
        for j in range(degree):
            z_j = current_roots[j]
            P_val = p(z_j)
            dP_val = dp(z_j)
            # 뉴턴 스텝 (N(z) = P/P')
            if abs(dP_val) < 1e-15: # 도함수가 0에 가까우면(중근 근처) 보호
                newton_step = 0
            else:
                newton_step = P_val / dP_val
            # Aberth 보정항 계산: sum(1 / (z_j - z_k))
            repulsion = 0
            for k in range(degree):
                if k != j:
                    denom = z_j - current_roots[k]
                    # 수치 안정성을 위해 매우 작은 값 처리
                    if abs(denom) < 1e-15: 
                        repulsion += 1e5 # 강하게 밀어냄
                    else:
                        repulsion += 1.0 / denom
            # Aberth 업데이트 공식
            # z_new = z - (P/P') / (1 - (P/P') * repulsion)
            denominator = 1 - newton_step * repulsion
            if abs(denominator) < 1e-15:
                updates[j] = 0
            else:
                updates[j] = newton_step / denominator
        roots = roots - updates
        # 수렴 판정
        if np.max(np.abs(updates)) < tol:
            print(f"Converged in {iteration + 1} iterations.")
            break
    return roots

# --- 실행 예제 ---
# 방정식: (x-1)^3 * (x+2) * (x-3) = 0
# 근: 1 (3중근), -2, 3
# 전개: x^5 - 4x^4 + x^3 + 10x^2 - 4x - 8
coeffs = [1, -4, 1, 10, -4, -8]

print(f"Target Roots: 1 (multiplicity 3), -2, 3")
print("-" * 30)
roots = aberth_ehrlich(coeffs)
print("Calculated Roots:")
# 결과를 보기 좋게 정렬 및 반올림하여 출력
roots_sorted = sorted(roots, key=lambda x: (x.real, x.imag))
for r in roots_sorted:
    print(f"{r.real:.5f} + {r.imag:.5f}j")
# 중근 근처에서는 수렴값이 미세하게 흩어질 수 있으나,
# DK 방법에 비해 훨씬 중심에 가깝게 모인다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def aberth_ehrlich_with_history(coeffs, max_iter=100, tol=1e-12):
    """
    Aberth-Ehrlich 방법을 사용하여 다항식의 근을 구하고,
    그 수렴 과정을 기록하여 반환한다.
    """
    p = np.poly1d(coeffs)
    dp = p.deriv()
    degree = len(coeffs) - 1
    # 초기값 설정: 원점 중심으로 약간 회전된 원 위에 배치
    # 중근 수렴 경로를 더 잘 보기 위해 반지름을 약간 키우고 오프셋을 주었다.
    radius = 3.0 
    offset = 0.1 + 0.1j # 초기 대칭성을 깨기 위한 약간의 오프셋
    roots = np.array([radius * np.exp(2j * np.pi * i / degree) + offset 
                      for i in range(degree)], dtype=complex)
    # 기록용 리스트 초기화
    for i in range(len(roots)):
        roots[i]=roots[i]*1e-2 + (np.random.random()-0.5)+1j*(np.random.random()-0.5)
    history = [roots.copy()]
    print("Iteration starting...")
    for iteration in range(max_iter):
        current_roots = roots.copy()
        updates = np.zeros_like(roots)
        for j in range(degree):
            z_j = current_roots[j]
            P_val = p(z_j)
            dP_val = dp(z_j)
            # 뉴턴 스텝 (N(z) = P/P')
            # 수치 안정성을 위해 분모가 너무 작으면 0으로 처리
            if abs(dP_val) < 1e-15:
                newton_step = 0
            else:
                newton_step = P_val / dP_val
            # Aberth 보정항 계산: sum(1 / (z_j - z_k))
            repulsion = 0
            for k in range(degree):
                if k != j:
                    denom = z_j - current_roots[k]
                    # 근들이 너무 가까워지면 튕겨내는 힘을 제한 (시각화 안정성 위함)
                    if abs(denom) < 1e-10: 
                        repulsion += 1e5 * (1+1j) # 임의의 큰 방향 벡터
                    else:
                        repulsion += 1.0 / denom
            # Aberth 업데이트 공식
            denominator = 1 - newton_step * repulsion
            if abs(denominator) < 1e-15:
                updates[j] = 0 # 분모가 0이면 업데이트 안 함
            else:
                updates[j] = newton_step / denominator
        roots = roots - updates
        # 현재 상태 기록
        history.append(roots.copy())
        # 수렴 판정
        if np.max(np.abs(updates)) < tol:
            print(f"Converged in {iteration + 1} iterations.")
            break
    return roots, np.array(history)

# --- 실행 및 시각화 설정 ---
# 방정식: (x-1)^3 * (x+2) * (x-3) = x^5 - 4x^4 + x^3 + 10x^2 - 4x - 8
coeffs = [1, -4, 1, 10, -4, -8]
# 함수 실행
final_roots, history = aberth_ehrlich_with_history(coeffs)
print(final_roots)
num_roots = len(final_roots)
num_iters = history.shape[0]
# --- 시각화 그리기 ---
plt.figure(figsize=(12, 10))
colors = plt.cm.rainbow(np.linspace(0, 1, num_roots))
# 실제 정답 위치 표시 (비교용)
actual_roots = [1, 1, 1, -2, 3]
plt.plot(np.real(actual_roots), np.imag(actual_roots), 'k*', markersize=15, label='Actual Roots', zorder=10)
for i in range(num_roots):
    # 각 근의 이동 경로 추출
    path = history[:, i]
    # 경로 선 그리기
    plt.plot(path.real, path.imag, '-', color=colors[i], alpha=0.6, linewidth=1.5)
    # 매 스텝마다 점 찍기 (경로 확인용)
    plt.plot(path.real, path.imag, '.', color=colors[i], alpha=0.3, markersize=5)
    # 시작점 (빨간색 X)
    plt.plot(path[0].real, path[0].imag, 'rx', markersize=12, markeredgewidth=2, label='Start' if i == 0 else "")
    # 끝점 (계산된 근, 해당 색상 원)
    plt.plot(path[-1].real, path[-1].imag, 'o', color=colors[i], markersize=10, markeredgecolor='k', label=f'Calc Root {i+1}')
# 그래프 꾸미기
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.grid(True, linestyle='--', alpha=0.7)
plt.title(f"Aberth-Ehrlich method trajectory\nEquation with triple root at x=1", fontsize=14)
plt.xlabel("Real part", fontsize=12)
plt.ylabel("Imaginary part", fontsize=12)
# 축 범위 설정 (근들이 잘 보이도록 조정)
plt.xlim(-4, 5)
plt.ylim(-4, 4)
# 범례 표시 (너무 많으면 가림)
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), loc='best')
plt.tight_layout()
plt.savefig('ae5.png')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 이전에 정의한 함수 재사용
def aberth_ehrlich_with_history(coeffs, max_iter=100, tol=1e-15): # tol을 더 낮춰 정밀도 확인
    p = np.poly1d(coeffs)
    dp = p.deriv()
    degree = len(coeffs) - 1
    # 초기값: 원점 주변의 작은 원 (정답인 단위 원과 겹치지 않게)
    radius = 0.5 
    offset = 0.0 + 0.0j 
    roots = np.array([radius * np.exp(2j * np.pi * i / degree) + offset 
                      for i in range(degree)], dtype=complex)
    history = [roots.copy()]
    for iteration in range(max_iter):
        current_roots = roots.copy()
        updates = np.zeros_like(roots)
        for j in range(degree):
            z_j = current_roots[j]
            P_val = p(z_j)
            dP_val = dp(z_j)
            if abs(dP_val) < 1e-15: newton_step = 0
            else: newton_step = P_val / dP_val
            repulsion = 0
            for k in range(degree):
                if k != j:
                    denom = z_j - current_roots[k]
                    if abs(denom) < 1e-15: repulsion += 1e5
                    else: repulsion += 1.0 / denom
            denominator = 1 - newton_step * repulsion
            if abs(denominator) < 1e-15: updates[j] = 0
            else: updates[j] = newton_step / denominator
        roots = roots - updates
        history.append(roots.copy())
        # 수렴 판정
        if np.max(np.abs(updates)) < tol:
            print(f"Converged in {iteration + 1} iterations. (Tolerance: {tol})")
            break
    return roots, np.array(history)

# --- 실행: x^5 - 1 = 0 ---
# 해: 1, 그리고 72도씩 회전한 복소수들 (서로 다름)
coeffs_simple = [1, 0, 0, 0, 0, -1] 
#coeffs_simple = [1, -4, 1, 10, -4, -8]
final_roots, history = aberth_ehrlich_with_history(coeffs_simple)
print(final_roots)
# --- 시각화 ---
plt.figure(figsize=(10, 10))
num_roots = len(final_roots)
colors = plt.cm.hsv(np.linspace(0, 1, num_roots)) # 무지개색
# 단위 원 (정답 위치 가이드)
theta = np.linspace(0, 2*np.pi, 100)
plt.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, label='Unit Circle')
for i in range(num_roots):
    path = history[:, i]
    plt.plot(path.real, path.imag, '.-', color=colors[i], alpha=0.6, linewidth=1.5)
    plt.plot(path[-1].real, path[-1].imag, 'o', color=colors[i], markersize=10, markeredgecolor='k')
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.grid(True, linestyle='--', alpha=0.7)
plt.title(f"Aberth-Ehrlich Method: Distinct Roots (x^5 - 1 = 0)", fontsize=14)
plt.xlim(-5.5, 5.5)
plt.ylim(-5.5, 5.5)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def aberth_ehrlich(coeffs, max_iter=100, tol=1e-12):
    # (이전과 동일한 알고리즘 구현 생략 - 핵심 로직은 동일)
    p = np.poly1d(coeffs)
    dp = p.deriv()
    degree = len(coeffs) - 1
    # 초기값: 1~20을 포함할 수 있는 넉넉한 반경 + 노이즈
    radius = 25.0 
    # 랜덤성을 주어 대칭성 문제 회피
    roots = np.array([radius * np.exp(2j * np.pi * i / degree) + (np.random.rand() + 1j*np.random.rand()) 
                      for i in range(degree)], dtype=complex)
    
    for iteration in range(max_iter):
        current_roots = roots.copy()
        updates = np.zeros_like(roots)
        for j in range(degree):
            z_j = current_roots[j]
            try:
                # poly1d는 고차항 계산 시 오차가 큼
                P_val = p(z_j) 
                dP_val = dp(z_j)
                if abs(dP_val) < 1e-15: newton_step = 0
                else: newton_step = P_val / dP_val
                
                repulsion = 0
                for k in range(degree):
                    if k != j:
                        denom = z_j - current_roots[k]
                        if abs(denom) < 1e-15: repulsion += 1e5
                        else: repulsion += 1.0 / denom
                
                denominator = 1 - newton_step * repulsion
                if abs(denominator) < 1e-15: updates[j] = 0
                else: updates[j] = newton_step / denominator
            except OverflowError:
                # 너무 큰 값으로 인해 계산 불가
                updates[j] = 0 

        roots = roots - updates
        if np.max(np.abs(updates)) < tol:
            break
    return roots

# --- 윌킨슨 다항식 생성 (x-1)...(x-20) ---
roots_true = np.arange(1, 21)
coeffs_wilkinson = np.poly(roots_true) # 계수 전개

print("Finding roots for Wilkinson's Polynomial (Degree 20)...")
roots_calc = aberth_ehrlich(coeffs_wilkinson)
print(roots_calc)
# --- 결과 시각화 ---
plt.figure(figsize=(10, 6))
plt.plot(roots_true.real, roots_true.imag, 'k*', markersize=10, label='True Roots (1~20)')
plt.plot(roots_calc.real, roots_calc.imag, 'ro', alpha=0.6, label='Calculated Roots (Aberth)')

plt.title("Failure of Aberth Method on Wilkinson Polynomial (Float64 Limit)")
plt.xlabel("Real Part")
plt.ylabel("Imaginary Part")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 윌킨슨 다항식 계수 생성 (근: 1 ~ 20)
true_roots = np.arange(1, 21)
coeffs = np.poly(true_roots)

print("NumPy(동반행렬 방식)로 윌킨슨 다항식 풀이 중...")
# np.roots는 내부적으로 동반행렬의 고유값을 구한다.
calc_roots = np.roots(coeffs)
print(calc_roots)
# 2. 결과 시각화
plt.figure(figsize=(10, 6))
plt.plot(true_roots, np.zeros_like(true_roots), 'k*', markersize=10, label='True Roots')
plt.plot(calc_roots.real, calc_roots.imag, 'rx', markersize=8, label='np.roots (Companion Matrix)')

plt.title("Failure of Companion Matrix Method (NumPy) on Wilkinson Polynomial")
plt.xlabel("Real Part")
plt.ylabel("Imaginary Part")
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

In [ ]:
!pip install mpmath

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

def draw_solvability_ladders():
    # 그래프 생성
    G = nx.DiGraph()
    
    # --- 데이터 정의: (단계, 그룹이름, 크기, 설명) ---
    # S2 (2차 방정식) - 성공
    s2_chain = [("S2", 2, "Start"), ("{e}", 1, "Solved!")]
    
    # S3 (3차 방정식) - 성공
    s3_chain = [("S3", 6, "Start"), ("A3", 3, "√"), ("{e}", 1, "∛ (Solved!)")]
    
    # S4 (4차 방정식) - 성공
    s4_chain = [("S4", 24, "Start"), ("A4", 12, "√"), ("V4", 4, "Resolvent Cubic"), 
                ("C2", 2, "√"), ("{e}", 1, "√ (Solved!)")]
    
    # S5 (5차 방정식) - 실패 (핵심!)
    s5_chain = [("S5", 120, "Start"), ("A5", 60, "√"), ("A5 ", 60, "Stuck! (Simple Group)")]

    # --- 그래프에 노드와 엣지 추가 ---
    chains = [s2_chain, s3_chain, s4_chain, s5_chain]
    titles = ["Quadratic (S2)", "Cubic (S3)", "Quartic (S4)", "Quintic (S5)"]
    pos = {}
    
    plt.figure(figsize=(14, 8))
    
    # 각 체인을 별도의 x축 라인에 배치
    for idx, chain in enumerate(chains):
        x_base = idx * 2
        for i, (name, size, label) in enumerate(chain):
            node_id = f"{titles[idx]}_{i}"
            G.add_node(node_id, label=f"{name}\n(Size {size})", subset=idx)
            pos[node_id] = (x_base, 10 - i * 2) # y좌표는 단계별로 내려감
            
            # 엣지 연결 (마지막 노드가 아니면)
            if i < len(chain) - 1:
                next_id = f"{titles[idx]}_{i+1}"
                G.add_edge(node_id, next_id)
                
            # 특수 처리: S5의 마지막 단계 (제자리 걸음 표현)
            if titles[idx] == "Quintic (S5)" and i == len(chain) - 1:
                G.add_edge(node_id, node_id, label="Cannot break")

    # --- 시각화 스타일 설정 ---
    node_colors = []
    for node in G.nodes():
        if "Solved" in node or "{e}" in G.nodes[node]['label']:
            node_colors.append('#4CAF50') # 성공 (초록)
        elif "Stuck" in G.nodes[node]['label'] or "A5 " in node:
            node_colors.append('#F44336') # 실패 (빨강)
        else:
            node_colors.append('#2196F3') # 진행 중 (파랑)

    # 그리기
    labels = nx.get_node_attributes(G, 'label')
    
    # 노드 그리기
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=3000, alpha=0.9, node_shape='o')
    nx.draw_networkx_labels(G, pos, labels, font_size=10, font_color='white', font_weight='bold')
    
    # 엣지 그리기
    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=20, width=2)
    
    # S5의 제자리 걸음(Self loop) 별도 표시
    s5_stuck_pos = pos[f"Quintic (S5)_{len(s5_chain)-1}"]
    plt.text(s5_stuck_pos[0] + 0.6, s5_stuck_pos[1], "↺ Cannot reduce\nfurther!", 
             fontsize=12, color='#F44336', fontweight='bold', ha='left')

    # 타이틀 및 설명
    plt.title("Visual Proof: Why the Quintic Formula Does Not Exist", fontsize=20)
    
    # x축 라벨 달기
    for idx, title in enumerate(titles):
        plt.text(idx * 2, 11, title, fontsize=14, ha='center', fontweight='bold', 
                 bbox=dict(facecolor='white', alpha=0.8, boxstyle='round'))

    plt.ylim(3, 12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

draw_solvability_ladders()

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

def draw_solvability_ladders():
    # 그래프 생성
    G = nx.DiGraph()
    
    # --- 데이터 정의: (단계, 그룹이름, 크기, 설명) ---
    # S2 (2차 방정식) - 성공
    s2_chain = [("S2", 2, "Start"), ("{e}", 1, "Solved!")]
    
    # S3 (3차 방정식) - 성공
    s3_chain = [("S3", 6, "Start"), ("A3", 3, "√"), ("{e}", 1, "∛ (Solved!)")]
    
    # S4 (4차 방정식) - 성공
    s4_chain = [("S4", 24, "Start"), ("A4", 12, "√"), ("V4", 4, "Resolvent Cubic"), 
                ("C2", 2, "√"), ("{e}", 1, "√ (Solved!)")]
    
    # S5 (5차 방정식) - 실패 (핵심!)
    s5_chain = [("S5", 120, "Start"), ("A5", 60, "√"), ("A5 ", 60, "Stuck! (Simple Group)")]

    # --- 그래프에 노드와 엣지 추가 ---
    chains = [s2_chain, s3_chain, s4_chain, s5_chain]
    titles = ["Quadratic (S2)", "Cubic (S3)", "Quartic (S4)", "Quintic (S5)"]
    pos = {}
    
    plt.figure(figsize=(14, 8))
    
    # 각 체인을 별도의 x축 라인에 배치
    for idx, chain in enumerate(chains):
        x_base = idx * 2
        for i, (name, size, label) in enumerate(chain):
            node_id = f"{titles[idx]}_{i}"
            G.add_node(node_id, label=f"{name}\n(Size {size})", subset=idx)
            pos[node_id] = (x_base, 10 - i * 2) # y좌표는 단계별로 내려감
            
            # 엣지 연결 (마지막 노드가 아니면)
            if i < len(chain) - 1:
                next_id = f"{titles[idx]}_{i+1}"
                G.add_edge(node_id, next_id)
                
            # 특수 처리: S5의 마지막 단계 (제자리 걸음 표현)
            if titles[idx] == "Quintic (S5)" and i == len(chain) - 1:
                G.add_edge(node_id, node_id, label="Cannot break")

    # --- 시각화 스타일 설정 ---
    node_colors = []
    for node in G.nodes():
        if "Solved" in node or "{e}" in G.nodes[node]['label']:
            node_colors.append('#4CAF50') # 성공 (초록)
        elif "Stuck" in G.nodes[node]['label'] or "A5 " in node:
            node_colors.append('#F44336') # 실패 (빨강)
        else:
            node_colors.append('#2196F3') # 진행 중 (파랑)

    # 그리기
    labels = nx.get_node_attributes(G, 'label')
    
    # 노드 그리기
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=3000, alpha=0.9, node_shape='o')
    nx.draw_networkx_labels(G, pos, labels, font_size=10, font_color='white', font_weight='bold')
    
    # 엣지 그리기
    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=20, width=2)
    
    # S5의 제자리 걸음(Self loop) 별도 표시
    s5_stuck_pos = pos[f"Quintic (S5)_{len(s5_chain)-1}"]
    plt.text(s5_stuck_pos[0] + 0.6, s5_stuck_pos[1], "↺ Cannot reduce\nfurther!", 
             fontsize=12, color='#F44336', fontweight='bold', ha='left')

    # 타이틀 및 설명
    plt.title("Visual Proof: Why the Quintic Formula Does Not Exist", fontsize=20)
    
    # x축 라벨 달기
    for idx, title in enumerate(titles):
        plt.text(idx * 2, 11, title, fontsize=14, ha='center', fontweight='bold', 
                 bbox=dict(facecolor='white', alpha=0.8, boxstyle='round'))

    plt.ylim(3, 12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

draw_solvability_ladders()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from sympy.combinatorics import Permutation

def draw_A5_cayley_graph():
    # 1. A5 생성 (생성자: 3-사이클과 2-2 호환)
    # A5는 (1 2 3)과 (1 2)(3 4) 두 개로 생성 가능하다.
    p1 = Permutation(0, 1, 2)       # (1 2 3)
    p2 = Permutation(0, 1)(2, 3)    # (1 2)(3 4)
    generators = [p1, p2]
    
    # 2. BFS로 A5의 모든 원소(60개) 탐색 및 그래프 연결
    G = nx.Graph()
    start_node = Permutation(4) # 항등원 (Identity)
    G.add_node(start_node)
    
    queue = [start_node]
    visited = {start_node}
    
    while queue:
        current = queue.pop(0)
        
        for gen in generators:
            next_perm = current * gen
            
            # 엣지 추가 (상호 연결)
            if not G.has_edge(current, next_perm):
                G.add_edge(current, next_perm)
            
            if next_perm not in visited:
                visited.add(next_perm)
                queue.append(next_perm)
                G.add_node(next_perm)
                
    print(f"Generated A5 Cayley Graph with {G.number_of_nodes()} nodes (Should be 60).")

    # 3. 시각화 (Kamada-Kawai layout이 대칭성을 잘 보여줌)
    plt.figure(figsize=(10, 10))
    pos = nx.kamada_kawai_layout(G)
    
    # 노드 그리기
    nx.draw_networkx_nodes(G, pos, node_size=100, node_color='#E91E63', alpha=0.8)
    # 엣지 그리기
    nx.draw_networkx_edges(G, pos, alpha=0.3, edge_color='gray')
    
    plt.title("Cayley Graph of A5 (The 'Indestructible' Simple Group)", fontsize=16)
    plt.axis('off')
    
    # 설명 추가
    plt.text(0, -1.2, 
             "This structure has 60 nodes and is mathematically 'Simple'.\nIt has NO normal subgroups to collapse into.\nThis complexity prevents the Quintic Formula.", 
             ha='center', fontsize=12, bbox=dict(facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

draw_A5_cayley_graph()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

def plot_fundamental_theorem_visualization():
    # 1. 복소 평면 그리드 생성
    # x축(실수부)과 y축(허수부)의 범위를 설정한다.
    r = 2  # 범위 (-2 ~ 2)
    x = np.linspace(-r, r, 400)
    y = np.linspace(-r, r, 400)
    X, Y = np.meshgrid(x, y)
    # 복소수 Z 생성 (Z = x + iy)
    Z = X + 1j * Y
    # 2. 다항식 정의: f(z) = z^3 - 1
    # 이 방정식은 3개의 근을 가진다: 1, -0.5+0.866i, -0.5-0.866i
    F = Z**3 - 1
    # 3. 높이 계산 (함숫값의 크기 |f(z)|)
    # 크기가 0인 지점이 바로 근이다.
    AbsF = np.abs(F)
    # 시각화를 위해 높이를 제한한다 (너무 높은 값은 잘라냄)
    # 이렇게 해야 바닥(0)에 닿는 부분이 더 잘 보인다.
    AbsF_clipped = np.clip(AbsF, 0, 5)
    # 4. 3D 플롯 그리기
    fig = plt.figure(figsize=(12, 6))
    # --- 첫 번째 그림: 3D 표면 플롯 ---
    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    surf = ax1.plot_surface(X, Y, AbsF_clipped, cmap=cm.viridis, 
                           linewidth=0, antialiased=False, alpha=0.9)
    # 바닥면(z=0) 표시 (근이 닿아야 하는 곳)
    ax1.contourf(X, Y, AbsF_clipped, zdir='z', offset=0, cmap=cm.viridis, alpha=0.3)
    ax1.set_title("3D Landscape of |f(z)| = |z^3 - 1|")
    ax1.set_xlabel("Real part")
    ax1.set_ylabel("Imaginary part")
    ax1.set_zlabel("|f(z)|")
    ax1.view_init(elev=30, azim=45) # 보는 각도 조절
    # --- 두 번째 그림: 2D 등고선 플롯 (위에서 본 모습) ---
    ax2 = fig.add_subplot(1, 2, 2)
    # 높이가 0에 가까운 곳을 진한 색으로 표현
    contour = ax2.contourf(X, Y, AbsF, levels=20, cmap=cm.viridis_r) 
    # 정확히 0에 가까운 지점에 점 찍기 (근사적인 위치)
    # 근의 위치 시각적 강조 (로컬 미니멈)
    roots_x = [1, -0.5, -0.5]
    roots_y = [0, np.sqrt(3)/2, -np.sqrt(3)/2]
    ax2.plot(roots_x, roots_y, 'rx', markersize=10, markeredgewidth=2, label='Roots (zeros)')
    ax2.set_title("2D Contour map (Darker = Closer to 0)")
    ax2.set_xlabel("Real part")
    ax2.set_ylabel("Imaginary part")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal')
    plt.tight_layout()
    plt.show()

# 프로그램 실행
plot_fundamental_theorem_visualization()

In [ ]:
import numpy as np
import scipy.sparse as sp

# 예시: 간단한 1차원 중심 차분 행렬 ([-0.5, 0, 0.5])
def get_1d_diff_matrix(N):
    # 실제로는 Fornberg 알고리즘 등을 사용해 고차 행렬을 생성
    diags = [np.ones(N)*0.5, -np.ones(N)*0.5]
    return sp.diags(diags, [1, -1], shape=(N, N))

# ------------------------------------------------
# 방법 1: 텐서곱 사용 (수학적으로 아름다우나, N이 작을 때만 가능)
# ------------------------------------------------
def method_tensor_product(N):
    Dx = get_1d_diff_matrix(N)
    Iy = sp.eye(N)
    Iz = sp.eye(N)
    
    # x방향 3D 미분 연산자 생성 (Dx ⊗ Iy ⊗ Iz)
    # 순서는 데이터 저장 방식(C-order vs F-order)에 따라 달라질 수 있음
    D_3d_x = sp.kron(sp.kron(Dx, Iy), Iz)
    
    print(f"3D 행렬 크기: {D_3d_x.shape}")
    return D_3d_x

# ------------------------------------------------
# 방법 2: 차원별 적용 (실전용, 메모리 절약)
# ------------------------------------------------
def method_apply_along_axis(data_3d, stencil_weights):
    # data_3d shape: (Nx, Ny, Nz)
    
    # x축 미분 (axis 0)
    # 실제로는 np.convolve나 상관함수 등을 사용하여 고속 처리
    grad_x = np.apply_along_axis(
        lambda m: np.convolve(m, stencil_weights, mode='same'), 
        axis=0, 
        arr=data_3d
    )
    
    # y축, z축도 동일하게 axis만 바꿔서 적용
    return grad_x

if __name__ == "__main__":
    N = 10  # N이 100만 되어도 방법 1은 메모리 터짐
    
    # 방법 1 확인
    try:
        method_tensor_product(N)
        print("텐서곱 행렬 생성 성공")
    except MemoryError:
        print("메모리 부족!")

In [ ]:
import numpy as np
from scipy.ndimage import correlate1d

# 데이터: [0, 10, 20, 30, 40] (기울기가 10인 직선)
data = np.array([0.0, 10.0, 20.0, 30.0, 40.0])

# 가중치: 중심 차분 계수 [-0.5, 0, 0.5]
# 의미: (오른쪽 값 * 0.5) + (내 위치 * 0) + (왼쪽 값 * -0.5)
# 수식적으로 (f(x+h) - f(x-h)) / 2 와 동일 (h=1 가정 시)
weights = np.array([-0.5, 0, 0.5])

result = correlate1d(data, weights, mode='constant', cval=0.0)

print(f"원본: {data}")
print(f"미분: {result}")

In [ ]:
import numpy as np
from scipy.ndimage import correlate1d

# 3x3 행렬
data = np.array([
    [1, 2, 3],
    [1, 2, 3],
    [1, 2, 3]
])
# 특징: 가로(axis=1)로는 값이 증가하지만, 세로(axis=0)로는 값이 변하지 않음.

weights = np.array([-0.5, 0, 0.5]) # 미분 필터

# 1. 가로 방향(axis=1) 미분
grad_x = correlate1d(data, weights, axis=1)

# 2. 세로 방향(axis=0) 미분
grad_y = correlate1d(data, weights, axis=0)

print("--- 가로 방향 미분 (변화 있음) ---")
print(grad_x)
# 결과: [[1, 1, 1], [1, 1, 1], ...] (변화율 1이 감지됨)

print("\n--- 세로 방향 미분 (변화 없음) ---")
print(grad_y)
# 결과: [[0, 0, 0], ...] (변화가 없으므로 0)

In [ ]:
!pip install PyWavelets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pywt
from scipy.linalg import toeplitz

def wavelet_operator_compression_fixed():
    # 1. 문제 설정: 크기 N=256 (2의 거듭제곱 권장)
    N = 256
    
    # 2. 밀집 행렬(Dense Matrix) 생성: 1/x 형태로 감소하는 상호작용
    col = 1.0 / (np.arange(N) + 1)
    A_dense = toeplitz(col)
    
    # 3. 웨이브렛 변환 행렬 구성
    w_name = 'db4'
    
    # [수정된 부분] W의 크기를 N x N으로 고정
    W = np.zeros((N, N))
    
    print(f"Matrix Size: {N}x{N}")
    
    # 기저 벡터들을 하나씩 변환하여 행렬 W를 구성
    for i in range(N):
        delta = np.zeros(N)
        delta[i] = 1
        
        # [핵심 수정] mode='periodization' 사용
        # 이 옵션을 써야 256개 입력 -> 128(cA) + 128(cD) = 256개 출력이 나온다.
        cA, cD = pywt.dwt(delta, w_name, mode='periodization')
        
        # 계수 합치기
        coeffs = np.concatenate([cA, cD])
        
        # 행렬의 i번째 열에 저장
        # 이제 shape이 (256,)으로 딱 맞으므로 에러가 나지 않는다.
        W[:, i] = coeffs
        
    # 4. 행렬 변환: A_wavelet = W * A * W.T (직교 기저 변환)
    # W는 직교 행렬에 가까우므로 역행렬은 전치행렬(W.T)과 비슷하다.
    # 정확한 수치해석을 위해서는 W의 역행렬을 써야 하지만, 
    # periodization 모드에서는 W가 거의 직교성을 가진다.
    A_wavelet = W @ A_dense @ W.T

    # 5. 희소화 (Thresholding) - 작은 값 제거
    # 압축 효과를 극적으로 보기 위해 threshold를 약간 높게 잡음
    threshold = 0.1 
    A_sparse = np.where(np.abs(A_wavelet) > threshold, A_wavelet, 0.0)

    # 0이 아닌 요소 개수(Non-zeros) 카운트
    nnz_dense = np.count_nonzero(A_dense)
    nnz_sparse = np.count_nonzero(A_sparse)
    ratio = (1 - nnz_sparse / nnz_dense) * 100

    print(f"Original Non-zeros: {nnz_dense}")
    print(f"Compressed Non-zeros: {nnz_sparse}")
    print(f"Compression Ratio: {ratio:.2f}% (Deleted {100-ratio:.2f}% of data)")

    # 6. 시각화 (Spy Plot)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # (1) 원본 (Dense)
    axes[0].spy(A_dense, markersize=1, color='black')
    axes[0].set_title(f"1. Original Dense Matrix\n(Full Interaction)")
    
    # (2) 압축본 (Sparse - Wavelet)
    axes[1].spy(A_sparse, markersize=1, color='red')
    axes[1].set_title(f"2. Wavelet Transformed Matrix\n(Finger-like Sparse Pattern)")

    plt.suptitle(f"Operator Compression using '{w_name}' (Periodization Mode)", fontsize=15)
    plt.show()

wavelet_operator_compression_fixed()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pywt
from scipy.linalg import toeplitz

def visualize_bcr_finger_pattern():
    # 1. 설정
    N = 512
    wavelet = 'db4'
    
    # 2. 적분 연산자 생성 (로그 커널: log|x-y|)
    # 원거리에서도 영향력이 천천히 줄어드는 대표적인 커널
    col = np.log(np.arange(N) + 1)
    A_dense = toeplitz(col)
    
    # 3. 2D 웨이브렛 변환 (Standard Form)
    # 행렬 이미지를 직접 2D DWT 하는 것과 수학적으로 동일한 효과
    # 레벨 3까지 분해
    coeffs = pywt.wavedec2(A_dense, wavelet, level=3)
    
    # 4. 시각화를 위해 계수들을 하나의 큰 행렬로 모으기
    # pywt.wavedec2의 결과는 리스트 형태이므로, 이를 이미지 형태로 합침
    arr, slices = pywt.coeffs_to_array(coeffs)
    
    # 5. 희소화 (Thresholding)
    threshold = np.max(np.abs(arr)) * 0.02 # 상위 2% 크기만 남김
    arr_sparse = np.where(np.abs(arr) > threshold, 1, 0) # 0 또는 1로 이진화

    # 6. 그리기
    plt.figure(figsize=(10, 10))
    plt.imshow(arr_sparse, cmap='Greys', interpolation='nearest')
    plt.title(f"BCR Algorithm Finger Pattern (N={N}, {wavelet})\nWhite=0, Black=Non-zero", fontsize=15)
    
    # 구조 설명 추가
    plt.text(N//2, N//2, "Interaction\nBand", ha='center', color='red', fontsize=12, fontweight='bold')
    plt.text(N//4, N//4, "Coarse\nScale", ha='center', color='blue', fontsize=10)
    plt.text(N-50, 50, "Fine\nScale", ha='center', color='green', fontsize=10)
    
    plt.axis('off')
    plt.show()

visualize_bcr_finger_pattern()

In [ ]:
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
from Crypto.Random import get_random_bytes
import base64

# -----------------
# 1. 키 및 IV 생성
# -----------------
# AES-256을 사용하므로 32바이트(256비트) 키가 필요하다.
key = get_random_bytes(32) 
# CBC 모드를 사용하므로 16바이트(128비트) IV가 필요하다.
# IV는 매번 다르게 생성되어야 암호화의 보안성이 높아진다.
iv = get_random_bytes(16) 

def encrypt(data, key, iv):
    """주어진 데이터를 AES-256-CBC 모드로 암호화한다."""
    
    # 텍스트 데이터를 바이트로 변환
    data_bytes = data.encode('utf-8')
    
    # 블록 크기(16바이트)에 맞게 데이터 패딩
    padded_data = pad(data_bytes, AES.block_size)
    
    # Cipher 객체 생성
    cipher = AES.new(key, AES.MODE_CBC, iv)
    
    # 암호화 수행
    ciphertext = cipher.encrypt(padded_data)
    
    # 암호문이 깨지지 않도록 Base64로 인코딩하여 반환
    return base64.b64encode(ciphertext).decode('utf-8')

def decrypt(enc_data, key, iv):
    """주어진 암호문을 AES-256-CBC 모드로 복호화한다."""
    
    # Base64로 디코딩하여 바이트 암호문으로 복원
    ciphertext = base64.b64decode(enc_data.encode('utf-8'))
    
    # Cipher 객체 생성 (암호화 시 사용한 키와 IV와 동일해야 함)
    cipher = AES.new(key, AES.MODE_CBC, iv)
    
    # 복호화 수행
    decrypted_padded_data = cipher.decrypt(ciphertext)
    
    # 패딩 제거
    decrypted_data = unpad(decrypted_padded_data, AES.block_size)
    
    # 바이트 데이터를 텍스트로 변환하여 반환
    return decrypted_data.decode('utf-8')

# -----------------
# 3. 실행 예시
# -----------------
original_text = "이것은 AES 암호화를 테스트하기 위한 비밀 메시지이다."

print(f"**원문 (Plaintext):** {original_text}")
print("-" * 50)

# 암호화
encrypted_message = encrypt(original_text, key, iv)
print(f"**암호문 (Ciphertext):** {encrypted_message}")
print("-" * 50)

# 복호화
decrypted_message = decrypt(encrypted_message, key, iv)
print(f"**복호화된 원문 (Decrypted):** {decrypted_message}")

# 키와 IV도 출력하여 이해를 돕는다 (실제 환경에서는 절대 출력하면 안 된다!)
print("\n--- Key & IV 정보 (참고용) ---")
print(f"Key (Base64): {base64.b64encode(key).decode()}")
print(f"IV (Base64):   {base64.b64encode(iv).decode()}")

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

# 콜라츠 추측 함수
def collatz(n):
    if n % 2 == 0:
        return n // 2
    else:
        return 3 * n + 1

# 콜라츠 추측을 따라 숫자들의 연결을 그래프 형태로 저장하는 함수
def generate_collatz_graph(start_num):
    G = nx.DiGraph()  # 방향 그래프 생성
    current = start_num
    while current != 1:
        next_num = collatz(current)
        G.add_edge(current, next_num)
        current = next_num
    G.add_edge(current, 1)  # 마지막 1을 추가
    return G

# 그래프 시각화 함수
def plot_collatz_graph(start_num):
    G = generate_collatz_graph(start_num)

    # 노드와 간선의 위치를 레이아웃으로 계산
    pos = nx.spring_layout(G, seed=42)

    # 그래프 그리기
    plt.figure(figsize=(10, 8))
    nx.draw(G, pos, with_labels=True, node_size=500, node_color='skyblue', font_size=12, font_weight='bold', edge_color='gray')
    plt.title(f"Collatz Conjecture for {start_num}", fontsize=16)
    plt.show()

# 실행 예시: 7부터 시작하는 콜라츠 추측 그래프 그리기
start_num = 7
plot_collatz_graph(start_num)
start_num = 11
plot_collatz_graph(start_num)

def collatz_sequence(n):
    """
    주어진 양의 정수 n에 대한 콜라츠 수열을 계산한다.
    """
    if n <= 0 or not isinstance(n, int):
        raise ValueError("입력은 1 이상의 양의 정수여야 한다.")
    
    sequence = [n]
    current = n
    
    # 현재 숫자가 1이 될 때까지 반복한다.
    while current != 1:
        if current % 2 == 0:
            # 짝수일 경우: 2로 나눈다.
            current = current // 2
        else:
            # 홀수일 경우: 3을 곱하고 1을 더한다.
            current = 3 * current + 1
        sequence.append(current)
        
    return sequence

# 예시: 6으로 시작하는 콜라츠 수열
start_number = 6
sequence = collatz_sequence(start_number)

print(f"**시작 숫자 {start_number}의 콜라츠 수열:**")
print(sequence)
print(f"**총 단계 수:** {len(sequence) - 1} 단계")

import networkx as nx
import matplotlib.pyplot as plt

def collatz_sequence(n):
    # (콜라츠 수열 생성 함수는 동일)
    if n <= 0 or not isinstance(n, int):
        raise ValueError("입력은 1 이상의 양의 정수여야 한다.")
    
    sequence = [n]
    current = n
    
    while current != 1:
        if current % 2 == 0:
            current = current // 2
        else:
            current = 3 * current + 1
        sequence.append(current)
        
    return sequence

def visualize_collatz_sequence(n):
    """
    주어진 숫자 n에 대한 콜라츠 수열을 생성하고 그래프로 시각화한다.
    (오류 수정: nx.draw_nodes -> nx.draw_networkx_nodes, 
              nx.draw_edges -> nx.draw_networkx_edges,
              nx.draw_labels -> nx.draw_networkx_labels)
    """
    try:
        sequence = collatz_sequence(n)
    except ValueError as e:
        print(f"오류: {e}")
        return

    # 그래프 객체 생성
    G = nx.DiGraph() 
    
    for i in range(len(sequence) - 1):
        u = sequence[i]
        v = sequence[i+1]
        G.add_edge(u, v)

    # --- 그래프 시각화 설정 ---
    
    # 노드 위치 결정 (예: 스프링 레이아웃)
    pos = nx.spring_layout(G, k=0.5, iterations=50) 
    
    plt.figure(figsize=(10, 6))
    plt.title(f"Collatz Sequence Graph starting from {n}")

    # 노드 그리기 (수정됨!)
    nx.draw_networkx_nodes(G, pos, node_size=1500, node_color='skyblue', alpha=0.9)
    
    # 엣지(화살표) 그리기 (수정됨!)
    nx.draw_networkx_edges(G, pos, edge_color='gray', width=2, arrowsize=20)
    
    # 노드 라벨(숫자) 그리기 (수정됨!)
    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')
    
    plt.show()

# 예시 실행: 13으로 시작하는 콜라츠 수열 시각화
start_number_vis = 13
print(f"**시작 숫자 {start_number_vis}의 콜라츠 수열:**")
print(collatz_sequence(start_number_vis))
visualize_collatz_sequence(start_number_vis)

In [ ]:
!pip uninstall pycrypto
!pip install pycryptodome

In [ ]:
!pip install fenics

In [ ]:
import numpy as np
from numpy.polynomial import Chebyshev, Polynomial

# 1. 윌킨슨 다항식의 근 (1 ~ 20)
roots = np.arange(1, 21)

# --- A. 일반 다항식 (Monomial Basis: 1, x, x^2...) ---
# domain 설정을 안 하면 기본적으로 1, x, x^2... 로 전개한다.
# 이 과정에서 엄청난 계수 팽창이 일어난다.
poly_mono = Polynomial.fromroots(roots)

# --- B. 체비쇼프 다항식 (Chebyshev Basis: T0, T1, T2...) ---
# domain=[1, 20]을 지정하면, 내부적으로 [-1, 1]로 매핑(window)하여 계산한다.
poly_cheb = Chebyshev.fromroots(roots, domain=[1, 20])

# 2. 계수 비교 (안정성 확인)
print("--- [일반 다항식 계수 (일부)] ---")
print(f"상수항: {poly_mono.coef[0]:.2e}")
print(f"x^19 계수: {poly_mono.coef[19]:.2e}")
# 결과: 계수가 10^19 승까지 커져서 부동소수점 정밀도를 위협한다.

print("\n--- [체비쇼프 다항식 계수 (일부)] ---")
# 체비쇼프 기저의 계수들 (c_k)
print(f"T_0 계수: {poly_cheb.coef[0]:.2e}")
print(f"T_19 계수: {poly_cheb.coef[19]:.2e}")
# 결과: 계수들이 훨씬 작고 안정적인 범위에 머무른다.

# 3. 값 계산 테스트 (x = 20.5 에서)
x_val = 20.5
x_val = 10.
true_val = np.prod(x_val - roots) # 실제 값 (직접 곱하기)

print(f"\n--- [값 계산 비교 (x={x_val})] ---")
print(f"True Value      : {true_val:.5e}")
print(f"Monomial Basis  : {poly_mono(x_val):.5e}")
print(f"Chebyshev Basis : {poly_cheb(x_val):.5e}")

In [ ]:
def bilinear_interpolation(x, y, points):
    '''
    x, y: 보간할 위치 (0~1 사이 정규화된 좌표라고 가정)
    points: [[Q11, Q12], [Q21, Q22]] 형태의 2x2 값 배열
            (좌하, 좌상, 우하, 우상 순서 배치에 주의 필요, 여기선 그리드 인덱스 기준)
            points[0][0] = (0,0) 값
            points[1][0] = (1,0) 값
            points[0][1] = (0,1) 값
            points[1][1] = (1,1) 값
    '''
    
    # 1. X축 방향 보간 (좌우)
    # R1: y=0 라인에서의 보간 (하단)
    r1 = (1 - x) * points[0][0] + x * points[1][0]
    # R2: y=1 라인에서의 보간 (상단)
    r2 = (1 - x) * points[0][1] + x * points[1][1]
    
    # 2. Y축 방향 보간 (상하)
    # R1과 R2 사이를 y 비율로 보간
    p = (1 - y) * r1 + y * r2
    
    return p

# 테스트 데이터 (값)
# 10  20
# 5   15
grid_values = [[5, 10], [15, 20]] 

# 정중앙 (0.5, 0.5) 값을 구하면?
# 예상: (5+10+15+20)/4 = 12.5
result = bilinear_interpolation(0.5, 0.5, grid_values)
print(f"Interpolated Value: {result}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def get_barycentric_weights(p, v1, v2, v3):
    """
    점 p(x, y)에 대한 삼각형 v1, v2, v3의 가중치(면적 좌표)를 계산
    """
    x, y = p
    x1, y1 = v1
    x2, y2 = v2
    x3, y3 = v3
    # 분모: 전체 삼각형 면적의 2배
    denom = (y2 - y3) * (x1 - x3) + (x3 - x2) * (y1 - y3)
    # 분자: 각 부분 삼각형 면적의 2배
    w1 = ((y2 - y3) * (x - x3) + (x3 - x2) * (y - y3)) / denom
    w2 = ((y3 - y1) * (x - x3) + (x1 - x3) * (y - y3)) / denom
    w3 = 1.0 - w1 - w2
    return w1, w2, w3
# --- 1. 데이터 설정 ---
V1 = np.array([0, 0])
V2 = np.array([10, 0])
V3 = np.array([0, 10])
# 각 꼭짓점의 스칼라 값 (예: 높이, 온도)
val_v1 = 0
val_v2 = 100
val_v3 = 50
# 보간할 점 P 위치
P = np.array([2, 2])
# --- 2. 계산 수행 ---
w1, w2, w3 = get_barycentric_weights(P, V1, V2, V3)
interpolated_val = w1 * val_v1 + w2 * val_v2 + w3 * val_v3
# --- 3. 시각화 (Matplotlib) ---
plt.figure(figsize=(8, 6))
# (1) 삼각형 그리기
# 닫힌 도형을 만들기 위해 시작점 V1을 리스트 마지막에 추가
triangle_x = [V1[0], V2[0], V3[0], V1[0]]
triangle_y = [V1[1], V2[1], V3[1], V1[1]]
plt.plot(triangle_x, triangle_y, 'k-', linewidth=2, label='Triangle boundary')
# (2) 점 P 찍기
plt.plot(P[0], P[1], 'ro', markersize=10, zorder=5, label='Point P')
# (3) 텍스트 주석: 꼭짓점 (Vertices)
# 위치를 조금씩 조정하여 점과 겹치지 않게 함
offset = 0.5
plt.text(V1[0]-offset, V1[1]-offset, f"V1\nVal={val_v1}", ha='right', va='top', fontsize=11, color='blue')
plt.text(V2[0]+offset, V2[1]-offset, f"V2\nVal={val_v2}", ha='left', va='top', fontsize=11, color='blue')
plt.text(V3[0]-offset, V3[1]+offset, f"V3\nVal={val_v3}", ha='right', va='bottom', fontsize=11, color='blue')
# (4) 텍스트 주석: 점 P (결과)
info_text = (f"  P({P[0]}, {P[1]})\n"
             f"  Result: {interpolated_val:.1f}\n"
             f"  Weights: ({w1:.2f}, {w2:.2f}, {w3:.2f})")
plt.text(P[0], P[1], info_text, fontsize=10, fontweight='bold', color='darkred', va='bottom')
# (5) 그래프 스타일 설정
plt.title("Barycentric interpolation visualization", fontsize=14)
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.axis('equal') # X, Y 축 비율을 같게 해서 삼각형 왜곡 방지
# 여백 조정 (텍스트 잘림 방지)
plt.xlim(min(triangle_x)-2, max(triangle_x)+4)
plt.ylim(min(triangle_y)-2, max(triangle_y)+2)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import Rbf # Radial Basis Function 라이브러리

def run_rbf_example():
    # 1. 데이터 생성 (무작위로 흩뿌려진 30개의 점)
    np.random.seed(42)
    x_data = np.random.rand(30) * 4 - 2 # -2 ~ 2
    y_data = np.random.rand(30) * 4 - 2 # -2 ~ 2
    
    # 참값 함수 (복잡한 2D 함수: Peaks function 유사)
    def true_func(x, y):
        return x * np.exp(-x**2 - y**2) + (x**2 + y**2) * 0.1

    z_data = true_func(x_data, y_data)

    # 2. 보간할 그리드 생성 (촘촘한 격자)
    ti = np.linspace(-2.0, 2.0, 100)
    XI, YI = np.meshgrid(ti, ti)

    # 3. RBF 보간 수행
    # function='multiquadric': 가장 일반적으로 쓰이는 RBF 커널 함수
    rbf_interp = Rbf(x_data, y_data, z_data, function='multiquadric')
    ZI = rbf_interp(XI, YI)

    # --- 시각화 ---
    fig = plt.figure(figsize=(12, 5))

    # (1) 원래 데이터 포인트와 참값 함수 형태
    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    ax1.scatter(x_data, y_data, z_data, c='r', s=50, label='Scattered Data', zorder=10)
    # 비교를 위해 참값 와이어프레임 살짝 표시
    ZT = true_func(XI, YI)
    ax1.plot_wireframe(XI, YI, ZT, color='gray', alpha=0.3)
    ax1.set_title("Scattered Input Data")
    ax1.legend()

    # (2) RBF로 복원된 곡면
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    surf = ax2.plot_surface(XI, YI, ZI, cmap='viridis', edgecolor='none', alpha=0.9)
    ax2.scatter(x_data, y_data, z_data, c='r', s=20, label='Data Points', zorder=10) # 데이터 위치 표시
    ax2.set_title("Reconstructed Surface (RBF)")
    fig.colorbar(surf, ax=ax2, shrink=0.5, aspect=10)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    run_rbf_example()

In [ ]:
import math

def trig_loss_demo():
    x = 1.0e-7 # 매우 작은 x
    
    # 1. 직접 계산 (위험)
    val_bad = 1.0 - math.cos(x)
    
    # 2. 반각 공식 이용 (안전)
    # 1 - cos(x) = 2 * sin^2(x/2)
    val_good = 2.0 * (math.sin(x/2.0))**2
    
    print(f"x = {x}")
    print(f"1. 직접 계산 (1 - cos x):   {val_bad}")
    print(f"2. 공식 변형 (2 sin^2 x/2): {val_good}")
    
    # 두 값의 비율 (1.0이어야 정상)
    print(f"비율 (Bad/Good): {val_bad/val_good}")

trig_loss_demo()

In [ ]:
def example_representation():
    print("--- 1. 표현 오차 (Representation Error) ---")
    result = 0.1 + 0.2
    
    print(f"0.1 + 0.2 = {result:.20f}")  # 소수점 20자리까지 출력
    print(f"0.1 + 0.2 == 0.3 : {result == 0.3}")
    
    # 해결책: math.isclose 사용
    import math
    print(f"math.isclose(result, 0.3) : {math.isclose(result, 0.3)}")

example_representation()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def compare_precision():
    # x를 10^-1 부터 10^-18 까지 점점 줄임
    x_values = np.logspace(-1, -18, 100)
    
    # 방법 1: 직접 뺄셈 (나쁜 방법)
    y_bad = 1.0 - np.cos(x_values)
    
    # 방법 2: 반각 공식 (좋은 방법) - 이를 참값(Ground Truth)으로 가정
    y_good = 2.0 * (np.sin(x_values / 2.0))**2
    
    # 상대 오차 계산 (Relative Error)
    # y_good이 0이 될 수 있으므로 안전장치 추가
    error = np.abs((y_bad - y_good) / (y_good + 1e-30))
    
    # 시각화
    plt.figure(figsize=(10, 6))
    plt.loglog(x_values, error, 'r-', linewidth=2, label='Relative error of (1 - cos x)')
    plt.axhline(y=1e-16, color='k', linestyle='--', label='Machine epsilon (Double)')
    
    plt.title("Catastrophic cancellation in 1 - cos(x)")
    plt.xlabel("x value (getting smaller ->)")
    plt.ylabel("Relative error")
    plt.gca().invert_xaxis() # x축을 작아지는 방향(오른쪽)으로
    plt.grid(True, which="both", ls="-")
    plt.legend()
    plt.show()

compare_precision()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def visualize_loss_of_significance():
    # 1. 설정: x를 10^-1부터 10^-18까지 로그 스케일로 생성
    # 숫자가 작아지는 과정을 보여주기 위해 범위를 넓게 잡음
    x = np.logspace(-1, -17, 1000)
    
    # 2. 계산 방법 두 가지
    
    # [Bad Method] 직접 뺄셈: 자릿수 상실 발생
    y_naive = 1.0 - np.cos(x)
    
    # [Good Method] 반각 공식: 수학적으로 동치이며 안정적
    # 1 - cos(x) = 2 * sin^2(x/2)
    y_robust = 2.0 * (np.sin(x / 2.0))**2
    
    # 3. 데이터 분석
    
    # (1) 상대 오차 (Relative Error)
    # Good Method를 참값(True Value)으로 가정
    # y_robust가 0일 경우 대비해 아주 작은 수(eps) 더함
    error = np.abs((y_naive - y_robust) / (y_robust + 1e-100))
    
    # (2) 정규화된 값 (Normalized Value)
    # 테일러 급수에 의해 1-cos(x)는 약 x^2/2 과 같음.
    # 따라서 (1-cos(x)) / (x^2/2) 의 값은 이론상 1.0이 나와야 함.
    # 이것이 1.0에서 얼마나 벗어나는지 확인.
    norm_factor = (x**2) / 2.0
    val_naive_norm = y_naive / norm_factor
    val_robust_norm = y_robust / norm_factor

    # --- 시각화 ---
    fig, ax = plt.subplots(2, 1, figsize=(10, 12))
    
    # 첫 번째 그래프: 상대 오차
    ax[0].loglog(x, error, 'r-', label='Direct: 1 - cos(x)', alpha=0.7)
    ax[0].loglog(x, np.zeros_like(x) + 1e-16, 'k--', label='Machine epsilon (Double)', alpha=0.5)
    
    ax[0].set_title("1. Relative error increase (Loss of significance)", fontsize=14, fontweight='bold')
    ax[0].set_xlabel("x (Log scale)", fontsize=18)
    ax[0].set_ylabel("Relative error", fontsize=18)
    ax[0].invert_xaxis() # x축을 큰 수 -> 작은 수 방향으로 뒤집음
    ax[0].grid(True, which="both", alpha=0.3)
    ax[0].legend(fontsize=12)
    
    # 설명 텍스트
    ax[0].text(1e-9, 1e-2, "Error explodes as x -> 0", color='red', fontsize=12, fontweight='bold')

    # 두 번째 그래프: 정규화된 값 비교
    ax[1].semilogx(x, val_naive_norm, 'r.', markersize=2, label='Direct: 1 - cos(x)')
    ax[1].semilogx(x, val_robust_norm, 'b-', linewidth=2, label='Robust: 2*sin^2(x/2)')
    
    ax[1].set_title("2. Normalized value behavior (Ideal = 1.0)", fontsize=14, fontweight='bold')
    ax[1].set_xlabel("x (Log scale)", fontsize=18)
    ax[1].set_ylabel("Calculated value / (x^2/2)", fontsize=18)
    ax[1].set_ylim(0, 2.0) # 1.0 주변을 확대해서 보기 위함
    ax[1].invert_xaxis()
    ax[1].grid(True, which="both", alpha=0.3)
    ax[1].legend(fontsize=12)
    
    # 설명 텍스트
    ax[1].text(1e-6, 1.5, "Discrete steps & collapse", color='red', fontsize=12, fontweight='bold')
    ax[1].text(1e-6, 1.1, "Stable (1.0)", color='blue', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    visualize_loss_of_significance()

In [ ]:
def example_absorption():
    print("\n--- 2. 흡수 현상 (Absorption) ---")
    large_num = 1.0e16   # 10의 16승 (매우 큰 수)
    small_num = 1.0      # 1 (상대적으로 매우 작은 수)
    
    result = large_num + small_num
    
    print(f"Large : {large_num:.1f}")
    print(f"Result: {result:.1f}")
    
    if result == large_num:
        print("-> 오류 발생! 1.0이 더해지지 않고 사라졌다.")
    else:
        print("-> 정상 계산됨.")

example_absorption()

In [ ]:
def example_associativity():
    print("\n--- 3. 결합 법칙 위배 ---")
    a = 1.0e16
    b = -1.0e16
    c = 1.0
    
    # Case 1: (큰 수 + 작은 수) 먼저 계산 -> 작은 수 흡수됨 -> 그 뒤에 큰 수 뺌
    case1 = (a + c) + b  
    
    # Case 2: (큰 수 - 큰 수) 먼저 계산 -> 0이 됨 -> 그 뒤에 작은 수 더함
    case2 = (a + b) + c  
    
    print(f"Case 1 ((a+c)+b): {case1} (오답)")
    print(f"Case 2 ((a+b)+c): {case2} (정답)")

example_associativity()

In [ ]:
def example_accumulation():
    print("\n--- 4. 누적 오차 (Accumulation) ---")
    sum_val = 0.0
    step = 0.1
    iterations = 1000000 # 100만 번
    
    for _ in range(iterations):
        sum_val += step
        
    expected = 100000.0 # 0.1 * 100만
    error = abs(sum_val - expected)
    
    print(f"계산된 합: {sum_val:.10f}")
    print(f"기대값   : {expected:.10f}")
    print(f"누적된 오차: {error:.10f}")

example_accumulation()

In [ ]:
def propagation_demo():
    print("--- 연쇄 오차 (Error propagation) 데모 ---")
    print("점화식: x_{n+1} = 10 * x_n - 9")
    print("초기값 x0 = 1.0 (이론상 결과는 항상 1.0 이어야 함)\n")
    
    # Case 1: 완벽한 1.0 (파이썬 정수/실수 처리상 1.0은 정확히 표현됨)
    # 하지만 아주 미세한 노이즈를 섞어봄 (예: 센서 측정 오차라고 가정)
    epsilon = 1e-15 # 0.000000000000001
    x_true = 1.0
    x_noise = 1.0 + epsilon # 아주 미세하게 틀린 값
    print(f"{'Step':<5} {'Ideal Value':<15} {'Propagated':<20} {'Error (x 10^n)':<20}")
    print("-" * 65)
    for i in range(21): # 20번 반복
        # 오차 출력
        err = abs(x_noise - x_true)
        print(f"{i:<5} {x_true:<15.1f} {x_noise:<20.8f} {err:.2e}")
        # 점화식 업데이트
        # x_new = 10 * x_old - 9
        x_true = 10 * x_true - 9      # 참값 유지 (논리적)
        x_noise = 10 * x_noise - 9    # 오차 전파
        # 만약 오차가 너무 커지면 중단
        if err > 100:
            print("\n!!! 오차가 너무 커져서 중단한다 !!!")
            break

propagation_demo()

In [ ]:
import numpy as np

# 1000x1000 행렬 생성 (데이터 타입: float64 -> Double Precision)
A = np.random.rand(1000, 1000)
B = np.random.rand(1000, 1000)
C = np.zeros((1000, 1000))

# 1. 일반적인 행렬 곱 (C = A * B)
# 내부적으로 DGEMM이 alpha=1, beta=0 으로 호출됨
C = np.dot(A, B) 

# 2. DGEMM의 원래 수식 형태 (C = 1.0 * A*B + 1.0 * C)
# 기존 C값에 A*B를 더함
C = 1.0 * np.dot(A, B) + 1.0 * C

In [ ]:
import shutil
import subprocess


def hassanat_dist_metric(x1, x2):
    tmp = 0.
    for i in range(len(x1)):
        min_ = min(x1[i], x2[i])
        max_ = max(x1[i], x2[i])
        if min_ >= 0:
            di = 1-((1+min_)/(1+max_))
        else:
            di = 1-((1+min_ + np.abs(min_))/(1+max_+np.abs(min_)))
        tmp = tmp+di
    return tmp


def hassanat_dist_metric1(x1, x2):
    tmp = 0.
    for i in range(len(x1)):
        min_ = min(x1[i], x2[i])
        max_ = max(x1[i], x2[i])
        if min_ >= 0:
            di = 1-(1+min_)/(1+max_)
        else:
            di = 1-1/(1+max_+np.abs(min_))
        tmp = tmp+di
    return tmp


def hassanat_dist_metric2(x1, x2):
    tmp = 1.
    for i in range(len(x1)):
        min_ = min(x1[i], x2[i])
        max_ = max(x1[i], x2[i])
        if min_ >= 0:
            di = 1-np.abs(x1[i]-x2[i])/(1+max_)
        else:
            di = 1-np.abs(x1[i]-x2[i])/(1+max_+np.abs(min_))
        tmp = tmp-di
    return tmp


def unified_distance(x1, x2):
    """모든 로직에서 공통으로 사용할 고속 Hassanat 거리 함수"""
    mins = np.minimum(x1, x2)
    maxs = np.maximum(x1, x2)
    eps = 1e-15

    di = np.where(
        mins >= 0,
        1.0 - (1.0 + mins) / (1.0 + maxs + eps),
        1.0 - 1.0 / (1.0 + maxs + np.abs(mins) + eps)
    )
    return np.sum(di, axis=-1)


def csadistance(x, y):
#   tmp = hassanat_dist_metric(x, y)
    tmp = unified_distance(x, y)
    return tmp


def repulsion_opt(work):
    # 행렬 형태의 거리를 한 번에 계산
    dist_matrix = unified_distance(work[:, np.newaxis, :], work[np.newaxis, :, :])
    iu1 = np.tril_indices(work.shape[0], k=-1)
    return np.sum(1.0 / (dist_matrix[iu1] + 1e-8))


def coulomb_opt(npop, ndim, old, lw, up):
    # 리스트로 들어온 lw, up을 넘파이 배열로 변환 (연산 효율성 및 브로드캐스팅용)
    lw = np.array(lw)
    up = np.array(up)
    
    new = old.copy()
    tmp0 = repulsion_opt(new)
    range_width = up - lw
    
    for _ in range(npop + 10):
        # 1. Noise 생성 시 range_width(ndim 크기)가 npop 행과 올바르게 곱해지도록 함
        # (npop, ndim) 형태의 난수에 (ndim,) 형태의 range_width가 브로드캐스팅됨
        noise = range_width * (np.random.random((npop, ndim)) - 0.5) / 4.0
        work = new + noise
        # 첫 번째 개체는 유지 (엘리트 보존 등)
        work[0, :] = old[0, :]
        # 2. 경계값 검사 및 재할당
        # out_mask는 (npop, ndim) 형태의 불리언 행렬
        out_mask = (work < lw) | (work > up)
        if np.any(out_mask):
            # (npop, ndim) 형태의 전체 무작위 행렬 생성 후 mask된 부분만 할당
            random_fill = lw + range_width * np.random.random((npop, ndim))
            work[out_mask] = random_fill[out_mask]
        tmp = repulsion_opt(work)
        if tmp < tmp0:
            tmp0 = tmp
            new = work.copy()
    return new


def save_dynamic_safely(filename, ostatic, static, odynamic, dynamic):
    tmp_filename = filename + ".tmp"
    bak_filename = filename + ".bak"
    try:
        with open(tmp_filename, 'wb') as f:
            np.save(f, ostatic)
            np.save(f, static)
            np.save(f, odynamic)
            np.save(f, dynamic)
        if os.path.isfile(filename):
            shutil.copy2(filename, bak_filename)
        os.replace(tmp_filename, filename)
    except Exception as e:
        print(f"[저장 오류] 데이터 저장 중 문제가 발생했습니다: {e}")
        if os.path.isfile(tmp_filename):
            os.remove(tmp_filename) # 실패한 임시 파일 삭제


def gen_directories(npop, pwd0):
    src_pbs = os.path.join(pwd0, 'CSA_SOLDIER.pbs')
    src_input = os.path.join(pwd0, 'INPUT')
    for i in range(npop):
        dir_name = str(i).zfill(4)
        target_dir = os.path.join(pwd0, dir_name)
        os.makedirs(target_dir, exist_ok=True)
        shutil.copy2(src_pbs, target_dir)
        shutil.copy2(src_input, target_dir)


def del_directories(npop, pwd0):
    for i in range(npop):
        target_dir = os.path.join(pwd0, str(i).zfill(4))
        if os.path.isdir(target_dir):
            shutil.rmtree(target_dir, ignore_errors=True)


def get_asubmission(pwd0, xvector, idirectory):
    target_dir = os.path.join(pwd0, str(idirectory).zfill(4))
    input_path = os.path.join(target_dir, 'input.txt')
    status_path = os.path.join(target_dir, 'STATUS')
    write_an_input(xvector, input_path)
    time.sleep(0.01)
#   scheduler_cmd = ['qsub', 'CSA_SOLDIER.pbs']
    scheduler_cmd = ['sbatch', 'CSA_SOLDIER.pbs']
    try:
        subprocess.run(scheduler_cmd, cwd=target_dir, check=True)
    except subprocess.CalledProcessError as e:
        print(f"Error submitting job in {target_dir}: {e}")
        return # 에러 발생 시 STATUS 업데이트를 막으려면 리턴
    with open(status_path, 'w') as f:
        f.write("ING\n")


def get_asolution(npop, ndim, pwd0):
    check_order = list(range(npop))
    np.random.shuffle(check_order)
    while True:
        for idirectory0 in check_order:
            folder_path = os.path.join(pwd0, str(idirectory0).zfill(4))
            fname = os.path.join(folder_path, 'STOP')
            gname = os.path.join(folder_path, 'STATUS')
            hname = os.path.join(folder_path, 'output.txt')
            if not os.path.isfile(fname):
                continue
            if os.path.isfile(gname):
                try:
                    with open(gname, 'r') as afile:
                        # read().split()은 모든 공백/줄바꿈을 제거하고 단어 리스트만 반환합니다.
                        # 예: "ING\nDONE\n\n" -> ['ING', 'DONE']
                        words = afile.read().split()
                    # 단어가 존재하고, 마지막 단어가 done 인지 확인
                    if words and words[-1].lower() == 'done':
                        if os.path.isfile(hname):
                            try:
                                xvector, e = get_an_output(hname, ndim)
                                os.remove(fname)
                                os.remove(hname)
                                return xvector, e, idirectory0
                            except (ValueError, IndexError, IOError):
                                continue
                except Exception:
                    continue
        time.sleep(0.5)
